In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
GPU available: False


In [ ]:
!git clone https://github.com/ming024/FastSpeech2.git

Cloning into 'FastSpeech2'...
remote: Enumerating objects: 994, done.
remote: Total 994 (delta 0), reused 0 (delta 0), pack-reused 994 (from 1)
Receiving objects: 100% (994/994), 330.32 MiB | 9.98 MiB/s, done.
Resolving deltas: 100% (162/162), done.


In [ ]:
%cd FastSpeech2

/content/FastSpeech2


In [ ]:
!ls

audio	     hifigan	 model		    README.md	      transformer
config	     img	 prepare_align.py   requirements.txt  utils
dataset.py   index.html  preprocessed_data  synthesize.py
demo	     lexicon	 preprocessor	    text
evaluate.py  LICENSE	 preprocess.py	    train.py


In [ ]:
!sed -n '1,200p' README.md

# FastSpeech 2 - PyTorch Implementation

This is a PyTorch implementation of Microsoft's text-to-speech system [**FastSpeech 2: Fast and High-Quality End-to-End Text to Speech**](https://arxiv.org/abs/2006.04558v1). 
This project is based on [xcmyz's implementation](https://github.com/xcmyz/FastSpeech) of FastSpeech. Feel free to use/modify the code.

There are several versions of FastSpeech 2.
This implementation is more similar to [version 1](https://arxiv.org/abs/2006.04558v1), which uses F0 values as the pitch features.
On the other hand, pitch spectrograms extracted by continuous wavelet transform are used as the pitch features in the [later versions](https://arxiv.org/abs/2006.04558).

![](./img/model.png)

# Updates
- 2021/7/8: Release the checkpoint and audio samples of a multi-speaker English TTS model trained on LibriTTS
- 2021/2/26: Support English and Mandarin TTS
- 2021/2/26: Support multi-speaker TTS (AISHELL-3 and LibriTTS)
- 2021/2/26: Support MelGAN and HiFi-GAN vocode

In [ ]:
!cat requirements.txt

g2p-en == 2.1.0
inflect == 4.1.0
librosa == 0.7.2
matplotlib == 3.2.2
numba == 0.48
numpy == 1.19.0
pypinyin==0.39.0
pyworld == 0.2.10
PyYAML==5.4.1
scikit-learn==0.23.2
scipy == 1.5.0
soundfile==0.10.3.post1
tensorboard == 2.2.2
tgt == 1.4.4
torch == 1.7.0
tqdm==4.46.1
unidecode == 1.1.1

In [ ]:
!find preprocessor -maxdepth 2 -type f

preprocessor/preprocessor.py
preprocessor/libritts.py
preprocessor/ljspeech.py
preprocessor/aishell3.py


In [ ]:
# These files handle converting text into model-readable representations
!find text -maxdepth 2 -type f

text/symbols.py
text/cleaners.py
text/numbers.py
text/cmudict.py
text/pinyin.py
text/__init__.py


In [ ]:
# Check how the repository prepares audio and text data
!sed -n '1,220p' preprocessor/preprocessor.py

import os
import random
import json

import tgt
import librosa
import numpy as np
import pyworld as pw
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import audio as Audio


class Preprocessor:
    def __init__(self, config):
        self.config = config
        self.in_dir = config["path"]["raw_path"]
        self.out_dir = config["path"]["preprocessed_path"]
        self.val_size = config["preprocessing"]["val_size"]
        self.sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
        self.hop_length = config["preprocessing"]["stft"]["hop_length"]

        assert config["preprocessing"]["pitch"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        assert config["preprocessing"]["energy"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        self.pitch_phoneme_averaging = (
            config["preprocessing"]["pitch"]["feature"] == 

In [ ]:
# Inspect the existing English text-processing logic
!find text -maxdepth 2 -type f

text/symbols.py
text/cleaners.py
text/numbers.py
text/cmudict.py
text/pinyin.py
text/__init__.py


In [ ]:
!cat config/LJSpeech/preprocess.yaml

dataset: "LJSpeech"

path:
  corpus_path: "/home/ming/Data/LJSpeech-1.1"
  lexicon_path: "lexicon/librispeech-lexicon.txt"
  raw_path: "./raw_data/LJSpeech"
  preprocessed_path: "./preprocessed_data/LJSpeech"

preprocessing:
  val_size: 512
  text:
    text_cleaners: ["english_cleaners"]
    language: "en"
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: 80
    mel_fmin: 0
    mel_fmax: 8000 # please set to 8000 for HiFi-GAN vocoder, set to null for MelGAN vocoder
  pitch:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True
  energy:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True


In [ ]:
!cat config/LJSpeech/model.yaml

transformer:
  encoder_layer: 4
  encoder_head: 2
  encoder_hidden: 256
  decoder_layer: 6
  decoder_head: 2
  decoder_hidden: 256
  conv_filter_size: 1024
  conv_kernel_size: [9, 1]
  encoder_dropout: 0.2
  decoder_dropout: 0.2

variance_predictor:
  filter_size: 256
  kernel_size: 3
  dropout: 0.5

variance_embedding:
  pitch_quantization: "linear" # support 'linear' or 'log', 'log' is allowed only if the pitch values are not normalized during preprocessing
  energy_quantization: "linear" # support 'linear' or 'log', 'log' is allowed only if the energy values are not normalized during preprocessing
  n_bins: 256

# gst:
#   use_gst: False
#   conv_filters: [32, 32, 64, 64, 128, 128]
#   gru_hidden: 128
#   token_size: 128
#   n_style_token: 10
#   attn_head: 4

multi_speaker: False

max_seq_len: 1000

vocoder:
  model: "HiFi-GAN" # support 'HiFi-GAN', 'MelGAN'
  speaker: "LJSpeech" # support  'LJSpeech', 'universal'


In [ ]:
!cat config/LJSpeech/train.yaml

path:
  ckpt_path: "./output/ckpt/LJSpeech"
  log_path: "./output/log/LJSpeech"
  result_path: "./output/result/LJSpeech"
optimizer:
  batch_size: 16
  betas: [0.9, 0.98]
  eps: 0.000000001
  weight_decay: 0.0
  grad_clip_thresh: 1.0
  grad_acc_step: 1
  warm_up_step: 4000
  anneal_steps: [300000, 400000, 500000]
  anneal_rate: 0.3
step:
  total_step: 900000
  log_step: 100
  synth_step: 1000
  val_step: 1000
  save_step: 100000


In [ ]:
!sed -n '1,200p' text/symbols.py

""" from https://github.com/keithito/tacotron """

"""
Defines the set of symbols used in text input to the model.

The default is a set of ASCII characters that works well for English or text that has been run through Unidecode. For other data, you can modify _characters. See TRAINING_DATA.md for details. """

from text import cmudict, pinyin

_pad = "_"
_punctuation = "!'(),.:;? "
_special = "-"
_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
_silences = ["@sp", "@spn", "@sil"]

# Prepend "@" to ARPAbet symbols to ensure uniqueness (some are the same as uppercase letters):
_arpabet = ["@" + s for s in cmudict.valid_symbols]
_pinyin = ["@" + s for s in pinyin.valid_symbols]

# Export all symbols:
symbols = (
    [_pad]
    + list(_special)
    + list(_punctuation)
    + list(_letters)
    + _arpabet
    + _pinyin
    + _silences
)


In [ ]:
!sed -n '1,220p' text/__init__.py

""" from https://github.com/keithito/tacotron """
import re
from text import cleaners
from text.symbols import symbols


# Mappings from symbol to numeric ID and vice versa:
_symbol_to_id = {s: i for i, s in enumerate(symbols)}
_id_to_symbol = {i: s for i, s in enumerate(symbols)}

# Regular expression matching text enclosed in curly braces:
_curly_re = re.compile(r"(.*?)\{(.+?)\}(.*)")


def text_to_sequence(text, cleaner_names):
    """Converts a string of text to a sequence of IDs corresponding to the symbols in the text.

    The text can optionally have ARPAbet sequences enclosed in curly braces embedded
    in it. For example, "Turn left on {HH AW1 S S T AH0 N} Street."

    Args:
      text: string to convert to a sequence
      cleaner_names: names of the cleaner functions to run the text through

    Returns:
      List of integers corresponding to the symbols in the text
    """
    sequence = []

    # Check for curly braces and treat their contents as ARPAbet:
    while len

In [ ]:
!sed -n '1,220p' text/cleaners.py

""" from https://github.com/keithito/tacotron """

'''
Cleaners are transformations that run over the input text at both training and eval time.

Cleaners can be selected by passing a comma-delimited list of cleaner names as the "cleaners"
hyperparameter. Some cleaners are English-specific. You'll typically want to use:
  1. "english_cleaners" for English text
  2. "transliteration_cleaners" for non-English text that can be transliterated to ASCII using
     the Unidecode library (https://pypi.python.org/pypi/Unidecode)
  3. "basic_cleaners" if you do not want to transliterate (in this case, you should also update
     the symbols in symbols.py to match your data).
'''


# Regular expression matching whitespace:
import re
from unidecode import unidecode
from .numbers import normalize_numbers
_whitespace_re = re.compile(r'\s+')

# List of (regular expression, replacement) pairs for abbreviations:
_abbreviations = [(re.compile('\\b%s\\.' % x[0], re.IGNORECASE), x[1]) for x in [
    ('mrs

In [ ]:
# Check how the model converts text into input IDs
!grep -R "text_to_sequence" -n .

# Check where the model defines its vocabulary size
!grep -R "len(symbols)" -n model text


./dataset.py:8:from text import text_to_sequence
./dataset.py:37:        phone = np.array(text_to_sequence(self.text[idx], self.cleaners))
./dataset.py:171:        phone = np.array(text_to_sequence(self.text[idx], self.cleaners))
./text/__init__.py:15:def text_to_sequence(text, cleaner_names):
./synthesize.py:15:from text import text_to_sequence
./synthesize.py:51:        text_to_sequence(
./synthesize.py:79:        text_to_sequence(


In [ ]:
!grep -R "n_src_vocab\|vocab_size\|src_vocab" -n model

In [ ]:
!sed -n '1,100p' dataset.py

import json
import math
import os

import numpy as np
from torch.utils.data import Dataset

from text import text_to_sequence
from utils.tools import pad_1D, pad_2D


class Dataset(Dataset):
    def __init__(
        self, filename, preprocess_config, train_config, sort=False, drop_last=False
    ):
        self.dataset_name = preprocess_config["dataset"]
        self.preprocessed_path = preprocess_config["path"]["preprocessed_path"]
        self.cleaners = preprocess_config["preprocessing"]["text"]["text_cleaners"]
        self.batch_size = train_config["optimizer"]["batch_size"]

        self.basename, self.speaker, self.text, self.raw_text = self.process_meta(
            filename
        )
        with open(os.path.join(self.preprocessed_path, "speakers.json")) as f:
            self.speaker_map = json.load(f)
        self.sort = sort
        self.drop_last = drop_last

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        basename = self.bas

In [ ]:
!grep -R "symbols" -n model

In [ ]:
!sed -n '1,180p' model/fastspeech2.py

import os
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformer import Encoder, Decoder, PostNet
from .modules import VarianceAdaptor
from utils.tools import get_mask_from_lengths


class FastSpeech2(nn.Module):
    """ FastSpeech2 """

    def __init__(self, preprocess_config, model_config):
        super(FastSpeech2, self).__init__()
        self.model_config = model_config

        self.encoder = Encoder(model_config)
        self.variance_adaptor = VarianceAdaptor(preprocess_config, model_config)
        self.decoder = Decoder(model_config)
        self.mel_linear = nn.Linear(
            model_config["transformer"]["decoder_hidden"],
            preprocess_config["preprocessing"]["mel"]["n_mel_channels"],
        )
        self.postnet = PostNet()

        self.speaker_emb = None
        if model_config["multi_speaker"]:
            with open(
                os.path.join(
                    preprocess_config["path"]["preprocessed_path"

In [ ]:
!sed -n '1,180p' transformer/Models.py

import torch
import torch.nn as nn
import numpy as np

import transformer.Constants as Constants
from .Layers import FFTBlock
from text.symbols import symbols


def get_sinusoid_encoding_table(n_position, d_hid, padding_idx=None):
    """ Sinusoid position encoding table """

    def cal_angle(position, hid_idx):
        return position / np.power(10000, 2 * (hid_idx // 2) / d_hid)

    def get_posi_angle_vec(position):
        return [cal_angle(position, hid_j) for hid_j in range(d_hid)]

    sinusoid_table = np.array(
        [get_posi_angle_vec(pos_i) for pos_i in range(n_position)]
    )

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # dim 2i
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # dim 2i+1

    if padding_idx is not None:
        # zero vector for padding dimension
        sinusoid_table[padding_idx] = 0.0

    return torch.FloatTensor(sinusoid_table)


class Encoder(nn.Module):
    """ Encoder """

    def __init__(self, config):
        

In [ ]:
!pip install -q datasets[audio] huggingface_hub soundfile

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import whoami

account = whoami()
print("Connected Hugging Face account:", account["name"])

Connected Hugging Face account: madhuri0625


In [ ]:
from datasets import load_dataset

english_dataset = load_dataset(
    "ai4bharat/Svarah",
    split="test",
    streaming=True,
    token=True
)

print("Successfully connected!")
print(english_dataset.features)

README.md:   0%|          | 0.00/4.30k [00:00<?, ?B/s]

Successfully connected!
{'audio_filepath': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'duration': Value('float64'), 'text': Value('string'), 'gender': Value('string'), 'age-group': Value('string'), 'primary_language': Value('string'), 'native_place_state': Value('string'), 'native_place_district': Value('string'), 'highest_qualification': Value('string'), 'job_category': Value('string'), 'occupation_domain': Value('string')}


In [ ]:
# Get one sample from the English dataset
sample = next(iter(english_dataset))

print("Transcript:")
print(sample["text"])

print("\nDuration:")
print(sample["duration"], "seconds")

print("\nSpeaker information:")
print("Gender:", sample["gender"])
print("Age group:", sample["age-group"])
print("Primary language:", sample["primary_language"])

print("\nAudio information:")
print("Sampling rate:", sample["audio_filepath"]["sampling_rate"])
print("Audio shape:", sample["audio_filepath"]["array"].shape)

Transcript:
Our district Hasana was the land of the great Hoysala dynasty who ruled over this region from 11th to 14th century CE.

Duration:
8.7595 seconds

Speaker information:
Gender: Female
Age group: 18-30
Primary language: Kannada

Audio information:
Sampling rate: 16000
Audio shape: (140152,)


In [ ]:
from itertools import islice

# Inspect the first 10 English recordings
for index, sample in enumerate(islice(english_dataset, 10), start=1):
    print(f"\nSample {index}")
    print("Duration:", round(sample["duration"], 2), "seconds")
    print("Primary language:", sample["primary_language"])
    print("Transcript:", sample["text"])


Sample 1
Duration: 8.76 seconds
Primary language: Kannada
Transcript: Our district Hasana was the land of the great Hoysala dynasty who ruled over this region from 11th to 14th century CE.

Sample 2
Duration: 3.1 seconds
Primary language: Kannada
Transcript: There is a gradual shift in the mentality

Sample 3
Duration: 0.45 seconds
Primary language: Dogri
Transcript: Wow

Sample 4
Duration: 2.72 seconds
Primary language: Maithili
Transcript: Nipah virus belongs to  the

Sample 5
Duration: 3.69 seconds
Primary language: Maithili
Transcript: How much money is left in my Arunachal Pradesh Rural Bank account?

Sample 6
Duration: 8.51 seconds
Primary language: Bengali
Transcript: Govinda has received 12 Filmfare Award nominations and 4 Zee Cine Awards.

Sample 7
Duration: 1.73 seconds
Primary language: Bodo
Transcript: There are many big rivers

Sample 8
Duration: 3.95 seconds
Primary language: Bodo
Transcript: where it is joined by other branches to form the Brahmaputra

Sample 9
Duration

In [ ]:
print("Audio object type:", type(sample["audio_filepath"]))

print("\nAudio object:")
print(sample["audio_filepath"])

print("\nDataset columns:")
print(english_dataset.features)

Audio object type: <class 'datasets.features._torchcodec.AudioDecoder'>

Audio object:

Dataset columns:
{'audio_filepath': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'duration': Value('float64'), 'text': Value('string'), 'gender': Value('string'), 'age-group': Value('string'), 'primary_language': Value('string'), 'native_place_state': Value('string'), 'native_place_district': Value('string'), 'highest_qualification': Value('string'), 'job_category': Value('string'), 'occupation_domain': Value('string')}


In [ ]:
from itertools import islice
from collections import Counter

language_counts = Counter()

for record in islice(english_dataset, 100):
    language_counts[record["primary_language"]] += 1

print("Primary language distribution:")

for language, count in language_counts.most_common():
    print(f"{language}: {count}")

Primary language distribution:
Maithili: 9
Kashmiri: 9
Kannada: 8
Nepali: 8
Urdu: 8
Malayalam: 7
Odia: 7
Telugu: 6
Marathi: 5
Konkani: 5
Tamil: 5
Bengali: 4
Hindi: 4
Punjabi: 4
Gujarati: 4
Dogri: 2
Bodo: 2
Assamese: 2
Sindhi: 1


In [ ]:
from datasets import load_dataset

# Load the metadata without decoding audio
english_metadata = load_dataset(
    "ai4bharat/Svarah",
    split="test",
    streaming=True,
    token=True,
)

total_duration = 0
total_samples = 0

for record in english_metadata:
    total_duration += record["duration"]
    total_samples += 1

print("Total recordings:", total_samples)
print("Total duration in hours:", round(total_duration / 3600, 2))

Total recordings: 6656
Total duration in hours: 9.61


In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import os
import pandas as pd
from datasets import load_dataset

# Create folders for the assignment
project_folder = "/content/drive/MyDrive/FastSpeech2_Assignment"
metadata_folder = os.path.join(project_folder, "metadata")
audio_folder = os.path.join(project_folder, "raw_data")

os.makedirs(metadata_folder, exist_ok=True)
os.makedirs(audio_folder, exist_ok=True)

# Load the Svarah dataset
dataset = load_dataset(
    "ai4bharat/Svarah",
    split="test",
    streaming=True,
    token=True
)

# Collect metadata for every recording
recording_details = []

for index, record in enumerate(dataset):
    recording_details.append({
        "recording_id": index,
        "duration": record["duration"],
        "text": record["text"],
        "gender": record["gender"],
        "age_group": record["age-group"],
        "primary_language": record["primary_language"]
    })

    if (index + 1) % 1000 == 0:
        print(f"Processed {index + 1} recordings")

# Save metadata as a CSV file
metadata_df = pd.DataFrame(recording_details)

metadata_path = os.path.join(metadata_folder, "svarah_metadata.csv")
metadata_df.to_csv(metadata_path, index=False)

print("\nMetadata collection completed!")
print("Total recordings:", len(metadata_df))
print("Total duration:", round(metadata_df["duration"].sum() / 3600, 2), "hours")
print("Saved to:", metadata_path)

Processed 1000 recordings
Processed 2000 recordings
Processed 3000 recordings
Processed 4000 recordings
Processed 5000 recordings
Processed 6000 recordings

Metadata collection completed!
Total recordings: 6656
Total duration: 9.61 hours
Saved to: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/svarah_metadata.csv


In [ ]:
import pandas as pd

# Load the metadata saved in Google Drive
metadata_path = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata/svarah_metadata.csv"
metadata_df = pd.read_csv(metadata_path)

# Check duration statistics
print("DURATION SUMMARY")
print(metadata_df["duration"].describe())

# Count recordings by duration
print("\nRECORDINGS BY DURATION")
print("Below 1 second:", (metadata_df["duration"] < 1).sum())
print("1–2 seconds:", ((metadata_df["duration"] >= 1) & (metadata_df["duration"] < 2)).sum())
print("2–5 seconds:", ((metadata_df["duration"] >= 2) & (metadata_df["duration"] < 5)).sum())
print("5–10 seconds:", ((metadata_df["duration"] >= 5) & (metadata_df["duration"] < 10)).sum())
print("10 seconds or more:", (metadata_df["duration"] >= 10).sum())

# Check transcript quality
empty_transcripts = metadata_df["text"].isna() | (metadata_df["text"].str.strip() == "")
print("\nTRANSCRIPT CHECK")
print("Empty transcripts:", empty_transcripts.sum())
print("Duplicate transcript rows:", metadata_df["text"].duplicated().sum())

# Check the duration available after basic filtering
usable_df = metadata_df[
    (metadata_df["duration"] >= 2) &
    (~empty_transcripts)
]

print("\nAFTER BASIC FILTERING")
print("Recordings:", len(usable_df))
print("Total duration:", round(usable_df["duration"].sum() / 3600, 2), "hours")

DURATION SUMMARY
count    6656.000000
mean        5.200120
std         4.568625
min         0.098000
25%         1.815234
50%         4.208344
75%         7.200000
max        29.871063
Name: duration, dtype: float64

RECORDINGS BY DURATION
Below 1 second: 1215
1–2 seconds: 561
2–5 seconds: 2076
5–10 seconds: 1994
10 seconds or more: 810

TRANSCRIPT CHECK
Empty transcripts: 0
Duplicate transcript rows: 1071

AFTER BASIC FILTERING
Recordings: 4880
Total duration: 9.18 hours


In [ ]:
# Display 20 recordings with their transcript and duration
sample_df = usable_df.sample(n=20, random_state=42)

for _, row in sample_df.iterrows():
    print(f"ID: {row['recording_id']}")
    print(f"Duration: {row['duration']:.2f} seconds")
    print(f"Primary language: {row['primary_language']}")
    print(f"Transcript: {row['text']}")
    print("-" * 80)

ID: 5083
Duration: 3.40 seconds
Primary language: Bodo
Transcript: canoeing is the central purpose of the activity.
--------------------------------------------------------------------------------
ID: 991
Duration: 9.63 seconds
Primary language: Kashmiri
Transcript: So every shrine has its own day which is celebrated by people when they come across and visit the shrine
--------------------------------------------------------------------------------
ID: 406
Duration: 6.88 seconds
Primary language: Hindi
Transcript: My favorite quote is from a classic play named A Woman of No Importance by Oscar Wilde.
--------------------------------------------------------------------------------
ID: 2859
Duration: 15.76 seconds
Primary language: Dogri
Transcript: Major challenges facing in our states regarding business is first e-commerce. Second one is a negative approach or lesser approach of administration towards retail business.
--------------------------------------------------------------------

In [ ]:
import re

# Keep the recordings that passed our duration and empty-transcript checks
usable_df = usable_df.copy()

# Identify transcripts containing characters outside basic English ASCII
usable_df["has_non_ascii"] = usable_df["text"].apply(
    lambda text: bool(re.search(r"[^\x00-\x7F]", str(text)))
)

# Check transcript lengths
usable_df["word_count"] = usable_df["text"].apply(
    lambda text: len(str(text).split())
)

print("Total usable recordings:", len(usable_df))
print("Transcripts with non-ASCII characters:", usable_df["has_non_ascii"].sum())
print("Transcripts with only 1–2 words:", (usable_df["word_count"] <= 2).sum())
print("Transcripts with 3 or more words:", (usable_df["word_count"] >= 3).sum())

print(
    "\nDuration of recordings with 3+ words:",
    round(
        usable_df.loc[usable_df["word_count"] >= 3, "duration"].sum() / 3600,
        2
    ),
    "hours"
)

# Display a few examples that contain non-ASCII characters, if any
print("\nExamples with non-ASCII characters:")
display(
    usable_df.loc[
        usable_df["has_non_ascii"],
        ["recording_id", "duration", "text"]
    ].head(10)
)

Total usable recordings: 4880
Transcripts with non-ASCII characters: 0
Transcripts with only 1–2 words: 44
Transcripts with 3 or more words: 4836

Duration of recordings with 3+ words: 9.14 hours

Examples with non-ASCII characters:


,recording_id,duration,text


In [ ]:
import os
import pandas as pd

# Keep recordings that passed our basic quality checks
english_candidates = usable_df[
    (usable_df["duration"] >= 2) &
    (usable_df["word_count"] >= 3) &
    (usable_df["text"].notna())
].copy()

# Randomize the recordings in a reproducible way
english_candidates = english_candidates.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Select approximately 5 hours for training
training_records = []
training_duration = 0

for _, record in english_candidates.iterrows():
    if training_duration >= 5 * 3600:
        break

    training_records.append(record)
    training_duration += record["duration"]

training_df = pd.DataFrame(training_records)

# Remove training recordings before selecting validation data
remaining_records = english_candidates[
    ~english_candidates["recording_id"].isin(training_df["recording_id"])
]

# Select approximately 1 hour for validation
validation_records = []
validation_duration = 0

for _, record in remaining_records.iterrows():
    if validation_duration >= 3600:
        break

    validation_records.append(record)
    validation_duration += record["duration"]

validation_df = pd.DataFrame(validation_records)

# Save both splits
split_folder = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata"
os.makedirs(split_folder, exist_ok=True)

training_df.to_csv(
    os.path.join(split_folder, "english_train.csv"),
    index=False
)

validation_df.to_csv(
    os.path.join(split_folder, "english_validation.csv"),
    index=False
)

# Display split information
print("ENGLISH DATASET SPLIT")
print("-" * 40)
print("Training recordings:", len(training_df))
print("Training duration:", round(training_df["duration"].sum() / 3600, 3), "hours")
print()
print("Validation recordings:", len(validation_df))
print("Validation duration:", round(validation_df["duration"].sum() / 3600, 3), "hours")

# Verify that there is no overlap
overlap = set(training_df["recording_id"]) & set(validation_df["recording_id"])
print("\nOverlapping recordings:", len(overlap))

print("\nSaved training and validation CSV files successfully.")

ENGLISH DATASET SPLIT
----------------------------------------
Training recordings: 2664
Training duration: 5.001 hours

Validation recordings: 505
Validation duration: 1.002 hours

Overlapping recordings: 0

Saved training and validation CSV files successfully.


In [ ]:
from datasets import load_dataset, Audio

# Load the specified Telugu dataset
telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

# Avoid automatic audio decoding during inspection
telugu_dataset = telugu_dataset.cast_column(
    "audio",
    Audio(decode=False)
)

print("Dataset features:")
print(telugu_dataset.features)

# Inspect the first five records
for index, sample in enumerate(telugu_dataset):
    print(f"\nSample {index + 1}")
    print("Text:", sample["text"])
    print("Gender:", sample["gender"])
    print("Audio information:", sample["audio"])

    if index == 4:
        break

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from datasets import load_dataset, Audio
import io
import wave

# Reload the Telugu dataset with audio decoding disabled
telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

telugu_dataset = telugu_dataset.cast_column(
    "audio",
    Audio(decode=False)
)

# Inspect five samples without printing raw audio
for index, sample in enumerate(telugu_dataset):

    audio_bytes = sample["audio"]["bytes"]

    # Read WAV properties directly from the audio bytes
    with wave.open(io.BytesIO(audio_bytes), "rb") as wav:
        sample_rate = wav.getframerate()
        channels = wav.getnchannels()
        bit_depth = wav.getsampwidth() * 8
        duration = wav.getnframes() / sample_rate

    gender = telugu_dataset.features["gender"].int2str(
        sample["gender"]
    )

    print(f"\nSample {index + 1}")
    print("Text:", sample["text"][:150])
    print("Gender:", gender)
    print("Duration:", round(duration, 2), "seconds")
    print("Sample rate:", sample_rate, "Hz")
    print("Channels:", channels)
    print("Bit depth:", bit_depth)

    if index == 4:
        break


Sample 1
Text: నరకవాసి.
Gender: female
Duration: 2.16 seconds
Sample rate: 16000 Hz
Channels: 1
Bit depth: 16

Sample 2
Text: బ్రహ్మదత్తుడు కాశీరాజ్యాన్ని పరిపాలించే కాలంలో ఆ నగరంలో ధనికుడైన ఒక గొప్పవర్తకుడుండేవాడు ఆయనకు మిత్ర విందకుడని ఒక కొడుకుండే వాడు ఈ మిత్రవిందకుడు ఎంతో 
Gender: female
Duration: 37.2 seconds
Sample rate: 16000 Hz
Channels: 1
Bit depth: 16

Sample 3
Text: ఇంతలో కార్తీక పౌర్ణమి వచ్చింది మిత్రవిందకుడితో తల్లి నాయనా ఇవాళ పుణ్యదినం తెల్లవార్లూ విహారంలో ధర్మోపదేశం చేస్తారు నీవు అక్కడ అందరితోపాటు పూజచేసుకుని 
Gender: female
Duration: 25.0 seconds
Sample rate: 16000 Hz
Channels: 1
Bit depth: 16

Sample 4
Text: డబ్బుకోసమని మిత్రవిందకుడు సరేనన్నాడు అతను ఉపదేశం వినటానికి వెళ్ళాడుగాని ఒక మూల పడుకుని నిద్రపోయాడు తెల్లవారుతూనే లేచి ముఖం కడుక్కుని ఇంటికి తిరిగి వచ్
Gender: female
Duration: 34.75 seconds
Sample rate: 16000 Hz
Channels: 1
Bit depth: 16

Sample 5
Text: ఆయనను ఇక్కడికి తీసుకురావటం దేనికమ్మా నా కాయనతో పనిలేదు అన్నాడు మిత్రవిందకుడు.
Gender: female
Duration: 8.27 secon

In [ ]:
from datasets import load_dataset, Audio
import io
import wave
import pandas as pd

telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

telugu_dataset = telugu_dataset.cast_column(
    "audio",
    Audio(decode=False)
)

records = []

for index, sample in enumerate(telugu_dataset):

    audio_bytes = sample["audio"]["bytes"]

    with wave.open(io.BytesIO(audio_bytes), "rb") as wav:
        duration = wav.getnframes() / wav.getframerate()

    records.append({
        "duration": duration,
        "gender": telugu_dataset.features["gender"].int2str(
            sample["gender"]
        ),
        "text_length": len(sample["text"])
    })

    if index == 99:
        break

duration_df = pd.DataFrame(records)

print("Recordings inspected:", len(duration_df))
print("Total duration:", round(duration_df["duration"].sum() / 3600, 2), "hours")
print("Average duration:", round(duration_df["duration"].mean(), 2), "seconds")
print("\nDuration statistics:")
print(duration_df["duration"].describe().round(2))

print("\nGender distribution:")
print(duration_df["gender"].value_counts())

Recordings inspected: 100
Total duration: 0.62 hours
Average duration: 22.38 seconds

Duration statistics:
count    100.00
mean      22.38
std       13.95
min        1.80
25%       12.35
50%       19.41
75%       29.12
max       61.21
Name: duration, dtype: float64

Gender distribution:
gender
female    100
Name: count, dtype: int64


In [ ]:
from datasets import load_dataset, Audio
from collections import Counter

telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

telugu_dataset = telugu_dataset.cast_column(
    "audio",
    Audio(decode=False)
)

gender_counts = Counter()
first_male_index = None

for index, sample in enumerate(telugu_dataset):

    gender = telugu_dataset.features["gender"].int2str(
        sample["gender"]
    )

    gender_counts[gender] += 1

    if gender == "male" and first_male_index is None:
        first_male_index = index

    if index == 999:
        break

print("Gender distribution in first 1,000 recordings:")
print(gender_counts)

print("First male recording index:", first_male_index)

Gender distribution in first 1,000 recordings:
Counter({'female': 1000})
First male recording index: None


In [ ]:
from datasets import load_dataset, Audio
from collections import Counter

telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

telugu_dataset = telugu_dataset.cast_column(
    "audio",
    Audio(decode=False)
)

gender_counts = Counter()
first_male_index = None

for index, sample in enumerate(telugu_dataset):

    gender = telugu_dataset.features["gender"].int2str(
        sample["gender"]
    )

    gender_counts[gender] += 1

    if gender == "male":
        first_male_index = index
        print("First male recording found at index:", index)
        break

    if (index + 1) % 1000 == 0:
        print("Checked", index + 1, "recordings")

print("\nGender counts up to this point:", gender_counts)

Checked 1000 recordings
Checked 2000 recordings
Checked 3000 recordings
Checked 4000 recordings
First male recording found at index: 4043

Gender counts up to this point: Counter({'female': 4043, 'male': 1})


In [ ]:
from datasets import load_dataset
from collections import Counter

telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

# Keep only gender metadata
gender_dataset = telugu_dataset.select_columns(["gender"])

gender_counts = Counter()
total_recordings = 0

for sample in gender_dataset:
    gender = gender_dataset.features["gender"].int2str(
        sample["gender"]
    )

    gender_counts[gender] += 1
    total_recordings += 1

print("Total recordings:", total_recordings)
print("Gender distribution:", gender_counts)

Total recordings: 8576
Gender distribution: Counter({'male': 4533, 'female': 4043})


In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/gdrive')

# Create the required folders
base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"
metadata_dir = os.path.join(base_dir, "metadata")

os.makedirs(metadata_dir, exist_ok=True)

print("Folder exists:", os.path.isdir(metadata_dir))
print("Folder path:", metadata_dir)

Mounted at /content/gdrive
Folder exists: True
Folder path: /content/drive/MyDrive/FastSpeech2_Assignment/metadata


In [ ]:
%whos

Variable              Type                          Data/Info
-------------------------------------------------------------
Audio                 type                          <class 'datasets.features.audio.Audio'>
Counter               type                          <class 'collections.Counter'>
TARGET_SR             int                           22050
account               dict                          n=11
audio_bytes           bytes                         b'RIFFp\xe3\x1d\x00WAVEfm<...>\x00\x00\x00\x00\x00\x00'
audio_folder          str                           /content/drive/MyDrive/Fa<...>eech2_Assignment/raw_data
base_dir              str                           /content/drive/MyDrive/FastSpeech2_Assignment
bit_depth             int                           16
channels              int                           1
count                 int                           1
dataset               IterableDataset               IterableDataset({\n    fe<...>],\n    num_shards: 3\n})
dr

In [ ]:
print(metadata_df.head())
print(metadata_df.columns)
print(metadata_df.shape)

   recording_id  duration                                               text  \
0             0  8.759500  Our district Hasana was the land of the great ...   
1             1  3.100062          There is a gradual shift in the mentality   
2             2  0.450062                                                Wow   
3             3  2.721125                        Nipah virus belongs to  the   
4             4  3.694313  How much money is left in my Arunachal Pradesh...   

   gender age_group primary_language  
0  Female     18-30          Kannada  
1  Female     30-45          Kannada  
2  Female     18-30            Dogri  
3  Female     45-60         Maithili  
4    Male     30-45         Maithili  
Index(['recording_id', 'duration', 'text', 'gender', 'age_group',
       'primary_language'],
      dtype='object')
(6656, 6)


In [ ]:
telugu_df = metadata_df[
    metadata_df["primary_language"].str.lower() == "telugu"
].copy()

print("Telugu recordings:", len(telugu_df))
print(telugu_df.head())

Telugu recordings: 334
    recording_id  duration                                               text  \
32            32  1.095687                                            trains,   
41            41  5.000000  and they went off into the space when earth wa...   
42            42  1.480000                                  that I came here.   
55            55  9.224438  You are 2 hours late than the delivery time fo...   
57            57  4.236313  What is your favorite painting style or techni...   

    gender age_group primary_language  
32  Female     45-60           Telugu  
41  Female     45-60           Telugu  
42  Female     45-60           Telugu  
55  Female     45-60           Telugu  
57  Female     45-60           Telugu  


In [ ]:
metadata_path = os.path.join(metadata_dir, "telugu_metadata.csv")

telugu_df.to_csv(metadata_path, index=False)

print("Telugu metadata saved successfully!")
print("Total recordings:", len(telugu_df))

print(
    "Total duration:",
    round(telugu_df["duration"].sum() / 3600, 2),
    "hours"
)

print("\nDuration by gender:")
print(
    telugu_df.groupby("gender")["duration"]
    .sum()
    .div(3600)
    .round(2)
)

Telugu metadata saved successfully!
Total recordings: 334
Total duration: 0.5 hours

Duration by gender:
gender
Female    0.41
Male      0.09
Name: duration, dtype: float64


In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"
metadata_dir = os.path.join(base_dir, "metadata")

os.makedirs(metadata_dir, exist_ok=True)

print("Metadata folder ready:", os.path.isdir(metadata_dir))

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Metadata folder ready: True


In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"
metadata_dir = os.path.join(base_dir, "metadata")

os.makedirs(metadata_dir, exist_ok=True)

print("Metadata folder ready:", os.path.isdir(metadata_dir))

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Metadata folder ready: True


In [ ]:
from datasets import load_dataset, Audio
import io
import wave
import pandas as pd
from tqdm.auto import tqdm

# Load the Telugu dataset
telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

telugu_dataset = telugu_dataset.cast_column(
    "audio",
    Audio(decode=False)
)

records = []

# Inspect every recording
for index, sample in enumerate(tqdm(telugu_dataset, total=8576)):

    audio_bytes = sample["audio"]["bytes"]

    with wave.open(io.BytesIO(audio_bytes), "rb") as wav:
        duration = wav.getnframes() / wav.getframerate()

    gender = telugu_dataset.features["gender"].int2str(
        sample["gender"]
    )

    records.append({
        "index": index,
        "text": sample["text"],
        "gender": gender,
        "duration": duration,
        "audio_path": sample["audio"]["path"]
    })

# Create a metadata DataFrame
telugu_df = pd.DataFrame(records)

# Save metadata to Google Drive
metadata_path = (
    "/content/drive/MyDrive/FastSpeech2_Assignment/"
    "metadata/telugu_metadata.csv"
)

telugu_df.to_csv(metadata_path, index=False)

# Display dataset statistics
print("Total recordings:", len(telugu_df))
print(
    "Total duration:",
    round(telugu_df["duration"].sum() / 3600, 2),
    "hours"
)

print("\nDuration by gender:")
print(
    telugu_df.groupby("gender")["duration"].sum().div(3600).round(2)
)

print("\nMetadata saved at:", metadata_path)

  0%|          | 0/8576 [00:00<?, ?it/s]

Total recordings: 8576
Total duration: 36.71 hours

Duration by gender:
gender
female    23.92
male      12.79
Name: duration, dtype: float64

Metadata saved at: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_metadata.csv


In [ ]:
import pandas as pd

metadata_path = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_metadata.csv"

telugu_df = pd.read_csv(metadata_path)

# Check transcript quality
empty_text = telugu_df["text"].isna().sum()
short_text = (telugu_df["text"].str.strip() == "").sum()

# Check recording duration
print("Total recordings:", len(telugu_df))
print("Empty transcripts:", empty_text)
print("Blank transcripts:", short_text)

print("\nDuration statistics (seconds):")
print(telugu_df["duration"].describe().round(2))

print("\nRecordings by duration:")
print(pd.cut(
    telugu_df["duration"],
    bins=[0, 1, 2, 5, 10, float("inf")],
    labels=["Under 1 sec", "1–2 sec", "2–5 sec", "5–10 sec", "Above 10 sec"]
).value_counts().sort_index())

print("\nDuplicate transcript rows:", telugu_df["text"].duplicated().sum())

print("\nGender distribution:")
print(telugu_df["gender"].value_counts())

Total recordings: 8576
Empty transcripts: 0
Blank transcripts: 0

Duration statistics (seconds):
count    8576.00
mean       15.41
std        12.44
min         0.66
25%         7.03
50%        10.97
75%        20.55
max       151.63
Name: duration, dtype: float64

Recordings by duration:
duration
Under 1 sec       12
1–2 sec          127
2–5 sec         1027
5–10 sec        2696
Above 10 sec    4714
Name: count, dtype: int64

Duplicate transcript rows: 113

Gender distribution:
gender
male      4533
female    4043
Name: count, dtype: int64


In [ ]:
import pandas as pd
import os

# Load the saved Telugu metadata
metadata_path = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_metadata.csv"
telugu_df = pd.read_csv(metadata_path)

# Keep recordings with usable text and at least 2 seconds of audio
usable_df = telugu_df[
    (telugu_df["text"].notna()) &
    (telugu_df["text"].str.strip() != "") &
    (telugu_df["duration"] >= 2)
].copy()

# Set reproducible random selection
random_state = 42

train_parts = []
validation_parts = []

# Select equal duration from female and male recordings
for gender in ["female", "male"]:

    gender_df = usable_df[
        usable_df["gender"] == gender
    ].sample(frac=1, random_state=random_state).reset_index(drop=True)

    # Select 2.5 hours for training
    train_target = 2.5 * 3600
    train_selected = []
    train_duration = 0

    for _, row in gender_df.iterrows():
        train_selected.append(row)
        train_duration += row["duration"]

        if train_duration >= train_target:
            break

    train_gender_df = pd.DataFrame(train_selected)
    train_parts.append(train_gender_df)

    # Remove training recordings before selecting validation
    remaining_df = gender_df[
        ~gender_df["index"].isin(train_gender_df["index"])
    ]

    # Select 0.5 hours for validation
    validation_target = 0.5 * 3600
    validation_selected = []
    validation_duration = 0

    for _, row in remaining_df.iterrows():
        validation_selected.append(row)
        validation_duration += row["duration"]

        if validation_duration >= validation_target:
            break

    validation_parts.append(pd.DataFrame(validation_selected))

# Combine both gender groups
telugu_train_df = pd.concat(train_parts).reset_index(drop=True)
telugu_validation_df = pd.concat(validation_parts).reset_index(drop=True)

# Verify that there is no overlap
overlap = set(telugu_train_df["index"]) & set(telugu_validation_df["index"])

# Save both splits
metadata_dir = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata"

telugu_train_df.to_csv(
    os.path.join(metadata_dir, "telugu_train.csv"),
    index=False
)

telugu_validation_df.to_csv(
    os.path.join(metadata_dir, "telugu_validation.csv"),
    index=False
)

# Display results
print("TRAINING DATA")
print("Recordings:", len(telugu_train_df))
print("Duration:", round(telugu_train_df["duration"].sum() / 3600, 3), "hours")
print(telugu_train_df["gender"].value_counts())

print("\nVALIDATION DATA")
print("Recordings:", len(telugu_validation_df))
print("Duration:", round(telugu_validation_df["duration"].sum() / 3600, 3), "hours")
print(telugu_validation_df["gender"].value_counts())

print("\nOverlap between splits:", len(overlap))
print("\nBoth CSV files saved successfully!")

TRAINING DATA
Recordings: 1281
Duration: 5.012 hours
gender
male      864
female    417
Name: count, dtype: int64

VALIDATION DATA
Recordings: 251
Duration: 1.003 hours
gender
male      171
female     80
Name: count, dtype: int64

Overlap between splits: 0

Both CSV files saved successfully!


In [ ]:
%cd /content

!git clone https://github.com/ming024/FastSpeech2.git

/content
fatal: destination path 'FastSpeech2' already exists and is not an empty directory.


In [ ]:
import os

repo_dir = "/content/FastSpeech2"

print("Repository exists:", os.path.isdir(repo_dir))

print("\nRepository files:")
for filename in os.listdir(repo_dir):
    print(filename)

Repository exists: True

Repository files:
model
preprocessed_data
index.html
preprocess.py
audio
README.md
demo
dataset.py
.git
img
utils
hifigan
text
evaluate.py
prepare_align.py
transformer
config
lexicon
preprocessor
synthesize.py
.gitignore
requirements.txt
train.py
LICENSE


In [ ]:
import os

repo_dir = "/content/FastSpeech2"

print("Repository exists:", os.path.isdir(repo_dir))

print("\nImportant files:")
for folder in ["config", "preprocessor", "text", "model", "audio"]:
    folder_path = os.path.join(repo_dir, folder)

    print(f"\n{folder}/")

    if os.path.isdir(folder_path):
        for filename in os.listdir(folder_path)[:15]:
            print("  ", filename)

print("\nRoot files:")
for filename in os.listdir(repo_dir):
    if filename.endswith(".py") or filename.endswith(".md"):
        print(" ", filename)

Repository exists: True

Important files:

config/
   LJSpeech_paper
   README.md
   LJSpeech
   AISHELL3
   LibriTTS

preprocessor/
   preprocessor.py
   libritts.py
   ljspeech.py
   aishell3.py

text/
   symbols.py
   cleaners.py
   numbers.py
   cmudict.py
   pinyin.py
   __init__.py

model/
   fastspeech2.py
   optimizer.py
   __init__.py
   loss.py
   modules.py

audio/
   stft.py
   audio_processing.py
   __init__.py
   tools.py

Root files:
  preprocess.py
  README.md
  dataset.py
  evaluate.py
  prepare_align.py
  synthesize.py
  train.py


In [ ]:
import os

repo_dir = "/content/FastSpeech2"

for folder in ["config/LJSpeech", "config/LibriTTS", "config/AISHELL3"]:
    folder_path = os.path.join(repo_dir, folder)

    print(f"\n{folder}:")
    if os.path.isdir(folder_path):
        for filename in os.listdir(folder_path):
            print("  ", filename)


config/LJSpeech:
   model.yaml
   preprocess.yaml
   train.yaml

config/LibriTTS:
   model.yaml
   preprocess.yaml
   train.yaml

config/AISHELL3:
   model.yaml
   preprocess.yaml
   train.yaml


In [ ]:
from pathlib import Path

config_dir = Path("/content/FastSpeech2/config/LJSpeech")

for filename in ["preprocess.yaml", "model.yaml", "train.yaml"]:
    print(f"\n{'='*20} {filename} {'='*20}\n")
    print((config_dir / filename).read_text())


==================== preprocess.yaml ====================

dataset: "LJSpeech"

path:
  corpus_path: "/home/ming/Data/LJSpeech-1.1"
  lexicon_path: "lexicon/librispeech-lexicon.txt"
  raw_path: "./raw_data/LJSpeech"
  preprocessed_path: "./preprocessed_data/LJSpeech"

preprocessing:
  val_size: 512
  text:
    text_cleaners: ["english_cleaners"]
    language: "en"
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: 80
    mel_fmin: 0
    mel_fmax: 8000 # please set to 8000 for HiFi-GAN vocoder, set to null for MelGAN vocoder
  pitch:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True
  energy:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True


==================== model.yaml ====================

transformer:
  encoder_layer: 4
  encoder_head: 2
  encoder_hidden: 256
  decoder_layer: 6
 

In [ ]:
import os

repo_dir = "/content/FastSpeech2"

print("Repository exists:", os.path.isdir(repo_dir))

if os.path.isdir(repo_dir):
    print("\nFiles in repository root:")
    print(os.listdir(repo_dir))
else:
    print("\nContents of /content:")
    print(os.listdir("/content"))

Repository exists: True

Files in repository root:
['model', 'preprocessed_data', 'index.html', 'preprocess.py', 'audio', 'README.md', 'demo', 'dataset.py', '.git', 'img', 'utils', 'hifigan', 'text', 'evaluate.py', 'prepare_align.py', 'transformer', 'config', 'lexicon', 'preprocessor', 'synthesize.py', '.gitignore', 'requirements.txt', 'train.py', 'LICENSE']


In [ ]:
%cd /content
!git clone https://github.com/ming024/FastSpeech2.git

/content
fatal: destination path 'FastSpeech2' already exists and is not an empty directory.


In [ ]:
import os

repo_dir = "/content/FastSpeech2"

print("Repository exists:", os.path.isdir(repo_dir))
print("preprocess.py exists:", os.path.isfile(
    os.path.join(repo_dir, "preprocess.py")
))

Repository exists: True
preprocess.py exists: True


In [ ]:
from pathlib import Path

files_to_inspect = [
    "/content/FastSpeech2/preprocess.py",
    "/content/FastSpeech2/preprocessor/preprocessor.py",
    "/content/FastSpeech2/preprocessor/ljspeech.py"
]

for file_path in files_to_inspect:
    print(f"\n{'='*25}\n{Path(file_path).name}\n{'='*25}")
    print(Path(file_path).read_text())


preprocess.py
import argparse

import yaml

from preprocessor.preprocessor import Preprocessor


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str, help="path to preprocess.yaml")
    args = parser.parse_args()

    config = yaml.load(open(args.config, "r"), Loader=yaml.FullLoader)
    preprocessor = Preprocessor(config)
    preprocessor.build_from_path()


preprocessor.py
import os
import random
import json

import tgt
import librosa
import numpy as np
import pyworld as pw
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import audio as Audio


class Preprocessor:
    def __init__(self, config):
        self.config = config
        self.in_dir = config["path"]["raw_path"]
        self.out_dir = config["path"]["preprocessed_path"]
        self.val_size = config["preprocessing"]["val_size"]
        self.sampling_rate = config["preprocessing"]["audio"]["sampling_rate"

In [ ]:
from pathlib import Path

files_to_inspect = [
    "/content/FastSpeech2/prepare_align.py",
    "/content/FastSpeech2/text/symbols.py",
    "/content/FastSpeech2/text/__init__.py",
    "/content/FastSpeech2/text/cleaners.py"
]

for file_path in files_to_inspect:
    print(f"\n{'='*25}\n{Path(file_path).name}\n{'='*25}")
    print(Path(file_path).read_text())


prepare_align.py
import argparse

import yaml

from preprocessor import ljspeech, aishell3, libritts


def main(config):
    if "LJSpeech" in config["dataset"]:
        ljspeech.prepare_align(config)
    if "AISHELL3" in config["dataset"]:
        aishell3.prepare_align(config)
    if "LibriTTS" in config["dataset"]:
        libritts.prepare_align(config)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("config", type=str, help="path to preprocess.yaml")
    args = parser.parse_args()

    config = yaml.load(open(args.config, "r"), Loader=yaml.FullLoader)
    main(config)


symbols.py
""" from https://github.com/keithito/tacotron """

"""
Defines the set of symbols used in text input to the model.

The default is a set of ASCII characters that works well for English or text that has been run through Unidecode. For other data, you can modify _characters. See TRAINING_DATA.md for details. """

from text import cmudict, pinyin

_pad = "_"
_punct

In [ ]:
from pathlib import Path

files_to_inspect = [
    "/content/FastSpeech2/dataset.py",
    "/content/FastSpeech2/train.py"
]

for file_path in files_to_inspect:
    print(f"\n{'='*25}\n{Path(file_path).name}\n{'='*25}")
    print(Path(file_path).read_text())


dataset.py
import json
import math
import os

import numpy as np
from torch.utils.data import Dataset

from text import text_to_sequence
from utils.tools import pad_1D, pad_2D


class Dataset(Dataset):
    def __init__(
        self, filename, preprocess_config, train_config, sort=False, drop_last=False
    ):
        self.dataset_name = preprocess_config["dataset"]
        self.preprocessed_path = preprocess_config["path"]["preprocessed_path"]
        self.cleaners = preprocess_config["preprocessing"]["text"]["text_cleaners"]
        self.batch_size = train_config["optimizer"]["batch_size"]

        self.basename, self.speaker, self.text, self.raw_text = self.process_meta(
            filename
        )
        with open(os.path.join(self.preprocessed_path, "speakers.json")) as f:
            self.speaker_map = json.load(f)
        self.sort = sort
        self.drop_last = drop_last

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        basenam

In [ ]:
from pathlib import Path

files_to_inspect = [
    "/content/FastSpeech2/model/fastspeech2.py",
    "/content/FastSpeech2/model/loss.py"
]

for file_path in files_to_inspect:
    print(f"\n{'='*25}\n{Path(file_path).name}\n{'='*25}")
    print(Path(file_path).read_text())


fastspeech2.py
import os
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformer import Encoder, Decoder, PostNet
from .modules import VarianceAdaptor
from utils.tools import get_mask_from_lengths


class FastSpeech2(nn.Module):
    """ FastSpeech2 """

    def __init__(self, preprocess_config, model_config):
        super(FastSpeech2, self).__init__()
        self.model_config = model_config

        self.encoder = Encoder(model_config)
        self.variance_adaptor = VarianceAdaptor(preprocess_config, model_config)
        self.decoder = Decoder(model_config)
        self.mel_linear = nn.Linear(
            model_config["transformer"]["decoder_hidden"],
            preprocess_config["preprocessing"]["mel"]["n_mel_channels"],
        )
        self.postnet = PostNet()

        self.speaker_emb = None
        if model_config["multi_speaker"]:
            with open(
                os.path.join(
                    preprocess_config["path"]["pr

In [ ]:
from pathlib import Path

file_path = "/content/FastSpeech2/model/modules.py"

print(Path(file_path).read_text())

import os
import json
import copy
import math
from collections import OrderedDict

import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F

from utils.tools import get_mask_from_lengths, pad

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class VarianceAdaptor(nn.Module):
    """Variance Adaptor"""

    def __init__(self, preprocess_config, model_config):
        super(VarianceAdaptor, self).__init__()
        self.duration_predictor = VariancePredictor(model_config)
        self.length_regulator = LengthRegulator()
        self.pitch_predictor = VariancePredictor(model_config)
        self.energy_predictor = VariancePredictor(model_config)

        self.pitch_feature_level = preprocess_config["preprocessing"]["pitch"][
            "feature"
        ]
        self.energy_feature_level = preprocess_config["preprocessing"]["energy"][
            "feature"
        ]
        assert self.pitch_feature_level in ["phoneme_level", "frame_l

In [ ]:
from pathlib import Path

files_to_inspect = [
    "/content/FastSpeech2/utils/model.py",
    "/content/FastSpeech2/hifigan/config.json"
]

for file_path in files_to_inspect:
    path = Path(file_path)

    print(f"\n{'='*25}\n{path.name}\n{'='*25}")

    if path.exists():
        print(path.read_text())
    else:
        print("File not found:", file_path)


model.py
import os
import json

import torch
import numpy as np

import hifigan
from model import FastSpeech2, ScheduledOptim


def get_model(args, configs, device, train=False):
    (preprocess_config, model_config, train_config) = configs

    model = FastSpeech2(preprocess_config, model_config).to(device)
    if args.restore_step:
        ckpt_path = os.path.join(
            train_config["path"]["ckpt_path"],
            "{}.pth.tar".format(args.restore_step),
        )
        ckpt = torch.load(ckpt_path)
        model.load_state_dict(ckpt["model"])

    if train:
        scheduled_optim = ScheduledOptim(
            model, train_config, model_config, args.restore_step
        )
        if args.restore_step:
            scheduled_optim.load_state_dict(ckpt["optimizer"])
        model.train()
        return model, scheduled_optim

    model.eval()
    model.requires_grad_ = False
    return model


def get_param_num(model):
    num_param = sum(param.numel() for param in model.para

In [ ]:
import os

vocoder_dir = "/content/FastSpeech2/hifigan"

print("HiFi-GAN files:")
for filename in os.listdir(vocoder_dir):
    path = os.path.join(vocoder_dir, filename)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{filename}: {size_mb:.2f} MB")

HiFi-GAN files:
generator_LJSpeech.pth.tar.zip: 49.44 MB
config.json: 0.00 MB
__init__.py: 0.00 MB
models.py: 0.01 MB
generator_universal.pth.tar.zip: 49.54 MB
LICENSE: 0.00 MB


In [ ]:
import os
import zipfile

vocoder_dir = "/content/FastSpeech2/hifigan"

zip_files = [
    "generator_universal.pth.tar.zip",
    "generator_LJSpeech.pth.tar.zip"
]

for filename in zip_files:
    zip_path = os.path.join(vocoder_dir, filename)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(vocoder_dir)

    print(f"Extracted: {filename}")

Extracted: generator_universal.pth.tar.zip
Extracted: generator_LJSpeech.pth.tar.zip


In [ ]:
import os

for filename in os.listdir("/content/FastSpeech2/hifigan"):
    if "generator" in filename:
        path = os.path.join("/content/FastSpeech2/hifigan", filename)
        print(filename, f"{os.path.getsize(path) / (1024 * 1024):.2f} MB")

generator_LJSpeech.pth.tar.zip 49.44 MB
generator_universal.pth.tar 53.20 MB
generator_LJSpeech.pth.tar 53.20 MB
generator_universal.pth.tar.zip 49.54 MB


In [ ]:
import torch
import os

vocoder_dir = "/content/FastSpeech2/hifigan"

for filename in [
    "generator_LJSpeech.pth.tar",
    "generator_universal.pth.tar"
]:
    path = os.path.join(vocoder_dir, filename)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print(f"\n{filename}")
    print("Checkpoint keys:", checkpoint.keys())

    if "generator" in checkpoint:
        weights = checkpoint["generator"]
        print("Number of generator parameters:", len(weights))
        first_key = next(iter(weights))
        print("First parameter:", first_key)
        print("Parameter shape:", weights[first_key].shape)


generator_LJSpeech.pth.tar
Checkpoint keys: dict_keys(['generator'])
Number of generator parameters: 234
First parameter: conv_pre.bias
Parameter shape: torch.Size([512])

generator_universal.pth.tar
Checkpoint keys: dict_keys(['generator'])
Number of generator parameters: 234
First parameter: conv_pre.bias
Parameter shape: torch.Size([512])


In [ ]:
import os
import sys

repo_path = "/content/FastSpeech2"

os.chdir(repo_path)

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Current directory:", os.getcwd())
print("Utils folder exists:", os.path.isdir("utils"))
print("Model file exists:", os.path.isfile("utils/model.py"))

Current directory: /content/FastSpeech2
Utils folder exists: True
Model file exists: True


In [ ]:
!pip install Unidecode

In [ ]:
import inspect
from utils.model import get_vocoder

print("Function signature:")
print(inspect.signature(get_vocoder))

print("\nVocoder loading code:")
print(inspect.getsource(get_vocoder))

Function signature:
(config, device)

Vocoder loading code:
def get_vocoder(config, device):
    name = config["vocoder"]["model"]
    speaker = config["vocoder"]["speaker"]

    if name == "MelGAN":
        if speaker == "LJSpeech":
            vocoder = torch.hub.load(
                "descriptinc/melgan-neurips", "load_melgan", "linda_johnson"
            )
        elif speaker == "universal":
            vocoder = torch.hub.load(
                "descriptinc/melgan-neurips", "load_melgan", "multi_speaker"
            )
        vocoder.mel2wav.eval()
        vocoder.mel2wav.to(device)
    elif name == "HiFi-GAN":
        with open("hifigan/config.json", "r") as f:
            config = json.load(f)
        config = hifigan.AttrDict(config)
        vocoder = hifigan.Generator(config)
        if speaker == "LJSpeech":
            ckpt = torch.load("hifigan/generator_LJSpeech.pth.tar")
        elif speaker == "universal":
            ckpt = torch.load("hifigan/generator_universal.pth.ta

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not currently available.")

PyTorch version: 2.11.0+cpu
CUDA available: False
GPU is not currently available.


In [ ]:
import os

repo_path = "/content/FastSpeech2"
vocoder_dir = os.path.join(repo_path, "hifigan")

print("Repository exists:", os.path.isdir(repo_path))
print("Vocoder folder exists:", os.path.isdir(vocoder_dir))

if os.path.isdir(vocoder_dir):
    for filename in [
        "generator_universal.pth.tar",
        "generator_LJSpeech.pth.tar"
    ]:
        print(filename, ":", os.path.isfile(
            os.path.join(vocoder_dir, filename)
        ))

Repository exists: True
Vocoder folder exists: True
generator_universal.pth.tar : True
generator_LJSpeech.pth.tar : True


In [ ]:
%cd /content

!git clone https://github.com/ming024/FastSpeech2.git

/content
fatal: destination path 'FastSpeech2' already exists and is not an empty directory.


In [ ]:
import os

repo_path = "/content/FastSpeech2"

print("Repository exists:", os.path.isdir(repo_path))
print("Vocoder folder exists:", os.path.isdir(
    os.path.join(repo_path, "hifigan")
))

Repository exists: True
Vocoder folder exists: True


In [ ]:
import os
import zipfile

vocoder_dir = "/content/FastSpeech2/hifigan"

for filename in [
    "generator_universal.pth.tar.zip",
    "generator_LJSpeech.pth.tar.zip"
]:
    zip_path = os.path.join(vocoder_dir, filename)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(vocoder_dir)

    print("Extracted:", filename)

Extracted: generator_universal.pth.tar.zip
Extracted: generator_LJSpeech.pth.tar.zip


In [ ]:
model_path = "/content/FastSpeech2/utils/model.py"

with open(model_path, "r") as f:
    code = f.read()

code = code.replace(
    'torch.load("hifigan/generator_universal.pth.tar")',
    'torch.load("hifigan/generator_universal.pth.tar", map_location=device)'
)

with open(model_path, "w") as f:
    f.write(code)

print("Updated vocoder checkpoint loading successfully!")

Updated vocoder checkpoint loading successfully!


In [ ]:
import importlib
import utils.model

importlib.reload(utils.model)

from utils.model import get_vocoder
import inspect

print(inspect.getsource(get_vocoder))


def get_vocoder(config, device):
    name = config["vocoder"]["model"]
    speaker = config["vocoder"]["speaker"]

    if name == "MelGAN":
        if speaker == "LJSpeech":
            vocoder = torch.hub.load(
                "descriptinc/melgan-neurips", "load_melgan", "linda_johnson"
            )
        elif speaker == "universal":
            vocoder = torch.hub.load(
                "descriptinc/melgan-neurips", "load_melgan", "multi_speaker"
            )
        vocoder.mel2wav.eval()
        vocoder.mel2wav.to(device)
    elif name == "HiFi-GAN":
        with open("hifigan/config.json", "r") as f:
            config = json.load(f)
        config = hifigan.AttrDict(config)
        vocoder = hifigan.Generator(config)
        if speaker == "LJSpeech":
            ckpt = torch.load("hifigan/generator_LJSpeech.pth.tar")
        elif speaker == "universal":
            ckpt = torch.load("hifigan/generator_universal.pth.tar", map_location=device)
        vocoder.load_state_dict(ckp

In [ ]:
import os
import sys
import torch

repo_path = "/content/FastSpeech2"
os.chdir(repo_path)

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from utils.model import get_vocoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocoder_config = {
    "vocoder": {
        "model": "HiFi-GAN",
        "speaker": "universal"
    }
}

vocoder = get_vocoder(vocoder_config, device)

print("Vocoder loaded successfully!")
print("Device:", device)
print("Vocoder type:", type(vocoder).__name__)

Removing weight norm...
Vocoder loaded successfully!
Device: cpu
Vocoder type: Generator


In [ ]:
import torch

# Create a test mel-spectrogram
dummy_mel = torch.zeros(1, 80, 20).to(device)

# Generate waveform
with torch.no_grad():
    generated_audio = vocoder(dummy_mel)

print("Input mel shape:", dummy_mel.shape)
print("Generated audio shape:", generated_audio.shape)
print("Audio generated successfully!")

Input mel shape: torch.Size([1, 80, 20])
Generated audio shape: torch.Size([1, 1, 5120])
Audio generated successfully!


In [ ]:
from google.colab import drive
import os

drive.mount("/content/gdrive")

metadata_dir = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata"

files = [
    "english_train.csv",
    "english_validation.csv",
    "telugu_train.csv",
    "telugu_validation.csv"
]

for filename in files:
    path = os.path.join(metadata_dir, filename)
    print(filename, ":", "Available" if os.path.exists(path) else "Missing")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
english_train.csv : Available
english_validation.csv : Available
telugu_train.csv : Available
telugu_validation.csv : Available


In [ ]:
import os
import pandas as pd

metadata_dir = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata"

files = [
    "english_train.csv",
    "english_validation.csv",
    "telugu_train.csv",
    "telugu_validation.csv"
]

for filename in files:
    df = pd.read_csv(os.path.join(metadata_dir, filename))

    print(f"\n{'='*50}")
    print(filename)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 3 rows:")
    display(df.head(3))


english_train.csv
Shape: (2664, 8)
Columns: ['recording_id', 'duration', 'text', 'gender', 'age_group', 'primary_language', 'has_non_ascii', 'word_count']

First 3 rows:


,recording_id,duration,text,gender,age_group,primary_language,has_non_ascii,word_count
0,1952,5.734625,so it definitely influences parts of our language,Male,18-30,Odia,False,8
1,3590,3.787375,I have reset my MPIN to 6279,Male,18-30,Assamese,False,7
2,1437,5.440000,"Of course, there is a lot of funding by the st...",Female,45-60,Telugu,False,14



english_validation.csv
Shape: (505, 8)
Columns: ['recording_id', 'duration', 'text', 'gender', 'age_group', 'primary_language', 'has_non_ascii', 'word_count']

First 3 rows:


,recording_id,duration,text,gender,age_group,primary_language,has_non_ascii,word_count
0,5403,7.560375,improving the healthcare infrastructure as wel...,Female,30-45,Kannada,False,14
1,4492,6.786562,I had a cooking disaster when I tried to prepa...,Female,30-45,Kannada,False,15
2,1381,10.960000,So first Baisak we used to celebrate New Year ...,Male,18-30,Nepali,False,15



telugu_train.csv
Shape: (1281, 5)
Columns: ['index', 'text', 'gender', 'duration', 'audio_path']

First 3 rows:


,index,text,gender,duration,audio_path
0,2123,కొంతదూరం అడవిలోంచి నడిచి అందరు ఒక కొండప్రాంతం ...,female,6.875687,train_telugufemale_02124.wav
1,2015,ఒకప్పుడు గోహ ఒక బిడారువెంట ప్రయాణంచెయ్యటం తటస్...,female,21.504250,train_telugufemale_02016.wav
2,2736,వాడు పారిపోదామనుకుంటే బయట విచ్చుకత్తులవాళ్ళు బ...,female,5.993313,train_telugufemale_02737.wav



telugu_validation.csv
Shape: (251, 5)
Columns: ['index', 'text', 'gender', 'duration', 'audio_path']

First 3 rows:


,index,text,gender,duration,audio_path
0,3123,ఆ తరువాత వారు కంచికి పోయి వరద రాజస్వామిని సేవి...,female,26.334000,train_telugufemale_03124.wav
1,3237,అన్నీ అయినై ఇంక తాంబూలం ఇయ్య వలసిన ఘట్టం వచ్చి...,female,20.877312,train_telugufemale_03238.wav
2,64,వంశ గౌరవం.,female,2.138812,train_telugufemale_00065.wav


In [ ]:
import os

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

folders = [
    "English/raw_audio/train",
    "English/raw_audio/validation",
    "Telugu/raw_audio/train",
    "Telugu/raw_audio/validation"
]

for folder in folders:
    path = os.path.join(base_dir, folder)
    os.makedirs(path, exist_ok=True)
    print("Ready:", path)


Ready: /content/drive/MyDrive/FastSpeech2_Assignment/English/raw_audio/train
Ready: /content/drive/MyDrive/FastSpeech2_Assignment/English/raw_audio/validation
Ready: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/train
Ready: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/validation


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from datasets import load_dataset, Audio
import pandas as pd
import itertools
import os

metadata_dir = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata"

# Read the saved split files
english_train = pd.read_csv(os.path.join(metadata_dir, "english_train.csv"))
telugu_train = pd.read_csv(os.path.join(metadata_dir, "telugu_train.csv"))

# Get the first selected recording ID from each dataset
english_id = int(english_train.iloc[0]["recording_id"])
telugu_id = int(telugu_train.iloc[0]["index"])

# Load English Svarah as a streaming dataset
english_dataset = load_dataset(
    "ai4bharat/Svarah",
    split="test",
    streaming=True
).cast_column("audio", Audio(decode=False))

# Retrieve the selected English recording
english_sample = next(
    itertools.islice(iter(english_dataset), english_id, english_id + 1)
)

# Load Telugu IndicTTS as a streaming dataset
telugu_dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
).cast_column("audio", Audio(decode=False))

# Retrieve the selected Telugu recording
telugu_sample = next(
    itertools.islice(iter(telugu_dataset), telugu_id, telugu_id + 1)
)

print("ENGLISH SAMPLE")
print("Recording ID:", english_id)
print("CSV text:", english_train.iloc[0]["text"])
print("Dataset text:", english_sample["text"])
print("Audio information:", english_sample["audio"])

print("\nTELUGU SAMPLE")
print("Recording ID:", telugu_id)
print("CSV text:", telugu_train.iloc[0]["text"])
print("Dataset text:", telugu_sample["text"])
print("Audio information:", telugu_sample["audio"])

ENGLISH SAMPLE
Recording ID: 1952
CSV text: so it definitely influences parts of our language
Dataset text: so it definitely influences parts of our language
Audio information: None

TELUGU SAMPLE
Recording ID: 2123
CSV text: కొంతదూరం అడవిలోంచి నడిచి అందరు ఒక కొండప్రాంతం చేరుకున్నారు.
Dataset text: కొంతదూరం అడవిలోంచి నడిచి అందరు ఒక కొండప్రాంతం చేరుకున్నారు.
Audio information: {'bytes': b'RIFF\x9a[\x03\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00datav[\x03\x00\t\x00\t\x00\x0e\x00\x0e\x00\t\x00\x05\x00\x00\x00\t\x00\t\x00\t\x00\x0e\x00\x0e\x00\x0e\x00\x0e\x00\x17\x00\x13\x00\x0e\x00\x13\x00\x17\x00\x13\x00\x0e\x00\x0e\x00\x0e\x00\x05\x00\x05\x00\t\x00\t\x00\x05\x00\x00\x00\x00\x00\x05\x00\t\x00\x13\x00\x13\x00\x0e\x00\x05\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x05\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\t\x00\t\x00\t\x00\t\x00\x05\x00\x00\x00\x00\x00\x00\x00\xfc\xff\x

In [ ]:
print("English dataset columns:")
print(english_sample.keys())

print("\nEnglish audio_filepath field:")
print(english_sample.get("audio_filepath"))

print("\nEnglish audio field:")
print(english_sample.get("audio"))

English dataset columns:
dict_keys(['audio_filepath', 'duration', 'text', 'gender', 'age-group', 'primary_language', 'native_place_state', 'native_place_district', 'highest_qualification', 'job_category', 'occupation_domain', 'audio'])

English audio_filepath field:

English audio field:
None


In [ ]:
from datasets import Audio
import itertools

# Apply audio decoding configuration to the correct column
english_dataset = english_dataset.cast_column(
    "audio_filepath",
    Audio(decode=False)
)

# Retrieve the same selected recording
english_sample = next(
    itertools.islice(
        iter(english_dataset),
        english_id,
        english_id + 1
    )
)

print("Recording ID:", english_id)
print("Transcript:", english_sample["text"])

print("\nAudio information:")
print(english_sample["audio_filepath"])

Recording ID: 1952
Transcript: so it definitely influences parts of our language

Audio information:
{'bytes': b'RIFF\xf8\xcc\x02\x00WAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\x80>\x00\x00\x00}\x00\x00\x02\x00\x10\x00data\xd4\xcc\x02\x00D\xff\xd7\xfe\xe4\xfeD\xff\xf2\xfey\xfe\xd2\xfeu\xff\x14\x00\xac\x00\x97\xff(\x00\x9c\xff\x97\xff\x95\xfeR\xffM\xff\x8a\x00%\x00\xfa\x00\xde\x00\x0e\x00\x88\x00\xd7\xfe\xb4\x00\x14\x00\x8f\x008\x00?\x00\xc9\x00\xd8\xffa\x00\x8e\x00b\x01i\x01\x10\x01\xe8\xff\xa0\x00\n\x00\xa5\xff\xff\xff7\x00\xf7\x00\xac\xff\xb7\xff\xe5\xfd$\xfe|\xfea\xfd\xfa\xfeP\x00\xbc\x00\xd1\xff?\x00\x93\xfe6\xfe\x88\xfe\x19\xfe\xb2\xfd>\xff\xce\xff\xaf\xffy\x00\xa6\x00\x13\x00i\x00\xbc\x00\xaa\xff@\x006\x00q\x00\xde\x00a\x01\x8f\x01\xb6\x00M\x01\xb7\x00\xad\x00\xdf\x01\x96\x00\x1a\x01\x0e\x01_\x01\x10\x01\x10\x00M\x00\xd6\xff\xeb\x00H\x00\x1e\x004\x00`\x00\x99\x00M\xff\x12\x00\xe7\xff\x02\x00\xfe\xff\n\x00\xd3\x00\xd4\xffU\x00\xe7\xff*\x00\xea\x00\xd7\x00\x91\x00\xe9\xff\xea\xff\x11\

In [ ]:
import os
from IPython.display import Audio, display

# Get the actual WAV bytes
audio_bytes = english_sample["audio_filepath"]["bytes"]

# Save the sample in Google Drive
sample_path = (
    "/content/drive/MyDrive/FastSpeech2_Assignment/"
    "English/raw_audio/train/sample_1952.wav"
)

with open(sample_path, "wb") as f:
    f.write(audio_bytes)

print("Audio saved:", sample_path)
print("File size:", os.path.getsize(sample_path), "bytes")

# Listen to the recording
display(Audio(audio_bytes))

Audio saved: /content/drive/MyDrive/FastSpeech2_Assignment/English/raw_audio/train/sample_1952.wav
File size: 183552 bytes


In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Audio

# 1. Set up paths
base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"
metadata_dir = os.path.join(base_dir, "metadata")

train_df = pd.read_csv(os.path.join(metadata_dir, "english_train.csv"))
val_df = pd.read_csv(os.path.join(metadata_dir, "english_validation.csv"))

# 2. Map each recording ID to its split and transcript
recordings = {}

for _, row in train_df.iterrows():
    recordings[int(row["recording_id"])] = ("train", row["text"])

for _, row in val_df.iterrows():
    recordings[int(row["recording_id"])] = ("validation", row["text"])

target_ids = set(recordings.keys())

# 3. Create output folders
train_dir = os.path.join(base_dir, "English", "raw_audio", "train")
val_dir = os.path.join(base_dir, "English", "raw_audio", "validation")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# 4. Load Svarah as a streaming dataset
print("Loading Svarah dataset...")

dataset = load_dataset(
    "ai4bharat/Svarah",
    split="test",
    streaming=True
)

# Svarah stores its audio under audio_filepath
dataset = dataset.cast_column(
    "audio_filepath",
    Audio(decode=False)
)

# 5. Extract selected recordings in one pass
saved_train = 0
saved_val = 0
skipped = 0
found_ids = set()

print("Starting English audio extraction...")

for idx, sample in enumerate(dataset):

    if idx in target_ids:
        split, transcript = recordings[idx]

        output_dir = train_dir if split == "train" else val_dir
        wav_path = os.path.join(output_dir, f"recording_{idx}.wav")
        lab_path = os.path.join(output_dir, f"recording_{idx}.lab")

        found_ids.add(idx)

        # Skip files that were already extracted
        if os.path.exists(wav_path) and os.path.exists(lab_path):
            skipped += 1
        else:
            audio_bytes = sample["audio_filepath"]["bytes"]

            if audio_bytes is None:
                raise ValueError(f"Audio bytes missing for recording {idx}")

            with open(wav_path, "wb") as f:
                f.write(audio_bytes)

            # Save the exact transcript alongside the audio
            with open(lab_path, "w", encoding="utf-8") as f:
                f.write(str(transcript))

            if split == "train":
                saved_train += 1
            else:
                saved_val += 1

        if len(found_ids) % 500 == 0:
            print(f"Processed {len(found_ids)} / {len(target_ids)} recordings")

    # Stop once every selected recording has been found
    if len(found_ids) == len(target_ids):
        break

# 6. Verify extraction
missing_ids = target_ids - found_ids

print("\nENGLISH EXTRACTION SUMMARY")
print("--------------------------")
print("Training recordings in CSV:", len(train_df))
print("Validation recordings in CSV:", len(val_df))
print("New training files saved:", saved_train)
print("New validation files saved:", saved_val)
print("Already extracted and skipped:", skipped)
print("Missing recording IDs:", len(missing_ids))

print("\nTraining folder:", train_dir)
print("Validation folder:", val_dir)

if missing_ids:
    print("Some recording IDs were not found. Please share this output.")
else:
    print("\nAll selected English recordings were found!")

Loading Svarah dataset...
Starting English audio extraction...
Processed 500 / 3169 recordings
Processed 1000 / 3169 recordings
Processed 1500 / 3169 recordings
Processed 2000 / 3169 recordings
Processed 2500 / 3169 recordings
Processed 3000 / 3169 recordings

ENGLISH EXTRACTION SUMMARY
--------------------------
Training recordings in CSV: 2664
Validation recordings in CSV: 505
New training files saved: 2664
New validation files saved: 505
Already extracted and skipped: 0
Missing recording IDs: 0

Training folder: /content/drive/MyDrive/FastSpeech2_Assignment/English/raw_audio/train
Validation folder: /content/drive/MyDrive/FastSpeech2_Assignment/English/raw_audio/validation

All selected English recordings were found!


In [ ]:
import os
import pandas as pd
from datasets import load_dataset, Audio

# 1. Set up paths
base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"
metadata_dir = os.path.join(base_dir, "metadata")

train_df = pd.read_csv(os.path.join(metadata_dir, "telugu_train.csv"))
val_df = pd.read_csv(os.path.join(metadata_dir, "telugu_validation.csv"))

# 2. Map recording IDs to their split, filename and transcript
recordings = {}

for _, row in train_df.iterrows():
    recordings[int(row["index"])] = (
        "train", row["audio_path"], row["text"]
    )

for _, row in val_df.iterrows():
    recordings[int(row["index"])] = (
        "validation", row["audio_path"], row["text"]
    )

target_ids = set(recordings.keys())

# 3. Create output folders
train_dir = os.path.join(base_dir, "Telugu", "raw_audio", "train")
val_dir = os.path.join(base_dir, "Telugu", "raw_audio", "validation")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# 4. Load IndicTTS Telugu in streaming mode
print("Loading IndicTTS Telugu dataset...")

dataset = load_dataset(
    "SPRINGLab/IndicTTS_Telugu",
    split="train",
    streaming=True
)

dataset = dataset.cast_column(
    "audio",
    Audio(decode=False)
)

# 5. Extract selected recordings in one pass
saved_train = 0
saved_val = 0
skipped = 0
found_ids = set()

print("Starting Telugu audio extraction...")

for idx, sample in enumerate(dataset):

    if idx in target_ids:
        split, filename, transcript = recordings[idx]

        output_dir = train_dir if split == "train" else val_dir
        wav_path = os.path.join(output_dir, os.path.basename(filename))
        lab_path = os.path.splitext(wav_path)[0] + ".lab"

        found_ids.add(idx)

        # Skip files that have already been extracted
        if os.path.exists(wav_path) and os.path.exists(lab_path):
            skipped += 1
        else:
            audio_bytes = sample["audio"]["bytes"]

            if audio_bytes is None:
                raise ValueError(f"Audio bytes missing for recording {idx}")

            with open(wav_path, "wb") as f:
                f.write(audio_bytes)

            with open(lab_path, "w", encoding="utf-8") as f:
                f.write(str(transcript))

            if split == "train":
                saved_train += 1
            else:
                saved_val += 1

        if len(found_ids) % 250 == 0:
            print(f"Processed {len(found_ids)} / {len(target_ids)} recordings")

    # Stop when all selected recordings have been found
    if len(found_ids) == len(target_ids):
        break

# 6. Verify extraction
missing_ids = target_ids - found_ids

print("\nTELUGU EXTRACTION SUMMARY")
print("-------------------------")
print("Training recordings in CSV:", len(train_df))
print("Validation recordings in CSV:", len(val_df))
print("New training files saved:", saved_train)
print("New validation files saved:", saved_val)
print("Already extracted and skipped:", skipped)
print("Missing recording IDs:", len(missing_ids))

print("\nTraining folder:", train_dir)
print("Validation folder:", val_dir)

if missing_ids:
    print("Some recording IDs were not found. Please share this output.")
else:
    print("\nAll selected Telugu recordings were found!")

Loading IndicTTS Telugu dataset...
Starting Telugu audio extraction...
Processed 250 / 1532 recordings
Processed 500 / 1532 recordings
Processed 750 / 1532 recordings
Processed 1000 / 1532 recordings
Processed 1250 / 1532 recordings
Processed 1500 / 1532 recordings

TELUGU EXTRACTION SUMMARY
-------------------------
Training recordings in CSV: 1281
Validation recordings in CSV: 251
New training files saved: 1281
New validation files saved: 251
Already extracted and skipped: 0
Missing recording IDs: 0

Training folder: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/train
Validation folder: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/validation

All selected Telugu recordings were found!


In [ ]:
import os
import wave
import random

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

folders = {
    "English Train": os.path.join(base_dir, "English/raw_audio/train"),
    "English Validation": os.path.join(base_dir, "English/raw_audio/validation"),
    "Telugu Train": os.path.join(base_dir, "Telugu/raw_audio/train"),
    "Telugu Validation": os.path.join(base_dir, "Telugu/raw_audio/validation")
}

print("AUDIO FILE VERIFICATION")
print("=======================")

for folder_name, folder_path in folders.items():

    wav_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".wav")
    ]

    lab_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".lab")
    ]

    print(f"\n{folder_name}")
    print("WAV files:", len(wav_files))
    print("LAB files:", len(lab_files))

    # Check that each WAV has a matching transcript
    missing_lab = [
        f for f in wav_files
        if not os.path.exists(
            os.path.join(folder_path, os.path.splitext(f)[0] + ".lab")
        )
    ]

    print("Missing transcripts:", len(missing_lab))

    # Inspect up to 3 audio files
    if wav_files:
        samples = random.sample(wav_files, min(3, len(wav_files)))

        for filename in samples:
            filepath = os.path.join(folder_path, filename)

            with wave.open(filepath, "rb") as audio:
                sample_rate = audio.getframerate()
                channels = audio.getnchannels()
                frames = audio.getnframes()
                duration = frames / sample_rate

            print(
                f"  {filename}: "
                f"{sample_rate} Hz, "
                f"{channels} channel(s), "
                f"{duration:.2f} seconds"
            )

In [ ]:
import os
import pandas as pd

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

train_dir = os.path.join(base_dir, "English/raw_audio/train")
metadata_dir = os.path.join(base_dir, "metadata")

train_df = pd.read_csv(
    os.path.join(metadata_dir, "english_train.csv")
)

# Expected recording IDs from the training CSV
expected_ids = set(train_df["recording_id"].astype(int))

# Actual recording IDs from WAV filenames
actual_files = [
    f for f in os.listdir(train_dir)
    if f.endswith(".wav")
]

actual_ids = {
    int(f.replace("recording_", "").replace(".wav", ""))
    for f in actual_files
    if f.startswith("recording_")
}

# Compare expected and actual IDs
extra_ids = actual_ids - expected_ids
missing_ids = expected_ids - actual_ids

print("ENGLISH TRAINING FILE AUDIT")
print("===========================")
print("Expected recordings:", len(expected_ids))
print("Actual recording IDs:", len(actual_ids))
print("Extra recording IDs:", sorted(extra_ids))
print("Missing recording IDs:", sorted(missing_ids))

# Check WAV files without matching transcript files
for recording_id in sorted(extra_ids):
    wav_path = os.path.join(
        train_dir, f"recording_{recording_id}.wav"
    )
    lab_path = os.path.join(
        train_dir, f"recording_{recording_id}.lab"
    )

    print("\nExtra file:", os.path.basename(wav_path))
    print("WAV exists:", os.path.exists(wav_path))
    print("LAB exists:", os.path.exists(lab_path))

In [ ]:
import os
import shutil

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

source = os.path.join(
    base_dir, "English/raw_audio/train/sample_1952.wav"
)

test_dir = os.path.join(
    base_dir, "English/sample_audio_test"
)

os.makedirs(test_dir, exist_ok=True)

if os.path.exists(source):
    destination = os.path.join(test_dir, "sample_1952.wav")
    shutil.move(source, destination)
    print("Test recording moved successfully!")
else:
    print("Test recording is already outside the training folder.")

print("Test audio folder:", test_dir)

Test recording moved successfully!
Test audio folder: /content/drive/MyDrive/FastSpeech2_Assignment/English/sample_audio_test


In [ ]:
import os
import wave
from collections import Counter

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

folders = {
    "English Train": f"{base_dir}/English/raw_audio/train",
    "English Validation": f"{base_dir}/English/raw_audio/validation",
    "Telugu Train": f"{base_dir}/Telugu/raw_audio/train",
    "Telugu Validation": f"{base_dir}/Telugu/raw_audio/validation"
}

for name, folder in folders.items():

    sample_rates = Counter()
    invalid_files = []

    wav_files = [
        f for f in os.listdir(folder)
        if f.lower().endswith(".wav")
    ]

    for filename in wav_files:
        path = os.path.join(folder, filename)

        try:
            with wave.open(path, "rb") as audio:
                sample_rates[audio.getframerate()] += 1
        except Exception:
            invalid_files.append(filename)

    print(f"\n{name}")
    print("-" * 35)
    print("Total WAV files:", len(wav_files))
    print("Sample rate distribution:", dict(sample_rates))
    print("Unreadable files:", len(invalid_files))


English Train
-----------------------------------
Total WAV files: 2664
Sample rate distribution: {16000: 2664}
Unreadable files: 0

English Validation
-----------------------------------
Total WAV files: 505
Sample rate distribution: {16000: 505}
Unreadable files: 0

Telugu Train
-----------------------------------
Total WAV files: 1281
Sample rate distribution: {48000: 864, 16000: 417}
Unreadable files: 0

Telugu Validation
-----------------------------------
Total WAV files: 251
Sample rate distribution: {16000: 80, 48000: 171}
Unreadable files: 0


In [ ]:
import os
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
from math import gcd

# Target sample rate required by our HiFi-GAN configuration
TARGET_SR = 22050

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

folders = {
    "English Train": f"{base_dir}/English/raw_audio/train",
    "English Validation": f"{base_dir}/English/raw_audio/validation",
    "Telugu Train": f"{base_dir}/Telugu/raw_audio/train",
    "Telugu Validation": f"{base_dir}/Telugu/raw_audio/validation"
}

# Separate output folders for processed audio
processed_base = os.path.join(base_dir, "processed_audio_22050")

total_processed = 0
total_skipped = 0
total_errors = 0

for folder_name, input_dir in folders.items():

    output_dir = os.path.join(
        processed_base,
        folder_name.replace(" ", "_").lower()
    )

    os.makedirs(output_dir, exist_ok=True)

    wav_files = [
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".wav")
    ]

    processed = 0
    skipped = 0
    errors = 0

    print(f"\nProcessing {folder_name}...")

    for filename in wav_files:

        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        # Avoid processing an already completed file again
        if os.path.exists(output_path):
            skipped += 1
            continue

        try:
            original_sr, audio = wavfile.read(input_path)

            # Convert audio to floating-point representation
            if np.issubdtype(audio.dtype, np.integer):
                max_value = max(
                    abs(np.iinfo(audio.dtype).min),
                    np.iinfo(audio.dtype).max
                )
                audio = audio.astype(np.float32) / max_value
            else:
                audio = audio.astype(np.float32)

            # Resample only when necessary
            if original_sr != TARGET_SR:
                common_divisor = gcd(original_sr, TARGET_SR)

                up = TARGET_SR // common_divisor
                down = original_sr // common_divisor

                audio = resample_poly(
                    audio,
                    up,
                    down,
                    axis=0
                )

            # Convert to 16-bit PCM WAV
            audio = np.clip(audio, -1.0, 1.0)
            audio = (audio * 32767).astype(np.int16)

            wavfile.write(
                output_path,
                TARGET_SR,
                audio
            )

            processed += 1

        except Exception as e:
            errors += 1
            print(f"Error in {filename}: {e}")

    total_processed += processed
    total_skipped += skipped
    total_errors += errors

    print("Newly processed:", processed)
    print("Already processed:", skipped)
    print("Errors:", errors)

print("\nRESAMPLING SUMMARY")
print("==================")
print("Newly processed files:", total_processed)
print("Already processed files:", total_skipped)
print("Errors:", total_errors)
print("Target sample rate:", TARGET_SR)
print("Output folder:", processed_base)


Processing English Train...
Newly processed: 2664
Already processed: 0
Errors: 0

Processing English Validation...
Newly processed: 505
Already processed: 0
Errors: 0

Processing Telugu Train...
Newly processed: 1281
Already processed: 0
Errors: 0

Processing Telugu Validation...
Newly processed: 251
Already processed: 0
Errors: 0

RESAMPLING SUMMARY
Newly processed files: 4701
Already processed files: 0
Errors: 0
Target sample rate: 22050
Output folder: /content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050


In [ ]:
import os
import wave
from collections import Counter

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"
processed_base = os.path.join(base_dir, "processed_audio_22050")

folders = {
    "English Train": "english_train",
    "English Validation": "english_validation",
    "Telugu Train": "telugu_train",
    "Telugu Validation": "telugu_validation"
}

for name, folder in folders.items():

    folder_path = os.path.join(processed_base, folder)

    sample_rates = Counter()
    unreadable = 0

    wav_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".wav")
    ]

    for filename in wav_files:
        try:
            with wave.open(
                os.path.join(folder_path, filename), "rb"
            ) as audio:
                sample_rates[audio.getframerate()] += 1
        except Exception:
            unreadable += 1

    print(f"\n{name}")
    print("WAV files:", len(wav_files))
    print("Sample rates:", dict(sample_rates))
    print("Unreadable files:", unreadable)


English Train
WAV files: 2664
Sample rates: {22050: 2664}
Unreadable files: 0

English Validation
WAV files: 505
Sample rates: {22050: 505}
Unreadable files: 0

Telugu Train
WAV files: 1281
Sample rates: {22050: 1281}
Unreadable files: 0

Telugu Validation
WAV files: 251
Sample rates: {22050: 251}
Unreadable files: 0


In [ ]:
import os
import pandas as pd
from collections import Counter

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"
metadata_dir = os.path.join(base_dir, "metadata")

datasets = {
    "English Train": "english_train.csv",
    "English Validation": "english_validation.csv",
    "Telugu Train": "telugu_train.csv",
    "Telugu Validation": "telugu_validation.csv"
}

for name, filename in datasets.items():

    df = pd.read_csv(os.path.join(metadata_dir, filename))
    texts = df["text"].dropna().astype(str)

    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)

    print("Total transcripts:", len(texts))
    print("Empty transcripts:", (texts.str.strip() == "").sum())

    # Display three sample transcripts
    print("\nSample transcripts:")
    for text in texts.head(3):
        print("-", text)

    # Count unique characters
    characters = Counter(
        char for text in texts for char in text
    )

    print("\nUnique characters:", len(characters))
    print("Character examples:", repr("".join(characters.keys())[:150]))


English Train
Total transcripts: 2664
Empty transcripts: 0

Sample transcripts:
- so it definitely influences parts of our language
- I have reset my MPIN to 6279
- Of course, there is a lot of funding by the state, the government-run schools,

Unique characters: 81
Character examples: 'so itdefnlyucpargIhvmMPN6279O,b-TR.10ZACz?wGkSWBYKEDqj"FJ\'H483U5x%LV&XQ@+!_\r\n:();'

English Validation
Total transcripts: 505
Empty transcripts: 0

Sample transcripts:
- improving the healthcare infrastructure as well as the healthcare providing services throughout the state
- I had a cooking disaster when I tried to prepare a classic North Karnataka dish
- So first Baisak we used to celebrate New Year by organizing traditional or cultural program.

Unique characters: 74
Character examples: 'improvng thealcfsuwdIkNKSBbYyz.W?TH,F1EV2CDJG50AP368xOX974+L\'q-RjUM%"&!()Z'

Telugu Train
Total transcripts: 1281
Empty transcripts: 0

Sample transcripts:
- కొంతదూరం అడవిలోంచి నడిచి అందరు ఒక కొండప్రాంతం చేరుకు

In [ ]:
import os

repo_path = "/content/FastSpeech2"

files_to_inspect = [
    "text/symbols.py",
    "text/cleaners.py",
    "text/__init__.py"
]

for filename in files_to_inspect:

    filepath = os.path.join(repo_path, filename)

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("File not found:", filepath)


text/symbols.py
""" from https://github.com/keithito/tacotron """

"""
Defines the set of symbols used in text input to the model.

The default is a set of ASCII characters that works well for English or text that has been run through Unidecode. For other data, you can modify _characters. See TRAINING_DATA.md for details. """

from text import cmudict, pinyin

_pad = "_"
_punctuation = "!'(),.:;? "
_special = "-"
_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
_silences = ["@sp", "@spn", "@sil"]

# Prepend "@" to ARPAbet symbols to ensure uniqueness (some are the same as uppercase letters):
_arpabet = ["@" + s for s in cmudict.valid_symbols]
_pinyin = ["@" + s for s in pinyin.valid_symbols]

# Export all symbols:
symbols = (
    [_pad]
    + list(_special)
    + list(_punctuation)
    + list(_letters)
    + _arpabet
    + _pinyin
    + _silences
)


text/cleaners.py
""" from https://github.com/keithito/tacotron """

'''
Cleaners are transformations that run over the 

In [ ]:
import os

repo_path = "/content/FastSpeech2"

files_to_inspect = [
    "preprocessor/preprocessor.py",
    "preprocessor/preprocessor_helpers.py",
    "preprocessor/libritts.py",
    "preprocessor/ljspeech.py",
    "dataset.py",
    "model/fastspeech2.py"
]

for filename in files_to_inspect:

    filepath = os.path.join(repo_path, filename)

    print("\n" + "=" * 75)
    print(filename)
    print("=" * 75)

    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()

        # Display the first 250 lines to keep output manageable
        lines = content.splitlines()
        print("\n".join(lines[:250]))

        if len(lines) > 250:
            print(f"\n... {len(lines) - 250} additional lines not displayed.")
    else:
        print("File not found.")


preprocessor/preprocessor.py
import os
import random
import json

import tgt
import librosa
import numpy as np
import pyworld as pw
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import audio as Audio


class Preprocessor:
    def __init__(self, config):
        self.config = config
        self.in_dir = config["path"]["raw_path"]
        self.out_dir = config["path"]["preprocessed_path"]
        self.val_size = config["preprocessing"]["val_size"]
        self.sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
        self.hop_length = config["preprocessing"]["stft"]["hop_length"]

        assert config["preprocessing"]["pitch"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        assert config["preprocessing"]["energy"]["feature"] in [
            "phoneme_level",
            "frame_level",
        ]
        self.pitch_phoneme_averaging = (
            config["preproces

In [ ]:
import os
import yaml

repo_path = "/content/FastSpeech2"

config_files = [
    "config/LJSpeech/preprocess.yaml",
    "config/LJSpeech/model.yaml",
    "config/LJSpeech/train.yaml"
]

for filename in config_files:

    filepath = os.path.join(repo_path, filename)

    print("\n" + "=" * 65)
    print(filename)
    print("=" * 65)

    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            config = yaml.safe_load(f)

        print(yaml.dump(config, sort_keys=False, allow_unicode=True))
    else:
        print("File not found:", filepath)


config/LJSpeech/preprocess.yaml
dataset: LJSpeech
path:
  corpus_path: /home/ming/Data/LJSpeech-1.1
  lexicon_path: lexicon/librispeech-lexicon.txt
  raw_path: ./raw_data/LJSpeech
  preprocessed_path: ./preprocessed_data/LJSpeech
preprocessing:
  val_size: 512
  text:
    text_cleaners:
    - english_cleaners
    language: en
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: 80
    mel_fmin: 0
    mel_fmax: 8000
  pitch:
    feature: phoneme_level
    normalization: true
  energy:
    feature: phoneme_level
    normalization: true


config/LJSpeech/model.yaml
transformer:
  encoder_layer: 4
  encoder_head: 2
  encoder_hidden: 256
  decoder_layer: 6
  decoder_head: 2
  decoder_hidden: 256
  conv_filter_size: 1024
  conv_kernel_size:
  - 9
  - 1
  encoder_dropout: 0.2
  decoder_dropout: 0.2
variance_predictor:
  filter_size: 256
  kernel_size: 3
  dropout: 0.5
variance_embeddin

In [ ]:
import importlib.util
import shutil
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

print("\nALIGNMENT TOOLS")
print("----------------")

print("MFA executable:", shutil.which("mfa"))

packages = [
    "tgt",
    "pyworld",
    "librosa",
    "scipy",
    "unidecode",
    "phonemizer",
    "indicnlp"
]

for package in packages:
    print(
        package,
        ":",
        "Installed" if importlib.util.find_spec(package) else "Not installed"
    )

GPU: Not available

ALIGNMENT TOOLS
----------------
MFA executable: None
tgt : Not installed
pyworld : Not installed
librosa : Installed
scipy : Installed
unidecode : Installed
phonemizer : Not installed
indicnlp : Not installed


In [ ]:
import torch
import os

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

CUDA available: False
CUDA device count: 0
CUDA_VISIBLE_DEVICES: None


In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch: 2.11.0+cpu
CUDA available: False
Selected device: cpu
Running on CPU


In [ ]:
import os

project_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

print("Project folder exists:", os.path.exists(project_dir))

if os.path.exists(project_dir):
    print("\nProject folders:")
    for folder in os.listdir(project_dir):
        print("-", folder)

Project folder exists: True

Project folders:
- raw_data
- English
- Telugu
- metadata
- processed_audio_22050


In [ ]:
import os

base_dir = "/content/drive/MyDrive/FastSpeech2_Assignment"

folders = {
    "English Train": "processed_audio_22050/english_train",
    "English Validation": "processed_audio_22050/english_validation",
    "Telugu Train": "processed_audio_22050/telugu_train",
    "Telugu Validation": "processed_audio_22050/telugu_validation"
}

for name, relative_path in folders.items():
    folder_path = os.path.join(base_dir, relative_path)

    wav_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".wav")
    ]

    print(f"{name}: {len(wav_files)} WAV files")

English Train: 2664 WAV files
English Validation: 505 WAV files
Telugu Train: 1281 WAV files
Telugu Validation: 251 WAV files


In [ ]:
import os

repo_path = "/content/FastSpeech2"

print("Repository exists:", os.path.exists(repo_path))

if os.path.exists(repo_path):
    print("\nRepository files:")
    print(os.listdir(repo_path)[:20])

Repository exists: True

Repository files:
['model', 'preprocessed_data', 'index.html', 'preprocess.py', 'audio', 'README.md', 'demo', 'dataset.py', '.git', 'img', 'utils', 'hifigan', 'text', 'evaluate.py', 'prepare_align.py', 'transformer', 'config', 'lexicon', 'preprocessor', 'synthesize.py']


In [ ]:
import importlib.util
import os

os.chdir("/content/FastSpeech2")

packages = [
    "torch",
    "librosa",
    "numpy",
    "scipy",
    "pandas",
    "pyworld",
    "tgt",
    "unidecode",
    "matplotlib"
]

print("PACKAGE CHECK")
print("----------------")

for package in packages:
    status = "Installed" if importlib.util.find_spec(package) else "Missing"
    print(f"{package}: {status}")

PACKAGE CHECK
----------------
torch: Installed
librosa: Installed
numpy: Installed
scipy: Installed
pandas: Installed
pyworld: Missing
tgt: Missing
unidecode: Installed
matplotlib: Installed


In [ ]:
!pip install pyworld --no-cache-dir

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyworld: filename=pyworld-0.3.5-cp313-cp313-linux_x86_64.whl size=1020160 sha256=a2c362d6bad0f6a3a4d682fb406061e510c7faea3cc0ac3ae9fe60ec08e33618
  Stored in directory: /tmp/pip-ephem-wheel-cache-e2j6ij9a/wheels/8b/c7/b2/0fb039381aa5f5b31c58216a65288315fda280becb7ae8f991
Successfully built pyworld


In [ ]:
!pip install tgt

In [ ]:
import pyworld
import tgt

print("pyworld: Installed")
print("tgt: Installed")
print("Both packages imported successfully!")

pyworld: Installed
tgt: Installed
Both packages imported successfully!


In [ ]:
import shutil

mfa_path = shutil.which("mfa")

if mfa_path:
    print("MFA is installed:", mfa_path)
else:
    print("MFA is not installed.")

MFA is not installed.


In [ ]:
import shutil

print("Conda:", shutil.which("conda"))
print("Mamba:", shutil.which("mamba"))

Conda: None
Mamba: None


In [ ]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Python executable: /usr/bin/python3


In [ ]:
import shutil

total, used, free = shutil.disk_usage("/content")

print("Total space:", round(total / (1024**3), 2), "GB")
print("Used space:", round(used / (1024**3), 2), "GB")
print("Free space:", round(free / (1024**3), 2), "GB")

Total space: 107.72 GB
Used space: 4.52 GB
Free space: 103.18 GB


In [ ]:
import requests

url = "https://github.com/conda-forge/miniforge/releases/latest"

response = requests.get(url, allow_redirects=True, timeout=30)

print("HTTP status:", response.status_code)
print("Installer page accessible:", response.ok)
print("Final URL:", response.url)

HTTP status: 200
Installer page accessible: True
Final URL: https://github.com/conda-forge/miniforge/releases/tag/26.9.4-1


In [ ]:
import platform

print("Operating system:", platform.system())
print("Architecture:", platform.machine())

Operating system: Linux
Architecture: x86_64


In [ ]:
%cd /content

!wget -q https://github.com/conda-forge/miniforge/releases/download/26.9.4-1/Miniforge3-26.9.4-1-Linux-x86_64.sh -O miniforge.sh

import os

print("Installer downloaded:", os.path.exists("/content/miniforge.sh"))
print("Installer size:", round(os.path.getsize("/content/miniforge.sh") / (1024**2), 2), "MB")

/content
Installer downloaded: True
Installer size: 122.19 MB


In [ ]:
!bash /content/miniforge.sh -b -p /content/miniforge3

PREFIX=/content/miniforge3
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.10.23-hbd8a1cb_0.conda
Extracting libgomp-16.2.0-he0feb66_7.conda
Extracting libzlib-1.3.2-h25fd6f3_3.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_2.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.14-9_cp314.conda
Extracting tzdata-2026e-h5d301ff_0.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_7.conda
Extracting ld_impl_linux-64-2.46.1-default_hbd61a6d_103.conda
Extracting libgcc-16.2.0-ha9f2e26_7.conda
Extracting bzip2-1.0.8-hda65f42_10.conda
Extracting c-ares-1.34.9-hebe6cf0_0.conda
Extracting keyutils-1.6.3-h7cc23a3_1.conda
Extracting libev-4.33-h280c20c_3.conda
Extracting libexpat-2.8.5-hd2095e1_0.conda
Extracting libffi-3.7.0-h81df57d_1.conda
Extracting libiconv-1.18-h0cb94f2_3.conda
Extracting liblzma-5.8.3-hb03c661_1.conda
Extracting libmpdec-4.0.0-hb03c661_2.conda
Extracting libpython-3.14.10-hd51c5da_100_cp31

In [ ]:
import subprocess
import os

conda = "/content/miniforge3/bin/conda"
mamba = "/content/miniforge3/bin/mamba"

print("Conda exists:", os.path.exists(conda))
print("Mamba exists:", os.path.exists(mamba))

print("\nConda version:")
subprocess.run([conda, "--version"])

print("\nMamba version:")
subprocess.run([mamba, "--version"])

Conda exists: True
Mamba exists: True

Conda version:

Mamba version:


CompletedProcess(args=['/content/miniforge3/bin/mamba', '--version'], returncode=0)

In [ ]:
!/content/miniforge3/bin/conda --version
!/content/miniforge3/bin/mamba --version

conda 26.9.4
2.9.0


In [ ]:
!/content/miniforge3/bin/mamba create -y -n mfa_env python=3.10

[+] 0.0s
[+] 0.1s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.2 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.2 sec)
Fetching and Parsing Packages' Shards                                                    ✔ Done (10.4 sec)

Resolving Environment                                                                     ✔ Done (0.3 sec)

Transaction

  Prefix: /content/miniforge3/envs/mfa_env

  Updating specs:

   - python=3.10
   - pip


  Package                Version  Build                 Channel           Size
────────────────────────────────────────────────────────────────────────────────
  Install:
────────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex            4.5  20_gnu                conda-forge     Cached
  + bzip2                  1.0.8  hda65f42_10           conda-forge     Cached
  + ca-certificates   2026.10.23  hbd

In [ ]:
!/content/miniforge3/bin/mamba install -y -n mfa_env -c conda-forge montreal-forced-aligner

Using Cached Shard Index for conda-forge/linux-64                                                   ✔ Done
Using Cached Shard Index for conda-forge/noarch                                                     ✔ Done
Fetching and Parsing Packages' Shards                                                    ✔ Done (52.2 sec)

Pinned packages:

  - python=3.10

Resolving Environment                                                                     ✔ Done (2.0 sec)

Transaction

  Prefix: /content/miniforge3/envs/mfa_env

  Updating specs:

   - montreal-forced-aligner


  Package                                      Version  Build                  Channel           Size
───────────────────────────────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────────────────────────────

  + _python_abi3_support                           1.0  hd8ed1ab_3             conda-forge     Cached
  + 

In [ ]:
import os
import subprocess

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"

commands = {
    "MFA VERSION": [MFA, "version"],
    "ACOUSTIC MODELS": [MFA, "model", "list", "acoustic"],
    "PRONUNCIATION DICTIONARIES": [MFA, "model", "list", "dictionary"],
    "G2P MODELS": [MFA, "model", "list", "g2p"],
}

for title, command in commands.items():
    print("\n" + "=" * 65)
    print(title)
    print("=" * 65)

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=180
    )

    print(result.stdout if result.stdout else "No standard output.")

    if result.stderr:
        print("Messages:", result.stderr[:2000])

    print("Return code:", result.returncode)


MFA VERSION
3.4.2

Return code: 0

ACOUSTIC MODELS
No standard output.
Messages: No models found

Return code: 0

PRONUNCIATION DICTIONARIES
No standard output.
Messages: No models found

Return code: 0

G2P MODELS
No standard output.
Messages: No models found

Return code: 0


In [ ]:
import os
import shutil
import subprocess
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. Project paths
# --------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"

METADATA = BASE / "metadata/english_train.csv"

AUDIO_DIR = BASE / "processed_audio_22050/english_train"
TRANSCRIPT_DIR = BASE / "English/raw_audio/train"

ALIGNMENT_DIR = BASE / "English/alignment"
CORPUS_DIR = ALIGNMENT_DIR / "smoke_test_corpus"
OUTPUT_DIR = ALIGNMENT_DIR / "smoke_test_output"

# --------------------------------------------------
# 2. Load existing metadata
# --------------------------------------------------

df = pd.read_csv(METADATA)

# Select 10 recordings for an initial alignment test
test_df = df.head(10)

speaker_dir = CORPUS_DIR / "speaker_1"
speaker_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 3. Prepare audio and transcript files
# --------------------------------------------------

copied = 0
missing = []

for _, row in test_df.iterrows():

    recording_id = int(row["recording_id"])

    audio_name = f"recording_{recording_id}.wav"

    source_audio = AUDIO_DIR / audio_name
    source_text = TRANSCRIPT_DIR / f"recording_{recording_id}.lab"

    destination_audio = speaker_dir / audio_name
    destination_text = speaker_dir / f"recording_{recording_id}.lab"

    if not source_audio.exists() or not source_text.exists():
        missing.append(recording_id)
        continue

    shutil.copy2(source_audio, destination_audio)
    shutil.copy2(source_text, destination_text)

    copied += 1

print("Recordings prepared:", copied)
print("Missing recordings:", missing)

if copied == 0:
    raise RuntimeError("No recordings were prepared. Check the source paths.")

# --------------------------------------------------
# 4. Run Indian English forced alignment
# --------------------------------------------------

command = [
    MFA,
    "align_hf",
    str(CORPUS_DIR),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT_DIR),
    "--dialect", "english_india_mfa",
    "--use_g2p",
    "--single_speaker",
    "-j", "2",
    "--overwrite"
]

print("\nStarting English alignment test...")
print("Model: MontrealCorpusTools/english_mfa")
print("Dictionary: english_india_mfa")

result = subprocess.run(command, text=True)

print("\nMFA return code:", result.returncode)

# --------------------------------------------------
# 5. Verify generated TextGrid files
# --------------------------------------------------

textgrids = list(OUTPUT_DIR.rglob("*.TextGrid"))

print("\nAlignment results")
print("-----------------")
print("TextGrid files generated:", len(textgrids))

for file in textgrids[:10]:
    print(file.name)

if result.returncode == 0 and len(textgrids) > 0:
    print("\nSUCCESS: English alignment test produced TextGrid files.")
else:
    print("\nAlignment needs troubleshooting. Check the MFA output above.")

Recordings prepared: 10
Missing recordings: []

Starting English alignment test...
Model: MontrealCorpusTools/english_mfa
Dictionary: english_india_mfa

MFA return code: 1

Alignment results
-----------------
TextGrid files generated: 0

Alignment needs troubleshooting. Check the MFA output above.


In [ ]:
import subprocess
from pathlib import Path

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
CORPUS = BASE / "English/alignment/smoke_test_corpus"
OUTPUT = BASE / "English/alignment/smoke_test_output"

# Run alignment with the corrected Indian English dialect name
command = [
    MFA,
    "align_hf",
    str(CORPUS),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT),
    "--dialect", "english_india",
    "--use_g2p",
    "--single_speaker",
    "-j", "2",
    "--overwrite",
    "--verbose"
]

print("Starting corrected English alignment...")
print("Dialect: english_india")
print("Model: MontrealCorpusTools/english_mfa")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("\n========== MFA OUTPUT ==========")
print(result.stdout)

print("\n========== MFA ERRORS / LOGS ==========")
print(result.stderr)

print("\nReturn code:", result.returncode)

# Verify alignment files
textgrids = list(OUTPUT.rglob("*.TextGrid"))

print("\n========== ALIGNMENT SUMMARY ==========")
print("TextGrid files generated:", len(textgrids))

for file in textgrids[:10]:
    print(file.name)

if result.returncode == 0 and len(textgrids) > 0:
    print("\nSUCCESS: English alignment test completed!")
else:
    print("\nAlignment still needs troubleshooting. The captured logs above will show why.")

Starting corrected English alignment...
Dialect: english_india
Model: MontrealCorpusTools/english_mfa

========== MFA OUTPUT ==========


========== MFA ERRORS / LOGS ==========

Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 1147.75it/s]
Traceback (most recent call last):
  File "/content/miniforge3/envs/mfa_env/bin/mfa", line 10, in <module>
    sys.exit(mfa_cli())
  File "/content/miniforge3/envs/mfa_env/lib/python3.10/site-packages/rich_click/rich_command.py", line 402, in __call__
    return super().__call__(*args, **kwargs)
  File "/content/miniforge3/envs/mfa_env/lib/python3.10/site-packages/click/core.py", line 1569, in __call__
    return self.main(*args, **kwargs)
  File "/content/miniforge3/envs/mfa_env/lib/python3.10/site-packages/rich_click/rich_command.py", line 216, in main
    rv = self.invoke(ctx)
  File "/content/miniforge3/envs/mfa_env/lib/python3.10/site-packages/click/core.py", line 1970, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx)

In [ ]:
import os
import shutil
import subprocess

CONDA = "/content/miniforge3/bin/mamba"
ENV = "/content/miniforge3/envs/mfa_env"
MFA = f"{ENV}/bin/mfa"

# 1. Install OpenFst in the existing MFA environment
print("Installing/verifying OpenFst...")

install = subprocess.run([
    CONDA, "install",
    "-y",
    "-n", "mfa_env",
    "-c", "conda-forge",
    "openfst"
], text=True)

if install.returncode != 0:
    raise RuntimeError("OpenFst installation failed.")

# 2. Add MFA environment binaries to PATH
os.environ["PATH"] = f"{ENV}/bin:" + os.environ["PATH"]

# 3. Verify required executables
required_tools = [
    "fstcompile",
    "fstinfo",
    "fstarcsort",
    "fstprint"
]

print("\nChecking OpenFst executables:")

for tool in required_tools:
    location = shutil.which(tool)
    print(f"{tool}: {location}")

# 4. Verify MFA
print("\nChecking MFA version:")

version = subprocess.run(
    [MFA, "version"],
    capture_output=True,
    text=True
)

print(version.stdout or version.stderr)

# 5. Verify third-party dependencies
print("\nChecking MFA dependencies:")

check = subprocess.run(
    [MFA, "validate"],
    capture_output=True,
    text=True,
    timeout=180
)

print(check.stdout)
if check.stderr:
    print(check.stderr)

print("\nValidation return code:", check.returncode)

Installing/verifying OpenFst...

Checking OpenFst executables:
fstcompile: /content/miniforge3/envs/mfa_env/bin/fstcompile
fstinfo: /content/miniforge3/envs/mfa_env/bin/fstinfo
fstarcsort: /content/miniforge3/envs/mfa_env/bin/fstarcsort
fstprint: /content/miniforge3/envs/mfa_env/bin/fstprint

Checking MFA version:
3.4.2


Checking MFA dependencies:

                                                                                
 Usage: mfa validate [OPTIONS] CORPUS_DIRECTORY DICTIONARY_PATH                 
                                                                                
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Missing argument 'CORPUS_DIRECTORY'.                                         │
╰──────────────────────────────────────────────────────────────────────────────╯
                                                                                


Validation return code: 2


In [ ]:
import os
import subprocess
from pathlib import Path

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
ENV_BIN = "/content/miniforge3/envs/mfa_env/bin"

# Ensure OpenFst executables are accessible
os.environ["PATH"] = ENV_BIN + ":" + os.environ["PATH"]

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
CORPUS = BASE / "English/alignment/smoke_test_corpus"
OUTPUT = BASE / "English/alignment/smoke_test_output"

OUTPUT.mkdir(parents=True, exist_ok=True)

command = [
    MFA,
    "align_hf",
    str(CORPUS),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT),
    "--dialect", "english_india",
    "--use_g2p",
    "--single_speaker",
    "-j", "2",
    "--overwrite",
    "--verbose"
]

print("Starting English alignment test...")
print("Input recordings:", len(list(CORPUS.rglob("*.wav"))))
print("Input transcripts:", len(list(CORPUS.rglob("*.lab"))))

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("\n========== MFA OUTPUT ==========")
print(result.stdout)

print("\n========== MFA ERRORS ==========")
print(result.stderr)

print("\nReturn code:", result.returncode)

textgrids = list(OUTPUT.rglob("*.TextGrid"))

print("\n========== ALIGNMENT SUMMARY ==========")
print("TextGrid files generated:", len(textgrids))

for file in textgrids:
    print(file.name)

if result.returncode == 0 and len(textgrids) == 10:
    print("\nSUCCESS: All 10 English recordings were aligned!")
elif textgrids:
    print(f"\nPARTIAL SUCCESS: {len(textgrids)} TextGrid files generated.")
else:
    print("\nAlignment did not complete. We need to inspect the error output.")

Starting English alignment test...
Input recordings: 10
Input transcripts: 10

========== MFA OUTPUT ==========
  10% ━━━╸                                10/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7  [ 0:00:02 < 0:00:00 , 2 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:22 < 0:00:00 , 8 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:12 < 0:00:00 , 5 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10  [ 0:00:02 < 0:00:00 , 9 it/s ]


========== MFA ERRORS ==========

Fetching 42 files: 100%|██████████| 42/42 

In [ ]:
import os
import shutil
import subprocess
import pandas as pd
from pathlib import Path
import tgt

# --------------------------------------------------
# 1. PATHS AND SETTINGS
# --------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
ENV_BIN = "/content/miniforge3/envs/mfa_env/bin"

os.environ["PATH"] = ENV_BIN + ":" + os.environ["PATH"]

# Existing metadata and audio folders
METADATA_DIR = BASE / "metadata"

PROCESSED_AUDIO = BASE / "processed_audio_22050"
RAW_AUDIO = BASE / "English/raw_audio"

# Temporary local corpus folders
LOCAL_CORPUS = Path("/content/english_mfa_corpus")

# Permanent alignment output folders
ALIGNMENT_OUTPUT = BASE / "English/alignment/full_alignment"

ALIGNMENT_OUTPUT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. CHECK THE SMOKE-TEST TEXTGRID STRUCTURE
# --------------------------------------------------

smoke_dir = BASE / "English/alignment/smoke_test_output"
smoke_files = list(smoke_dir.rglob("*.TextGrid"))

if not smoke_files:
    raise FileNotFoundError("No smoke-test TextGrid files found.")

sample_grid = tgt.io.read_textgrid(str(smoke_files[0]))

tier_names = [tier.name for tier in sample_grid.tiers]

print("Sample TextGrid:", smoke_files[0].name)
print("Available tiers:", tier_names)

if "phones" not in tier_names:
    raise RuntimeError(
        "The required 'phones' tier was not found. "
        "Do not start full alignment until this is resolved."
    )

print("TextGrid structure verified.")

# --------------------------------------------------
# 3. PREPARE TRAINING AND VALIDATION CORPORA
# --------------------------------------------------

splits = {
    "train": {
        "csv": METADATA_DIR / "english_train.csv",
        "audio": PROCESSED_AUDIO / "english_train",
        "transcripts": RAW_AUDIO / "train"
    },
    "validation": {
        "csv": METADATA_DIR / "english_validation.csv",
        "audio": PROCESSED_AUDIO / "english_validation",
        "transcripts": RAW_AUDIO / "validation"
    }
}

for split_name, paths in splits.items():

    print(f"\nPreparing {split_name} corpus...")

    df = pd.read_csv(paths["csv"])

    speaker_dir = LOCAL_CORPUS / split_name / "speaker_1"
    speaker_dir.mkdir(parents=True, exist_ok=True)

    copied = 0
    missing = []

    for recording_id in df["recording_id"]:

        filename = f"recording_{int(recording_id)}"

        wav_source = paths["audio"] / f"{filename}.wav"
        lab_source = paths["transcripts"] / f"{filename}.lab"

        wav_target = speaker_dir / f"{filename}.wav"
        lab_target = speaker_dir / f"{filename}.lab"

        if not wav_source.exists() or not lab_source.exists():
            missing.append(filename)
            continue

        # Link audio files to avoid copying large WAV files.
        if not wav_target.exists():
            os.symlink(wav_source, wav_target)

        shutil.copy2(lab_source, lab_target)
        copied += 1

    print("Expected recordings:", len(df))
    print("Prepared recordings:", copied)
    print("Missing recordings:", len(missing))

    if missing:
        raise RuntimeError(
            f"{split_name}: missing files. Examples: {missing[:10]}"
        )

# --------------------------------------------------
# 4. ALIGN TRAINING AND VALIDATION SEPARATELY
# --------------------------------------------------

for split_name in ["train", "validation"]:

    corpus_dir = LOCAL_CORPUS / split_name
    output_dir = ALIGNMENT_OUTPUT / split_name

    output_dir.mkdir(parents=True, exist_ok=True)

    expected = len(list(corpus_dir.rglob("*.wav")))

    print(f"\n{'=' * 55}")
    print(f"Starting full English {split_name} alignment")
    print("Recordings:", expected)
    print(f"{'=' * 55}")

    command = [
        MFA,
        "align_hf",
        str(corpus_dir),
        "MontrealCorpusTools/english_mfa",
        str(output_dir),
        "--dialect", "english_india",
        "--use_g2p",
        "--single_speaker",
        "-j", "2",
        "--overwrite"
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print("\nMFA output:")
    print(result.stdout[-5000:] if result.stdout else "No standard output.")

    if result.stderr:
        print("\nMFA messages:")
        print(result.stderr[-5000:])

    textgrids = list(output_dir.rglob("*.TextGrid"))

    print("\nExpected TextGrids:", expected)
    print("Generated TextGrids:", len(textgrids))
    print("Return code:", result.returncode)

    if result.returncode != 0 or len(textgrids) != expected:
        raise RuntimeError(
            f"{split_name} alignment was incomplete. "
            "Check the MFA output above before proceeding."
        )

    print(f"{split_name.upper()} ALIGNMENT SUCCESSFUL!")

# --------------------------------------------------
# 5. FINAL SUMMARY
# --------------------------------------------------

train_count = len(
    list((ALIGNMENT_OUTPUT / "train").rglob("*.TextGrid"))
)

val_count = len(
    list((ALIGNMENT_OUTPUT / "validation").rglob("*.TextGrid"))
)

print("\n========== FINAL ENGLISH ALIGNMENT ==========")
print("Training TextGrids:", train_count)
print("Validation TextGrids:", val_count)
print("Total TextGrids:", train_count + val_count)
print("Saved in:", ALIGNMENT_OUTPUT)

if train_count == 2664 and val_count == 505:
    print("\nSUCCESS: All English recordings aligned!")
else:
    print("\nWARNING: Check the counts before continuing.")

Sample TextGrid: recording_1437.TextGrid
Available tiers: ['words', 'phones']
TextGrid structure verified.

Preparing train corpus...
Expected recordings: 2664
Prepared recordings: 2664
Missing recordings: 0

Preparing validation corpus...
Expected recordings: 505
Prepared recordings: 505
Missing recordings: 0

Starting full English train alignment
Recordings: 2664

MFA output:
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/100  [ 0:00:00 < 0:00:00 , 362 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/2,664  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1,365/1,365  [ 0:07:42 < 0:00:00 , 3 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/2,664  [ 0:00:46 < 0:00:00 , 47 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/2,664  [ 0:00:02 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/2,664  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/2,664  [ 0:02:34 < 0:00:00 , 17 it/s ]
  99% ━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 2,647/2,664  [ 0:

RuntimeError: train alignment was incomplete. Check the MFA output above before proceeding.

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

metadata_path = BASE / "metadata/english_train.csv"
audio_dir = BASE / "processed_audio_22050/english_train"
transcript_dir = BASE / "English/raw_audio/train"
output_dir = BASE / "English/alignment/full_alignment/train"

df = pd.read_csv(metadata_path)

expected_ids = set(df["recording_id"].astype(int))

generated_ids = {
    int(path.stem.replace("recording_", ""))
    for path in output_dir.glob("recording_*.TextGrid")
    if path.stem.replace("recording_", "").isdigit()
}

missing_ids = sorted(expected_ids - generated_ids)

print("Expected recordings:", len(expected_ids))
print("Generated TextGrids:", len(generated_ids))
print("Missing alignments:", len(missing_ids))

print("\n========== MISSING RECORDINGS ==========")

for recording_id in missing_ids:
    name = f"recording_{recording_id}"

    wav = audio_dir / f"{name}.wav"
    lab = transcript_dir / f"{name}.lab"

    print(f"\n{name}")
    print("WAV exists:", wav.exists())
    print("LAB exists:", lab.exists())

    if wav.exists():
        print("WAV size:", wav.stat().st_size, "bytes")

    if lab.exists():
        transcript = lab.read_text(encoding="utf-8").strip()
        print("Transcript:", repr(transcript))
        print("Transcript length:", len(transcript))

print("\n========== ALIGNMENT FILES ==========")

for path in output_dir.iterdir():
    if path.is_file():
        print(path.name)

Streaming output truncated to the last 5000 lines.

recording_4851
WAV exists: True
LAB exists: True
WAV size: 562270 bytes
Transcript: 'And those ancestors will have a separate temple and people will pray to them and separate kind of puja and all the things will be done for them'
Transcript length: 143

recording_4852
WAV exists: True
LAB exists: True
WAV size: 809288 bytes
Transcript: 'Goa has also emerged as a manufacturing base for several leading companies in areas such as fertilizers, tires and tubes, cement, electrical machinery, fish net making machines, automatic washing machines, printing circuit boards, pharmaceuticals and pharmaceutical machinery.'
Transcript length: 276

recording_4870
WAV exists: True
LAB exists: True
WAV size: 272136 bytes
Transcript: 'is the book titled Rich Dad Poor Dad'
Transcript length: 36

recording_4874
WAV exists: True
LAB exists: True
WAV size: 115562 bytes
Transcript: 'delivered after 5 hours of ordering them'
Transcript length: 40

recording_4

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

df = pd.read_csv(BASE / "metadata/english_train.csv")

expected_ids = set(df["recording_id"].astype(int))

output_dir = BASE / "English/alignment/full_alignment/train"

generated_ids = {
    int(p.stem.replace("recording_", ""))
    for p in output_dir.rglob("*.TextGrid")
    if p.stem.replace("recording_", "").isdigit()
}

missing_ids = sorted(expected_ids - generated_ids)

print("Expected:", len(expected_ids))
print("Generated:", len(generated_ids))
print("Missing:", len(missing_ids))
print("\nMissing recording IDs:")
print(missing_ids)

print("\nWAV files missing:",
      sum(not (BASE / f"processed_audio_22050/english_train/recording_{i}.wav").exists()
          for i in missing_ids))

print("LAB files missing:",
      sum(not (BASE / f"English/raw_audio/train/recording_{i}.lab").exists()
          for i in missing_ids))

Expected: 2664
Generated: 2647
Missing: 17

Missing recording IDs:
[282, 1558, 2157, 2282, 2362, 2371, 3253, 3274, 3397, 3515, 3983, 4688, 4707, 5283, 5595, 5712, 6486]

WAV files missing: 0
LAB files missing: 0


In [ ]:
import os
import shutil
import subprocess
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# 1. PATHS
# --------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
ENV_BIN = "/content/miniforge3/envs/mfa_env/bin"

os.environ["PATH"] = ENV_BIN + ":" + os.environ["PATH"]

metadata_path = BASE / "metadata/english_train.csv"
audio_dir = BASE / "processed_audio_22050/english_train"
transcript_dir = BASE / "English/raw_audio/train"

full_output = BASE / "English/alignment/full_alignment/train"

retry_corpus = Path("/content/english_mfa_retry_corpus")
retry_output = BASE / "English/alignment/retry_output"

retry_corpus.mkdir(parents=True, exist_ok=True)
retry_output.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. IDENTIFY THE 17 MISSING RECORDINGS
# --------------------------------------------------

df = pd.read_csv(metadata_path)

expected_ids = set(df["recording_id"].astype(int))

existing_ids = {
    int(p.stem.replace("recording_", ""))
    for p in full_output.rglob("*.TextGrid")
    if p.stem.startswith("recording_")
    and p.stem.replace("recording_", "").isdigit()
}

missing_ids = sorted(expected_ids - existing_ids)

print("Recordings to retry:", len(missing_ids))
print("Missing IDs:", missing_ids)

if not missing_ids:
    print("No missing recordings. Retry is unnecessary.")

else:

    # --------------------------------------------------
    # 3. PREPARE RETRY CORPUS
    # --------------------------------------------------

    speaker_dir = retry_corpus / "speaker_1"
    speaker_dir.mkdir(parents=True, exist_ok=True)

    for recording_id in missing_ids:

        filename = f"recording_{recording_id}"

        wav_source = audio_dir / f"{filename}.wav"
        lab_source = transcript_dir / f"{filename}.lab"

        wav_target = speaker_dir / f"{filename}.wav"
        lab_target = speaker_dir / f"{filename}.lab"

        if not wav_source.exists() or not lab_source.exists():
            raise FileNotFoundError(
                f"Missing source file for {filename}"
            )

        if wav_target.exists() or wav_target.is_symlink():
            wav_target.unlink()

        os.symlink(wav_source, wav_target)
        shutil.copy2(lab_source, lab_target)

    print("\nRetry corpus prepared:", len(missing_ids))

    # --------------------------------------------------
    # 4. RUN MFA ON ONLY THE MISSING RECORDINGS
    # --------------------------------------------------

    command = [
        MFA,
        "align_hf",
        str(retry_corpus),
        "MontrealCorpusTools/english_mfa",
        str(retry_output),
        "--dialect", "english_india",
        "--use_g2p",
        "--single_speaker",
        "-j", "2",
        "--overwrite",
        "--verbose"
    ]

    print("\nStarting alignment retry...")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print("\nReturn code:", result.returncode)

    if result.stdout:
        print("\nMFA output:")
        print(result.stdout[-4000:])

    if result.stderr:
        print("\nMFA messages:")
        print(result.stderr[-6000:])

    # --------------------------------------------------
    # 5. VERIFY RETRY RESULTS
    # --------------------------------------------------

    retry_files = list(retry_output.rglob("*.TextGrid"))

    retry_ids = {
        int(p.stem.replace("recording_", ""))
        for p in retry_files
        if p.stem.startswith("recording_")
        and p.stem.replace("recording_", "").isdigit()
    }

    print("\nRetry expected:", len(missing_ids))
    print("Retry generated:", len(retry_ids))
    print("Still missing:", sorted(set(missing_ids) - retry_ids))

    # --------------------------------------------------
    # 6. MERGE SUCCESSFUL ALIGNMENTS
    # --------------------------------------------------

    if result.returncode == 0:

        copied = 0

        for textgrid in retry_files:

            target = full_output / textgrid.name

            if not target.exists():
                shutil.copy2(textgrid, target)
                copied += 1

        print("\nNew TextGrids copied:", copied)

    else:
        print("\nMFA retry failed. Existing alignments were preserved.")

    # --------------------------------------------------
    # 7. FINAL TRAIN ALIGNMENT COUNT
    # --------------------------------------------------

    final_ids = {
        int(p.stem.replace("recording_", ""))
        for p in full_output.rglob("*.TextGrid")
        if p.stem.startswith("recording_")
        and p.stem.replace("recording_", "").isdigit()
    }

    print("\n========== FINAL TRAIN ALIGNMENT ==========")
    print("Expected:", len(expected_ids))
    print("Generated:", len(final_ids))
    print("Remaining:", len(expected_ids - final_ids))

Recordings to retry: 17
Missing IDs: [282, 1558, 2157, 2282, 2362, 2371, 3253, 3274, 3397, 3515, 3983, 4688, 4707, 5283, 5595, 5712, 6486]

Retry corpus prepared: 17

Starting alignment retry...

Return code: 0

MFA output:
  17% ━━━━━╸                              17/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12  [ 0:00:02 < 0:00:00 , 3 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17  [ 0:00:03 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17  [ 0:00:03 < 0:00:00 , ? it/s ]
  88% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸     15/17  [ 0:00:01 < -:--:-- , ? it/s ]
  88% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸     15/17  [ 0:00:01 < -:--:-- , ? it/s ]
  88% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸     15/17

In [ ]:
import pandas as pd
import soundfile as sf
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

df = pd.read_csv(BASE / "metadata/english_train.csv")

audio_dir = BASE / "processed_audio_22050/english_train"
lab_dir = BASE / "English/raw_audio/train"
alignment_dir = BASE / "English/alignment/full_alignment/train"

for recording_id in [3253, 3515]:

    filename = f"recording_{recording_id}"

    row = df[df["recording_id"] == recording_id]

    wav_path = audio_dir / f"{filename}.wav"
    lab_path = lab_dir / f"{filename}.lab"

    print("\n" + "=" * 55)
    print(filename)

    print("Transcript:")
    print(lab_path.read_text(encoding="utf-8"))

    info = sf.info(str(wav_path))
    print("Duration:", round(info.duration, 2), "seconds")
    print("Sample rate:", info.samplerate)
    print("Audio channels:", info.channels)

    print("WAV exists:", wav_path.exists())
    print("LAB exists:", lab_path.exists())

    matching = list(alignment_dir.rglob(f"{filename}.TextGrid"))
    print("TextGrid exists:", bool(matching))


recording_3253
Transcript:
You are 2 hours late than the delivery time for my products Dettol Shaving Cream and Good knight Activ+ Low Smoke Mosquito Coil
Duration: 10.16 seconds
Sample rate: 22050
Audio channels: 1
WAV exists: True
LAB exists: True
TextGrid exists: False

recording_3515
Transcript:
So that was one of the shows which I liked watching.
Duration: 2.0 seconds
Sample rate: 22050
Audio channels: 1
WAV exists: True
LAB exists: True
TextGrid exists: False


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import soundfile as sf

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
audio_dir = BASE / "processed_audio_22050/english_train"
lab_dir = BASE / "English/raw_audio/train"

for recording_id in [3253, 3515]:

    filename = f"recording_{recording_id}"

    wav_path = audio_dir / f"{filename}.wav"
    lab_path = lab_dir / f"{filename}.lab"

    audio, sr = sf.read(wav_path)

    print("\n" + "=" * 60)
    print(filename)
    print("Transcript:", lab_path.read_text(encoding="utf-8"))
    print("Duration:", round(len(audio) / sr, 2), "seconds")

    display(Audio(audio, rate=sr))


recording_3253
Transcript: You are 2 hours late than the delivery time for my products Dettol Shaving Cream and Good knight Activ+ Low Smoke Mosquito Coil
Duration: 10.16 seconds



recording_3515
Transcript: So that was one of the shows which I liked watching.
Duration: 2.0 seconds


In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

lab_dir = BASE / "English/raw_audio/train"
diagnostic_dir = BASE / "English/alignment/manual_review"

diagnostic_dir.mkdir(parents=True, exist_ok=True)

# Original transcript for recording 3253
original_3253 = (
    lab_dir / "recording_3253.lab"
).read_text(encoding="utf-8")

# Pronunciation-friendly diagnostic transcript
modified_3253 = (
    "You are two hours late than the delivery time for my products "
    "Dettol Shaving Cream and Good Knight Active Plus Low Smoke "
    "Mosquito Coil"
)

# Save both versions separately
(diagnostic_dir / "recording_3253_original.lab").write_text(
    original_3253, encoding="utf-8"
)

(diagnostic_dir / "recording_3253_retry.lab").write_text(
    modified_3253, encoding="utf-8"
)

# Preserve the incomplete recording's original transcript for review
original_3515 = (
    lab_dir / "recording_3515.lab"
).read_text(encoding="utf-8")

(diagnostic_dir / "recording_3515_review.lab").write_text(
    original_3515, encoding="utf-8"
)

print("Diagnostic files saved:", diagnostic_dir)
print("\n3253 original:", original_3253)
print("\n3253 retry:", modified_3253)
print("\n3515 original:", original_3515)

Diagnostic files saved: /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment/manual_review

3253 original: You are 2 hours late than the delivery time for my products Dettol Shaving Cream and Good knight Activ+ Low Smoke Mosquito Coil

3253 retry: You are two hours late than the delivery time for my products Dettol Shaving Cream and Good Knight Active Plus Low Smoke Mosquito Coil

3515 original: So that was one of the shows which I liked watching.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
ENV_BIN = "/content/miniforge3/envs/mfa_env/bin"
os.environ["PATH"] = ENV_BIN + ":" + os.environ["PATH"]

# Temporary corpus for recording 3253 only
corpus_dir = Path("/content/mfa_3253_retry/speaker_1")
corpus_dir.mkdir(parents=True, exist_ok=True)

audio_source = (
    BASE / "processed_audio_22050/english_train/recording_3253.wav"
)

transcript_source = (
    BASE / "English/alignment/manual_review/recording_3253_retry.lab"
)

# Prepare input files
shutil.copy2(audio_source, corpus_dir / "recording_3253.wav")
shutil.copy2(transcript_source, corpus_dir / "recording_3253.lab")

output_dir = BASE / "English/alignment/retry_3253"
output_dir.mkdir(parents=True, exist_ok=True)

command = [
    MFA,
    "align_hf",
    str(corpus_dir.parent),
    "MontrealCorpusTools/english_mfa",
    str(output_dir),
    "--dialect", "english_india",
    "--use_g2p",
    "--single_speaker",
    "-j", "2",
    "--overwrite",
    "--verbose"
]

print("Retrying recording 3253...")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

if result.stdout:
    print("\nMFA output:")
    print(result.stdout[-3000:])

if result.stderr:
    print("\nMFA messages:")
    print(result.stderr[-4000:])

generated = list(output_dir.rglob("recording_3253.TextGrid"))

print("\nTextGrid generated:", bool(generated))

# Copy the result without overwriting existing files
if result.returncode == 0 and generated:
    destination = BASE / "English/alignment/full_alignment/train/recording_3253.TextGrid"

    if not destination.exists():
        shutil.copy2(generated[0], destination)

    print("Recording 3253 alignment saved.")
else:
    print("Recording 3253 still needs manual review.")

Retrying recording 3253...
Return code: 1

MFA output:
   1%                                      1/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:00 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:03 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]


MFA messages:
ied as 2, but due to only having 1           
          utterances, MFA will only use 1 jobs.                                 
 INFO     Normalizing text...                                                   
 INFO     Generating pronunciations...                                          
 INFO     Generating MFCCs...                                            

In [ ]:
import pandas as pd
import tgt
from pathlib import Path

# Project paths
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

metadata_path = BASE / "metadata/english_train.csv"
alignment_dir = BASE / "English/alignment/full_alignment/train"

# Load expected recording IDs
df = pd.read_csv(metadata_path)
expected_ids = set(df["recording_id"].astype(int))

# Find generated TextGrids recursively
textgrid_files = list(alignment_dir.rglob("*.TextGrid"))

generated_ids = {
    int(p.stem.replace("recording_", ""))
    for p in textgrid_files
    if p.stem.startswith("recording_")
    and p.stem.replace("recording_", "").isdigit()
}

missing_ids = sorted(expected_ids - generated_ids)

# Verify TextGrid tiers
valid_files = 0
invalid_files = []

for path in textgrid_files:
    try:
        grid = tgt.io.read_textgrid(str(path))
        tier_names = [tier.name for tier in grid.tiers]

        if "phones" in tier_names and "words" in tier_names:
            valid_files += 1
        else:
            invalid_files.append(path.name)

    except Exception as error:
        invalid_files.append(path.name)
        print("Error reading:", path.name, error)

print("========== ENGLISH TRAIN VERIFICATION ==========")
print("Expected recordings:", len(expected_ids))
print("Generated TextGrids:", len(generated_ids))
print("Valid TextGrids:", valid_files)
print("Missing recordings:", missing_ids)
print("Invalid TextGrids:", invalid_files)

print("\nExcluded recordings:")
print("3253 - MFA alignment unsuccessful")
print("3515 - Incomplete audio")

if valid_files == 2662 and len(missing_ids) == 2:
    print("\nTRAINING ALIGNMENT VERIFICATION PASSED.")
else:
    print("\nPlease inspect the results before proceeding.")

========== ENGLISH TRAIN VERIFICATION ==========
Expected recordings: 2664
Generated TextGrids: 2662
Valid TextGrids: 2662
Missing recordings: [3253, 3515]
Invalid TextGrids: []

Excluded recordings:
3253 - MFA alignment unsuccessful
3515 - Incomplete audio

TRAINING ALIGNMENT VERIFICATION PASSED.


In [ ]:
import os
import subprocess
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
ENV_BIN = "/content/miniforge3/envs/mfa_env/bin"

os.environ["PATH"] = ENV_BIN + ":" + os.environ["PATH"]

corpus_dir = Path("/content/english_mfa_corpus/validation")

output_dir = BASE / "English/alignment/full_alignment/validation"
output_dir.mkdir(parents=True, exist_ok=True)

# Check validation corpus
wav_files = list(corpus_dir.rglob("*.wav"))
lab_files = list(corpus_dir.rglob("*.lab"))

print("Validation WAV files:", len(wav_files))
print("Validation LAB files:", len(lab_files))

if len(wav_files) != 505 or len(lab_files) != 505:
    raise RuntimeError("Validation corpus is incomplete.")

# Run MFA
command = [
    MFA,
    "align_hf",
    str(corpus_dir),
    "MontrealCorpusTools/english_mfa",
    str(output_dir),
    "--dialect", "english_india",
    "--use_g2p",
    "--single_speaker",
    "-j", "2",
    "--overwrite"
]

print("\nStarting English validation alignment...")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("\nReturn code:", result.returncode)

if result.stdout:
    print(result.stdout[-3000:])

if result.stderr:
    print(result.stderr[-4000:])

# Verify output
textgrid_files = list(output_dir.rglob("*.TextGrid"))

print("\nExpected TextGrids:", 505)
print("Generated TextGrids:", len(textgrid_files))

if result.returncode == 0 and len(textgrid_files) == 505:
    print("\nENGLISH VALIDATION ALIGNMENT SUCCESSFUL!")
else:
    print("\nValidation alignment needs further inspection.")

Validation WAV files: 505
Validation LAB files: 505

Starting English validation alignment...

Return code: 0
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/100  [ 0:00:00 < 0:00:00 , 216 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/505  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326/326  [ 0:01:52 < 0:00:00 , 3 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/505  [ 0:00:08 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/505  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/505  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/505  [ 0:00:31 < 0:00:00 , 15 it/s ]
  99% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 501/505  [ 0:00:03 < -:--:-- , ? it/s ]
  99% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 501/505  [ 0:00:18 < 0:00:01 , 29 it/s ]
  99% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 501/505  [ 0:00:02 < 0:00:01 , 496 it/s ]


Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 1092.94it/s]
 INFO    

In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

# Expected validation recording IDs
df = pd.read_csv(BASE / "metadata/english_validation.csv")
expected_ids = set(df["recording_id"].astype(int))

# Find all generated TextGrids, including nested folders
alignment_dir = BASE / "English/alignment/full_alignment/validation"
textgrid_files = list(alignment_dir.rglob("*.TextGrid"))

generated_ids = {
    int(p.stem.replace("recording_", ""))
    for p in textgrid_files
    if p.stem.startswith("recording_")
    and p.stem.replace("recording_", "").isdigit()
}

missing_ids = sorted(expected_ids - generated_ids)

print("Expected validation recordings:", len(expected_ids))
print("Generated TextGrids:", len(generated_ids))
print("Missing count:", len(missing_ids))
print("Missing recording IDs:", missing_ids)

# Check that the corresponding WAV and LAB files exist
audio_dir = BASE / "processed_audio_22050/english_validation"
transcript_dir = BASE / "English/raw_audio/validation"

print("\n========== MISSING RECORDING DETAILS ==========")

for recording_id in missing_ids:
    filename = f"recording_{recording_id}"
    wav_path = audio_dir / f"{filename}.wav"
    lab_path = transcript_dir / f"{filename}.lab"

    transcript = lab_path.read_text(encoding="utf-8").strip() if lab_path.exists() else "MISSING"

    print(f"\nRecording: {filename}")
    print("WAV exists:", wav_path.exists())
    print("LAB exists:", lab_path.exists())
    print("Transcript:", transcript)

Expected validation recordings: 505
Generated TextGrids: 501
Missing count: 4
Missing recording IDs: [3617, 3653, 4893, 6055]

========== MISSING RECORDING DETAILS ==========

Recording: recording_3617
WAV exists: True
LAB exists: True
Transcript: The different religions practiced in our area

Recording: recording_3653
WAV exists: True
LAB exists: True
Transcript: Show me the email of salary reciept of last 15 days

Recording: recording_4893
WAV exists: True
LAB exists: True
Transcript: leftovers, the leaves and the grass to dry and rot

Recording: recording_6055
WAV exists: True
LAB exists: True
Transcript: They are celebrated widely and there are some other local festivals


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
ENV_BIN = "/content/miniforge3/envs/mfa_env/bin"
os.environ["PATH"] = ENV_BIN + ":" + os.environ["PATH"]

missing_ids = [3617, 3653, 4893, 6055]

audio_dir = BASE / "processed_audio_22050/english_validation"
transcript_dir = BASE / "English/raw_audio/validation"

retry_corpus = Path("/content/english_validation_retry_corpus")
speaker_dir = retry_corpus / "speaker_1"

retry_output = BASE / "English/alignment/retry_validation"
final_output = BASE / "English/alignment/full_alignment/validation"

speaker_dir.mkdir(parents=True, exist_ok=True)
retry_output.mkdir(parents=True, exist_ok=True)

# Prepare only the four missing recordings
for recording_id in missing_ids:
    filename = f"recording_{recording_id}"

    wav_source = audio_dir / f"{filename}.wav"
    lab_source = transcript_dir / f"{filename}.lab"

    if not wav_source.exists() or not lab_source.exists():
        raise FileNotFoundError(f"Missing WAV or LAB for {filename}")

    shutil.copy2(wav_source, speaker_dir / wav_source.name)
    shutil.copy2(lab_source, speaker_dir / lab_source.name)

print("Retry corpus prepared.")
print("WAV files:", len(list(speaker_dir.glob("*.wav"))))
print("LAB files:", len(list(speaker_dir.glob("*.lab"))))

# Run MFA on only these four recordings
command = [
    MFA,
    "align_hf",
    str(retry_corpus),
    "MontrealCorpusTools/english_mfa",
    str(retry_output),
    "--dialect", "english_india",
    "--use_g2p",
    "--single_speaker",
    "-j", "2",
    "--overwrite"
]

print("\nStarting retry alignment...")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("\nReturn code:", result.returncode)

if result.stdout:
    print(result.stdout[-2500:])

if result.stderr:
    print(result.stderr[-3500:])

# Copy newly generated TextGrids into the final validation folder
generated = list(retry_output.rglob("*.TextGrid"))
copied = 0

for source in generated:
    destination = final_output / source.name

    if not destination.exists():
        shutil.copy2(source, destination)
        copied += 1

print("\nRetry TextGrids generated:", len(generated))
print("New TextGrids copied:", copied)

# Final validation count
final_files = list(final_output.rglob("*.TextGrid"))
print("Total validation TextGrids now:", len(final_files))

Retry corpus prepared.
WAV files: 4
LAB files: 4

Starting retry alignment...

Return code: 0
   4% ━                                    4/100  [ 0:00:01 < -:--:-- , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , 4 it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:03 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:01 < 0:00:00 , ? it/s ]
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4  [ 0:00:02 < 0:00:00 , 3 it/s ]


Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 892.36it/s]
 INFO     Setting up corpu

In [ ]:
import pandas as pd
import tgt
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
metadata_dir = BASE / "metadata"

splits = {
    "train": {
        "metadata": metadata_dir / "english_train.csv",
        "alignment": BASE / "English/alignment/full_alignment/train",
        "excluded": {3253, 3515}
    },
    "validation": {
        "metadata": metadata_dir / "english_validation.csv",
        "alignment": BASE / "English/alignment/full_alignment/validation",
        "excluded": set()
    }
}

for split_name, config in splits.items():

    df = pd.read_csv(config["metadata"])
    expected_ids = set(df["recording_id"].astype(int))

    textgrid_files = list(config["alignment"].rglob("*.TextGrid"))

    valid_ids = set()
    invalid_ids = []

    for path in textgrid_files:
        try:
            grid = tgt.io.read_textgrid(str(path))
            tiers = [tier.name for tier in grid.tiers]

            recording_id = int(
                path.stem.replace("recording_", "")
            )

            if "phones" in tiers and "words" in tiers:
                valid_ids.add(recording_id)
            else:
                invalid_ids.append(recording_id)

        except Exception as error:
            invalid_ids.append(path.stem)
            print("Could not read:", path, error)

    missing_ids = sorted(expected_ids - valid_ids)

    print(f"\n========== {split_name.upper()} ==========")
    print("Expected:", len(expected_ids))
    print("Valid alignments:", len(valid_ids))
    print("Missing or invalid:", missing_ids)
    print("Invalid tier files:", invalid_ids)

    # Keep only recordings with valid alignments
    aligned_df = df[
        df["recording_id"].astype(int).isin(valid_ids)
    ].copy()

    output_path = metadata_dir / f"english_{split_name}_aligned.csv"
    aligned_df.to_csv(output_path, index=False)

    print("Saved:", output_path)
    print("Usable recordings:", len(aligned_df))
    print("Usable audio hours:", round(aligned_df["duration"].sum() / 3600, 3))

    if valid_ids != expected_ids - config["excluded"]:
        raise RuntimeError(
            f"{split_name} alignment audit did not match the expected IDs."
        )

print("\nENGLISH ALIGNMENT AUDIT COMPLETED.")


========== TRAIN ==========
Expected: 2664
Valid alignments: 2662
Missing or invalid: [3253, 3515]
Invalid tier files: []
Saved: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/english_train_aligned.csv
Usable recordings: 2662
Usable audio hours: 4.998

========== VALIDATION ==========
Expected: 505
Valid alignments: 505
Missing or invalid: []
Invalid tier files: []
Saved: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/english_validation_aligned.csv
Usable recordings: 505
Usable audio hours: 1.002

ENGLISH ALIGNMENT AUDIT COMPLETED.


In [ ]:
import os
import subprocess

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
os.environ["PATH"] = (
    "/content/miniforge3/envs/mfa_env/bin:"
    + os.environ["PATH"]
)

for model_type in ["acoustic", "g2p", "dictionary"]:

    print(f"\n========== {model_type.upper()} MODELS ==========")

    result = subprocess.run(
        [MFA, "model", "list", model_type],
        capture_output=True,
        text=True
    )

    print(result.stdout if result.stdout else result.stderr)


========== ACOUSTIC MODELS ==========
No models found


========== G2P MODELS ==========
No models found


========== DICTIONARY MODELS ==========
No models found



In [ ]:
import os
import subprocess

MFA = "/content/miniforge3/envs/mfa_env/bin/mfa"
os.environ["PATH"] = (
    "/content/miniforge3/envs/mfa_env/bin:"
    + os.environ["PATH"]
)

for model_type in ["acoustic", "g2p", "dictionary"]:

    print(f"\n========== DOWNLOADABLE {model_type.upper()} MODELS ==========")

    result = subprocess.run(
        [MFA, "model", "download", model_type],
        capture_output=True,
        text=True,
        timeout=180
    )

    print("Return code:", result.returncode)
    print(result.stdout or "(No standard output)")
    if result.stderr:
        print("Details:", result.stderr[-2000:])


========== DOWNLOADABLE ACOUSTIC MODELS ==========
Return code: 0
{
│   'abkhaz_cv': ['v2.0.0'],
│   'armenian_cv': ['v2.0.0'],
│   'bashkir_cv': ['v2.0.0'],
│   'basque_cv': ['v2.0.0'],
│   'belarusian_cv': ['v2.0.0'],
│   'bulgarian_cv': ['v2.0.0'],
│   'bulgarian_mfa': ['v2.0.0', 'v2.0.0a', 'v3.0.0'],
│   'chuvash_cv': ['v2.0.0'],
│   'croatian_mfa': ['v2.0.0', 'v2.0.0a'],
│   'czech_cv': ['v2.0.0'],
│   'czech_mfa': ['v2.0.0', 'v2.0.0a', 'v3.3.0'],
│   'dutch_cv': ['v2.0.0'],
│   'english_mfa': ['v2.0.0', 'v2.0.0a', 'v2.2.1', 'v3.0.0', 'v3.1.0'],
│   'english_us_arpa': ['v2.0.0', 'v2.0.0a', 'v3.0.0'],
│   'french_mfa': ['v2.0.0', 'v2.0.0a', 'v3.0.0'],
│   'georgian_cv': ['v2.0.0'],
│   'german_mfa': ['v2.0.0', 'v2.0.0a', 'v3.0.0'],
│   'greek_cv': ['v2.0.0'],
│   'guarani_cv': ['v2.0.0'],
│   'hausa_cv': ['v2.0.0'],
│   'hausa_mfa': ['v2.0.0', 'v2.0.0a', 'v3.0.0'],
│   'hungarian_cv': ['v2.0.0'],
│   'italian_cv': ['v2.0.0'],
│   'japanese_mfa': ['v2.0.1a', 'v3.0.0'],
│   'kazakh_

In [ ]:
import requests

url = "https://storage.googleapis.com/test_public_bucket/aligner_models.zip"

response = requests.head(url, allow_redirects=True, timeout=30)

print("HTTP status:", response.status_code)
print("Archive available:", response.ok)

size = response.headers.get("Content-Length")

if size:
    print("Archive size (GB):", round(int(size) / (1024**3), 2))
else:
    print("Archive size: Not provided by server")

print("Content type:", response.headers.get("Content-Type"))

HTTP status: 403
Archive available: False
Archive size (GB): 0.0
Content type: application/xml; charset=UTF-8


In [ ]:
import sys
import torch
import importlib.util

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

for package in ["transformers", "torchaudio", "soundfile", "librosa"]:
    print(
        package,
        "installed" if importlib.util.find_spec(package) else "not installed"
    )

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
CUDA available: False
transformers installed
torchaudio installed
soundfile installed
librosa installed


In [ ]:
import torch
import torchaudio

print("PyTorch version:", torch.__version__)
print("TorchAudio version:", torchaudio.__version__)

print("\nMMS_FA available:",
      hasattr(torchaudio.pipelines, "MMS_FA"))

print("forced_align available:",
      hasattr(torchaudio.functional, "forced_align"))

print("MMS_FA multilingual bundle check:")

try:
    bundle = torchaudio.pipelines.MMS_FA
    print("MMS_FA loaded successfully")
    print("Bundle:", bundle)
except Exception as e:
    print("MMS_FA unavailable:", repr(e))

PyTorch version: 2.11.0+cpu
TorchAudio version: 2.11.0+cpu

MMS_FA available: True
forced_align available: True
MMS_FA multilingual bundle check:
MMS_FA loaded successfully
Bundle: Wav2Vec2FABundle(_path='https://dl.fbaipublicfiles.com/mms/torchaudio/ctc_alignment_mling_uroman/model.pt', _params={'extractor_mode': 'layer_norm', 'extractor_conv_layer_config': [(512, 10, 5), (512, 3, 2), (512, 3, 2), (512, 3, 2), (512, 3, 2), (512, 2, 2), (512, 2, 2)], 'extractor_conv_bias': True, 'encoder_embed_dim': 1024, 'encoder_projection_dropout': 0.0, 'encoder_pos_conv_kernel': 128, 'encoder_pos_conv_groups': 16, 'encoder_num_layers': 24, 'encoder_num_heads': 16, 'encoder_attention_dropout': 0.0, 'encoder_ff_interm_features': 4096, 'encoder_ff_interm_dropout': 0.1, 'encoder_dropout': 0.0, 'encoder_layer_norm_first': True, 'encoder_layer_drop': 0.1, 'aux_num_out': 28}, _sample_rate=16000, _normalize_waveform=True, _model_type='Wav2Vec2', _labels=('a', 'i', 'e', 'n', 'o', 'u', 't', 's', 'r', 'm', 'k

In [ ]:
import os
import shutil
import importlib.util
import torch
import torchaudio

# Available system memory
try:
    import psutil
    memory = psutil.virtual_memory()

    print("Total RAM (GB):", round(memory.total / (1024**3), 2))
    print("Available RAM (GB):", round(memory.available / (1024**3), 2))
except ImportError:
    print("psutil is not installed")

# Available disk space
disk = shutil.disk_usage("/content")

print("\nAvailable disk space (GB):", round(disk.free / (1024**3), 2))

# Check whether uroman is already available
print("\nuroman package installed:",
      importlib.util.find_spec("uroman") is not None)

print("TorchAudio:", torchaudio.__version__)
print("MMS_FA sample rate:", torchaudio.pipelines.MMS_FA.sample_rate)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Total RAM (GB): 12.67
Available RAM (GB): 9.2

Available disk space (GB): 99.03

uroman package installed: False
TorchAudio: 2.11.0+cpu
MMS_FA sample rate: 16000
Device: cpu


In [ ]:
import subprocess
from pathlib import Path
import pandas as pd
import shutil

# Clone the official Uroman repository
uroman_dir = Path("/content/uroman")

if not uroman_dir.exists():
    result = subprocess.run(
        [
            "git", "clone",
            "https://github.com/isi-nlp/uroman.git",
            str(uroman_dir)
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

print("Uroman repository ready.")

# Check Perl
perl_path = shutil.which("perl")
if perl_path is None:
    raise RuntimeError("Perl is not available in this runtime.")

script_path = uroman_dir / "bin/uroman.pl"

if not script_path.exists():
    raise FileNotFoundError("Uroman script not found.")

# Load three Telugu transcripts from the training metadata
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
df = pd.read_csv(BASE / "metadata/telugu_train.csv")

samples = df.sample(n=3, random_state=42)["text"].tolist()

print("\n========== TELUGU ROMANIZATION TEST ==========")

for i, text in enumerate(samples, 1):

    result = subprocess.run(
        ["perl", str(script_path)],
        input=str(text) + "\n",
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("Romanization failed:", result.stderr)
        continue

    print(f"\nSample {i}")
    print("Original Telugu:", text)
    print("Romanized:", result.stdout.strip())

print("\nUroman test completed.")

Uroman repository ready.


FileNotFoundError: Uroman script not found.

In [ ]:
!pip install -q uroman

import uroman as ur

# Load the romanizer once
romanizer = ur.Uroman()

# Read three Telugu training transcripts
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
df = pd.read_csv(BASE / "metadata/telugu_train.csv")

samples = df.sample(n=3, random_state=42)["text"].tolist()

print("========== TELUGU ROMANIZATION TEST ==========")

for i, text in enumerate(samples, 1):
    romanized = romanizer.romanize_string(str(text))

    print(f"\nSample {i}")
    print("Original Telugu:", text)
    print("Romanized:", romanized)

print("\nUroman test completed.")

AttributeError: module 'uroman' has no attribute 'Uroman'

In [ ]:
import uroman as ur
import inspect

print("Module location:", ur.__file__)
print("Available names:", [name for name in dir(ur) if not name.startswith("_")])

try:
    from uroman.uroman import Uroman

    print("\nUroman class imported successfully.")

    romanizer = Uroman()

    print("Test:", romanizer.romanize_string("నమస్కారం"))

except Exception as error:
    print("\nImport/test error:", repr(error))

Module location: None
Available names: []

Uroman class imported successfully.
Test: namaskaaram


In [ ]:
import pandas as pd
from pathlib import Path
from uroman.uroman import Uroman

# Initialize Uroman
romanizer = Uroman()

# Load Telugu training metadata
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
df = pd.read_csv(BASE / "metadata/telugu_train.csv")

# Select three reproducible samples
samples = df.sample(n=3, random_state=42)["text"].tolist()

print("========== TELUGU ROMANIZATION TEST ==========")

for i, text in enumerate(samples, 1):
    romanized = romanizer.romanize_string(str(text))

    print(f"\nSample {i}")
    print("Original Telugu:", text)
    print("Romanized:", romanized)

print("\nRomanization test completed.")

========== TELUGU ROMANIZATION TEST ==========

Sample 1
Original Telugu: దీని ముఖ్య విషయాలు పిల్లల మానసిక శాస్త్రము మరియు సామాజిక మానసిక శాస్త్రము.
Romanized: diini mukhya vissayaalu pillala maanasika shaastramu mariyu saamaajika maanasika shaastramu.

Sample 2
Original Telugu: గ్రామములో రెండు ప్రభుత్వ పాఠశాలలు కలవు.
Romanized: graamamuloo remddu prabhutva paatthashaalalu kalavu.

Sample 3
Original Telugu: వీరపల్లి పోలీసులు తెలిపిన వివరాల ప్రకారం, గన్నవరం డీపో ఆర్టీసీ బస్సు, ఆగిరిపల్లి నుంచి గన్నవరం వెళుతోంది.
Romanized: viirapalli pooliisulu telipina vivaraala prakaaram, gannavaram ddiipoo aarttiisii bassu, aagiripalli numci gannavaram vellutoomdi.

Romanization test completed.


In [ ]:
import re
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio

from pathlib import Path
from scipy.signal import resample_poly
from IPython.display import Audio, display
from uroman.uroman import Uroman

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
metadata_path = BASE / "metadata/telugu_train.csv"
audio_dir = BASE / "processed_audio_22050/telugu_train"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# 2. Load three reproducible Telugu samples
# --------------------------------------------------

df = pd.read_csv(metadata_path)
samples = df.sample(n=3, random_state=42).reset_index(drop=True)

romanizer = Uroman()

# --------------------------------------------------
# 3. Initialize MMS_FA model
# --------------------------------------------------

bundle = torchaudio.pipelines.MMS_FA

print("Loading MMS_FA model. This may take a few minutes...")

model = bundle.get_model(with_star=False).to(device)
model.eval()

tokenizer = bundle.get_tokenizer()
aligner = bundle.get_aligner()

print("MMS_FA loaded successfully.")
print("Device:", device)
print("Expected sample rate:", bundle.sample_rate)

# --------------------------------------------------
# 4. Normalize romanized text
# --------------------------------------------------

def normalize_uroman(text):
    text = text.lower()
    text = text.replace("’", "'")
    text = re.sub(r"([^a-z' ])", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# --------------------------------------------------
# 5. Align each sample
# --------------------------------------------------

for i, row in samples.iterrows():

    original_text = str(row["text"])
    audio_path = audio_dir / str(row["audio_path"])

    if not audio_path.exists():
        print("Audio not found:", audio_path)
        continue

    # Romanize Telugu transcript
    romanized = romanizer.romanize_string(original_text)
    normalized = normalize_uroman(romanized)

    words = normalized.split()

    if not words:
        print("Empty normalized transcript. Skipping.")
        continue

    # Load audio
    waveform, original_sr = sf.read(audio_path)

    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)

    # Resample to MMS_FA's 16 kHz
    if original_sr != bundle.sample_rate:
        waveform = resample_poly(
            waveform,
            bundle.sample_rate,
            original_sr
        )

    waveform = torch.tensor(
        waveform,
        dtype=torch.float32
    ).unsqueeze(0)

    # Generate model emissions
    with torch.inference_mode():
        emission, _ = model(waveform.to(device))

        # Tokenize and align
        tokens = tokenizer(words)
        token_spans = aligner(emission[0], tokens)

    # Convert model frames into seconds
    time_ratio = (
        waveform.shape[1]
        / emission.shape[1]
        / bundle.sample_rate
    )

    print("\n" + "=" * 55)
    print("TELUGU SAMPLE", i + 1)
    print("=" * 55)

    print("Original Telugu:", original_text)
    print("Romanized:", romanized)
    print("Normalized:", normalized)
    print("Audio duration:", round(waveform.shape[1] / bundle.sample_rate, 2), "sec")

    print("\nWord-level timing results:")

    for word, spans in zip(words, token_spans):

        if len(spans) == 0:
            print(word, "-> No alignment")
            continue

        start = spans[0].start * time_ratio
        end = spans[-1].end * time_ratio

        confidence = sum(
            span.score * len(span) for span in spans
        ) / sum(len(span) for span in spans)

        print(
            f"{word:20s} "
            f"{start:.3f}s - {end:.3f}s "
            f"| score: {confidence:.3f}"
        )

    print("\nAudio preview:")
    display(Audio(
        waveform.squeeze(0).numpy(),
        rate=bundle.sample_rate
    ))

print("\nThree-sample Telugu MMS_FA test completed.")

Loading MMS_FA model. This may take a few minutes...
Downloading: "https://dl.fbaipublicfiles.com/mms/torchaudio/ctc_alignment_mling_uroman/model.pt" to /root/.cache/torch/hub/checkpoints/model.pt


100%|██████████| 1.18G/1.18G [00:20<00:00, 62.2MB/s]


MMS_FA loaded successfully.
Device: cpu
Expected sample rate: 16000

TELUGU SAMPLE 1
Original Telugu: దీని ముఖ్య విషయాలు పిల్లల మానసిక శాస్త్రము మరియు సామాజిక మానసిక శాస్త్రము.
Romanized: diini mukhya vissayaalu pillala maanasika shaastramu mariyu saamaajika maanasika shaastramu.
Normalized: diini mukhya vissayaalu pillala maanasika shaastramu mariyu saamaajika maanasika shaastramu
Audio duration: 6.62 sec

Word-level timing results:
diini                0.762s - 1.003s | score: 0.995
mukhya               1.043s - 1.324s | score: 0.995
vissayaalu           1.364s - 1.825s | score: 0.968
pillala              2.266s - 2.587s | score: 0.951
maanasika            2.647s - 3.129s | score: 0.999
shaastramu           3.189s - 3.650s | score: 0.959
mariyu               4.011s - 4.252s | score: 0.999
saamaajika           4.352s - 4.914s | score: 0.998
maanasika            4.974s - 5.415s | score: 0.973
shaastramu           5.475s - 5.917s | score: 0.940

Audio preview:



TELUGU SAMPLE 2
Original Telugu: గ్రామములో రెండు ప్రభుత్వ పాఠశాలలు కలవు.
Romanized: graamamuloo remddu prabhutva paatthashaalalu kalavu.
Normalized: graamamuloo remddu prabhutva paatthashaalalu kalavu
Audio duration: 5.28 sec

Word-level timing results:
graamamuloo          0.743s - 2.590s | score: 0.874
remddu               2.931s - 3.172s | score: 0.986
prabhutva            3.232s - 3.614s | score: 0.994
paatthashaalalu      3.654s - 4.276s | score: 0.848
kalavu               4.336s - 4.597s | score: 0.999

Audio preview:



TELUGU SAMPLE 3
Original Telugu: వీరపల్లి పోలీసులు తెలిపిన వివరాల ప్రకారం, గన్నవరం డీపో ఆర్టీసీ బస్సు, ఆగిరిపల్లి నుంచి గన్నవరం వెళుతోంది.
Romanized: viirapalli pooliisulu telipina vivaraala prakaaram, gannavaram ddiipoo aarttiisii bassu, aagiripalli numci gannavaram vellutoomdi.
Normalized: viirapalli pooliisulu telipina vivaraala prakaaram gannavaram ddiipoo aarttiisii bassu aagiripalli numci gannavaram vellutoomdi
Audio duration: 8.2 sec

Word-level timing results:
viirapalli           0.842s - 1.363s | score: 0.992
pooliisulu           1.403s - 1.824s | score: 0.854
telipina             1.844s - 2.165s | score: 0.967
vivaraala            2.205s - 2.585s | score: 0.997
prakaaram            2.606s - 3.026s | score: 1.000
gannavaram           3.447s - 3.868s | score: 0.999
ddiipoo              3.908s - 4.189s | score: 0.830
aarttiisii           4.329s - 4.790s | score: 0.838
bassu                4.810s - 5.071s | score: 0.990
aagiripalli          5.572s - 6.133s | score: 0.956
numci 


Three-sample Telugu MMS_FA test completed.


In [ ]:
import sys
import subprocess

# Install Uroman in the current Colab Python environment
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "uroman"
])

# Verify the import
from uroman.uroman import Uroman

romanizer = Uroman()
print("Uroman installed successfully!")
print("Telugu test:", romanizer.romanize_string("నమస్కారం"))

Uroman installed successfully!
Telugu test: namaskaaram


In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

MYDRIVE = Path("/content/drive/MyDrive")

# Search for the Telugu training metadata file
matches = list(MYDRIVE.rglob("telugu_train.csv"))

print("\nSearching for Telugu training metadata...")

if matches:
    for path in matches:
        print("FOUND:", path)
        print("Parent folder:", path.parent)
else:
    print("telugu_train.csv was not found in Google Drive.")

    print("\nTop-level folders in MyDrive:")
    for item in MYDRIVE.iterdir():
        if item.is_dir():
            print(" -", item.name)

Mounted at /content/drive

Searching for Telugu training metadata...
FOUND: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train.csv
Parent folder: /content/drive/MyDrive/FastSpeech2_Assignment/metadata


In [ ]:

# STEP 53: FULL TELUGU MMS_FA ALIGNMENT — CPU + CHECKPOINTING
# Create a NEW cell. Do not rerun Step 52.

import re
import json
import gc
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio

from pathlib import Path
from scipy.signal import resample_poly
from uroman.uroman import Uroman

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

DATASETS = {
    "train": {
        "csv": BASE / "metadata/telugu_train.csv",
        "audio_dir": BASE / "processed_audio_22050/telugu_train",
        "output": BASE / "metadata/telugu_train_mmsfa.jsonl",
        "errors": BASE / "metadata/telugu_train_mmsfa_errors.jsonl",
    },
    "validation": {
        "csv": BASE / "metadata/telugu_validation.csv",
        "audio_dir": BASE / "processed_audio_22050/telugu_validation",
        "output": BASE / "metadata/telugu_validation_mmsfa.jsonl",
        "errors": BASE / "metadata/telugu_validation_mmsfa_errors.jsonl",
    },
}

DEVICE = torch.device("cpu")
SAVE_EVERY = 10

torch.set_num_threads(2)

print("Device:", DEVICE)
print("Loading Telugu metadata and MMS_FA...")

# ---------------------------------------------------------
# 2. LOAD MODEL ONCE
# ---------------------------------------------------------

bundle = torchaudio.pipelines.MMS_FA
model = bundle.get_model(with_star=False).to(DEVICE)
model.eval()

tokenizer = bundle.get_tokenizer()
aligner = bundle.get_aligner()
romanizer = Uroman()

SAMPLE_RATE = bundle.sample_rate

print("MMS_FA loaded.")
print("Required sample rate:", SAMPLE_RATE)

# ---------------------------------------------------------
# 3. TEXT NORMALIZATION
# ---------------------------------------------------------

def normalize_uroman(text):
    text = str(text).lower().replace("’", "'")
    text = re.sub(r"([^a-z' ])", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ---------------------------------------------------------
# 4. ALIGNMENT HELPERS
# ---------------------------------------------------------

def get_span_value(span, field):
    value = getattr(span, field, None)
    if value is None:
        return None
    return float(value) if field == "score" else int(value)


def align_one_record(row, audio_dir):
    original_text = str(row["text"])
    audio_name = str(row["audio_path"])
    audio_path = audio_dir / audio_name

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio not found: {audio_path}")

    # Telugu -> Uroman transliteration -> MMS_FA normalization
    romanized = romanizer.romanize_string(original_text)
    normalized = normalize_uroman(romanized)
    words = normalized.split()

    if not words:
        raise ValueError("Transcript became empty after normalization.")

    # Load and resample audio
    waveform, original_sr = sf.read(audio_path)

    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)

    if original_sr != SAMPLE_RATE:
        waveform = resample_poly(
            waveform,
            SAMPLE_RATE,
            original_sr
        )

    waveform = torch.tensor(
        waveform,
        dtype=torch.float32
    ).unsqueeze(0)

    # MMS_FA inference
    with torch.inference_mode():
        emission, _ = model(waveform.to(DEVICE))
        tokens = tokenizer(words)
        token_spans = aligner(emission[0], tokens)

    # Convert model frame indices to seconds
    seconds_per_frame = (
        waveform.shape[1] /
        emission.shape[1] /
        SAMPLE_RATE
    )

    word_results = []
    character_results = []
    alignment_warnings = []

    for word_index, (word, spans) in enumerate(zip(words, token_spans)):

        valid_spans = [
            span for span in spans
            if get_span_value(span, "start") is not None
            and get_span_value(span, "end") is not None
        ]

        if not valid_spans:
            alignment_warnings.append(
                f"No valid spans for word: {word}"
            )
            continue

        start_frame = min(
            get_span_value(span, "start")
            for span in valid_spans
        )
        end_frame = max(
            get_span_value(span, "end")
            for span in valid_spans
        )

        scores = [
            get_span_value(span, "score")
            for span in valid_spans
            if get_span_value(span, "score") is not None
        ]

        word_score = (
            float(np.mean(scores))
            if scores else None
        )

        word_results.append({
            "word_index": word_index,
            "word": word,
            "start": round(start_frame * seconds_per_frame, 4),
            "end": round(end_frame * seconds_per_frame, 4),
            "duration": round(
                (end_frame - start_frame) * seconds_per_frame, 4
            ),
            "score": word_score,
        })

        # The aligner returns token spans within each word.
        # Map them to individual romanized characters only
        # when the number of spans matches the word length.
        if len(valid_spans) == len(word):
            for char_index, (char, span) in enumerate(
                zip(word, valid_spans)
            ):
                char_start = get_span_value(span, "start")
                char_end = get_span_value(span, "end")
                char_score = get_span_value(span, "score")

                character_results.append({
                    "word_index": word_index,
                    "word": word,
                    "char_index": char_index,
                    "character": char,
                    "start": round(
                        char_start * seconds_per_frame, 4
                    ),
                    "end": round(
                        char_end * seconds_per_frame, 4
                    ),
                    "duration": round(
                        (char_end - char_start) * seconds_per_frame,
                        4
                    ),
                    "score": char_score,
                })
        else:
            alignment_warnings.append(
                f"Character-span count mismatch for '{word}': "
                f"{len(valid_spans)} spans vs {len(word)} characters"
            )

    if not word_results:
        raise ValueError("No valid word timings were produced.")

    return {
        "audio_path": audio_name,
        "original_text": original_text,
        "romanized_text": romanized,
        "normalized_text": normalized,
        "audio_duration": round(
            len(waveform[0]) / SAMPLE_RATE, 4
        ),
        "word_spans": word_results,
        "character_spans": character_results,
        "character_alignment_complete": (
            len(character_results) == sum(len(w) for w in words)
        ),
        "alignment_warnings": alignment_warnings,
    }


# ---------------------------------------------------------
# 5. RESUMABLE DATASET PROCESSING
# ---------------------------------------------------------

def process_dataset(split_name, config):
    print("\n" + "=" * 65)
    print(f"PROCESSING TELUGU {split_name.upper()}")
    print("=" * 65)

    df = pd.read_csv(config["csv"])
    output_path = config["output"]
    error_path = config["errors"]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Read previously completed records.
    # This allows the cell to resume after a runtime interruption.
    completed = set()

    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    completed.add(record["audio_path"])
                except Exception:
                    continue

    print("Total recordings:", len(df))
    print("Already completed:", len(completed))

    pending = df[
        ~df["audio_path"].astype(str).isin(completed)
    ].reset_index(drop=True)

    print("Remaining recordings:", len(pending))

    success_count = 0
    failure_count = 0

    with open(output_path, "a", encoding="utf-8") as out_file, \
         open(error_path, "a", encoding="utf-8") as error_file:

        for index, row in pending.iterrows():

            audio_name = str(row["audio_path"])

            try:
                result = align_one_record(
                    row,
                    config["audio_dir"]
                )

                out_file.write(
                    json.dumps(result, ensure_ascii=False) + "\n"
                )
                out_file.flush()

                success_count += 1

            except Exception as exc:
                error_record = {
                    "audio_path": audio_name,
                    "error": str(exc),
                }

                error_file.write(
                    json.dumps(error_record, ensure_ascii=False) + "\n"
                )
                error_file.flush()

                failure_count += 1

            processed = index + 1

            if processed % SAVE_EVERY == 0:
                print(
                    f"{split_name}: {processed}/{len(pending)} "
                    f"| Successful: {success_count} "
                    f"| Failed: {failure_count}"
                )

            # Release per-record inference memory
            if processed % 25 == 0:
                gc.collect()

    print(f"\n{split_name.upper()} COMPLETE")
    print("New successful alignments:", success_count)
    print("New failures:", failure_count)
    print("Alignment file:", output_path)
    print("Error file:", error_path)


# ---------------------------------------------------------
# 6. RUN TRAINING AND VALIDATION
# ---------------------------------------------------------

process_dataset("train", DATASETS["train"])
process_dataset("validation", DATASETS["validation"])

# ---------------------------------------------------------
# 7. FINAL SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 65)
print("TELUGU MMS_FA ALIGNMENT RUN FINISHED")
print("=" * 65)

for split_name, config in DATASETS.items():
    count = 0
    if config["output"].exists():
        with open(config["output"], "r", encoding="utf-8") as f:
            count = sum(1 for line in f if line.strip())

    print(f"{split_name.title()} aligned recordings: {count}")
    print(f"Saved at: {config['output']}")

print("\nProgress is saved. If Colab disconnects, rerun this same cell.")
print("Previously completed recordings will be skipped.")


Device: cpu
Loading Telugu metadata and MMS_FA...
MMS_FA loaded.
Required sample rate: 16000

PROCESSING TELUGU TRAIN
Total recordings: 1281
Already completed: 0
Remaining recordings: 1281
train: 10/1281 | Successful: 0 | Failed: 10
train: 20/1281 | Successful: 0 | Failed: 20
train: 30/1281 | Successful: 0 | Failed: 30
train: 40/1281 | Successful: 0 | Failed: 40
train: 50/1281 | Successful: 0 | Failed: 50
train: 60/1281 | Successful: 0 | Failed: 60
train: 70/1281 | Successful: 0 | Failed: 70
train: 80/1281 | Successful: 0 | Failed: 80
train: 90/1281 | Successful: 0 | Failed: 90
train: 100/1281 | Successful: 0 | Failed: 100
train: 110/1281 | Successful: 0 | Failed: 110
train: 120/1281 | Successful: 0 | Failed: 120
train: 130/1281 | Successful: 0 | Failed: 130
train: 140/1281 | Successful: 0 | Failed: 140
train: 150/1281 | Successful: 0 | Failed: 150
train: 160/1281 | Successful: 0 | Failed: 160
train: 170/1281 | Successful: 0 | Failed: 170
train: 180/1281 | Successful: 0 | Failed: 180
t

In [ ]:
import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/metadata")

for split in ["train", "validation"]:
    error_file = BASE / f"telugu_{split}_mmsfa_errors.jsonl"

    print("\n" + "=" * 60)
    print(f"{split.upper()} — FIRST 5 ERRORS")
    print("=" * 60)

    if not error_file.exists():
        print("Error file not found.")
        continue

    with open(error_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i == 5:
                break
            record = json.loads(line)
            print("\nAudio:", record.get("audio_path"))
            print("Error:", record.get("error"))


TRAIN — FIRST 5 ERRORS

Audio: train_telugufemale_02124.wav
Error: Audio not found: /content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050/telugu_train/train_telugufemale_02124.wav

Audio: train_telugufemale_02016.wav
Error: Audio not found: /content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050/telugu_train/train_telugufemale_02016.wav

Audio: train_telugufemale_02737.wav
Error: Audio not found: /content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050/telugu_train/train_telugufemale_02737.wav

Audio: train_telugufemale_00215.wav
Error: Audio not found: /content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050/telugu_train/train_telugufemale_00215.wav

Audio: train_telugufemale_03593.wav
Error: Audio not found: /content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050/telugu_train/train_telugufemale_03593.wav

VALIDATION — FIRST 5 ERRORS

Audio: train_telugufemale_03124.wav
Error: Audio not found: /content/drive/MyDrive/FastSpee

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

# Check the expected audio directories
folders = [
    BASE / "processed_audio_22050/telugu_train",
    BASE / "processed_audio_22050/telugu_validation",
    BASE / "Telugu/raw_audio/train",
    BASE / "Telugu/raw_audio/validation",
]

print("AUDIO DIRECTORY CHECK")
for folder in folders:
    print(f"\n{folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        files = list(folder.glob("*.wav"))
        print("WAV files directly inside:", len(files))
        print("Examples:", [f.name for f in files[:5]])

# Search for actual locations of sample files
sample_names = [
    "train_telugufemale_02124.wav",
    "train_telugufemale_02016.wav",
    "train_telugufemale_03124.wav",
]

print("\nSEARCHING FOR SAMPLE RECORDINGS")

for filename in sample_names:
    matches = list(BASE.rglob(filename))
    print(f"\n{filename}")
    if matches:
        for match in matches[:5]:
            print("FOUND:", match)
    else:
        print("NOT FOUND inside assignment folder.")

# Check metadata filename columns
for split in ["train", "validation"]:
    csv_path = BASE / f"metadata/telugu_{split}.csv"
    df = pd.read_csv(csv_path)

    print(f"\n{split.upper()} METADATA")
    print("Columns:", df.columns.tolist())
    print(df[["audio_path", "text"]].head(3).to_string(index=False))

AUDIO DIRECTORY CHECK

/content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050/telugu_train
Exists: False

/content/drive/MyDrive/FastSpeech2_Assignment/processed_audio_22050/telugu_validation
Exists: False

/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/train
Exists: True
WAV files directly inside: 1281
Examples: ['train_telugumale_01838.wav', 'train_telugumale_01842.wav', 'train_telugumale_01858.wav', 'train_telugumale_01860.wav', 'train_telugumale_01861.wav']

/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/validation
Exists: True
WAV files directly inside: 251
Examples: ['train_telugufemale_00060.wav', 'train_telugufemale_00065.wav', 'train_telugufemale_00078.wav', 'train_telugufemale_00146.wav', 'train_telugufemale_00256.wav']

SEARCHING FOR SAMPLE RECORDINGS

train_telugufemale_02124.wav
FOUND: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/train/train_telugufemale_02124.wav

train_telugufemale_02016.wav
FOUND: /content/d

In [8]:

# STEP 56: FULL TELUGU MMS_FA ALIGNMENT — CORRECTED AUDIO PATHS
# NEW replacement for Step 53.
# Do not rerun Step 52.

import re
import json
import gc
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio

from pathlib import Path
from scipy.signal import resample_poly
from uroman.uroman import Uroman

# ---------------------------------------------------------
# 1. PATHS AND SETTINGS
# ---------------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
META = BASE / "metadata"

DATASETS = {
    "train": {
        "csv": META / "telugu_train.csv",
        "audio_dir": BASE / "Telugu/raw_audio/train",
        "output": META / "telugu_train_mmsfa.jsonl",
        "errors": META / "telugu_train_mmsfa_retry_errors.jsonl",
    },
    "validation": {
        "csv": META / "telugu_validation.csv",
        "audio_dir": BASE / "Telugu/raw_audio/validation",
        "output": META / "telugu_validation_mmsfa.jsonl",
        "errors": META / "telugu_validation_mmsfa_retry_errors.jsonl",
    },
}

DEVICE = torch.device("cpu")
SAVE_EVERY = 10

torch.set_num_threads(2)

# ---------------------------------------------------------
# 2. LOAD MODEL
# ---------------------------------------------------------

print("Loading MMS_FA...")

bundle = torchaudio.pipelines.MMS_FA
model = bundle.get_model(with_star=False).to(DEVICE)
model.eval()

tokenizer = bundle.get_tokenizer()
aligner = bundle.get_aligner()
romanizer = Uroman()

TARGET_SR = bundle.sample_rate

print("MMS_FA loaded.")
print("Device:", DEVICE)
print("Target sample rate:", TARGET_SR)

# ---------------------------------------------------------
# 3. TEXT NORMALIZATION
# ---------------------------------------------------------

def normalize_uroman(text):
    text = str(text).lower().replace("’", "'")
    text = re.sub(r"([^a-z' ])", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ---------------------------------------------------------
# 4. ALIGN ONE RECORDING
# ---------------------------------------------------------

def align_one_record(row, audio_dir):

    audio_name = str(row["audio_path"])
    original_text = str(row["text"])

    audio_path = audio_dir / audio_name

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio not found: {audio_path}")

    # Telugu transcript to Uroman representation
    romanized = romanizer.romanize_string(original_text)
    normalized = normalize_uroman(romanized)
    words = normalized.split()

    if not words:
        raise ValueError("Empty transcript after normalization.")

    # Read original audio
    waveform, original_sr = sf.read(audio_path)

    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)

    # Resample to MMS_FA's required 16 kHz
    if original_sr != TARGET_SR:
        waveform = resample_poly(
            waveform,
            TARGET_SR,
            original_sr
        )

    waveform = torch.tensor(
        waveform,
        dtype=torch.float32
    ).unsqueeze(0)

    # Run forced alignment
    with torch.inference_mode():
        emission, _ = model(waveform)
        tokens = tokenizer(words)
        token_spans = aligner(emission[0], tokens)

    seconds_per_frame = (
        waveform.shape[1] /
        emission.shape[1] /
        TARGET_SR
    )

    word_results = []
    character_results = []
    warnings = []

    for word_index, (word, spans) in enumerate(
        zip(words, token_spans)
    ):

        if not spans:
            warnings.append(f"No alignment for word: {word}")
            continue

        start_frames = [int(span.start) for span in spans]
        end_frames = [int(span.end) for span in spans]

        start_frame = min(start_frames)
        end_frame = max(end_frames)

        scores = [
            float(span.score)
            for span in spans
            if hasattr(span, "score")
        ]

        word_score = float(np.mean(scores)) if scores else None

        word_results.append({
            "word_index": word_index,
            "word": word,
            "start": round(start_frame * seconds_per_frame, 4),
            "end": round(end_frame * seconds_per_frame, 4),
            "duration": round(
                (end_frame - start_frame) * seconds_per_frame, 4
            ),
            "score": word_score,
        })

        # Save character timing only if every character
        # has a corresponding alignment span.
        if len(spans) == len(word):

            for char_index, (char, span) in enumerate(
                zip(word, spans)
            ):
                character_results.append({
                    "word_index": word_index,
                    "word": word,
                    "char_index": char_index,
                    "character": char,
                    "start": round(
                        int(span.start) * seconds_per_frame, 4
                    ),
                    "end": round(
                        int(span.end) * seconds_per_frame, 4
                    ),
                    "duration": round(
                        (int(span.end) - int(span.start))
                        * seconds_per_frame, 4
                    ),
                    "score": float(span.score),
                })

        else:
            warnings.append(
                f"Character count mismatch for {word}: "
                f"{len(spans)} spans, {len(word)} characters"
            )

    if not word_results:
        raise ValueError("No valid word timings generated.")

    return {
        "audio_path": audio_name,
        "original_text": original_text,
        "romanized_text": romanized,
        "normalized_text": normalized,
        "original_sample_rate": int(original_sr),
        "alignment_sample_rate": TARGET_SR,
        "audio_duration": round(
            waveform.shape[1] / TARGET_SR, 4
        ),
        "word_spans": word_results,
        "character_spans": character_results,
        "character_alignment_complete": (
            len(character_results) == sum(len(w) for w in words)
        ),
        "alignment_warnings": warnings,
    }


# ---------------------------------------------------------
# 5. PROCESS DATASET WITH RESUME SUPPORT
# ---------------------------------------------------------

def process_dataset(split, config):

    print("\n" + "=" * 65)
    print("TELUGU", split.upper())
    print("=" * 65)

    df = pd.read_csv(config["csv"])
    audio_dir = config["audio_dir"]
    output_path = config["output"]
    error_path = config["errors"]

    if not audio_dir.exists():
        raise FileNotFoundError(
            f"Audio directory does not exist: {audio_dir}"
        )

    # Find previously completed recordings
    completed = set()

    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    completed.add(
                        json.loads(line)["audio_path"]
                    )
                except Exception:
                    continue

    pending = df[
        ~df["audio_path"].astype(str).isin(completed)
    ].reset_index(drop=True)

    print("Total recordings:", len(df))
    print("Already completed:", len(completed))
    print("Remaining:", len(pending))
    print("Audio folder:", audio_dir)

    # New error file so previous incorrect-path errors
    # are not mixed with this corrected run.
    success_count = 0
    failure_count = 0

    with open(output_path, "a", encoding="utf-8") as out_file, \
         open(error_path, "w", encoding="utf-8") as error_file:

        for index, row in pending.iterrows():

            try:
                result = align_one_record(row, audio_dir)

                out_file.write(
                    json.dumps(result, ensure_ascii=False) + "\n"
                )
                out_file.flush()

                success_count += 1

            except Exception as exc:

                error_file.write(
                    json.dumps({
                        "audio_path": str(row["audio_path"]),
                        "error": repr(exc),
                    }, ensure_ascii=False) + "\n"
                )
                error_file.flush()

                failure_count += 1

                # Fail fast on the first recording.
                # Avoid repeating hundreds of failed attempts.
                if success_count == 0 and failure_count == 1:
                    raise RuntimeError(
                        f"First {split} alignment failed. "
                        f"Actual error: {repr(exc)}"
                    ) from exc

            processed = index + 1

            if processed % SAVE_EVERY == 0:
                print(
                    f"{split}: {processed}/{len(pending)} "
                    f"| Success: {success_count} "
                    f"| Failed: {failure_count}"
                )

            if processed % 25 == 0:
                gc.collect()

    print(f"\n{split.upper()} FINISHED")
    print("New successes:", success_count)
    print("New failures:", failure_count)
    print("Output:", output_path)
    print("Errors:", error_path)


# ---------------------------------------------------------
# 6. RUN BOTH SPLITS
# ---------------------------------------------------------

process_dataset("train", DATASETS["train"])
process_dataset("validation", DATASETS["validation"])


# ---------------------------------------------------------
# 7. FINAL SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 65)
print("TELUGU ALIGNMENT SUMMARY")
print("=" * 65)

for split, config in DATASETS.items():

    count = 0

    if config["output"].exists():
        with open(config["output"], "r", encoding="utf-8") as f:
            count = sum(1 for line in f if line.strip())

    print(f"{split.title()}: {count} aligned recordings")

print("\nAlignment processing completed.")


Loading MMS_FA...
MMS_FA loaded.
Device: cpu
Target sample rate: 16000

TELUGU TRAIN
Total recordings: 1281
Already completed: 0
Remaining: 1281
Audio folder: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio/train
train: 10/1281 | Success: 10 | Failed: 0
train: 20/1281 | Success: 20 | Failed: 0
train: 30/1281 | Success: 30 | Failed: 0
train: 40/1281 | Success: 40 | Failed: 0
train: 50/1281 | Success: 50 | Failed: 0
train: 60/1281 | Success: 60 | Failed: 0
train: 70/1281 | Success: 70 | Failed: 0
train: 80/1281 | Success: 80 | Failed: 0
train: 90/1281 | Success: 90 | Failed: 0
train: 100/1281 | Success: 100 | Failed: 0
train: 110/1281 | Success: 110 | Failed: 0
train: 120/1281 | Success: 120 | Failed: 0
train: 130/1281 | Success: 130 | Failed: 0
train: 140/1281 | Success: 140 | Failed: 0
train: 150/1281 | Success: 150 | Failed: 0
train: 160/1281 | Success: 160 | Failed: 0
train: 170/1281 | Success: 170 | Failed: 0
train: 180/1281 | Success: 180 | Failed: 0
train: 190/1281 

In [2]:
# ============================================================
# STEP 57: VERIFY TELUGU ALIGNMENT OUTPUTS
# ============================================================

import os
import json
from collections import Counter

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"
META = os.path.join(BASE, "metadata")

FILES = {
    "train": os.path.join(META, "telugu_train_mmsfa.jsonl"),
    "validation": os.path.join(META, "telugu_validation_mmsfa.jsonl")
}

def inspect_jsonl(path, name):
    print("\n" + "=" * 65)
    print(name.upper(), "ALIGNMENT VERIFICATION")
    print("=" * 65)

    if not os.path.exists(path):
        print("ERROR: File not found:", path)
        return

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    print("WARNING: Invalid JSON line detected")

    print("Records:", len(records))

    if not records:
        print("No records found.")
        return

    print("\nAvailable fields:")
    print(list(records[0].keys()))

    print("\nFirst record (shortened):")
    print(json.dumps(records[0], ensure_ascii=False, indent=2)[:3500])

    print("\nField types:")
    for key, value in records[0].items():
        print(f"{key}: {type(value).__name__}")

    # Check duplicate record identifiers using common ID fields
    id_key = next(
        (k for k in ["id", "utt_id", "filename", "audio_path"]
         if k in records[0]),
        None
    )

    if id_key:
        ids = [r.get(id_key) for r in records]
        print("\nIdentifier field:", id_key)
        print("Unique identifiers:", len(set(ids)))
        print("Duplicate identifiers:", len(ids) - len(set(ids)))

    return records


train_records = inspect_jsonl(FILES["train"], "Telugu train")
val_records = inspect_jsonl(FILES["validation"], "Telugu validation")

print("\n" + "=" * 65)
print("VERIFICATION COMPLETE")
print("=" * 65)


TELUGU TRAIN ALIGNMENT VERIFICATION
ERROR: File not found: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train_mmsfa.jsonl

TELUGU VALIDATION ALIGNMENT VERIFICATION
ERROR: File not found: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_validation_mmsfa.jsonl

VERIFICATION COMPLETE


In [3]:
# ============================================================
# STEP 57A: LOCATE EXISTING TELUGU ALIGNMENT FILES
# ============================================================

import os

# Mount Google Drive if needed
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive"

print("\nChecking project folder...")

for folder in os.listdir(ROOT):
    if "FastSpeech2" in folder:
        print("Found folder:", os.path.join(ROOT, folder))

BASE = os.path.join(ROOT, "FastSpeech2_Assignment")

print("\nProject folder exists:", os.path.exists(BASE))

if os.path.exists(BASE):

    print("\nMetadata folder contents:")
    META = os.path.join(BASE, "metadata")

    if os.path.exists(META):
        for f in os.listdir(META):
            if "telugu" in f.lower() or "mmsfa" in f.lower():
                path = os.path.join(META, f)
                print(f, "|", os.path.getsize(path), "bytes")
    else:
        print("Metadata folder not found.")

    print("\nSearching for alignment JSONL files...")

    for root, dirs, files in os.walk(BASE):
        for file in files:
            if "telugu" in file.lower() and "mmsfa" in file.lower():
                print(os.path.join(root, file))

else:
    print("FastSpeech2_Assignment folder not found.")

Mounted at /content/drive

Checking project folder...
Found folder: /content/drive/MyDrive/FastSpeech2_Assignment

Project folder exists: True

Metadata folder contents:
telugu_metadata.csv | 4267148 bytes
telugu_train.csv | 613474 bytes
telugu_validation.csv | 122816 bytes
telugu_train_mmsfa_errors.jsonl | 234810 bytes
telugu_validation_mmsfa_errors.jsonl | 47257 bytes
telugu_validation_mmsfa_retry_errors.jsonl | 0 bytes
telugu_train_mmsfa_retry_errors.jsonl | 288 bytes
telugu_train_mmsfa.jsonl | 39454921 bytes
telugu_validation_mmsfa.jsonl | 7961106 bytes

Searching for alignment JSONL files...
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train_mmsfa_errors.jsonl
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_validation_mmsfa_errors.jsonl
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_validation_mmsfa_retry_errors.jsonl
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train_mmsfa_retry_errors.jsonl
/content/drive/MyDrive/Fas

In [4]:
BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"
META = os.path.join(BASE, "metadata")

In [5]:
# ============================================================
# STEP 58: TELUGU ALIGNMENT STRUCTURE CHECK
# ============================================================

import os
import json

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata"

FILES = {
    "Train": os.path.join(BASE, "telugu_train_mmsfa.jsonl"),
    "Validation": os.path.join(BASE, "telugu_validation_mmsfa.jsonl")
}

for split, path in FILES.items():

    print("\n" + "=" * 60)
    print(split.upper())
    print("=" * 60)

    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    print("Total records:", len(records))

    if records:
        sample = records[0]

        print("\nAvailable fields:")
        print(list(sample.keys()))

        print("\nFirst record:")
        print(json.dumps(sample, ensure_ascii=False, indent=2)[:4000])

        print("\nAlignment-related fields:")
        for key, value in sample.items():
            if any(x in key.lower() for x in
                   ["span", "word", "char", "align", "text"]):
                print(key, ":", str(value)[:1000])

print("\nStructure check complete.")


TRAIN
Total records: 1280

Available fields:
['audio_path', 'original_text', 'romanized_text', 'normalized_text', 'original_sample_rate', 'alignment_sample_rate', 'audio_duration', 'word_spans', 'character_spans', 'character_alignment_complete', 'alignment_warnings']

First record:
{
  "audio_path": "train_telugufemale_02124.wav",
  "original_text": "కొంతదూరం అడవిలోంచి నడిచి అందరు ఒక కొండప్రాంతం చేరుకున్నారు.",
  "romanized_text": "komtaduuram addaviloomci naddici amdaru oka komddapraamtam ceerukunnaaru.",
  "normalized_text": "komtaduuram addaviloomci naddici amdaru oka komddapraamtam ceerukunnaaru",
  "original_sample_rate": 16000,
  "alignment_sample_rate": 16000,
  "audio_duration": 6.8757,
  "word_spans": [
    {
      "word_index": 0,
      "word": "komtaduuram",
      "start": 0.4009,
      "end": 1.1226,
      "duration": 0.7216,
      "score": 0.9982179024002769
    },
    {
      "word_index": 1,
      "word": "addaviloomci",
      "start": 1.5636,
      "end": 2.2251,
     

In [6]:
# ============================================================
# STEP 59: COMPLETE TELUGU ALIGNMENT QUALITY AUDIT
# ============================================================

import os
import json
import numpy as np

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment/metadata"

FILES = {
    "Train": os.path.join(BASE, "telugu_train_mmsfa.jsonl"),
    "Validation": os.path.join(BASE, "telugu_validation_mmsfa.jsonl")
}

for split, path in FILES.items():

    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    complete = 0
    incomplete = 0
    missing_words = 0
    missing_chars = 0
    invalid_times = 0
    low_confidence = 0
    scores = []

    for r in records:

        if r.get("character_alignment_complete", False):
            complete += 1
        else:
            incomplete += 1

        words = r.get("word_spans", [])
        chars = r.get("character_spans", [])

        if not words:
            missing_words += 1

        if not chars:
            missing_chars += 1

        for span in words + chars:
            start = span.get("start", -1)
            end = span.get("end", -1)
            score = span.get("score", None)

            if start < 0 or end <= start:
                invalid_times += 1

            if score is not None:
                scores.append(score)
                if score < 0.70:
                    low_confidence += 1

    print("\n" + "=" * 60)
    print(split.upper(), "AUDIT")
    print("=" * 60)

    print("Total records:", len(records))
    print("Complete character alignments:", complete)
    print("Incomplete character alignments:", incomplete)
    print("Missing word spans:", missing_words)
    print("Missing character spans:", missing_chars)
    print("Invalid time spans:", invalid_times)
    print("Low-confidence spans (<0.70):", low_confidence)

    if scores:
        print("Mean alignment score:", round(float(np.mean(scores)), 4))

print("\nTELUGU ALIGNMENT AUDIT COMPLETE")


TRAIN AUDIT
Total records: 1280
Complete character alignments: 1280
Incomplete character alignments: 0
Missing word spans: 0
Missing character spans: 0
Invalid time spans: 0
Low-confidence spans (<0.70): 10251
Mean alignment score: 0.9515

VALIDATION AUDIT
Total records: 251
Complete character alignments: 251
Incomplete character alignments: 0
Missing word spans: 0
Missing character spans: 0
Invalid time spans: 0
Low-confidence spans (<0.70): 2095
Mean alignment score: 0.9508

TELUGU ALIGNMENT AUDIT COMPLETE


In [8]:
!pip install pyworld

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyworld: filename=pyworld-0.3.5-cp313-cp313-linux_x86_64.whl size=1020155 sha256=da8a7692f4596f466ad065821d855b42e80fe53d762bb12460fd323bd6026cc8
  Stored in directory: /root/.cache/pip/wheels/8b/c7/b2/0fb039381aa5f5b31c58216a65288315fda280becb7ae8f991
Successfully built pyworld


In [9]:
import pyworld
print("PyWorld installed successfully!")
print("Version:", pyworld.__version__)

PyWorld installed successfully!
Version: 0.3.5


/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [10]:
# ============================================================
# STEP 60: TELUGU ACOUSTIC FEATURE EXTRACTION
# Features: Mel Spectrogram, Pitch (F0), Energy, Duration
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import pyworld

from scipy.signal import resample_poly
from math import gcd
from collections import defaultdict

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"

SAMPLE_RATE = 22050
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
N_MELS = 80
FMIN = 0
FMAX = 8000

SPLITS = {
    "train": {
        "alignment": os.path.join(
            BASE, "metadata", "telugu_train_mmsfa.jsonl"
        ),
        "audio": os.path.join(
            BASE, "Telugu", "raw_audio", "train"
        ),
        "output": os.path.join(
            BASE, "Telugu", "features", "train"
        )
    },
    "validation": {
        "alignment": os.path.join(
            BASE, "metadata", "telugu_validation_mmsfa.jsonl"
        ),
        "audio": os.path.join(
            BASE, "Telugu", "raw_audio", "validation"
        ),
        "output": os.path.join(
            BASE, "Telugu", "features", "validation"
        )
    }
}

for config in SPLITS.values():
    os.makedirs(config["output"], exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD ALIGNMENT RECORDS
# ------------------------------------------------------------

def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


# ------------------------------------------------------------
# 3. CONVERT CHARACTER ALIGNMENTS TO DURATION TARGETS
# ------------------------------------------------------------

def create_duration_targets(record):

    text = record["normalized_text"].strip()
    words = text.split()

    character_spans = record["character_spans"]

    grouped = defaultdict(list)

    for span in character_spans:
        grouped[span["word_index"]].append(span)

    tokens = []
    durations = []
    alignment_scores = []

    word_spans = sorted(
        record["word_spans"],
        key=lambda x: x["word_index"]
    )

    if len(words) != len(word_spans):
        raise ValueError("Normalized text and aligned word counts differ")

    for i, (word, word_span) in enumerate(zip(words, word_spans)):

        word_index = word_span["word_index"]

        spans = sorted(
            grouped[word_index],
            key=lambda x: x["char_index"]
        )

        aligned_word = "".join(
            span["character"] for span in spans
        )

        if aligned_word != word:
            raise ValueError(
                f"Character mismatch: {word} != {aligned_word}"
            )

        for span in spans:

            duration_seconds = span["end"] - span["start"]

            duration_frames = max(
                1,
                int(round(
                    duration_seconds * SAMPLE_RATE / HOP_LENGTH
                ))
            )

            tokens.append(span["character"])
            durations.append(duration_frames)

            alignment_scores.append(
                float(span.get("score", np.nan))
            )

        # Preserve spaces in the text representation.
        # Space tokens receive zero duration.
        if i < len(words) - 1:
            tokens.append(" ")
            durations.append(0)

    return (
        np.array(tokens),
        np.array(durations, dtype=np.int32),
        alignment_scores
    )


# ------------------------------------------------------------
# 4. EXTRACT MEL, PITCH AND ENERGY
# ------------------------------------------------------------

def extract_acoustic_features(audio_path):

    audio, original_sr = sf.read(
        audio_path,
        dtype="float32"
    )

    # Convert stereo to mono
    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)

    # Resample to 22050 Hz
    if original_sr != SAMPLE_RATE:

        factor = gcd(int(original_sr), SAMPLE_RATE)

        audio = resample_poly(
            audio,
            SAMPLE_RATE // factor,
            original_sr // factor
        ).astype(np.float32)

    audio = np.asarray(audio, dtype=np.float32)

    # Mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=1.0,
        center=True
    )

    mel = np.log(np.maximum(mel, 1e-5)).astype(np.float32)

    # Frame-level RMS energy
    energy = librosa.feature.rms(
        y=audio,
        frame_length=WIN_LENGTH,
        hop_length=HOP_LENGTH,
        center=True
    )[0].astype(np.float32)

    # Pitch extraction using WORLD
    frame_period = HOP_LENGTH / SAMPLE_RATE * 1000

    audio_double = audio.astype(np.float64)

    f0, time_axis = pyworld.dio(
        audio_double,
        SAMPLE_RATE,
        frame_period=frame_period,
        f0_floor=65.0,
        f0_ceil=500.0
    )

    f0 = pyworld.stonemask(
        audio_double,
        f0,
        time_axis,
        SAMPLE_RATE
    )

    # Align F0 frames to mel frames
    mel_times = (
        np.arange(mel.shape[1]) * HOP_LENGTH / SAMPLE_RATE
    )

    pitch = np.interp(
        mel_times,
        time_axis,
        f0
    ).astype(np.float32)

    # Ensure matching acoustic feature lengths
    target_frames = mel.shape[1]

    energy = librosa.util.fix_length(
        energy,
        size=target_frames
    )

    pitch = librosa.util.fix_length(
        pitch,
        size=target_frames
    )

    return mel, pitch, energy


# ------------------------------------------------------------
# 5. PROCESS BOTH DATA SPLITS
# ------------------------------------------------------------

for split, config in SPLITS.items():

    print("\n" + "=" * 65)
    print("PROCESSING TELUGU:", split.upper())
    print("=" * 65)

    records = load_jsonl(config["alignment"])

    output_dir = config["output"]
    audio_dir = config["audio"]

    success = 0
    skipped = 0
    failed = 0

    manifest = []
    error_path = os.path.join(
        BASE, "metadata",
        f"telugu_{split}_feature_errors.jsonl"
    )

    for index, record in enumerate(records):

        filename = record["audio_path"]
        output_path = os.path.join(
            output_dir,
            os.path.splitext(filename)[0] + ".npz"
        )

        # Resume support: do not repeat completed work
        if os.path.exists(output_path):

            try:
                with np.load(output_path) as data:
                    manifest.append({
                        "audio_path": filename,
                        "feature_file": output_path,
                        "mel_frames": data["mel"].shape[1],
                        "token_count": len(data["tokens"]),
                        "duration_frames": int(
                            data["durations"].sum()
                        )
                    })

                skipped += 1
                continue

            except Exception:
                os.remove(output_path)

        audio_path = os.path.join(audio_dir, filename)

        try:

            if not os.path.exists(audio_path):
                raise FileNotFoundError(audio_path)

            tokens, durations, scores = create_duration_targets(
                record
            )

            mel, pitch, energy = extract_acoustic_features(
                audio_path
            )

            # Save one recording at a time
            np.savez_compressed(
                output_path,
                mel=mel,
                pitch=pitch,
                energy=energy,
                tokens=tokens,
                durations=durations,
                mean_alignment_score=np.float32(
                    np.nanmean(scores)
                ),
                low_confidence_count=np.int32(
                    sum(s < 0.70 for s in scores)
                ),
                sample_rate=np.int32(SAMPLE_RATE),
                hop_length=np.int32(HOP_LENGTH)
            )

            manifest.append({
                "audio_path": filename,
                "feature_file": output_path,
                "mel_frames": mel.shape[1],
                "token_count": len(tokens),
                "duration_frames": int(durations.sum())
            })

            success += 1

        except Exception as e:

            failed += 1

            with open(error_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "audio_path": filename,
                    "error": str(e)
                }, ensure_ascii=False) + "\n")

        processed = index + 1

        if processed % 25 == 0:
            print(
                f"{split}: {processed}/{len(records)} | "
                f"New: {success} | Skipped: {skipped} | "
                f"Failed: {failed}"
            )

    # Save manifest
    manifest_path = os.path.join(
        BASE, "metadata",
        f"telugu_{split}_features_manifest.csv"
    )

    pd.DataFrame(manifest).to_csv(
        manifest_path,
        index=False
    )

    print("\n", split.upper(), "COMPLETE")
    print("Newly extracted:", success)
    print("Already available:", skipped)
    print("Failed:", failed)
    print("Manifest:", manifest_path)

print("\n" + "=" * 65)
print("TELUGU FEATURE EXTRACTION FINISHED")
print("=" * 65)


PROCESSING TELUGU: TRAIN
train: 25/1280 | New: 25 | Skipped: 0 | Failed: 0
train: 50/1280 | New: 50 | Skipped: 0 | Failed: 0
train: 75/1280 | New: 75 | Skipped: 0 | Failed: 0
train: 100/1280 | New: 100 | Skipped: 0 | Failed: 0
train: 125/1280 | New: 125 | Skipped: 0 | Failed: 0
train: 150/1280 | New: 150 | Skipped: 0 | Failed: 0
train: 175/1280 | New: 175 | Skipped: 0 | Failed: 0
train: 200/1280 | New: 200 | Skipped: 0 | Failed: 0
train: 225/1280 | New: 225 | Skipped: 0 | Failed: 0
train: 250/1280 | New: 250 | Skipped: 0 | Failed: 0
train: 275/1280 | New: 275 | Skipped: 0 | Failed: 0
train: 300/1280 | New: 300 | Skipped: 0 | Failed: 0
train: 325/1280 | New: 325 | Skipped: 0 | Failed: 0
train: 350/1280 | New: 350 | Skipped: 0 | Failed: 0
train: 375/1280 | New: 375 | Skipped: 0 | Failed: 0
train: 400/1280 | New: 400 | Skipped: 0 | Failed: 0
train: 425/1280 | New: 425 | Skipped: 0 | Failed: 0
train: 450/1280 | New: 450 | Skipped: 0 | Failed: 0
train: 475/1280 | New: 475 | Skipped: 0 | Fa

In [11]:
# ============================================================
# STEP 61: VERIFY TELUGU ACOUSTIC FEATURES
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"

for split in ["train", "validation"]:

    folder = os.path.join(
        BASE, "Telugu", "features", split
    )

    files = glob.glob(os.path.join(folder, "*.npz"))

    print("\n" + "=" * 60)
    print(split.upper(), "FEATURE VERIFICATION")
    print("=" * 60)

    print("Total feature files:", len(files))

    if not files:
        print("ERROR: No feature files found")
        continue

    valid = 0
    invalid = 0
    shapes = []

    for path in files:

        try:
            with np.load(path) as data:

                mel = data["mel"]
                pitch = data["pitch"]
                energy = data["energy"]
                tokens = data["tokens"]
                durations = data["durations"]

                assert mel.shape[0] == 80
                assert mel.shape[1] == len(pitch)
                assert mel.shape[1] == len(energy)
                assert len(tokens) == len(durations)

                assert np.isfinite(mel).all()
                assert np.isfinite(pitch).all()
                assert np.isfinite(energy).all()

                assert np.all(durations >= 0)

                valid += 1
                shapes.append(mel.shape[1])

        except Exception as e:
            invalid += 1
            print("Invalid:", os.path.basename(path), str(e))

    print("Valid files:", valid)
    print("Invalid files:", invalid)

    if shapes:
        print("Average mel frames:", round(np.mean(shapes), 2))
        print("Minimum mel frames:", min(shapes))
        print("Maximum mel frames:", max(shapes))

print("\nFEATURE VERIFICATION FINISHED")


TRAIN FEATURE VERIFICATION
Total feature files: 1280
Valid files: 1280
Invalid files: 0
Average mel frames: 1214.27
Minimum mel frames: 177
Maximum mel frames: 11429

VALIDATION FEATURE VERIFICATION
Total feature files: 251
Valid files: 251
Invalid files: 0
Average mel frames: 1239.41
Minimum mel frames: 185
Maximum mel frames: 5905

FEATURE VERIFICATION FINISHED


In [12]:
# ============================================================
# STEP 62: INSPECT FASTSPEECH2 CONFIGURATION
# ============================================================

import os

REPO = "/content/FastSpeech2"

files_to_check = [
    "config/LJSpeech/preprocess.yaml",
    "config/LJSpeech/model.yaml",
    "config/LJSpeech/train.yaml",
    "preprocessor/preprocessor.py",
    "model/fastspeech2.py",
    "train.py",
    "text/__init__.py"
]

for relative_path in files_to_check:

    path = os.path.join(REPO, relative_path)

    print("\n" + "=" * 65)
    print(relative_path)
    print("=" * 65)

    if os.path.exists(path):

        try:
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()

            print(content[:10000])

        except Exception as e:
            print("Could not read:", e)

    else:
        print("File not found")


config/LJSpeech/preprocess.yaml
File not found

config/LJSpeech/model.yaml
File not found

config/LJSpeech/train.yaml
File not found

preprocessor/preprocessor.py
File not found

model/fastspeech2.py
File not found

train.py
File not found

text/__init__.py
File not found


In [13]:
import os

print("Checking /content:\n")

for item in os.listdir("/content"):
    path = os.path.join("/content", item)
    print(item, "|", "Directory" if os.path.isdir(path) else "File")

print("\nChecking expected repository:")
print(os.path.exists("/content/FastSpeech2"))

print("\nChecking Google Drive project:")
print(os.path.exists(
    "/content/drive/MyDrive/FastSpeech2_Assignment"
))

Checking /content:

.config | Directory
drive | Directory
sample_data | Directory

Checking expected repository:
False

Checking Google Drive project:
True


In [14]:
# ============================================================
# STEP 62B: RESTORE FASTSPEECH2 REPOSITORY
# ============================================================

%cd /content

!git clone https://github.com/ming024/FastSpeech2.git

%cd /content/FastSpeech2

print("\nRepository restored!")

print("\nTop-level files and folders:")
!ls -la

print("\nConfiguration files:")
!find . -maxdepth 3 -type f \( -name "*.yaml" -o -name "*.yml" \)

print("\nMain model files:")
!find . -maxdepth 3 -type f -iname "*fastspeech*"

/content
Cloning into 'FastSpeech2'...
remote: Enumerating objects: 994, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 994 (delta 11), reused 7 (delta 7), pack-reused 978 (from 1)
Receiving objects: 100% (994/994), 330.31 MiB | 19.94 MiB/s, done.
Resolving deltas: 100% (176/176), done.
/content/FastSpeech2

Repository restored!

Top-level files and folders:
total 144
drwxr-xr-x 15 root root  4096 Nov  4 15:17 .
drwxr-xr-x  1 root root  4096 Nov  4 15:16 ..
drwxr-xr-x  2 root root  4096 Nov  4 15:17 audio
drwxr-xr-x  6 root root  4096 Nov  4 15:17 config
-rw-r--r--  1 root root  7912 Nov  4 15:17 dataset.py
drwxr-xr-x  5 root root  4096 Nov  4 15:17 demo
-rw-r--r--  1 root root  3715 Nov  4 15:17 evaluate.py
drwxr-xr-x  8 root root  4096 Nov  4 15:17 .git
-rw-r--r--  1 root root  1350 Nov  4 15:17 .gitignore
drwxr-xr-x  2 root root  4096 Nov  4 15:17 hifigan
drwxr-xr-x  2 root root  4096 Nov  4 15:17 img
-rw-r--r--  1 ro

In [15]:
# ============================================================
# STEP 63: INSPECT FASTSPEECH 2 ARCHITECTURE AND CONFIGURATION
# ============================================================

import os

REPO = "/content/FastSpeech2"

files_to_inspect = [
    "config/LJSpeech/preprocess.yaml",
    "config/LJSpeech/model.yaml",
    "config/LJSpeech/train.yaml",
    "model/fastspeech2.py",
    "dataset.py",
    "train.py",
    "text/__init__.py",
    "text/cleaners.py"
]

for relative_path in files_to_inspect:

    path = os.path.join(REPO, relative_path)

    print("\n" + "=" * 75)
    print("FILE:", relative_path)
    print("=" * 75)

    if not os.path.isfile(path):
        print("File not found")
        continue

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    print(content[:12000])

print("\nARCHITECTURE INSPECTION COMPLETE")


FILE: config/LJSpeech/preprocess.yaml
dataset: "LJSpeech"

path:
  corpus_path: "/home/ming/Data/LJSpeech-1.1"
  lexicon_path: "lexicon/librispeech-lexicon.txt"
  raw_path: "./raw_data/LJSpeech"
  preprocessed_path: "./preprocessed_data/LJSpeech"

preprocessing:
  val_size: 512
  text:
    text_cleaners: ["english_cleaners"]
    language: "en"
  audio:
    sampling_rate: 22050
    max_wav_value: 32768.0
  stft:
    filter_length: 1024
    hop_length: 256
    win_length: 1024
  mel:
    n_mel_channels: 80
    mel_fmin: 0
    mel_fmax: 8000 # please set to 8000 for HiFi-GAN vocoder, set to null for MelGAN vocoder
  pitch:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True
  energy:
    feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
    normalization: True


FILE: config/LJSpeech/model.yaml
transformer:
  encoder_layer: 4
  encoder_head: 2
  encoder_hidden: 256
  decoder_layer: 6
  decoder_head: 2
  decoder_hidden: 256
  c

In [16]:
# ============================================================
# STEP 64: INSPECT VARIANCE ADAPTOR AND TEXT VOCABULARY
# ============================================================

import os

REPO = "/content/FastSpeech2"

files = [
    "model/modules.py",
    "model/loss.py",
    "text/symbols.py",
    "text/cleaners.py",
    "preprocessor/preprocessor.py",
    "utils/tools.py"
]

for relative_path in files:

    path = os.path.join(REPO, relative_path)

    print("\n" + "=" * 75)
    print("FILE:", relative_path)
    print("=" * 75)

    if not os.path.isfile(path):
        print("File not found")
        continue

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    # Print relevant sections without flooding the notebook
    if relative_path == "model/modules.py":
        start = content.find("class VarianceAdaptor")
        print(content[start:start + 14000])

    elif relative_path == "text/symbols.py":
        print(content[:10000])

    elif relative_path == "model/loss.py":
        print(content[:10000])

    elif relative_path == "preprocessor/preprocessor.py":
        print(content[:10000])

    else:
        print(content[:7000])

print("\nSTEP 64 INSPECTION COMPLETE")


FILE: model/modules.py
class VarianceAdaptor(nn.Module):
    """Variance Adaptor"""

    def __init__(self, preprocess_config, model_config):
        super(VarianceAdaptor, self).__init__()
        self.duration_predictor = VariancePredictor(model_config)
        self.length_regulator = LengthRegulator()
        self.pitch_predictor = VariancePredictor(model_config)
        self.energy_predictor = VariancePredictor(model_config)

        self.pitch_feature_level = preprocess_config["preprocessing"]["pitch"][
            "feature"
        ]
        self.energy_feature_level = preprocess_config["preprocessing"]["energy"][
            "feature"
        ]
        assert self.pitch_feature_level in ["phoneme_level", "frame_level"]
        assert self.energy_feature_level in ["phoneme_level", "frame_level"]

        pitch_quantization = model_config["variance_embedding"]["pitch_quantization"]
        energy_quantization = model_config["variance_embedding"]["energy_quantization"]
        n_b

In [18]:
!pip install Unidecode

In [19]:
# ============================================================
# STEP 65: TELUGU FEATURE AND DURATION COMPATIBILITY AUDIT
# ============================================================

import os
import glob
import numpy as np
import json
from collections import Counter

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"
REPO = "/content/FastSpeech2"

TRAIN_DIR = f"{BASE}/Telugu/features/train"
VAL_DIR = f"{BASE}/Telugu/features/validation"

sys_path_added = REPO not in __import__("sys").path
if sys_path_added:
    import sys
    sys.path.insert(0, REPO)

from text.symbols import symbols

symbol_set = set(symbols)

def audit_split(split_name, folder):

    files = sorted(glob.glob(os.path.join(folder, "*.npz")))

    print(f"\n{'='*65}")
    print(f"{split_name}: {len(files)} feature files")
    print("="*65)

    duration_mismatches = 0
    token_mismatches = 0
    missing_symbols = Counter()

    mel_lengths = []
    token_lengths = []
    duration_sums = []
    duration_errors = []

    pitch_values = []
    energy_values = []

    for path in files:

        with np.load(path, allow_pickle=True) as data:

            mel = data["mel"]
            pitch = data["pitch"]
            energy = data["energy"]
            tokens = data["tokens"]
            durations = data["durations"]

            # Current mel layout is expected to be [80, frames]
            mel_frames = mel.shape[-1] if mel.shape[0] == 80 else mel.shape[0]

            token_list = [str(x) for x in tokens.tolist()]
            duration_list = np.asarray(durations).astype(int)

            mel_lengths.append(mel_frames)
            token_lengths.append(len(token_list))
            duration_sums.append(int(duration_list.sum()))

            if len(token_list) != len(duration_list):
                token_mismatches += 1

            if int(duration_list.sum()) != mel_frames:
                duration_mismatches += 1
                duration_errors.append(
                    int(duration_list.sum()) - mel_frames
                )

            for char in token_list:
                if char not in symbol_set:
                    missing_symbols[char] += 1

            pitch_values.extend(np.asarray(pitch).reshape(-1)[::20].tolist())
            energy_values.extend(np.asarray(energy).reshape(-1)[::20].tolist())

    print("Token-duration length mismatches:", token_mismatches)
    print("Duration sum vs mel frame mismatches:", duration_mismatches)

    if duration_errors:
        print("Duration discrepancy range:",
              min(duration_errors), "to", max(duration_errors), "frames")

    print("Average mel frames:", round(np.mean(mel_lengths), 2))
    print("Average tokens:", round(np.mean(token_lengths), 2))
    print("Average duration sum:", round(np.mean(duration_sums), 2))

    print("Characters missing from original vocabulary:",
          dict(missing_symbols))

    print("Sampled pitch range:",
          round(float(np.nanmin(pitch_values)), 4),
          "to", round(float(np.nanmax(pitch_values)), 4))

    print("Sampled energy range:",
          round(float(np.nanmin(energy_values)), 6),
          "to", round(float(np.nanmax(energy_values)), 6))

    # Inspect one example
    if files:
        with np.load(files[0], allow_pickle=True) as data:
            print("\nExample file:", os.path.basename(files[0]))
            print("Mel shape:", data["mel"].shape)
            print("Pitch shape:", data["pitch"].shape)
            print("Energy shape:", data["energy"].shape)
            print("Token count:", len(data["tokens"]))
            print("Duration count:", len(data["durations"]))
            print("Duration sum:", int(np.sum(data["durations"])))

            print("First 80 tokens:",
                  "".join(str(x) for x in data["tokens"].tolist())[:80])

audit_split("TELUGU TRAIN", TRAIN_DIR)
audit_split("TELUGU VALIDATION", VAL_DIR)

print("\nSTEP 65 AUDIT COMPLETE")


TELUGU TRAIN: 1280 feature files
Token-duration length mismatches: 0
Duration sum vs mel frame mismatches: 1280
Duration discrepancy range: -8627 to -138 frames
Average mel frames: 1214.27
Average tokens: 192.48
Average duration sum: 358.82
Characters missing from original vocabulary: {}
Sampled pitch range: 0.0 to 478.5612
Sampled energy range: 0.0 to 0.458576

Example file: train_telugufemale_00009.npz
Mel shape: (80, 1509)
Pitch shape: (1509,)
Energy shape: (1509,)
Token count: 210
Duration count: 210
Duration sum: 399
First 80 tokens: mitravimdakuddu talli maatta vinaka pooyitiiraalani muurkhimcaaddu aavidda tana 

TELUGU VALIDATION: 251 feature files
Token-duration length mismatches: 0
Duration sum vs mel frame mismatches: 251
Duration discrepancy range: -4078 to -157 frames
Average mel frames: 1239.41
Average tokens: 197.98
Average duration sum: 369.53
Characters missing from original vocabulary: {}
Sampled pitch range: 0.0 to 467.2492
Sampled energy range: 0.0 to 0.451382

Exam

In [22]:
import os
import json
import numpy as np

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"

feature_file = (
    f"{BASE}/Telugu/features/train/"
    "train_telugufemale_00009.npz"
)

alignment_file = (
    f"{BASE}/Telugu/metadata/telugu_train_mmsfa.jsonl"
)

# Load feature data
with np.load(feature_file, allow_pickle=True) as d:
    mel = d["mel"]
    pitch = d["pitch"]
    energy = d["energy"]
    tokens = d["tokens"].tolist()
    durations = d["durations"]

print("FEATURE DETAILS")
print("Mel shape:", mel.shape)
print("Pitch frames:", len(pitch))
print("Energy frames:", len(energy))
print("Token count:", len(tokens))
print("Duration sum:", int(np.sum(durations)))
print("First 20 durations:", durations[:20])

# Find matching alignment record
target_name = os.path.basename(feature_file)

with open(alignment_file, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        if os.path.basename(record["audio_path"]) == target_name.replace(".npz", ".wav"):
            print("\nMATCHED ALIGNMENT RECORD")
            print("Audio path:", record["audio_path"])
            print("Original sample rate:", record["original_sample_rate"])
            print("Alignment sample rate:", record["alignment_sample_rate"])
            print("Audio duration:", record["audio_duration"])
            print("Character alignment complete:",
                  record["character_alignment_complete"])

            print("\nROMANIZED TEXT:")
            print(record["normalized_text"][:300])

            print("\nFIRST 5 CHARACTER SPANS:")
            for span in record["character_spans"][:5]:
                print(span)

            print("\nLAST 5 CHARACTER SPANS:")
            for span in record["character_spans"][-5:]:
                print(span)

            print("\nFIRST 5 WORD SPANS:")
            for span in record["word_spans"][:5]:
                print(span)

            break
    else:
        print("Matching alignment record not found.")

FEATURE DETAILS
Mel shape: (80, 1509)
Pitch frames: 1509
Energy frames: 1509
Token count: 210
Duration sum: 399
First 20 durations: [2 2 2 3 2 2 2 2 2 2 2 2 2 2 2 0 2 3 2 2]


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/metadata/telugu_train_mmsfa.jsonl'

In [21]:
import os
import glob
import json

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"

print("Searching for Telugu alignment files...\n")

patterns = [
    "**/*telugu*train*mmsfa*.jsonl",
    "**/*telugu*validation*mmsfa*.jsonl",
    "**/*mmsfa*.jsonl"
]

found = set()

for pattern in patterns:
    for path in glob.glob(os.path.join(BASE, pattern), recursive=True):
        found.add(path)

for path in sorted(found):
    print(path, "|", os.path.getsize(path), "bytes")

print("\nSearching for the specific audio alignment record...")

target_audio = "train_telugufemale_00009.wav"

for path in sorted(found):
    if "train" not in os.path.basename(path).lower():
        continue

    with open(path, encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            if os.path.basename(record.get("audio_path", "")) == target_audio:
                print("\nMATCH FOUND:", path)
                print("Audio duration:", record.get("audio_duration"))
                print("Alignment sample rate:", record.get("alignment_sample_rate"))
                print("Character alignment complete:",
                      record.get("character_alignment_complete"))
                print("Romanized text:", record.get("normalized_text"))

                print("\nFirst 5 character spans:")
                for span in record.get("character_spans", [])[:5]:
                    print(span)

                print("\nLast 5 character spans:")
                for span in record.get("character_spans", [])[-5:]:
                    print(span)

                break
        else:
            continue
        break
else:
    print("\nSpecific alignment record not found in the discovered files.")

Searching for Telugu alignment files...

/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train_mmsfa.jsonl | 39454921 bytes
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train_mmsfa_errors.jsonl | 234810 bytes
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train_mmsfa_retry_errors.jsonl | 288 bytes
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_validation_mmsfa.jsonl | 7961106 bytes
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_validation_mmsfa_errors.jsonl | 47257 bytes
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_validation_mmsfa_retry_errors.jsonl | 0 bytes

Searching for the specific audio alignment record...

MATCH FOUND: /content/drive/MyDrive/FastSpeech2_Assignment/metadata/telugu_train_mmsfa.jsonl
Audio duration: 17.5104
Alignment sample rate: 16000
Character alignment complete: True
Romanized text: mitravimdakuddu talli maatta vinaka pooyitiiraalani muurkhimcaaddu aavidda tana kodduku 

In [23]:
import os
import json
import numpy as np

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"

feature_path = (
    f"{BASE}/Telugu/features/train/"
    "train_telugufemale_00009.npz"
)

alignment_path = (
    f"{BASE}/metadata/telugu_train_mmsfa.jsonl"
)

target_audio = "train_telugufemale_00009.wav"

# Load features
with np.load(feature_path, allow_pickle=True) as d:
    mel = d["mel"]
    tokens = d["tokens"].tolist()
    durations = np.asarray(d["durations"])

# Find alignment record
record = None

with open(alignment_path, encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if os.path.basename(item["audio_path"]) == target_audio:
            record = item
            break

if record is None:
    raise ValueError("Alignment record not found")

chars = record["character_spans"]
words = record["word_spans"]

# Character timing statistics
starts = np.array([x["start"] for x in chars])
ends = np.array([x["end"] for x in chars])
span_durations = np.array([x["duration"] for x in chars])

gaps = starts[1:] - ends[:-1]

print("=" * 65)
print("AUDIO AND FEATURE INFORMATION")
print("=" * 65)

print("Audio duration:", record["audio_duration"])
print("Mel frames:", mel.shape[-1])
print("Mel duration:", mel.shape[-1] * 256 / 22050)
print("Expected frames from audio duration:",
      round(record["audio_duration"] * 22050 / 256))

print("\n" + "=" * 65)
print("CHARACTER ALIGNMENT COVERAGE")
print("=" * 65)

print("Character spans:", len(chars))
print("First character start:", starts[0])
print("Last character end:", ends[-1])
print("Total character-span duration:", span_durations.sum())
print("Total positive gap duration:", np.maximum(gaps, 0).sum())
print("Total negative gap duration:", np.minimum(gaps, 0).sum())
print("Mean character duration:", span_durations.mean())
print("Mean positive gap:", np.maximum(gaps, 0).mean())

print("\nLargest 15 gaps:")
for i in np.argsort(gaps)[-15:][::-1]:
    print(
        f"Between {chars[i]['character']!r} and "
        f"{chars[i+1]['character']!r}: "
        f"{gaps[i]:.4f}s"
    )

print("\n" + "=" * 65)
print("WORD SPAN INFORMATION")
print("=" * 65)

print("Word spans:", len(words))

for word in words[:10]:
    print(word)

print("\nLast 5 word spans:")
for word in words[-5:]:
    print(word)

print("\n" + "=" * 65)
print("DURATION TARGET CHECK")
print("=" * 65)

print("Duration sum:", durations.sum())
print("Mel frame count:", mel.shape[-1])
print("Duration sum in seconds:",
      durations.sum() * 256 / 22050)

print("\nSTEP 68 COMPLETE")

AUDIO AND FEATURE INFORMATION
Audio duration: 17.5104
Mel frames: 1509
Mel duration: 17.519455782312924
Expected frames from audio duration: 1508

CHARACTER ALIGNMENT COVERAGE
Character spans: 191
First character start: 0.4403
Last character end: 16.6499
Total character-span duration: 4.16
Total positive gap duration: 12.047200000000005
Total negative gap duration: 0.0
Mean character duration: 0.021780104712041885
Mean positive gap: 0.06340631578947371

Largest 15 gaps:
Between 'i' and 'm': 1.0006s
Between 'u' and 'a': 0.8205s
Between 'i' and 't': 0.6004s
Between 'i' and 'v': 0.6003s
Between 'a' and 't': 0.5603s
Between 'u' and 't': 0.5403s
Between 'a' and 'p': 0.5003s
Between 'u' and 't': 0.4802s
Between 'i' and 'v': 0.4003s
Between 'i' and 'b': 0.1000s
Between 'i' and 'm': 0.1000s
Between 'a' and 'k': 0.0801s
Between 'l' and 'l': 0.0801s
Between 'l' and 'l': 0.0801s
Between 'o' and 'y': 0.0801s

WORD SPAN INFORMATION
Word spans: 20
{'word_index': 0, 'word': 'mitravimdakuddu', 'start'

In [24]:
import os
import json
import numpy as np

BASE = "/content/drive/MyDrive/FastSpeech2_Assignment"

FEATURE = f"{BASE}/Telugu/features/train/train_telugufemale_00009.npz"
ALIGNMENT = f"{BASE}/metadata/telugu_train_mmsfa.jsonl"

SR = 22050
HOP = 256

# Load feature data
with np.load(FEATURE, allow_pickle=True) as d:
    tokens = [str(x) for x in d["tokens"].tolist()]
    mel_frames = d["mel"].shape[-1]

# Load matching alignment
record = None

with open(ALIGNMENT, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if os.path.basename(r["audio_path"]) == "train_telugufemale_00009.wav":
            record = r
            break

if record is None:
    raise ValueError("Alignment not found")

words = record["word_spans"]
chars = record["character_spans"]

# Initialize durations for every token, including spaces
new_durations = np.zeros(len(tokens), dtype=np.int64)

# Identify token positions belonging to each word
word_positions = []
cursor = 0

for word in words:
    word_text = word["word"]
    positions = []

    for char in word_text:
        if cursor >= len(tokens) or tokens[cursor] != char:
            raise ValueError(
                f"Token mismatch at {cursor}: "
                f"expected {char}, got {tokens[cursor] if cursor < len(tokens) else 'END'}"
            )
        positions.append(cursor)
        cursor += 1

    word_positions.append(positions)

    # Skip the separating space token
    if cursor < len(tokens) and tokens[cursor] == " ":
        cursor += 1

if cursor != len(tokens):
    raise ValueError("Unmatched tokens remain")

# Group character spans by word index
char_groups = {}

for span in chars:
    char_groups.setdefault(span["word_index"], []).append(span)

# Convert a timestamp to a mel-frame boundary
def frame_boundary(seconds):
    return int(round(seconds * SR / HOP))

# Distribute each word's duration across its characters
for word_index, (word, positions) in enumerate(zip(words, word_positions)):

    group = char_groups[word_index]

    if len(group) != len(positions):
        raise ValueError(f"Character count mismatch in word {word_index}")

    weights = np.array(
        [max(float(x["duration"]), 0.001) for x in group]
    )

    start_frame = frame_boundary(word["start"])
    end_frame = frame_boundary(word["end"])

    word_frames = end_frame - start_frame

    if word_frames < 0:
        raise ValueError("Invalid word timing")

    exact = weights / weights.sum() * word_frames
    allocated = np.floor(exact).astype(int)

    remainder = word_frames - allocated.sum()

    # Largest-remainder allocation ensures exact word duration
    order = np.argsort(-(exact - allocated))

    for i in order[:remainder]:
        allocated[i] += 1

    for position, duration in zip(positions, allocated):
        new_durations[position] = duration

# Assign inter-word gaps and leading/trailing silence to spaces/edge tokens
# This preserves all mel frames.
for i in range(len(words) - 1):

    space_position = word_positions[i][-1] + 1

    if tokens[space_position] != " ":
        raise ValueError("Expected space token between words")

    gap_start = frame_boundary(words[i]["end"])
    gap_end = frame_boundary(words[i + 1]["start"])

    new_durations[space_position] = max(0, gap_end - gap_start)

# Include leading and trailing silence
leading = frame_boundary(words[0]["start"])
trailing = mel_frames - frame_boundary(words[-1]["end"])

new_durations[word_positions[0][0]] += max(0, leading)
new_durations[word_positions[-1][-1]] += max(0, trailing)

print("ORIGINAL DURATION SUM:", sum(
    np.load(FEATURE, allow_pickle=True)["durations"]
))
print("NEW DURATION SUM:", new_durations.sum())
print("MEL FRAME COUNT:", mel_frames)
print("DURATION MATCH:", new_durations.sum() == mel_frames)
print("TOKEN COUNT:", len(tokens))
print("NEW DURATION COUNT:", len(new_durations))
print("LEADING SILENCE FRAMES:", leading)
print("TRAILING SILENCE FRAMES:", trailing)
print("SPACE DURATIONS:", new_durations[
    np.array(tokens) == " "
])

print("\nFIRST 40 CORRECTED DURATIONS:")
print(new_durations[:40])

ORIGINAL DURATION SUM: 399
NEW DURATION SUM: 1509
MEL FRAME COUNT: 1509
DURATION MATCH: True
TOKEN COUNT: 210
NEW DURATION COUNT: 210
LEADING SILENCE FRAMES: 38
TRAILING SILENCE FRAMES: 75
SPACE DURATIONS: [41  5  4 43  8 71 48  7  5  5 51  8 86 47  5 52  5  3 35]

FIRST 40 CORRECTED DURATIONS:
[43  5  5  9  5  5  5  4  4  4  4  4  4  4  4 41  5  9  4  4  4  5  4  4
  4  4  4  4  4  5  5  5  4  4  4 43  6  6  6  5]


In [25]:
import os
import json
import numpy as np
from pathlib import Path
from collections import defaultdict

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

SR = 22050
HOP = 256

JOBS = [
    {
        "split": "train",
        "features": BASE / "Telugu/features/train",
        "alignment": BASE / "metadata/telugu_train_mmsfa.jsonl",
    },
    {
        "split": "validation",
        "features": BASE / "Telugu/features/validation",
        "alignment": BASE / "metadata/telugu_validation_mmsfa.jsonl",
    },
]

def frame_boundary(seconds):
    return int(round(float(seconds) * SR / HOP))

def correct_record(record, feature_path):
    with np.load(feature_path, allow_pickle=True) as data:
        tokens = [str(x) for x in data["tokens"].tolist()]
        mel_frames = data["mel"].shape[-1]
        original_durations = data["durations"].copy()

    words = record["word_spans"]
    chars = record["character_spans"]

    corrected = np.zeros(len(tokens), dtype=np.int64)
    word_positions = []
    cursor = 0

    # Match romanized words to the exact feature tokens
    for word in words:
        positions = []

        for char in word["word"]:
            if cursor >= len(tokens) or tokens[cursor] != char:
                raise ValueError(f"Token mismatch: {feature_path.name}")

            positions.append(cursor)
            cursor += 1

        word_positions.append(positions)

        if cursor < len(tokens) and tokens[cursor] == " ":
            cursor += 1

    if cursor != len(tokens):
        raise ValueError(f"Unmatched tokens: {feature_path.name}")

    # Group character alignments by word
    groups = defaultdict(list)

    for span in chars:
        groups[span["word_index"]].append(span)

    # Allocate each word's frame duration across its characters
    for word_index, (word, positions) in enumerate(zip(words, word_positions)):
        group = groups[word_index]

        if len(group) != len(positions):
            raise ValueError(f"Character alignment mismatch: {feature_path.name}")

        weights = np.array(
            [max(float(x["duration"]), 0.001) for x in group]
        )

        start = frame_boundary(word["start"])
        end = frame_boundary(word["end"])
        word_frames = end - start

        if word_frames < 0:
            raise ValueError(f"Negative word duration: {feature_path.name}")

        exact = weights / weights.sum() * word_frames
        allocated = np.floor(exact).astype(int)

        remainder = word_frames - allocated.sum()
        order = np.argsort(-(exact - allocated))

        for i in order[:remainder]:
            allocated[i] += 1

        corrected[positions] = allocated

    # Assign gaps between words to space tokens
    for i in range(len(words) - 1):
        space_position = word_positions[i][-1] + 1

        if tokens[space_position] != " ":
            raise ValueError(f"Missing space token: {feature_path.name}")

        gap = (
            frame_boundary(words[i + 1]["start"])
            - frame_boundary(words[i]["end"])
        )

        corrected[space_position] = max(0, gap)

    # Account for leading and trailing silence
    leading = frame_boundary(words[0]["start"])
    trailing = mel_frames - frame_boundary(words[-1]["end"])

    corrected[word_positions[0][0]] += max(0, leading)
    corrected[word_positions[-1][-1]] += max(0, trailing)

    if corrected.sum() != mel_frames:
        raise ValueError(
            f"Frame mismatch in {feature_path.name}: "
            f"{corrected.sum()} vs {mel_frames}"
        )

    return corrected, original_durations


total_updated = 0
total_failed = 0
reports = []

for job in JOBS:
    split = job["split"]
    feature_dir = job["features"]

    alignment_by_name = {}

    with open(job["alignment"], encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            name = Path(record["audio_path"]).stem
            alignment_by_name[name] = record

    files = sorted(feature_dir.glob("*.npz"))
    updated = 0
    failed = []

    for feature_path in files:
        record = alignment_by_name.get(feature_path.stem)

        if record is None:
            failed.append((feature_path.name, "Alignment missing"))
            continue

        try:
            new_durations, old_durations = correct_record(
                record, feature_path
            )

            with np.load(feature_path, allow_pickle=True) as data:
                arrays = {key: data[key] for key in data.files}

            arrays["durations"] = new_durations

            # Save atomically
            temp_path = feature_path.with_suffix(".tmp.npz")
            np.savez_compressed(temp_path, **arrays)
            os.replace(temp_path, feature_path)

            updated += 1

        except Exception as e:
            failed.append((feature_path.name, str(e)))

    total_updated += updated
    total_failed += len(failed)

    print(f"\n{split.upper()}")
    print("Feature files:", len(files))
    print("Successfully corrected:", updated)
    print("Failed:", len(failed))

    if failed[:10]:
        print("First failures:", failed[:10])

    reports.append({
        "split": split,
        "files": len(files),
        "updated": updated,
        "failed": len(failed),
    })

print("\n" + "=" * 55)
print("TOTAL UPDATED:", total_updated)
print("TOTAL FAILED:", total_failed)
print("=" * 55)

if total_failed:
    print("Do not proceed to training yet. Share the failures.")
else:
    print("All Telugu duration targets corrected.")


TRAIN
Feature files: 1280
Successfully corrected: 1280
Failed: 0

VALIDATION
Feature files: 251
Successfully corrected: 251
Failed: 0

TOTAL UPDATED: 1531
TOTAL FAILED: 0
All Telugu duration targets corrected.


In [26]:
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

SPLITS = {
    "TRAIN": BASE / "Telugu/features/train",
    "VALIDATION": BASE / "Telugu/features/validation",
}

for split, folder in SPLITS.items():
    files = sorted(folder.glob("*.npz"))

    valid = 0
    failures = []
    total_frames = 0
    total_tokens = 0
    total_spaces = 0

    for path in files:
        try:
            with np.load(path, allow_pickle=True) as d:
                mel = d["mel"]
                pitch = d["pitch"]
                energy = d["energy"]
                durations = d["durations"].astype(np.int64)
                tokens = [str(x) for x in d["tokens"].tolist()]

            checks = {
                "mel_duration": mel.shape[-1] == len(pitch) == len(energy),
                "token_duration": len(tokens) == len(durations),
                "duration_sum": durations.sum() == mel.shape[-1],
                "nonnegative": np.all(durations >= 0),
                "finite_mel": np.isfinite(mel).all(),
                "finite_pitch": np.isfinite(pitch).all(),
                "finite_energy": np.isfinite(energy).all(),
            }

            if all(checks.values()):
                valid += 1
            else:
                failures.append((path.name, checks))

            total_frames += mel.shape[-1]
            total_tokens += len(tokens)
            total_spaces += sum(t == " " for t in tokens)

        except Exception as e:
            failures.append((path.name, str(e)))

    print("\n" + "=" * 55)
    print(split)
    print("=" * 55)
    print("Total files:", len(files))
    print("Valid files:", valid)
    print("Failed files:", len(failures))
    print("Total mel frames:", total_frames)
    print("Total tokens:", total_tokens)
    print("Total space tokens:", total_spaces)

    if failures:
        print("\nFirst failures:")
        for item in failures[:5]:
            print(item)

print("\nVERIFICATION COMPLETE")


TRAIN
Total files: 1280
Valid files: 1280
Failed files: 0
Total mel frames: 1554270
Total tokens: 246370
Total space tokens: 23608

VALIDATION
Total files: 251
Valid files: 251
Failed files: 0
Total mel frames: 311092
Total tokens: 49693
Total space tokens: 4750

VERIFICATION COMPLETE


In [27]:
from pathlib import Path

REPO = Path("/content/FastSpeech2")

files_to_inspect = [
    "dataset.py",
    "text/__init__.py",
    "text/symbols.py",
    "model/fastspeech2.py",
    "model/loss.py",
    "config/LJSpeech/model.yaml",
]

for filename in files_to_inspect:
    path = REPO / filename

    print("\n" + "=" * 80)
    print(f"FILE: {filename}")
    print("=" * 80)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    # Print relevant source sections without dumping the entire model
    if filename == "dataset.py":
        ranges = [(1, 240)]
    elif filename == "text/__init__.py":
        ranges = [(1, 180)]
    elif filename == "text/symbols.py":
        ranges = [(1, 100)]
    elif filename == "model/fastspeech2.py":
        ranges = [(1, 180), (180, 360)]
    elif filename == "model/loss.py":
        ranges = [(1, 200)]
    else:
        ranges = [(1, 240)]

    for start, end in ranges:
        for i in range(start - 1, min(end, len(lines))):
            print(f"{i+1:4}: {lines[i]}")

# Inspect one existing Telugu feature file
import numpy as np

feature_path = (
    "/content/drive/MyDrive/FastSpeech2_Assignment/"
    "Telugu/features/train/train_telugufemale_00009.npz"
)

with np.load(feature_path, allow_pickle=True) as d:
    print("\n" + "=" * 80)
    print("TELUGU FEATURE FILE")
    print("=" * 80)

    for key in d.files:
        value = d[key]
        print(
            key,
            "| shape:", getattr(value, "shape", None),
            "| dtype:", getattr(value, "dtype", None)
        )


FILE: dataset.py
   1: import json
   2: import math
   3: import os
   4: 
   5: import numpy as np
   6: from torch.utils.data import Dataset
   7: 
   8: from text import text_to_sequence
   9: from utils.tools import pad_1D, pad_2D
  10: 
  11: 
  12: class Dataset(Dataset):
  13:     def __init__(
  14:         self, filename, preprocess_config, train_config, sort=False, drop_last=False
  15:     ):
  16:         self.dataset_name = preprocess_config["dataset"]
  17:         self.preprocessed_path = preprocess_config["path"]["preprocessed_path"]
  18:         self.cleaners = preprocess_config["preprocessing"]["text"]["text_cleaners"]
  19:         self.batch_size = train_config["optimizer"]["batch_size"]
  20: 
  21:         self.basename, self.speaker, self.text, self.raw_text = self.process_meta(
  22:             filename
  23:         )
  24:         with open(os.path.join(self.preprocessed_path, "speakers.json")) as f:
  25:             self.speaker_map = json.load(f)
  26: 

In [28]:
from pathlib import Path

REPO = Path("/content/FastSpeech2")

files = [
    "config/LJSpeech/preprocess.yaml",
    "config/LJSpeech/train.yaml",
    "model/modules.py",
    "transformer/Models.py",
    "utils/tools.py",
    "text/cleaners.py",
]

for filename in files:
    path = REPO / filename

    print("\n" + "=" * 75)
    print("FILE:", filename)
    print("=" * 75)

    if not path.exists():
        print("NOT FOUND")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    # Keep the output focused on configuration and model interfaces
    if filename.endswith("preprocess.yaml") or filename.endswith("train.yaml"):
        start, end = 1, len(lines)
    elif filename == "model/modules.py":
        start, end = 1, min(len(lines), 260)
    elif filename == "transformer/Models.py":
        start, end = 1, min(len(lines), 180)
    elif filename == "utils/tools.py":
        start, end = 1, min(len(lines), 150)
    else:
        start, end = 1, min(len(lines), 180)

    for i in range(start - 1, end):
        print(f"{i+1:4}: {lines[i]}")

print("\nINSPECTION COMPLETE")


FILE: config/LJSpeech/preprocess.yaml
   1: dataset: "LJSpeech"
   2: 
   3: path:
   4:   corpus_path: "/home/ming/Data/LJSpeech-1.1"
   5:   lexicon_path: "lexicon/librispeech-lexicon.txt"
   6:   raw_path: "./raw_data/LJSpeech"
   7:   preprocessed_path: "./preprocessed_data/LJSpeech"
   8: 
   9: preprocessing:
  10:   val_size: 512
  11:   text:
  12:     text_cleaners: ["english_cleaners"]
  13:     language: "en"
  14:   audio:
  15:     sampling_rate: 22050
  16:     max_wav_value: 32768.0
  17:   stft:
  18:     filter_length: 1024
  19:     hop_length: 256
  20:     win_length: 1024
  21:   mel:
  22:     n_mel_channels: 80
  23:     mel_fmin: 0
  24:     mel_fmax: 8000 # please set to 8000 for HiFi-GAN vocoder, set to null for MelGAN vocoder
  25:   pitch:
  26:     feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
  27:     normalization: True
  28:   energy:
  29:     feature: "phoneme_level" # support 'phoneme_level' or 'frame_level'
  30:     normaliza

In [4]:
from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

# Check assignment folder
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

print("\nGOOGLE DRIVE STATUS")
print("Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Assignment folder exists:", BASE.exists())

if BASE.exists():

    print("\nASSIGNMENT FOLDER CONTENTS:")
    for item in BASE.iterdir():
        print("📁" if item.is_dir() else "📄", item.name)

    feature_train = BASE / "Telugu/features/train"
    feature_val = BASE / "Telugu/features/validation"

    print("\nTELUGU FEATURE CHECK")
    print("Training folder:", feature_train.exists())
    print("Validation folder:", feature_val.exists())

    if feature_train.exists():
        print("Training NPZ files:", len(list(feature_train.glob("*.npz"))))

    if feature_val.exists():
        print("Validation NPZ files:", len(list(feature_val.glob("*.npz"))))

else:
    print("\nAssignment folder not found.")
    print("Check the folder name in your Google Drive.")

Mounted at /content/drive

GOOGLE DRIVE STATUS
Drive mounted: True
Assignment folder exists: True

ASSIGNMENT FOLDER CONTENTS:
📁 metadata
📁 raw_data
📁 English
📁 Telugu

TELUGU FEATURE CHECK
Training folder: True
Validation folder: True
Training NPZ files: 1280
Validation NPZ files: 251


In [5]:
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

for split in ["train", "validation"]:

    folder = BASE / f"Telugu/features/{split}"

    print("\n" + "=" * 60)
    print("CHECKING:", split.upper())
    print("Folder:", folder)
    print("Folder exists:", folder.exists())
    print("=" * 60)

    if not folder.exists():
        raise FileNotFoundError(
            f"Folder not found: {folder}\n"
            "Check whether Google Drive is mounted."
        )

    files = sorted(folder.glob("*.npz"))

    print("NPZ files found:", len(files))

    if len(files) == 0:
        raise RuntimeError(
            f"No NPZ files found in {folder}. "
            "Do not proceed until the path is corrected."
        )

    mel_lengths = []
    token_lengths = []
    pitch_values = []
    energy_values = []

    for path in files:
        with np.load(path, allow_pickle=True) as d:
            mel_lengths.append(d["mel"].shape[-1])
            token_lengths.append(len(d["tokens"]))
            pitch_values.append(d["pitch"])
            energy_values.append(d["energy"])

    mel_lengths = np.array(mel_lengths)
    token_lengths = np.array(token_lengths)

    pitch_values = np.concatenate(pitch_values)
    energy_values = np.concatenate(energy_values)

    print("\nTotal files:", len(files))

    for name, values in [
        ("Mel frames", mel_lengths),
        ("Token count", token_lengths),
    ]:
        print(f"\n{name}:")
        for q in [0, 50, 90, 95, 99, 100]:
            print(f"  {q:3d}th percentile: {np.percentile(values, q):.1f}")

    print("\nPitch:")
    print("  Min:", pitch_values.min())
    print("  Max:", pitch_values.max())
    print("  Mean:", pitch_values.mean())
    print("  Std:", pitch_values.std())

    print("\nEnergy:")
    print("  Min:", energy_values.min())
    print("  Max:", energy_values.max())
    print("  Mean:", energy_values.mean())
    print("  Std:", energy_values.std())

print("\nLENGTH AND STATISTICS AUDIT COMPLETE")


CHECKING: TRAIN
Folder: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/features/train
Folder exists: True
NPZ files found: 1280

Total files: 1280

Mel frames:
    0th percentile: 177.0
   50th percentile: 857.0
   90th percentile: 2541.0
   95th percentile: 3200.5
   99th percentile: 4892.0
  100th percentile: 11429.0

Token count:
    0th percentile: 14.0
   50th percentile: 145.0
   90th percentile: 391.0
   95th percentile: 469.2
   99th percentile: 721.1
  100th percentile: 1508.0

Pitch:
  Min: 0.0
  Max: 528.8987
  Mean: 109.76953
  Std: 103.30357

Energy:
  Min: 0.0
  Max: 0.5278873
  Mean: 0.052461125
  Std: 0.05535233

CHECKING: VALIDATION
Folder: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/features/validation
Folder exists: True
NPZ files found: 251

Total files: 251

Mel frames:
    0th percentile: 185.0
   50th percentile: 834.0
   90th percentile: 2683.0
   95th percentile: 3413.0
   99th percentile: 4608.5
  100th percentile: 5905.0

Token count:
    0th 

In [9]:
%pip install Unidecode

In [10]:
import sys
from pathlib import Path

REPO = Path("/content/FastSpeech2")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from text.symbols import symbols

symbol_to_id = {symbol: i for i, symbol in enumerate(symbols)}

print("FastSpeech2 vocabulary imported successfully!")
print("Vocabulary size:", len(symbols))
print("Padding ID:", symbol_to_id["_"])

FastSpeech2 vocabulary imported successfully!
Vocabulary size: 360
Padding ID: 0


In [11]:
from pathlib import Path
import sys
import subprocess

REPO = Path("/content/FastSpeech2")

# Clone only if the repository is missing
if not (REPO / "text" / "symbols.py").exists():
    if REPO.exists():
        print("Incomplete repository found. Removing it...")
        import shutil
        shutil.rmtree(REPO)

    print("Cloning FastSpeech2 repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/ming024/FastSpeech2.git", str(REPO)],
        check=True
    )

# Add repository to Python path
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from text.symbols import symbols

print("Repository restored successfully!")
print("Vocabulary size:", len(symbols))
print("Symbols file:", REPO / "text" / "symbols.py")

Repository restored successfully!
Vocabulary size: 360
Symbols file: /content/FastSpeech2/text/symbols.py


In [12]:
# ================================================================
# FASTSPEECH 2 - TELUGU DATASET PREPARATION
# STEP 74: CHUNKING + VOCABULARY + STATISTICS + CONFIGURATION
# ================================================================

import os
import sys
import json
import csv
import shutil
import numpy as np
import yaml

from pathlib import Path
from collections import Counter

# -------------------- PATHS --------------------

REPO = Path("/content/FastSpeech2")
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

SOURCE_DIRS = {
    "train": BASE / "Telugu/features/train",
    "validation": BASE / "Telugu/features/validation",
}

OUTPUT_ROOT = BASE / "Telugu/fs2_prepared"

CHUNK_DIRS = {
    "train": OUTPUT_ROOT / "train",
    "validation": OUTPUT_ROOT / "validation",
}

CONFIG_DIR = BASE / "Telugu/config"

for folder in CHUNK_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- SETTINGS --------------------

SAMPLE_RATE = 22050
HOP_LENGTH = 256
N_MELS = 80

MAX_MEL_FRAMES = 2048
MAX_TEXT_TOKENS = 900

# -------------------- REPOSITORY VOCABULARY --------------------

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from text.symbols import symbols

symbol_to_id = {symbol: i for i, symbol in enumerate(symbols)}

PAD_ID = symbol_to_id["_"]

print("Repository vocabulary size:", len(symbols))
print("Padding ID:", PAD_ID)

# -------------------- WORD GROUPING --------------------

def get_word_groups(tokens, durations):
    """
    Identify words using spaces.
    Attach each inter-word space to the preceding word.
    """

    groups = []
    start = 0

    for i, token in enumerate(tokens):
        if token == " ":
            groups.append((start, i + 1))
            start = i + 1

    if start < len(tokens):
        groups.append((start, len(tokens)))

    if not groups:
        raise ValueError("No words found")

    # Every group must contain at least one non-space token
    for start, end in groups:
        if not any(t != " " for t in tokens[start:end]):
            raise ValueError("Empty word group detected")

    return groups


def make_chunks(tokens, durations, mel_frames):
    """
    Greedy word-boundary chunking.

    Each chunk has:
    - Maximum 2048 mel frames
    - Maximum 900 text tokens

    Frame boundaries are obtained from cumulative durations.
    """

    groups = get_word_groups(tokens, durations)

    chunks = []
    current_start = groups[0][0]
    current_end = current_start
    current_frames = 0

    for group_start, group_end in groups:

        group_frames = int(durations[group_start:group_end].sum())
        group_tokens = group_end - group_start

        if group_frames > MAX_MEL_FRAMES:
            raise ValueError(
                f"Single word group exceeds frame limit: {group_frames}"
            )

        if group_tokens > MAX_TEXT_TOKENS:
            raise ValueError(
                f"Single word group exceeds token limit: {group_tokens}"
            )

        proposed_frames = current_frames + group_frames
        proposed_tokens = group_end - current_start

        if (
            current_end > current_start
            and (
                proposed_frames > MAX_MEL_FRAMES
                or proposed_tokens > MAX_TEXT_TOKENS
            )
        ):
            chunks.append((current_start, current_end))

            current_start = group_start
            current_end = group_end
            current_frames = group_frames

        else:
            current_end = group_end
            current_frames = proposed_frames

    if current_end > current_start:
        chunks.append((current_start, current_end))

    # Verify exact token/frame coverage
    if chunks[0][0] != 0 or chunks[-1][1] != len(tokens):
        raise ValueError("Chunk boundaries do not cover all tokens")

    for i in range(1, len(chunks)):
        if chunks[i - 1][1] != chunks[i][0]:
            raise ValueError("Token overlap or gap between chunks")

    frame_cursor = 0
    output = []

    for start, end in chunks:
        chunk_duration = durations[start:end]
        chunk_frames = int(chunk_duration.sum())

        output.append({
            "token_start": start,
            "token_end": end,
            "frame_start": frame_cursor,
            "frame_end": frame_cursor + chunk_frames,
        })

        frame_cursor += chunk_frames

    if frame_cursor != mel_frames:
        raise ValueError(
            f"Chunk frame coverage mismatch: {frame_cursor} vs {mel_frames}"
        )

    return output


# -------------------- FEATURE STATISTICS --------------------

pitch_min = float("inf")
pitch_max = float("-inf")
energy_min = float("inf")
energy_max = float("-inf")

pitch_sum = 0.0
pitch_sq_sum = 0.0
pitch_count = 0

energy_sum = 0.0
energy_sq_sum = 0.0
energy_count = 0

# -------------------- MANIFESTS --------------------

manifests = {
    "train": [],
    "validation": [],
}

all_failures = []
total_chunks = Counter()
total_originals = Counter()

# -------------------- PROCESS BOTH SPLITS --------------------

for split in ["train", "validation"]:

    print("\n" + "=" * 65)
    print("PROCESSING:", split.upper())
    print("=" * 65)

    source_files = sorted(SOURCE_DIRS[split].glob("*.npz"))

    if not source_files:
        raise FileNotFoundError(
            f"No source feature files found: {SOURCE_DIRS[split]}"
        )

    for file_index, source_path in enumerate(source_files, start=1):

        try:
            with np.load(source_path, allow_pickle=True) as data:

                mel = data["mel"].astype(np.float32)
                pitch = data["pitch"].astype(np.float32)
                energy = data["energy"].astype(np.float32)
                tokens = [str(x) for x in data["tokens"].tolist()]
                durations = data["durations"].astype(np.int64)

            # Validate source arrays
            mel_frames = mel.shape[-1]

            if mel.shape[0] != N_MELS:
                raise ValueError("Unexpected mel channel count")

            if not (
                len(pitch) == len(energy) == mel_frames
            ):
                raise ValueError("Frame feature lengths differ")

            if not (
                len(tokens) == len(durations)
                and durations.sum() == mel_frames
            ):
                raise ValueError("Duration/token/frame mismatch")

            if np.any(durations < 0):
                raise ValueError("Negative duration found")

            if not (
                np.isfinite(mel).all()
                and np.isfinite(pitch).all()
                and np.isfinite(energy).all()
            ):
                raise ValueError("Non-finite acoustic features")

            # Verify every character belongs to model vocabulary
            unknown = set(tokens) - set(symbol_to_id)

            if unknown:
                raise ValueError(
                    f"Unknown vocabulary symbols: {unknown}"
                )

            # Collect statistics from original full recordings
            pitch_min = min(pitch_min, float(pitch.min()))
            pitch_max = max(pitch_max, float(pitch.max()))
            energy_min = min(energy_min, float(energy.min()))
            energy_max = max(energy_max, float(energy.max()))

            pitch_sum += float(pitch.sum())
            pitch_sq_sum += float(np.square(pitch).sum())
            pitch_count += len(pitch)

            energy_sum += float(energy.sum())
            energy_sq_sum += float(np.square(energy).sum())
            energy_count += len(energy)

            # Create word-boundary chunks
            chunks = make_chunks(tokens, durations, mel_frames)

            original_id = source_path.stem

            for chunk_index, chunk in enumerate(chunks):

                ts = chunk["token_start"]
                te = chunk["token_end"]
                fs = chunk["frame_start"]
                fe = chunk["frame_end"]

                chunk_tokens = tokens[ts:te]
                chunk_durations = durations[ts:te]

                chunk_mel = mel[:, fs:fe]
                chunk_pitch = pitch[fs:fe]
                chunk_energy = energy[fs:fe]

                # Character-to-ID mapping
                text_ids = np.array(
                    [symbol_to_id[t] for t in chunk_tokens],
                    dtype=np.int64
                )

                # Final chunk-level consistency checks
                if len(text_ids) != len(chunk_durations):
                    raise ValueError("Chunk token/duration mismatch")

                if chunk_durations.sum() != chunk_mel.shape[-1]:
                    raise ValueError("Chunk duration/mel mismatch")

                if not (
                    len(chunk_pitch)
                    == len(chunk_energy)
                    == chunk_mel.shape[-1]
                ):
                    raise ValueError("Chunk frame mismatch")

                if chunk_mel.shape[-1] == 0:
                    raise ValueError("Empty mel chunk")

                chunk_id = f"{original_id}_chunk{chunk_index:03d}"

                output_path = CHUNK_DIRS[split] / f"{chunk_id}.npz"

                np.savez_compressed(
                    output_path,
                    mel=chunk_mel,
                    pitch=chunk_pitch,
                    energy=chunk_energy,
                    tokens=np.array(chunk_tokens),
                    text_ids=text_ids,
                    durations=chunk_durations,
                    original_id=np.array(original_id),
                    chunk_index=np.array(chunk_index),
                    sample_rate=np.array(SAMPLE_RATE),
                    hop_length=np.array(HOP_LENGTH),
                )

                manifests[split].append({
                    "chunk_id": chunk_id,
                    "original_id": original_id,
                    "feature_path": str(output_path),
                    "text": "".join(chunk_tokens),
                    "text_length": len(text_ids),
                    "mel_frames": chunk_mel.shape[-1],
                    "duration_sum": int(chunk_durations.sum()),
                    "chunk_index": chunk_index,
                })

                total_chunks[split] += 1

            total_originals[split] += 1

        except Exception as e:
            all_failures.append({
                "split": split,
                "file": source_path.name,
                "error": str(e),
            })

        if file_index % 100 == 0:
            print(
                f"Processed {file_index}/{len(source_files)} "
                f"recordings | chunks: {total_chunks[split]}"
            )

    print("Original recordings:", total_originals[split])
    print("Generated chunks:", total_chunks[split])


# -------------------- STOP ON DATA ERRORS --------------------

if all_failures:
    error_path = OUTPUT_ROOT / "preparation_errors.json"
    with open(error_path, "w", encoding="utf-8") as f:
        json.dump(all_failures, f, indent=2)

    raise RuntimeError(
        f"{len(all_failures)} recordings failed. "
        f"See {error_path}"
    )


# -------------------- SAVE MANIFESTS --------------------

for split in ["train", "validation"]:

    manifest_path = OUTPUT_ROOT / f"{split}_manifest.csv"

    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=list(manifests[split][0].keys())
        )
        writer.writeheader()
        writer.writerows(manifests[split])


# -------------------- GLOBAL STATISTICS --------------------

pitch_mean = pitch_sum / pitch_count
pitch_std = max(
    1e-8,
    (pitch_sq_sum / pitch_count - pitch_mean ** 2) ** 0.5
)

energy_mean = energy_sum / energy_count
energy_std = max(
    1e-8,
    (energy_sq_sum / energy_count - energy_mean ** 2) ** 0.5
)

stats = {
    "pitch": [
        pitch_min, pitch_max, pitch_mean, pitch_std
    ],
    "energy": [
        energy_min, energy_max, energy_mean, energy_std
    ],
}

PREPROCESSED_DIR = OUTPUT_ROOT / "model_metadata"
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

with open(PREPROCESSED_DIR / "stats.json", "w") as f:
    json.dump(stats, f, indent=2)

with open(PREPROCESSED_DIR / "speakers.json", "w") as f:
    json.dump({"telugu": 0}, f, indent=2)


# -------------------- FASTSPEECH 2 CONFIGURATION --------------------

preprocess_config = {
    "dataset": "Telugu_MMS_FA",
    "path": {
        "corpus_path": str(BASE / "Telugu"),
        "lexicon_path": "",
        "raw_path": str(BASE / "Telugu/raw_audio"),
        "preprocessed_path": str(PREPROCESSED_DIR),
    },
    "preprocessing": {
        "val_size": 0,
        "text": {
            "text_cleaners": ["basic_cleaners"],
            "language": "te_romanized",
        },
        "audio": {
            "sampling_rate": SAMPLE_RATE,
            "max_wav_value": 32768.0,
        },
        "stft": {
            "filter_length": 1024,
            "hop_length": HOP_LENGTH,
            "win_length": 1024,
        },
        "mel": {
            "n_mel_channels": N_MELS,
            "mel_fmin": 0,
            "mel_fmax": 8000,
        },
        "pitch": {
            "feature": "frame_level",
            "normalization": False,
        },
        "energy": {
            "feature": "frame_level",
            "normalization": False,
        },
    },
}

model_config = {
    "transformer": {
        "encoder_layer": 2,
        "encoder_head": 2,
        "encoder_hidden": 128,
        "decoder_layer": 2,
        "decoder_head": 2,
        "decoder_hidden": 128,
        "conv_filter_size": 512,
        "conv_kernel_size": [9, 1],
        "encoder_dropout": 0.1,
        "decoder_dropout": 0.1,
    },
    "variance_predictor": {
        "filter_size": 128,
        "kernel_size": 3,
        "dropout": 0.2,
    },
    "variance_embedding": {
        "pitch_quantization": "linear",
        "energy_quantization": "linear",
        "n_bins": 256,
    },
    "multi_speaker": False,
    "max_seq_len": MAX_MEL_FRAMES,
    "vocoder": {
        "model": "HiFi-GAN",
        "speaker": "LJSpeech",
    },
}

train_config = {
    "path": {
        "ckpt_path": str(BASE / "Telugu/checkpoints"),
        "log_path": str(BASE / "Telugu/logs"),
        "result_path": str(BASE / "Telugu/results"),
    },
    "optimizer": {
        "batch_size": 1,
        "betas": [0.9, 0.98],
        "eps": 1e-9,
        "weight_decay": 0.0,
        "grad_clip_thresh": 1.0,
        "grad_acc_step": 1,
        "warm_up_step": 100,
        "anneal_steps": [10000, 20000],
        "anneal_rate": 0.3,
    },
    "step": {
        "total_step": 5000,
        "log_step": 10,
        "synth_step": 500,
        "val_step": 100,
        "save_step": 100,
    },
}

# -------------------- SAVE CONFIGURATION FILES --------------------

configs = {
    "preprocess.yaml": preprocess_config,
    "model.yaml": model_config,
    "train.yaml": train_config,
}

for filename, config in configs.items():
    with open(CONFIG_DIR / filename, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            config,
            f,
            sort_keys=False,
            allow_unicode=True
        )

# -------------------- FINAL VERIFICATION --------------------

print("\n" + "=" * 70)
print("TELUGU DATASET PREPARATION SUMMARY")
print("=" * 70)

for split in ["train", "validation"]:

    records = manifests[split]

    frame_lengths = np.array(
        [r["mel_frames"] for r in records]
    )

    token_lengths = np.array(
        [r["text_length"] for r in records]
    )

    print(f"\n{split.upper()}")
    print("Original recordings:", total_originals[split])
    print("Generated chunks:", len(records))
    print("Maximum mel frames:", frame_lengths.max())
    print("Maximum text tokens:", token_lengths.max())
    print("Average mel frames:", round(frame_lengths.mean(), 2))

    assert frame_lengths.max() <= MAX_MEL_FRAMES
    assert token_lengths.max() <= MAX_TEXT_TOKENS

print("\nGLOBAL FEATURE STATISTICS")
print(json.dumps(stats, indent=2))

print("\nCONFIGURATION FILES")
for filename in configs:
    print(CONFIG_DIR / filename)

print("\nMETADATA FILES")
print(PREPROCESSED_DIR / "stats.json")
print(PREPROCESSED_DIR / "speakers.json")

print("\n" + "=" * 70)
print("STEP 74 COMPLETED SUCCESSFULLY")
print("Original feature files were not modified.")
print("=" * 70)

Repository vocabulary size: 360
Padding ID: 0

PROCESSING: TRAIN
Processed 100/1280 recordings | chunks: 138
Processed 200/1280 recordings | chunks: 265
Processed 300/1280 recordings | chunks: 398
Processed 400/1280 recordings | chunks: 568
Processed 500/1280 recordings | chunks: 681
Processed 600/1280 recordings | chunks: 781
Processed 700/1280 recordings | chunks: 901
Processed 800/1280 recordings | chunks: 1023
Processed 900/1280 recordings | chunks: 1135
Processed 1000/1280 recordings | chunks: 1240
Processed 1100/1280 recordings | chunks: 1340
Processed 1200/1280 recordings | chunks: 1440
Original recordings: 1280
Generated chunks: 1520

PROCESSING: VALIDATION
Processed 100/251 recordings | chunks: 136
Processed 200/251 recordings | chunks: 250
Original recordings: 251
Generated chunks: 301

TELUGU DATASET PREPARATION SUMMARY

TRAIN
Original recordings: 1280
Generated chunks: 1520
Maximum mel frames: 2048
Maximum text tokens: 464
Average mel frames: 1022.55

VALIDATION
Original re

In [13]:
# ================================================================
# STEP 75A: TELUGU SPEAKER CONFIGURATION + TRAINING PREFLIGHT
# ================================================================

import json
import yaml
import ast
import numpy as np
from pathlib import Path
from collections import Counter

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

TRAIN_DIR = BASE / "Telugu/fs2_prepared/train"
VAL_DIR = BASE / "Telugu/fs2_prepared/validation"
META_DIR = BASE / "Telugu/fs2_prepared/model_metadata"
CONFIG_DIR = BASE / "Telugu/config"

# ------------------------------------------------
# 1. VERIFY PREPARED DATA
# ------------------------------------------------

print("=" * 65)
print("1. VERIFYING PREPARED DATA")
print("=" * 65)

assert TRAIN_DIR.exists(), f"Missing: {TRAIN_DIR}"
assert VAL_DIR.exists(), f"Missing: {VAL_DIR}"

train_files = sorted(TRAIN_DIR.glob("*.npz"))
val_files = sorted(VAL_DIR.glob("*.npz"))

print("Training chunks:", len(train_files))
print("Validation chunks:", len(val_files))

assert train_files and val_files

# ------------------------------------------------
# 2. IDENTIFY SPEAKER CATEGORIES
# ------------------------------------------------

print("\n" + "=" * 65)
print("2. IDENTIFYING SPEAKERS")
print("=" * 65)

speaker_counts = Counter()

for file in train_files + val_files:
    with np.load(file, allow_pickle=True) as data:
        original_id = str(data["original_id"].item()).lower()

    if "female" in original_id:
        speaker_counts["female"] += 1
    elif "male" in original_id:
        speaker_counts["male"] += 1
    else:
        raise ValueError(
            f"Cannot identify speaker from original ID: {original_id}"
        )

print("Speaker chunk counts:", dict(speaker_counts))

assert set(speaker_counts) == {"female", "male"}

# ------------------------------------------------
# 3. SAVE TWO-SPEAKER CONFIGURATION
# ------------------------------------------------

speakers = {
    "female": 0,
    "male": 1
}

with open(META_DIR / "speakers.json", "w") as f:
    json.dump(speakers, f, indent=2)

model_config_path = CONFIG_DIR / "model.yaml"

with open(model_config_path, "r") as f:
    model_config = yaml.safe_load(f)

model_config["multi_speaker"] = True

with open(model_config_path, "w") as f:
    yaml.safe_dump(model_config, f, sort_keys=False)

print("\nSpeaker configuration saved:")
print(json.dumps(speakers, indent=2))
print("multi_speaker:", model_config["multi_speaker"])

# ------------------------------------------------
# 4. INSPECT EXACT REPOSITORY INTERFACES
# ------------------------------------------------

print("\n" + "=" * 65)
print("3. INSPECTING FASTSPEECH 2 SOURCE CODE")
print("=" * 65)

def show_class_methods(filepath, class_name):
    source_path = REPO / filepath
    source = source_path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name == class_name:
            print(f"\n{class_name} in {filepath}")

            for item in node.body:
                if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    print(
                        f"  {item.name}{ast.unparse(item.args)}"
                    )
            return

    print(f"Class {class_name} not found in {filepath}")


show_class_methods("model/fastspeech2.py", "FastSpeech2")
show_class_methods("model/loss.py", "FastSpeech2Loss")

# ------------------------------------------------
# 5. INSPECT ORIGINAL TRAINING IMPLEMENTATION
# ------------------------------------------------

print("\n" + "=" * 65)
print("4. ORIGINAL TRAINING IMPLEMENTATION REFERENCES")
print("=" * 65)

train_source = (REPO / "train.py").read_text(
    encoding="utf-8"
).splitlines()

keywords = [
    "FastSpeech2(",
    "FastSpeech2Loss(",
    "model(",
    "criterion(",
    "to_device(",
    "optimizer.step",
    "optimizer.zero_grad",
]

for keyword in keywords:
    matches = [
        (i + 1, line.strip())
        for i, line in enumerate(train_source)
        if keyword in line
    ]

    print(f"\n{keyword}")
    for line_number, line in matches[:8]:
        print(f"  Line {line_number}: {line}")

# ------------------------------------------------
# 6. VERIFY ONE CHUNK
# ------------------------------------------------

print("\n" + "=" * 65)
print("5. SAMPLE CHUNK VERIFICATION")
print("=" * 65)

sample = train_files[0]

with np.load(sample, allow_pickle=True) as data:
    original_id = str(data["original_id"].item()).lower()

    print("Sample:", sample.name)
    print("Original recording:", original_id)
    print("Text tokens:", len(data["text_ids"]))
    print("Mel shape:", data["mel"].shape)
    print("Pitch shape:", data["pitch"].shape)
    print("Energy shape:", data["energy"].shape)
    print("Duration sum:", int(data["durations"].sum()))

    speaker_id = (
        0 if "female" in original_id else
        1 if "male" in original_id else -1
    )

    print("Assigned speaker ID:", speaker_id)

print("\n" + "=" * 65)
print("STEP 75A COMPLETED")
print("Speaker conditioning configured.")
print("No model training started.")
print("=" * 65)

1. VERIFYING PREPARED DATA
Training chunks: 1520
Validation chunks: 301

2. IDENTIFYING SPEAKERS
Speaker chunk counts: {'female': 713, 'male': 1108}

Speaker configuration saved:
{
  "female": 0,
  "male": 1
}
multi_speaker: True

3. INSPECTING FASTSPEECH 2 SOURCE CODE

FastSpeech2 in model/fastspeech2.py
  __init__self, preprocess_config, model_config
  forwardself, speakers, texts, src_lens, max_src_len, mels=None, mel_lens=None, max_mel_len=None, p_targets=None, e_targets=None, d_targets=None, p_control=1.0, e_control=1.0, d_control=1.0

FastSpeech2Loss in model/loss.py
  __init__self, preprocess_config, model_config
  forwardself, inputs, predictions

4. ORIGINAL TRAINING IMPLEMENTATION REFERENCES

FastSpeech2(

FastSpeech2Loss(
  Line 44: Loss = FastSpeech2Loss(preprocess_config, model_config).to(device)

model(
  Line 41: model, optimizer = get_model(args, configs, device, train=True)
  Line 82: output = model(*(batch[2:]))

criterion(

to_device(
  Line 79: batch = to_device(bat

In [14]:
# ================================================================
# STEP 75B: FASTSPEECH 2 TRAINING INTERFACE INSPECTION
# ================================================================

import ast
from pathlib import Path

REPO = Path("/content/FastSpeech2")

def print_function(filepath, class_name, function_name):
    path = REPO / filepath
    tree = ast.parse(path.read_text(encoding="utf-8"))

    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef) and node.name == class_name:
            for method in node.body:
                if isinstance(method, ast.FunctionDef) and method.name == function_name:
                    print(f"\n{'='*70}")
                    print(f"{filepath} -> {class_name}.{function_name}")
                    print("="*70)
                    print(ast.get_source_segment(
                        path.read_text(encoding="utf-8"), method
                    ))
                    return

    print(f"NOT FOUND: {class_name}.{function_name}")


def print_top_function(filepath, function_name):
    path = REPO / filepath
    tree = ast.parse(path.read_text(encoding="utf-8"))

    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            print(f"\n{'='*70}")
            print(f"{filepath} -> {function_name}")
            print("="*70)
            print(ast.get_source_segment(
                path.read_text(encoding="utf-8"), node
            ))
            return


# Model initialization and forward
print_function("model/fastspeech2.py", "FastSpeech2", "__init__")
print_function("model/fastspeech2.py", "FastSpeech2", "forward")

# Loss calculation
print_function("model/loss.py", "FastSpeech2Loss", "forward")

# Original dataset and batch construction
print_function("dataset.py", "Dataset", "__getitem__")
print_function("dataset.py", "Dataset", "reprocess")
print_function("dataset.py", "Dataset", "collate_fn")

# Training model construction
print_top_function("utils/model.py", "get_model")

print("\nINSPECTION COMPLETE")


model/fastspeech2.py -> FastSpeech2.__init__
def __init__(self, preprocess_config, model_config):
        super(FastSpeech2, self).__init__()
        self.model_config = model_config

        self.encoder = Encoder(model_config)
        self.variance_adaptor = VarianceAdaptor(preprocess_config, model_config)
        self.decoder = Decoder(model_config)
        self.mel_linear = nn.Linear(
            model_config["transformer"]["decoder_hidden"],
            preprocess_config["preprocessing"]["mel"]["n_mel_channels"],
        )
        self.postnet = PostNet()

        self.speaker_emb = None
        if model_config["multi_speaker"]:
            with open(
                os.path.join(
                    preprocess_config["path"]["preprocessed_path"], "speakers.json"
                ),
                "r",
            ) as f:
                n_speaker = len(json.load(f))
            self.speaker_emb = nn.Embedding(
                n_speaker,
                model_config["transformer"

In [15]:
# ================================================================
# STEP 75B: FASTSPEECH 2 MODEL + LOSS + BACKWARD PASS TEST
# ================================================================

import sys
import json
import yaml
import numpy as np
import torch
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from model.fastspeech2 import FastSpeech2
from model.loss import FastSpeech2Loss

# ------------------------------------------------
# 1. DEVICE AND CONFIGURATION
# ------------------------------------------------

device = torch.device("cpu")

CONFIG_DIR = BASE / "Telugu/config"
META_DIR = BASE / "Telugu/fs2_prepared/model_metadata"
TRAIN_DIR = BASE / "Telugu/fs2_prepared/train"

with open(CONFIG_DIR / "preprocess.yaml") as f:
    preprocess_config = yaml.safe_load(f)

with open(CONFIG_DIR / "model.yaml") as f:
    model_config = yaml.safe_load(f)

with open(META_DIR / "speakers.json") as f:
    speaker_map = json.load(f)

print("Device:", device)
print("Speakers:", speaker_map)
print("Multi-speaker:", model_config["multi_speaker"])

# ------------------------------------------------
# 2. LOAD ONE PREPARED CHUNK
# ------------------------------------------------

files = sorted(TRAIN_DIR.glob("*.npz"))

assert len(files) > 0, "No training chunks found."

sample_path = files[0]

with np.load(sample_path, allow_pickle=True) as data:
    original_id = str(data["original_id"].item()).lower()

    mel = data["mel"].astype(np.float32).T
    pitch = data["pitch"].astype(np.float32)
    energy = data["energy"].astype(np.float32)
    text = data["text_ids"].astype(np.int64)
    duration = data["durations"].astype(np.int64)

if "female" in original_id:
    speaker_id = speaker_map["female"]
elif "male" in original_id:
    speaker_id = speaker_map["male"]
else:
    raise ValueError("Unknown speaker in sample.")

# ------------------------------------------------
# 3. CONVERT TO MODEL TENSORS
# ------------------------------------------------

speakers = torch.tensor([speaker_id], dtype=torch.long)
texts = torch.tensor(text[None, :], dtype=torch.long)
src_lens = torch.tensor([len(text)], dtype=torch.long)

mels = torch.tensor(mel[None, :, :], dtype=torch.float32)
mel_lens = torch.tensor([mel.shape[0]], dtype=torch.long)

p_targets = torch.tensor(pitch[None, :], dtype=torch.float32)
e_targets = torch.tensor(energy[None, :], dtype=torch.float32)
d_targets = torch.tensor(duration[None, :], dtype=torch.long)

max_src_len = len(text)
max_mel_len = mel.shape[0]

# Same 12-item structure used by the original repository.
batch = (
    [sample_path.stem],
    ["sample"],
    speakers,
    texts,
    src_lens,
    max_src_len,
    mels,
    mel_lens,
    max_mel_len,
    p_targets,
    e_targets,
    d_targets
)

print("\nSample information:")
print("Speaker ID:", speaker_id)
print("Text shape:", texts.shape)
print("Mel shape:", mels.shape)
print("Pitch shape:", p_targets.shape)
print("Energy shape:", e_targets.shape)
print("Duration shape:", d_targets.shape)

assert int(d_targets.sum()) == max_mel_len

# ------------------------------------------------
# 4. INITIALIZE MODEL AND LOSS
# ------------------------------------------------

print("\nInitializing FastSpeech 2...")

model = FastSpeech2(
    preprocess_config,
    model_config
).to(device)

criterion = FastSpeech2Loss(
    preprocess_config,
    model_config
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.0
)

# ------------------------------------------------
# 5. FORWARD PASS
# ------------------------------------------------

print("\nRunning forward pass...")

model.train()
optimizer.zero_grad()

predictions = model(*(batch[2:]))

print("Forward pass successful.")

for i, output in enumerate(predictions):
    if torch.is_tensor(output):
        print(
            f"Output {i}:",
            tuple(output.shape),
            "finite:",
            bool(torch.isfinite(output).all())
        )

# ------------------------------------------------
# 6. CALCULATE FASTSPEECH 2 LOSSES
# ------------------------------------------------

print("\nCalculating losses...")

losses = criterion(batch, predictions)

loss_names = [
    "total",
    "mel",
    "postnet_mel",
    "pitch",
    "energy",
    "duration"
]

for name, value in zip(loss_names, losses):
    print(f"{name}: {value.item():.6f}")

# ------------------------------------------------
# 7. BACKWARD PASS
# ------------------------------------------------

print("\nRunning backward pass...")

losses[0].backward()

gradient_count = sum(
    1 for parameter in model.parameters()
    if parameter.grad is not None
)

print("Parameters with gradients:", gradient_count)

# Verify gradients are finite.
bad_gradients = sum(
    1 for parameter in model.parameters()
    if parameter.grad is not None
    and not torch.isfinite(parameter.grad).all()
)

print("Non-finite gradients:", bad_gradients)

assert bad_gradients == 0
assert gradient_count > 0

optimizer.step()

print("\n" + "=" * 65)
print("STEP 75B COMPLETED SUCCESSFULLY")
print("Model forward pass: PASS")
print("Loss calculation: PASS")
print("Backward pass: PASS")
print("Optimizer update: PASS")
print("Full training has NOT started.")
print("=" * 65)

Device: cpu
Speakers: {'female': 0, 'male': 1}
Multi-speaker: True

Sample information:
Speaker ID: 0
Text shape: torch.Size([1, 210])
Mel shape: torch.Size([1, 1509, 80])
Pitch shape: torch.Size([1, 1509])
Energy shape: torch.Size([1, 1509])
Duration shape: torch.Size([1, 210])

Initializing FastSpeech 2...

Running forward pass...
Forward pass successful.
Output 0: (1, 1509, 80) finite: True
Output 1: (1, 1509, 80) finite: True
Output 2: (1, 1509) finite: True
Output 3: (1, 1509) finite: True
Output 4: (1, 210) finite: True
Output 5: (1, 210) finite: True
Output 6: (1, 210) finite: True
Output 7: (1, 1509) finite: True
Output 8: (1,) finite: True
Output 9: (1,) finite: True

Calculating losses...
total: 24655.566406
mel: 6.758092
postnet_mel: 6.780599
pitch: 24638.562500
energy: 0.356192
duration: 3.109322

Running backward pass...
Parameters with gradients: 120
Non-finite gradients: 0

STEP 75B COMPLETED SUCCESSFULLY
Model forward pass: PASS
Loss calculation: PASS
Backward pass: PAS

In [16]:
# ================================================================
# STEP 75C: INSPECT PITCH NORMALIZATION IN FASTSPEECH 2
# ================================================================

import ast
from pathlib import Path

REPO = Path("/content/FastSpeech2")

def inspect_class(filepath, class_name, methods=None):
    path = REPO / filepath
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef) and node.name == class_name:
            for method in node.body:
                if isinstance(method, ast.FunctionDef):
                    if methods is None or method.name in methods:
                        print(f"\n{'='*65}")
                        print(f"{class_name}.{method.name}")
                        print("="*65)
                        print(ast.get_source_segment(source, method))


inspect_class(
    "model/modules.py",
    "VarianceAdaptor",
    ["__init__", "forward"]
)

inspect_class(
    "model/loss.py",
    "FastSpeech2Loss",
    ["__init__", "forward"]
)

print("\nPITCH CONFIGURATION:")
import yaml

with open(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/config/preprocess.yaml"
) as f:
    config = yaml.safe_load(f)

print(config["preprocessing"]["pitch"])
print("\nSaved pitch statistics:")
import json

with open(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/fs2_prepared/model_metadata/stats.json"
) as f:
    stats = json.load(f)

print(stats["pitch"])


VarianceAdaptor.__init__
def __init__(self, preprocess_config, model_config):
        super(VarianceAdaptor, self).__init__()
        self.duration_predictor = VariancePredictor(model_config)
        self.length_regulator = LengthRegulator()
        self.pitch_predictor = VariancePredictor(model_config)
        self.energy_predictor = VariancePredictor(model_config)

        self.pitch_feature_level = preprocess_config["preprocessing"]["pitch"][
            "feature"
        ]
        self.energy_feature_level = preprocess_config["preprocessing"]["energy"][
            "feature"
        ]
        assert self.pitch_feature_level in ["phoneme_level", "frame_level"]
        assert self.energy_feature_level in ["phoneme_level", "frame_level"]

        pitch_quantization = model_config["variance_embedding"]["pitch_quantization"]
        energy_quantization = model_config["variance_embedding"]["energy_quantization"]
        n_bins = model_config["variance_embedding"]["n_bins"]
        asser

In [17]:
# ================================================================
# STEP 75D: CHECK PITCH / ENERGY EMBEDDING AND NORMALIZATION
# ================================================================

import ast
from pathlib import Path

REPO = Path("/content/FastSpeech2")

def inspect_methods(filepath, class_name, method_names):
    path = REPO / filepath
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef) and node.name == class_name:
            for method in node.body:
                if isinstance(method, ast.FunctionDef):
                    if method.name in method_names:
                        print(f"\n{'='*65}")
                        print(f"{class_name}.{method.name}")
                        print("="*65)
                        print(ast.get_source_segment(source, method))

inspect_methods(
    "model/fastspeech2.py",
    "VarianceAdaptor",
    ["get_pitch_embedding", "get_energy_embedding"]
)

# Find pitch normalization logic in preprocessing code
print("\n" + "="*65)
print("PREPROCESSING NORMALIZATION REFERENCES")
print("="*65)

for filename in ["preprocessor/preprocessor.py", "preprocess.py"]:
    path = REPO / filename
    if not path.exists():
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        if any(word in line.lower() for word in [
            "pitch", "normalization", "mean", "std"
        ]):
            print(f"{filename}:{i+1}: {line.strip()}")

print("\nInspection complete. No data or configuration changed.")


PREPROCESSING NORMALIZATION REFERENCES
preprocessor/preprocessor.py:25: assert config["preprocessing"]["pitch"]["feature"] in [
preprocessor/preprocessor.py:33: self.pitch_phoneme_averaging = (
preprocessor/preprocessor.py:34: config["preprocessing"]["pitch"]["feature"] == "phoneme_level"
preprocessor/preprocessor.py:40: self.pitch_normalization = config["preprocessing"]["pitch"]["normalization"]
preprocessor/preprocessor.py:41: self.energy_normalization = config["preprocessing"]["energy"]["normalization"]
preprocessor/preprocessor.py:55: os.makedirs((os.path.join(self.out_dir, "pitch")), exist_ok=True)
preprocessor/preprocessor.py:62: pitch_scaler = StandardScaler()
preprocessor/preprocessor.py:65: # Compute pitch, energy, duration, and mel-spectrogram
preprocessor/preprocessor.py:67: for i, speaker in enumerate(tqdm(os.listdir(self.in_dir))):
preprocessor/preprocessor.py:69: for wav_name in os.listdir(os.path.join(self.in_dir, speaker)):
preprocessor/preprocessor.py:82: info, pitch,

In [18]:
# ================================================================
# STEP 75E: LOCATE VARIANCE EMBEDDING IMPLEMENTATION
# ================================================================

import ast
from pathlib import Path

REPO = Path("/content/FastSpeech2")

for filename in [
    "model/fastspeech2.py",
    "model/modules.py"
]:
    path = REPO / filename
    source = path.read_text(encoding="utf-8")
    tree = ast.parse(source)

    print("\n" + "=" * 70)
    print("FILE:", filename)
    print("=" * 70)

    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            if node.name in ["VarianceAdaptor", "FastSpeech2"]:
                print("\nCLASS:", node.name)

                for method in node.body:
                    if isinstance(method, ast.FunctionDef):
                        if any(k in method.name for k in [
                            "pitch", "energy", "embedding"
                        ]):
                            print("\nMETHOD:", method.name)
                            print(ast.get_source_segment(source, method))

print("\nNo files or configurations were modified.")


FILE: model/fastspeech2.py

CLASS: FastSpeech2

FILE: model/modules.py

CLASS: VarianceAdaptor

METHOD: get_pitch_embedding
def get_pitch_embedding(self, x, target, mask, control):
        prediction = self.pitch_predictor(x, mask)
        if target is not None:
            embedding = self.pitch_embedding(torch.bucketize(target, self.pitch_bins))
        else:
            prediction = prediction * control
            embedding = self.pitch_embedding(
                torch.bucketize(prediction, self.pitch_bins)
            )
        return prediction, embedding

METHOD: get_energy_embedding
def get_energy_embedding(self, x, target, mask, control):
        prediction = self.energy_predictor(x, mask)
        if target is not None:
            embedding = self.energy_embedding(torch.bucketize(target, self.energy_bins))
        else:
            prediction = prediction * control
            embedding = self.energy_embedding(
                torch.bucketize(prediction, self.energy_bins)
  

In [19]:
# ================================================================
# STEP 75F: TRAINING-ONLY PITCH AND ENERGY NORMALIZATION
# ================================================================

import json
import yaml
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

TRAIN_DIR = BASE / "Telugu/fs2_prepared/train"
VAL_DIR = BASE / "Telugu/fs2_prepared/validation"
META_DIR = BASE / "Telugu/fs2_prepared/model_metadata"
CONFIG_DIR = BASE / "Telugu/config"

# ------------------------------------------------
# 1. COMPUTE STATISTICS USING TRAINING DATA ONLY
# ------------------------------------------------

pitch_sum = 0.0
pitch_sq_sum = 0.0
pitch_count = 0

energy_sum = 0.0
energy_sq_sum = 0.0
energy_count = 0

pitch_min = float("inf")
pitch_max = float("-inf")
energy_min = float("inf")
energy_max = float("-inf")

train_files = sorted(TRAIN_DIR.glob("*.npz"))

assert len(train_files) > 0

print("Computing training-only statistics...")

for i, path in enumerate(train_files, 1):

    with np.load(path, allow_pickle=True) as data:
        pitch = data["pitch"].astype(np.float64)
        energy = data["energy"].astype(np.float64)

    pitch_sum += pitch.sum()
    pitch_sq_sum += np.square(pitch).sum()
    pitch_count += len(pitch)

    energy_sum += energy.sum()
    energy_sq_sum += np.square(energy).sum()
    energy_count += len(energy)

    pitch_min = min(pitch_min, float(pitch.min()))
    pitch_max = max(pitch_max, float(pitch.max()))

    energy_min = min(energy_min, float(energy.min()))
    energy_max = max(energy_max, float(energy.max()))

    if i % 300 == 0:
        print(f"Processed {i}/{len(train_files)} chunks")

pitch_mean = pitch_sum / pitch_count
pitch_std = max(
    np.sqrt(max(0.0, pitch_sq_sum / pitch_count - pitch_mean**2)),
    1e-8
)

energy_mean = energy_sum / energy_count
energy_std = max(
    np.sqrt(max(0.0, energy_sq_sum / energy_count - energy_mean**2)),
    1e-8
)

# ------------------------------------------------
# 2. CALCULATE NORMALIZED BIN BOUNDARIES
# ------------------------------------------------

normalized_pitch_min = (pitch_min - pitch_mean) / pitch_std
normalized_pitch_max = (pitch_max - pitch_mean) / pitch_std

normalized_energy_min = (energy_min - energy_mean) / energy_std
normalized_energy_max = (energy_max - energy_mean) / energy_std

stats = {
    "pitch": [
        float(normalized_pitch_min),
        float(normalized_pitch_max),
        0.0,
        1.0
    ],
    "energy": [
        float(normalized_energy_min),
        float(normalized_energy_max),
        0.0,
        1.0
    ],
    "normalization_parameters": {
        "pitch_mean": float(pitch_mean),
        "pitch_std": float(pitch_std),
        "energy_mean": float(energy_mean),
        "energy_std": float(energy_std)
    }
}

# ------------------------------------------------
# 3. SAVE MODEL STATISTICS
# ------------------------------------------------

META_DIR.mkdir(parents=True, exist_ok=True)

stats_path = META_DIR / "stats.json"

with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)

# ------------------------------------------------
# 4. ENABLE NORMALIZATION IN CONFIG
# ------------------------------------------------

config_path = CONFIG_DIR / "preprocess.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config["preprocessing"]["pitch"]["normalization"] = True
config["preprocessing"]["energy"]["normalization"] = True

with open(config_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# ------------------------------------------------
# 5. VERIFY VALIDATION FITS TRAINING SCALE
# ------------------------------------------------

print("\nChecking validation values against training statistics...")

val_pitch_min = float("inf")
val_pitch_max = float("-inf")
val_energy_min = float("inf")
val_energy_max = float("-inf")

for path in sorted(VAL_DIR.glob("*.npz")):

    with np.load(path, allow_pickle=True) as data:
        p = data["pitch"]
        e = data["energy"]

    val_pitch_min = min(val_pitch_min, float(p.min()))
    val_pitch_max = max(val_pitch_max, float(p.max()))

    val_energy_min = min(val_energy_min, float(e.min()))
    val_energy_max = max(val_energy_max, float(e.max()))

print("\nNORMALIZATION PARAMETERS")
print(json.dumps(stats, indent=2))

print("\nSTANDARDIZED VALIDATION RANGE")
print(
    "Pitch:",
    (val_pitch_min - pitch_mean) / pitch_std,
    "to",
    (val_pitch_max - pitch_mean) / pitch_std
)
print(
    "Energy:",
    (val_energy_min - energy_mean) / energy_std,
    "to",
    (val_energy_max - energy_mean) / energy_std
)

print("\nSaved:", stats_path)
print("Updated:", config_path)

print("\nSTEP 75F COMPLETED")
print("Original NPZ features remain unchanged.")

Computing training-only statistics...
Processed 300/1520 chunks
Processed 600/1520 chunks
Processed 900/1520 chunks
Processed 1200/1520 chunks
Processed 1500/1520 chunks

Checking validation values against training statistics...

NORMALIZATION PARAMETERS
{
  "pitch": [
    -1.0625918313858391,
    4.057256970053094,
    0.0,
    1.0
  ],
  "energy": [
    -0.9477671838391939,
    8.58909065259882,
    0.0,
    1.0
  ],
  "normalization_parameters": {
    "pitch_mean": 109.76953432377077,
    "pitch_std": 103.30357441258384,
    "energy_mean": 0.05246112019682791,
    "energy_std": 0.05535232817865627
  }
}

STANDARDIZED VALIDATION RANGE
Pitch: -1.0625918313858391 to 3.8127070736761164
Energy: -0.9477671838391939 to 8.291943575724433

Saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/fs2_prepared/model_metadata/stats.json
Updated: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/config/preprocess.yaml

STEP 75F COMPLETED
Original NPZ features remain unchanged.


In [20]:
# ================================================================
# STEP 76A: TELUGU FASTSPEECH 2 TRAINING
# CPU TRAINING + VALIDATION + CHECKPOINTS + LOSS PLOTS
# ================================================================

import os
import sys
import json
import yaml
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path

# ------------------------------------------------
# 1. PATHS AND SETTINGS
# ------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

TRAIN_DIR = BASE / "Telugu/fs2_prepared/train"
VAL_DIR = BASE / "Telugu/fs2_prepared/validation"
META_DIR = BASE / "Telugu/fs2_prepared/model_metadata"

CKPT_DIR = BASE / "Telugu/checkpoints"
LOG_DIR = BASE / "Telugu/logs"
RESULT_DIR = BASE / "Telugu/results"

for folder in [CKPT_DIR, LOG_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MAX_STEPS = 300
VALIDATE_EVERY = 100
SAVE_EVERY = 100
VALIDATION_SAMPLES = 24
LEARNING_RATE = 1e-4
SEED = 42

RESUME = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_num_threads(2)

device = torch.device("cpu")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from model.fastspeech2 import FastSpeech2
from model.loss import FastSpeech2Loss

# ------------------------------------------------
# 2. LOAD CONFIGURATION
# ------------------------------------------------

with open(BASE / "Telugu/config/preprocess.yaml") as f:
    preprocess_config = yaml.safe_load(f)

with open(BASE / "Telugu/config/model.yaml") as f:
    model_config = yaml.safe_load(f)

with open(META_DIR / "speakers.json") as f:
    speaker_map = json.load(f)

with open(META_DIR / "stats.json") as f:
    stats = json.load(f)

norm = stats["normalization_parameters"]

pitch_mean = norm["pitch_mean"]
pitch_std = norm["pitch_std"]

energy_mean = norm["energy_mean"]
energy_std = norm["energy_std"]

print("Device:", device)
print("Speakers:", speaker_map)
print("Pitch normalization:", pitch_mean, pitch_std)
print("Energy normalization:", energy_mean, energy_std)

assert model_config["multi_speaker"] is True
assert preprocess_config["preprocessing"]["pitch"]["normalization"]
assert preprocess_config["preprocessing"]["energy"]["normalization"]

# ------------------------------------------------
# 3. LOAD FILE LISTS
# ------------------------------------------------

train_files = sorted(TRAIN_DIR.glob("*.npz"))
val_files = sorted(VAL_DIR.glob("*.npz"))

assert len(train_files) > 0
assert len(val_files) > 0

print("\nTraining chunks:", len(train_files))
print("Validation chunks:", len(val_files))

# ------------------------------------------------
# 4. CUSTOM SAMPLE LOADER
# ------------------------------------------------

def load_sample(path):

    with np.load(path, allow_pickle=True) as data:

        original_id = str(data["original_id"].item()).lower()

        mel = data["mel"].astype(np.float32).T
        pitch = data["pitch"].astype(np.float32)
        energy = data["energy"].astype(np.float32)

        text = data["text_ids"].astype(np.int64)
        duration = data["durations"].astype(np.int64)

    if "female" in original_id:
        speaker_id = speaker_map["female"]
    elif "male" in original_id:
        speaker_id = speaker_map["male"]
    else:
        raise ValueError(f"Unknown speaker: {original_id}")

    # Apply training-derived normalization.
    pitch = (pitch - pitch_mean) / pitch_std
    energy = (energy - energy_mean) / energy_std

    # Validate sample dimensions.
    assert mel.shape[0] == len(pitch)
    assert len(pitch) == len(energy)
    assert len(text) == len(duration)
    assert int(duration.sum()) == mel.shape[0]

    assert np.isfinite(mel).all()
    assert np.isfinite(pitch).all()
    assert np.isfinite(energy).all()

    return {
        "speaker": speaker_id,
        "text": text,
        "mel": mel,
        "pitch": pitch,
        "energy": energy,
        "duration": duration
    }


def make_batch(path):

    sample = load_sample(path)

    speakers = torch.tensor(
        [sample["speaker"]], dtype=torch.long
    )

    texts = torch.tensor(
        sample["text"][None, :], dtype=torch.long
    )

    src_lens = torch.tensor(
        [len(sample["text"])], dtype=torch.long
    )

    mels = torch.tensor(
        sample["mel"][None, :, :], dtype=torch.float32
    )

    mel_lens = torch.tensor(
        [sample["mel"].shape[0]], dtype=torch.long
    )

    p_targets = torch.tensor(
        sample["pitch"][None, :], dtype=torch.float32
    )

    e_targets = torch.tensor(
        sample["energy"][None, :], dtype=torch.float32
    )

    d_targets = torch.tensor(
        sample["duration"][None, :], dtype=torch.long
    )

    return (
        [path.stem],
        [""],
        speakers,
        texts,
        src_lens,
        len(sample["text"]),
        mels,
        mel_lens,
        sample["mel"].shape[0],
        p_targets,
        e_targets,
        d_targets
    )

# ------------------------------------------------
# 5. INITIALIZE MODEL AND OPTIMIZER
# ------------------------------------------------

print("\nInitializing model...")

model = FastSpeech2(
    preprocess_config,
    model_config
).to(device)

criterion = FastSpeech2Loss(
    preprocess_config,
    model_config
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.0
)

start_step = 0

history = {
    "step": [],
    "train_total": [],
    "train_mel": [],
    "train_postnet_mel": [],
    "train_pitch": [],
    "train_energy": [],
    "train_duration": [],
    "validation_total": [],
    "validation_mel": [],
    "validation_postnet_mel": [],
    "validation_pitch": [],
    "validation_energy": [],
    "validation_duration": []
}

history_path = LOG_DIR / "training_history.json"

# ------------------------------------------------
# 6. RESTORE CHECKPOINT IF AVAILABLE
# ------------------------------------------------

latest_ckpt = CKPT_DIR / "latest.pth.tar"

if RESUME and latest_ckpt.exists():

    print("\nRestoring previous checkpoint...")

    checkpoint = torch.load(
        latest_ckpt,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    start_step = checkpoint["step"]

    if history_path.exists():
        with open(history_path) as f:
            history = json.load(f)

    print("Resumed from step:", start_step)

else:
    print("Starting new training run.")

# ------------------------------------------------
# 7. VALIDATION FUNCTION
# ------------------------------------------------

def run_validation():

    model.eval()

    selected = random.sample(
        val_files,
        min(VALIDATION_SAMPLES, len(val_files))
    )

    values = []

    with torch.no_grad():

        for path in selected:

            batch = make_batch(path)

            predictions = model(*(batch[2:]))

            losses = criterion(batch, predictions)

            values.append([
                float(x.item()) for x in losses
            ])

    model.train()

    return np.mean(values, axis=0).tolist()

# ------------------------------------------------
# 8. CHECKPOINT FUNCTION
# ------------------------------------------------

def save_checkpoint(step):

    checkpoint = {
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "history": history,
        "speaker_map": speaker_map,
        "normalization": norm
    }

    torch.save(checkpoint, latest_ckpt)

    torch.save(
        checkpoint,
        CKPT_DIR / f"{step}.pth.tar"
    )

    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    print("Checkpoint saved:", latest_ckpt)

# ------------------------------------------------
# 9. TRAINING LOOP
# ------------------------------------------------

print("\n" + "=" * 65)
print("STARTING TELUGU FASTSPEECH 2 TRAINING")
print("=" * 65)

model.train()

for step in range(start_step + 1, MAX_STEPS + 1):

    path = random.choice(train_files)

    batch = make_batch(path)

    optimizer.zero_grad(set_to_none=True)

    predictions = model(*(batch[2:]))

    losses = criterion(batch, predictions)

    total_loss = losses[0]

    if not torch.isfinite(total_loss):
        raise RuntimeError(
            f"Non-finite training loss at step {step}"
        )

    total_loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    loss_values = [
        float(value.item()) for value in losses
    ]

    if step == 1 or step % 10 == 0:
        print(
            f"Step {step:4d}/{MAX_STEPS} | "
            f"Total: {loss_values[0]:.4f} | "
            f"Mel: {loss_values[1]:.4f} | "
            f"Postnet: {loss_values[2]:.4f} | "
            f"Pitch: {loss_values[3]:.4f} | "
            f"Energy: {loss_values[4]:.4f} | "
            f"Duration: {loss_values[5]:.4f}"
        )

    # Validation
    if step % VALIDATE_EVERY == 0:

        avg_train = np.mean(
            [
                # Recent displayed training losses are approximated
                # by this step's loss; validation is independently averaged.
                loss_values
            ],
            axis=0
        )

        val_values = run_validation()

        history["step"].append(step)

        train_names = [
            "train_total",
            "train_mel",
            "train_postnet_mel",
            "train_pitch",
            "train_energy",
            "train_duration"
        ]

        val_names = [
            "validation_total",
            "validation_mel",
            "validation_postnet_mel",
            "validation_pitch",
            "validation_energy",
            "validation_duration"
        ]

        for name, value in zip(train_names, avg_train):
            history[name].append(float(value))

        for name, value in zip(val_names, val_values):
            history[name].append(float(value))

        print("\nVALIDATION AT STEP", step)

        for name, value in zip(val_names, val_values):
            print(f"{name}: {value:.5f}")

        save_checkpoint(step)

# ------------------------------------------------
# 10. SAVE FINAL MODEL AND HISTORY
# ------------------------------------------------

save_checkpoint(MAX_STEPS)

# ------------------------------------------------
# 11. PLOT TRAINING AND VALIDATION LOSSES
# ------------------------------------------------

if history["step"]:

    plt.figure(figsize=(10, 6))

    plt.plot(
        history["step"],
        history["train_total"],
        marker="o",
        label="Training loss"
    )

    plt.plot(
        history["step"],
        history["validation_total"],
        marker="s",
        label="Validation loss"
    )

    plt.xlabel("Training step")
    plt.ylabel("Total loss")
    plt.title("Telugu FastSpeech 2 Training and Validation Loss")

    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(
        RESULT_DIR / "training_validation_loss.png",
        dpi=200
    )

    plt.show()

    # Individual loss components
    components = [
        ("mel", "Mel loss"),
        ("postnet_mel", "Postnet mel loss"),
        ("pitch", "Pitch loss"),
        ("energy", "Energy loss"),
        ("duration", "Duration loss")
    ]

    for key, label in components:

        plt.figure(figsize=(8, 5))

        plt.plot(
            history["step"],
            history[f"train_{key}"],
            marker="o",
            label="Training"
        )

        plt.plot(
            history["step"],
            history[f"validation_{key}"],
            marker="s",
            label="Validation"
        )

        plt.xlabel("Training step")
        plt.ylabel(label)
        plt.title(f"Telugu FastSpeech 2 - {label}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()

        plt.savefig(
            RESULT_DIR / f"{key}_loss.png",
            dpi=200
        )

        plt.show()

print("\n" + "=" * 65)
print("TELUGU TRAINING RUN FINISHED")
print("Checkpoint:", latest_ckpt)
print("History:", history_path)
print("Loss plots:", RESULT_DIR)
print("=" * 65)

Device: cpu
Speakers: {'female': 0, 'male': 1}
Pitch normalization: 109.76953432377077 103.30357441258384
Energy normalization: 0.05246112019682791 0.05535232817865627

Training chunks: 1520
Validation chunks: 301

Initializing model...
Starting new training run.

STARTING TELUGU FASTSPEECH 2 TRAINING
Step    1/300 | Total: 18.7719 | Mel: 6.3286 | Postnet: 6.3581 | Pitch: 1.3142 | Energy: 1.4115 | Duration: 3.3596
Step   10/300 | Total: 14.3176 | Mel: 5.8367 | Postnet: 5.8492 | Pitch: 1.2705 | Energy: 0.7624 | Duration: 0.5988
Step   20/300 | Total: 13.6403 | Mel: 5.9822 | Postnet: 5.9796 | Pitch: 0.8057 | Energy: 0.5237 | Duration: 0.3491
Step   30/300 | Total: 10.9675 | Mel: 4.8085 | Postnet: 4.8135 | Pitch: 0.6064 | Energy: 0.4512 | Duration: 0.2878
Step   40/300 | Total: 12.4165 | Mel: 5.0600 | Postnet: 5.0580 | Pitch: 0.6230 | Energy: 1.3268 | Duration: 0.3486
Step   50/300 | Total: 12.6018 | Mel: 5.4430 | Postnet: 5.4185 | Pitch: 0.8477 | Energy: 0.4921 | Duration: 0.4004
Step   

In [21]:
from pathlib import Path
import json
import torch
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

checkpoint = BASE / "Telugu/checkpoints/latest.pth.tar"
history = BASE / "Telugu/logs/training_history.json"
results = BASE / "Telugu/results"

print("CHECKPOINT:", checkpoint.exists())
print("HISTORY:", history.exists())

if checkpoint.exists():
    print("Checkpoint size:", round(checkpoint.stat().st_size / 1024**2, 2), "MB")

if history.exists():
    with open(history) as f:
        data = json.load(f)

    print("\nHistory structure:", type(data).__name__)

    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, list):
                print(key, "entries:", len(value))
            else:
                print(key, ":", value)

print("\nRESULT FILES:")
for p in sorted(results.rglob("*")):
    if p.is_file():
        print(p.relative_to(BASE))

print("\nVERIFICATION COMPLETE")

CHECKPOINT: True
HISTORY: True
Checkpoint size: 89.81 MB

History structure: dict
step entries: 3
train_total entries: 3
train_mel entries: 3
train_postnet_mel entries: 3
train_pitch entries: 3
train_energy entries: 3
train_duration entries: 3
validation_total entries: 3
validation_mel entries: 3
validation_postnet_mel entries: 3
validation_pitch entries: 3
validation_energy entries: 3
validation_duration entries: 3

RESULT FILES:
Telugu/results/duration_loss.png
Telugu/results/energy_loss.png
Telugu/results/mel_loss.png
Telugu/results/pitch_loss.png
Telugu/results/postnet_mel_loss.png
Telugu/results/training_validation_loss.png

VERIFICATION COMPLETE


In [22]:
# ============================================================
# STEP 78A: LOAD TELUGU FASTSPEECH 2 + CHECK TEST TOKENIZATION
# ============================================================

import os
import sys
import json
import yaml
import torch
import torchaudio
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

sys.path.insert(0, str(REPO))
os.chdir(REPO)

DEVICE = torch.device("cpu")

# ------------------------------------------------------------
# 1. Load configurations
# ------------------------------------------------------------

config_dir = BASE / "Telugu/config"

with open(config_dir / "preprocess.yaml") as f:
    preprocess_config = yaml.safe_load(f)

with open(config_dir / "model.yaml") as f:
    model_config = yaml.safe_load(f)

with open(config_dir / "train.yaml") as f:
    train_config = yaml.safe_load(f)

metadata_dir = BASE / "Telugu/fs2_prepared/model_metadata"

with open(metadata_dir / "stats.json") as f:
    stats = json.load(f)

with open(metadata_dir / "speakers.json") as f:
    speakers = json.load(f)

print("Configurations loaded.")
print("Speakers:", speakers)

# ------------------------------------------------------------
# 2. Initialize FastSpeech 2 model
# ------------------------------------------------------------

from model.fastspeech2 import FastSpeech2

model = FastSpeech2(
    preprocess_config,
    model_config
).to(DEVICE)

checkpoint_path = BASE / "Telugu/checkpoints/latest.pth.tar"

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

print("\nCheckpoint keys:", list(checkpoint.keys()))

# Support common checkpoint formats
if "model" in checkpoint:
    model_state = checkpoint["model"]
elif "model_state_dict" in checkpoint:
    model_state = checkpoint["model_state_dict"]
else:
    raise KeyError("Could not find model weights in checkpoint.")

model.load_state_dict(model_state)
model.eval()

print("\nFastSpeech 2 checkpoint loaded successfully.")

# ------------------------------------------------------------
# 3. Check MMS_FA tokenizer
# ------------------------------------------------------------

bundle = torchaudio.pipelines.MMS_FA
tokenizer = bundle.get_tokenizer()

print("\nMMS_FA tokenizer:", type(tokenizer))
print("Tokenizer callable:", callable(tokenizer))
print("Tokenizer methods:", [
    name for name in dir(tokenizer)
    if not name.startswith("_")
][:40])

# ------------------------------------------------------------
# 4. Telugu test sentences
# ------------------------------------------------------------

test_sentences = {
    "T1": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
    "T2a": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
    "T2b": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
    "T3": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
    "T4": "ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి."
}

print("\nFive test sentences loaded.")

for key, sentence in test_sentences.items():
    print(f"\n{key}: {sentence}")

# ------------------------------------------------------------
# 5. Inspect tokenizer output for one sentence
# ------------------------------------------------------------

sample_text = test_sentences["T2a"]

try:
    token_output = tokenizer(sample_text)
    print("\nTokenizer output:")
    print(token_output)
except Exception as e:
    print("\nTokenizer direct call failed:")
    print(type(e).__name__, str(e))

print("\nSTEP 78A COMPLETE")

Configurations loaded.
Speakers: {'female': 0, 'male': 1}

Checkpoint keys: ['step', 'model', 'optimizer', 'history', 'speaker_map', 'normalization']

FastSpeech 2 checkpoint loaded successfully.

MMS_FA tokenizer: <class 'torchaudio.pipelines._wav2vec2.aligner.Tokenizer'>
Tokenizer callable: True
Tokenizer methods: ['dictionary']

Five test sentences loaded.

T1: న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.

T2a: తాడుపై అతని పట్టు చాలా బలంగా ఉంది.

T2b: ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.

T3: పులి అడవిలో నిశ్శబ్దంగా నడిచింది.

T4: ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి.

Tokenizer direct call failed:
KeyError 'త'

STEP 78A COMPLETE


In [23]:
# ============================================================
# STEP 78B: TELUGU INFERENCE SMOKE TEST
# ============================================================

import numpy as np
import torch
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
VAL_DIR = BASE / "Telugu/fs2_prepared/validation"
OUT_DIR = BASE / "Telugu/inference_smoke_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load one prepared validation sample
files = sorted(VAL_DIR.glob("*.npz"))
assert files, "No prepared validation chunks found."

sample_path = files[0]

with np.load(sample_path, allow_pickle=True) as d:
    text_ids = torch.tensor(d["text_ids"].astype(np.int64), dtype=torch.long)
    speaker_name = str(d["speaker"].item()) if "speaker" in d else None
    original_id = str(d["original_id"].item()) if "original_id" in d else ""

# Derive speaker from filename if metadata is not present
filename = sample_path.name.lower()
speaker_id = 0 if "female" in filename else 1

texts = text_ids.unsqueeze(0).to(DEVICE)
src_lens = torch.tensor([texts.shape[1]], device=DEVICE)
speakers_tensor = torch.tensor([speaker_id], device=DEVICE)

print("Sample:", sample_path.name)
print("Text tokens:", texts.shape[1])
print("Speaker ID:", speaker_id)

# Run model without teacher-forced duration/pitch/energy
with torch.no_grad():
    predictions = model(
        speakers_tensor,
        texts,
        src_lens,
        texts.shape[1]
    )

# Extract predictions
mel_pred = predictions[1][0].cpu().numpy()
pitch_pred = predictions[2][0].cpu().numpy()
energy_pred = predictions[3][0].cpu().numpy()
duration_pred = predictions[4][0].cpu().numpy()

print("\nPREDICTION SHAPES")
print("Mel:", mel_pred.shape)
print("Pitch:", pitch_pred.shape)
print("Energy:", energy_pred.shape)
print("Duration:", duration_pred.shape)

# Check outputs
for name, arr in {
    "mel": mel_pred,
    "pitch": pitch_pred,
    "energy": energy_pred,
    "duration": duration_pred
}.items():
    assert np.isfinite(arr).all(), f"Non-finite values in {name}"

# Save raw prediction arrays
np.savez_compressed(
    OUT_DIR / "smoke_test_predictions.npz",
    mel=mel_pred,
    pitch=pitch_pred,
    energy=energy_pred,
    duration=duration_pred
)

print("\nSaved predictions:", OUT_DIR / "smoke_test_predictions.npz")
print("STEP 78B COMPLETE")

Sample: train_telugufemale_00060_chunk000.npz
Text tokens: 66
Speaker ID: 0

PREDICTION SHAPES
Mel: (449, 80)
Pitch: (449,)
Energy: (449,)
Duration: (66,)

Saved predictions: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference_smoke_test/smoke_test_predictions.npz
STEP 78B COMPLETE


In [24]:
# ============================================================
# STEP 78C: PLOT TELUGU SMOKE-TEST PREDICTIONS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
OUT_DIR = BASE / "Telugu/inference_smoke_test"

data = np.load(OUT_DIR / "smoke_test_predictions.npz")

duration = data["duration"]
pitch = data["pitch"]
energy = data["energy"]

plots = {
    "duration": duration,
    "pitch": pitch,
    "energy": energy
}

for name, values in plots.items():
    plt.figure(figsize=(12, 4))
    plt.plot(values)
    plt.title(f"Telugu FastSpeech 2 - Predicted {name.capitalize()}")
    plt.xlabel("Index")
    plt.ylabel(name.capitalize())
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}_smoke_test.png", dpi=200)
    plt.show()
    plt.close()

print("Saved preliminary variance-adaptor plots:")
for name in plots:
    print(OUT_DIR / f"{name}_smoke_test.png")

print("STEP 78C COMPLETE")

Saved preliminary variance-adaptor plots:
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference_smoke_test/duration_smoke_test.png
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference_smoke_test/pitch_smoke_test.png
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference_smoke_test/energy_smoke_test.png
STEP 78C COMPLETE


In [25]:
# ============================================================
# STEP 78D: PREPARE FINAL TELUGU RESULTS DIRECTORIES
# ============================================================

from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
TELUGU = BASE / "Telugu"

folders = [
    TELUGU / "results/test_generated_audios",
    TELUGU / "results/variance_adaptor_plots/duration",
    TELUGU / "results/variance_adaptor_plots/pitch",
    TELUGU / "results/variance_adaptor_plots/energy",
    TELUGU / "inference/test_predictions",
    TELUGU / "inference/logs",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print("READY:", folder)

print("\nSTEP 78D COMPLETE")

READY: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/test_generated_audios
READY: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/variance_adaptor_plots/duration
READY: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/variance_adaptor_plots/pitch
READY: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/variance_adaptor_plots/energy
READY: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions
READY: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference/logs

STEP 78D COMPLETE


In [26]:
# ============================================================
# STEP 78E: SAVE TELUGU TEST SENTENCES
# ============================================================

import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
OUT = BASE / "Telugu/inference/test_sentences.json"

test_sentences = {
    "T1": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
    "T2a": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
    "T2b": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
    "T3": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
    "T4": "ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి."
}

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(test_sentences, f, ensure_ascii=False, indent=2)

print("Saved:", OUT)

for key, sentence in test_sentences.items():
    print(f"{key}: {sentence}")

print("\nSTEP 78E COMPLETE")

Saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_sentences.json
T1: న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.
T2a: తాడుపై అతని పట్టు చాలా బలంగా ఉంది.
T2b: ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.
T3: పులి అడవిలో నిశ్శబ్దంగా నడిచింది.
T4: ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి.

STEP 78E COMPLETE


In [27]:
# ============================================================
# STEP 78F: SAVE TELUGU TRAINING SUMMARY
# ============================================================

import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
HISTORY = BASE / "Telugu/logs/training_history.json"
OUT = BASE / "Telugu/inference/logs/training_summary.json"

with open(HISTORY, "r") as f:
    history = json.load(f)

summary = {
    "language": "Telugu",
    "architecture": "FastSpeech 2",
    "device": "CPU",
    "completed_steps": history["step"][-1],
    "validation_records": len(history["step"]),
    "initial_validation_loss": history["validation_total"][0],
    "final_validation_loss": history["validation_total"][-1],
    "final_validation_components": {
        "mel": history["validation_mel"][-1],
        "postnet_mel": history["validation_postnet_mel"][-1],
        "pitch": history["validation_pitch"][-1],
        "energy": history["validation_energy"][-1],
        "duration": history["validation_duration"][-1]
    },
    "checkpoint": str(BASE / "Telugu/checkpoints/latest.pth.tar")
}

with open(OUT, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nSaved:", OUT)
print("STEP 78F COMPLETE")

{
  "language": "Telugu",
  "architecture": "FastSpeech 2",
  "device": "CPU",
  "completed_steps": 300,
  "validation_records": 3,
  "initial_validation_loss": 9.893165151278177,
  "final_validation_loss": 6.9097413420677185,
  "final_validation_components": {
    "mel": 2.734986980756124,
    "postnet_mel": 3.114191840092341,
    "pitch": 0.5171841631333033,
    "energy": 0.4133523106575012,
    "duration": 0.13002612115815282
  },
  "checkpoint": "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/latest.pth.tar"
}

Saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference/logs/training_summary.json
STEP 78F COMPLETE


In [28]:
# ============================================================
# STEP 78G: INSPECT AVAILABLE VOCODER CHECKPOINTS
# ============================================================

from pathlib import Path

REPO = Path("/content/FastSpeech2")
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

search_roots = [
    REPO / "hifigan",
    REPO / "checkpoints",
    BASE / "Telugu",
    BASE / "English"
]

extensions = {".pth", ".pt", ".ckpt", ".tar"}

found = []

for root in search_roots:
    if root.exists():
        for file in root.rglob("*"):
            if file.is_file() and file.suffix.lower() in extensions:
                found.append(file)

print("VOCODER / MODEL CHECKPOINT FILES")
print("=" * 55)

for file in found:
    print(file, "|", round(file.stat().st_size / 1024**2, 2), "MB")

if not found:
    print("No checkpoint files found in the inspected locations.")

print("\nHiFi-GAN directory exists:", (REPO / "hifigan").exists())
print("STEP 78G COMPLETE")

VOCODER / MODEL CHECKPOINT FILES
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/latest.pth.tar | 89.81 MB
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/100.pth.tar | 89.8 MB
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/200.pth.tar | 89.8 MB
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/300.pth.tar | 89.8 MB

HiFi-GAN directory exists: True
STEP 78G COMPLETE


In [29]:
# ============================================================
# STEP 79: INSPECT HIFI-GAN RESOURCES
# ============================================================

from pathlib import Path

REPO = Path("/content/FastSpeech2")
HIFI = REPO / "hifigan"

print("HIFI-GAN DIRECTORY STRUCTURE")
print("=" * 60)

for path in sorted(HIFI.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO))

print("\nPOTENTIAL VOCODER CHECKPOINTS")
print("=" * 60)

for ext in ["*.pth", "*.pt", "*.ckpt", "*.tar"]:
    matches = list(HIFI.rglob(ext))
    for file in matches:
        print(file, round(file.stat().st_size / 1024**2, 2), "MB")

print("\nSTEP 79 COMPLETE")

HIFI-GAN DIRECTORY STRUCTURE
hifigan/LICENSE
hifigan/__init__.py
hifigan/config.json
hifigan/generator_LJSpeech.pth.tar.zip
hifigan/generator_universal.pth.tar.zip
hifigan/models.py

POTENTIAL VOCODER CHECKPOINTS

STEP 79 COMPLETE


In [30]:
# ============================================================
# STEP 80: EXTRACT AND INSPECT HIFI-GAN CHECKPOINTS
# ============================================================

import zipfile
import json
from pathlib import Path

REPO = Path("/content/FastSpeech2")
HIFI = REPO / "hifigan"

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
OUT = BASE / "Telugu/vocoder"

OUT.mkdir(parents=True, exist_ok=True)

archives = [
    HIFI / "generator_LJSpeech.pth.tar.zip",
    HIFI / "generator_universal.pth.tar.zip"
]

for archive in archives:
    print("\n" + "=" * 60)
    print("ARCHIVE:", archive.name)

    if not archive.exists():
        print("Archive missing!")
        continue

    with zipfile.ZipFile(archive, "r") as z:
        print("Contents:")
        for item in z.infolist():
            print(item.filename, round(item.file_size / 1024**2, 2), "MB")

        # Extract into separate folders
        name = archive.stem.replace(".pth.tar", "")
        destination = OUT / name
        destination.mkdir(parents=True, exist_ok=True)

        z.extractall(destination)
        print("Extracted to:", destination)

# Inspect HiFi-GAN configuration
config_path = HIFI / "config.json"

with open(config_path) as f:
    hifi_config = json.load(f)

print("\nHIFI-GAN CONFIGURATION")
print("=" * 60)

for key in [
    "sampling_rate",
    "num_mels",
    "n_fft",
    "hop_size",
    "win_size",
    "fmin",
    "fmax"
]:
    print(key, ":", hifi_config.get(key))

print("\nEXTRACTED CHECKPOINTS")
for file in OUT.rglob("*.pth*"):
    print(file, round(file.stat().st_size / 1024**2, 2), "MB")

print("\nSTEP 80 COMPLETE")


ARCHIVE: generator_LJSpeech.pth.tar.zip
Contents:
generator_LJSpeech.pth.tar 53.2 MB
Extracted to: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/vocoder/generator_LJSpeech

ARCHIVE: generator_universal.pth.tar.zip
Contents:
generator_universal.pth.tar 53.2 MB
Extracted to: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/vocoder/generator_universal

HIFI-GAN CONFIGURATION
sampling_rate : 22050
num_mels : 80
n_fft : 1024
hop_size : 256
win_size : 1024
fmin : 0
fmax : 8000

EXTRACTED CHECKPOINTS
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/vocoder/generator_LJSpeech/generator_LJSpeech.pth.tar 53.2 MB
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/vocoder/generator_universal/generator_universal.pth.tar 53.2 MB

STEP 80 COMPLETE


In [31]:
# ============================================================
# STEP 81: HIFI-GAN CHECKPOINT COMPATIBILITY TEST
# ============================================================

import sys
import json
import torch
from pathlib import Path

REPO = Path("/content/FastSpeech2")
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
HIFI = REPO / "hifigan"

sys.path.insert(0, str(REPO))

from hifigan.models import Generator

DEVICE = torch.device("cpu")

with open(HIFI / "config.json") as f:
    hifi_config = json.load(f)

checkpoint_paths = {
    "universal": BASE / "Telugu/vocoder/generator_universal/generator_universal.pth.tar",
    "ljspeech": BASE / "Telugu/vocoder/generator_LJSpeech/generator_LJSpeech.pth.tar"
}

results = {}

for name, checkpoint_path in checkpoint_paths.items():

    print("\n" + "=" * 65)
    print("TESTING:", name.upper())
    print("=" * 65)

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False
        )

        print("Checkpoint type:", type(checkpoint))

        if isinstance(checkpoint, dict):
            print("Checkpoint keys:", list(checkpoint.keys())[:20])

        # Identify generator weights
        if isinstance(checkpoint, dict) and "generator" in checkpoint:
            state_dict = checkpoint["generator"]
        elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif isinstance(checkpoint, dict) and "model" in checkpoint:
            state_dict = checkpoint["model"]
        else:
            state_dict = checkpoint

        generator = Generator(hifi_config).to(DEVICE)

        load_result = generator.load_state_dict(
            state_dict,
            strict=True
        )

        generator.eval()

        print("Checkpoint loaded successfully.")
        print("Load result:", load_result)

        # Dummy mel: [batch, mel_channels, frames]
        dummy_mel = torch.randn(1, 80, 20, device=DEVICE)

        with torch.no_grad():
            waveform = generator(dummy_mel)

        waveform = waveform.squeeze().cpu()

        assert torch.isfinite(waveform).all()

        print("Input mel shape:", tuple(dummy_mel.shape))
        print("Generated waveform shape:", tuple(waveform.shape))
        print("Waveform duration:", round(len(waveform) / 22050, 3), "seconds")
        print("Amplitude range:", waveform.min().item(), waveform.max().item())

        results[name] = {
            "success": True,
            "waveform_samples": len(waveform),
            "duration_seconds": len(waveform) / 22050
        }

        print("RESULT: SUCCESS")

        del generator, checkpoint, state_dict, waveform

    except Exception as e:
        results[name] = {
            "success": False,
            "error": str(e)
        }

        print("RESULT: FAILED")
        print(type(e).__name__, str(e)[:1000])

print("\n" + "=" * 65)
print("FINAL VOCODER TEST RESULTS")
print("=" * 65)

for name, result in results.items():
    print(name, result)

output = BASE / "Telugu/inference/logs/vocoder_compatibility.json"
output.parent.mkdir(parents=True, exist_ok=True)

with open(output, "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved:", output)
print("STEP 81 COMPLETE")


TESTING: UNIVERSAL
Checkpoint type: <class 'dict'>
Checkpoint keys: ['generator']
RESULT: FAILED
AttributeError 'dict' object has no attribute 'resblock_kernel_sizes'

TESTING: LJSPEECH
Checkpoint type: <class 'dict'>
Checkpoint keys: ['generator']
RESULT: FAILED
AttributeError 'dict' object has no attribute 'resblock_kernel_sizes'

FINAL VOCODER TEST RESULTS
universal {'success': False, 'error': "'dict' object has no attribute 'resblock_kernel_sizes'"}
ljspeech {'success': False, 'error': "'dict' object has no attribute 'resblock_kernel_sizes'"}

Saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference/logs/vocoder_compatibility.json
STEP 81 COMPLETE


In [32]:
# STEP 81B: FRESH HIFI-GAN CONFIGURATION TEST

import json
import torch
import sys
from pathlib import Path
from types import SimpleNamespace

REPO = Path("/content/FastSpeech2")
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

sys.path.insert(0, str(REPO))

from hifigan.models import Generator

# Load config
with open(REPO / "hifigan/config.json") as f:
    config_dict = json.load(f)

print("CONFIG TYPE:", type(config_dict))
print("CONFIG KEYS:", list(config_dict.keys()))

# Convert dictionary to attribute-based object
h = SimpleNamespace(**config_dict)

print("CONFIG OBJECT:", type(h))
print("RESBLOCK KERNEL SIZES:", h.resblock_kernel_sizes)

# Initialize generator
generator = Generator(h).cpu()

print("\nGENERATOR INITIALIZED SUCCESSFULLY")

# Inspect universal checkpoint
checkpoint_path = (
    BASE / "Telugu/vocoder/generator_universal/"
    "generator_universal.pth.tar"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCHECKPOINT KEYS:", list(checkpoint.keys()))

state_dict = checkpoint["generator"]

print("WEIGHT COUNT:", len(state_dict))

# Load weights
generator.load_state_dict(state_dict, strict=True)
generator.eval()

print("\nUNIVERSAL CHECKPOINT LOADED SUCCESSFULLY")

# Test waveform generation
dummy_mel = torch.randn(1, 80, 20)

with torch.no_grad():
    waveform = generator(dummy_mel)

waveform = waveform.squeeze()

print("\nWAVEFORM SHAPE:", tuple(waveform.shape))
print("WAVEFORM DURATION:", waveform.numel() / 22050)
print("FINITE:", torch.isfinite(waveform).all().item())

print("\nSTEP 81B COMPLETE")

CONFIG TYPE: <class 'dict'>
CONFIG KEYS: ['resblock', 'num_gpus', 'batch_size', 'learning_rate', 'adam_b1', 'adam_b2', 'lr_decay', 'seed', 'upsample_rates', 'upsample_kernel_sizes', 'upsample_initial_channel', 'resblock_kernel_sizes', 'resblock_dilation_sizes', 'segment_size', 'num_mels', 'num_freq', 'n_fft', 'hop_size', 'win_size', 'sampling_rate', 'fmin', 'fmax', 'fmax_for_loss', 'num_workers', 'dist_config']
CONFIG OBJECT: <class 'types.SimpleNamespace'>
RESBLOCK KERNEL SIZES: [3, 7, 11]


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



GENERATOR INITIALIZED SUCCESSFULLY

CHECKPOINT KEYS: ['generator']
WEIGHT COUNT: 234

UNIVERSAL CHECKPOINT LOADED SUCCESSFULLY

WAVEFORM SHAPE: (5120,)
WAVEFORM DURATION: 0.23219954648526078
FINITE: True

STEP 81B COMPLETE


In [33]:
# ============================================================
# STEP 82: END-TO-END TELUGU AUDIO GENERATION TEST
# ============================================================

import sys
import gc
import json
import torch
import numpy as np
import torchaudio

from pathlib import Path
from types import SimpleNamespace

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

sys.path.insert(0, str(REPO))

from hifigan.models import Generator

DEVICE = torch.device("cpu")

# ------------------------------------------------------------
# 1. Load FastSpeech 2 if not already available
# ------------------------------------------------------------

if "model" not in globals():
    import yaml
    from model.fastspeech2 import FastSpeech2

    with open(BASE / "Telugu/config/preprocess.yaml") as f:
        preprocess_config = yaml.safe_load(f)

    with open(BASE / "Telugu/config/model.yaml") as f:
        model_config = yaml.safe_load(f)

    model = FastSpeech2(preprocess_config, model_config).to(DEVICE)

    fs2_checkpoint = torch.load(
        BASE / "Telugu/checkpoints/latest.pth.tar",
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(fs2_checkpoint["model"])
    model.eval()

model.eval()
print("FastSpeech 2 ready.")

# ------------------------------------------------------------
# 2. Load one existing validation chunk
# ------------------------------------------------------------

val_dir = BASE / "Telugu/fs2_prepared/validation"
sample_path = sorted(val_dir.glob("*.npz"))[0]

with np.load(sample_path, allow_pickle=True) as d:
    text_ids = torch.tensor(
        d["text_ids"].astype(np.int64),
        dtype=torch.long
    )

filename = sample_path.name.lower()
speaker_id = 0 if "female" in filename else 1

texts = text_ids.unsqueeze(0).to(DEVICE)
src_lens = torch.tensor([texts.shape[1]], device=DEVICE)
speakers = torch.tensor([speaker_id], device=DEVICE)

print("Validation sample:", sample_path.name)
print("Text tokens:", texts.shape[1])
print("Speaker ID:", speaker_id)

# ------------------------------------------------------------
# 3. Generate mel spectrogram
# ------------------------------------------------------------

with torch.no_grad():
    predictions = model(
        speakers,
        texts,
        src_lens,
        texts.shape[1]
    )

mel_prediction = predictions[1]

print("Predicted mel shape:", tuple(mel_prediction.shape))

# ------------------------------------------------------------
# 4. Load universal HiFi-GAN
# ------------------------------------------------------------

hifi_dir = REPO / "hifigan"

with open(hifi_dir / "config.json") as f:
    hifi_config = SimpleNamespace(**json.load(f))

vocoder_path = (
    BASE / "Telugu/vocoder/generator_universal/"
    "generator_universal.pth.tar"
)

vocoder_checkpoint = torch.load(
    vocoder_path,
    map_location=DEVICE,
    weights_only=False
)

vocoder = Generator(hifi_config).to(DEVICE)
vocoder.load_state_dict(
    vocoder_checkpoint["generator"],
    strict=True
)
vocoder.eval()

print("Universal HiFi-GAN loaded.")

# ------------------------------------------------------------
# 5. Convert mel to waveform
# ------------------------------------------------------------

mel_input = mel_prediction.transpose(1, 2).contiguous()

with torch.no_grad():
    waveform = vocoder(mel_input)

waveform = waveform.squeeze().cpu()

assert torch.isfinite(waveform).all()

# Prevent clipping while preserving relative amplitude
peak = waveform.abs().max()

if peak > 0:
    waveform = waveform * (0.95 / peak)

# ------------------------------------------------------------
# 6. Save WAV
# ------------------------------------------------------------

output_dir = BASE / "Telugu/inference_smoke_test"
output_dir.mkdir(parents=True, exist_ok=True)

wav_path = output_dir / "telugu_validation_smoke_test.wav"

torchaudio.save(
    str(wav_path),
    waveform.unsqueeze(0),
    22050
)

print("\nAUDIO GENERATED SUCCESSFULLY")
print("File:", wav_path)
print("Duration:", round(waveform.numel() / 22050, 2), "seconds")
print("Sample rate: 22050 Hz")
print("Peak amplitude:", round(waveform.abs().max().item(), 4))

# Clean up
del vocoder, vocoder_checkpoint, waveform
gc.collect()

print("\nSTEP 82 COMPLETE")

FastSpeech 2 ready.
Validation sample: train_telugufemale_00060_chunk000.npz
Text tokens: 66
Speaker ID: 0
Predicted mel shape: (1, 449, 80)
Universal HiFi-GAN loaded.

AUDIO GENERATED SUCCESSFULLY
File: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference_smoke_test/telugu_validation_smoke_test.wav
Duration: 5.21 seconds
Sample rate: 22050 Hz
Peak amplitude: 0.95

STEP 82 COMPLETE


In [34]:
# ============================================================
# STEP 83A: INSPECT TELUGU ROMANIZATION PIPELINE
# ============================================================

import json
import sys
import inspect
import importlib.util
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

sys.path.insert(0, str(REPO))

# ------------------------------------------------------------
# 1. Inspect an existing MMS_FA alignment record
# ------------------------------------------------------------

alignment_file = (
    BASE / "metadata/telugu_train_mmsfa.jsonl"
)

with open(alignment_file, encoding="utf-8") as f:
    first_record = json.loads(next(f))

print("EXISTING TRAINING ROMANIZATION")
print("=" * 60)
print("Original Telugu:")
print(first_record["original_text"])
print("\nRomanized text:")
print(first_record["romanized_text"])
print("\nNormalized text:")
print(first_record["normalized_text"])

# ------------------------------------------------------------
# 2. Inspect installed romanization packages
# ------------------------------------------------------------

print("\nROMANIZATION PACKAGES")
print("=" * 60)

for package in ["uroman", "torchaudio", "indic_transliteration"]:
    print(package, ":", importlib.util.find_spec(package))

# ------------------------------------------------------------
# 3. Inspect MMS_FA tokenizer
# ------------------------------------------------------------

import torchaudio

bundle = torchaudio.pipelines.MMS_FA
tokenizer = bundle.get_tokenizer()

print("\nMMS_FA TOKENIZER")
print("=" * 60)
print("Tokenizer class:", type(tokenizer))
print("Tokenizer module:", type(tokenizer).__module__)

try:
    print("Tokenizer signature:", inspect.signature(tokenizer))
except Exception as e:
    print("Signature unavailable:", e)

try:
    print("Tokenizer source:")
    print(inspect.getsource(type(tokenizer))[:5000])
except Exception as e:
    print("Source unavailable:", e)

print("\nSTEP 83A COMPLETE")

EXISTING TRAINING ROMANIZATION
Original Telugu:
కొంతదూరం అడవిలోంచి నడిచి అందరు ఒక కొండప్రాంతం చేరుకున్నారు.

Romanized text:
komtaduuram addaviloomci naddici amdaru oka komddapraamtam ceerukunnaaru.

Normalized text:
komtaduuram addaviloomci naddici amdaru oka komddapraamtam ceerukunnaaru

ROMANIZATION PACKAGES
uroman : None
torchaudio : ModuleSpec(name='torchaudio', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7bf7d8ceb050>, origin='/usr/local/lib/python3.13/dist-packages/torchaudio/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/torchaudio'])
indic_transliteration : None

MMS_FA TOKENIZER
Tokenizer class: <class 'torchaudio.pipelines._wav2vec2.aligner.Tokenizer'>
Tokenizer module: torchaudio.pipelines._wav2vec2.aligner
Tokenizer signature: (transcript: List[str]) -> List[List[int]]
Tokenizer source:
class Tokenizer(ITokenizer):
    def __init__(self, dictionary: Dict[str, int]):
        self.dictionary = dictionary

    def __call

In [35]:
# ============================================================
# STEP 83B: LOCATE UROMAN IMPLEMENTATION
# ============================================================

from pathlib import Path
import importlib.util
import os

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

print("CHECKING PYTHON MODULES")
print("=" * 55)

for module in ["uroman", "uroman.romanize", "indic_transliteration"]:
    try:
        print(module, ":", importlib.util.find_spec(module))
    except Exception as e:
        print(module, ":", e)

print("\nCHECKING COMMON UROMAN LOCATIONS")
print("=" * 55)

search_roots = [
    Path("/content"),
    Path("/root/.cache"),
    BASE
]

matches = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if "uroman" in path.name.lower():
            matches.append(path)

            if len(matches) >= 100:
                break

    if len(matches) >= 100:
        break

for path in matches:
    print(path)

if not matches:
    print("No Uroman files found in the inspected locations.")

print("\nCHECKING ALIGNMENT METADATA")
print("=" * 55)

alignment = BASE / "metadata/telugu_train_mmsfa.jsonl"

print("Alignment file exists:", alignment.exists())
print("Alignment file size:", alignment.stat().st_size if alignment.exists() else None)

print("\nSTEP 83B COMPLETE")

CHECKING PYTHON MODULES
uroman : None
uroman.romanize : No module named 'uroman'
indic_transliteration : None

CHECKING COMMON UROMAN LOCATIONS
No Uroman files found in the inspected locations.

CHECKING ALIGNMENT METADATA
Alignment file exists: True
Alignment file size: 39454921

STEP 83B COMPLETE


In [36]:
# ============================================================
# STEP 83C: INSTALL AND VERIFY UROMAN
# ============================================================

import sys
import subprocess
import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO_DIR = Path("/content/uroman")

# ------------------------------------------------------------
# 1. Obtain the original Uroman implementation
# ------------------------------------------------------------

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/isi-nlp/uroman.git",
            str(REPO_DIR)
        ],
        check=True
    )

# ------------------------------------------------------------
# 2. Install it
# ------------------------------------------------------------

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)],
    check=True
)

# ------------------------------------------------------------
# 3. Import Uroman
# ------------------------------------------------------------

from uroman import Uroman

uroman = Uroman()

print("Uroman imported successfully.")

# ------------------------------------------------------------
# 4. Verify against existing training alignment
# ------------------------------------------------------------

alignment_file = BASE / "metadata/telugu_train_mmsfa.jsonl"

with open(alignment_file, encoding="utf-8") as f:
    record = json.loads(next(f))

original_text = record["original_text"]
expected = record["romanized_text"]

generated = uroman.romanize_string(original_text)

print("\nREFERENCE TEXT:")
print(original_text)

print("\nEXISTING TRAINING ROMANIZATION:")
print(expected)

print("\nNEW UROMAN OUTPUT:")
print(generated)

print("\nEXACT MATCH:", generated == expected)

# Save diagnostic
result = {
    "original_text": original_text,
    "existing_romanization": expected,
    "new_romanization": generated,
    "exact_match": generated == expected
}

output = BASE / "Telugu/inference/logs/uroman_verification.json"
output.parent.mkdir(parents=True, exist_ok=True)

with open(output, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved:", output)
print("\nSTEP 83C COMPLETE")

ImportError: cannot import name 'Uroman' from 'uroman' (unknown location)

In [37]:
# ============================================================
# STEP 83D: FIX UROMAN IMPORT AND VERIFY ROMANIZATION
# ============================================================

import sys
import json
import importlib
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
UROMAN_REPO = Path("/content/uroman")

# Make the repository root take priority over /content
sys.path.insert(0, str(UROMAN_REPO))

# Clear any incorrectly cached namespace package
sys.modules.pop("uroman", None)

# Correct official import
import uroman as ur

print("Imported module:", ur.__file__)
print("Uroman class available:", hasattr(ur, "Uroman"))

uroman = ur.Uroman()

# Load an actual training example
alignment_file = BASE / "metadata/telugu_train_mmsfa.jsonl"

with open(alignment_file, encoding="utf-8") as f:
    record = json.loads(next(f))

original = record["original_text"]
expected = record["romanized_text"]

generated = uroman.romanize_string(original)

print("\nORIGINAL TELUGU:")
print(original)

print("\nTRAINING ROMANIZATION:")
print(expected)

print("\nNEW ROMANIZATION:")
print(generated)

print("\nEXACT MATCH:", generated == expected)

# Save verification
output = BASE / "Telugu/inference/logs/uroman_verification.json"

with open(output, "w", encoding="utf-8") as f:
    json.dump({
        "original_text": original,
        "training_romanization": expected,
        "new_romanization": generated,
        "exact_match": generated == expected
    }, f, ensure_ascii=False, indent=2)

print("\nSaved:", output)
print("STEP 83D COMPLETE")

Imported module: /content/uroman/uroman/__init__.py
Uroman class available: True

ORIGINAL TELUGU:
కొంతదూరం అడవిలోంచి నడిచి అందరు ఒక కొండప్రాంతం చేరుకున్నారు.

TRAINING ROMANIZATION:
komtaduuram addaviloomci naddici amdaru oka komddapraamtam ceerukunnaaru.

NEW ROMANIZATION:
komtaduuram addaviloomci naddici amdaru oka komddapraamtam ceerukunnaaru.

EXACT MATCH: True

Saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference/logs/uroman_verification.json
STEP 83D COMPLETE


In [40]:
# ============================================================
# STEP 84: ROMANIZE AND VALIDATE TELUGU TEST SENTENCES
# ============================================================

import sys
import json
import re
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

sys.path.insert(0, "/content/uroman")

import uroman as ur
uroman = ur.Uroman()

from text.symbols import symbols

# Build the character-to-ID mapping
symbol_to_id = {symbol: idx for idx, symbol in enumerate(symbols)}

# ------------------------------------------------------------
# 1. Exact assignment test sentences
# ------------------------------------------------------------

test_sentences = {
    "T1": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
    "T2a": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
    "T2b": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
    "T3": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
    "T4": "ఈ సమావేశం రెండువేల ఇరవై ఆరు నవంబర్ పన్నెండవ తేదీ ఉదయం పది గంటల నలభై ఐదు నిమిషాలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి."
}

# ------------------------------------------------------------
# 2. Romanize using verified Uroman
# ------------------------------------------------------------

results = {}
all_valid = True

for key, original in test_sentences.items():

    romanized = uroman.romanize_string(original)

    # Match the whitespace normalization used in training
    normalized = re.sub(r"\s+", " ", romanized).strip()

    # FastSpeech 2 training uses character-level symbols
    chars = list(normalized)

    unknown = sorted(set(c for c in chars if c not in symbol_to_id))

    token_ids = [
        symbol_to_id[c]
        for c in chars
        if c in symbol_to_id
    ]

    valid = len(unknown) == 0 and len(token_ids) == len(chars)

    results[key] = {
        "original_telugu": original,
        "romanized": romanized,
        "normalized": normalized,
        "characters": chars,
        "token_ids": token_ids,
        "unknown_characters": unknown,
        "vocabulary_valid": valid
    }

    all_valid = all_valid and valid

    print("\n" + "=" * 60)
    print(key)
    print("Original:", original)
    print("Romanized:", romanized)
    print("Normalized:", normalized)
    print("Character count:", len(chars))
    print("Token count:", len(token_ids))
    print("Unknown characters:", unknown)
    print("Vocabulary valid:", valid)

# ------------------------------------------------------------
# 3. Save complete mapping
# ------------------------------------------------------------

output = BASE / "Telugu/inference/test_sentence_token_mapping.json"
output.parent.mkdir(parents=True, exist_ok=True)

with open(output, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print("ALL FIVE SENTENCES VOCABULARY VALID:", all_valid)
print("Saved:", output)

print("\nSTEP 84 COMPLETE")


T1
Original: న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.
Romanized: nyuu roophaarma koolaajikal parishoodhanaloo nuutana aussadhaala prabhaavaanni vishleessimcaaru.
Normalized: nyuu roophaarma koolaajikal parishoodhanaloo nuutana aussadhaala prabhaavaanni vishleessimcaaru.
Character count: 96
Token count: 96
Unknown characters: []
Vocabulary valid: True

T2a
Original: తాడుపై అతని పట్టు చాలా బలంగా ఉంది.
Romanized: taaddupai atani pattttu caalaa balamgaa umdi.
Normalized: taaddupai atani pattttu caalaa balamgaa umdi.
Character count: 45
Token count: 45
Unknown characters: []
Vocabulary valid: True

T2b
Original: ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.
Romanized: aame pattttu ciiranu dharimci veeddukaku vaccimdi.
Normalized: aame pattttu ciiranu dharimci veeddukaku vaccimdi.
Character count: 50
Token count: 50
Unknown characters: []
Vocabulary valid: True

T3
Original: పులి అడవిలో నిశ్శబ్దంగా నడిచింది.
Romanized: puli addaviloo nishshabdamgaa naddicimdi.
Norma

In [41]:
# ================================================================
# STEP 85: TELUGU FASTSPEECH 2 INFERENCE + AUDIO + 15 PLOTS
# ================================================================
# RUN NOW: New cell
# DO NOT rerun: preprocessing, alignment, feature extraction, training
# Speaker: female (speaker ID 0) for all five test sentences
# Output: 5 WAV files + 15 plots + prediction NPZ files
# ================================================================

import os
import re
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. PATHS AND SETTINGS
# ------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
TELUGU = BASE / "Telugu"

AUDIO_DIR = TELUGU / "results/test_generated_audios"
DURATION_DIR = TELUGU / "results/variance_adaptor_plots/duration"
PITCH_DIR = TELUGU / "results/variance_adaptor_plots/pitch"
ENERGY_DIR = TELUGU / "results/variance_adaptor_plots/energy"
PRED_DIR = TELUGU / "inference/test_predictions"
LOG_DIR = TELUGU / "inference/logs"

for folder in [
    AUDIO_DIR, DURATION_DIR, PITCH_DIR,
    ENERGY_DIR, PRED_DIR, LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 22050
SPEAKER_ID = 0       # female
DEVICE = torch.device("cpu")

print("Device:", DEVICE)
print("Output directory:", TELUGU)

# ------------------------------------------------
# 2. CHECK REQUIRED OBJECTS
# ------------------------------------------------

required_objects = ["model", "generator", "uroman", "symbol_to_id"]

missing = [x for x in required_objects if x not in globals()]

if missing:
    raise RuntimeError(
        f"Missing objects: {missing}\n"
        "This cell requires the model and HiFi-GAN generator "
        "from the successful Step 82 smoke test, plus Uroman "
        "and symbol_to_id from Step 84."
    )

model = model.to(DEVICE)
model.eval()

generator = generator.to(DEVICE)
generator.eval()

print("FastSpeech 2 model: loaded")
print("HiFi-GAN generator: loaded")
print("Uroman: loaded")
print("Vocabulary size:", len(symbol_to_id))

# ------------------------------------------------
# 3. TEST SENTENCES
# ------------------------------------------------
# T4 is expanded into spoken Telugu for model input.
# The original numeric sentence is retained in the metadata.

test_sentences = {
    "T1": {
        "original": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
        "spoken": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు."
    },
    "T2a": {
        "original": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
        "spoken": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది."
    },
    "T2b": {
        "original": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
        "spoken": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది."
    },
    "T3": {
        "original": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
        "spoken": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది."
    },
    "T4": {
        "original": "ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి.",
        "spoken": "ఈ సమావేశం రెండువేల ఇరవై ఆరు నవంబర్ పన్నెండవ తేదీ ఉదయం పది గంటల నలభై ఐదు నిమిషాలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి."
    }
}

# ------------------------------------------------
# 4. TEXT NORMALIZATION AND TOKEN MAPPING
# ------------------------------------------------

def prepare_text(text):
    """
    Follow the romanized-character representation used
    by the Telugu FastSpeech 2 pipeline.
    """

    romanized = uroman.romanize_string(text)

    # Match the punctuation-free normalized-text convention.
    normalized = re.sub(r"[^\w\s]", "", romanized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    normalized = normalized.lower()

    chars = list(normalized)
    unknown = sorted(set(c for c in chars if c not in symbol_to_id))

    if unknown:
        raise ValueError(
            f"Unknown vocabulary characters: {unknown}\n"
            f"Text: {text}\nRomanized: {romanized}"
        )

    ids = [symbol_to_id[c] for c in chars]

    if len(ids) == 0:
        raise ValueError("Empty token sequence.")

    return romanized, normalized, chars, ids


# ------------------------------------------------
# 5. PREPARE AND VALIDATE ALL SENTENCES
# ------------------------------------------------

prepared = {}

print("\n" + "=" * 65)
print("TEST SENTENCE TOKENIZATION")
print("=" * 65)

for name, item in test_sentences.items():

    romanized, normalized, chars, ids = prepare_text(item["spoken"])

    prepared[name] = {
        "original": item["original"],
        "spoken": item["spoken"],
        "romanized": romanized,
        "normalized": normalized,
        "tokens": chars,
        "token_ids": ids
    }

    print(f"\n{name}")
    print("Romanized:", romanized)
    print("Normalized:", normalized)
    print("Token count:", len(ids))
    print("Unknown characters: []")
    print("Vocabulary valid: True")

print("\nAll five sentences passed vocabulary validation.")

# ------------------------------------------------
# 6. CHECK MODEL OUTPUT SHAPES BEFORE GENERATION
# ------------------------------------------------

print("\nChecking model inference output...")

first_name = list(prepared.keys())[0]
first_ids = prepared[first_name]["token_ids"]

text_tensor = torch.tensor(
    [first_ids], dtype=torch.long, device=DEVICE
)

src_lens = torch.tensor(
    [len(first_ids)], dtype=torch.long, device=DEVICE
)

speaker_tensor = torch.tensor(
    [SPEAKER_ID], dtype=torch.long, device=DEVICE
)

with torch.no_grad():
    smoke_outputs = model(
        speaker_tensor,
        text_tensor,
        src_lens,
        int(src_lens.max().item())
    )

if len(smoke_outputs) < 6:
    raise RuntimeError(
        f"Unexpected model output count: {len(smoke_outputs)}"
    )

print("Model output count:", len(smoke_outputs))
for i, value in enumerate(smoke_outputs[:6]):
    if torch.is_tensor(value):
        print(f"Output {i}: shape={tuple(value.shape)}")

# Expected FastSpeech 2 outputs:
# 0 = mel before postnet
# 1 = mel after postnet
# 2 = pitch predictions
# 3 = energy predictions
# 4 = log-duration predictions
# 5 = rounded predicted durations

# ------------------------------------------------
# 7. INFERENCE FUNCTION
# ------------------------------------------------

def run_inference(token_ids):

    text = torch.tensor(
        [token_ids], dtype=torch.long, device=DEVICE
    )

    src_lens = torch.tensor(
        [len(token_ids)], dtype=torch.long, device=DEVICE
    )

    speakers = torch.tensor(
        [SPEAKER_ID], dtype=torch.long, device=DEVICE
    )

    with torch.no_grad():

        outputs = model(
            speakers,
            text,
            src_lens,
            int(src_lens.max().item())
        )

        mel_prediction = outputs[1]
        pitch_prediction = outputs[2]
        energy_prediction = outputs[3]
        log_duration_prediction = outputs[4]
        duration_prediction = outputs[5]

        # Remove batch dimension.
        mel = mel_prediction[0].detach().cpu().numpy()
        pitch = pitch_prediction[0].detach().cpu().numpy()
        energy = energy_prediction[0].detach().cpu().numpy()
        log_duration = log_duration_prediction[0].detach().cpu().numpy()
        durations = duration_prediction[0].detach().cpu().numpy()

    return {
        "mel": mel,
        "pitch": pitch,
        "energy": energy,
        "log_duration": log_duration,
        "durations": durations
    }


# ------------------------------------------------
# 8. VOCODER FUNCTION
# ------------------------------------------------

def generate_waveform(mel):

    # FastSpeech 2 output: [time, mel_channels]
    # HiFi-GAN input: [batch, mel_channels, time]

    mel_tensor = torch.tensor(
        mel.T[None, :, :],
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():
        waveform = generator(mel_tensor)

    waveform = waveform.squeeze().detach().cpu().numpy()
    waveform = np.nan_to_num(waveform)

    peak = np.max(np.abs(waveform))

    if peak > 1e-8:
        waveform = 0.95 * waveform / peak

    return waveform


# ------------------------------------------------
# 9. PLOTTING FUNCTIONS
# ------------------------------------------------

def save_duration_plot(name, tokens, durations):

    durations = np.asarray(durations).reshape(-1)

    # Duration predictor output is token-level.
    n = min(len(tokens), len(durations))

    plt.figure(figsize=(max(12, n * 0.16), 5))
    plt.bar(np.arange(n), durations[:n])
    plt.xlabel("Input character position")
    plt.ylabel("Predicted duration (mel frames)")
    plt.title(f"{name} - Predicted Duration")
    plt.xticks(
        np.arange(n),
        tokens[:n],
        rotation=90,
        fontsize=7
    )
    plt.tight_layout()
    plt.savefig(DURATION_DIR / f"{name}_duration.png", dpi=180)
    plt.close()


def save_pitch_plot(name, pitch):

    pitch = np.asarray(pitch).reshape(-1)

    plt.figure(figsize=(12, 4))
    plt.plot(pitch)
    plt.xlabel("Mel frame")
    plt.ylabel("Predicted pitch")
    plt.title(f"{name} - Predicted Pitch")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PITCH_DIR / f"{name}_pitch.png", dpi=180)
    plt.close()


def save_energy_plot(name, energy):

    energy = np.asarray(energy).reshape(-1)

    plt.figure(figsize=(12, 4))
    plt.plot(energy)
    plt.xlabel("Mel frame")
    plt.ylabel("Predicted energy")
    plt.title(f"{name} - Predicted Energy")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(ENERGY_DIR / f"{name}_energy.png", dpi=180)
    plt.close()


# ------------------------------------------------
# 10. GENERATE ALL FIVE AUDIOS
# ------------------------------------------------

results_summary = []

print("\n" + "=" * 65)
print("GENERATING TELUGU TEST AUDIOS")
print("=" * 65)

for name, item in prepared.items():

    print(f"\nProcessing {name}...")

    prediction = run_inference(item["token_ids"])

    mel = prediction["mel"]
    pitch = prediction["pitch"]
    energy = prediction["energy"]
    durations = prediction["durations"]
    log_duration = prediction["log_duration"]

    # Basic validity checks.
    if mel.ndim != 2 or mel.shape[1] != 80:
        raise RuntimeError(
            f"{name}: unexpected mel shape {mel.shape}"
        )

    if not np.isfinite(mel).all():
        raise RuntimeError(f"{name}: non-finite mel values.")

    if not np.isfinite(durations).all():
        raise RuntimeError(f"{name}: non-finite duration values.")

    # Generate waveform using universal HiFi-GAN.
    waveform = generate_waveform(mel)

    if len(waveform) == 0:
        raise RuntimeError(f"{name}: generated empty waveform.")

    # Save WAV.
    wav_path = AUDIO_DIR / f"{name}.wav"

    sf.write(
        str(wav_path),
        waveform,
        SAMPLE_RATE,
        subtype="PCM_16"
    )

    # Save predictions.
    prediction_path = PRED_DIR / f"{name}_predictions.npz"

    np.savez_compressed(
        prediction_path,
        mel=mel,
        pitch=pitch,
        energy=energy,
        durations=durations,
        log_duration=log_duration,
        tokens=np.asarray(item["tokens"]),
        token_ids=np.asarray(item["token_ids"]),
        original_text=item["original"],
        spoken_text=item["spoken"],
        romanized_text=item["romanized"],
        normalized_text=item["normalized"],
        speaker_id=SPEAKER_ID
    )

    # Save three variance-adaptor plots.
    save_duration_plot(name, item["tokens"], durations)
    save_pitch_plot(name, pitch)
    save_energy_plot(name, energy)

    duration_seconds = len(waveform) / SAMPLE_RATE

    results_summary.append({
        "sentence": name,
        "input_characters": len(item["tokens"]),
        "mel_frames": int(mel.shape[0]),
        "pitch_values": int(np.asarray(pitch).size),
        "energy_values": int(np.asarray(energy).size),
        "duration_values": int(np.asarray(durations).size),
        "audio_seconds": round(duration_seconds, 3),
        "audio_file": str(wav_path),
        "prediction_file": str(prediction_path)
    })

    print("Audio saved:", wav_path.name)
    print("Audio duration:", round(duration_seconds, 3), "seconds")
    print("Mel shape:", mel.shape)
    print("Pitch shape:", np.asarray(pitch).shape)
    print("Energy shape:", np.asarray(energy).shape)
    print("Duration shape:", np.asarray(durations).shape)


# ------------------------------------------------
# 11. SAVE GENERATION SUMMARY
# ------------------------------------------------

summary_path = LOG_DIR / "telugu_inference_summary.json"

summary_data = {
    "generated_at": datetime.now().isoformat(),
    "model": "FastSpeech 2",
    "vocoder": "HiFi-GAN universal",
    "device": "CPU",
    "sample_rate": SAMPLE_RATE,
    "speaker_id": SPEAKER_ID,
    "speaker_description": "Female speaker",
    "sentences": results_summary
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, ensure_ascii=False, indent=2)


# ------------------------------------------------
# 12. FINAL VERIFICATION
# ------------------------------------------------

print("\n" + "=" * 65)
print("STEP 85 COMPLETE")
print("=" * 65)

print("\nGenerated audio files:")
for path in sorted(AUDIO_DIR.glob("*.wav")):
    info = sf.info(str(path))
    print(
        path.name,
        "|", round(info.duration, 2), "seconds",
        "|", info.samplerate, "Hz"
    )

print("\nDuration plots:", len(list(DURATION_DIR.glob("*.png"))))
print("Pitch plots:", len(list(PITCH_DIR.glob("*.png"))))
print("Energy plots:", len(list(ENERGY_DIR.glob("*.png"))))
print("Prediction files:", len(list(PRED_DIR.glob("*_predictions.npz"))))

print("\nSummary saved:", summary_path)
print("\nAudio folder:", AUDIO_DIR)
print("Plot folders:", TELUGU / "results/variance_adaptor_plots")
print("Prediction folder:", PRED_DIR)

print("\nIMPORTANT:")
print("Listen to all five WAV files before judging pronunciation.")
print("The universal HiFi-GAN vocoder is not confirmed to be Telugu-trained.")
print("These results are experimental outputs, not proof of natural Telugu speech.")

Device: cpu
Output directory: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu
FastSpeech 2 model: loaded
HiFi-GAN generator: loaded
Uroman: loaded
Vocabulary size: 360

TEST SENTENCE TOKENIZATION

T1
Romanized: nyuu roophaarma koolaajikal parishoodhanaloo nuutana aussadhaala prabhaavaanni vishleessimcaaru.
Normalized: nyuu roophaarma koolaajikal parishoodhanaloo nuutana aussadhaala prabhaavaanni vishleessimcaaru
Token count: 95
Unknown characters: []
Vocabulary valid: True

T2a
Romanized: taaddupai atani pattttu caalaa balamgaa umdi.
Normalized: taaddupai atani pattttu caalaa balamgaa umdi
Token count: 44
Unknown characters: []
Vocabulary valid: True

T2b
Romanized: aame pattttu ciiranu dharimci veeddukaku vaccimdi.
Normalized: aame pattttu ciiranu dharimci veeddukaku vaccimdi
Token count: 49
Unknown characters: []
Vocabulary valid: True

T3
Romanized: puli addaviloo nishshabdamgaa naddicimdi.
Normalized: puli addaviloo nishshabdamgaa naddicimdi
Token count: 40
Unknown characters:

In [42]:
# ================================================================
# STEP 86: TELUGU AUDIO QUALITY AUDIT
# ================================================================

from pathlib import Path
from IPython.display import Audio, display
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu")
AUDIO_DIR = BASE / "results/test_generated_audios"
AUDIT_DIR = BASE / "results/audio_quality_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

sentences = ["T1", "T2a", "T2b", "T3", "T4"]
audit = []

for name in sentences:

    path = AUDIO_DIR / f"{name}.wav"

    if not path.exists():
        print("Missing:", path)
        continue

    audio, sr = sf.read(path)

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    duration = len(audio) / sr
    peak = float(np.max(np.abs(audio)))
    rms = float(np.sqrt(np.mean(audio ** 2)))
    silence_ratio = float(np.mean(np.abs(audio) < 0.001))
    clipped_ratio = float(np.mean(np.abs(audio) >= 0.999))

    audit.append({
        "Sentence": name,
        "Duration (s)": round(duration, 3),
        "Sample rate": sr,
        "Peak amplitude": round(peak, 5),
        "RMS": round(rms, 5),
        "Near-silence ratio (%)": round(silence_ratio * 100, 2),
        "Clipped samples (%)": round(clipped_ratio * 100, 4)
    })

    print("\n" + "=" * 55)
    print(name, "|", round(duration, 2), "seconds")
    print("Listen to this audio:")
    display(Audio(audio, rate=sr))

    # Save waveform plot.
    time_axis = np.arange(len(audio)) / sr

    plt.figure(figsize=(13, 3))
    plt.plot(time_axis, audio, linewidth=0.5)
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.title(f"{name} - Generated Telugu Audio Waveform")
    plt.tight_layout()
    plt.savefig(AUDIT_DIR / f"{name}_waveform.png", dpi=160)
    plt.show()
    plt.close()

df = pd.DataFrame(audit)

print("\n" + "=" * 65)
print("AUDIO QUALITY AUDIT SUMMARY")
print("=" * 65)

display(df)

df.to_csv(AUDIT_DIR / "telugu_audio_quality.csv", index=False)

print("Audit saved:", AUDIT_DIR)



T1 | 6.83 seconds
Listen to this audio:



T2a | 3.49 seconds
Listen to this audio:



T2b | 4.21 seconds
Listen to this audio:



T3 | 3.22 seconds
Listen to this audio:



T4 | 12.17 seconds
Listen to this audio:



AUDIO QUALITY AUDIT SUMMARY


,Sentence,Duration (s),Sample rate,Peak amplitude,RMS,Near-silence ratio (%),Clipped samples (%)
0,T1,6.827,22050,0.95001,0.23530,0.37,0.0
1,T2a,3.495,22050,0.95001,0.24622,0.32,0.0
2,T2b,4.214,22050,0.95001,0.24761,0.38,0.0
3,T3,3.216,22050,0.95001,0.24364,0.30,0.0
4,T4,12.167,22050,0.95001,0.23638,0.35,0.0


Audit saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/audio_quality_audit


In [43]:
# STEP 86B: PACKAGE ALL TELUGU TEST AUDIOS

from pathlib import Path
import zipfile
from google.colab import files

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu")
AUDIO_DIR = BASE / "results/test_generated_audios"

ZIP_PATH = BASE / "results/Telugu_Test_Audios.zip"

sentences = ["T1", "T2a", "T2b", "T3", "T4"]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:

    for name in sentences:
        audio_path = AUDIO_DIR / f"{name}.wav"

        if not audio_path.exists():
            print("Missing:", name)
            continue

        z.write(audio_path, arcname=f"{name}.wav")
        print("Added:", name)

print("\nZIP created:", ZIP_PATH)
print("Download starting...")

files.download(str(ZIP_PATH))

Added: T1
Added: T2a
Added: T2b
Added: T3
Added: T4

ZIP created: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/Telugu_Test_Audios.zip
Download starting...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
# ================================================================
# STEP 87: FASTSPEECH 2 / HIFIGAN MEL COMPATIBILITY DIAGNOSTIC
# ================================================================

import numpy as np
import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu")

FEATURE_DIR = BASE / "features/train"
PRED_DIR = BASE / "inference/test_predictions"
VOCODER_CONFIG = Path("/content/FastSpeech2/hifigan/config.json")

def inspect_npz(path, key="mel"):
    data = np.load(path, allow_pickle=True)
    print("\nFile:", path.name)
    print("Keys:", data.files)

    if key not in data.files:
        print("Missing key:", key)
        return

    arr = np.asarray(data[key], dtype=np.float64)

    print("Shape:", arr.shape)
    print("Min:", np.nanmin(arr))
    print("Max:", np.nanmax(arr))
    print("Mean:", np.nanmean(arr))
    print("Std:", np.nanstd(arr))
    print("NaN:", np.isnan(arr).sum())
    print("Inf:", np.isinf(arr).sum())

print("=" * 65)
print("1. TRAINING MEL FEATURE")
print("=" * 65)

train_files = sorted(FEATURE_DIR.glob("*.npz"))

if not train_files:
    raise FileNotFoundError("No Telugu training feature files found.")

inspect_npz(train_files[0])

print("\n" + "=" * 65)
print("2. GENERATED MEL PREDICTIONS")
print("=" * 65)

prediction_files = sorted(PRED_DIR.glob("*_predictions.npz"))

for path in prediction_files:
    inspect_npz(path, "mel")

print("\n" + "=" * 65)
print("3. HIFIGAN CONFIGURATION")
print("=" * 65)

if VOCODER_CONFIG.exists():
    with open(VOCODER_CONFIG, "r") as f:
        config = json.load(f)

    for key in [
        "sampling_rate", "num_mels", "n_fft",
        "hop_size", "win_size", "fmin", "fmax"
    ]:
        print(key, ":", config.get(key, "Not specified"))
else:
    print("HiFi-GAN config not found.")

print("\n" + "=" * 65)
print("4. CHECKPOINTS")
print("=" * 65)

for path in sorted((BASE / "checkpoints").glob("*.pth.tar")):
    print(path.name, "|", round(path.stat().st_size / (1024**2), 2), "MB")

print("\nSTEP 87 DIAGNOSTIC COMPLETE")

1. TRAINING MEL FEATURE

File: train_telugufemale_00009.npz
Keys: ['mel', 'pitch', 'energy', 'tokens', 'durations', 'mean_alignment_score', 'low_confidence_count', 'sample_rate', 'hop_length']
Shape: (80, 1509)
Min: -11.512925148010254
Max: 1.1536953449249268
Mean: -6.723828378207654
Std: 2.604683433057743
NaN: 0
Inf: 0

2. GENERATED MEL PREDICTIONS

File: T1_predictions.npz
Keys: ['mel', 'pitch', 'energy', 'durations', 'log_duration', 'tokens', 'token_ids', 'original_text', 'spoken_text', 'romanized_text', 'normalized_text', 'speaker_id']
Shape: (588, 80)
Min: -6.452662944793701
Max: -1.1587300300598145
Mean: -3.039835998172663
Std: 0.9168557564139657
NaN: 0
Inf: 0

File: T2a_predictions.npz
Keys: ['mel', 'pitch', 'energy', 'durations', 'log_duration', 'tokens', 'token_ids', 'original_text', 'spoken_text', 'romanized_text', 'normalized_text', 'speaker_id']
Shape: (301, 80)
Min: -6.392207145690918
Max: -1.2248934507369995
Mean: -3.062393626169707
Std: 0.9162704083141219
NaN: 0
Inf: 0



In [46]:
# ================================================================
# STEP 88: INDEPENDENT HIFIGAN VOCODER TEST
# ================================================================

import torch
import numpy as np
import soundfile as sf
import json
from pathlib import Path
from types import SimpleNamespace
from IPython.display import Audio, display

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu")
REPO = Path("/content/FastSpeech2")

DEVICE = torch.device("cpu")
SR = 22050

# ------------------------------------------------
# 1. LOAD REAL AND PREDICTED MEL
# ------------------------------------------------

real_path = BASE / "features/train/train_telugufemale_00009.npz"
pred_path = BASE / "inference/test_predictions/T1_predictions.npz"

real_data = np.load(real_path, allow_pickle=True)
pred_data = np.load(pred_path, allow_pickle=True)

real_mel = np.asarray(real_data["mel"], dtype=np.float32)
pred_mel = np.asarray(pred_data["mel"], dtype=np.float32)

# Real mel: [80, frames]
# Predicted mel: [frames, 80]
assert real_mel.shape[0] == 80
assert pred_mel.shape[1] == 80

print("Real mel:", real_mel.shape)
print("Predicted mel:", pred_mel.shape)

# ------------------------------------------------
# 2. LOAD HIFIGAN CONFIGURATION
# ------------------------------------------------

with open(REPO / "hifigan/config.json", "r") as f:
    h = SimpleNamespace(**json.load(f))

# ------------------------------------------------
# 3. LOAD BOTH VOCODERS
# ------------------------------------------------

from hifigan.models import Generator

def load_vocoder(checkpoint_path):

    vocoder = Generator(h).to(DEVICE).eval()

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    state = checkpoint["generator"]

    vocoder.load_state_dict(state, strict=True)

    return vocoder

VOCODER_DIR = BASE / "vocoder"

universal_path = (
    VOCODER_DIR / "generator_universal" /
    "generator_universal.pth.tar"
)

ljspeech_path = (
    VOCODER_DIR / "generator_LJSpeech" /
    "generator_LJSpeech.pth.tar"
)

print("Universal checkpoint exists:", universal_path.exists())
print("LJSpeech checkpoint exists:", ljspeech_path.exists())

if not universal_path.exists() or not ljspeech_path.exists():
    raise FileNotFoundError(
        "Vocoder checkpoint missing. Check Telugu/vocoder folder."
    )

# Extracted checkpoints should already exist.
# If these files are absent, stop rather than downloading again.

if not universal_path.exists():
    raise FileNotFoundError(universal_path)

if not ljspeech_path.exists():
    raise FileNotFoundError(ljspeech_path)

universal = load_vocoder(universal_path)
ljspeech = load_vocoder(ljspeech_path)

print("Both vocoders loaded successfully.")

# ------------------------------------------------
# 4. GENERATE WAVEFORMS
# ------------------------------------------------

def vocode(vocoder, mel):

    # mel must be [80, frames]
    x = torch.tensor(
        mel[None, :, :],
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():
        y = vocoder(x)

    y = y.squeeze().cpu().numpy()
    y = np.nan_to_num(y)

    # Do not peak-normalize here: retain raw output levels.
    return y


tests = {
    "universal_real_training": (universal, real_mel),
    "universal_predicted_T1": (universal, pred_mel.T),
    "ljspeech_real_training": (ljspeech, real_mel),
    "ljspeech_predicted_T1": (ljspeech, pred_mel.T)
}

OUT = BASE / "inference/vocoder_diagnostic"
OUT.mkdir(parents=True, exist_ok=True)

results = {}

for name, (vocoder, mel) in tests.items():

    print("\nTesting:", name)

    waveform = vocode(vocoder, mel)

    peak = float(np.max(np.abs(waveform)))
    rms = float(np.sqrt(np.mean(waveform ** 2)))
    duration = len(waveform) / SR

    output_path = OUT / f"{name}.wav"

    sf.write(
        str(output_path),
        waveform,
        SR,
        subtype="PCM_16"
    )

    results[name] = {
        "duration": duration,
        "peak": peak,
        "rms": rms,
        "finite": bool(np.isfinite(waveform).all()),
        "file": str(output_path)
    }

    print("Duration:", round(duration, 3))
    print("Peak:", peak)
    print("RMS:", rms)
    print("Saved:", output_path)

    print("Listen:")
    display(Audio(waveform, rate=SR))

# ------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------

summary_path = OUT / "vocoder_diagnostic.json"

with open(summary_path, "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "=" * 65)
print("STEP 88 COMPLETE")
print("=" * 65)
print(json.dumps(results, indent=2))
print("\nSaved in:", OUT)

Output hidden; open in https://colab.research.google.com to view.

In [47]:
# ================================================================
# STEP 89: CHECKPOINT AND TRAINING HISTORY INSPECTION
# ================================================================

import torch
import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu")
CHECKPOINT_DIR = BASE / "checkpoints"
HISTORY_PATH = BASE / "logs/training_history.json"

checkpoint_path = CHECKPOINT_DIR / "300.pth.tar"

print("=" * 60)
print("CHECKPOINT INSPECTION")
print("=" * 60)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint:", checkpoint_path)
print("Checkpoint size:",
      round(checkpoint_path.stat().st_size / (1024**2), 2), "MB")

print("\nCheckpoint keys:")
for key, value in checkpoint.items():
    if torch.is_tensor(value):
        print(key, "| Tensor", tuple(value.shape))
    elif isinstance(value, dict):
        print(key, "| Dictionary", len(value), "entries")
    else:
        print(key, "|", type(value).__name__, "|", value)

print("\n" + "=" * 60)
print("TRAINING HISTORY")
print("=" * 60)

if HISTORY_PATH.exists():
    with open(HISTORY_PATH, "r") as f:
        history = json.load(f)

    print(json.dumps(history, indent=2))
else:
    print("Training history not found.")

print("\n" + "=" * 60)
print("TRAINING FILES")
print("=" * 60)

for path in sorted(BASE.glob("**/*.py")):
    if "train" in path.name.lower():
        print(path)

print("\nSTEP 89 COMPLETE")

CHECKPOINT INSPECTION
Checkpoint: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/300.pth.tar
Checkpoint size: 89.8 MB

Checkpoint keys:
step | int | 300
model | Dictionary 139 entries
optimizer | Dictionary 2 entries
history | Dictionary 13 entries
speaker_map | Dictionary 2 entries
normalization | Dictionary 4 entries

TRAINING HISTORY
{
  "step": [
    100,
    200,
    300
  ],
  "train_total": [
    11.093661308288574,
    8.841996192932129,
    6.579644680023193
  ],
  "train_mel": [
    4.779876708984375,
    3.9302239418029785,
    2.7016289234161377
  ],
  "train_postnet_mel": [
    4.743826866149902,
    3.834397554397583,
    2.523439645767212
  ],
  "train_pitch": [
    0.7721270322799683,
    0.5067895650863647,
    0.7235786318778992
  ],
  "train_energy": [
    0.5193660259246826,
    0.339690625667572,
    0.39245596528053284
  ],
  "train_duration": [
    0.2784644067287445,
    0.23089446127414703,
    0.23854169249534607
  ],
  "validation_total": [


In [50]:
# ================================================================
# STEP 90: INSPECT OPTIMIZER AND TRAINING STATE
# ================================================================

import torch
import json
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu")
checkpoint_path = BASE / "checkpoints/300.pth.tar"

ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("=" * 60)
print("OPTIMIZER STATE")
print("=" * 60)

optimizer_state = ckpt["optimizer"]

print("Optimizer keys:", list(optimizer_state.keys()))

for key, value in optimizer_state.items():

    if isinstance(value, dict):
        print(f"\n{key}:")
        print("Entries:", len(value))

        if key == "param_groups":
            for group in value:
                print({
                    "lr": group.get("lr"),
                    "betas": group.get("betas"),
                    "eps": group.get("eps"),
                    "weight_decay": group.get("weight_decay")
                })

        elif key == "state":
            if value:
                first = next(iter(value.values()))
                print("First parameter state keys:", list(first.keys()))
                for k, v in first.items():
                    if torch.is_tensor(v):
                        print(k, tuple(v.shape))
                    else:
                        print(k, v)
            else:
                print("No optimizer states found.")

    else:
        print(key, value)

print("\n" + "=" * 60)
print("MODEL STATE")
print("=" * 60)

model_state = ckpt["model"]
print("Model parameter tensors:", len(model_state))

for key in list(model_state.keys())[:12]:
    value = model_state[key]
    print(key, tuple(value.shape) if torch.is_tensor(value) else type(value))

print("\n" + "=" * 60)
print("CHECKPOINT HISTORY")
print("=" * 60)

print(json.dumps(ckpt["history"], indent=2))

print("\n" + "=" * 60)
print("CURRENT RUNTIME OBJECTS")
print("=" * 60)

for name in [
    "model", "optimizer", "train_chunks",
    "val_chunks", "train_files", "val_files",
    "train_dataset", "val_dataset",
    "train_loader", "val_loader",
    "config", "preprocess_config"
]:
    if name in globals():
        obj = globals()[name]
        print(name, ":", type(obj).__name__)

print("\nSTEP 90 COMPLETE")

OPTIMIZER STATE
Optimizer keys: ['state', 'param_groups']

state:
Entries: 120
First parameter state keys: ['step', 'exp_avg', 'exp_avg_sq']
step ()
exp_avg (361, 128)
exp_avg_sq (361, 128)
param_groups [{'lr': 0.0001, 'betas': (0.9, 0.98), 'eps': 1e-09, 'weight_decay': 0.0, 'amsgrad': False, 'maximize': False, 'foreach': None, 'capturable': False, 'differentiable': False, 'fused': None, 'decoupled_weight_decay': True, 'params': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123]}]

MODEL STATE
Model parameter tensors: 139
encoder.posi

In [51]:
# ================================================================
# STEP 91: INSPECT EXISTING TRAINING FUNCTIONS
# ================================================================

import inspect

print("=" * 65)
print("AVAILABLE TRAINING FUNCTIONS")
print("=" * 65)

keywords = [
    "batch", "loss", "train", "valid",
    "dataset", "checkpoint", "save"
]

for name, obj in list(globals().items()):

    if not callable(obj):
        continue

    if not any(k in name.lower() for k in keywords):
        continue

    try:
        print("\nFUNCTION:", name)
        print("SIGNATURE:", inspect.signature(obj))

        if inspect.isfunction(obj):
            try:
                print(inspect.getsource(obj)[:7000])
            except Exception:
                print("Source unavailable.")

    except Exception as e:
        print("Inspection error:", str(e))


print("\n" + "=" * 65)
print("TRAINING FILES")
print("=" * 65)

print("Training files:", len(train_files))
print("Validation files:", len(val_files))

print("\nTraining sample:")
print(train_files[0])

print("\nValidation sample:")
print(val_files[0])

print("\n" + "=" * 65)
print("STEP 91 COMPLETE")
print("=" * 65)

AVAILABLE TRAINING FUNCTIONS

FUNCTION: FastSpeech2Loss
SIGNATURE: (preprocess_config, model_config)

FUNCTION: make_batch
SIGNATURE: (path)
def make_batch(path):

    sample = load_sample(path)

    speakers = torch.tensor(
        [sample["speaker"]], dtype=torch.long
    )

    texts = torch.tensor(
        sample["text"][None, :], dtype=torch.long
    )

    src_lens = torch.tensor(
        [len(sample["text"])], dtype=torch.long
    )

    mels = torch.tensor(
        sample["mel"][None, :, :], dtype=torch.float32
    )

    mel_lens = torch.tensor(
        [sample["mel"].shape[0]], dtype=torch.long
    )

    p_targets = torch.tensor(
        sample["pitch"][None, :], dtype=torch.float32
    )

    e_targets = torch.tensor(
        sample["energy"][None, :], dtype=torch.float32
    )

    d_targets = torch.tensor(
        sample["duration"][None, :], dtype=torch.long
    )

    return (
        [path.stem],
        [""],
        speakers,
        texts,
        src_lens,
        

In [52]:
import inspect
import numpy as np
from pathlib import Path

print("=" * 65)
print("1. LOAD_SAMPLE FUNCTION")
print("=" * 65)

print(inspect.signature(load_sample))
print(inspect.getsource(load_sample))

print("\n" + "=" * 65)
print("2. TRAINING FILE FORMAT")
print("=" * 65)

for path in [train_files[0], val_files[0]]:
    path = Path(path)
    print("\nFile:", path)
    data = np.load(path, allow_pickle=True)

    for key in data.files:
        value = data[key]
        if isinstance(value, np.ndarray):
            print(f"{key}: shape={value.shape}, dtype={value.dtype}")
        else:
            print(f"{key}: {value}")

print("\n" + "=" * 65)
print("3. TRAINING AND VALIDATION PATHS")
print("=" * 65)

print("Training files:", len(train_files))
print("Validation files:", len(val_files))

print("Training path:", train_files[0].parent)
print("Validation path:", val_files[0].parent)

print("\nPrepared training chunks:")
prepared_train = list(
    Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/fs2_prepared/train").glob("*.npz")
)
print("Count:", len(prepared_train))
print("Example:", prepared_train[0] if prepared_train else "None")

print("\n" + "=" * 65)
print("STEP 92 COMPLETE")
print("=" * 65)

1. LOAD_SAMPLE FUNCTION
(path)
def load_sample(path):

    with np.load(path, allow_pickle=True) as data:

        original_id = str(data["original_id"].item()).lower()

        mel = data["mel"].astype(np.float32).T
        pitch = data["pitch"].astype(np.float32)
        energy = data["energy"].astype(np.float32)

        text = data["text_ids"].astype(np.int64)
        duration = data["durations"].astype(np.int64)

    if "female" in original_id:
        speaker_id = speaker_map["female"]
    elif "male" in original_id:
        speaker_id = speaker_map["male"]
    else:
        raise ValueError(f"Unknown speaker: {original_id}")

    # Apply training-derived normalization.
    pitch = (pitch - pitch_mean) / pitch_std
    energy = (energy - energy_mean) / energy_std

    # Validate sample dimensions.
    assert mel.shape[0] == len(pitch)
    assert len(pitch) == len(energy)
    assert len(text) == len(duration)
    assert int(duration.sum()) == mel.shape[0]

    assert np.isfinite(me

In [53]:
# ================================================================
# STEP 93 — RESUME TELUGU FASTSPEECH2 TRAINING FROM STEP 300
# ================================================================

import random
import numpy as np
import torch
from pathlib import Path

# ------------------------------------------------
# 1. VERIFY REQUIRED OBJECTS
# ------------------------------------------------

required = [
    "model", "optimizer", "criterion", "history",
    "make_batch", "run_validation", "save_checkpoint",
    "speaker_map", "norm"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing required objects: {missing}. "
        "Do not start training until these are restored."
    )

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

CKPT_DIR = BASE / "checkpoints"
latest_ckpt = CKPT_DIR / "latest.pth.tar"

# ------------------------------------------------
# 2. USE THE CORRECT CHUNKED DATA
# ------------------------------------------------

train_files = sorted(
    (BASE / "fs2_prepared" / "train").glob("*.npz")
)

val_files = sorted(
    (BASE / "fs2_prepared" / "validation").glob("*.npz")
)

assert len(train_files) == 1520, (
    f"Expected 1520 training chunks, found {len(train_files)}"
)

assert len(val_files) == 301, (
    f"Expected 301 validation chunks, found {len(val_files)}"
)

# Confirm the loader can read both formats.
train_sample = load_sample(train_files[0])
val_sample = load_sample(val_files[0])

assert train_sample["mel"].shape[0] == int(
    train_sample["duration"].sum()
)
assert val_sample["mel"].shape[0] == int(
    val_sample["duration"].sum()
)

print("Training chunks:", len(train_files))
print("Validation chunks:", len(val_files))
print("Training sample:", train_files[0].name)
print("Validation sample:", val_files[0].name)

# ------------------------------------------------
# 3. RESTORE STEP-300 MODEL AND OPTIMIZER
# ------------------------------------------------

checkpoint_path = CKPT_DIR / "300.pth.tar"

assert checkpoint_path.exists(), (
    f"Checkpoint not found: {checkpoint_path}"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["step"] == 300, (
    f"Expected step 300, found {checkpoint['step']}"
)

model.load_state_dict(checkpoint["model"])
optimizer.load_state_dict(checkpoint["optimizer"])

history = checkpoint["history"]

print("\nRestored checkpoint:", checkpoint_path)
print("Starting step:", checkpoint["step"])
print("Optimizer state restored.")

# Ensure the model is on CPU.
device = next(model.parameters()).device
print("Model device:", device)

# ------------------------------------------------
# 4. CONFIGURATION
# ------------------------------------------------

START_STEP = 300
ADDITIONAL_STEPS = 300
END_STEP = START_STEP + ADDITIONAL_STEPS

VALIDATION_INTERVAL = 100
VALIDATION_SAMPLES = 24

model.train()

# ------------------------------------------------
# 5. RESUME TRAINING
# ------------------------------------------------

print("\nStarting resumed training...")
print(f"Steps: {START_STEP + 1} to {END_STEP}")

for step in range(START_STEP + 1, END_STEP + 1):

    # Randomly select one prepared training chunk.
    path = random.choice(train_files)

    batch = make_batch(path)

    optimizer.zero_grad(set_to_none=True)

    predictions = model(*(batch[2:]))

    losses = criterion(batch, predictions)

    total_loss = sum(losses)

    if not torch.isfinite(total_loss):
        raise RuntimeError(
            f"Non-finite loss at step {step}: {total_loss.item()}"
        )

    total_loss.backward()
    optimizer.step()

    # ------------------------------------------------
    # VALIDATION AND CHECKPOINTING
    # ------------------------------------------------

    if step % VALIDATION_INTERVAL == 0:

        model.eval()

        validation_losses = run_validation()

        model.train()

        train_values = [
            float(loss.detach().item())
            for loss in losses
        ]

        history["step"].append(step)

        names = [
            "total",
            "mel",
            "postnet_mel",
            "pitch",
            "energy",
            "duration"
        ]

        # Training total is the sum of component losses.
        history.setdefault("train_total", []).append(
            float(total_loss.detach().item())
        )

        for i, name in enumerate(names[1:]):
            history.setdefault(f"train_{name}", []).append(
                train_values[i]
            )

        history.setdefault("val_total", []).append(
            float(sum(validation_losses))
        )

        for i, name in enumerate(names[1:]):
            history.setdefault(f"val_{name}", []).append(
                float(validation_losses[i])
            )

        print("\n" + "=" * 55)
        print(f"STEP {step}")
        print(f"Training total:   {total_loss.item():.4f}")
        print(f"Validation total: {sum(validation_losses):.4f}")

        for i, name in enumerate(names[1:]):
            print(
                f"{name:16s} "
                f"Train={train_values[i]:.4f} "
                f"Val={validation_losses[i]:.4f}"
            )

        save_checkpoint(step)

        print("=" * 55)

print("\nSTEP 93 COMPLETE")
print("Final step:", END_STEP)
print("Latest checkpoint:", latest_ckpt)

Training chunks: 1520
Validation chunks: 301
Training sample: train_telugufemale_00009_chunk000.npz
Validation sample: train_telugufemale_00060_chunk000.npz

Restored checkpoint: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/300.pth.tar
Starting step: 300
Optimizer state restored.
Model device: cpu

Starting resumed training...
Steps: 301 to 600

STEP 400
Training total:   12.6099
Validation total: 12.5419
mel              Train=6.3049 Val=6.2710
postnet_mel      Train=2.6675 Val=2.6405
pitch            Train=2.5670 Val=2.5353
energy           Train=0.4768 Val=0.5634
duration         Train=0.4488 Val=0.4007
Checkpoint saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/latest.pth.tar

STEP 500
Training total:   12.6797
Validation total: 12.4244
mel              Train=6.3399 Val=6.2122
postnet_mel      Train=2.7415 Val=2.6400
pitch            Train=2.6171 Val=2.5435
energy           Train=0.5172 Val=0.5171
duration         Train=0.3183 Val=0.3929
C

In [54]:
# ================================================================
# STEP 94 — FAIR CHECKPOINT COMPARISON
# Evaluate steps 300, 400, 500 and 600 on ALL validation chunks.
# ================================================================

import torch
import numpy as np
from pathlib import Path

checkpoint_dir = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints"
)

checkpoint_steps = [300, 400, 500, 600]

results = {}

for step in checkpoint_steps:

    path = checkpoint_dir / f"{step}.pth.tar"

    if not path.exists():
        print("Missing checkpoint:", path)
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(ckpt["model"])
    model.eval()

    all_losses = []

    with torch.no_grad():

        for sample_path in val_files:

            batch = make_batch(sample_path)
            predictions = model(*(batch[2:]))

            losses = criterion(batch, predictions)

            all_losses.append([
                float(x.item()) for x in losses
            ])

    mean_losses = np.mean(all_losses, axis=0)

    results[step] = {
        "mel": float(mean_losses[0]),
        "postnet_mel": float(mean_losses[1]),
        "pitch": float(mean_losses[2]),
        "energy": float(mean_losses[3]),
        "duration": float(mean_losses[4]),
        "total": float(mean_losses.sum())
    }

    print("\n" + "=" * 55)
    print("CHECKPOINT:", step)
    for name, value in results[step].items():
        print(f"{name:15s}: {value:.5f}")

# ------------------------------------------------
# SELECT BEST CHECKPOINT BY FULL VALIDATION LOSS
# ------------------------------------------------

best_step = min(
    results,
    key=lambda s: results[s]["total"]
)

best_path = checkpoint_dir / f"{best_step}.pth.tar"
best_ckpt = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(best_ckpt["model"])
optimizer.load_state_dict(best_ckpt["optimizer"])
model.eval()

print("\n" + "=" * 55)
print("BEST CHECKPOINT:", best_step)
print("FULL VALIDATION LOSS:", results[best_step]["total"])
print("Best model restored in memory.")
print("=" * 55)

print("\nSTEP 94 COMPLETE")


CHECKPOINT: 300
mel            : 6.72942
postnet_mel    : 2.63118
pitch          : 3.02650
energy         : 0.53458
duration       : 0.40391
total          : 13.45884

CHECKPOINT: 400
mel            : 6.23921
postnet_mel    : 2.63118
pitch          : 2.53629
energy         : 0.53458
duration       : 0.40391
total          : 12.47842

CHECKPOINT: 500
mel            : 6.21427
postnet_mel    : 2.63118
pitch          : 2.51135
energy         : 0.53458
duration       : 0.40391
total          : 12.42854

CHECKPOINT: 600
mel            : 6.26125
postnet_mel    : 2.63118
pitch          : 2.55834
energy         : 0.53458
duration       : 0.40391
total          : 12.52251

BEST CHECKPOINT: 500
FULL VALIDATION LOSS: 12.42853794407805
Best model restored in memory.

STEP 94 COMPLETE


In [2]:
from pathlib import Path
from google.colab import drive

# Mount Drive if it is not already mounted.
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

print("Assignment folder exists:", ROOT.exists())

# Search for checkpoint 500.
matches = list(ROOT.rglob("500.pth.tar"))

print("\nCheckpoint 500 matches:")
for path in matches:
    print(path, "| Size:", round(path.stat().st_size / (1024**2), 2), "MB")

# Also locate latest checkpoint.
latest = list(ROOT.rglob("latest.pth.tar"))

print("\nLatest checkpoints:")
for path in latest:
    print(path)

Mounted at /content/drive
Assignment folder exists: True

Checkpoint 500 matches:
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/500.pth.tar | Size: 89.8 MB

Latest checkpoints:
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints/latest.pth.tar


In [5]:
from pathlib import Path
import torch

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

print("Checkpoint files:")
for p in sorted((BASE / "checkpoints").glob("*.pth.tar")):
    print(p.name, round(p.stat().st_size / (1024**2), 2), "MB")

print("\nConfiguration files:")
for p in [
    BASE / "config" / "preprocess.yaml",
    BASE / "config" / "model.yaml",
    BASE / "config" / "train.yaml"
]:
    print(p, "EXISTS" if p.exists() else "MISSING")

print("\nRepository:", Path("/content/FastSpeech2").exists())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Checkpoint files:
100.pth.tar 89.8 MB
200.pth.tar 89.8 MB
300.pth.tar 89.8 MB
400.pth.tar 89.8 MB
500.pth.tar 89.8 MB
600.pth.tar 89.81 MB
latest.pth.tar 89.81 MB

Configuration files:
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/config/preprocess.yaml EXISTS
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/config/model.yaml EXISTS
/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/config/train.yaml EXISTS

Repository: False
PyTorch: 2.11.0+cpu
CUDA: False


In [7]:
# ================================================================
# STEP 95C — RESTORE REPOSITORY AND INSPECT CONFIGURATION
# ================================================================

import os
import sys
import subprocess
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

REPO = Path("/content/FastSpeech2")

# 1. Clone repository again
if not REPO.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/ming024/FastSpeech2.git",
         str(REPO)],
        check=True
    )

os.chdir(REPO)
sys.path.insert(0, str(REPO))

# 2. Install missing text dependency
subprocess.run(
    [sys.executable, "-m", "pip", "install", "Unidecode", "-q"],
    check=True
)

# 3. Load saved YAML configurations
import yaml

for name in ["preprocess", "model", "train"]:
    path = BASE / "config" / f"{name}.yaml"

    with open(path) as f:
        config_data = yaml.safe_load(f)

    print(f"\n{name.upper()} CONFIGURATION")
    print(config_data)

# 4. Inspect model constructor
import inspect
from model.fastspeech2 import FastSpeech2

print("\nMODEL CONSTRUCTOR:")
print(inspect.signature(FastSpeech2))

print("\nMODEL INITIALIZATION SOURCE:")
print(inspect.getsource(FastSpeech2.__init__))

# 5. Verify checkpoint
checkpoint_path = BASE / "checkpoints" / "500.pth.tar"

assert checkpoint_path.exists()
print("\nCheckpoint 500 verified:", checkpoint_path)
print("Repository restored successfully.")
print("\nSTEP 95C COMPLETE")


PREPROCESS CONFIGURATION
{'dataset': 'Telugu_MMS_FA', 'path': {'corpus_path': '/content/drive/MyDrive/FastSpeech2_Assignment/Telugu', 'lexicon_path': '', 'raw_path': '/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio', 'preprocessed_path': '/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/fs2_prepared/model_metadata'}, 'preprocessing': {'val_size': 0, 'text': {'text_cleaners': ['basic_cleaners'], 'language': 'te_romanized'}, 'audio': {'sampling_rate': 22050, 'max_wav_value': 32768.0}, 'stft': {'filter_length': 1024, 'hop_length': 256, 'win_length': 1024}, 'mel': {'n_mel_channels': 80, 'mel_fmin': 0, 'mel_fmax': 8000}, 'pitch': {'feature': 'frame_level', 'normalization': True}, 'energy': {'feature': 'frame_level', 'normalization': True}}}

MODEL CONFIGURATION
{'transformer': {'encoder_layer': 2, 'encoder_head': 2, 'encoder_hidden': 128, 'decoder_layer': 2, 'decoder_head': 2, 'decoder_hidden': 128, 'conv_filter_size': 512, 'conv_kernel_size': [9, 1], 'encoder_dropout': 

In [9]:
# ================================================================
# STEP 95D — REBUILD FASTSPEECH2 AND RESTORE CHECKPOINT 500
# ================================================================

import json
import yaml
import torch
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

REPO = Path("/content/FastSpeech2")

# 1. Load configurations
with open(BASE / "config" / "preprocess.yaml") as f:
    preprocess_config = yaml.safe_load(f)

with open(BASE / "config" / "model.yaml") as f:
    model_config = yaml.safe_load(f)

# 2. Ensure repository imports work
import sys
sys.path.insert(0, str(REPO))

# 3. Rebuild FastSpeech2 model
from model.fastspeech2 import FastSpeech2

model = FastSpeech2(
    preprocess_config,
    model_config
)

model.cpu()

# 4. Restore checkpoint 500
checkpoint_path = BASE / "checkpoints" / "500.pth.tar"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("Model successfully reconstructed.")
print("Checkpoint:", checkpoint["step"])
print("Device:", next(model.parameters()).device)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Model evaluation mode:", not model.training)

# 5. Verify speaker mapping
speaker_path = (
    BASE / "fs2_prepared" / "model_metadata" / "speakers.json"
)

with open(speaker_path) as f:
    speaker_map = json.load(f)

print("Speaker map:", speaker_map)

print("\nSTEP 95D COMPLETE")

Model successfully reconstructed.
Checkpoint: 500
Device: cpu
Parameters: 8183361
Model evaluation mode: True
Speaker map: {'female': 0, 'male': 1}

STEP 95D COMPLETE


In [14]:
import uroman
print("Module file:", uroman.__file__)
print("Available names:", [x for x in dir(uroman) if "roman" in x.lower()])

Module file: None
Available names: []


In [17]:
import os
import sys
from pathlib import Path

ROOT = Path("/content/uroman")

print("Repository exists:", ROOT.exists())

if ROOT.exists():
    print("\nTop-level files and folders:")
    for item in ROOT.iterdir():
        print(item.name, "DIR" if item.is_dir() else "FILE")

    print("\nPython files containing Uroman:")
    for file in ROOT.rglob("*.py"):
        try:
            content = file.read_text(errors="ignore")
            if "class Uroman" in content:
                print("\nFOUND:", file)
        except Exception:
            pass

print("\nPython import paths:")
for p in sys.path:
    if "uroman" in p.lower():
        print(p)

Repository exists: True

Top-level files and folders:
requirements.txt FILE
pyproject.toml FILE
.gitignore FILE
uroman DIR
README.md FILE
text DIR
how-to-release.md FILE
lib DIR
.git DIR
LICENSE.txt FILE
uroman.ipynb FILE
test DIR

Python files containing Uroman:

FOUND: /content/uroman/uroman/uroman.py

Python import paths:
/content/uroman
/content/uroman
/content/uroman
/content/uroman
/content/uroman
/content/uroman


In [20]:
# ================================================================
# STEP 95H — FIX UROMAN MODULE PATH AND CACHE
# ================================================================

import sys
import importlib
from pathlib import Path

# Actual folder containing uroman.py
UROMAN_PACKAGE = Path("/content/uroman/uroman")

assert (UROMAN_PACKAGE / "uroman.py").exists(), (
    "uroman.py not found!"
)

# Clear previously cached incorrect namespace module.
for name in list(sys.modules):
    if name == "uroman" or name.startswith("uroman."):
        del sys.modules[name]

# Put the actual implementation folder first.
sys.path.insert(0, str(UROMAN_PACKAGE))
importlib.invalidate_caches()

# Import Uroman from the actual Python file.
from uroman import Uroman

uroman = Uroman()

test_text = "పులి అడవిలో నిశ్శబ్దంగా నడిచింది."
romanized = uroman.romanize_string(test_text)

print("Original:", test_text)
print("Romanized:", romanized)
print("Imported from:", sys.modules["uroman"].__file__)
print("STEP 95H COMPLETE")

Original: పులి అడవిలో నిశ్శబ్దంగా నడిచింది.
Romanized: puli addaviloo nishshabdamgaa naddicimdi.
Imported from: /content/uroman/uroman/uroman.py
STEP 95H COMPLETE


In [22]:
# ================================================================
# STEP 95I — RESTORE UNIVERSAL HIFIGAN VOCODER
# ================================================================

import zipfile
from pathlib import Path
import json

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

REPO = Path("/content/FastSpeech2")

# Original vocoder archive in the cloned repository
archive = REPO / "hifigan" / "generator_universal.pth.tar.zip"

# Destination
vocoder_dir = BASE / "vocoder" / "generator_universal"
vocoder_dir.mkdir(parents=True, exist_ok=True)

assert archive.exists(), f"Archive missing: {archive}"

# Extract checkpoint if needed
checkpoint_path = vocoder_dir / "generator_universal.pth.tar"

if not checkpoint_path.exists():
    with zipfile.ZipFile(archive, "r") as z:
        z.extractall(vocoder_dir)

# Find config.json in the repository
config_candidates = list(
    (REPO / "hifigan").rglob("config.json")
)

assert config_candidates, "HiFi-GAN config.json not found"

source_config = config_candidates[0]
destination_config = vocoder_dir / "config.json"

import shutil
shutil.copy2(source_config, destination_config)

print("Vocoder checkpoint:", checkpoint_path.exists())
print("Vocoder config:", destination_config.exists())
print("Config source:", source_config)

with open(destination_config) as f:
    config = json.load(f)

print("Sample rate:", config.get("sampling_rate"))
print("Mel channels:", config.get("num_mels"))
print("\nSTEP 95I COMPLETE")

Vocoder checkpoint: True
Vocoder config: True
Config source: /content/FastSpeech2/hifigan/config.json
Sample rate: 22050
Mel channels: 80

STEP 95I COMPLETE


In [23]:
from hifigan.models import Generator

In [24]:
# ================================================================
# STEP 95 — TEST TELUGU SPEECH USING CHECKPOINT 500
# ================================================================

import torch
import numpy as np
import soundfile as sf
from pathlib import Path
from types import SimpleNamespace
from scipy.signal import resample_poly

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

# ------------------------------------------------
# 1. RESTORE CHECKPOINT 500
# ------------------------------------------------

checkpoint = torch.load(
    BASE / "checkpoints" / "500.pth.tar",
    map_location="cpu",
    weights_only=False
)

model.load_state_dict(checkpoint["model"])
model.eval()

print("Loaded checkpoint:", checkpoint["step"])

# ------------------------------------------------
# 2. PREPARE TEST TEXT
# ------------------------------------------------

original_text = "పులి అడవిలో నిశ్శబ్దంగా నడిచింది."

import sys
sys.path.insert(0, "/content/uroman")

import uroman as ur
uroman = ur.Uroman()

romanized_text = uroman.romanize_string(original_text)

from text.symbols import symbols

symbol_to_id = {
    symbol: idx for idx, symbol in enumerate(symbols)
}

tokens = list(romanized_text)

text_ids = np.array(
    [symbol_to_id[ch] for ch in tokens],
    dtype=np.int64
)

print("Original:", original_text)
print("Romanized:", romanized_text)
print("Token count:", len(text_ids))

# ------------------------------------------------
# 3. GENERATE FASTSPEECH2 MEL
# ------------------------------------------------

text_tensor = torch.tensor(
    text_ids[None, :],
    dtype=torch.long
)

src_lens = torch.tensor(
    [len(text_ids)],
    dtype=torch.long
)

speakers = torch.tensor([0], dtype=torch.long)

with torch.no_grad():

    output = model(
        speakers,
        text_tensor,
        src_lens,
        len(text_ids)
    )

    mel_prediction = output[1][0].cpu().numpy()

    predicted_pitch = output[2][0].cpu().numpy()
    predicted_energy = output[3][0].cpu().numpy()
    predicted_duration = output[4][0].cpu().numpy()

print("Mel shape:", mel_prediction.shape)
print("Mel range:", mel_prediction.min(), mel_prediction.max())
print("Mel mean:", mel_prediction.mean())
print("Mel std:", mel_prediction.std())

# ------------------------------------------------
# 4. LOAD UNIVERSAL HIFIGAN VOCODER
# ------------------------------------------------

import json
from hifigan.models import Generator

vocoder_dir = BASE / "vocoder" / "generator_universal"

with open(vocoder_dir / "config.json") as f:
    vocoder_config = SimpleNamespace(**json.load(f))

generator = Generator(vocoder_config)

vocoder_checkpoint = torch.load(
    vocoder_dir / "generator_universal.pth.tar",
    map_location="cpu",
    weights_only=False
)

generator.load_state_dict(
    vocoder_checkpoint["generator"]
)

generator.eval()
generator.remove_weight_norm()

# ------------------------------------------------
# 5. CONVERT MEL TO AUDIO
# ------------------------------------------------

mel_tensor = torch.tensor(
    mel_prediction.T[None, :, :],
    dtype=torch.float32
)

with torch.no_grad():
    waveform = generator(mel_tensor).squeeze().cpu().numpy()

peak = np.max(np.abs(waveform))

if peak > 0:
    waveform = waveform / peak * 0.95

# ------------------------------------------------
# 6. SAVE AUDIO AND PREDICTIONS
# ------------------------------------------------

output_dir = BASE / "results" / "test_generated_audios"
output_dir.mkdir(parents=True, exist_ok=True)

audio_path = output_dir / "T3_checkpoint500.wav"

sf.write(
    audio_path,
    waveform,
    22050
)

prediction_dir = BASE / "inference" / "test_predictions"
prediction_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    prediction_dir / "T3_checkpoint500_predictions.npz",
    tokens=np.array(tokens),
    mel=mel_prediction,
    pitch=predicted_pitch,
    energy=predicted_energy,
    duration=predicted_duration
)

print("\nAUDIO SAVED:", audio_path)
print("Duration:", round(len(waveform) / 22050, 2), "seconds")
print("Sample rate: 22050 Hz")
print("STEP 95 COMPLETE")

Loaded checkpoint: 500
Original: పులి అడవిలో నిశ్శబ్దంగా నడిచింది.
Romanized: puli addaviloo nishshabdamgaa naddicimdi.
Token count: 41
Mel shape: (253, 80)
Mel range: -6.413075 -1.2781682
Mel mean: -3.240225
Mel std: 0.90477926


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...

AUDIO SAVED: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/test_generated_audios/T3_checkpoint500.wav
Duration: 2.94 seconds
Sample rate: 22050 Hz
STEP 95 COMPLETE


In [25]:
from IPython.display import Audio, display
display(Audio(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/results/test_generated_audios/T3_checkpoint500.wav"
))

In [26]:
# ================================================================
# STEP 96 — COMPARE PREDICTED MEL WITH REAL TELUGU MEL
# ================================================================

import numpy as np
import torch
import inspect
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

# 1. Inspect FastSpeech2 output implementation
print("=" * 60)
print("1. FASTSPEECH2 FORWARD IMPLEMENTATION")
print("=" * 60)

print(inspect.getsource(model.forward))

# 2. Load actual Telugu training mel
real_path = (
    BASE / "fs2_prepared" / "train" /
    "train_telugufemale_00009_chunk000.npz"
)

with np.load(real_path, allow_pickle=True) as data:
    real_mel = data["mel"].astype(np.float32).T

# 3. Load checkpoint-500 predicted mel
pred_path = (
    BASE / "inference" / "test_predictions" /
    "T3_checkpoint500_predictions.npz"
)

with np.load(pred_path, allow_pickle=True) as data:
    predicted_mel = data["mel"].astype(np.float32)

# 4. Compare distributions
def report(name, array):
    print(f"\n{name}")
    print("Shape:", array.shape)
    print("Min:", float(array.min()))
    print("Max:", float(array.max()))
    print("Mean:", float(array.mean()))
    print("Std:", float(array.std()))
    print("Median:", float(np.median(array)))
    print("25th percentile:", float(np.percentile(array, 25)))
    print("75th percentile:", float(np.percentile(array, 75)))
    print("Finite:", bool(np.isfinite(array).all()))

print("\n" + "=" * 60)
print("2. MEL DISTRIBUTION COMPARISON")
print("=" * 60)

report("REAL TRAINING MEL", real_mel)
report("PREDICTED T3 MEL", predicted_mel)

# 5. Check output ordering and shapes from model source
print("\n" + "=" * 60)
print("3. MODEL OUTPUT RETURN STATEMENT")
print("=" * 60)

source = inspect.getsource(model.forward)

for line in source.splitlines():
    if "return " in line:
        print(line.strip())

print("\nSTEP 96 COMPLETE")

1. FASTSPEECH2 FORWARD IMPLEMENTATION
    def forward(
        self,
        speakers,
        texts,
        src_lens,
        max_src_len,
        mels=None,
        mel_lens=None,
        max_mel_len=None,
        p_targets=None,
        e_targets=None,
        d_targets=None,
        p_control=1.0,
        e_control=1.0,
        d_control=1.0,
    ):
        src_masks = get_mask_from_lengths(src_lens, max_src_len)
        mel_masks = (
            get_mask_from_lengths(mel_lens, max_mel_len)
            if mel_lens is not None
            else None
        )

        output = self.encoder(texts, src_masks)

        if self.speaker_emb is not None:
            output = output + self.speaker_emb(speakers).unsqueeze(1).expand(
                -1, max_src_len, -1
            )

        (
            output,
            p_predictions,
            e_predictions,
            log_d_predictions,
            d_rounded,
            mel_lens,
            mel_masks,
        ) = self.variance_ad

In [27]:
# ================================================================
# STEP 97 — TEACHER-FORCED ACOUSTIC MODEL DIAGNOSTIC
# ================================================================

import numpy as np
import torch
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Telugu"
)

sample_path = (
    BASE / "fs2_prepared" / "train" /
    "train_telugufemale_00009_chunk000.npz"
)

# Normalization values used during training
pitch_mean = 109.76953432377077
pitch_std = 103.30357441258384
energy_mean = 0.05246112019682791
energy_std = 0.05535232817865627

# ------------------------------------------------
# 1. LOAD ONE PREPARED TRAINING CHUNK
# ------------------------------------------------

with np.load(sample_path, allow_pickle=True) as data:

    mel = data["mel"].astype(np.float32).T
    text = data["text_ids"].astype(np.int64)
    pitch = data["pitch"].astype(np.float32)
    energy = data["energy"].astype(np.float32)
    duration = data["durations"].astype(np.int64)
    original_id = str(data["original_id"].item()).lower()

speaker_id = 0 if "female" in original_id else 1

# Normalize pitch and energy using training statistics
pitch = (pitch - pitch_mean) / pitch_std
energy = (energy - energy_mean) / energy_std

assert mel.shape[0] == len(pitch) == len(energy)
assert len(text) == len(duration)
assert duration.sum() == mel.shape[0]

# ------------------------------------------------
# 2. CREATE TENSORS
# ------------------------------------------------

speakers = torch.tensor([speaker_id], dtype=torch.long)
texts = torch.tensor(text[None, :], dtype=torch.long)
src_lens = torch.tensor([len(text)], dtype=torch.long)

mels = torch.tensor(mel[None, :, :], dtype=torch.float32)
mel_lens = torch.tensor([mel.shape[0]], dtype=torch.long)

p_targets = torch.tensor(pitch[None, :], dtype=torch.float32)
e_targets = torch.tensor(energy[None, :], dtype=torch.float32)
d_targets = torch.tensor(duration[None, :], dtype=torch.long)

# ------------------------------------------------
# 3. TEACHER-FORCED MODEL FORWARD
# ------------------------------------------------

model.eval()

with torch.no_grad():

    forced_output = model(
        speakers,
        texts,
        src_lens,
        len(text),
        mels,
        mel_lens,
        mel.shape[0],
        p_targets,
        e_targets,
        d_targets
    )

# Model returns:
# 0 = mel before postnet
# 1 = mel after postnet
# 2 = pitch predictions
# 3 = energy predictions
# 4 = log-duration predictions
# 5 = rounded durations

mel_before_postnet = forced_output[0][0].cpu().numpy()
mel_after_postnet = forced_output[1][0].cpu().numpy()

# ------------------------------------------------
# 4. DISTRIBUTION REPORT
# ------------------------------------------------

def report(name, x):
    x = np.asarray(x)
    print(f"\n{name}")
    print("Shape:", x.shape)
    print("Min:", float(x.min()))
    print("Max:", float(x.max()))
    print("Mean:", float(x.mean()))
    print("Std:", float(x.std()))
    print("Finite:", bool(np.isfinite(x).all()))

print("=" * 60)
print("TEACHER-FORCED MEL COMPARISON")
print("=" * 60)

report("REAL TARGET MEL", mel)
report("MODEL MEL BEFORE POSTNET", mel_before_postnet)
report("MODEL MEL AFTER POSTNET", mel_after_postnet)

# ------------------------------------------------
# 5. CHECK PREDICTED VARIANCE ADAPTOR VALUES
# ------------------------------------------------

report(
    "PREDICTED PITCH",
    forced_output[2][0].cpu().numpy()
)

report(
    "PREDICTED ENERGY",
    forced_output[3][0].cpu().numpy()
)

report(
    "PREDICTED DURATION",
    forced_output[4][0].cpu().numpy()
)

print("\nTarget mel frames:", mel.shape[0])
print("Target text tokens:", len(text))
print("Duration sum:", int(duration.sum()))

print("\nSTEP 97 COMPLETE")

TEACHER-FORCED MEL COMPARISON

REAL TARGET MEL
Shape: (1509, 80)
Min: -11.512925148010254
Max: 1.1536953449249268
Mean: -6.723828315734863
Std: 2.6046833992004395
Finite: True

MODEL MEL BEFORE POSTNET
Shape: (1509, 80)
Min: -5.863694190979004
Max: -2.849274158477783
Mean: -4.451935291290283
Std: 0.5260211825370789
Finite: True

MODEL MEL AFTER POSTNET
Shape: (1509, 80)
Min: -7.616171836853027
Max: -1.3390034437179565
Mean: -4.367366790771484
Std: 1.4938241243362427
Finite: True

PREDICTED PITCH
Shape: (1509,)
Min: -0.9848856925964355
Max: 1.4197713136672974
Mean: 0.031744930893182755
Std: 0.7323170304298401
Finite: True

PREDICTED ENERGY
Shape: (1509,)
Min: -0.8639967441558838
Max: 2.112699270248413
Mean: 0.26049989461898804
Std: 0.9508087635040283
Finite: True

PREDICTED DURATION
Shape: (210,)
Min: 1.5442537069320679
Max: 4.312723636627197
Mean: 1.8482617139816284
Std: 0.350679874420166
Finite: True

Target mel frames: 1509
Target text tokens: 210
Duration sum: 1509

STEP 97 COMPLETE

In [28]:
# ============================================================
# STEP 98 — TEACHER-FORCED MEL RECONSTRUCTION DIAGNOSTICS
# RUN NOW — NEW CELL
# DO NOT RETRAIN OR REGENERATE AUDIO
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

target = mel.astype(np.float32)
before = mel_before_postnet.astype(np.float32)
after = mel_after_postnet.astype(np.float32)

def metrics(name, prediction):
    error = prediction - target

    mae = np.mean(np.abs(error))
    rmse = np.sqrt(np.mean(error ** 2))
    correlation = np.corrcoef(
        target.flatten(), prediction.flatten()
    )[0, 1]

    print(f"\n{name}")
    print("MAE:", round(float(mae), 5))
    print("RMSE:", round(float(rmse), 5))
    print("Correlation:", round(float(correlation), 5))
    print("Prediction mean:", round(float(prediction.mean()), 5))
    print("Prediction std:", round(float(prediction.std()), 5))

print("=" * 60)
print("TEACHER-FORCED RECONSTRUCTION METRICS")
print("=" * 60)

metrics("BEFORE POSTNET", before)
metrics("AFTER POSTNET", after)

# Per-mel-bin statistics
target_bin_mean = target.mean(axis=0)
after_bin_mean = after.mean(axis=0)

plt.figure(figsize=(12, 5))
plt.plot(target_bin_mean, label="Real target")
plt.plot(after_bin_mean, label="Model after postnet")
plt.title("Mean Mel Value per Mel Bin")
plt.xlabel("Mel Bin")
plt.ylabel("Mean Log-Mel Value")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plot_path = (
    "/content/drive/MyDrive/FastSpeech2_Assignment/"
    "Telugu/inference_smoke_test/step98_mel_bin_comparison.png"
)

plt.savefig(plot_path, dpi=150)
plt.show()

print("\nSaved:", plot_path)

# Compare spectrograms
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

for ax, data, title in zip(
    axes,
    [target, before, after],
    [
        "Real Target Mel",
        "Model Mel Before Postnet",
        "Model Mel After Postnet"
    ]
):
    ax.imshow(data.T, origin="lower", aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("Time Frame")
    ax.set_ylabel("Mel Bin")

plt.tight_layout()
plt.show()

print("\nSTEP 98 COMPLETE")

TEACHER-FORCED RECONSTRUCTION METRICS

BEFORE POSTNET
MAE: 2.80841
RMSE: 3.38889
Correlation: 0.26933
Prediction mean: -4.45194
Prediction std: 0.52602

AFTER POSTNET
MAE: 2.56274
RMSE: 3.0305
Correlation: 0.69198
Prediction mean: -4.36737
Prediction std: 1.49382

Saved: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/inference_smoke_test/step98_mel_bin_comparison.png

STEP 98 COMPLETE


In [29]:
# ============================================================
# STEP 99 — INSPECT FASTSPEECH2 LOSS IMPLEMENTATION
# RUN NOW — NEW CELL
# NO TRAINING / NO CHECKPOINT MODIFICATIONS
# ============================================================

import inspect
import sys
from pathlib import Path

REPO = Path("/content/FastSpeech2")
sys.path.insert(0, str(REPO))

from model.loss import FastSpeech2Loss

print("=" * 65)
print("FASTSpeech2 LOSS CLASS")
print("=" * 65)

print(inspect.getsource(FastSpeech2Loss))

print("\n" + "=" * 65)
print("LOSS FORWARD METHOD")
print("=" * 65)

print(inspect.getsource(FastSpeech2Loss.forward))

print("\n" + "=" * 65)
print("CURRENT TRAINING CONFIGURATION")
print("=" * 65)

print("Batch size:", train_config["optimizer"]["batch_size"]
      if "train_config" in globals() else "Inspect config manually")

print("\nSTEP 99 COMPLETE")

FASTSpeech2 LOSS CLASS
class FastSpeech2Loss(nn.Module):
    """ FastSpeech2 Loss """

    def __init__(self, preprocess_config, model_config):
        super(FastSpeech2Loss, self).__init__()
        self.pitch_feature_level = preprocess_config["preprocessing"]["pitch"][
            "feature"
        ]
        self.energy_feature_level = preprocess_config["preprocessing"]["energy"][
            "feature"
        ]
        self.mse_loss = nn.MSELoss()
        self.mae_loss = nn.L1Loss()

    def forward(self, inputs, predictions):
        (
            mel_targets,
            _,
            _,
            pitch_targets,
            energy_targets,
            duration_targets,
        ) = inputs[6:]
        (
            mel_predictions,
            postnet_mel_predictions,
            pitch_predictions,
            energy_predictions,
            log_duration_predictions,
            _,
            src_masks,
            mel_masks,
            _,
            _,
        ) = predictions

In [31]:
# ============================================================
# STEP 100A — CHECK CURRENT RUNTIME FUNCTIONS
# RUN NOW — INSPECTION ONLY
# ============================================================

import inspect

function_names = [
    "make_batch",
    "load_sample",
    "run_validation",
    "save_checkpoint",
]

for name in function_names:
    obj = globals().get(name)

    print("\n" + "=" * 55)
    print(name)

    if obj is None:
        print("NOT DEFINED in current runtime")
    else:
        print(inspect.getsource(obj))

print("\n" + "=" * 55)
print("AVAILABLE DATA VARIABLES")

for name in ["train_files", "val_files", "model", "optimizer",
             "criterion", "preprocess_config", "model_config"]:

    obj = globals().get(name)

    if obj is None:
        print(name, ": NOT DEFINED")
    elif isinstance(obj, (list, tuple)):
        print(name, ": available, count =", len(obj))
    else:
        print(name, ": available, type =", type(obj).__name__)

print("\nSTEP 100A COMPLETE")


make_batch
NOT DEFINED in current runtime

load_sample
NOT DEFINED in current runtime

run_validation
NOT DEFINED in current runtime

save_checkpoint
NOT DEFINED in current runtime

AVAILABLE DATA VARIABLES
train_files : NOT DEFINED
val_files : NOT DEFINED
model : available, type = FastSpeech2
optimizer : NOT DEFINED
criterion : NOT DEFINED
preprocess_config : available, type = dict
model_config : available, type = dict

STEP 100A COMPLETE


In [32]:
# ============================================================
# STEP 100B — RESTORE TRAINING DATA REFERENCES
# RUN NOW — NO TRAINING
# ============================================================

from pathlib import Path
import json
import torch
import numpy as np

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Telugu")

TRAIN_DIR = BASE / "fs2_prepared" / "train"
VAL_DIR = BASE / "fs2_prepared" / "validation"
CKPT_DIR = BASE / "checkpoints"

# Rebuild file lists from prepared chunks
train_files = sorted(TRAIN_DIR.glob("*.npz"))
val_files = sorted(VAL_DIR.glob("*.npz"))

print("=" * 60)
print("DATA FILES")
print("=" * 60)

print("Training chunks:", len(train_files))
print("Validation chunks:", len(val_files))

print("\nTraining examples:")
for f in train_files[:5]:
    print(f.name)

print("\nValidation examples:")
for f in val_files[:5]:
    print(f.name)

# Check checkpoint availability
print("\n" + "=" * 60)
print("CHECKPOINTS")
print("=" * 60)

for f in sorted(CKPT_DIR.glob("*.pth.tar")):
    print(f.name, f"{f.stat().st_size / (1024**2):.2f} MB")

# Inspect the existing checkpoint without changing the model
checkpoint_path = CKPT_DIR / "500.pth.tar"
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint step:", checkpoint.get("step"))
print("Checkpoint keys:", list(checkpoint.keys()))
print("Saved speaker map:", checkpoint.get("speaker_map"))
print("Saved normalization:", checkpoint.get("normalization"))

# Check a prepared training sample
sample_path = train_files[0]

with np.load(sample_path, allow_pickle=True) as d:
    print("\n" + "=" * 60)
    print("SAMPLE STRUCTURE")
    print("=" * 60)

    for key in d.files:
        value = d[key]
        print(key, getattr(value, "shape", None), getattr(value, "dtype", None))

print("\nSTEP 100B COMPLETE")

DATA FILES
Training chunks: 1520
Validation chunks: 301

Training examples:
train_telugufemale_00009_chunk000.npz
train_telugufemale_00013_chunk000.npz
train_telugufemale_00015_chunk000.npz
train_telugufemale_00015_chunk001.npz
train_telugufemale_00018_chunk000.npz

Validation examples:
train_telugufemale_00060_chunk000.npz
train_telugufemale_00065_chunk000.npz
train_telugufemale_00078_chunk000.npz
train_telugufemale_00146_chunk000.npz
train_telugufemale_00146_chunk001.npz

CHECKPOINTS
100.pth.tar 89.80 MB
200.pth.tar 89.80 MB
300.pth.tar 89.80 MB
400.pth.tar 89.80 MB
500.pth.tar 89.80 MB
600.pth.tar 89.81 MB
latest.pth.tar 89.81 MB

Checkpoint step: 500
Checkpoint keys: ['step', 'model', 'optimizer', 'history', 'speaker_map', 'normalization']
Saved speaker map: {'female': 0, 'male': 1}
Saved normalization: {'pitch_mean': 109.76953432377077, 'pitch_std': 103.30357441258384, 'energy_mean': 0.05246112019682791, 'energy_std': 0.05535232817865627}

SAMPLE STRUCTURE
mel (80, 1509) float32
p

In [33]:
# ============================================================
# STEP 100C — CHECK MODEL AND CHECKPOINT CONSISTENCY
# RUN NOW — NO TRAINING
# ============================================================

import inspect
import torch
import numpy as np

from model.fastspeech2 import FastSpeech2
from model.loss import FastSpeech2Loss

print("=" * 65)
print("1. MODEL FORWARD SIGNATURE")
print("=" * 65)

print(inspect.signature(model.forward))

print("\n" + "=" * 65)
print("2. MODEL FORWARD IMPLEMENTATION")
print("=" * 65)

print(inspect.getsource(FastSpeech2.forward))

print("\n" + "=" * 65)
print("3. LOSS INITIALIZATION")
print("=" * 65)

criterion = FastSpeech2Loss(preprocess_config, model_config)
print("Loss function initialized successfully.")

print("\n" + "=" * 65)
print("4. CHECKPOINT TRAINING HISTORY")
print("=" * 65)

for step in [300, 400, 500, 600]:
    path = CKPT_DIR / f"{step}.pth.tar"

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print(f"\nCheckpoint {step}")
    print("Stored step:", ckpt.get("step"))

    history = ckpt.get("history", {})

    if isinstance(history, dict):
        for key, value in history.items():
            print(key, ":", value)

    else:
        print("History format:", type(history))

print("\n" + "=" * 65)
print("5. OPTIMIZER STATE IN CHECKPOINT 500")
print("=" * 65)

ckpt = torch.load(
    CKPT_DIR / "500.pth.tar",
    map_location="cpu",
    weights_only=False
)

opt_state = ckpt["optimizer"]

print("Optimizer parameter groups:")

for group in opt_state["param_groups"]:
    print({
        "lr": group["lr"],
        "betas": group["betas"],
        "eps": group["eps"],
        "weight_decay": group["weight_decay"]
    })

print("Optimizer state entries:", len(opt_state["state"]))

print("\nSTEP 100C COMPLETE")

1. MODEL FORWARD SIGNATURE
(speakers, texts, src_lens, max_src_len, mels=None, mel_lens=None, max_mel_len=None, p_targets=None, e_targets=None, d_targets=None, p_control=1.0, e_control=1.0, d_control=1.0)

2. MODEL FORWARD IMPLEMENTATION
    def forward(
        self,
        speakers,
        texts,
        src_lens,
        max_src_len,
        mels=None,
        mel_lens=None,
        max_mel_len=None,
        p_targets=None,
        e_targets=None,
        d_targets=None,
        p_control=1.0,
        e_control=1.0,
        d_control=1.0,
    ):
        src_masks = get_mask_from_lengths(src_lens, max_src_len)
        mel_masks = (
            get_mask_from_lengths(mel_lens, max_mel_len)
            if mel_lens is not None
            else None
        )

        output = self.encoder(texts, src_masks)

        if self.speaker_emb is not None:
            output = output + self.speaker_emb(speakers).unsqueeze(1).expand(
                -1, max_src_len, -1
            )

        (
 

In [34]:
# ============================================================
# STEP 101 — ENGLISH PROJECT INVENTORY
# RUN NOW — INSPECTION ONLY
# ============================================================

from pathlib import Path
import json

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

print("=" * 65)
print("ENGLISH PROJECT DIRECTORY")
print("=" * 65)

if not ENG.exists():
    print("English directory does not exist!")
else:
    for path in sorted(ENG.rglob("*")):
        if path.is_file():
            relative = path.relative_to(ENG)
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"{relative} | {size_mb:.2f} MB")

print("\n" + "=" * 65)
print("IMPORTANT DIRECTORY COUNTS")
print("=" * 65)

folders = [
    "raw_audio/train",
    "raw_audio/validation",
    "alignment/full_alignment/train",
    "alignment/full_alignment/validation",
    "features/train",
    "features/validation",
    "checkpoints",
    "results/test_generated_audios",
    "results/variance_adaptor_plots/duration",
    "results/variance_adaptor_plots/pitch",
    "results/variance_adaptor_plots/energy",
]

for folder in folders:
    path = ENG / folder
    if path.exists():
        count = sum(1 for f in path.rglob("*") if f.is_file())
        print(f"{folder}: {count} files")
    else:
        print(f"{folder}: NOT FOUND")

print("\nSTEP 101 COMPLETE")

Streaming output truncated to the last 5000 lines.
raw_audio/train/recording_252.lab | 0.00 MB
raw_audio/train/recording_252.wav | 0.12 MB
raw_audio/train/recording_2520.lab | 0.00 MB
raw_audio/train/recording_2520.wav | 0.10 MB
raw_audio/train/recording_2521.lab | 0.00 MB
raw_audio/train/recording_2521.wav | 0.07 MB
raw_audio/train/recording_2524.lab | 0.00 MB
raw_audio/train/recording_2524.wav | 0.37 MB
raw_audio/train/recording_2526.lab | 0.00 MB
raw_audio/train/recording_2526.wav | 0.16 MB
raw_audio/train/recording_2527.lab | 0.00 MB
raw_audio/train/recording_2527.wav | 0.29 MB
raw_audio/train/recording_2529.lab | 0.00 MB
raw_audio/train/recording_2529.wav | 0.21 MB
raw_audio/train/recording_2531.lab | 0.00 MB
raw_audio/train/recording_2531.wav | 0.28 MB
raw_audio/train/recording_2535.lab | 0.00 MB
raw_audio/train/recording_2535.wav | 0.23 MB
raw_audio/train/recording_2537.lab | 0.00 MB
raw_audio/train/recording_2537.wav | 0.06 MB
raw_audio/train/recording_2539.lab | 0.00 MB
raw_au

In [35]:
# ============================================================
# STEP 102 — ENGLISH FULL FILE RECOVERY & AUDIT
# RUN NOW — NEW CELL
# READ-ONLY: DOES NOT REPROCESS OR OVERWRITE FILES
# ============================================================

from pathlib import Path
from collections import Counter
import json

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

assert ENG.exists(), f"English folder not found: {ENG}"

# Collect all files once
all_files = [p for p in ENG.rglob("*") if p.is_file()]

def matching_files(extensions):
    extensions = {e.lower() for e in extensions}
    return [p for p in all_files if p.suffix.lower() in extensions]

groups = {
    "Audio WAV": matching_files([".wav"]),
    "TextGrid alignments": matching_files([".TextGrid", ".textgrid"]),
    "Feature NPZ": matching_files([".npz"]),
    "Feature NPY": matching_files([".npy"]),
    "Checkpoints": matching_files([".pth", ".pt", ".ckpt"]),
    "Configs": matching_files([".yaml", ".yml", ".json"]),
    "Metadata": matching_files([".csv", ".tsv"]),
    "Plots": matching_files([".png", ".jpg", ".jpeg"]),
    "Generated audio": matching_files([".wav"]),
    "Reports": matching_files([".pdf", ".md", ".txt"]),
}

print("=" * 70)
print("ENGLISH PROJECT — COMPLETE FILE AUDIT")
print("=" * 70)
print("English root:", ENG)
print("Total files:", len(all_files))

for name, files in groups.items():
    print(f"{name}: {len(files)}")

print("\n" + "=" * 70)
print("DIRECTORY STRUCTURE")
print("=" * 70)

for folder in sorted(p for p in ENG.rglob("*") if p.is_dir()):
    relative = folder.relative_to(ENG)
    count = sum(1 for p in folder.iterdir() if p.is_file())
    print(f"{relative}/  ({count} direct files)")

print("\n" + "=" * 70)
print("ALIGNMENT FILES — FIRST 20")
print("=" * 70)

for p in groups["TextGrid alignments"][:20]:
    print(p.relative_to(ENG))

print("\n" + "=" * 70)
print("FEATURE FILES — FIRST 20")
print("=" * 70)

for p in (groups["Feature NPZ"] + groups["Feature NPY"])[:20]:
    print(p.relative_to(ENG))

print("\n" + "=" * 70)
print("CHECKPOINTS")
print("=" * 70)

for p in groups["Checkpoints"]:
    print(p.relative_to(ENG), f"{p.stat().st_size / 1024**2:.2f} MB")

print("\n" + "=" * 70)
print("CONFIGURATION AND METADATA FILES")
print("=" * 70)

for p in (groups["Configs"] + groups["Metadata"])[:100]:
    print(p.relative_to(ENG))

print("\n" + "=" * 70)
print("SAMPLE RAW AUDIO COUNTS")
print("=" * 70)

for split in ["train", "validation"]:
    folder = ENG / "raw_audio" / split
    if folder.exists():
        wavs = list(folder.glob("*.wav"))
        labs = list(folder.glob("*.lab"))
        print(split, "| WAV:", len(wavs), "| LAB:", len(labs))
    else:
        print(split, "| folder missing")

# Save a compact audit report for later use
audit = {
    "english_root": str(ENG),
    "total_files": len(all_files),
    "counts": {name: len(files) for name, files in groups.items()},
    "directories": [
        str(p.relative_to(ENG))
        for p in sorted(ENG.rglob("*")) if p.is_dir()
    ],
    "textgrids": [str(p.relative_to(ENG)) for p in groups["TextGrid alignments"]],
    "features_npz": [str(p.relative_to(ENG)) for p in groups["Feature NPZ"]],
    "features_npy": [str(p.relative_to(ENG)) for p in groups["Feature NPY"]],
    "checkpoints": [str(p.relative_to(ENG)) for p in groups["Checkpoints"]],
}

audit_path = ENG / "english_full_audit.json"
audit_path.parent.mkdir(parents=True, exist_ok=True)

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)

print("\nAudit saved:", audit_path)
print("\nSTEP 102 COMPLETE")

ENGLISH PROJECT — COMPLETE FILE AUDIT
English root: /content/drive/MyDrive/FastSpeech2_Assignment/English
Total files: 6339
Audio WAV: 3170
TextGrid alignments: 0
Feature NPZ: 0
Feature NPY: 0
Checkpoints: 0
Configs: 0
Metadata: 0
Plots: 0
Generated audio: 3170
Reports: 0

DIRECTORY STRUCTURE
raw_audio/  (0 direct files)
raw_audio/train/  (5328 direct files)
raw_audio/validation/  (1010 direct files)
raw_data/  (0 direct files)
sample_audio_test/  (1 direct files)

ALIGNMENT FILES — FIRST 20

FEATURE FILES — FIRST 20

CHECKPOINTS

CONFIGURATION AND METADATA FILES

SAMPLE RAW AUDIO COUNTS
train | WAV: 2664 | LAB: 2664
validation | WAV: 505 | LAB: 505

Audit saved: /content/drive/MyDrive/FastSpeech2_Assignment/English/english_full_audit.json

STEP 102 COMPLETE


In [36]:
# ================================================================
# STEP 103 — ENGLISH DATASET & TRANSCRIPT READINESS AUDIT
# RUN NOW — NEW CELL
# READ-ONLY: DOES NOT MODIFY OR OVERWRITE DATA
# ================================================================

from pathlib import Path
import soundfile as sf
import pandas as pd
import numpy as np
import json

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

def inspect_split(split):
    folder = ENG / "raw_audio" / split
    wavs = sorted(folder.glob("*.wav"))
    labs = sorted(folder.glob("*.lab"))

    lab_map = {p.stem: p for p in labs}
    wav_map = {p.stem: p for p in wavs}

    missing_lab = sorted(set(wav_map) - set(lab_map))
    missing_wav = sorted(set(lab_map) - set(wav_map))

    records = []
    for wav in wavs:
        try:
            info = sf.info(str(wav))
            transcript = ""
            lab_path = lab_map.get(wav.stem)

            if lab_path and lab_path.exists():
                transcript = lab_path.read_text(
                    encoding="utf-8",
                    errors="replace"
                ).strip()

            records.append({
                "filename": wav.name,
                "duration": info.duration,
                "sample_rate": info.samplerate,
                "channels": info.channels,
                "transcript": transcript,
                "transcript_chars": len(transcript),
                "has_transcript": bool(transcript)
            })
        except Exception as e:
            print("FILE ERROR:", wav.name, str(e))

    df = pd.DataFrame(records)

    print("\n" + "=" * 65)
    print("SPLIT:", split.upper())
    print("=" * 65)

    print("WAV files:", len(wavs))
    print("LAB files:", len(labs))
    print("Missing LAB:", len(missing_lab))
    print("Missing WAV:", len(missing_wav))

    if len(df):
        total_hours = df["duration"].sum() / 3600

        print("Total duration:", round(total_hours, 4), "hours")
        print("Average duration:", round(df["duration"].mean(), 3), "seconds")
        print("Sample rates:", df["sample_rate"].value_counts().to_dict())
        print("Channels:", df["channels"].value_counts().to_dict())
        print("Empty transcripts:", int((~df["has_transcript"]).sum()))
        print("Transcript characters — min:",
              int(df["transcript_chars"].min()))
        print("Transcript characters — max:",
              int(df["transcript_chars"].max()))

        print("\nFIRST 5 TRANSCRIPTS:")
        for _, row in df.head(5).iterrows():
            print(row["filename"], "=>", row["transcript"][:250])

        print("\nRANDOM 5 TRANSCRIPTS:")
        for _, row in df.sample(
            min(5, len(df)), random_state=42
        ).iterrows():
            print(row["filename"], "=>", row["transcript"][:250])

    return df

train_df = inspect_split("train")
val_df = inspect_split("validation")

# Check overlap
train_names = set(train_df["filename"].str.replace(".wav", "", regex=False))
val_names = set(val_df["filename"].str.replace(".wav", "", regex=False))

print("\n" + "=" * 65)
print("SPLIT VALIDATION")
print("=" * 65)
print("Overlapping recording IDs:", len(train_names & val_names))

# Save audit
audit = {
    "train_files": len(train_df),
    "validation_files": len(val_df),
    "train_hours": float(train_df["duration"].sum() / 3600),
    "validation_hours": float(val_df["duration"].sum() / 3600),
    "overlapping_ids": len(train_names & val_names),
    "train_empty_transcripts": int((~train_df["has_transcript"]).sum()),
    "validation_empty_transcripts": int((~val_df["has_transcript"]).sum())
}

out = ENG / "english_dataset_readiness_audit.json"
out.write_text(json.dumps(audit, indent=2))

print("\nAudit saved:", out)
print("STEP 103 COMPLETE")


SPLIT: TRAIN
WAV files: 2664
LAB files: 2664
Missing LAB: 0
Missing WAV: 0
Total duration: 5.0014 hours
Average duration: 6.759 seconds
Sample rates: {16000: 2664}
Channels: {1: 2664}
Empty transcripts: 0
Transcript characters — min: 8
Transcript characters — max: 392

FIRST 5 TRANSCRIPTS:
recording_0.wav => Our district Hasana was the land of the great Hoysala dynasty who ruled over this region from 11th to 14th century CE.
recording_10.wav => Among the poets, I remember William Wordsworth, T.S.
recording_1001.wav => Their holistic approach to the disease of addiction
recording_1003.wav => The book is really inspiring and can put a new hope in our lives.
recording_1005.wav => And in between, during daytime, also whenever I get time, I do meditation and

RANDOM 5 TRANSCRIPTS:
recording_4318.wav => I do yoga on a regular basis
recording_1293.wav => Corruption is one of that reason. Confidence, lack of patience, everything is demoting the sports in the state.
recording_3496.wav => What 

In [37]:
# ================================================================
# STEP 104 — CHECK ENGLISH MFA ALIGNMENT ENVIRONMENT
# RUN NOW — NEW CELL
# READ-ONLY ENVIRONMENT CHECK
# Does not run alignment or modify dataset files.
# ================================================================

import shutil
import subprocess
import importlib.util
from pathlib import Path
import json

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

print("=" * 65)
print("ENGLISH MFA ENVIRONMENT CHECK")
print("=" * 65)

# 1. Check MFA executable
mfa_path = shutil.which("mfa")
print("\nMFA executable:", mfa_path)

if mfa_path:
    result = subprocess.run(
        ["mfa", "version"],
        capture_output=True,
        text=True
    )
    print("MFA version output:")
    print(result.stdout[:1000] or result.stderr[:1000])

# 2. Check MFA Python package
for package in ["montreal_forced_aligner", "tgt"]:
    print(
        f"{package}:",
        importlib.util.find_spec(package) is not None
    )

# 3. Check available MFA models
if mfa_path:
    result = subprocess.run(
        ["mfa", "model", "list", "acoustic"],
        capture_output=True,
        text=True
    )
    print("\nACOUSTIC MODELS:")
    print((result.stdout + result.stderr)[:5000])

    result = subprocess.run(
        ["mfa", "model", "list", "g2p"],
        capture_output=True,
        text=True
    )
    print("\nG2P MODELS:")
    print((result.stdout + result.stderr)[:5000])

# 4. Check alignment-related existing files across assignment
print("\nEXISTING ALIGNMENT ARTIFACTS:")

for folder in BASE.rglob("*"):
    if folder.is_file() and folder.suffix.lower() in [
        ".textgrid", ".json", ".csv"
    ]:
        if any(word in str(folder).lower() for word in [
            "alignment", "align", "english"
        ]):
            print(folder)

# 5. Save environment audit
audit = {
    "mfa_executable": mfa_path,
    "mfa_python_package": (
        importlib.util.find_spec("montreal_forced_aligner") is not None
    ),
    "tgt_available": importlib.util.find_spec("tgt") is not None
}

output = ENG / "english_mfa_environment_audit.json"
output.write_text(json.dumps(audit, indent=2))

print("\nAudit saved:", output)
print("\nSTEP 104 COMPLETE")

ENGLISH MFA ENVIRONMENT CHECK

MFA executable: None
montreal_forced_aligner: False
tgt: False

EXISTING ALIGNMENT ARTIFACTS:
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/english_train.csv
/content/drive/MyDrive/FastSpeech2_Assignment/metadata/english_validation.csv
/content/drive/MyDrive/FastSpeech2_Assignment/English/english_full_audit.json
/content/drive/MyDrive/FastSpeech2_Assignment/English/english_dataset_readiness_audit.json

Audit saved: /content/drive/MyDrive/FastSpeech2_Assignment/English/english_mfa_environment_audit.json

STEP 104 COMPLETE


In [38]:
# ================================================================
# STEP 105 — INSTALL MONTREAL FORCED ALIGNER
# RUN NOW — NEW CELL
# Installs MFA and verifies its command.
# Does not run alignment or modify dataset files.
# ================================================================

import sys
import subprocess
import shutil

print("Python:", sys.version)

# Install MFA
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "montreal-forced-aligner"
    ],
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        "MFA installation failed. Share the complete error output."
    )

# Verify executable
mfa_path = shutil.which("mfa")

if not mfa_path:
    # Check common Colab executable directory
    candidate = "/usr/local/bin/mfa"
    if shutil.which(candidate):
        mfa_path = candidate

if not mfa_path:
    raise RuntimeError(
        "MFA was installed but the executable was not found. "
        "Do not reinstall; share the output."
    )

print("\nMFA executable:", mfa_path)

# Version check
version = subprocess.run(
    [mfa_path, "version"],
    capture_output=True,
    text=True
)

print("MFA version:")
print(version.stdout or version.stderr)

# Verify existing PyTorch installation
import torch
import torchaudio

print("\nPyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())

print("\nSTEP 105 COMPLETE")

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]

MFA executable: /usr/local/bin/mfa
MFA version:
Traceback (most recent call last):
  File "/usr/local/bin/mfa", line 3, in <module>
    from montreal_forced_aligner.command_line.mfa import mfa_cli
  File "/usr/local/lib/python3.13/dist-packages/montreal_forced_aligner/__init__.py", line 4, in <module>
    import montreal_forced_aligner.acoustic_modeling as acoustic_modeling
  File "/usr/local/lib/python3.13/dist-packages/montreal_forced_aligner/acoustic_modeling/__init__.py", line 7, in <module>
    from montreal_forced_aligner.acoustic_modeling.base import AcousticModelTrainingMixin  # noqa
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/montreal_forced_aligner/acoustic_modeling/base.py", line 12, in <module>
    from _kalpy.gmm import AccumAmDiagGmm
ModuleNotFoundError: No module named '_kalpy'


PyTorch: 2.11.0+cpu
TorchAudio: 2.11.0+c

In [39]:
# ================================================================
# STEP 106 — INSTALL MFA IN A SEPARATE CONDA ENVIRONMENT
# RUN NOW — NEW CELL
# Keeps Colab Python and PyTorch environment untouched.
# ================================================================

import os
import subprocess
import shutil
from pathlib import Path

MINIFORGE = Path("/content/miniforge3")
MFA_ENV = Path("/content/mfa-env")

# 1. Install Miniforge if it is not already installed
if not (MINIFORGE / "bin/conda").exists():
    installer = "/content/Miniforge3.sh"

    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
            "-O", installer
        ],
        check=True
    )

    subprocess.run(
        ["bash", installer, "-b", "-p", str(MINIFORGE)],
        check=True
    )

conda = str(MINIFORGE / "bin/conda")

print("Miniforge:", conda)

# 2. Create isolated MFA environment
if not (MFA_ENV / "bin/mfa").exists():
    subprocess.run(
        [
            conda, "create",
            "-y",
            "-p", str(MFA_ENV),
            "-c", "conda-forge",
            "python=3.11",
            "montreal-forced-aligner"
        ],
        check=True
    )

# 3. Verify MFA
mfa = str(MFA_ENV / "bin/mfa")

result = subprocess.run(
    [mfa, "version"],
    capture_output=True,
    text=True
)

print("\nMFA VERSION:")
print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("MFA verification failed. Share the error.")

# 4. Verify help command
help_result = subprocess.run(
    [mfa, "--help"],
    capture_output=True,
    text=True
)

if help_result.returncode != 0:
    raise RuntimeError("MFA help command failed.")

print("\nMFA COMMAND VERIFIED")

# 5. Verify original Colab PyTorch remains unchanged
import torch
import torchaudio

print("\nCOLAB ENVIRONMENT:")
print("Python:", os.sys.version)
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("CUDA:", torch.cuda.is_available())

print("\nMFA executable:", mfa)
print("STEP 106 COMPLETE")

Miniforge: /content/miniforge3/bin/conda

MFA VERSION:
3.4.3



MFA COMMAND VERIFIED

COLAB ENVIRONMENT:
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
TorchAudio: 2.11.0+cpu
CUDA: False

MFA executable: /content/mfa-env/bin/mfa
STEP 106 COMPLETE


In [40]:
# ================================================================
# STEP 107 — CHECK AND DOWNLOAD ENGLISH MFA MODELS
# RUN NOW — NEW CELL
# Does not run alignment.
# ================================================================

import subprocess
from pathlib import Path

MFA = Path("/content/mfa-env/bin/mfa")

assert MFA.exists(), "MFA executable not found."

def run_mfa(args):
    result = subprocess.run(
        [str(MFA)] + args,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    print("Return code:", result.returncode)
    return result

print("=" * 65)
print("AVAILABLE ACOUSTIC MODELS")
print("=" * 65)

acoustic = run_mfa(["model", "list", "acoustic"])
if acoustic.returncode != 0:
    raise RuntimeError("Could not list acoustic models.")

print("\n" + "=" * 65)
print("AVAILABLE G2P MODELS")
print("=" * 65)

g2p = run_mfa(["model", "list", "g2p"])
if g2p.returncode != 0:
    raise RuntimeError("Could not list G2P models.")

# Save model catalogue for reproducibility
output = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/English"
)
output.mkdir(parents=True, exist_ok=True)

(output / "mfa_acoustic_model_catalogue.txt").write_text(
    acoustic.stdout + "\n" + acoustic.stderr
)

(output / "mfa_g2p_model_catalogue.txt").write_text(
    g2p.stdout + "\n" + g2p.stderr
)

print("\nModel catalogues saved.")
print("STEP 107 COMPLETE")

AVAILABLE ACOUSTIC MODELS

STDERR: No models found

Return code: 0

AVAILABLE G2P MODELS

STDERR: No models found

Return code: 0

Model catalogues saved.
STEP 107 COMPLETE


In [41]:
# ================================================================
# STEP 108 — MFA ALIGN_HF COMMAND AND CORPUS CHECK
# RUN NOW — NEW CELL
# READ-ONLY CHECK; DOES NOT START ALIGNMENT
# ================================================================

import subprocess
from pathlib import Path

MFA = "/content/mfa-env/bin/mfa"
ENG = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")

# 1. Inspect the Hugging Face alignment command
result = subprocess.run(
    [MFA, "align_hf", "--help"],
    capture_output=True,
    text=True
)

print("ALIGN_HF HELP")
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("align_hf command is unavailable.")

# 2. Verify the Indian English dialect option
help_text = result.stdout + result.stderr

for option in ["--dialect", "--use_g2p", "--num_jobs"]:
    print(option, "available:", option in help_text)

# 3. Verify a few existing corpus pairs
for split in ["train", "validation"]:
    folder = ENG / "raw_audio" / split
    wavs = sorted(folder.glob("*.wav"))

    print(f"\n{split.upper()} SAMPLE PAIRS")
    for wav in wavs[:5]:
        lab = wav.with_suffix(".lab")
        print(
            wav.name,
            "| LAB exists:", lab.exists(),
            "| Transcript:", lab.read_text(
                encoding="utf-8",
                errors="replace"
            ).strip()[:100] if lab.exists() else "MISSING"
        )

print("\nSTEP 108 COMPLETE")

ALIGN_HF HELP
                                                                                
 Usage: mfa align_hf [OPTIONS] CORPUS_DIRECTORY MODEL_ID OUTPUT_DIRECTORY       
                                                                                
 Align a corpus with an MFA model hosted on Hugging Face.                       
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --config_path           -c      FILE                   Path to config file   │
│                                                        to use for aligning.  │
│ --speaker_characters    -s      TEXT                   Number of characters  │
│                                                        of file names to use  │
│                                                        for determining       │
│                                                        speaker, default is   │
│             

In [42]:
# ================================================================
# STEP 109 — ENGLISH MFA ALIGNMENT PILOT
# RUN NOW — NEW CELL
# Only 10 training recordings; original dataset untouched.
# ================================================================

import subprocess
from pathlib import Path
import shutil

MFA = "/content/mfa-env/bin/mfa"
ENG = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")

PILOT = ENG / "alignment_pilot" / "corpus"
OUTPUT = ENG / "alignment_pilot" / "output"
TEMP = ENG / "alignment_pilot" / "temporary"

PILOT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
TEMP.mkdir(parents=True, exist_ok=True)

# Prepare a separate 10-file pilot corpus
source = ENG / "raw_audio" / "train"

for wav in sorted(source.glob("*.wav"))[:10]:
    lab = wav.with_suffix(".lab")

    if not lab.exists():
        raise FileNotFoundError(f"Missing transcript: {lab}")

    shutil.copy2(wav, PILOT / wav.name)
    shutil.copy2(lab, PILOT / lab.name)

print("Pilot WAV files:", len(list(PILOT.glob("*.wav"))))
print("Pilot LAB files:", len(list(PILOT.glob("*.lab"))))

# Run Hugging Face model alignment
command = [
    MFA,
    "align_hf",
    str(PILOT),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT),
    "--dialect", "english_india",
    "--use_g2p",
    "--num_jobs", "2",
    "--single_speaker",
    "--temporary_directory", str(TEMP),
    "--no_clean",
    "--verbose"
]

print("\nRunning command:")
print(" ".join(command))

result = subprocess.run(
    command,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

log_path = ENG / "alignment_pilot" / "pilot_alignment.log"
log_path.write_text(result.stdout)

print("\nMFA OUTPUT — LAST 12000 CHARACTERS")
print(result.stdout[-12000:])

print("\nReturn code:", result.returncode)

textgrids = list(OUTPUT.rglob("*.TextGrid")) + list(
    OUTPUT.rglob("*.textgrid")
)

print("TextGrid files generated:", len(textgrids))

if result.returncode != 0 or len(textgrids) == 0:
    print("\nPILOT FAILED OR PRODUCED NO ALIGNMENTS.")
    print("Do not run full alignment.")
else:
    print("\nPILOT ALIGNMENT SUCCESSFUL.")
    print("Inspect the TextGrids before proceeding.")

print("\nLog saved:", log_path)
print("STEP 109 COMPLETE")

Pilot WAV files: 10
Pilot LAB files: 10

Running command:
/content/mfa-env/bin/mfa align_hf /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment_pilot/corpus MontrealCorpusTools/english_mfa /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment_pilot/output --dialect english_india --use_g2p --num_jobs 2 --single_speaker --temporary_directory /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment_pilot/temporary --no_clean --verbose

MFA OUTPUT — LAST 12000 CHARACTERS

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 42 files: 100%|██████████| 42/42 [00:04<00:00,  9.38it/s]
Traceback (most recent call last):
  File "/content/mfa-env/bin/mfa", line 10, in <module>
    sys.exit(mfa_cli())
             ^^^^^^^^^
  File "/content/mfa-env/lib/python3.11/site-packages/rich_click/rich_command.py", line 401, in _

In [43]:
# ================================================================
# STEP 110 — FIX OPENFST / FSTCOMPILE
# RUN NOW — NEW CELL
# ================================================================

import subprocess
import os
from pathlib import Path

CONDA = "/content/miniforge3/bin/conda"
ENV = "/content/mfa-env"

# 1. Install OpenFst into the existing MFA environment
result = subprocess.run(
    [
        CONDA, "install",
        "-y",
        "-p", ENV,
        "-c", "conda-forge",
        "openfst"
    ],
    text=True
)

if result.returncode != 0:
    raise RuntimeError("OpenFst installation failed.")

# 2. Activate environment paths for this process
env = os.environ.copy()
env["PATH"] = f"{ENV}/bin:" + env.get("PATH", "")

# 3. Verify fstcompile
fstcompile = subprocess.run(
    ["which", "fstcompile"],
    capture_output=True,
    text=True,
    env=env
)

print("fstcompile location:", fstcompile.stdout.strip())

if fstcompile.returncode != 0:
    raise RuntimeError("fstcompile is still unavailable.")

# 4. Verify MFA
mfa = f"{ENV}/bin/mfa"

version = subprocess.run(
    [mfa, "version"],
    capture_output=True,
    text=True,
    env=env
)

print("\nMFA version:", version.stdout.strip())

# 5. Verify original Colab PyTorch remains unchanged
import torch
import torchaudio

print("\nPyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)

print("\nSTEP 110 COMPLETE")

fstcompile location: /content/mfa-env/bin/fstcompile

MFA version: 3.4.3

PyTorch: 2.11.0+cpu
TorchAudio: 2.11.0+cpu

STEP 110 COMPLETE


In [44]:
# ================================================================
# STEP 111 — RETRY ENGLISH MFA PILOT
# RUN NOW — NEW CELL
# Reuses downloaded model and existing 10 WAV/LAB pairs.
# ================================================================

import os
import subprocess
from pathlib import Path

MFA = "/content/mfa-env/bin/mfa"
ENV = "/content/mfa-env"

ENG = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")
PILOT = ENG / "alignment_pilot" / "corpus"
OUTPUT = ENG / "alignment_pilot" / "output"
TEMP = ENG / "alignment_pilot" / "temporary"

assert len(list(PILOT.glob("*.wav"))) == 10
assert len(list(PILOT.glob("*.lab"))) == 10

OUTPUT.mkdir(parents=True, exist_ok=True)
TEMP.mkdir(parents=True, exist_ok=True)

# Ensure MFA can find OpenFst binaries
env = os.environ.copy()
env["PATH"] = f"{ENV}/bin:" + env.get("PATH", "")

command = [
    MFA, "align_hf",
    str(PILOT),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT),
    "--dialect", "english_india",
    "--use_g2p",
    "--num_jobs", "2",
    "--single_speaker",
    "--temporary_directory", str(TEMP),
    "--no_clean",
    "--verbose"
]

print("Running English alignment pilot...")
print(" ".join(command))

result = subprocess.run(
    command,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

log = ENG / "alignment_pilot" / "pilot_alignment_retry.log"
log.write_text(result.stdout)

print("\nMFA OUTPUT — LAST 12000 CHARACTERS")
print(result.stdout[-12000:])

textgrids = (
    list(OUTPUT.rglob("*.TextGrid")) +
    list(OUTPUT.rglob("*.textgrid"))
)

print("\nReturn code:", result.returncode)
print("TextGrid files:", len(textgrids))

if result.returncode == 0 and len(textgrids) > 0:
    print("\nPILOT ALIGNMENT SUCCESSFUL.")
    print("Inspect alignment quality before full processing.")
else:
    print("\nPILOT NOT SUCCESSFUL. DO NOT RUN FULL ALIGNMENT.")

print("Log:", log)
print("STEP 111 COMPLETE")

Running English alignment pilot...
/content/mfa-env/bin/mfa align_hf /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment_pilot/corpus MontrealCorpusTools/english_mfa /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment_pilot/output --dialect english_india --use_g2p --num_jobs 2 --single_speaker --temporary_directory /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment_pilot/temporary --no_clean --verbose

MFA OUTPUT — LAST 12000 CHARACTERS

Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 1388.34it/s]
 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
  10% ━━━╸                                10/100  [ 0:00:01 < 0:01:38 , 1 it/s ]
 INFO     Found 1 speaker across 10 files, average number of utterances per     
          speaker: 10.0                                                         
 INFO     Initializing multiprocessing jobs... 

In [45]:
# ================================================================
# STEP 112 — ENGLISH PILOT ALIGNMENT QUALITY AUDIT
# RUN NOW — NEW CELL
# READ-ONLY
# ================================================================

from pathlib import Path
import json
import re

ENG = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")
PILOT = ENG / "alignment_pilot"
CORPUS = PILOT / "corpus"
OUTPUT = PILOT / "output"

textgrids = sorted(
    list(OUTPUT.rglob("*.TextGrid")) +
    list(OUTPUT.rglob("*.textgrid"))
)

print("TextGrid count:", len(textgrids))

if not textgrids:
    raise RuntimeError("No TextGrid files found.")

def inspect_textgrid(path):
    content = path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    # Extract interval boundaries and labels
    xmin = [float(x) for x in re.findall(
        r"(?m)^\s*xmin\s*=\s*([0-9.eE+-]+)", content
    )]
    xmax = [float(x) for x in re.findall(
        r"(?m)^\s*xmax\s*=\s*([0-9.eE+-]+)", content
    )]
    labels = re.findall(
        r'(?m)^\s*text\s*=\s*"(.*?)"\s*$',
        content
    )

    nonempty = [x for x in labels if x.strip()]
    invalid = (
        len(xmin) != len(xmax) or
        any(b < a for a, b in zip(xmin, xmax))
    )

    return {
        "file": path.name,
        "intervals": len(labels),
        "nonempty_intervals": len(nonempty),
        "first_boundary": min(xmin) if xmin else None,
        "last_boundary": max(xmax) if xmax else None,
        "invalid_boundaries": invalid,
        "sample_labels": nonempty[:8]
    }

reports = []

for tg in textgrids:
    report = inspect_textgrid(tg)
    reports.append(report)

    print("\n", report["file"])
    print("Intervals:", report["intervals"])
    print("Nonempty labels:", report["nonempty_intervals"])
    print("Invalid boundaries:", report["invalid_boundaries"])
    print("Sample labels:", report["sample_labels"])

# Check audio duration against TextGrid endpoint
print("\n" + "=" * 60)
print("AUDIO / ALIGNMENT DURATION COMPARISON")
print("=" * 60)

import soundfile as sf

for report in reports:
    wav = CORPUS / (Path(report["file"]).stem + ".wav")

    if wav.exists():
        audio_duration = sf.info(str(wav)).duration
        grid_duration = report["last_boundary"]

        print(
            report["file"],
            "| Audio:", round(audio_duration, 3),
            "| TextGrid:", round(grid_duration or 0, 3),
            "| Difference:",
            round(abs(audio_duration - (grid_duration or 0)), 3)
        )

summary = {
    "textgrid_count": len(textgrids),
    "invalid_boundary_files": sum(
        r["invalid_boundaries"] for r in reports
    ),
    "total_nonempty_intervals": sum(
        r["nonempty_intervals"] for r in reports
    ),
    "reports": reports
}

save_path = PILOT / "pilot_quality_audit.json"
save_path.write_text(json.dumps(summary, indent=2))

print("\nAudit saved:", save_path)
print("STEP 112 COMPLETE")

TextGrid count: 10

 recording_0.TextGrid
Intervals: 108
Nonempty labels: 100
Invalid boundaries: False
Sample labels: ['our', 'district', 'hasana', 'was', 'the', 'land', 'of', 'the']

 recording_10.TextGrid
Intervals: 55
Nonempty labels: 45
Invalid boundaries: False
Sample labels: ['among', 'the', 'poets', 'i', 'remember', 'william', 'wordsworth', 't']

 recording_1001.TextGrid
Intervals: 49
Nonempty labels: 41
Invalid boundaries: False
Sample labels: ['their', 'holistic', 'approach', 'to', 'the', 'disease', 'of', 'addiction']

 recording_1003.TextGrid
Intervals: 63
Nonempty labels: 55
Invalid boundaries: False
Sample labels: ['the', 'book', 'is', 'really', 'inspiring', 'and', 'can', 'put']

 recording_1005.TextGrid
Intervals: 78
Nonempty labels: 64
Invalid boundaries: False
Sample labels: ['and', 'in', 'between', 'during', 'daytime', 'also', 'whenever', 'i']

 recording_1008.TextGrid
Intervals: 80
Nonempty labels: 76
Invalid boundaries: False
Sample labels: ['can', 'you', 'track', 't

In [46]:
# ================================================================
# STEP 113 — FULL ENGLISH TRAINING ALIGNMENT
# RUN NOW — NEW CELL
# Processes 2,664 training recordings.
# ================================================================

import os
import subprocess
from pathlib import Path
import json
from datetime import datetime

MFA = "/content/mfa-env/bin/mfa"
ENV = "/content/mfa-env"

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

CORPUS = ENG / "raw_audio" / "train"
OUTPUT = ENG / "alignment" / "full_alignment" / "train"
TEMP = ENG / "alignment" / "temporary" / "train"
LOG_DIR = ENG / "alignment" / "logs"

for folder in [OUTPUT, TEMP, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Safety checks
wav_count = len(list(CORPUS.glob("*.wav")))
lab_count = len(list(CORPUS.glob("*.lab")))

print("Training WAV files:", wav_count)
print("Training LAB files:", lab_count)

if wav_count != 2664 or lab_count != 2664:
    raise RuntimeError("Unexpected training corpus count.")

existing = (
    list(OUTPUT.rglob("*.TextGrid")) +
    list(OUTPUT.rglob("*.textgrid"))
)

if existing:
    raise RuntimeError(
        f"{len(existing)} TextGrids already exist. "
        "Stopping to avoid overwriting."
    )

# MFA environment
env = os.environ.copy()
env["PATH"] = f"{ENV}/bin:" + env.get("PATH", "")

command = [
    MFA, "align_hf",
    str(CORPUS),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT),
    "--dialect", "english_india",
    "--use_g2p",
    "--num_jobs", "2",
    "--single_speaker",
    "--temporary_directory", str(TEMP),
    "--no_clean",
    "--verbose"
]

print("\nStarting full training alignment...")
print("Started:", datetime.now().isoformat())

result = subprocess.run(
    command,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

log_path = LOG_DIR / "english_train_alignment.log"
log_path.write_text(result.stdout)

textgrids = (
    list(OUTPUT.rglob("*.TextGrid")) +
    list(OUTPUT.rglob("*.textgrid"))
)

summary = {
    "split": "train",
    "input_wav_count": wav_count,
    "input_lab_count": lab_count,
    "textgrid_count": len(textgrids),
    "return_code": result.returncode,
    "completed_at": datetime.now().isoformat(),
    "log_file": str(log_path)
}

summary_path = LOG_DIR / "english_train_alignment_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("\nALIGNMENT OUTPUT — LAST 10000 CHARACTERS")
print(result.stdout[-10000:])

print("\nReturn code:", result.returncode)
print("TextGrid files:", len(textgrids))
print("Summary:", summary_path)
print("Log:", log_path)

if result.returncode == 0:
    print("\nTRAINING ALIGNMENT COMMAND COMPLETED.")
    print("Expected TextGrids:", wav_count)
    print("Actual TextGrids:", len(textgrids))
else:
    print("\nTRAINING ALIGNMENT FAILED. DO NOT RUN VALIDATION.")

print("\nSTEP 113 COMPLETE")

Training WAV files: 2664
Training LAB files: 2664

Starting full training alignment...
Started: 2026-11-05T06:22:18.542009

ALIGNMENT OUTPUT — LAST 10000 CHARACTERS

Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 742.42it/s]
 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/100  [ 0:00:16 < 0:00:00 , 103 it/s ]
 INFO     Found 1 speaker across 2664 files, average number of utterances per   
          speaker: 2664.0                                                       
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2,664/2,664  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Generating pronunciations...                                          
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [47]:
# ================================================================
# STEP 114 — AUDIT MISSING ENGLISH TRAIN ALIGNMENTS
# RUN NOW — NEW CELL
# READ-ONLY
# ================================================================

from pathlib import Path
import json

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

CORPUS = ENG / "raw_audio" / "train"
OUTPUT = ENG / "alignment" / "full_alignment" / "train"

wav_stems = {p.stem for p in CORPUS.glob("*.wav")}
tg_stems = {
    p.stem for p in OUTPUT.rglob("*")
    if p.is_file() and p.suffix.lower() == ".textgrid"
}

missing = sorted(wav_stems - tg_stems)
unexpected = sorted(tg_stems - wav_stems)

print("Training WAV files:", len(wav_stems))
print("TextGrid files:", len(tg_stems))
print("Missing alignments:", len(missing))
print("Unexpected TextGrids:", len(unexpected))

print("\nMISSING RECORDINGS:")
for name in missing:
    lab = CORPUS / f"{name}.lab"
    print(name, "| Transcript:", lab.read_text(
        encoding="utf-8", errors="replace"
    ).strip() if lab.exists() else "MISSING LAB")

print("\nUNEXPECTED TEXTGRIDS:", unexpected[:20])

summary = {
    "expected": len(wav_stems),
    "generated": len(tg_stems),
    "missing_count": len(missing),
    "missing_recordings": missing,
    "unexpected_textgrids": unexpected
}

path = ENG / "alignment" / "logs" / "english_train_missing_audit.json"
path.write_text(json.dumps(summary, indent=2))

print("\nAudit saved:", path)
print("STEP 114 COMPLETE")

Training WAV files: 2664
TextGrid files: 2647
Missing alignments: 17
Unexpected TextGrids: 0

MISSING RECORDINGS:
recording_1558 | Transcript: it is considered to be more elite has a lot of prestige associated with it
recording_2157 | Transcript: the people who practice those different religions, they have a high level of tolerance and respect for each other's religion
recording_2282 | Transcript: it's not only the computers, but using smartphones is also very important in the present world
recording_2362 | Transcript: I tried to prepare my own compost
recording_2371 | Transcript: and people cooperate with one another whenever the issue of religious festivals comes rather they participate in each other's festivals.
recording_282 | Transcript: so that the plants can be saved during the winter
recording_3253 | Transcript: You are 2 hours late than the delivery time for my products Dettol Shaving Cream and Good knight Activ+ Low Smoke Mosquito Coil
recording_3274 | Transcript: they are a 

In [48]:
# ================================================================
# STEP 115 — DIAGNOSE MFA TRAINING ALIGNMENT FAILURES
# RUN NOW — NEW CELL
# READ-ONLY
# ================================================================

from pathlib import Path
import re
import json

ENG = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")

LOG = ENG / "alignment" / "logs" / "english_train_alignment.log"
TEMP = ENG / "alignment" / "temporary" / "train"

missing = [
    "recording_1558", "recording_2157", "recording_2282",
    "recording_2362", "recording_2371", "recording_282",
    "recording_3253", "recording_3274", "recording_3397",
    "recording_3515", "recording_3983", "recording_4688",
    "recording_4707", "recording_5283", "recording_5595",
    "recording_5712", "recording_6486"
]

print("=" * 65)
print("MFA LOG DIAGNOSTICS")
print("=" * 65)

if LOG.exists():
    lines = LOG.read_text(errors="replace").splitlines()

    keywords = [
        "error", "warning", "failed", "no alignment",
        "unalignable", "skipping", "could not"
    ]

    matches = [
        (i, line) for i, line in enumerate(lines)
        if any(k in line.lower() for k in keywords)
    ]

    print("Total log lines:", len(lines))
    print("Diagnostic lines:", len(matches))

    for i, line in matches[-100:]:
        print(f"{i + 1}: {line[:500]}")
else:
    print("MFA log not found.")

print("\n" + "=" * 65)
print("TEMPORARY FILE SEARCH")
print("=" * 65)

found = {}

if TEMP.exists():
    for name in missing:
        matches = [
            str(p) for p in TEMP.rglob("*")
            if p.is_file() and name in p.name
        ]
        found[name] = matches
        print(name, "=>", matches[:5])
else:
    print("Temporary directory not found.")

output = {
    "missing_recordings": missing,
    "diagnostic_lines": [
        {"line": i + 1, "text": line}
        for i, line in matches
    ] if LOG.exists() else [],
    "temporary_files": found
}

save = ENG / "alignment" / "logs" / "english_train_failure_diagnostics.json"
save.write_text(json.dumps(output, indent=2))

print("\nSaved:", save)
print("STEP 115 COMPLETE")

MFA LOG DIAGNOSTICS
Total log lines: 36
Diagnostic lines: 0

TEMPORARY FILE SEARCH
recording_1558 => []
recording_2157 => []
recording_2282 => []
recording_2362 => []
recording_2371 => []
recording_282 => []
recording_3253 => []
recording_3274 => []
recording_3397 => []
recording_3515 => []
recording_3983 => []
recording_4688 => []
recording_4707 => []
recording_5283 => []
recording_5595 => []
recording_5712 => []
recording_6486 => []

Saved: /content/drive/MyDrive/FastSpeech2_Assignment/English/alignment/logs/english_train_failure_diagnostics.json
STEP 115 COMPLETE


In [49]:
# ================================================================
# STEP 116 — RETRY ONLY 17 MISSING TRAIN ALIGNMENTS
# RUN NOW — NEW CELL
# Original dataset and 2,647 successful alignments untouched.
# ================================================================

import os
import shutil
import subprocess
from pathlib import Path
import json

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

MFA = "/content/mfa-env/bin/mfa"
ENV = "/content/mfa-env"

missing = [
    "recording_1558", "recording_2157", "recording_2282",
    "recording_2362", "recording_2371", "recording_282",
    "recording_3253", "recording_3274", "recording_3397",
    "recording_3515", "recording_3983", "recording_4688",
    "recording_4707", "recording_5283", "recording_5595",
    "recording_5712", "recording_6486"
]

SOURCE = ENG / "raw_audio" / "train"
RETRY = ENG / "alignment" / "retry_17"
CORPUS = RETRY / "corpus"
OUTPUT = RETRY / "output"
TEMP = RETRY / "temporary"

for folder in [CORPUS, OUTPUT, TEMP]:
    folder.mkdir(parents=True, exist_ok=True)

# Prepare isolated retry corpus
for name in missing:
    wav = SOURCE / f"{name}.wav"
    lab = SOURCE / f"{name}.lab"

    if not wav.exists() or not lab.exists():
        raise FileNotFoundError(f"Missing source pair: {name}")

    shutil.copy2(wav, CORPUS / wav.name)
    shutil.copy2(lab, CORPUS / lab.name)

print("Retry WAV files:", len(list(CORPUS.glob("*.wav"))))
print("Retry LAB files:", len(list(CORPUS.glob("*.lab"))))

# Do not overwrite a previous retry result
existing = list(OUTPUT.rglob("*.TextGrid")) + list(
    OUTPUT.rglob("*.textgrid")
)
if existing:
    raise RuntimeError(
        "Retry output already contains TextGrids. "
        "Inspect them before running again."
    )

env = os.environ.copy()
env["PATH"] = f"{ENV}/bin:" + env.get("PATH", "")

command = [
    MFA, "align_hf",
    str(CORPUS),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT),
    "--dialect", "english_india",
    "--use_g2p",
    "--num_jobs", "2",
    "--single_speaker",
    "--temporary_directory", str(TEMP),
    "--no_clean",
    "--verbose"
]

print("\nStarting isolated retry...")
result = subprocess.run(
    command,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

log = RETRY / "retry_alignment.log"
log.write_text(result.stdout)

textgrids = list(OUTPUT.rglob("*.TextGrid")) + list(
    OUTPUT.rglob("*.textgrid")
)

generated = {p.stem for p in textgrids}
successful = sorted(set(missing) & generated)
still_missing = sorted(set(missing) - generated)

summary = {
    "attempted": len(missing),
    "generated": len(successful),
    "still_missing": still_missing,
    "return_code": result.returncode
}

(RETRY / "retry_summary.json").write_text(
    json.dumps(summary, indent=2)
)

print("\nMFA OUTPUT — LAST 8000 CHARACTERS")
print(result.stdout[-8000:])

print("\nRetry return code:", result.returncode)
print("Retry TextGrids:", len(textgrids))
print("Successful:", successful)
print("Still missing:", still_missing)

print("\nSTEP 116 COMPLETE")

Retry WAV files: 17
Retry LAB files: 17

Starting isolated retry...

MFA OUTPUT — LAST 8000 CHARACTERS

Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 921.31it/s]
 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
  17% ━━━━━╸                              17/100  [ 0:00:01 < 0:00:48 , 2 it/s ]
 INFO     Found 1 speaker across 17 files, average number of utterances per     
          speaker: 17.0                                                         
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Generating pronunciations...                                          
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12  [ 0:00:02 < 0:00:00 , 4 it/s ]
 INFO     Generati

In [50]:
# STEP 117 — MERGE RECOVERED ALIGNMENTS
# RUN NOW | NEW CELL | SAFE: no existing files overwritten

from pathlib import Path
import shutil
import json

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")

RETRY_OUT = BASE / "alignment/retry_17/output"
MAIN_OUT = BASE / "alignment/full_alignment/train"
LOG_DIR = BASE / "alignment/logs"

MAIN_OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

retry_files = (
    list(RETRY_OUT.rglob("*.TextGrid")) +
    list(RETRY_OUT.rglob("*.textgrid"))
)

copied = []
already_present = []

for src in retry_files:
    dest = MAIN_OUT / src.name

    if dest.exists():
        already_present.append(src.stem)
    else:
        shutil.copy2(src, dest)
        copied.append(src.stem)

# Verify against original training audio
wav_dir = BASE / "raw_audio/train"
wav_stems = {p.stem for p in wav_dir.glob("*.wav")}

grid_stems = {
    p.stem for p in MAIN_OUT.iterdir()
    if p.suffix.lower() == ".textgrid"
}

missing = sorted(wav_stems - grid_stems)
unexpected = sorted(grid_stems - wav_stems)

summary = {
    "training_audio_count": len(wav_stems),
    "textgrid_count": len(grid_stems),
    "copied_from_retry": sorted(copied),
    "already_present": sorted(already_present),
    "missing": missing,
    "unexpected": unexpected,
    "alignment_success_rate": round(
        100 * len(wav_stems & grid_stems) / len(wav_stems), 3
    )
}

(LOG_DIR / "english_train_final_alignment_audit.json").write_text(
    json.dumps(summary, indent=2)
)

print(json.dumps(summary, indent=2))
print("\nSTEP 117 COMPLETE")

{
  "training_audio_count": 2664,
  "textgrid_count": 2662,
  "copied_from_retry": [
    "recording_1558",
    "recording_2157",
    "recording_2282",
    "recording_2362",
    "recording_2371",
    "recording_282",
    "recording_3274",
    "recording_3397",
    "recording_3983",
    "recording_4688",
    "recording_4707",
    "recording_5283",
    "recording_5595",
    "recording_5712",
    "recording_6486"
  ],
  "already_present": [],
  "missing": [
    "recording_3253",
    "recording_3515"
  ],
  "unexpected": [],
  "alignment_success_rate": 99.925
}

STEP 117 COMPLETE


In [51]:
# ================================================================
# STEP 118 — ENGLISH VALIDATION MFA ALIGNMENT
# RUN NOW | NEW CELL
# ================================================================

from pathlib import Path
import subprocess
import os
import json
import time

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENG = BASE / "English"

CORPUS = ENG / "raw_audio/validation"
OUTPUT = ENG / "alignment/full_alignment/validation"
TEMP = ENG / "alignment/temporary/validation"
LOG_DIR = ENG / "alignment/logs"

for folder in [OUTPUT, TEMP, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

wav_files = list(CORPUS.glob("*.wav"))
lab_files = list(CORPUS.glob("*.lab"))

print("Validation WAV files:", len(wav_files))
print("Validation LAB files:", len(lab_files))

assert len(wav_files) == 505
assert {p.stem for p in wav_files} == {p.stem for p in lab_files}

existing = list(OUTPUT.rglob("*.TextGrid")) + list(
    OUTPUT.rglob("*.textgrid")
)

if existing:
    raise RuntimeError(
        f"Validation output already contains {len(existing)} TextGrids. "
        "Inspect existing results before rerunning."
    )

env = os.environ.copy()
env["PATH"] = "/content/mfa-env/bin:" + env.get("PATH", "")

command = [
    "/content/mfa-env/bin/mfa",
    "align_hf",
    str(CORPUS),
    "MontrealCorpusTools/english_mfa",
    str(OUTPUT),
    "--dialect", "english_india",
    "--use_g2p",
    "--num_jobs", "2",
    "--single_speaker",
    "--temporary_directory", str(TEMP),
    "--no_clean",
    "--verbose"
]

print("\nStarting English validation alignment...")
start = time.time()

result = subprocess.run(
    command,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

elapsed = time.time() - start

log_path = LOG_DIR / "english_validation_alignment.log"
log_path.write_text(result.stdout)

grids = list(OUTPUT.rglob("*.TextGrid")) + list(
    OUTPUT.rglob("*.textgrid")
)

wav_stems = {p.stem for p in wav_files}
grid_stems = {p.stem for p in grids}

summary = {
    "validation_audio_count": len(wav_stems),
    "textgrid_count": len(grid_stems),
    "missing": sorted(wav_stems - grid_stems),
    "unexpected": sorted(grid_stems - wav_stems),
    "alignment_success_rate": round(
        100 * len(wav_stems & grid_stems) / len(wav_stems), 3
    ),
    "return_code": result.returncode,
    "elapsed_seconds": round(elapsed, 2)
}

summary_path = LOG_DIR / "english_validation_alignment_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print("\nMFA OUTPUT — LAST 4000 CHARACTERS")
print(result.stdout[-4000:])

print("\nVALIDATION ALIGNMENT SUMMARY")
print(json.dumps(summary, indent=2))

print("\nSTEP 118 COMPLETE")

Validation WAV files: 505
Validation LAB files: 505

Starting English validation alignment...

MFA OUTPUT — LAST 4000 CHARACTERS

Fetching 42 files: 100%|██████████| 42/42 [00:00<00:00, 1574.80it/s]
 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/100  [ 0:00:04 < 0:00:00 , 60 it/s ]
 INFO     Found 1 speaker across 505 files, average number of utterances per    
          speaker: 505.0                                                        
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505/505  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Generating pronunciations...                                          
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326/326  [ 0:01:52 < 0:00:00 , 

In [52]:
# ================================================================
# STEP 119 — ENGLISH TEXTGRID STRUCTURE INSPECTION
# RUN NOW | NEW CELL | READ-ONLY
# ================================================================

from pathlib import Path
import json
import soundfile as sf

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")

GRID_DIR = BASE / "alignment/full_alignment/train"
AUDIO_DIR = BASE / "raw_audio/train"

grids = sorted(GRID_DIR.glob("*.TextGrid"))
if not grids:
    grids = sorted(GRID_DIR.glob("*.textgrid"))

assert grids, "No training TextGrids found."

# Inspect a few examples
selected = grids[:5]

try:
    from praatio import textgrid
except ImportError:
    print("Installing praatio...")
    %pip -q install praatio
    from praatio import textgrid

summary = []

for grid_path in selected:
    tg = textgrid.openTextgrid(
        str(grid_path),
        includeEmptyIntervals=True
    )

    tiers = []

    for tier_name in tg.tierNames:
        tier = tg.getTier(tier_name)
        intervals = tier.entries

        labels = [
            str(entry.label).strip()
            for entry in intervals
            if str(entry.label).strip()
        ]

        tiers.append({
            "tier": tier_name,
            "interval_count": len(intervals),
            "nonempty_labels": len(labels),
            "sample_labels": labels[:25]
        })

    wav_path = AUDIO_DIR / f"{grid_path.stem}.wav"
    audio_duration = None

    if wav_path.exists():
        info = sf.info(str(wav_path))
        audio_duration = info.duration

    summary.append({
        "recording": grid_path.stem,
        "audio_duration": audio_duration,
        "tiers": tiers
    })

print(json.dumps(summary, indent=2, ensure_ascii=False))

# Check the two unresolved training recordings
print("\nUNRESOLVED TRAINING RECORDINGS")

for name in ["recording_3253", "recording_3515"]:
    wav = AUDIO_DIR / f"{name}.wav"
    lab = AUDIO_DIR / f"{name}.lab"

    print({
        "recording": name,
        "wav_exists": wav.exists(),
        "lab_exists": lab.exists(),
        "transcript": lab.read_text(errors="replace")[:300]
        if lab.exists() else None
    })

print("\nSTEP 119 COMPLETE")

[
  {
    "recording": "recording_0",
    "audio_duration": 8.7595,
    "tiers": [
      {
        "tier": "words",
        "interval_count": 26,
        "nonempty_labels": 22,
        "sample_labels": [
          "our",
          "district",
          "hasana",
          "was",
          "the",
          "land",
          "of",
          "the",
          "great",
          "hoysala",
          "dynasty",
          "who",
          "ruled",
          "over",
          "this",
          "region",
          "from",
          "11th",
          "to",
          "14th",
          "century",
          "ce"
        ]
      },
      {
        "tier": "phones",
        "interval_count": 82,
        "nonempty_labels": 78,
        "sample_labels": [
          "aː",
          "ɖ",
          "ɪ",
          "s",
          "ʈʲ",
          "ɹ",
          "ɪ",
          "k",
          "ʈ",
          "ç",
          "eː",
          "s",
          "ə",
          "n",
          "ə",
          "ʋ",
         

In [53]:
# ================================================================
# STEP 120 — ENGLISH PHONE VOCABULARY COMPATIBILITY AUDIT
# RUN NOW | NEW CELL | READ-ONLY
# ================================================================

from pathlib import Path
from collections import Counter
import json
import sys

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")
GRID_DIR = BASE / "alignment/full_alignment/train"

sys.path.insert(0, "/content/FastSpeech2")

from praatio import textgrid
from text.symbols import symbols

# All supported FastSpeech 2 symbols
model_symbols = set(symbols)

# Collect every non-empty MFA phone label
phone_counts = Counter()
tier_names = Counter()
examples = {}

for path in GRID_DIR.iterdir():
    if path.suffix.lower() != ".textgrid":
        continue

    tg = textgrid.openTextgrid(
        str(path),
        includeEmptyIntervals=True
    )

    for tier_name in tg.tierNames:
        tier_names[tier_name] += 1

        if tier_name.lower() != "phones":
            continue

        tier = tg.getTier(tier_name)

        for entry in tier.entries:
            label = str(entry.label).strip()
            if label:
                phone_counts[label] += 1
                examples.setdefault(label, path.stem)

supported = {
    phone for phone in phone_counts
    if phone in model_symbols
}

unsupported = {
    phone: count
    for phone, count in phone_counts.items()
    if phone not in model_symbols
}

summary = {
    "aligned_recordings_scanned": len(list(GRID_DIR.glob("*.TextGrid"))),
    "unique_mfa_phones": len(phone_counts),
    "total_phone_intervals": sum(phone_counts.values()),
    "unique_phones_supported_directly": len(supported),
    "unique_phones_not_supported_directly": len(unsupported),
    "unsupported_phone_examples": [
        {
            "phone": phone,
            "count": count,
            "recording": examples[phone]
        }
        for phone, count in sorted(
            unsupported.items(),
            key=lambda x: -x[1]
        )[:100]
    ],
    "model_symbol_examples": sorted(model_symbols)[:100],
    "tier_names": dict(tier_names)
}

output = BASE / "alignment/logs/english_phone_vocabulary_audit.json"
output.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nSaved:", output)
print("\nSTEP 120 COMPLETE")

{
  "aligned_recordings_scanned": 2662,
  "unique_mfa_phones": 64,
  "total_phone_intervals": 139645,
  "unique_phones_supported_directly": 14,
  "unique_phones_not_supported_directly": 50,
  "unsupported_phone_examples": [
    {
      "phone": "ə",
      "count": 15577,
      "recording": "recording_914"
    },
    {
      "phone": "ɪ",
      "count": 11041,
      "recording": "recording_914"
    },
    {
      "phone": "ʈ",
      "count": 8099,
      "recording": "recording_6141"
    },
    {
      "phone": "ʋ",
      "count": 5112,
      "recording": "recording_389"
    },
    {
      "phone": "ɖ",
      "count": 4913,
      "recording": "recording_914"
    },
    {
      "phone": "ɛ",
      "count": 3960,
      "recording": "recording_914"
    },
    {
      "phone": "ɹ",
      "count": 3881,
      "recording": "recording_6132"
    },
    {
      "phone": "d̪",
      "count": 3352,
      "recording": "recording_389"
    },
    {
      "phone": "eː",
      "count": 2917,
      "reco

In [54]:
# ================================================================
# STEP 121 — INSPECT ENGLISH TEXT PIPELINE
# RUN NOW | NEW CELL | READ-ONLY
# ================================================================

from pathlib import Path

REPO = Path("/content/FastSpeech2")

files = [
    REPO / "text/symbols.py",
    REPO / "text/__init__.py",
    REPO / "text/cleaners.py",
    REPO / "model/fastspeech2.py",
    REPO / "model/modules.py",
]

for path in files:
    print("\n" + "=" * 75)
    print("FILE:", path)
    print("=" * 75)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    content = path.read_text(errors="replace")

    if path.name == "symbols.py":
        print(content[:14000])

    elif path.name == "__init__.py":
        print(content[:10000])

    elif path.name == "cleaners.py":
        print(content[:10000])

    else:
        lines = content.splitlines()

        keywords = [
            "src_word_emb",
            "len(symbols)",
            "text_to_sequence",
            "sequence_to_text",
            "text_cleaners",
            "symbols"
        ]

        for i, line in enumerate(lines):
            if any(k in line for k in keywords):
                start = max(0, i - 4)
                end = min(len(lines), i + 8)

                print(f"\n--- Lines {start+1} to {end} ---")
                print("\n".join(lines[start:end]))

print("\nSTEP 121 COMPLETE")


FILE: /content/FastSpeech2/text/symbols.py
""" from https://github.com/keithito/tacotron """

"""
Defines the set of symbols used in text input to the model.

The default is a set of ASCII characters that works well for English or text that has been run through Unidecode. For other data, you can modify _characters. See TRAINING_DATA.md for details. """

from text import cmudict, pinyin

_pad = "_"
_punctuation = "!'(),.:;? "
_special = "-"
_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
_silences = ["@sp", "@spn", "@sil"]

# Prepend "@" to ARPAbet symbols to ensure uniqueness (some are the same as uppercase letters):
_arpabet = ["@" + s for s in cmudict.valid_symbols]
_pinyin = ["@" + s for s in pinyin.valid_symbols]

# Export all symbols:
symbols = (
    [_pad]
    + list(_special)
    + list(_punctuation)
    + list(_letters)
    + _arpabet
    + _pinyin
    + _silences
)


FILE: /content/FastSpeech2/text/__init__.py
""" from https://github.com/keithito/tacotron ""

In [55]:
# STEP 122 — INSPECT MODEL EMBEDDING AND TEXT INPUT
# RUN NOW | NEW CELL | READ-ONLY

from pathlib import Path

files = {
    "FastSpeech2": Path("/content/FastSpeech2/model/fastspeech2.py"),
    "Modules": Path("/content/FastSpeech2/model/modules.py"),
    "Encoder": Path("/content/FastSpeech2/model/transformer/Models.py"),
}

for name, path in files.items():
    print("\n" + "=" * 80)
    print(name, "-", path)
    print("=" * 80)

    if not path.exists():
        print("Not found")
        continue

    lines = path.read_text(errors="replace").splitlines()

    if name == "FastSpeech2":
        ranges = [(0, min(180, len(lines)))]
    else:
        ranges = [
            (i, min(i + 35, len(lines)))
            for i, line in enumerate(lines)
            if any(key in line.lower() for key in [
                "embedding", "src_word_emb", "n_src_vocab",
                "class encoder", "class fastspeech"
            ])
        ]

    printed = set()

    for start, end in ranges:
        if start in printed:
            continue
        printed.add(start)
        print(f"\n--- Lines {start+1}-{end} ---")
        print("\n".join(lines[start:end]))

print("\nSTEP 122 COMPLETE")


FastSpeech2 - /content/FastSpeech2/model/fastspeech2.py

--- Lines 1-110 ---
import os
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformer import Encoder, Decoder, PostNet
from .modules import VarianceAdaptor
from utils.tools import get_mask_from_lengths


class FastSpeech2(nn.Module):
    """ FastSpeech2 """

    def __init__(self, preprocess_config, model_config):
        super(FastSpeech2, self).__init__()
        self.model_config = model_config

        self.encoder = Encoder(model_config)
        self.variance_adaptor = VarianceAdaptor(preprocess_config, model_config)
        self.decoder = Decoder(model_config)
        self.mel_linear = nn.Linear(
            model_config["transformer"]["decoder_hidden"],
            preprocess_config["preprocessing"]["mel"]["n_mel_channels"],
        )
        self.postnet = PostNet()

        self.speaker_emb = None
        if model_config["multi_speaker"]:
            with open(
                o

In [56]:
# STEP 123 — FIND ENCODER VOCABULARY DEFINITION

from pathlib import Path

repo = Path("/content/FastSpeech2")

for path in repo.rglob("*.py"):
    if "transformer" not in str(path).lower():
        continue

    content = path.read_text(errors="replace")
    lines = content.splitlines()

    matches = [
        i for i, line in enumerate(lines)
        if any(term in line.lower() for term in [
            "class encoder",
            "src_word_emb",
            "n_src_vocab",
            "nn.embedding"
        ])
    ]

    if matches:
        print("\n" + "=" * 70)
        print(path)
        print("=" * 70)

        shown = set()
        for i in matches:
            start = max(0, i - 8)
            end = min(len(lines), i + 15)

            if start in shown:
                continue

            shown.add(start)
            print("\n".join(
                f"{j+1}: {lines[j]}"
                for j in range(start, end)
            ))

print("\nSTEP 123 COMPLETE")


/content/FastSpeech2/transformer/Models.py
25: 
26:     if padding_idx is not None:
27:         # zero vector for padding dimension
28:         sinusoid_table[padding_idx] = 0.0
29: 
30:     return torch.FloatTensor(sinusoid_table)
31: 
32: 
33: class Encoder(nn.Module):
34:     """ Encoder """
35: 
36:     def __init__(self, config):
37:         super(Encoder, self).__init__()
38: 
39:         n_position = config["max_seq_len"] + 1
40:         n_src_vocab = len(symbols) + 1
41:         d_word_vec = config["transformer"]["encoder_hidden"]
42:         n_layers = config["transformer"]["encoder_layer"]
43:         n_head = config["transformer"]["encoder_head"]
44:         d_k = d_v = (
45:             config["transformer"]["encoder_hidden"]
46:             // config["transformer"]["encoder_head"]
47:         )
32: 
33: class Encoder(nn.Module):
34:     """ Encoder """
35: 
36:     def __init__(self, config):
37:         super(Encoder, self).__init__()
38: 
39:         n_position = config

In [57]:
# STEP 124 — INSPECT ENCODER IMPORTS AND ENGLISH CONFIG

from pathlib import Path

repo = Path("/content/FastSpeech2")

files = [
    repo / "transformer/Models.py",
    repo / "config/LJSpeech/preprocess.yaml",
    repo / "config/LJSpeech/model.yaml",
    repo / "config/LJSpeech/train.yaml",
]

for path in files:
    print("\n" + "=" * 75)
    print(path)
    print("=" * 75)

    if path.exists():
        print(path.read_text(errors="replace")[:12000])
    else:
        print("Not found")

print("\nSTEP 124 COMPLETE")


/content/FastSpeech2/transformer/Models.py
import torch
import torch.nn as nn
import numpy as np

import transformer.Constants as Constants
from .Layers import FFTBlock
from text.symbols import symbols


def get_sinusoid_encoding_table(n_position, d_hid, padding_idx=None):
    """ Sinusoid position encoding table """

    def cal_angle(position, hid_idx):
        return position / np.power(10000, 2 * (hid_idx // 2) / d_hid)

    def get_posi_angle_vec(position):
        return [cal_angle(position, hid_j) for hid_j in range(d_hid)]

    sinusoid_table = np.array(
        [get_posi_angle_vec(pos_i) for pos_i in range(n_position)]
    )

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # dim 2i
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # dim 2i+1

    if padding_idx is not None:
        # zero vector for padding dimension
        sinusoid_table[padding_idx] = 0.0

    return torch.FloatTensor(sinusoid_table)


class Encoder(nn.Module):
    """ Encoder "

In [2]:
%pip install -q praatio

# Verify installation
import praatio
from praatio import textgrid

print("praatio installed successfully")
print("Version:", getattr(praatio, "__version__", "installed"))

praatio installed successfully
Version: installed


In [6]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive if it is not already mounted
drive.mount('/content/gdrive')

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"

print("\nAssignment folder exists:", BASE.exists())
print("English folder exists:", ENGLISH.exists())

# Inspect English folder
if ENGLISH.exists():
    print("\nContents of English folder:")
    for p in sorted(ENGLISH.iterdir()):
        print(" -", p.name)

    print("\nAlignment folder contents:")
    alignment = ENGLISH / "alignment"
    if alignment.exists():
        for p in sorted(alignment.rglob("*")):
            if p.is_dir():
                print("DIR :", p)
    else:
        print("Alignment folder not found.")
else:
    print("English folder not found. Check the assignment folder in Google Drive.")

Mounted at /content/gdrive

Assignment folder exists: True
English folder exists: True

Contents of English folder:
 - config

Alignment folder contents:
Alignment folder not found.


In [10]:
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")

print("Assignment root:", ROOT)
print("\nTop-level folders:")

for p in sorted(ROOT.iterdir()):
    print(("DIR  " if p.is_dir() else "FILE "), p.name)

print("\nSearching for existing alignment files...")

textgrids = list(ROOT.rglob("*.TextGrid")) + list(ROOT.rglob("*.textgrid"))

print("TextGrid files found:", len(textgrids))

for p in textgrids[:20]:
    print(p)

print("\nSearching for English audio files...")

wav_files = list((ROOT / "English").rglob("*.wav"))

print("English WAV files found:", len(wav_files))

for p in wav_files[:10]:
    print(p)

print("\nEnglish folder structure:")

english = ROOT / "English"

for p in sorted(english.rglob("*")):
    if p.is_dir():
        print("DIR :", p.relative_to(ROOT))

Assignment root: /content/gdrive/MyDrive/FastSpeech2_Assignment

Top-level folders:
DIR   English
DIR   Telugu
DIR   metadata
DIR   raw_data

Searching for existing alignment files...
TextGrid files found: 3188
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/retry_17/output/recording_6486.TextGrid
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/retry_17/output/recording_1558.TextGrid
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/retry_17/output/recording_3983.TextGrid
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/retry_17/output/recording_4707.TextGrid
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/retry_17/output/recording_5283.TextGrid
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/retry_17/output/recording_2362.TextGrid
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/retry_17/output/recording_5595.TextGrid
/content/gdrive/MyDrive/FastSpeech2_Assignment/English

In [17]:
from pathlib import Path
from collections import Counter
from praatio import textgrid
import json
import yaml
import shutil

# ============================================================
# PATHS
# ============================================================

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"

GRID_DIR = ENGLISH / "alignment/full_alignment/train"
CONFIG_DIR = ENGLISH / "config"
REPO = Path("/content/FastSpeech2")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

assert GRID_DIR.exists(), f"Missing alignment folder: {GRID_DIR}"
assert REPO.exists(), "FastSpeech2 repository not found. Restore the repository first."

# ============================================================
# 1. EXTRACT PHONE INVENTORY FROM MFA TEXTGRIDS
# ============================================================

phone_counts = Counter()
textgrid_files = sorted(GRID_DIR.glob("*.TextGrid"))

print("Training TextGrids found:", len(textgrid_files))
assert len(textgrid_files) > 0, "No TextGrid files found."

for i, path in enumerate(textgrid_files):
    tg = textgrid.openTextgrid(
        str(path),
        includeEmptyIntervals=True
    )

    tier_names = tg.tierNames
    phone_tier_name = next(
        (name for name in tier_names
         if name.lower() == "phones"),
        None
    )

    if phone_tier_name is None:
        raise ValueError(f"No phones tier in {path.name}")

    tier = tg.getTier(phone_tier_name)

    for interval in tier.entries:
        label = str(interval.label).strip()
        if label:
            phone_counts[label] += 1

    if (i + 1) % 500 == 0:
        print(f"Scanned {i + 1}/{len(textgrid_files)} files")

print("\nUnique MFA phones:", len(phone_counts))
print("Total labelled phone intervals:", sum(phone_counts.values()))

# ============================================================
# 2. CREATE DETERMINISTIC VOCABULARY
# ============================================================

PAD = "<pad>"
UNK = "<unk>"
SIL = "<sil>"

phone_labels = sorted(phone_counts.keys())

assert PAD not in phone_labels
assert UNK not in phone_labels
assert SIL not in phone_labels

vocab = [PAD, UNK, SIL] + phone_labels

phone_to_id = {phone: idx for idx, phone in enumerate(vocab)}
id_to_phone = {str(idx): phone for idx, phone in enumerate(vocab)}

vocab_data = {
    "vocabulary": vocab,
    "phone_to_id": phone_to_id,
    "id_to_phone": id_to_phone,
    "phone_counts": dict(phone_counts),
    "special_tokens": {
        "pad": PAD,
        "unk": UNK,
        "silence": SIL
    },
    "vocab_size": len(vocab)
}

VOCAB_PATH = CONFIG_DIR / "english_ipa_vocabulary.json"

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)

print("\nVocabulary saved:", VOCAB_PATH)
print("Vocabulary size:", len(vocab))

# ============================================================
# 3. COPY ORIGINAL FASTSPEECH2 CONFIGS
# ============================================================

SOURCE_CONFIG = REPO / "config/LJSpeech"

config_files = {
    "preprocess.yaml": SOURCE_CONFIG / "preprocess.yaml",
    "model.yaml": SOURCE_CONFIG / "model.yaml",
    "train.yaml": SOURCE_CONFIG / "train.yaml"
}

for name, source in config_files.items():
    assert source.exists(), f"Missing source config: {source}"
    shutil.copy2(source, CONFIG_DIR / name)

print("\nOriginal configuration files copied.")

# ============================================================
# 4. MODIFY PREPROCESS CONFIGURATION
# ============================================================

PREPROCESS_PATH = CONFIG_DIR / "preprocess.yaml"

with open(PREPROCESS_PATH, "r") as f:
    preprocess = yaml.safe_load(f)

preprocess["dataset"] = "Svarah_MFA_IPA"

preprocess.setdefault("path", {})
preprocess["path"]["corpus_path"] = str(ENGLISH / "raw_audio")
preprocess["path"]["raw_path"] = str(ENGLISH / "raw_audio")
preprocess["path"]["preprocessed_path"] = str(ENGLISH / "processed")

preprocess.setdefault("preprocessing", {})
preprocessing = preprocess["preprocessing"]

preprocessing["text"] = {
    "language": "en_ipa",
    "text_cleaners": [],
    "use_phonemes": True
}

preprocessing.setdefault("pitch", {})
preprocessing["pitch"]["feature"] = "phoneme_level"

preprocessing.setdefault("energy", {})
preprocessing["energy"]["feature"] = "phoneme_level"

with open(PREPROCESS_PATH, "w") as f:
    yaml.safe_dump(preprocess, f, sort_keys=False)

# ============================================================
# 5. MODIFY MODEL CONFIGURATION FOR CPU EXPERIMENTATION
# ============================================================

MODEL_PATH = CONFIG_DIR / "model.yaml"

with open(MODEL_PATH, "r") as f:
    model = yaml.safe_load(f)

model["input_vocabulary_size"] = len(vocab)

transformer = model["transformer"]
transformer["encoder_layer"] = 2
transformer["decoder_layer"] = 2
transformer["encoder_head"] = 2
transformer["decoder_head"] = 2
transformer["encoder_hidden"] = 128
transformer["decoder_hidden"] = 128
transformer["conv_filter_size"] = 512
transformer["conv_kernel_size"] = [9, 1]
transformer["encoder_dropout"] = 0.1
transformer["decoder_dropout"] = 0.1

variance = model["variance_predictor"]
variance["filter_size"] = 128
variance["dropout"] = 0.2

model["max_seq_len"] = 2048
model["multi_speaker"] = False

with open(MODEL_PATH, "w") as f:
    yaml.safe_dump(model, f, sort_keys=False)

# ============================================================
# 6. MODIFY TRAIN CONFIGURATION
# ============================================================

TRAIN_PATH = CONFIG_DIR / "train.yaml"

with open(TRAIN_PATH, "r") as f:
    train = yaml.safe_load(f)

train.setdefault("path", {})
train["path"]["ckpt_path"] = str(ENGLISH / "checkpoints")
train["path"]["log_path"] = str(ENGLISH / "logs")
train["path"]["result_path"] = str(ENGLISH / "results")

Path(train["path"]["ckpt_path"]).mkdir(parents=True, exist_ok=True)
Path(train["path"]["log_path"]).mkdir(parents=True, exist_ok=True)
Path(train["path"]["result_path"]).mkdir(parents=True, exist_ok=True)

with open(TRAIN_PATH, "w") as f:
    yaml.safe_dump(train, f, sort_keys=False)

# ============================================================
# 7. VERIFY OUTPUTS
# ============================================================

print("\n========== STEP 125D COMPLETE ==========")
print("Vocabulary:", VOCAB_PATH.exists())
print("Preprocess config:", PREPROCESS_PATH.exists())
print("Model config:", MODEL_PATH.exists())
print("Train config:", TRAIN_PATH.exists())

print("\nVocabulary size:", len(vocab))
print("First 15 vocabulary entries:", vocab[:15])
print("Special token IDs:", {
    PAD: phone_to_id[PAD],
    UNK: phone_to_id[UNK],
    SIL: phone_to_id[SIL]
})

print("\nEnglish configuration directory:")
for p in sorted(CONFIG_DIR.iterdir()):
    if p.is_file():
        print(" -", p.name)

print("\nIMPORTANT: Do not start training yet.")

Training TextGrids found: 2662
Scanned 500/2662 files
Scanned 1000/2662 files
Scanned 1500/2662 files
Scanned 2000/2662 files
Scanned 2500/2662 files

Unique MFA phones: 64
Total labelled phone intervals: 139645

Vocabulary saved: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/english_ipa_vocabulary.json
Vocabulary size: 67

Original configuration files copied.

========== STEP 125D COMPLETE ==========
Vocabulary: True
Preprocess config: True
Model config: True
Train config: True

Vocabulary size: 67
First 15 vocabulary entries: ['<pad>', '<unk>', '<sil>', 'a', 'aj', 'aw', 'aː', 'b', 'bʲ', 'c', 'cʷ', 'dʒ', 'd̪', 'eː', 'f']
Special token IDs: {'<pad>': 0, '<unk>': 1, '<sil>': 2}

English configuration directory:
 - english_ipa_vocabulary.json
 - model.yaml
 - preprocess.yaml
 - train.yaml

IMPORTANT: Do not start training yet.


In [12]:
from pathlib import Path

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"

TRAIN_GRID_DIR = ENGLISH / "alignment/full_alignment/train"
VAL_GRID_DIR = ENGLISH / "alignment/full_alignment/validation"

for name, folder in [
    ("Training", TRAIN_GRID_DIR),
    ("Validation", VAL_GRID_DIR)
]:
    files = list(folder.glob("*.TextGrid")) if folder.exists() else []
    print(f"{name}:")
    print("  Folder exists:", folder.exists())
    print("  TextGrid files:", len(files))
    print("  Path:", folder)

Training:
  Folder exists: True
  TextGrid files: 2662
  Path: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/full_alignment/train
Validation:
  Folder exists: True
  TextGrid files: 501
  Path: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignment/full_alignment/validation


In [15]:
from pathlib import Path
import subprocess

REPO = Path("/content/FastSpeech2")

if REPO.exists():
    print("Repository already exists:", REPO)
else:
    print("Cloning FastSpeech2 repository...")

    result = subprocess.run(
        [
            "git", "clone",
            "https://github.com/ming024/FastSpeech2.git",
            str(REPO)
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Repository cloning failed.")

assert (REPO / "config/LJSpeech").exists()
assert (REPO / "transformer/Models.py").exists()

print("\nFastSpeech2 repository restored successfully.")
print("Repository:", REPO)
print("LJSpeech configs:", list((REPO / "config/LJSpeech").iterdir()))

Cloning FastSpeech2 repository...


FastSpeech2 repository restored successfully.
Repository: /content/FastSpeech2
LJSpeech configs: [PosixPath('/content/FastSpeech2/config/LJSpeech/preprocess.yaml'), PosixPath('/content/FastSpeech2/config/LJSpeech/model.yaml'), PosixPath('/content/FastSpeech2/config/LJSpeech/train.yaml')]


In [19]:
from pathlib import Path
import json
import yaml

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
CONFIG = BASE / "English/config"
REPO = Path("/content/FastSpeech2")

files = [
    CONFIG / "english_ipa_vocabulary.json",
    CONFIG / "preprocess.yaml",
    CONFIG / "model.yaml",
    CONFIG / "train.yaml",
]

print("========== VERIFICATION ==========")

for file in files:
    print(file.name, ":", "EXISTS" if file.exists() else "MISSING")

print("\nFastSpeech2 repository:", REPO.exists())
print("Encoder source:", (REPO / "transformer/Models.py").exists())

if all(f.exists() for f in files):
    with open(CONFIG / "english_ipa_vocabulary.json", encoding="utf-8") as f:
        vocab = json.load(f)

    with open(CONFIG / "model.yaml") as f:
        model = yaml.safe_load(f)

    print("\nVocabulary size:", vocab["vocab_size"])
    print("Silence token ID:", vocab["phone_to_id"]["<sil>"])
    print("Configured vocabulary size:", model["input_vocabulary_size"])

    assert vocab["vocab_size"] == model["input_vocabulary_size"]

    print("\nALL CHECKS PASSED.")
else:
    print("\nSome configuration files are missing.")

========== VERIFICATION ==========
english_ipa_vocabulary.json : EXISTS
preprocess.yaml : EXISTS
model.yaml : EXISTS
train.yaml : EXISTS

FastSpeech2 repository: True
Encoder source: True

Vocabulary size: 67
Silence token ID: 2
Configured vocabulary size: 67

ALL CHECKS PASSED.


In [2]:
%pip install -q praatio
from praatio import textgrid
print("praatio ready")

praatio ready


In [3]:
# ================================================================
# STEP 125 — CREATE ENGLISH-SPECIFIC IPA VOCABULARY
# RUN NOW | NEW CELL
# ================================================================

from pathlib import Path
from praatio import textgrid
from collections import Counter
import json
import yaml

REPO = Path("/content/FastSpeech2")
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/English")

GRID_DIR = BASE / "alignment/full_alignment/train"
CONFIG_DIR = BASE / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# Collect every phone from successful training alignments
phone_counts = Counter()

for path in GRID_DIR.iterdir():
    if path.suffix.lower() != ".textgrid":
        continue

    tg = textgrid.openTextgrid(
        str(path),
        includeEmptyIntervals=True
    )

    for tier_name in tg.tierNames:
        if tier_name.lower() != "phones":
            continue

        for entry in tg.getTier(tier_name).entries:
            phone = str(entry.label).strip()
            if phone:
                phone_counts[phone] += 1

# Explicit special tokens
PAD = "<pad>"
SPN = "spn"
UNK = "<unk>"

# Preserve every observed MFA phone as an independent token.
# Sort for deterministic IDs.
phones = sorted(phone_counts.keys())

vocab = [PAD, UNK] + phones

phone_to_id = {phone: idx for idx, phone in enumerate(vocab)}
id_to_phone = {str(idx): phone for phone, idx in phone_to_id.items()}

vocab_data = {
    "pad_token": PAD,
    "unk_token": UNK,
    "vocabulary_size": len(vocab),
    "phone_to_id": phone_to_id,
    "id_to_phone": id_to_phone,
    "phone_counts": dict(phone_counts)
}

vocab_path = CONFIG_DIR / "english_ipa_vocabulary.json"
vocab_path.write_text(
    json.dumps(vocab_data, indent=2, ensure_ascii=False)
)

# Copy LJSpeech configs as starting templates
source_config = REPO / "config/LJSpeech"

for filename in ["preprocess.yaml", "model.yaml", "train.yaml"]:
    source = source_config / filename
    destination = CONFIG_DIR / filename

    if not destination.exists():
        destination.write_text(source.read_text())

# Adapt the copied English preprocessing config
preprocess_path = CONFIG_DIR / "preprocess.yaml"
preprocess = yaml.safe_load(preprocess_path.read_text())

preprocess["dataset"] = "Svarah_MFA_IPA"
preprocess["path"]["corpus_path"] = str(BASE)
preprocess["path"]["raw_path"] = str(BASE / "raw_audio")
preprocess["path"]["preprocessed_path"] = str(BASE / "processed")
preprocess["preprocessing"]["text"]["language"] = "en_ipa"
preprocess["preprocessing"]["text"]["text_cleaners"] = []

preprocess["preprocessing"]["pitch"]["feature"] = "phoneme_level"
preprocess["preprocessing"]["energy"]["feature"] = "phoneme_level"

preprocess_path.write_text(
    yaml.safe_dump(preprocess, sort_keys=False)
)

# Adapt the model config
model_path = CONFIG_DIR / "model.yaml"
model_config = yaml.safe_load(model_path.read_text())

# Use a smaller architecture suitable for CPU experimentation
model_config["transformer"]["encoder_layer"] = 2
model_config["transformer"]["decoder_layer"] = 2
model_config["transformer"]["encoder_head"] = 2
model_config["transformer"]["decoder_head"] = 2
model_config["transformer"]["encoder_hidden"] = 128
model_config["transformer"]["decoder_hidden"] = 128
model_config["transformer"]["conv_filter_size"] = 512
model_config["transformer"]["encoder_dropout"] = 0.1
model_config["transformer"]["decoder_dropout"] = 0.1

model_config["variance_predictor"]["filter_size"] = 128
model_config["variance_predictor"]["dropout"] = 0.2
model_config["multi_speaker"] = False
model_config["max_seq_len"] = 2048

model_config["input_vocabulary_size"] = len(vocab)

model_path.write_text(
    yaml.safe_dump(model_config, sort_keys=False)
)

print("Vocabulary size:", len(vocab))
print("Unique observed phones:", len(phones))
print("Special tokens:", [PAD, UNK])
print("Most common phones:", phone_counts.most_common(15))

print("\nCreated:")
print(vocab_path)
print(preprocess_path)
print(model_path)

print("\nSTEP 125 COMPLETE")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FastSpeech2_Assignment/English/alignment/full_alignment/train'

In [6]:
from pathlib import Path
from google.colab import drive

# Mount Drive at the location used in our latest steps
drive.mount("/content/gdrive")

# Search both possible Drive mount locations
roots = [
    Path("/content/gdrive/MyDrive/FastSpeech2_Assignment"),
    Path("/content/drive/MyDrive/FastSpeech2_Assignment")
]

for root in roots:
    print("\nChecking:", root)
    print("Exists:", root.exists())

    if root.exists():
        english = root / "English"
        config = english / "config"

        print("English folder:", english.exists())
        print("Config folder:", config.exists())

        if config.exists():
            print("Config files:")
            for f in sorted(config.iterdir()):
                print(" -", f.name)

# Search for the vocabulary file in MyDrive
for drive_root in [
    Path("/content/gdrive/MyDrive"),
    Path("/content/drive/MyDrive")
]:
    if drive_root.exists():
        matches = list(drive_root.rglob("english_ipa_vocabulary.json"))
        print(f"\nVocabulary matches under {drive_root}:")
        for match in matches:
            print(match)

Mounted at /content/gdrive

Checking: /content/gdrive/MyDrive/FastSpeech2_Assignment
Exists: True
English folder: True
Config folder: True
Config files:
 - english_ipa_vocabulary.json
 - model.yaml
 - preprocess.yaml
 - train.yaml

Checking: /content/drive/MyDrive/FastSpeech2_Assignment
Exists: True
English folder: True
Config folder: True
Config files:

Vocabulary matches under /content/gdrive/MyDrive:
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/english_ipa_vocabulary.json

Vocabulary matches under /content/drive/MyDrive:


In [9]:
from pathlib import Path
import subprocess

REPO = Path("/content/FastSpeech2")

# Clone only if the repository is missing
if not (REPO / "transformer/Models.py").exists():

    print("Restoring FastSpeech2 repository...")

    result = subprocess.run(
        [
            "git", "clone",
            "https://github.com/ming024/FastSpeech2.git",
            "/content/FastSpeech2"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Repository restoration failed.")

else:
    print("Repository already available.")

# Verify
MODEL_FILE = REPO / "transformer/Models.py"

assert MODEL_FILE.exists(), "Models.py is still missing!"

print("\nRepository restored successfully.")
print("Models.py:", MODEL_FILE)
print("File size:", MODEL_FILE.stat().st_size, "bytes")

Restoring FastSpeech2 repository...

Repository restored successfully.
Models.py: /content/FastSpeech2/transformer/Models.py
File size: 5820 bytes


In [10]:
from pathlib import Path
import shutil
import json
import yaml

# Exact verified paths
CONFIG_DIR = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config"
)
REPO = Path("/content/FastSpeech2")
MODEL_FILE = REPO / "transformer/Models.py"
BACKUP_FILE = REPO / "transformer/Models.py.step126_backup"

VOCAB_FILE = CONFIG_DIR / "english_ipa_vocabulary.json"
MODEL_CONFIG_FILE = CONFIG_DIR / "model.yaml"

# Verify files
for p in [VOCAB_FILE, MODEL_CONFIG_FILE, MODEL_FILE]:
    print(p, "EXISTS:", p.exists())

assert VOCAB_FILE.is_file()
assert MODEL_CONFIG_FILE.is_file()
assert MODEL_FILE.is_file()

with open(VOCAB_FILE, encoding="utf-8") as f:
    vocab = json.load(f)

with open(MODEL_CONFIG_FILE, encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert vocab["vocab_size"] == config["input_vocabulary_size"]

# Preserve original source
if not BACKUP_FILE.exists():
    shutil.copy2(MODEL_FILE, BACKUP_FILE)

source = MODEL_FILE.read_text(encoding="utf-8")

old = "n_src_vocab = len(symbols) + 1"

new = """n_src_vocab = model_config.get(
            "input_vocabulary_size",
            len(symbols) + 1
        )"""

if new in source:
    print("\nPatch already applied.")
elif old in source:
    source = source.replace(old, new, 1)
    MODEL_FILE.write_text(source, encoding="utf-8")
    print("\nPatch applied successfully.")
else:
    raise RuntimeError("Could not locate the expected encoder line.")

print("\nVocabulary size:", vocab["vocab_size"])
print("Configured vocabulary size:", config["input_vocabulary_size"])
print("Original backup:", BACKUP_FILE.exists())
print("STEP 126 COMPLETE")

/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/english_ipa_vocabulary.json EXISTS: True
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/model.yaml EXISTS: True
/content/FastSpeech2/transformer/Models.py EXISTS: True

Patch applied successfully.

Vocabulary size: 67
Configured vocabulary size: 67
Original backup: True
STEP 126 COMPLETE


In [12]:
%pip install -q Unidecode

from unidecode import unidecode

print("Unidecode installed successfully.")

Unidecode installed successfully.


In [13]:
import sys
sys.path.insert(0, "/content/FastSpeech2")

import torch
from transformer.Models import Encoder

print("PyTorch:", torch.__version__)
print("FastSpeech2 Encoder import: SUCCESS")

PyTorch: 2.11.0+cu130
FastSpeech2 Encoder import: SUCCESS


In [15]:
from pathlib import Path

MODEL_FILE = Path("/content/FastSpeech2/transformer/Models.py")

source = MODEL_FILE.read_text(encoding="utf-8")

old = """n_src_vocab = model_config.get(
            "input_vocabulary_size",
            len(symbols) + 1
        )"""

new = """n_src_vocab = config.get(
            "input_vocabulary_size",
            len(symbols) + 1
        )"""

if old in source:
    source = source.replace(old, new, 1)
    MODEL_FILE.write_text(source, encoding="utf-8")
    print("Encoder variable corrected successfully.")

elif new in source:
    print("Correction already applied.")

else:
    raise RuntimeError("Expected patched code was not found.")

print("\nCorrected encoder code:")
for i, line in enumerate(source.splitlines()):
    if "n_src_vocab" in line:
        print(f"{i+1}: {line.strip()}")

Encoder variable corrected successfully.

Corrected encoder code:
40: n_src_vocab = config.get(
60: n_src_vocab, d_word_vec, padding_idx=Constants.PAD


In [17]:
import sys
import importlib
from pathlib import Path

sys.path.insert(0, "/content/FastSpeech2")

# Reload the actual module from the corrected source file
import transformer.Models as models_module

importlib.reload(models_module)

# Get the freshly loaded Encoder class
Encoder = models_module.Encoder

# Verify the source contains config.get, not model_config.get
source = Path(
    "/content/FastSpeech2/transformer/Models.py"
).read_text()

start = source.find("n_src_vocab =")

print("Encoder source:")
print(source[start:start + 150])

assert "n_src_vocab = config.get(" in source
assert "n_src_vocab = model_config.get(" not in source

print("\nEncoder module reloaded successfully.")

Encoder source:
n_src_vocab = config.get(
            "input_vocabulary_size",
            len(symbols) + 1
        )
        d_word_vec = config["transformer"]["enco

Encoder module reloaded successfully.


In [18]:
import json
import yaml
import torch

from pathlib import Path

CONFIG_DIR = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config"
)

with open(CONFIG_DIR / "model.yaml") as f:
    model_config = yaml.safe_load(f)

with open(CONFIG_DIR / "english_ipa_vocabulary.json") as f:
    vocab = json.load(f)

encoder = Encoder(model_config)

embedding = encoder.src_word_emb

print("Vocabulary size:", vocab["vocab_size"])
print("Embedding shape:", tuple(embedding.weight.shape))
print("Embedding entries:", embedding.num_embeddings)
print("Padding index:", embedding.padding_idx)

assert embedding.num_embeddings == 67
assert embedding.padding_idx == 0

print("\nSTEP 127 PASSED")

Vocabulary size: 67
Embedding shape: (67, 128)
Embedding entries: 67
Padding index: 0

STEP 127 PASSED


In [19]:
import sys
import json
import yaml
import torch
from pathlib import Path

sys.path.insert(0, "/content/FastSpeech2")

from transformer.Models import Encoder

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
CONFIG_DIR = BASE / "English/config"

with open(CONFIG_DIR / "model.yaml") as f:
    model_config = yaml.safe_load(f)

with open(CONFIG_DIR / "english_ipa_vocabulary.json") as f:
    vocab = json.load(f)

print("Configured vocabulary size:", model_config["input_vocabulary_size"])
print("Actual vocabulary size:", vocab["vocab_size"])

# Construct encoder using the English model configuration
encoder = Encoder(model_config)

embedding = encoder.src_word_emb

print("\nEmbedding shape:", tuple(embedding.weight.shape))
print("Expected vocabulary entries:", vocab["vocab_size"])
print("Actual embedding entries:", embedding.num_embeddings)

assert embedding.num_embeddings == vocab["vocab_size"]
assert embedding.padding_idx == 0

print("\nSTEP 127 PASSED")

Configured vocabulary size: 67
Actual vocabulary size: 67

Embedding shape: (67, 128)
Expected vocabulary entries: 67
Actual embedding entries: 67

STEP 127 PASSED


In [20]:
%pip install -q praatio soundfile

from pathlib import Path
from praatio import textgrid
import soundfile as sf
import numpy as np

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"

GRID_DIR = ENGLISH / "alignment/full_alignment/train"
AUDIO_DIR = ENGLISH / "raw_audio/train"

# Select one aligned training example
grid_files = sorted(GRID_DIR.glob("*.TextGrid"))
assert grid_files, "No training TextGrid files found."

grid_path = grid_files[0]
audio_path = AUDIO_DIR / (grid_path.stem + ".wav")

assert audio_path.exists(), f"Matching audio missing: {audio_path}"

# Read audio
audio, sr = sf.read(audio_path)
if audio.ndim == 2:
    audio = audio.mean(axis=1)

audio_duration = len(audio) / sr

# Read TextGrid
tg = textgrid.openTextgrid(
    str(grid_path),
    includeEmptyIntervals=True
)

print("========== AUDIO ==========")
print("File:", audio_path.name)
print("Sample rate:", sr)
print("Samples:", len(audio))
print("Duration:", round(audio_duration, 4), "seconds")

print("\n========== TEXTGRID ==========")
print("File:", grid_path.name)
print("Tiers:", tg.tierNames)

phone_tier_name = next(
    (name for name in tg.tierNames if name.lower() == "phones"),
    None
)

assert phone_tier_name is not None, "Phones tier not found."

phone_tier = tg.getTier(phone_tier_name)
entries = phone_tier.entries

nonempty = [e for e in entries if str(e[2]).strip()]
empty = [e for e in entries if not str(e[2]).strip()]

print("\nPhone intervals:", len(entries))
print("Labelled intervals:", len(nonempty))
print("Blank intervals:", len(empty))

print("First 15 intervals:")
for start, end, label in entries[:15]:
    print(
        f"{start:.4f} - {end:.4f} | "
        f"{repr(label)} | duration={end-start:.4f}"
    )

print("\n========== COVERAGE ==========")
print("TextGrid start:", phone_tier.minTimestamp)
print("TextGrid end:", phone_tier.maxTimestamp)
print("Audio duration:", audio_duration)
print(
    "Endpoint difference:",
    round(abs(phone_tier.maxTimestamp - audio_duration), 5),
    "seconds"
)

print("\nSTEP 128A DIAGNOSTIC COMPLETE")

========== AUDIO ==========
File: recording_0.wav
Sample rate: 16000
Samples: 140152
Duration: 8.7595 seconds

========== TEXTGRID ==========
File: recording_0.TextGrid
Tiers: ('words', 'phones')

Phone intervals: 82
Labelled intervals: 78
Blank intervals: 4
First 15 intervals:
0.0000 - 0.1700 | 'aː' | duration=0.1700
0.1700 - 0.3200 | 'ɖ' | duration=0.1500
0.3200 - 0.3800 | 'ɪ' | duration=0.0600
0.3800 - 0.4900 | 's' | duration=0.1100
0.4900 - 0.5400 | 'ʈʲ' | duration=0.0500
0.5400 - 0.5700 | 'ɹ' | duration=0.0300
0.5700 - 0.6100 | 'ɪ' | duration=0.0400
0.6100 - 0.7000 | 'k' | duration=0.0900
0.7000 - 0.7400 | 'ʈ' | duration=0.0400
0.7400 - 0.7900 | 'ç' | duration=0.0500
0.7900 - 0.9300 | 'eː' | duration=0.1400
0.9300 - 1.0500 | 's' | duration=0.1200
1.0500 - 1.0900 | 'ə' | duration=0.0400
1.0900 - 1.1700 | 'n' | duration=0.0800
1.1700 - 1.3500 | 'ə' | duration=0.1800

========== COVERAGE ==========
TextGrid start: 0.0
TextGrid end: 8.7595
Audio duration: 8.7595
Endpoint difference: 0

In [21]:
from pathlib import Path
from praatio import textgrid
import json
import yaml
import numpy as np

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"

GRID_DIR = ENGLISH / "alignment/full_alignment/train"
AUDIO_DIR = ENGLISH / "raw_audio/train"
CONFIG_DIR = ENGLISH / "config"

# Load vocabulary
with open(CONFIG_DIR / "english_ipa_vocabulary.json", encoding="utf-8") as f:
    vocab_data = json.load(f)

phone_to_id = vocab_data["phone_to_id"]

# Load preprocessing configuration
with open(CONFIG_DIR / "preprocess.yaml") as f:
    preprocess = yaml.safe_load(f)

print("========== ACOUSTIC CONFIGURATION ==========")
print(yaml.safe_dump(preprocess.get("preprocessing", {}), sort_keys=False))

# Inspect one sample
grid_path = sorted(GRID_DIR.glob("*.TextGrid"))[0]
tg = textgrid.openTextgrid(str(grid_path), includeEmptyIntervals=True)
tier = tg.getTier("phones")

entries = tier.entries

mapped_ids = []
unknown_phones = []
blank_count = 0

for start, end, label in entries:
    label = str(label).strip()

    if not label:
        blank_count += 1
        mapped_ids.append(phone_to_id["<sil>"])
    elif label in phone_to_id:
        mapped_ids.append(phone_to_id[label])
    else:
        unknown_phones.append(label)

# Check interval continuity
gaps = []
for i in range(len(entries) - 1):
    difference = entries[i + 1][0] - entries[i][1]
    if abs(difference) > 1e-6:
        gaps.append((i, difference))

print("\n========== VOCABULARY CHECK ==========")
print("Vocabulary size:", len(phone_to_id))
print("Phone intervals:", len(entries))
print("Mapped intervals:", len(mapped_ids))
print("Blank intervals mapped to <sil>:", blank_count)
print("Unknown phones:", sorted(set(unknown_phones)))

print("\n========== TIMING CHECK ==========")
print("Uncovered interval boundaries:", len(gaps))
print("First 10 mapped IDs:", mapped_ids[:10])
print("Duration sum:", round(sum(e[1] - e[0] for e in entries), 5))

assert not unknown_phones, "Some MFA phones are missing from the vocabulary."
assert not gaps, "TextGrid has uncovered boundaries."
assert len(mapped_ids) == len(entries)

print("\nSTEP 128B DIAGNOSTIC COMPLETE")

========== ACOUSTIC CONFIGURATION ==========
val_size: 512
text:
  language: en_ipa
  text_cleaners: []
  use_phonemes: true
audio:
  sampling_rate: 22050
  max_wav_value: 32768.0
stft:
  filter_length: 1024
  hop_length: 256
  win_length: 1024
mel:
  n_mel_channels: 80
  mel_fmin: 0
  mel_fmax: 8000
pitch:
  feature: phoneme_level
  normalization: true
energy:
  feature: phoneme_level
  normalization: true


========== VOCABULARY CHECK ==========
Vocabulary size: 67
Phone intervals: 82
Mapped intervals: 82
Blank intervals mapped to <sil>: 4
Unknown phones: []

========== TIMING CHECK ==========
Uncovered interval boundaries: 0
First 10 mapped IDs: [6, 41, 51, 29, 58, 54, 51, 20, 57, 34]
Duration sum: 8.7595

STEP 128B DIAGNOSTIC COMPLETE


In [22]:
from pathlib import Path

REPO = Path("/content/FastSpeech2")

# Find relevant preprocessing source files
keywords = [
    "audio",
    "preprocessor",
    "preprocess",
    "pitch",
    "energy",
    "dataset"
]

print("========== RELEVANT SOURCE FILES ==========")

for p in sorted(REPO.rglob("*.py")):
    if any(k in str(p.relative_to(REPO)).lower() for k in keywords):
        print(p.relative_to(REPO))

# Inspect the repository's audio processing implementation
files_to_inspect = [
    REPO / "audio/tools.py",
    REPO / "preprocessor/preprocessor.py",
    REPO / "preprocessor/compute_f0.py",
    REPO / "dataset.py"
]

for path in files_to_inspect:
    print("\n" + "=" * 65)
    print("FILE:", path.relative_to(REPO) if path.exists() else path)
    print("=" * 65)

    if path.exists():
        content = path.read_text(encoding="utf-8")
        print(content[:14000])
    else:
        print("File not found at this location.")

========== RELEVANT SOURCE FILES ==========
audio/__init__.py
audio/audio_processing.py
audio/stft.py
audio/tools.py
dataset.py
preprocess.py
preprocessor/aishell3.py
preprocessor/libritts.py
preprocessor/ljspeech.py
preprocessor/preprocessor.py

FILE: audio/tools.py
import torch
import numpy as np
from scipy.io.wavfile import write

from audio.audio_processing import griffin_lim


def get_mel_from_wav(audio, _stft):
    audio = torch.clip(torch.FloatTensor(audio).unsqueeze(0), -1, 1)
    audio = torch.autograd.Variable(audio, requires_grad=False)
    melspec, energy = _stft.mel_spectrogram(audio)
    melspec = torch.squeeze(melspec, 0).numpy().astype(np.float32)
    energy = torch.squeeze(energy, 0).numpy().astype(np.float32)

    return melspec, energy


def inv_mel_spec(mel, out_filename, _stft, griffin_iters=60):
    mel = torch.stack([mel])
    mel_decompress = _stft.spectral_de_normalize(mel)
    mel_decompress = mel_decompress.transpose(1, 2).data.cpu()
    spec_from_mel_scaling

In [23]:
# ============================================================
# FASTSPEECH2 ENGLISH: STEP 128C–129
# CONSOLIDATED PREPROCESSING SETUP
# ============================================================

from pathlib import Path
import sys
import os
import json
import yaml
import importlib.util
import subprocess
import shutil

# -------------------- PATHS --------------------

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"
REPO = Path("/content/FastSpeech2")
CONFIG = ENGLISH / "config"

TRAIN_AUDIO = ENGLISH / "raw_audio/train"
VAL_AUDIO = ENGLISH / "raw_audio/validation"

TRAIN_GRID = ENGLISH / "alignment/full_alignment/train"
VAL_GRID = ENGLISH / "alignment/full_alignment/validation"

OUTPUT = ENGLISH / "processed"
OUTPUT_TRAIN = OUTPUT / "train"
OUTPUT_VAL = OUTPUT / "validation"
OUTPUT_STATS = OUTPUT / "stats"

for folder in [OUTPUT_TRAIN, OUTPUT_VAL, OUTPUT_STATS]:
    folder.mkdir(parents=True, exist_ok=True)

# -------------------- VERIFY PROJECT FILES --------------------

required = [
    REPO / "transformer/Models.py",
    REPO / "audio/tools.py",
    CONFIG / "english_ipa_vocabulary.json",
    CONFIG / "preprocess.yaml",
    CONFIG / "model.yaml",
    TRAIN_AUDIO,
    VAL_AUDIO,
    TRAIN_GRID,
    VAL_GRID
]

for p in required:
    assert p.exists(), f"Missing required file/folder: {p}"

# -------------------- LOAD CONFIGURATIONS --------------------

with open(CONFIG / "english_ipa_vocabulary.json", encoding="utf-8") as f:
    vocab = json.load(f)

with open(CONFIG / "preprocess.yaml") as f:
    preprocess = yaml.safe_load(f)

with open(CONFIG / "model.yaml") as f:
    model = yaml.safe_load(f)

phone_to_id = vocab["phone_to_id"]

# -------------------- CHECK DEPENDENCIES --------------------

packages = {
    "numpy": "numpy",
    "scipy": "scipy",
    "soundfile": "soundfile",
    "librosa": "librosa",
    "pyworld": "pyworld",
    "praatio": "praatio",
    "torch": "torch",
    "yaml": "PyYAML"
}

missing = []

for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        missing.append(package)

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])

# -------------------- IMPORT AND VERSION CHECK --------------------

import numpy as np
import scipy
import soundfile as sf
import librosa
import pyworld
import torch
from praatio import textgrid

print("\n========== ENVIRONMENT ==========")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("NumPy:", np.__version__)
print("Librosa:", librosa.__version__)
print("PyWorld:", pyworld.__version__)

# -------------------- AUDIO CONFIGURATION --------------------

audio_cfg = preprocess["preprocessing"]["audio"]
stft_cfg = preprocess["preprocessing"]["stft"]
mel_cfg = preprocess["preprocessing"]["mel"]

SR = int(audio_cfg["sampling_rate"])
N_FFT = int(stft_cfg["filter_length"])
HOP = int(stft_cfg["hop_length"])
WIN = int(stft_cfg["win_length"])
N_MELS = int(mel_cfg["n_mel_channels"])
FMIN = float(mel_cfg["mel_fmin"])
FMAX = float(mel_cfg["mel_fmax"])

print("\n========== AUDIO SETTINGS ==========")
print("Sample rate:", SR)
print("FFT:", N_FFT)
print("Hop:", HOP)
print("Window:", WIN)
print("Mel channels:", N_MELS)
print("Frequency range:", FMIN, "-", FMAX)

print("\n========== DATASET COUNTS ==========")
print("Training WAV:", len(list(TRAIN_AUDIO.glob("*.wav"))))
print("Training TextGrid:", len(list(TRAIN_GRID.glob("*.TextGrid"))))
print("Validation WAV:", len(list(VAL_AUDIO.glob("*.wav"))))
print("Validation TextGrid:", len(list(VAL_GRID.glob("*.TextGrid"))))

print("\n========== VOCABULARY ==========")
print("Vocabulary size:", vocab["vocab_size"])
print("Silence ID:", phone_to_id["<sil>"])
print("Padding ID:", phone_to_id["<pad>"])

# -------------------- INSPECT REPOSITORY FEATURE FUNCTIONS --------------------

source_files = [
    REPO / "audio/tools.py",
    REPO / "preprocessor/preprocessor.py",
    REPO / "preprocessor/compute_f0.py",
    REPO / "dataset.py"
]

print("\n========== FASTSPEECH2 FEATURE IMPLEMENTATION ==========")

search_terms = [
    "get_mel_from_wav",
    "get_pitch",
    "get_energy",
    "spectrogram",
    "mel_spectrogram",
    "pyworld",
    "pitch_feature",
    "energy_feature",
    "duration",
    "np.save",
    "np.load"
]

for file in source_files:
    if not file.exists():
        continue

    print("\n" + "=" * 55)
    print("FILE:", file.relative_to(REPO))
    print("=" * 55)

    lines = file.read_text(encoding="utf-8").splitlines()

    selected = set()

    for i, line in enumerate(lines):
        if any(term in line for term in search_terms):
            for j in range(max(0, i-3), min(len(lines), i+9)):
                selected.add(j)

    for i in sorted(selected):
        print(f"{i+1:4}: {lines[i]}")

# -------------------- PREPARE PIPELINE MANIFEST --------------------

manifest = {
    "language": "English",
    "dataset": "Svarah",
    "vocabulary_size": vocab["vocab_size"],
    "sample_rate": SR,
    "n_fft": N_FFT,
    "hop_length": HOP,
    "n_mels": N_MELS,
    "pitch_level": "phoneme",
    "energy_level": "phoneme",
    "silence_token": "<sil>",
    "silence_token_id": phone_to_id["<sil>"],
    "training_audio_count": len(list(TRAIN_AUDIO.glob("*.wav"))),
    "validation_audio_count": len(list(VAL_AUDIO.glob("*.wav"))),
    "training_alignment_count": len(list(TRAIN_GRID.glob("*.TextGrid"))),
    "validation_alignment_count": len(list(VAL_GRID.glob("*.TextGrid")))
}

with open(OUTPUT / "preprocessing_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n========== SETUP COMPLETE ==========")
print("Output directory:", OUTPUT)
print("Manifest saved:", OUTPUT / "preprocessing_manifest.json")
print("English preprocessing environment is ready.")
print("\nDO NOT START TRAINING YET.")

Installing missing packages: ['pyworld']


/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



========== ENVIRONMENT ==========
Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
NumPy: 2.4.6
Librosa: 0.11.0
PyWorld: 0.3.5

========== AUDIO SETTINGS ==========
Sample rate: 22050
FFT: 1024
Hop: 256
Window: 1024
Mel channels: 80
Frequency range: 0.0 - 8000.0

========== DATASET COUNTS ==========
Training WAV: 2664
Training TextGrid: 2662
Validation WAV: 505
Validation TextGrid: 501

========== VOCABULARY ==========
Vocabulary size: 67
Silence ID: 2
Padding ID: 0

========== FASTSPEECH2 FEATURE IMPLEMENTATION ==========

FILE: audio/tools.py
   5: from audio.audio_processing import griffin_lim
   6: 
   7: 
   8: def get_mel_from_wav(audio, _stft):
   9:     audio = torch.clip(torch.FloatTensor(audio).unsqueeze(0), -1, 1)
  10:     audio = torch.autograd.Variable(audio, requires_grad=False)
  11:     melspec, energy = _stft.mel_spectrogram(audio)
  12:     melspec = torch.squeeze(melspec, 0).numpy().astype(np.float32)
  13:     energy = torch.squeeze(energy, 0).numpy().as

In [30]:
from pathlib import Path
import sys
import importlib

repo = Path("/content/FastSpeech2")
stft_file = repo / "audio" / "stft.py"

source = stft_file.read_text()

# Add librosa import if missing
if "import librosa" not in source:
    source = "import librosa\n" + source
    stft_file.write_text(source)
    print("Added librosa import.")
else:
    print("librosa import already exists.")

# Reload the module
sys.path.insert(0, str(repo))
import audio.stft as stft_module
importlib.reload(stft_module)

# Initialize STFT
stft = stft_module.TacotronSTFT(
    filter_length=1024,
    hop_length=256,
    win_length=1024,
    n_mel_channels=80,
    sampling_rate=22050,
    mel_fmin=0,
    mel_fmax=8000
)

print("\nSUCCESS: TacotronSTFT initialized.")
print("Mel basis shape:", tuple(stft.mel_basis.shape))

Added librosa import.

SUCCESS: TacotronSTFT initialized.
Mel basis shape: (80, 513)


In [33]:

# ================================================================
# FASTSPEECH2 ENGLISH: STEP 129-130
# COMPLETE, RESUMABLE ACOUSTIC FEATURE EXTRACTION
# ================================================================
# Extracts:
#   1. IPA phone IDs from MFA TextGrids
#   2. Phone-level durations
#   3. Phone-level pitch (F0)
#   4. Phone-level energy
#   5. FastSpeech2-compatible mel spectrograms
#
# Uses the already-patched audio/stft.py.
# Reuses valid existing .npz files.
# ================================================================

import os
import sys
import csv
import json
import time
import warnings
import importlib
from pathlib import Path

import numpy as np
import librosa
import pyworld as pw
import torch
import yaml

from praatio import textgrid
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ================================================================
# 1. PATHS AND CONFIGURATION
# ================================================================

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"
REPO = Path("/content/FastSpeech2")

CONFIG_DIR = ENGLISH / "config"

TRAIN_AUDIO = ENGLISH / "raw_audio/train"
VAL_AUDIO = ENGLISH / "raw_audio/validation"

TRAIN_GRID = ENGLISH / "alignment/full_alignment/train"
VAL_GRID = ENGLISH / "alignment/full_alignment/validation"

OUT = ENGLISH / "processed"
OUT_TRAIN = OUT / "train"
OUT_VAL = OUT / "validation"
OUT_STATS = OUT / "stats"

for directory in [OUT_TRAIN, OUT_VAL, OUT_STATS]:
    directory.mkdir(parents=True, exist_ok=True)

assert BASE.exists(), f"Assignment directory not found: {BASE}"
assert REPO.exists(), f"Repository not found: {REPO}"

assert (CONFIG_DIR / "english_ipa_vocabulary.json").exists()
assert (CONFIG_DIR / "preprocess.yaml").exists()

# ================================================================
# 2. LOAD VOCABULARY AND PREPROCESSING CONFIG
# ================================================================

with open(
    CONFIG_DIR / "english_ipa_vocabulary.json",
    encoding="utf-8"
) as f:
    vocab_data = json.load(f)

with open(
    CONFIG_DIR / "preprocess.yaml",
    encoding="utf-8"
) as f:
    preprocess = yaml.safe_load(f)

phone_to_id = vocab_data["phone_to_id"]
VOCAB_SIZE = int(vocab_data["vocab_size"])
SIL_ID = phone_to_id["<sil>"]

pc = preprocess["preprocessing"]

SR = int(pc["audio"]["sampling_rate"])
N_FFT = int(pc["stft"]["filter_length"])
HOP = int(pc["stft"]["hop_length"])
WIN = int(pc["stft"]["win_length"])

N_MELS = int(pc["mel"]["n_mel_channels"])
FMIN = float(pc["mel"]["mel_fmin"])
FMAX = float(pc["mel"]["mel_fmax"])

# ================================================================
# 3. FASTSPEECH2 IMPORTS
# ================================================================

sys.path.insert(0, str(REPO))

# Use the already-patched repository implementation.
import audio.stft as stft_module
import audio.tools as audio_tools

importlib.reload(stft_module)
importlib.reload(audio_tools)

TacotronSTFT = stft_module.TacotronSTFT
get_mel_from_wav = audio_tools.get_mel_from_wav

# Initialize STFT using the configuration values.
stft = TacotronSTFT(
    filter_length=N_FFT,
    hop_length=HOP,
    win_length=WIN,
    n_mel_channels=N_MELS,
    sampling_rate=SR,
    mel_fmin=FMIN,
    mel_fmax=FMAX
)

# ================================================================
# 4. DEVICE AND ENVIRONMENT
# ================================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 70)
print("FASTSPEECH2 ENGLISH FEATURE EXTRACTION")
print("=" * 70)

print("Device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("PyTorch:", torch.__version__)
print("Librosa:", librosa.__version__)
print("Sample rate:", SR)
print("FFT:", N_FFT)
print("Hop length:", HOP)
print("Mel channels:", N_MELS)
print("Vocabulary size:", VOCAB_SIZE)

print("\nSTFT initialized successfully.")

# ================================================================
# 5. VALIDATE INPUT DIRECTORIES
# ================================================================

for directory in [
    TRAIN_AUDIO,
    VAL_AUDIO,
    TRAIN_GRID,
    VAL_GRID
]:
    assert directory.exists(), f"Missing directory: {directory}"

print("\nInput file counts:")
print("Training WAV:", len(list(TRAIN_AUDIO.glob("*.wav"))))
print("Training TextGrid:", len(list(TRAIN_GRID.glob("*.TextGrid"))))
print("Validation WAV:", len(list(VAL_AUDIO.glob("*.wav"))))
print("Validation TextGrid:", len(list(VAL_GRID.glob("*.TextGrid"))))

# ================================================================
# 6. TRANSCRIPT EXTRACTION
# ================================================================

def read_transcript(wav_path, tg):

    lab_path = wav_path.with_suffix(".lab")

    if lab_path.exists():
        return lab_path.read_text(
            encoding="utf-8"
        ).strip()

    try:
        words = tg.getTier("words")

        return " ".join(
            str(entry[2]).strip()
            for entry in words.entries
            if str(entry[2]).strip()
        )

    except Exception:
        return ""


# ================================================================
# 7. MFA PHONE AND DURATION EXTRACTION
# ================================================================

def extract_phone_data(tg, mel_frames):

    tier = tg.getTier("phones")
    entries = tier.entries

    if not entries:
        raise ValueError("Empty phones tier")

    ids = []
    labels = []
    boundaries = [0]

    for start, end, label in entries:

        label = str(label).strip()

        # Blank MFA intervals are mapped to silence.
        if not label:
            label = "<sil>"

        if label not in phone_to_id:
            raise ValueError(
                f"Unknown MFA phone: {label}"
            )

        ids.append(phone_to_id[label])
        labels.append(label)

        frame_end = int(round(end * SR / HOP))
        boundaries.append(frame_end)

    boundaries = np.asarray(
        boundaries,
        dtype=np.int64
    )

    # Ensure boundaries cover the complete mel sequence.
    boundaries = np.clip(
        boundaries,
        0,
        mel_frames
    )

    boundaries = np.maximum.accumulate(boundaries)

    boundaries[0] = 0
    boundaries[-1] = mel_frames

    durations = np.diff(boundaries)

    assert len(ids) == len(durations)
    assert np.all(durations >= 0)
    assert int(durations.sum()) == mel_frames

    return (
        np.asarray(ids, dtype=np.int64),
        durations.astype(np.int64),
        labels,
        boundaries
    )


# ================================================================
# 8. ACOUSTIC FEATURE EXTRACTION
# ================================================================

def extract_acoustic_features(wav_path, tg_path):

    # ------------------------------------------------------------
    # A. Load and resample audio
    # ------------------------------------------------------------

    wav, _ = librosa.load(
        str(wav_path),
        sr=SR,
        mono=True
    )

    wav = np.asarray(
        wav,
        dtype=np.float32
    )

    if len(wav) < N_FFT:
        raise ValueError("Audio is too short")

    # ------------------------------------------------------------
    # B. FastSpeech2 mel and frame energy
    # ------------------------------------------------------------

    mel, frame_energy = get_mel_from_wav(
        wav,
        stft
    )

    mel = np.asarray(
        mel,
        dtype=np.float32
    )

    frame_energy = np.asarray(
        frame_energy,
        dtype=np.float32
    ).reshape(-1)

    if mel.ndim != 2:
        raise ValueError(
            f"Unexpected mel shape: {mel.shape}"
        )

    if mel.shape[0] != N_MELS:
        raise ValueError(
            f"Unexpected mel dimension: {mel.shape}"
        )

    mel_frames = mel.shape[1]

    # ------------------------------------------------------------
    # C. Match energy frame count
    # ------------------------------------------------------------

    if len(frame_energy) > mel_frames:

        frame_energy = frame_energy[:mel_frames]

    elif len(frame_energy) < mel_frames:

        if len(frame_energy) == 0:
            frame_energy = np.zeros(
                mel_frames,
                dtype=np.float32
            )
        else:
            frame_energy = np.pad(
                frame_energy,
                (0, mel_frames - len(frame_energy)),
                mode="edge"
            )

    # ------------------------------------------------------------
    # D. Read MFA TextGrid
    # ------------------------------------------------------------

    tg = textgrid.openTextgrid(
        str(tg_path),
        includeEmptyIntervals=True
    )

    text_ids, durations, labels, boundaries = (
        extract_phone_data(
            tg,
            mel_frames
        )
    )

    # ------------------------------------------------------------
    # E. WORLD F0 extraction
    # ------------------------------------------------------------

    wav64 = wav.astype(np.float64)

    frame_period = HOP * 1000.0 / SR

    f0, f0_times = pw.dio(
        wav64,
        SR,
        frame_period=frame_period
    )

    f0 = pw.stonemask(
        wav64,
        f0,
        f0_times,
        SR
    )

    f0 = np.nan_to_num(
        f0,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    # ------------------------------------------------------------
    # F. Interpolate voiced F0 onto mel-frame timeline
    # ------------------------------------------------------------

    valid = np.where(f0 > 0)[0]

    target_times = (
        np.arange(mel_frames) * HOP / SR
    )

    if len(valid) >= 2:

        f0_frames = np.interp(
            target_times,
            f0_times[valid],
            f0[valid]
        ).astype(np.float32)

    elif len(valid) == 1:

        f0_frames = np.full(
            mel_frames,
            f0[valid[0]],
            dtype=np.float32
        )

    else:

        f0_frames = np.zeros(
            mel_frames,
            dtype=np.float32
        )

    # ------------------------------------------------------------
    # G. Aggregate pitch and energy at phone level
    # ------------------------------------------------------------

    phone_pitch = np.zeros(
        len(text_ids),
        dtype=np.float32
    )

    phone_energy = np.zeros(
        len(text_ids),
        dtype=np.float32
    )

    for i, (start, end) in enumerate(
        zip(boundaries[:-1], boundaries[1:])
    ):

        if end > start:

            phone_pitch[i] = float(
                np.mean(f0_frames[start:end])
            )

            phone_energy[i] = float(
                np.mean(frame_energy[start:end])
            )

    # ------------------------------------------------------------
    # H. Numerical safety and final shapes
    # ------------------------------------------------------------

    mel = np.nan_to_num(
        mel,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    phone_pitch = np.nan_to_num(phone_pitch)
    phone_energy = np.nan_to_num(phone_energy)

    return {
        "text": text_ids,
        "duration": durations,
        "pitch": phone_pitch,
        "energy": phone_energy,
        "mel": mel.T.astype(np.float32),
        "labels": labels,
        "frames": mel_frames,
        "voiced_frames": int(
            np.sum(f0_frames > 0)
        )
    }


# ================================================================
# 9. RESUMABLE SPLIT PROCESSOR
# ================================================================

def process_split(
    name,
    audio_dir,
    grid_dir,
    output_dir
):

    audio_files = sorted(
        audio_dir.glob("*.wav")
    )

    processed = []
    reused = 0
    missing_alignment = 0
    failed = []

    pitch_values = []
    energy_values = []

    metadata_path = output_dir / "metadata.csv"

    for wav_path in tqdm(
        audio_files,
        desc=f"English {name}"
    ):

        basename = wav_path.stem

        grid_path = (
            grid_dir / f"{basename}.TextGrid"
        )

        output_path = (
            output_dir / f"{basename}.npz"
        )

        # Missing alignment
        if not grid_path.exists():

            missing_alignment += 1
            continue

        # --------------------------------------------------------
        # Reuse valid existing output
        # --------------------------------------------------------

        if output_path.exists():

            try:

                with np.load(
                    output_path,
                    allow_pickle=False
                ) as data:

                    valid_existing = (
                        data["text"].ndim == 1
                        and data["duration"].ndim == 1
                        and data["pitch"].ndim == 1
                        and data["energy"].ndim == 1
                        and data["mel"].ndim == 2
                        and data["mel"].shape[1] == N_MELS
                        and len(data["text"]) == len(data["duration"])
                        and len(data["text"]) == len(data["pitch"])
                        and len(data["text"]) == len(data["energy"])
                        and int(data["duration"].sum()) == len(data["mel"])
                    )

                    if valid_existing:

                        processed.append(basename)

                        pitch_values.append(
                            data["pitch"].copy()
                        )

                        energy_values.append(
                            data["energy"].copy()
                        )

                        reused += 1
                        continue

            except Exception:
                pass

        # --------------------------------------------------------
        # Extract new features
        # --------------------------------------------------------

        try:

            features = extract_acoustic_features(
                wav_path,
                grid_path
            )

            tg = textgrid.openTextgrid(
                str(grid_path),
                includeEmptyIntervals=True
            )

            transcript = read_transcript(
                wav_path,
                tg
            )

            # Atomic save to prevent partial files.
            temporary = output_dir / f"{basename}.tmp"

            with open(temporary, "wb") as f:

                np.savez_compressed(
                    f,
                    text=features["text"],
                    duration=features["duration"],
                    pitch=features["pitch"],
                    energy=features["energy"],
                    mel=features["mel"],
                    raw_text=np.asarray(transcript),
                    speaker=np.asarray("Svarah")
                )

            temporary.replace(output_path)

            processed.append(basename)

            pitch_values.append(
                features["pitch"]
            )

            energy_values.append(
                features["energy"]
            )

        except Exception as exc:

            failed.append({
                "id": basename,
                "error": repr(exc)
            })

    # ============================================================
    # Write metadata
    # ============================================================

    with open(
        metadata_path,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "id",
            "feature_file",
            "speaker"
        ])

        for basename in processed:

            writer.writerow([
                basename,
                f"{basename}.npz",
                "Svarah"
            ])

    # ============================================================
    # Processing report
    # ============================================================

    report = {
        "split": name,
        "audio_files": len(audio_files),
        "processed_or_reused": len(processed),
        "newly_processed": len(processed) - reused,
        "reused": reused,
        "missing_alignments": missing_alignment,
        "failed_count": len(failed),
        "failed": failed,
        "feature_directory": str(output_dir)
    }

    with open(
        output_dir / "processing_report.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            report,
            f,
            indent=2
        )

    return (
        processed,
        pitch_values,
        energy_values,
        failed,
        report
    )


# ================================================================
# 10. PROCESS TRAINING DATA
# ================================================================

start_time = time.time()

print("\nStarting English training feature extraction...")

(
    train_ids,
    train_pitch,
    train_energy,
    train_errors,
    train_report
) = process_split(
    "training",
    TRAIN_AUDIO,
    TRAIN_GRID,
    OUT_TRAIN
)

# ================================================================
# 11. PROCESS VALIDATION DATA
# ================================================================

print("\nStarting English validation feature extraction...")

(
    val_ids,
    val_pitch,
    val_energy,
    val_errors,
    val_report
) = process_split(
    "validation",
    VAL_AUDIO,
    VAL_GRID,
    OUT_VAL
)

# ================================================================
# 12. TRAINING-ONLY NORMALIZATION STATISTICS
# ================================================================

def calculate_stats(
    arrays,
    positive_only=False
):

    if not arrays:

        return {
            "mean": 0.0,
            "std": 1.0,
            "count": 0
        }

    values = np.concatenate([
        np.asarray(a, dtype=np.float64).ravel()
        for a in arrays
    ])

    values = values[
        np.isfinite(values)
    ]

    if positive_only:

        values = values[
            values > 0
        ]

    if len(values) == 0:

        return {
            "mean": 0.0,
            "std": 1.0,
            "count": 0
        }

    mean = float(np.mean(values))
    std = float(np.std(values))

    if std < 1e-8:
        std = 1.0

    return {
        "mean": mean,
        "std": std,
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "count": int(len(values))
    }


pitch_stats = calculate_stats(
    train_pitch,
    positive_only=True
)

energy_stats = calculate_stats(
    train_energy
)

stats = {
    "sampling_rate": SR,
    "hop_length": HOP,
    "n_mels": N_MELS,
    "n_fft": N_FFT,
    "win_length": WIN,
    "vocabulary_size": VOCAB_SIZE,
    "silence_id": SIL_ID,
    "pitch": pitch_stats,
    "energy": energy_stats,
    "training_samples": len(train_ids),
    "validation_samples": len(val_ids)
}

with open(
    OUT_STATS / "stats.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        stats,
        f,
        indent=2
    )

# ================================================================
# 13. FINAL VERIFICATION
# ================================================================

elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("ENGLISH FEATURE EXTRACTION SUMMARY")
print("=" * 70)

print("\nTRAINING")
print(json.dumps(train_report, indent=2))

print("\nVALIDATION")
print(json.dumps(val_report, indent=2))

print("\nPitch statistics:")
print(json.dumps(pitch_stats, indent=2))

print("\nEnergy statistics:")
print(json.dumps(energy_stats, indent=2))

print("\nElapsed minutes:", round(elapsed / 60, 2))

# ================================================================
# 14. SAMPLE FEATURE VERIFICATION
# ================================================================

sample_files = sorted(
    OUT_TRAIN.glob("*.npz")
)

if not sample_files:

    raise RuntimeError(
        "No training feature files were generated."
    )

sample_path = sample_files[0]

with np.load(
    sample_path,
    allow_pickle=False
) as sample:

    print("\n" + "=" * 70)
    print("SAMPLE FEATURE VERIFICATION")
    print("=" * 70)

    print("File:", sample_path.name)
    print("Text shape:", sample["text"].shape)
    print("Duration shape:", sample["duration"].shape)
    print("Pitch shape:", sample["pitch"].shape)
    print("Energy shape:", sample["energy"].shape)
    print("Mel shape:", sample["mel"].shape)

    print(
        "Duration sum:",
        int(sample["duration"].sum())
    )

    print(
        "Mel frames:",
        len(sample["mel"])
    )

    print(
        "Mel min/max:",
        float(sample["mel"].min()),
        float(sample["mel"].max())
    )

    print(
        "Pitch min/max:",
        float(sample["pitch"].min()),
        float(sample["pitch"].max())
    )

    print(
        "Energy min/max:",
        float(sample["energy"].min()),
        float(sample["energy"].max())
    )

    assert len(sample["text"]) == len(sample["duration"])
    assert len(sample["text"]) == len(sample["pitch"])
    assert len(sample["text"]) == len(sample["energy"])

    assert int(sample["duration"].sum()) == len(sample["mel"])
    assert sample["mel"].shape[1] == N_MELS

    assert np.all(sample["text"] >= 0)
    assert np.all(sample["text"] < VOCAB_SIZE)

    assert np.isfinite(sample["mel"]).all()
    assert np.isfinite(sample["pitch"]).all()
    assert np.isfinite(sample["energy"]).all()

# ================================================================
# 15. COMPLETION
# ================================================================

print("\n" + "=" * 70)
print("STEP 129-130 COMPLETED")
print("=" * 70)

print("Training features:", len(train_ids))
print("Validation features:", len(val_ids))
print("Training errors:", len(train_errors))
print("Validation errors:", len(val_errors))

print("\nOutput locations:")
print("Training:", OUT_TRAIN)
print("Validation:", OUT_VAL)
print("Statistics:", OUT_STATS / "stats.json")

print("\nReview failed recordings before training.")


FASTSPEECH2 ENGLISH FEATURE EXTRACTION
Device: cuda
GPU: Tesla T4
PyTorch: 2.11.0+cu130
Librosa: 0.11.0
Sample rate: 22050
FFT: 1024
Hop length: 256
Mel channels: 80
Vocabulary size: 67

STFT initialized successfully.

Input file counts:
Training WAV: 2664
Training TextGrid: 2662
Validation WAV: 505
Validation TextGrid: 501

Starting English training feature extraction...


English training:   0%|          | 0/2664 [00:00<?, ?it/s]


Starting English validation feature extraction...


English validation:   0%|          | 0/505 [00:00<?, ?it/s]


ENGLISH FEATURE EXTRACTION SUMMARY

TRAINING
{
  "split": "training",
  "audio_files": 2664,
  "processed_or_reused": 2662,
  "newly_processed": 2662,
  "reused": 0,
  "missing_alignments": 2,
  "failed_count": 0,
  "failed": [],
  "feature_directory": "/content/gdrive/MyDrive/FastSpeech2_Assignment/English/processed/train"
}

VALIDATION
{
  "split": "validation",
  "audio_files": 505,
  "processed_or_reused": 501,
  "newly_processed": 501,
  "reused": 0,
  "missing_alignments": 4,
  "failed_count": 0,
  "failed": [],
  "feature_directory": "/content/gdrive/MyDrive/FastSpeech2_Assignment/English/processed/validation"
}

Pitch statistics:
{
  "mean": 179.5105018095426,
  "std": 59.33071361434518,
  "min": 64.46283721923828,
  "max": 758.3485107421875,
  "count": 151841
}

Energy statistics:
{
  "mean": 26.42564903151013,
  "std": 26.422304389107822,
  "min": 0.0,
  "max": 226.0308380126953,
  "count": 152163
}

Elapsed minutes: 35.15

SAMPLE FEATURE VERIFICATION
File: recording_0.npz
T

In [34]:

# ================================================================
# STEP 131: FASTSPEECH2 MODEL AND TRAINING INTERFACE INSPECTION
# READ-ONLY — DOES NOT MODIFY YOUR PROJECT
# ================================================================

import ast
import inspect
import sys
from pathlib import Path

REPO = Path("/content/FastSpeech2")
sys.path.insert(0, str(REPO))

files_to_check = [
    REPO / "model" / "fastspeech2.py",
    REPO / "model" / "loss.py",
    REPO / "train.py",
    REPO / "dataset.py",
    REPO / "transformer" / "Models.py",
]

print("=" * 75)
print("STEP 131: FASTSPEECH2 INTEGRATION INSPECTION")
print("=" * 75)

for path in files_to_check:
    print(f"\n{'=' * 75}")
    print("FILE:", path.relative_to(REPO))
    print("EXISTS:", path.exists())
    print("=" * 75)

    if not path.exists():
        continue

    source = path.read_text(encoding="utf-8")

    try:
        tree = ast.parse(source)

        print("\nCLASSES AND FUNCTION SIGNATURES:")

        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                if node.col_offset == 0:
                    args = [
                        arg.arg for arg in node.args.args
                    ]
                    print(
                        f"  FUNCTION {node.name}({', '.join(args)}) "
                        f"[line {node.lineno}]"
                    )

            elif isinstance(node, ast.ClassDef):
                if node.col_offset == 0:
                    print(
                        f"\n  CLASS {node.name} [line {node.lineno}]"
                    )

                    for child in node.body:
                        if isinstance(child, ast.FunctionDef):
                            args = [
                                arg.arg for arg in child.args.args
                            ]
                            print(
                                f"    {child.name}({', '.join(args)})"
                                f" [line {child.lineno}]"
                            )

    except Exception as e:
        print("AST inspection error:", repr(e))

    # Print important source sections
    lines = source.splitlines()

    keywords = [
        "def forward",
        "def __getitem__",
        "def collate_fn",
        "def collate",
        "def train",
        "def evaluate",
        "class FastSpeech2Loss",
        "class Dataset",
        "optimizer",
        "criterion",
        "mel_loss",
        "duration_loss",
        "pitch_loss",
        "energy_loss",
    ]

    matched = []

    for i, line in enumerate(lines):
        if any(keyword in line for keyword in keywords):
            start = max(0, i - 2)
            end = min(len(lines), i + 12)

            matched.append((start, end))

    # Merge overlapping sections
    merged = []

    for start, end in matched:
        if merged and start <= merged[-1][1]:
            merged[-1] = (
                merged[-1][0],
                max(merged[-1][1], end)
            )
        else:
            merged.append((start, end))

    print("\nIMPORTANT SOURCE SECTIONS:")

    for start, end in merged[:15]:
        print(f"\n--- Lines {start + 1} to {end} ---")

        for i in range(start, end):
            print(f"{i + 1:4}: {lines[i]}")

print("\n" + "=" * 75)
print("STEP 131 INSPECTION COMPLETED")
print("=" * 75)


STEP 131: FASTSPEECH2 INTEGRATION INSPECTION

FILE: model/fastspeech2.py
EXISTS: True

CLASSES AND FUNCTION SIGNATURES:

  CLASS FastSpeech2 [line 13]
    __init__(self, preprocess_config, model_config) [line 16]
    forward(self, speakers, texts, src_lens, max_src_len, mels, mel_lens, max_mel_len, p_targets, e_targets, d_targets, p_control, e_control, d_control) [line 43]

IMPORTANT SOURCE SECTIONS:

--- Lines 41 to 54 ---
  41:             )
  42: 
  43:     def forward(
  44:         self,
  45:         speakers,
  46:         texts,
  47:         src_lens,
  48:         max_src_len,
  49:         mels=None,
  50:         mel_lens=None,
  51:         max_mel_len=None,
  52:         p_targets=None,
  53:         e_targets=None,
  54:         d_targets=None,

FILE: model/loss.py
EXISTS: True

CLASSES AND FUNCTION SIGNATURES:

  CLASS FastSpeech2Loss [line 5]
    __init__(self, preprocess_config, model_config) [line 8]
    forward(self, inputs, predictions) [line 19]

IMPORTANT SOURCE 

In [35]:

# ================================================================
# STEP 132: COMPLETE FASTSPEECH2 TRAINING INTERFACE INSPECTION
# READ ONLY — NO FILES MODIFIED
# ================================================================

from pathlib import Path
import yaml
import inspect
import sys

REPO = Path("/content/FastSpeech2")
ENGLISH = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/English"
)

sys.path.insert(0, str(REPO))

def print_file(path, start=1, end=None):
    print("\n" + "=" * 85)
    print("FILE:", path)
    print("=" * 85)

    lines = Path(path).read_text(
        encoding="utf-8"
    ).splitlines()

    if end is None:
        end = len(lines)

    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")


# 1. Complete model architecture and forward pass
print_file(
    REPO / "model/fastspeech2.py"
)

# 2. Complete FastSpeech2 loss
print_file(
    REPO / "model/loss.py"
)

# 3. Original training loop
print_file(
    REPO / "train.py"
)

# 4. Model creation and optimizer implementation
for candidate in [
    REPO / "utils/model.py",
    REPO / "optimizer.py",
    REPO / "utils/tools.py"
]:
    if candidate.exists():
        print_file(candidate)

# 5. English configuration files
for filename in [
    "preprocess.yaml",
    "model.yaml",
    "train.yaml"
]:
    path = ENGLISH / "config" / filename

    if path.exists():
        print_file(path)

print("\n" + "=" * 85)
print("STEP 132 INSPECTION COMPLETED")
print("=" * 85)



FILE: /content/FastSpeech2/model/fastspeech2.py
   1: import os
   2: import json
   3: 
   4: import torch
   5: import torch.nn as nn
   6: import torch.nn.functional as F
   7: 
   8: from transformer import Encoder, Decoder, PostNet
   9: from .modules import VarianceAdaptor
  10: from utils.tools import get_mask_from_lengths
  11: 
  12: 
  13: class FastSpeech2(nn.Module):
  14:     """ FastSpeech2 """
  15: 
  16:     def __init__(self, preprocess_config, model_config):
  17:         super(FastSpeech2, self).__init__()
  18:         self.model_config = model_config
  19: 
  20:         self.encoder = Encoder(model_config)
  21:         self.variance_adaptor = VarianceAdaptor(preprocess_config, model_config)
  22:         self.decoder = Decoder(model_config)
  23:         self.mel_linear = nn.Linear(
  24:             model_config["transformer"]["decoder_hidden"],
  25:             preprocess_config["preprocessing"]["mel"]["n_mel_channels"],
  26:         )
  27:         self.po

In [36]:

# ================================================================
# STEP 133: VARIANCE ADAPTOR, EMBEDDING AND OPTIMIZER INSPECTION
# READ ONLY — NO FILES MODIFIED
# ================================================================

from pathlib import Path

REPO = Path("/content/FastSpeech2")

files = [
    "model/modules.py",
    "transformer/Models.py",
    "transformer/Layers.py",
    "model/optimizer.py",
    "model/__init__.py",
]

for relative_path in files:

    path = REPO / relative_path

    print("\n" + "=" * 85)
    print("FILE:", relative_path)
    print("=" * 85)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    # Print complete relevant source.
    for i, line in enumerate(lines, 1):
        print(f"{i:4}: {line}")

# Inspect saved English statistics
import json

stats_path = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/"
    "English/processed/stats/stats.json"
)

print("\n" + "=" * 85)
print("ENGLISH FEATURE STATISTICS")
print("=" * 85)

with open(stats_path, encoding="utf-8") as f:
    stats = json.load(f)

print(json.dumps(stats, indent=2))

print("\nSTEP 133 INSPECTION COMPLETED")



FILE: model/modules.py
   1: import os
   2: import json
   3: import copy
   4: import math
   5: from collections import OrderedDict
   6: 
   7: import torch
   8: import torch.nn as nn
   9: import numpy as np
  10: import torch.nn.functional as F
  11: 
  12: from utils.tools import get_mask_from_lengths, pad
  13: 
  14: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  15: 
  16: 
  17: class VarianceAdaptor(nn.Module):
  18:     """Variance Adaptor"""
  19: 
  20:     def __init__(self, preprocess_config, model_config):
  21:         super(VarianceAdaptor, self).__init__()
  22:         self.duration_predictor = VariancePredictor(model_config)
  23:         self.length_regulator = LengthRegulator()
  24:         self.pitch_predictor = VariancePredictor(model_config)
  25:         self.energy_predictor = VariancePredictor(model_config)
  26: 
  27:         self.pitch_feature_level = preprocess_config["preprocessing"]["pitch"][
  28:             "feature"
 

In [39]:
# ============================================================
# ENGLISH FASTSPEECH 2 — FIX MISSING stats.json
# RUN NOW, BEFORE RUNNING THE TRAINING CELL AGAIN
# ============================================================

import os
import json
import shutil

ENGLISH_ROOT = (
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/English"
)

SOURCE_STATS = os.path.join(
    ENGLISH_ROOT, "processed", "stats", "stats.json"
)

EXPECTED_STATS = os.path.join(
    ENGLISH_ROOT, "processed", "stats.json"
)

print("Checking statistics file...")

if not os.path.exists(SOURCE_STATS):
    raise FileNotFoundError(
        f"Original statistics file not found: {SOURCE_STATS}"
    )

os.makedirs(os.path.dirname(EXPECTED_STATS), exist_ok=True)

shutil.copy2(SOURCE_STATS, EXPECTED_STATS)

with open(EXPECTED_STATS, "r") as f:
    stats = json.load(f)

print("\nStatistics copied successfully.")
print("Source:", SOURCE_STATS)
print("Destination:", EXPECTED_STATS)
print("Pitch statistics:", stats["pitch"])
print("Energy statistics:", stats["energy"])

assert os.path.exists(EXPECTED_STATS)

print("\nSUCCESS: FastSpeech 2 can now locate stats.json.")

Checking statistics file...

Statistics copied successfully.
Source: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/processed/stats/stats.json
Destination: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/processed/stats.json
Pitch statistics: {'mean': 179.5105018095426, 'std': 59.33071361434518, 'min': 64.46283721923828, 'max': 758.3485107421875, 'count': 151841}
Energy statistics: {'mean': 26.42564903151013, 'std': 26.422304389107822, 'min': 0.0, 'max': 226.0308380126953, 'count': 152163}

SUCCESS: FastSpeech 2 can now locate stats.json.


In [41]:

# =====================================================================
# ENGLISH FASTSPEECH 2 — COMPLETE REPLACEMENT TRAINING PIPELINE
# Dataset: Svarah | IPA vocabulary: 67 tokens | GPU: Tesla T4
#
# Includes:
# 1. Environment and configuration
# 2. Compatibility patches
# 3. Clean module imports (fixes Encoder super() conflict)
# 4. Custom IPA dataset and collate function
# 5. FastSpeech2 model and loss
# 6. Pre-training verification
# 7. Training, validation and checkpointing
# 8. Automatic resume
# 9. CSV logs and loss plots
#
# Does NOT rerun alignment or feature extraction.
# =====================================================================

import os
import sys
import json
import time
import random
import importlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =====================================================================
# 1. ENVIRONMENT
# =====================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

REPO = Path("/content/FastSpeech2")
ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = ROOT / "English"

CONFIG_DIR = ENGLISH / "config"
PROCESSED_DIR = ENGLISH / "processed"

TRAIN_DIR = PROCESSED_DIR / "train"
VAL_DIR = PROCESSED_DIR / "validation"
STATS_PATH = PROCESSED_DIR / "stats/stats.json"
VOCAB_PATH = CONFIG_DIR / "english_ipa_vocabulary.json"

CKPT_DIR = ENGLISH / "checkpoints"
LOG_DIR = ENGLISH / "logs"
RESULT_DIR = ENGLISH / "results"

for folder in [CKPT_DIR, LOG_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

assert REPO.exists(), "FastSpeech2 repository not found."
assert TRAIN_DIR.exists(), "English training features missing."
assert VAL_DIR.exists(), "English validation features missing."
assert STATS_PATH.exists(), "Feature statistics missing."
assert VOCAB_PATH.exists(), "IPA vocabulary missing."

os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("=" * 75)
print("ENGLISH FASTSPEECH 2 TRAINING")
print("=" * 75)

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )

# =====================================================================
# 2. CONFIGURATIONS AND VOCABULARY
# =====================================================================

def load_yaml(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

preprocess_config = load_yaml(CONFIG_DIR / "preprocess.yaml")
model_config = load_yaml(CONFIG_DIR / "model.yaml")
train_config = load_yaml(CONFIG_DIR / "train.yaml")

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = json.load(f)

phone_to_id = vocab["phone_to_id"]
vocab_size = len(phone_to_id)

model_config["input_vocabulary_size"] = vocab_size

preprocess_config["path"]["preprocessed_path"] = str(
    PROCESSED_DIR
)

print("\nVocabulary size:", vocab_size)
print("PAD:", phone_to_id.get("<pad>"))
print("UNK:", phone_to_id.get("<unk>"))
print("SIL:", phone_to_id.get("<sil>"))

assert vocab_size == 67

# =====================================================================
# 3. FEATURE STATISTICS
# =====================================================================

with open(STATS_PATH, "r", encoding="utf-8") as f:
    feature_stats = json.load(f)

def read_stat(feature, key):
    values = feature_stats[feature]

    if isinstance(values, dict):
        return float(values[key])

    legacy_index = {
        "min": 0,
        "max": 1,
        "mean": 2,
        "std": 3
    }

    return float(values[legacy_index[key]])

PITCH_MEAN = read_stat("pitch", "mean")
PITCH_STD = max(read_stat("pitch", "std"), 1e-8)

ENERGY_MEAN = read_stat("energy", "mean")
ENERGY_STD = max(read_stat("energy", "std"), 1e-8)

print("\nPitch mean/std:", PITCH_MEAN, PITCH_STD)
print("Energy mean/std:", ENERGY_MEAN, ENERGY_STD)

# =====================================================================
# 4. PATCH VARIANCE ADAPTOR
#
# Original repository expects list statistics.
# Our stats.json stores dictionaries.
# Normalize quantization bounds consistently with the targets.
# Fix energy control to use e_control.
# =====================================================================

modules_path = REPO / "model/modules.py"
modules_source = modules_path.read_text(encoding="utf-8")

old_stats = '''            pitch_min, pitch_max = stats["pitch"][:2]
            energy_min, energy_max = stats["energy"][:2]'''

new_stats = '''            def read_feature_bounds(feature):
                values = stats[feature]

                if isinstance(values, dict):
                    minimum = float(values["min"])
                    maximum = float(values["max"])
                    mean = float(values["mean"])
                    std = max(float(values["std"]), 1e-8)
                else:
                    minimum = float(values[0])
                    maximum = float(values[1])
                    mean = float(values[2])
                    std = max(float(values[3]), 1e-8)

                if preprocess_config["preprocessing"][feature].get(
                    "normalization", False
                ):
                    minimum = (minimum - mean) / std
                    maximum = (maximum - mean) / std

                return minimum, maximum

            pitch_min, pitch_max = read_feature_bounds("pitch")
            energy_min, energy_max = read_feature_bounds("energy")'''

if old_stats in modules_source:
    modules_source = modules_source.replace(
        old_stats, new_stats, 1
    )
elif "def read_feature_bounds(feature):" not in modules_source:
    raise RuntimeError(
        "Could not safely patch variance-adaptor statistics."
    )

old_energy = "x, energy_target, src_mask, p_control"

if old_energy in modules_source:
    modules_source = modules_source.replace(
        old_energy,
        "x, energy_target, src_mask, e_control",
        1
    )
elif "x, energy_target, src_mask, e_control" not in modules_source:
    raise RuntimeError(
        "Could not safely patch energy control."
    )

modules_path.write_text(
    modules_source,
    encoding="utf-8"
)

print("\nVariance adaptor compatibility patches applied.")

# =====================================================================
# 5. FIX ENCODER/DECODER SUPER() RELOADING ISSUE
#
# The previous cell reloaded a class while another module still held
# a reference to its older version.
#
# This cell changes explicit super(Class, self) to super().
# Then it clears and freshly imports the dependent modules.
# =====================================================================

models_path = REPO / "transformer/Models.py"
models_source = models_path.read_text(encoding="utf-8")

models_source = models_source.replace(
    "super(Encoder, self).__init__()",
    "super().__init__()"
)

models_source = models_source.replace(
    "super(Decoder, self).__init__()",
    "super().__init__()"
)

models_path.write_text(
    models_source,
    encoding="utf-8"
)

# Remove stale module objects from the current Python session.
# This is an in-memory cleanup; it does not delete project files.

for module_name in list(sys.modules.keys()):
    if (
        module_name == "model"
        or module_name.startswith("model.")
        or module_name == "transformer"
        or module_name.startswith("transformer.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

# Import clean classes from the patched source.
from model import FastSpeech2, FastSpeech2Loss, ScheduledOptim

print("Fresh model imports successful.")

# =====================================================================
# 6. CUSTOM ENGLISH IPA DATASET
# =====================================================================

MAX_SEQUENCE_LENGTH = int(model_config["max_seq_len"])

def find_npz_value(data, keys, description):
    for key in keys:
        if key in data:
            return data[key]

    raise KeyError(
        f"Missing {description}. "
        f"Available keys: {list(data.keys())}"
    )


class EnglishIPADataset(Dataset):

    def __init__(self, folder, split):

        self.files = sorted(Path(folder).glob("*.npz"))
        self.samples = []
        self.rejected = []

        for file_path in self.files:

            try:

                with np.load(
                    file_path,
                    allow_pickle=True
                ) as data:

                    text = find_npz_value(
                        data,
                        ["text", "phone_ids", "phoneme_ids", "tokens"],
                        "IPA token IDs"
                    )

                    duration = find_npz_value(
                        data,
                        ["duration", "durations"],
                        "duration"
                    )

                    pitch = find_npz_value(
                        data,
                        ["pitch", "pitches"],
                        "pitch"
                    )

                    energy = find_npz_value(
                        data,
                        ["energy", "energies"],
                        "energy"
                    )

                    mel = find_npz_value(
                        data,
                        ["mel", "mel_target", "mel_spectrogram"],
                        "mel"
                    )

                text = np.asarray(text).reshape(-1).astype(np.int64)
                duration = np.asarray(duration).reshape(-1).astype(np.float32)
                pitch = np.asarray(pitch).reshape(-1).astype(np.float32)
                energy = np.asarray(energy).reshape(-1).astype(np.float32)
                mel = np.asarray(mel, dtype=np.float32)

                if mel.ndim != 2:
                    raise ValueError("Mel must be two-dimensional.")

                if mel.shape[0] == 80 and mel.shape[1] != 80:
                    mel = mel.T

                if mel.shape[1] != 80:
                    raise ValueError("Mel must have 80 channels.")

                if not (
                    len(text) == len(duration)
                    == len(pitch) == len(energy)
                ):
                    raise ValueError("Feature lengths do not match.")

                if len(text) == 0 or len(mel) == 0:
                    raise ValueError("Empty sample.")

                if not all([
                    np.all(np.isfinite(text)),
                    np.all(np.isfinite(duration)),
                    np.all(np.isfinite(pitch)),
                    np.all(np.isfinite(energy)),
                    np.all(np.isfinite(mel))
                ]):
                    raise ValueError("Non-finite values found.")

                if np.any(duration < 0):
                    raise ValueError("Negative durations.")

                duration = np.rint(duration).astype(np.int64)

                if int(duration.sum()) != len(mel):
                    raise ValueError(
                        "Duration sum does not equal mel frames."
                    )

                if np.any(text < 0) or np.any(text >= vocab_size):
                    raise ValueError("Invalid IPA token ID.")

                # Encoder input must fit its positional embedding.
                if len(text) > MAX_SEQUENCE_LENGTH:
                    raise ValueError("Text exceeds maximum sequence length.")

                # Decoder also uses the configured maximum length during
                # training. Do not silently truncate targets.
                if len(mel) > MAX_SEQUENCE_LENGTH:
                    raise ValueError(
                        "Mel exceeds maximum decoder sequence length."
                    )

                # Normalize targets consistently with adaptor bins.
                pitch = (pitch - PITCH_MEAN) / PITCH_STD
                energy = (energy - ENERGY_MEAN) / ENERGY_STD

                self.samples.append({
                    "id": file_path.stem,
                    "text": text,
                    "duration": duration,
                    "pitch": pitch.astype(np.float32),
                    "energy": energy.astype(np.float32),
                    "mel": mel.astype(np.float32)
                })

            except Exception as exc:
                self.rejected.append(
                    (file_path.name, str(exc))
                )

        print(f"\n{split} dataset")
        print("NPZ files:", len(self.files))
        print("Valid:", len(self.samples))
        print("Rejected:", len(self.rejected))

        if self.rejected:
            print("First five rejected:")
            for name, reason in self.rejected[:5]:
                print(name, "->", reason)

        if not self.samples:
            raise RuntimeError(f"No usable samples in {split}.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


# =====================================================================
# 7. COLLATE FUNCTION
# =====================================================================

def english_collate(batch):

    batch_size = len(batch)

    src_lengths = [len(x["text"]) for x in batch]
    mel_lengths = [len(x["mel"]) for x in batch]

    max_src_len = max(src_lengths)
    max_mel_len = max(mel_lengths)

    texts = torch.zeros(
        batch_size, max_src_len, dtype=torch.long
    )

    durations = torch.zeros(
        batch_size, max_src_len, dtype=torch.long
    )

    pitches = torch.zeros(
        batch_size, max_src_len, dtype=torch.float32
    )

    energies = torch.zeros(
        batch_size, max_src_len, dtype=torch.float32
    )

    mels = torch.zeros(
        batch_size, max_mel_len, 80, dtype=torch.float32
    )

    for i, item in enumerate(batch):

        src_len = src_lengths[i]
        mel_len = mel_lengths[i]

        texts[i, :src_len] = torch.from_numpy(item["text"])
        durations[i, :src_len] = torch.from_numpy(item["duration"])
        pitches[i, :src_len] = torch.from_numpy(item["pitch"])
        energies[i, :src_len] = torch.from_numpy(item["energy"])
        mels[i, :mel_len] = torch.from_numpy(item["mel"])

    return (
        [x["id"] for x in batch],
        [x["id"] for x in batch],
        torch.zeros(batch_size, dtype=torch.long),
        texts,
        torch.tensor(src_lengths, dtype=torch.long),
        max_src_len,
        mels,
        torch.tensor(mel_lengths, dtype=torch.long),
        max_mel_len,
        pitches,
        energies,
        durations
    )


# =====================================================================
# 8. DATASETS AND LOADERS
# =====================================================================

train_dataset = EnglishIPADataset(TRAIN_DIR, "TRAIN")
val_dataset = EnglishIPADataset(VAL_DIR, "VALIDATION")

BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=english_collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=english_collate
)

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch size:", BATCH_SIZE)

# =====================================================================
# 9. DEVICE TRANSFER
# =====================================================================

def move_batch_to_device(batch):

    return (
        batch[0],
        batch[1],
        batch[2].to(DEVICE),
        batch[3].to(DEVICE),
        batch[4].to(DEVICE),
        batch[5],
        batch[6].to(DEVICE),
        batch[7].to(DEVICE),
        batch[8],
        batch[9].to(DEVICE),
        batch[10].to(DEVICE),
        batch[11].to(DEVICE)
    )


# =====================================================================
# 10. MODEL AND LOSS INITIALIZATION
# =====================================================================

model = FastSpeech2(
    preprocess_config,
    model_config
).to(DEVICE)

criterion = FastSpeech2Loss(
    preprocess_config,
    model_config
).to(DEVICE)

print("\nModel initialized.")
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print(
    "Embedding vocabulary:",
    model.encoder.src_word_emb.num_embeddings
)

assert model.encoder.src_word_emb.num_embeddings == vocab_size

# =====================================================================
# 11. PRE-TRAINING FORWARD PASS
# =====================================================================

test_batch = move_batch_to_device(
    next(iter(train_loader))
)

model.train()

with torch.no_grad():
    test_output = model(*(test_batch[2:]))
    test_losses = criterion(test_batch, test_output)

loss_names = [
    "Total loss",
    "Mel loss",
    "Postnet mel loss",
    "Pitch loss",
    "Energy loss",
    "Duration loss"
]

print("\nPRE-TRAINING VERIFICATION")

for name, loss in zip(loss_names, test_losses):
    print(f"{name}: {loss.item():.6f}")

assert all(
    torch.isfinite(x).item()
    for x in test_losses
), "Non-finite loss. Training stopped."

del test_batch, test_output, test_losses

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nForward pass verified successfully.")

# =====================================================================
# 12. TRAINING SETTINGS
# =====================================================================

MAX_STEPS = 6000
LOG_EVERY = 50
SAVE_EVERY = 500
VALIDATE_EVERY = 1000

optimizer = ScheduledOptim(
    model,
    train_config,
    model_config,
    current_step=0
)

checkpoint_path = CKPT_DIR / "latest_checkpoint.pth.tar"
best_checkpoint_path = CKPT_DIR / "best_checkpoint.pth.tar"
history_path = LOG_DIR / "english_training_history.csv"

history_columns = [
    "step",
    "train_total",
    "train_mel",
    "train_postnet",
    "train_pitch",
    "train_energy",
    "train_duration",
    "validation_total",
    "validation_mel",
    "validation_postnet",
    "validation_pitch",
    "validation_energy",
    "validation_duration"
]

history = {
    key: [] for key in history_columns
}

start_step = 0
best_val_loss = float("inf")

# =====================================================================
# 13. AUTOMATIC RESUME
# =====================================================================

if checkpoint_path.exists():

    print("\nExisting checkpoint found. Resuming training.")

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model"])

    optimizer._optimizer.load_state_dict(
        checkpoint["optimizer"]
    )

    start_step = int(checkpoint["step"])
    optimizer.current_step = start_step

    best_val_loss = checkpoint.get(
        "best_val_loss",
        float("inf")
    )

    if history_path.exists():

        previous = pd.read_csv(history_path)

        for key in history:
            if key in previous.columns:
                history[key] = previous[key].tolist()

    print("Resume step:", start_step)

else:
    print("\nStarting new training session.")

# =====================================================================
# 14. VALIDATION FUNCTION
# =====================================================================

def run_validation():

    model.eval()

    accumulated = np.zeros(6, dtype=np.float64)
    count = 0

    with torch.no_grad():

        for batch in val_loader:

            batch = move_batch_to_device(batch)

            output = model(*(batch[2:]))

            losses = criterion(batch, output)

            accumulated += np.array([
                x.item() for x in losses
            ])

            count += 1

    model.train()

    if count == 0:
        raise RuntimeError("Validation loader is empty.")

    return (accumulated / count).tolist()


# =====================================================================
# 15. CHECKPOINT FUNCTION
# =====================================================================

def save_checkpoint(step):

    state = {
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer._optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "vocabulary_size": vocab_size,
        "seed": SEED
    }

    torch.save(state, checkpoint_path)

    print("Saved latest checkpoint at step:", step)


# =====================================================================
# 16. HISTORY AND PLOTS
# =====================================================================

def save_history_and_plots():

    df = pd.DataFrame(history)
    df.to_csv(history_path, index=False)

    if df.empty:
        return

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(
        df["step"],
        df["train_total"],
        label="Training total loss"
    )

    ax.set_title("English FastSpeech2 Training Loss")
    ax.set_xlabel("Training steps")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)
    ax.legend()

    fig.tight_layout()

    fig.savefig(
        RESULT_DIR / "english_training_loss.png",
        dpi=180
    )

    plt.close(fig)

    valid = df.dropna(subset=["validation_total"])

    if not valid.empty:

        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(
            valid["step"],
            valid["validation_total"],
            marker="o",
            label="Validation total loss"
        )

        ax.set_title("English FastSpeech2 Validation Loss")
        ax.set_xlabel("Training steps")
        ax.set_ylabel("Loss")
        ax.grid(alpha=0.3)
        ax.legend()

        fig.tight_layout()

        fig.savefig(
            RESULT_DIR / "english_validation_loss.png",
            dpi=180
        )

        plt.close(fig)


# =====================================================================
# 17. COMPLETE TRAINING LOOP
# =====================================================================

print("\n" + "=" * 75)
print("ENGLISH FASTSPEECH 2 TRAINING STARTED")
print("=" * 75)

print("Maximum steps:", MAX_STEPS)
print("Initial step:", start_step)
print("Batch size:", BATCH_SIZE)
print("Validation interval:", VALIDATE_EVERY)

model.train()

step = start_step
running_losses = np.zeros(6, dtype=np.float64)
running_count = 0

session_start = time.time()

try:

    while step < MAX_STEPS:

        for batch in train_loader:

            if step >= MAX_STEPS:
                break

            batch = move_batch_to_device(batch)

            optimizer.zero_grad()

            output = model(*(batch[2:]))

            losses = criterion(batch, output)

            total_loss = losses[0]

            if not torch.isfinite(total_loss):
                raise RuntimeError(
                    f"Non-finite loss at step {step + 1}"
                )

            total_loss.backward()

            nn.utils.clip_grad_norm_(
                model.parameters(),
                train_config["optimizer"]["grad_clip_thresh"]
            )

            optimizer.step_and_update_lr()

            step += 1

            running_losses += np.array([
                x.item() for x in losses
            ])

            running_count += 1

            # ---------------------------------------------------------
            # TRAINING LOG
            # ---------------------------------------------------------

            if step % LOG_EVERY == 0:

                avg = running_losses / max(running_count, 1)

                row = {
                    "step": step,
                    "train_total": avg[0],
                    "train_mel": avg[1],
                    "train_postnet": avg[2],
                    "train_pitch": avg[3],
                    "train_energy": avg[4],
                    "train_duration": avg[5],
                    "validation_total": np.nan,
                    "validation_mel": np.nan,
                    "validation_postnet": np.nan,
                    "validation_pitch": np.nan,
                    "validation_energy": np.nan,
                    "validation_duration": np.nan
                }

                for key in history:
                    history[key].append(row[key])

                elapsed = (
                    time.time() - session_start
                ) / 60

                print(
                    f"\nStep {step}/{MAX_STEPS} | "
                    f"Total={avg[0]:.4f} | "
                    f"Mel={avg[1]:.4f} | "
                    f"Postnet={avg[2]:.4f} | "
                    f"Pitch={avg[3]:.4f} | "
                    f"Energy={avg[4]:.4f} | "
                    f"Duration={avg[5]:.4f} | "
                    f"Time={elapsed:.1f} min"
                )

                running_losses[:] = 0
                running_count = 0

                save_history_and_plots()

            # ---------------------------------------------------------
            # VALIDATION
            # ---------------------------------------------------------

            if step % VALIDATE_EVERY == 0:

                print("\nValidation at step:", step)

                val = run_validation()

                print(
                    f"Validation total={val[0]:.4f} | "
                    f"Mel={val[1]:.4f} | "
                    f"Postnet={val[2]:.4f} | "
                    f"Pitch={val[3]:.4f} | "
                    f"Energy={val[4]:.4f} | "
                    f"Duration={val[5]:.4f}"
                )

                if (
                    history["step"]
                    and history["step"][-1] == step
                ):

                    history["validation_total"][-1] = val[0]
                    history["validation_mel"][-1] = val[1]
                    history["validation_postnet"][-1] = val[2]
                    history["validation_pitch"][-1] = val[3]
                    history["validation_energy"][-1] = val[4]
                    history["validation_duration"][-1] = val[5]

                if val[0] < best_val_loss:

                    best_val_loss = val[0]

                    print(
                        "New best validation loss:",
                        best_val_loss
                    )

                    save_checkpoint(step)

                    torch.save(
                        {
                            "step": step,
                            "model": model.state_dict(),
                            "optimizer": optimizer._optimizer.state_dict(),
                            "best_val_loss": best_val_loss,
                            "model_config": model_config,
                            "preprocess_config": preprocess_config,
                            "vocabulary_size": vocab_size
                        },
                        best_checkpoint_path
                    )

                else:
                    save_checkpoint(step)

                save_history_and_plots()

            # ---------------------------------------------------------
            # PERIODIC CHECKPOINT
            # ---------------------------------------------------------

            elif step % SAVE_EVERY == 0:

                save_checkpoint(step)

        # End of epoch: save a recovery checkpoint.
        save_checkpoint(step)
        save_history_and_plots()

except KeyboardInterrupt:

    print("\nTraining interrupted. Saving current progress.")

    save_checkpoint(step)
    save_history_and_plots()

except Exception as error:

    print("\nTraining error:", repr(error))

    try:
        save_checkpoint(step)
        save_history_and_plots()
    except Exception as save_error:
        print("Checkpoint save error:", repr(save_error))

    raise

# =====================================================================
# 18. FINAL SAVE AND SUMMARY
# =====================================================================

save_checkpoint(step)
save_history_and_plots()

elapsed = (time.time() - session_start) / 60

print("\n" + "=" * 75)
print("ENGLISH FASTSPEECH 2 TRAINING SESSION FINISHED")
print("=" * 75)

print("Final step:", step)
print("Best validation loss:", best_val_loss)
print("Elapsed minutes:", round(elapsed, 2))

print("\nFILES SAVED")

print("Latest checkpoint:")
print(checkpoint_path)

print("\nBest checkpoint:")
print(best_checkpoint_path)

print("\nTraining history:")
print(history_path)

print("\nTraining loss plot:")
print(RESULT_DIR / "english_training_loss.png")

print("\nValidation loss plot:")
print(RESULT_DIR / "english_validation_loss.png")

print("\nENGLISH TRAINING PIPELINE COMPLETED")


ENGLISH FASTSPEECH 2 TRAINING
Device: cuda
PyTorch: 2.11.0+cu130
GPU: Tesla T4
GPU memory: 14.56 GB

Vocabulary size: 67
PAD: 0
UNK: 1
SIL: 2

Pitch mean/std: 179.5105018095426 59.33071361434518
Energy mean/std: 26.42564903151013 26.422304389107822

Variance adaptor compatibility patches applied.
Fresh model imports successful.

TRAIN dataset
NPZ files: 2662
Valid: 2630
Rejected: 32
First five rejected:
recording_1359.npz -> Mel exceeds maximum decoder sequence length.
recording_1364.npz -> Mel exceeds maximum decoder sequence length.
recording_1388.npz -> Mel exceeds maximum decoder sequence length.
recording_1445.npz -> Mel exceeds maximum decoder sequence length.
recording_1660.npz -> Mel exceeds maximum decoder sequence length.

VALIDATION dataset
NPZ files: 501
Valid: 496
Rejected: 5
First five rejected:
recording_1196.npz -> Mel exceeds maximum decoder sequence length.
recording_2724.npz -> Mel exceeds maximum decoder sequence length.
recording_4137.npz -> Mel exceeds maximum dec

In [5]:
# ============================================================
# RECOVER ENGLISH FASTSPEECH 2 CHECKPOINT
# ============================================================

import os
import torch
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = BASE / "English"

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled in this runtime.")
    print("Colab: Runtime > Change runtime type > T4 GPU")

print("\nAssignment folder exists:", BASE.exists())
print("English folder exists:", ENGLISH.exists())

# Inspect checkpoint directory
CHECKPOINT_DIR = ENGLISH / "checkpoints"

print("\nCheckpoint directory:", CHECKPOINT_DIR)
print("Exists:", CHECKPOINT_DIR.exists())

if CHECKPOINT_DIR.exists():
    for file in CHECKPOINT_DIR.iterdir():
        if file.is_file():
            print(file.name, f"{file.stat().st_size / (1024**2):.2f} MB")

# Search for checkpoints inside the assignment folder
print("\nSearching for saved checkpoints...")

found = []

if BASE.exists():
    for file in BASE.rglob("*.pth.tar"):
        found.append(file)

    for file in BASE.rglob("*.pt"):
        found.append(file)

for file in found:
    print("FOUND:", file)

if not found:
    print("\nNo checkpoint found inside the assignment folder.")
    print("Check whether training used a different Google Drive folder.")
else:
    best = [
        f for f in found
        if "best_checkpoint" in f.name.lower()
    ]

    latest = [
        f for f in found
        if "latest_checkpoint" in f.name.lower()
    ]

    print("\nBest checkpoints:", len(best))
    print("Latest checkpoints:", len(latest))

Mounted at /content/drive

GPU available: False
Colab: Runtime > Change runtime type > T4 GPU

Assignment folder exists: True
English folder exists: True

Checkpoint directory: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/checkpoints
Exists: False

Searching for saved checkpoints...

No checkpoint found inside the assignment folder.
Check whether training used a different Google Drive folder.


In [8]:
# ============================================================
# GOOGLE DRIVE — SAFE MOUNT AND CHECKPOINT RECOVERY
# ============================================================

from pathlib import Path
from google.colab import drive
import os
import torch

MOUNT_POINT = "/content/drive_recovery"

os.makedirs(MOUNT_POINT, exist_ok=True)

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)
else:
    print("Recovery Drive already mounted.")

print("\nGPU available:", torch.cuda.is_available())

# Search the newly mounted Drive
MYDRIVE = Path(MOUNT_POINT) / "MyDrive"

print("\nMyDrive exists:", MYDRIVE.exists())

matches = []

if MYDRIVE.exists():
    for pattern in [
        "best_checkpoint.pth.tar",
        "latest_checkpoint.pth.tar"
    ]:
        matches.extend(MYDRIVE.rglob(pattern))

if matches:
    print("\nCHECKPOINTS FOUND:")
    for p in matches:
        print("\nPath:", p)
        print("Size:", round(p.stat().st_size / (1024**2), 2), "MB")
else:
    print("\nNo checkpoints found in this Drive.")

Mounted at /content/drive_recovery

GPU available: False

MyDrive exists: True

CHECKPOINTS FOUND:

Path: /content/drive_recovery/MyDrive/FastSpeech2_Assignment/English/checkpoints/best_checkpoint.pth.tar
Size: 89.38 MB

Path: /content/drive_recovery/MyDrive/FastSpeech2_Assignment/English/checkpoints/latest_checkpoint.pth.tar
Size: 89.38 MB


In [10]:
# ============================================================
# GOOGLE DRIVE — SAFE MOUNT AND CHECKPOINT RECOVERY
# ============================================================

from pathlib import Path
from google.colab import drive
import os
import torch

MOUNT_POINT = "/content/drive_recovery"

os.makedirs(MOUNT_POINT, exist_ok=True)

if not os.path.ismount(MOUNT_POINT):
    drive.mount(MOUNT_POINT)
else:
    print("Recovery Drive already mounted.")

print("\nGPU available:", torch.cuda.is_available())

# Search the newly mounted Drive
MYDRIVE = Path(MOUNT_POINT) / "MyDrive"

print("\nMyDrive exists:", MYDRIVE.exists())

matches = []

if MYDRIVE.exists():
    for pattern in [
        "best_checkpoint.pth.tar",
        "latest_checkpoint.pth.tar"
    ]:
        matches.extend(MYDRIVE.rglob(pattern))

if matches:
    print("\nCHECKPOINTS FOUND:")
    for p in matches:
        print("\nPath:", p)
        print("Size:", round(p.stat().st_size / (1024**2), 2), "MB")
else:
    print("\nNo checkpoints found in this Drive.")

Recovery Drive already mounted.

GPU available: False

MyDrive exists: True

CHECKPOINTS FOUND:

Path: /content/drive_recovery/MyDrive/FastSpeech2_Assignment/English/checkpoints/best_checkpoint.pth.tar
Size: 89.38 MB

Path: /content/drive_recovery/MyDrive/FastSpeech2_Assignment/English/checkpoints/latest_checkpoint.pth.tar
Size: 89.38 MB


In [3]:
from pathlib import Path
import os

roots = [
    Path("/content/gdrive/MyDrive"),
    Path("/content/drive_recovery/MyDrive")
]

for root in roots:
    print("\n" + "=" * 60)
    print("Checking:", root)
    print("Exists:", root.exists())

    if not root.exists():
        continue

    assignment = root / "FastSpeech2_Assignment"
    print("Assignment folder exists:", assignment.exists())

    if assignment.exists():
        for name in [
            "best_checkpoint.pth.tar",
            "latest_checkpoint.pth.tar"
        ]:
            matches = list(assignment.rglob(name))
            for path in matches:
                print("\nFOUND CHECKPOINT:", path)
                print("Size (MB):", round(path.stat().st_size / 1024**2, 2))

        vocab = list(assignment.rglob("english_ipa_vocabulary.json"))
        for path in vocab:
            print("\nFOUND VOCABULARY:", path)

print("\nSEARCH COMPLETE")


Checking: /content/gdrive/MyDrive
Exists: True
Assignment folder exists: True

Checking: /content/drive_recovery/MyDrive
Exists: False

SEARCH COMPLETE


In [5]:
from pathlib import Path
import os

print("GOOGLE DRIVE CHECK")

for mount in ["/content/gdrive", "/content/drive_recovery"]:
    root = Path(mount)
    print(f"\n{mount}")
    print("Exists:", root.exists())

    mydrive = root / "MyDrive"
    print("MyDrive exists:", mydrive.exists())

    if mydrive.exists():
        folder = mydrive / "FastSpeech2_Assignment"
        print("Assignment exists:", folder.exists())

        if folder.exists():
            english = folder / "English"
            print("English exists:", english.exists())

            for p in english.rglob("*checkpoint*"):
                if p.is_file():
                    print("CHECKPOINT:", p)
                    print("SIZE MB:", round(p.stat().st_size / 1048576, 2))

            for p in english.rglob("*vocabulary*.json"):
                print("VOCABULARY:", p)

print("\nDIAGNOSTIC FINISHED")

GOOGLE DRIVE CHECK

/content/gdrive
Exists: True
MyDrive exists: True
Assignment exists: True
English exists: True

/content/drive_recovery
Exists: False
MyDrive exists: False

DIAGNOSTIC FINISHED


In [6]:
# ============================================================
# FINAL CHECKPOINT LOCATION SEARCH
# ============================================================

from pathlib import Path

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")

print("ASSIGNMENT DIRECTORY:", BASE)

print("\nENGLISH FOLDER CONTENTS")
english = BASE / "English"

for p in sorted(english.rglob("*")):
    if p.is_file():
        print(
            "FILE:",
            p.relative_to(BASE),
            "| Size:",
            round(p.stat().st_size / 1024**2, 2),
            "MB"
        )
    elif p.is_dir() and p.name in [
        "checkpoints", "results", "logs"
    ]:
        print("FOLDER:", p.relative_to(BASE))

print("\nSEARCHING ENTIRE ASSIGNMENT FOR MODEL CHECKPOINTS")

checkpoint_files = []

for pattern in ["*.pth", "*.pth.tar", "*.pt", "*.ckpt"]:
    checkpoint_files.extend(BASE.rglob(pattern))

# Remove duplicates
checkpoint_files = list(set(checkpoint_files))

if checkpoint_files:
    for p in sorted(checkpoint_files):
        print("\nFOUND:", p)
        print("SIZE:", round(p.stat().st_size / 1024**2, 2), "MB")
else:
    print("NO MODEL CHECKPOINTS FOUND.")

print("\nSEARCH COMPLETE")

ASSIGNMENT DIRECTORY: /content/gdrive/MyDrive/FastSpeech2_Assignment

ENGLISH FOLDER CONTENTS
FOLDER: English/results

SEARCHING ENTIRE ASSIGNMENT FOR MODEL CHECKPOINTS
NO MODEL CHECKPOINTS FOUND.

SEARCH COMPLETE


In [8]:
# ============================================================
# FASTSPEECH 2 — SEARCH FOR TRAINED MODEL CHECKPOINTS
# ============================================================

from pathlib import Path
import os

search_roots = [
    Path("/content"),
    Path("/content/gdrive/MyDrive"),
]

patterns = [
    "*.pth",
    "*.pth.tar",
    "*.pt",
    "*.ckpt"
]

found = set()

for root in search_roots:
    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for pattern in patterns:
        for path in root.rglob(pattern):
            if path.is_file():
                found.add(path)

for path in sorted(found):
    print(
        "\nFOUND:", path,
        "\nSIZE:", round(path.stat().st_size / (1024**2), 2), "MB"
    )

if not found:
    print("\nNO CHECKPOINT FILES FOUND.")


Searching: /content

Searching: /content/gdrive/MyDrive

NO CHECKPOINT FILES FOUND.


In [10]:
from pathlib import Path

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")

checks = {
    "English vocabulary": BASE / "English/config/english_ipa_vocabulary.json",
    "English preprocess config": BASE / "English/config/preprocess.yaml",
    "English model config": BASE / "English/config/model.yaml",
    "English train config": BASE / "English/config/train.yaml",
    "English training features": BASE / "English/processed/train",
    "English validation features": BASE / "English/processed/validation",
    "English training history": BASE / "English/logs/english_training_history.csv",
}

for name, path in checks.items():
    if path.is_dir():
        count = sum(1 for _ in path.glob("*.npz"))
        print(f"{name}: {'FOUND' if path.exists() else 'MISSING'} | NPZ files: {count}")
    else:
        print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

English vocabulary: MISSING
English preprocess config: MISSING
English model config: MISSING
English train config: MISSING
English training features: MISSING
English validation features: MISSING
English training history: MISSING


In [12]:
from pathlib import Path
import os

print("CURRENT GOOGLE DRIVE")

root = Path("/content/gdrive/MyDrive")

print("\nMyDrive exists:", root.exists())

print("\nTop-level folders and files:")
for item in sorted(root.iterdir()):
    print(
        "[DIR] " if item.is_dir() else "[FILE]",
        item.name
    )

print("\nSearching for project-related folders:")
for name in [
    "FastSpeech2_Assignment",
    "FastSpeech2",
    "English",
]:
    matches = list(root.rglob(name))
    print(f"\n{name}:")
    for p in matches[:20]:
        print(p)

CURRENT GOOGLE DRIVE

MyDrive exists: True

Top-level folders and files:
[DIR]  FastSpeech2_Assignment

Searching for project-related folders:

FastSpeech2_Assignment:
/content/gdrive/MyDrive/FastSpeech2_Assignment

FastSpeech2:

English:
/content/gdrive/MyDrive/FastSpeech2_Assignment/English


In [14]:
from pathlib import Path

locations = [
    Path("/content"),
    Path("/content/FastSpeech2"),
    Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
]

keywords = [
    "english_ipa_vocabulary.json",
    "english_training_history.csv",
    "best_checkpoint.pth.tar",
    "latest_checkpoint.pth.tar",
    "stats.json"
]

for root in locations:
    print(f"\n{'='*55}\nSEARCHING: {root}")

    if not root.exists():
        print("Location missing")
        continue

    for keyword in keywords:
        matches = list(root.rglob(keyword))
        for match in matches[:5]:
            print("FOUND:", match)

    for folder_name in ["processed", "checkpoints", "config"]:
        matches = [
            p for p in root.rglob(folder_name)
            if p.is_dir()
        ]
        for match in matches[:5]:
            print("FOLDER:", match)


SEARCHING: /content

SEARCHING: /content/FastSpeech2
Location missing

SEARCHING: /content/gdrive/MyDrive/FastSpeech2_Assignment


In [16]:
# ============================================================
# FASTSPEECH 2 — CHECK TRAINING ARTIFACTS
# ============================================================

from pathlib import Path
import os

locations = [
    Path("/content"),
    Path("/content/gdrive/MyDrive/FastSpeech2_Assignment"),
]

extensions = {".pth", ".pt", ".ckpt", ".npz", ".json", ".yaml", ".csv"}

for root in locations:
    print(f"\n{'='*60}")
    print("LOCATION:", root)

    if not root.exists():
        print("Not found")
        continue

    counts = {}

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in extensions:
            counts[path.suffix] = counts.get(path.suffix, 0) + 1

    print("Artifact counts:", counts)

    for name in [
        "best_checkpoint.pth.tar",
        "latest_checkpoint.pth.tar",
        "english_ipa_vocabulary.json",
        "english_training_history.csv"
    ]:
        matches = list(root.rglob(name))
        for match in matches:
            print("FOUND:", match)

print("\nCHECK COMPLETE")


LOCATION: /content
Artifact counts: {'.yaml': 2, '.json': 2, '.csv': 4}

LOCATION: /content/gdrive/MyDrive/FastSpeech2_Assignment
Artifact counts: {}

CHECK COMPLETE


In [18]:
# ============================================================
# FASTSPEECH 2 — IN-MEMORY MODEL RECOVERY
# Does not retrain or overwrite existing checkpoints
# ============================================================

import os
import torch
from pathlib import Path

print("GPU:", torch.cuda.is_available())

# Check available training variables
for name in [
    "model", "optimizer", "scheduler",
    "step", "global_step", "best_val_loss",
    "history", "best_checkpoint"
]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'NOT AVAILABLE'}")

# Check whether a trained model remains in RAM
if "model" not in globals():
    print("\nNo model object exists in current RAM.")

elif not isinstance(model, torch.nn.Module):
    print("\nThe variable 'model' is not a PyTorch model.")

else:
    print("\nModel found in RAM!")
    print("Model type:", type(model).__name__)

    # Save without overwriting anything
    SAVE_DIR = Path(
        "/content/gdrive/MyDrive/FastSpeech2_Assignment/"
        "English/checkpoints"
    )
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    SAVE_PATH = SAVE_DIR / "recovered_in_memory_checkpoint.pth"

    if SAVE_PATH.exists():
        print("Recovery file already exists:", SAVE_PATH)
    else:
        torch.save(
            {
                "model": model.state_dict(),
                "step": globals().get(
                    "global_step",
                    globals().get("step", -1)
                ),
                "recovered_from_ram": True
            },
            SAVE_PATH
        )

        print("Saved:", SAVE_PATH)
        print("Size (MB):", round(SAVE_PATH.stat().st_size / 1024**2, 2))

        # Verify the file
        saved = torch.load(
            SAVE_PATH,
            map_location="cpu",
            weights_only=False
        )

        print("Verified tensors:", len(saved["model"]))
        print("Recorded step:", saved["step"])
        print("RECOVERY SUCCESSFUL")

GPU: True
model: NOT AVAILABLE
optimizer: NOT AVAILABLE
scheduler: NOT AVAILABLE
step: NOT AVAILABLE
global_step: NOT AVAILABLE
best_val_loss: NOT AVAILABLE
history: NOT AVAILABLE
best_checkpoint: NOT AVAILABLE

No model object exists in current RAM.


In [20]:
from pathlib import Path

print("CHECKING COLAB FILES\n")

for folder in [
    Path("/content"),
    Path("/content/gdrive/MyDrive")
]:
    print("\nLOCATION:", folder)

    if not folder.exists():
        continue

    for p in folder.iterdir():
        print(
            "DIR " if p.is_dir() else "FILE",
            p.name
        )

print("\nCHECKING NOTEBOOK BACKUPS")

for root in [
    Path("/content"),
    Path("/content/gdrive/MyDrive")
]:
    for pattern in ["*.ipynb", "*.py", "*.pth", "*.tar"]:
        for p in root.rglob(pattern):
            print(p)

CHECKING COLAB FILES


LOCATION: /content
DIR  .config
DIR  gdrive
DIR  sample_data

LOCATION: /content/gdrive/MyDrive
DIR  FastSpeech2_Assignment

CHECKING NOTEBOOK BACKUPS


In [21]:
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")

print("ASSIGNMENT FOLDER CONTENTS\n")

for p in sorted(ROOT.rglob("*")):
    relative = p.relative_to(ROOT)
    if p.is_dir():
        print(f"[DIR]  {relative}")
    else:
        print(f"[FILE] {relative}  ({p.stat().st_size / 1024:.1f} KB)")

print("\nTOTAL FILES:", sum(1 for p in ROOT.rglob("*") if p.is_file()))

ASSIGNMENT FOLDER CONTENTS

[DIR]  English
[DIR]  English/results
[DIR]  English/results/predictions
[DIR]  English/results/test_generated_audios

TOTAL FILES: 0


In [24]:

# ================================================================
# ENGLISH FASTSPEECH 2 — INFERENCE + AUDIO + VARIANCE PLOTS
# Uses the trained BEST checkpoint
# Generates 5 audios + 15 variance adaptor plots
# ================================================================

import os
import sys
import json
import yaml
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

from pathlib import Path

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")
ENGLISH = ROOT / "English"

CHECKPOINT = ENGLISH / "checkpoints/best_checkpoint.pth.tar"
VOCAB_FILE = ENGLISH / "config/english_ipa_vocabulary.json"

PREPROCESS_FILE = ENGLISH / "config/preprocess.yaml"
MODEL_FILE = ENGLISH / "config/model.yaml"
TRAIN_FILE = ENGLISH / "config/train.yaml"

OUTPUT_DIR = ENGLISH / "results/test_generated_audios"
PREDICTION_DIR = ENGLISH / "results/predictions"
PLOT_ROOT = ENGLISH / "results/variance_adaptor_plots"

for folder in [
    OUTPUT_DIR,
    PREDICTION_DIR,
    PLOT_ROOT / "duration",
    PLOT_ROOT / "pitch",
    PLOT_ROOT / "energy"
]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------
# 2. DEVICE AND FILE VALIDATION
# ------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

required_files = [
    CHECKPOINT,
    VOCAB_FILE,
    PREPROCESS_FILE,
    MODEL_FILE,
    TRAIN_FILE
]

missing_files = [str(p) for p in required_files if not p.is_file()]

if missing_files:
    raise FileNotFoundError(
        "Required files are missing:\n" +
        "\n".join(missing_files) +
        "\n\nRestore these files before running inference."
    )

if not REPO.exists():
    raise FileNotFoundError(
        f"FastSpeech2 repository not found: {REPO}"
    )

# ------------------------------------------------
# 3. LOAD CONFIGURATIONS
# ------------------------------------------------

with open(PREPROCESS_FILE, "r") as f:
    preprocess_config = yaml.safe_load(f)

with open(MODEL_FILE, "r") as f:
    model_config = yaml.safe_load(f)

with open(TRAIN_FILE, "r") as f:
    train_config = yaml.safe_load(f)

# ------------------------------------------------
# 4. LOAD VOCABULARY
# ------------------------------------------------

with open(VOCAB_FILE, "r") as f:
    vocab_data = json.load(f)

phone_to_id = vocab_data["phone_to_id"]
id_to_phone = vocab_data["id_to_phone"]

vocab_size = len(phone_to_id)
model_config["input_vocabulary_size"] = vocab_size

assert "<pad>" in phone_to_id
assert "<unk>" in phone_to_id

print("Vocabulary size:", vocab_size)

# ------------------------------------------------
# 5. LOAD FASTSPEECH 2 MODEL
# ------------------------------------------------

sys.path.insert(0, str(REPO))
os.chdir(REPO)

from model import FastSpeech2

model = FastSpeech2(
    preprocess_config,
    model_config
).to(device)

checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=False
)

if not isinstance(checkpoint, dict):
    raise ValueError("Invalid checkpoint format.")

state = checkpoint.get(
    "model",
    checkpoint.get("state_dict", None)
)

if state is None:
    # Allow a raw state_dict checkpoint.
    if all(torch.is_tensor(v) for v in checkpoint.values()):
        state = checkpoint
    else:
        raise ValueError(
            "Could not locate model weights in checkpoint."
        )

model.load_state_dict(state, strict=True)
model.eval()

print("Best checkpoint loaded.")
print("Checkpoint step:", checkpoint.get("step", "Not recorded"))

# ------------------------------------------------
# 6. LOAD VOCODER
# ------------------------------------------------

from utils.model import get_vocoder

vocoder = get_vocoder(model_config, device)
vocoder.eval()

print("Vocoder loaded.")

# ------------------------------------------------
# 7. TEST SENTENCES
# ------------------------------------------------
# Enter the exact assignment sentences and IPA transcriptions.
# IPA must use the same phone inventory as the training vocabulary.
# Separate individual phones using spaces.

ENGLISH_TESTS = {
    "E1": {
        "text": "REPLACE WITH EXACT E1 SENTENCE",
        "ipa": "REPLACE WITH IPA PHONES"
    },
    "E2a": {
        "text": "REPLACE WITH EXACT E2(a) SENTENCE",
        "ipa": "REPLACE WITH IPA PHONES"
    },
    "E2b": {
        "text": "REPLACE WITH EXACT E2(b) SENTENCE",
        "ipa": "REPLACE WITH IPA PHONES"
    },
    "E3": {
        "text": "REPLACE WITH EXACT E3 SENTENCE",
        "ipa": "REPLACE WITH IPA PHONES"
    },
    "E4": {
        "text": "REPLACE WITH EXACT E4 SENTENCE",
        "ipa": "REPLACE WITH IPA PHONES"
    }
}

# ------------------------------------------------
# 8. IPA ENCODING
# ------------------------------------------------

def encode_ipa(ipa):

    phones = ipa.strip().split()

    if not phones:
        raise ValueError("Empty IPA transcription.")

    unknown = [
        phone for phone in phones
        if phone not in phone_to_id
    ]

    if unknown:
        raise ValueError(
            f"Unknown IPA phones: {unknown}"
        )

    return [phone_to_id[phone] for phone in phones]


# ------------------------------------------------
# 9. PLOT FUNCTION
# ------------------------------------------------

def save_variance_plot(values, title, ylabel, output_path):

    values = np.asarray(values).reshape(-1)

    plt.figure(figsize=(12, 4))

    plt.plot(
        np.arange(len(values)),
        values,
        linewidth=1.5
    )

    plt.title(title)
    plt.xlabel("Input phone position")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(output_path, dpi=200)
    plt.close()


# ------------------------------------------------
# 10. INFERENCE
# ------------------------------------------------

results = []

audio_config = preprocess_config["preprocessing"]["audio"]
sample_rate = audio_config["sampling_rate"]

with torch.no_grad():

    for test_id, item in ENGLISH_TESTS.items():

        text = item["text"].strip()
        ipa = item["ipa"].strip()

        if (
            not text or not ipa
            or "REPLACE" in text
            or "REPLACE" in ipa
        ):
            raise ValueError(
                f"Exact text and IPA are required for {test_id}."
            )

        token_ids = encode_ipa(ipa)
        input_length = len(token_ids)

        texts = torch.tensor(
            [token_ids],
            dtype=torch.long,
            device=device
        )

        src_lens = torch.tensor(
            [input_length],
            dtype=torch.long,
            device=device
        )

        speakers = torch.zeros(
            1,
            dtype=torch.long,
            device=device
        )

        # FastSpeech 2 inference
        predictions = model(
            speakers,
            texts,
            src_lens,
            input_length,
            p_control=1.0,
            e_control=1.0,
            d_control=1.0
        )

        if not isinstance(predictions, (tuple, list)):
            raise TypeError(
                "Unexpected FastSpeech2 output format."
            )

        if len(predictions) < 5:
            raise ValueError(
                "FastSpeech2 returned fewer than five outputs."
            )

        # Standard repository output:
        # mel, postnet_mel, log_duration, pitch, energy, ...
        postnet_mel = predictions[1]
        log_duration = predictions[2]
        pitch = predictions[3]
        energy = predictions[4]

        # Remove batch dimension
        mel = postnet_mel[0].detach()

        log_duration = (
            log_duration[0, :input_length]
            .detach().cpu().numpy()
        )

        pitch = (
            pitch[0, :input_length]
            .detach().cpu().numpy()
        )

        energy = (
            energy[0, :input_length]
            .detach().cpu().numpy()
        )

        # Convert predicted log durations to approximate frame counts.
        duration = np.maximum(
            np.round(np.exp(log_duration) - 1),
            0
        ).astype(np.int32)

        # ------------------------------------------------
        # VOCODER
        # ------------------------------------------------

        mel_for_vocoder = mel.transpose(0, 1).unsqueeze(0)

        waveform = vocoder(mel_for_vocoder)

        waveform = (
            waveform.squeeze()
            .detach()
            .cpu()
            .numpy()
        )

        if waveform.ndim != 1 or len(waveform) == 0:
            raise ValueError(
                f"Invalid waveform generated for {test_id}."
            )

        # ------------------------------------------------
        # SAVE AUDIO
        # ------------------------------------------------

        audio_path = OUTPUT_DIR / f"{test_id}.wav"

        sf.write(
            audio_path,
            waveform,
            sample_rate
        )

        # ------------------------------------------------
        # SAVE PREDICTIONS
        # ------------------------------------------------

        prediction_path = (
            PREDICTION_DIR /
            f"{test_id}_predictions.npz"
        )

        np.savez_compressed(
            prediction_path,
            duration=duration,
            pitch=pitch,
            energy=energy,
            mel=mel.cpu().numpy(),
            ipa=ipa,
            text=text
        )

        # ------------------------------------------------
        # SAVE THREE VARIANCE ADAPTOR PLOTS
        # ------------------------------------------------

        save_variance_plot(
            duration,
            f"{test_id} - Predicted Duration",
            "Duration (frames)",
            PLOT_ROOT / "duration" / f"{test_id}_duration.png"
        )

        save_variance_plot(
            pitch,
            f"{test_id} - Predicted Pitch",
            "Pitch (normalized)",
            PLOT_ROOT / "pitch" / f"{test_id}_pitch.png"
        )

        save_variance_plot(
            energy,
            f"{test_id} - Predicted Energy",
            "Energy (normalized)",
            PLOT_ROOT / "energy" / f"{test_id}_energy.png"
        )

        results.append({
            "Test": test_id,
            "Sentence": text,
            "IPA phones": input_length,
            "Mel frames": mel.shape[0],
            "Duration frames": int(duration.sum()),
            "Audio seconds": len(waveform) / sample_rate,
            "Audio file": str(audio_path)
        })

        print(
            f"{test_id} generated | "
            f"Phones: {input_length} | "
            f"Mel frames: {mel.shape[0]} | "
            f"Audio: {len(waveform)/sample_rate:.2f}s"
        )

# ------------------------------------------------
# 11. SAVE SUMMARY
# ------------------------------------------------

summary = pd.DataFrame(results)

summary_path = ENGLISH / "results/english_inference_summary.csv"

summary.to_csv(
    summary_path,
    index=False
)

print("\n==============================================")
print("ENGLISH INFERENCE FINISHED")
print("==============================================")

print(summary.to_string(index=False))

print("\nAudios:", OUTPUT_DIR)
print("Predictions:", PREDICTION_DIR)
print("Variance plots:", PLOT_ROOT)
print("Summary:", summary_path)


Device: cuda


FileNotFoundError: Required files are missing:
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/checkpoints/best_checkpoint.pth.tar
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/english_ipa_vocabulary.json
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/preprocess.yaml
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/model.yaml
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/config/train.yaml

Restore these files before running inference.

In [25]:
from pathlib import Path
import json

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")

folders = [
    "English/config",
    "English/checkpoints",
    "English/logs",
    "English/processed/train",
    "English/processed/validation",
    "English/results/test_generated_audios",
    "English/results/predictions",
    "English/results/variance_adaptor_plots/duration",
    "English/results/variance_adaptor_plots/pitch",
    "English/results/variance_adaptor_plots/energy",
    "Telugu/config",
    "Telugu/checkpoints",
    "Telugu/logs",
    "Telugu/results/test_generated_audios",
    "Telugu/results/variance_adaptor_plots/duration",
    "Telugu/results/variance_adaptor_plots/pitch",
    "Telugu/results/variance_adaptor_plots/energy",
    "Report"
]

for folder in folders:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

print("ASSIGNMENT RECOVERY REPORT")
print("=" * 55)

for language in ["English", "Telugu"]:
    base = ROOT / language
    files = [p for p in base.rglob("*") if p.is_file()]

    print(f"\n{language}: {len(files)} files")

    for p in files:
        print(f"  {p.relative_to(ROOT)} ({p.stat().st_size:,} bytes)")

    if not files:
        print("  No saved files found.")

print("\nFolder structure verified.")
print("Drive root:", ROOT)

ASSIGNMENT RECOVERY REPORT

English: 0 files
  No saved files found.

Telugu: 0 files
  No saved files found.

Folder structure verified.
Drive root: /content/gdrive/MyDrive/FastSpeech2_Assignment


In [27]:
# ================================================================
# CELL 2 — ENVIRONMENT SETUP (MOUNT-SAFE VERSION)
# ================================================================

from pathlib import Path
import subprocess
import sys
import torch
import os

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

# Check existing Drive mount; do not mount twice
if ROOT.exists():
    print("Google Drive is already accessible.")
else:
    from google.colab import drive
    drive.mount("/content/gdrive")
    ROOT.mkdir(parents=True, exist_ok=True)

# Clone repository only if missing
if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/ming024/FastSpeech2.git",
            str(REPO)
        ],
        check=True
    )
else:
    print("FastSpeech2 repository already exists.")

# Install additional dependencies only
packages = [
    "Unidecode",
    "pyworld",
    "praatio",
    "librosa==0.11.0",
    "soundfile",
    "pyyaml",
    "matplotlib",
    "pandas",
    "tqdm"
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    check=True
)

print("\nENVIRONMENT CHECK")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Drive root:", ROOT)
print("Repository:", REPO)
print("\nSETUP COMPLETE")

Google Drive is already accessible.

ENVIRONMENT CHECK
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
Drive root: /content/gdrive/MyDrive/FastSpeech2_Assignment
Repository: /content/FastSpeech2

SETUP COMPLETE


In [32]:
# =====================================================
# HUGGING FACE LOGIN — INTERACTIVE AUTHENTICATION
# =====================================================

!pip -q install huggingface_hub datasets

from huggingface_hub import notebook_login, whoami

# Opens a secure Hugging Face login/token prompt
notebook_login()

# Verify authentication
user = whoami()

print("Successfully authenticated!")
print("Hugging Face account:", user["name"])

# Test access to required dataset
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/Svarah",
    split="test"
)

print("Svarah dataset access verified!")
print("Total recordings:", len(dataset))
print("Columns:", dataset.column_names)

Successfully authenticated!
Hugging Face account: madhuri0625


data/test-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  365MB            

data/test-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  360MB            

data/test-00001-of-00003.parquet: downloading bytes:           |  0.00B            

data/test-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  370MB            

data/test-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/6656 [00:00<?, ? examples/s]

Svarah dataset access verified!
Total recordings: 6656
Columns: ['audio_filepath', 'duration', 'text', 'gender', 'age-group', 'primary_language', 'native_place_state', 'native_place_district', 'highest_qualification', 'job_category', 'occupation_domain']


In [34]:
!pip -q install huggingface_hub datasets

from huggingface_hub import notebook_login, whoami

print("Opening Hugging Face authentication...")
notebook_login()

user = whoami()

print("\nLogin successful!")
print("Account:", user["name"])

from datasets import load_dataset

print("\nChecking Svarah dataset access...")

dataset = load_dataset(
    "ai4bharat/Svarah",
    split="test"
)

print("\nSUCCESS!")
print("Dataset rows:", len(dataset))
print("Columns:", dataset.column_names)

Opening Hugging Face authentication...

Login successful!
Account: madhuri0625

Checking Svarah dataset access...

SUCCESS!
Dataset rows: 6656
Columns: ['audio_filepath', 'duration', 'text', 'gender', 'age-group', 'primary_language', 'native_place_state', 'native_place_district', 'highest_qualification', 'job_category', 'occupation_domain']


In [35]:
# ================================================================
# CELL 4 — ENGLISH SVARAH COMPLETE DATA PREPARATION
# ================================================================

import os, json, random, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = ROOT / "English"
DATA_ROOT = ENGLISH / "raw_audio"
META_ROOT = ENGLISH / "metadata"

for folder in [
    DATA_ROOT / "train",
    DATA_ROOT / "validation",
    META_ROOT
]:
    folder.mkdir(parents=True, exist_ok=True)

# Dataset already loaded in your previous cell.
if "dataset" not in globals():
    from datasets import load_dataset
    dataset = load_dataset("ai4bharat/Svarah", split="test")

assert len(dataset) == 6656
assert all(c in dataset.column_names for c in
           ["audio_filepath", "duration", "text"])

print("Dataset:", len(dataset), "recordings")

# ------------------------------------------------
# 1. SELECT APPROXIMATELY 5H TRAIN + 1H VALIDATION
# ------------------------------------------------

SEED = 42
TRAIN_TARGET = 5 * 3600
VAL_TARGET = 1 * 3600

rng = random.Random(SEED)
indices = list(range(len(dataset)))
rng.shuffle(indices)

durations = np.array(dataset["duration"], dtype=float)

train_indices = []
val_indices = []

train_seconds = 0.0
val_seconds = 0.0

for idx in indices:
    if train_seconds >= TRAIN_TARGET:
        break
    if train_seconds + durations[idx] <= TRAIN_TARGET + 30:
        train_indices.append(idx)
        train_seconds += durations[idx]

train_set = set(train_indices)

for idx in indices:
    if idx in train_set:
        continue
    if val_seconds >= VAL_TARGET:
        break
    if val_seconds + durations[idx] <= VAL_TARGET + 30:
        val_indices.append(idx)
        val_seconds += durations[idx]

assert not (set(train_indices) & set(val_indices))

print("\nSPLIT SUMMARY")
print("Train:", len(train_indices), "recordings,",
      round(train_seconds / 3600, 4), "hours")
print("Validation:", len(val_indices), "recordings,",
      round(val_seconds / 3600, 4), "hours")

# ------------------------------------------------
# 2. SAVE WAV AND TRANSCRIPTS
# ------------------------------------------------

def extract_audio(record):
    """
    Handles audio_filepath as either:
    - decoded audio dictionary
    - local path
    - bytes dictionary
    """
    audio = record["audio_filepath"]

    if isinstance(audio, dict):
        if audio.get("array") is not None:
            return np.asarray(audio["array"]), int(audio["sampling_rate"])

        if audio.get("bytes") is not None:
            import io
            waveform, sr = sf.read(io.BytesIO(audio["bytes"]))
            return waveform, sr

        if audio.get("path") is not None:
            return sf.read(audio["path"])

    if isinstance(audio, str):
        return sf.read(audio)

    raise TypeError(
        f"Unsupported audio format: {type(audio)}"
    )


def save_split(selected_indices, split_name):

    output_dir = DATA_ROOT / split_name
    records = []

    for idx in tqdm(selected_indices, desc=f"Saving {split_name}"):

        record = dataset[int(idx)]
        text = str(record["text"]).strip()

        if not text:
            continue

        stem = f"recording_{idx}"
        wav_path = output_dir / f"{stem}.wav"
        lab_path = output_dir / f"{stem}.lab"

        if not wav_path.exists() or wav_path.stat().st_size == 0:
            waveform, sr = extract_audio(record)
            waveform = np.asarray(waveform, dtype=np.float32)

            if waveform.ndim == 2:
                waveform = waveform.mean(axis=1)

            sf.write(wav_path, waveform, sr, subtype="PCM_16")

        lab_path.write_text(text, encoding="utf-8")

        info = sf.info(str(wav_path))

        records.append({
            "file": stem,
            "dataset_index": int(idx),
            "audio_path": str(wav_path),
            "transcript_path": str(lab_path),
            "text": text,
            "duration_seconds": float(info.duration),
            "sampling_rate": int(info.samplerate),
            "split": split_name
        })

    return pd.DataFrame(records)


train_df = save_split(train_indices, "train")
val_df = save_split(val_indices, "validation")

# ------------------------------------------------
# 3. SAVE METADATA
# ------------------------------------------------

train_csv = META_ROOT / "english_train.csv"
val_csv = META_ROOT / "english_validation.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)

pd.concat([train_df, val_df], ignore_index=True).to_csv(
    META_ROOT / "english_metadata.csv",
    index=False
)

report = {
    "dataset": "ai4bharat/Svarah",
    "dataset_split": "test",
    "seed": SEED,
    "selection": "shuffled recording-level duration-limited split",
    "train_indices": train_indices,
    "validation_indices": val_indices,
    "train_hours": float(train_df.duration_seconds.sum() / 3600),
    "validation_hours": float(val_df.duration_seconds.sum() / 3600),
    "train_count": len(train_df),
    "validation_count": len(val_df),
    "overlap_count": len(set(train_indices) & set(val_indices))
}

(META_ROOT / "english_split_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------
# 4. VERIFY SAVED FILES
# ------------------------------------------------

for split_name, df in [
    ("train", train_df),
    ("validation", val_df)
]:
    wavs = list((DATA_ROOT / split_name).glob("*.wav"))
    labs = list((DATA_ROOT / split_name).glob("*.lab"))

    assert len(wavs) == len(df), (
        f"{split_name}: WAV count mismatch"
    )
    assert len(labs) == len(df), (
        f"{split_name}: LAB count mismatch"
    )

    print(f"\n{split_name.upper()} VERIFIED")
    print("WAV files:", len(wavs))
    print("LAB files:", len(labs))
    print("Duration:", round(df.duration_seconds.sum()/3600, 4), "hours")

print("\n" + "=" * 55)
print("ENGLISH DATA PREPARATION COMPLETE")
print("=" * 55)
print("Saved to:", DATA_ROOT)
print("Metadata:", META_ROOT)
print("Next: English alignment and acoustic feature extraction")

Dataset: 6656 recordings

SPLIT SUMMARY
Train: 3449 recordings, 5.0 hours
Validation: 695 recordings, 1.0061 hours


Saving train:   0%|          | 0/3449 [00:00<?, ?it/s]

TypeError: Unsupported audio format: <class 'datasets.features._torchcodec.AudioDecoder'>

In [36]:
# ================================================================
# ENGLISH SVARAH — AUDIO EXTRACTION (TORCHCODEC FIX)
# Reuses existing dataset, train_indices and val_indices
# ================================================================

import numpy as np
import pandas as pd
import soundfile as sf
import json
import torch

from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
ENGLISH = ROOT / "English"
DATA_ROOT = ENGLISH / "raw_audio"
META_ROOT = ENGLISH / "metadata"

# Decode AudioDecoder objects
def extract_audio(record):

    audio = record["audio_filepath"]

    if hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        waveform = samples.data
        sr = int(samples.sample_rate)

        if isinstance(waveform, torch.Tensor):
            waveform = waveform.cpu().numpy()

        waveform = np.asarray(waveform)

        if waveform.ndim == 2:
            waveform = waveform.mean(axis=0)

        return waveform.astype(np.float32), sr

    raise TypeError(f"Unexpected audio type: {type(audio)}")


def save_split(indices, split_name):

    folder = DATA_ROOT / split_name
    folder.mkdir(parents=True, exist_ok=True)

    rows = []
    failures = []

    for idx in tqdm(indices, desc=split_name):

        try:
            record = dataset[int(idx)]
            text = str(record["text"]).strip()

            if not text:
                raise ValueError("Empty transcript")

            stem = f"recording_{idx}"
            wav = folder / f"{stem}.wav"
            lab = folder / f"{stem}.lab"

            if not wav.exists() or wav.stat().st_size < 1000:

                waveform, sr = extract_audio(record)

                if waveform.ndim != 1 or len(waveform) == 0:
                    raise ValueError("Invalid waveform")

                sf.write(wav, waveform, sr, subtype="PCM_16")

            info = sf.info(str(wav))
            lab.write_text(text, encoding="utf-8")

            rows.append({
                "file": stem,
                "dataset_index": int(idx),
                "audio_path": str(wav),
                "transcript_path": str(lab),
                "text": text,
                "sampling_rate": info.samplerate,
                "duration_seconds": info.duration,
                "split": split_name
            })

        except Exception as e:
            failures.append({
                "index": int(idx),
                "error": str(e)
            })

    return pd.DataFrame(rows), failures


assert "dataset" in globals()
assert "train_indices" in globals()
assert "val_indices" in globals()

print("Saving training recordings...")
train_df, train_errors = save_split(train_indices, "train")

print("\nSaving validation recordings...")
val_df, val_errors = save_split(val_indices, "validation")

META_ROOT.mkdir(parents=True, exist_ok=True)

train_df.to_csv(META_ROOT / "english_train.csv", index=False)
val_df.to_csv(META_ROOT / "english_validation.csv", index=False)

pd.concat([train_df, val_df]).to_csv(
    META_ROOT / "english_metadata.csv",
    index=False
)

report = {
    "dataset": "ai4bharat/Svarah",
    "train_saved": len(train_df),
    "validation_saved": len(val_df),
    "train_hours": float(train_df.duration_seconds.sum()/3600),
    "validation_hours": float(val_df.duration_seconds.sum()/3600),
    "train_errors": train_errors,
    "validation_errors": val_errors
}

with open(META_ROOT / "english_extraction_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\n========== EXTRACTION REPORT ==========")
print("Training saved:", len(train_df))
print("Training failures:", len(train_errors))
print("Validation saved:", len(val_df))
print("Validation failures:", len(val_errors))

print("Training hours:", round(train_df.duration_seconds.sum()/3600, 4))
print("Validation hours:", round(val_df.duration_seconds.sum()/3600, 4))

print("\nReport saved to:", META_ROOT)
print("EXTRACTION FINISHED")

Saving training recordings...


train:   0%|          | 0/3449 [00:00<?, ?it/s]


Saving validation recordings...


validation:   0%|          | 0/695 [00:00<?, ?it/s]


========== EXTRACTION REPORT ==========
Training saved: 3449
Training failures: 0
Validation saved: 695
Validation failures: 0
Training hours: 5.0
Validation hours: 1.0061

Report saved to: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/metadata
EXTRACTION FINISHED


In [37]:
# ================================================================
# ENGLISH FASTSPEECH 2 — FORCED ALIGNMENT (TRAIN + VALIDATION)
# RUN NOW in a NEW Colab cell.
# Does not download the dataset or re-extract audio.
# ================================================================

import os, sys, glob, json, csv, subprocess, shutil, time
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")
TRAIN_DIR = ROOT / "raw_audio" / "train"
VAL_DIR   = ROOT / "raw_audio" / "validation"
META_DIR  = ROOT / "metadata"
ALIGN_DIR = ROOT / "alignments"
TRAIN_ALIGN = ALIGN_DIR / "train"
VAL_ALIGN   = ALIGN_DIR / "validation"

for p in [TRAIN_DIR, VAL_DIR, META_DIR, TRAIN_ALIGN, VAL_ALIGN]:
    p.mkdir(parents=True, exist_ok=True)

print("1. CHECKING SAVED AUDIO")
train_audio = list(TRAIN_DIR.rglob("*.wav"))
val_audio   = list(VAL_DIR.rglob("*.wav"))

print("Training WAV files:", len(train_audio))
print("Validation WAV files:", len(val_audio))

if not train_audio or not val_audio:
    raise RuntimeError(
        "WAV files were not found in the expected folders. "
        "Check English/raw_audio/train and English/raw_audio/validation."
    )

print("\n2. CHECKING METADATA")
metadata_files = (
    list(META_DIR.glob("*.csv")) +
    list(META_DIR.glob("*.json")) +
    list(META_DIR.glob("*.jsonl"))
)
print("Metadata files:")
for f in metadata_files:
    print(" -", f.name)

if not metadata_files:
    raise RuntimeError("No metadata CSV/JSON files found in English/metadata.")

# Load metadata tables
tables = []
for f in metadata_files:
    try:
        if f.suffix == ".csv":
            df = pd.read_csv(f)
        elif f.suffix == ".jsonl":
            df = pd.read_json(f, lines=True)
        else:
            df = pd.read_json(f)
        if len(df):
            print(f"\n{f.name}: {len(df)} rows")
            print("Columns:", list(df.columns))
            tables.append((f, df))
    except Exception as e:
        print("Could not read", f.name, ":", str(e))

# Build audio-stem -> transcript mapping from metadata
transcript_map = {}

def find_text_column(columns):
    candidates = ["text", "transcript", "sentence", "normalized_text", "utterance"]
    lower = {str(c).lower(): c for c in columns}
    for c in candidates:
        if c in lower:
            return lower[c]
    return None

def find_audio_column(columns):
    candidates = [
        "filename", "file_name", "audio", "audio_filepath",
        "audio_path", "path", "wav", "wav_path", "recording"
    ]
    lower = {str(c).lower(): c for c in columns}
    for c in candidates:
        if c in lower:
            return lower[c]
    return None

for source, df in tables:
    text_col = find_text_column(df.columns)
    audio_col = find_audio_column(df.columns)

    if text_col is None:
        continue

    # If a table contains only transcript rows, use its index/stem when possible.
    if audio_col is None:
        print(f"Skipping automatic mapping from {source.name}: no audio filename column.")
        continue

    for _, row in df.iterrows():
        audio_value = str(row[audio_col])
        text_value = str(row[text_col]).strip()

        if not text_value or text_value.lower() == "nan":
            continue

        stem = Path(audio_value).stem
        transcript_map[stem] = text_value

print("\nTranscript mappings found:", len(transcript_map))

# Match audio files to transcripts. Also check common metadata sidecar formats.
def get_transcript(wav_path):
    stem = wav_path.stem

    if stem in transcript_map:
        return transcript_map[stem]

    # Search common sidecar transcript files.
    for suffix in [".lab", ".txt", ".text"]:
        sidecar = wav_path.with_suffix(suffix)
        if sidecar.exists():
            return sidecar.read_text(encoding="utf-8").strip()

    return None

# Prepare MFA corpus folders with audio + .lab pairs.
# Keep original WAVs untouched.
MFA_TRAIN = ROOT / "mfa_corpus" / "train"
MFA_VAL   = ROOT / "mfa_corpus" / "validation"

for folder in [MFA_TRAIN, MFA_VAL]:
    folder.mkdir(parents=True, exist_ok=True)

def prepare_mfa_corpus(audio_files, destination, split_name):
    copied, missing = 0, []
    for wav in tqdm(audio_files, desc=f"Preparing {split_name}"):
        transcript = get_transcript(wav)

        if not transcript:
            missing.append(wav.name)
            continue

        target_wav = destination / wav.name
        target_lab = destination / (wav.stem + ".lab")

        if wav.resolve() != target_wav.resolve():
            if not target_wav.exists():
                shutil.copy2(wav, target_wav)

        target_lab.write_text(transcript, encoding="utf-8")
        copied += 1

    print(f"{split_name}: prepared={copied}, missing_transcripts={len(missing)}")
    if missing[:10]:
        print("Examples missing transcript:", missing[:10])
    return copied, missing

train_prepared, train_missing = prepare_mfa_corpus(
    train_audio, MFA_TRAIN, "Training"
)
val_prepared, val_missing = prepare_mfa_corpus(
    val_audio, MFA_VAL, "Validation"
)

if train_prepared == 0 or val_prepared == 0:
    raise RuntimeError(
        "Could not pair audio with transcripts. "
        "The metadata filename column may need mapping to the saved WAV filenames. "
        "Check the printed metadata columns and sample filenames."
    )

print("\n3. CHECKING MONTREAL FORCED ALIGNER")
mfa_path = shutil.which("mfa")

if mfa_path is None:
    print("MFA command not found.")
    print(
        "\nThis Colab runtime uses Python 3.13. Installing MFA directly with pip "
        "may break the existing PyTorch/NumPy environment. Stopping safely rather "
        "than changing your working environment."
    )
    print("\nPrepared corpus folders:")
    print("Training:", MFA_TRAIN)
    print("Validation:", MFA_VAL)
    raise RuntimeError(
        "MFA is not installed in this runtime. Audio/transcript pairing is prepared; "
        "we need a compatible MFA environment before running alignment."
    )

print("MFA executable:", mfa_path)
subprocess.run(["mfa", "version"], check=False)

print("\n4. CHECKING ENGLISH MFA MODELS")
model_list = subprocess.run(
    ["mfa", "model", "list"],
    capture_output=True, text=True
)
print((model_list.stdout or model_list.stderr)[:5000])

print("\n5. DOWNLOADING ENGLISH MODELS IF NEEDED")
# Standard MFA English US ARPA models.
# These are ARPAbet models, not the IPA-phone alignment used in the
# earlier English experiment. We do not silently substitute them.
raise RuntimeError(
    "Alignment has not been launched. The standard MFA English US model "
    "uses ARPAbet phones, while the earlier experiment used an IPA phone inventory. "
    "We must confirm/use the same phone representation before generating labels."
)

1. CHECKING SAVED AUDIO
Training WAV files: 3449
Validation WAV files: 695

2. CHECKING METADATA
Metadata files:
 - english_validation.csv
 - english_metadata.csv
 - english_train.csv
 - english_extraction_report.json

english_validation.csv: 695 rows
Columns: ['file', 'dataset_index', 'audio_path', 'transcript_path', 'text', 'sampling_rate', 'duration_seconds', 'split']

english_metadata.csv: 4144 rows
Columns: ['file', 'dataset_index', 'audio_path', 'transcript_path', 'text', 'sampling_rate', 'duration_seconds', 'split']

english_train.csv: 3449 rows
Columns: ['file', 'dataset_index', 'audio_path', 'transcript_path', 'text', 'sampling_rate', 'duration_seconds', 'split']

Transcript mappings found: 4144


Preparing Training:   0%|          | 0/3449 [00:00<?, ?it/s]

Training: prepared=3449, missing_transcripts=0


Preparing Validation:   0%|          | 0/695 [00:00<?, ?it/s]

Validation: prepared=695, missing_transcripts=0

3. CHECKING MONTREAL FORCED ALIGNER
MFA command not found.

This Colab runtime uses Python 3.13. Installing MFA directly with pip may break the existing PyTorch/NumPy environment. Stopping safely rather than changing your working environment.

Prepared corpus folders:
Training: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/mfa_corpus/train
Validation: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/mfa_corpus/validation


RuntimeError: MFA is not installed in this runtime. Audio/transcript pairing is prepared; we need a compatible MFA environment before running alignment.

In [ ]:
# ================================================================
# ENGLISH FASTSPEECH 2 — MFA INSTALLATION + FORCED ALIGNMENT
# RUN NOW in a NEW CELL. Do not rerun extraction or corpus preparation.
# ================================================================

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

TRAIN_CORPUS = ROOT / "mfa_corpus" / "train"
VAL_CORPUS   = ROOT / "mfa_corpus" / "validation"

ALIGN_ROOT = ROOT / "alignments"
TRAIN_OUT  = ALIGN_ROOT / "train"
VAL_OUT    = ALIGN_ROOT / "validation"

MFA_ENV = Path("/content/mfa_env")
MICROMAMBA = Path("/content/bin/micromamba")

for p in [TRAIN_OUT, VAL_OUT]:
    p.mkdir(parents=True, exist_ok=True)

# ---------- 1. VERIFY PREPARED CORPORA ----------

def count_pairs(folder):
    wavs = list(folder.glob("*.wav"))
    labs = list(folder.glob("*.lab"))
    wav_stems = {p.stem for p in wavs}
    lab_stems = {p.stem for p in labs}
    return wavs, labs, wav_stems & lab_stems, wav_stems - lab_stems

train_wavs, train_labs, train_pairs, train_unpaired = count_pairs(TRAIN_CORPUS)
val_wavs, val_labs, val_pairs, val_unpaired = count_pairs(VAL_CORPUS)

print("TRAIN WAV:", len(train_wavs))
print("TRAIN LAB:", len(train_labs))
print("TRAIN PAIRED:", len(train_pairs))
print("VALIDATION WAV:", len(val_wavs))
print("VALIDATION LAB:", len(val_labs))
print("VALIDATION PAIRED:", len(val_pairs))

if not train_pairs or not val_pairs:
    raise RuntimeError("Prepared MFA corpus is missing audio/transcript pairs.")

if train_unpaired or val_unpaired:
    raise RuntimeError(
        "Some audio files do not have matching .lab transcripts. "
        "Resolve those pairs before alignment."
    )

# ---------- 2. INSTALL MICROMAMBA ----------

if not MICROMAMBA.exists():
    print("\nInstalling micromamba...")
    subprocess.run(
        "mkdir -p /content/bin && "
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xvj -C /content/bin --strip-components=1 bin/micromamba",
        shell=True,
        check=True
    )

print("\nMicromamba:")
subprocess.run([str(MICROMAMBA), "--version"], check=True)

# ---------- 3. CREATE ISOLATED PYTHON 3.11 MFA ENVIRONMENT ----------

mfa_executable = MFA_ENV / "bin" / "mfa"

if not mfa_executable.exists():
    print("\nCreating isolated MFA environment.")
    print("This may take several minutes.")

    subprocess.run(
        [
            str(MICROMAMBA), "create",
            "-y",
            "-p", str(MFA_ENV),
            "-c", "conda-forge",
            "python=3.11",
            "montreal-forced-aligner"
        ],
        check=True
    )
else:
    print("\nExisting MFA environment found.")

def mfa_run(args, capture=False):
    command = [
        str(MICROMAMBA), "run",
        "-p", str(MFA_ENV),
        "mfa"
    ] + args

    print("\nCOMMAND:", " ".join(command))

    return subprocess.run(
        command,
        check=True,
        text=True,
        capture_output=capture
    )

# ---------- 4. VERIFY MFA ----------

mfa_run(["version"])

# ---------- 5. DOWNLOAD ENGLISH MODELS ----------

print("\nDownloading English acoustic model...")
mfa_run(["model", "download", "acoustic", "english_us_arpa"])

print("\nDownloading English pronunciation dictionary...")
mfa_run(["model", "download", "dictionary", "english_us_arpa"])

# ---------- 6. ALIGN TRAINING CORPUS ----------

print("\n" + "=" * 65)
print("ALIGNING ENGLISH TRAINING DATA")
print("=" * 65)

start_train = time.time()

mfa_run([
    "align",
    str(TRAIN_CORPUS),
    "english_us_arpa",
    "english_us_arpa",
    str(TRAIN_OUT),
    "--clean",
    "--overwrite",
    "--num_jobs", "2"
])

train_elapsed = time.time() - start_train

# ---------- 7. ALIGN VALIDATION CORPUS ----------

print("\n" + "=" * 65)
print("ALIGNING ENGLISH VALIDATION DATA")
print("=" * 65)

start_val = time.time()

mfa_run([
    "align",
    str(VAL_CORPUS),
    "english_us_arpa",
    "english_us_arpa",
    str(VAL_OUT),
    "--clean",
    "--overwrite",
    "--num_jobs", "2"
])

val_elapsed = time.time() - start_val

# ---------- 8. VERIFY OUTPUTS ----------

train_textgrids = list(TRAIN_OUT.rglob("*.TextGrid")) + list(TRAIN_OUT.rglob("*.textgrid"))
val_textgrids = list(VAL_OUT.rglob("*.TextGrid")) + list(VAL_OUT.rglob("*.textgrid"))

report = {
    "aligner": "Montreal Forced Aligner",
    "python_environment": "Isolated Python 3.11",
    "acoustic_model": "english_us_arpa",
    "dictionary": "english_us_arpa",
    "phone_representation": "ARPAbet",
    "training_audio_count": len(train_wavs),
    "training_paired_count": len(train_pairs),
    "training_textgrid_count": len(train_textgrids),
    "training_alignment_seconds": round(train_elapsed, 2),
    "validation_audio_count": len(val_wavs),
    "validation_paired_count": len(val_pairs),
    "validation_textgrid_count": len(val_textgrids),
    "validation_alignment_seconds": round(val_elapsed, 2)
}

report_path = ALIGN_ROOT / "english_alignment_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n" + "=" * 65)
print("ENGLISH ALIGNMENT REPORT")
print("=" * 65)

for key, value in report.items():
    print(f"{key}: {value}")

print("\nReport:", report_path)
print("Training alignments:", TRAIN_OUT)
print("Validation alignments:", VAL_OUT)

if len(train_textgrids) == 0 or len(val_textgrids) == 0:
    raise RuntimeError(
        "Alignment command finished but TextGrid files were not found. "
        "Inspect the MFA output before continuing."
    )

print("\nALIGNMENT STAGE FINISHED.")

TRAIN WAV: 3449
TRAIN LAB: 3449
TRAIN PAIRED: 3449
VALIDATION WAV: 695
VALIDATION LAB: 695
VALIDATION PAIRED: 695

Installing micromamba...

Micromamba:

Creating isolated MFA environment.
This may take several minutes.

COMMAND: /content/bin/micromamba run -p /content/mfa_env mfa version


COMMAND: /content/bin/micromamba run -p /content/mfa_env mfa model download acoustic english_us_arpa


COMMAND: /content/bin/micromamba run -p /content/mfa_env mfa model download dictionary english_us_arpa

ALIGNING ENGLISH TRAINING DATA

COMMAND: /content/bin/micromamba run -p /content/mfa_env mfa align /content/gdrive/MyDrive/FastSpeech2_Assignment/English/mfa_corpus/train english_us_arpa english_us_arpa /content/gdrive/MyDrive/FastSpeech2_Assignment/English/alignments/train --clean --overwrite --num_jobs 2

ALIGNING ENGLISH VALIDATION DATA

COMMAND: /content/bin/micromamba run -p /content/mfa_env mfa align /content/gdrive/MyDrive/FastSpeech2_Assignment/English/mfa_corpus/validation english_us_arp

In [1]:
# ================================================================
# MFA ALIGNMENT RECOVERY CHECK — RUN NOW
# Checks saved TextGrids and whether either split completed.
# Does NOT rerun alignment or reinstall anything.
# ================================================================

from pathlib import Path
import json, os, glob

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")
ALIGN = ROOT / "alignments"

def inspect_split(name):
    folder = ALIGN / name
    grids = (
        list(folder.rglob("*.TextGrid")) +
        list(folder.rglob("*.textgrid"))
    ) if folder.exists() else []

    wav_folder = ROOT / "mfa_corpus" / name
    wavs = list(wav_folder.glob("*.wav")) if wav_folder.exists() else []

    print(f"\n{name.upper()}")
    print("-" * 45)
    print("Input WAV files:", len(wavs))
    print("Saved TextGrid files:", len(grids))
    print("Output folder exists:", folder.exists())

    if grids:
        print("Sample outputs:")
        for f in grids[:5]:
            print(" ", f.name, f.stat().st_size, "bytes")

    return len(grids), len(wavs)

train_grids, train_wavs = inspect_split("train")
val_grids, val_wavs = inspect_split("validation")

print("\n" + "=" * 45)
print("MFA ENVIRONMENT")
print("=" * 45)

mfa = Path("/content/mfa_env/bin/mfa")
print("MFA executable exists:", mfa.exists())

print("\n" + "=" * 45)
print("ALIGNMENT STATUS")
print("=" * 45)

if train_grids == train_wavs and val_grids == val_wavs:
    print("Both splits appear complete.")
elif train_grids > 0 or val_grids > 0:
    print("Partial alignment found. We can resume only the incomplete split.")
else:
    print("No TextGrid outputs found. Alignment needs to be rerun.")

report = {
    "training_wavs": train_wavs,
    "training_textgrids": train_grids,
    "validation_wavs": val_wavs,
    "validation_textgrids": val_grids,
    "mfa_environment_exists": mfa.exists()
}

print("\n", json.dumps(report, indent=2))


TRAIN
---------------------------------------------
Input WAV files: 0
Saved TextGrid files: 0
Output folder exists: False

VALIDATION
---------------------------------------------
Input WAV files: 0
Saved TextGrid files: 0
Output folder exists: False

MFA ENVIRONMENT
MFA executable exists: False

ALIGNMENT STATUS
Both splits appear complete.

 {
  "training_wavs": 0,
  "training_textgrids": 0,
  "validation_wavs": 0,
  "validation_textgrids": 0,
  "mfa_environment_exists": false
}


In [2]:
# ================================================================
# FASTSPEECH 2 — CHECK DRIVE DATA AFTER COLAB INTERRUPTION
# RUN NOW. READ-ONLY: DOES NOT DELETE OR MODIFY ANYTHING.
# ================================================================

from pathlib import Path
import os

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

print("GOOGLE DRIVE")
print("=" * 55)
print("Drive mount exists:", Path("/content/gdrive/MyDrive").exists())
print("Assignment folder exists:", BASE.exists())

folders = [
    "raw_audio/train",
    "raw_audio/validation",
    "metadata",
    "mfa_corpus/train",
    "mfa_corpus/validation",
    "alignments/train",
    "alignments/validation",
]

print("\nFOLDER CHECK")
print("=" * 55)

for folder in folders:
    p = BASE / folder
    print(f"\n{folder}")
    print("Exists:", p.exists())

    if p.exists():
        files = list(p.iterdir())
        print("Immediate files:", len(files))
        print("Examples:", [x.name for x in files[:5]])

        if "raw_audio" in folder or "mfa_corpus" in folder:
            print("WAV files:", len(list(p.rglob("*.wav"))))
            print("LAB files:", len(list(p.rglob("*.lab"))))

print("\nLOCAL MFA ENVIRONMENT")
print("=" * 55)
print("MFA environment exists:", Path("/content/mfa_env").exists())
print("MFA executable exists:", Path("/content/mfa_env/bin/mfa").exists())

print("\nCHECK COMPLETE — NO FILES MODIFIED.")

GOOGLE DRIVE
Drive mount exists: False
Assignment folder exists: False

FOLDER CHECK

raw_audio/train
Exists: False

raw_audio/validation
Exists: False

metadata
Exists: False

mfa_corpus/train
Exists: False

mfa_corpus/validation
Exists: False

alignments/train
Exists: False

alignments/validation
Exists: False

LOCAL MFA ENVIRONMENT
MFA environment exists: False
MFA executable exists: False

CHECK COMPLETE — NO FILES MODIFIED.


In [4]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

from pathlib import Path

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

print("\nDrive mounted:", Path("/content/gdrive/MyDrive").exists())
print("Assignment folder exists:", BASE.exists())

if BASE.exists():
    for folder in ["raw_audio/train", "raw_audio/validation", "metadata"]:
        p = BASE / folder
        print(f"{folder}:",
              len(list(p.rglob("*"))) if p.exists() else "NOT FOUND")
else:
    print("\nAssignment folder was not found at the expected location.")
    print("Do not recreate or overwrite it yet.")

Mounted at /content/gdrive

Drive mounted: True
Assignment folder exists: True
raw_audio/train: 5328
raw_audio/validation: 1010
metadata: NOT FOUND


In [5]:
# ================================================================
# FASTSPEECH 2 — DRIVE RECOVERY AND FILE INVENTORY
# RUN NOW. READ-ONLY. DOES NOT MODIFY OR DELETE FILES.
# ================================================================

from pathlib import Path
from collections import Counter

BASE = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

print("=" * 65)
print("ENGLISH PROJECT FOLDER")
print("=" * 65)

for item in sorted(BASE.iterdir()):
    print(("DIR  " if item.is_dir() else "FILE "), item.name)

print("\n" + "=" * 65)
print("AUDIO FOLDER INVENTORY")
print("=" * 65)

for name in ["raw_audio/train", "raw_audio/validation"]:
    folder = BASE / name
    files = [p for p in folder.rglob("*") if p.is_file()]
    extensions = Counter(p.suffix.lower() for p in files)

    print(f"\n{name}")
    print("Total files:", len(files))
    print("File types:", dict(extensions))
    print("WAV count:", sum(1 for p in files if p.suffix.lower() == ".wav"))
    print("LAB count:", sum(1 for p in files if p.suffix.lower() == ".lab"))
    print("Examples:", [p.name for p in files[:8]])

print("\n" + "=" * 65)
print("SEARCHING FOR METADATA AND ALIGNMENTS")
print("=" * 65)

patterns = ["*.csv", "*.json", "*.jsonl", "*.TextGrid",
            "*.textgrid", "*.yaml", "*.yml"]

for pattern in patterns:
    matches = list(BASE.rglob(pattern))
    print(f"\n{pattern}: {len(matches)}")
    for p in matches[:8]:
        print(" ", p.relative_to(BASE))

print("\nINVENTORY COMPLETE — NO FILES MODIFIED.")

ENGLISH PROJECT FOLDER
DIR   alignment
DIR   alignment_pilot
DIR   checkpoints
DIR   config
FILE  english_dataset_readiness_audit.json
FILE  english_full_audit.json
FILE  english_mfa_environment_audit.json
DIR   logs
FILE  mfa_acoustic_model_catalogue.txt
FILE  mfa_g2p_model_catalogue.txt
DIR   processed
DIR   raw_audio
DIR   raw_data
DIR   results
DIR   sample_audio_test

AUDIO FOLDER INVENTORY

raw_audio/train
Total files: 5328
File types: {'.wav': 2664, '.lab': 2664}
WAV count: 2664
LAB count: 2664
Examples: ['recording_5432.wav', 'recording_5432.lab', 'recording_5438.wav', 'recording_5438.lab', 'recording_5441.wav', 'recording_5441.lab', 'recording_5446.wav', 'recording_5446.lab']

raw_audio/validation
Total files: 1010
File types: {'.wav': 505, '.lab': 505}
WAV count: 505
LAB count: 505
Examples: ['recording_55.wav', 'recording_55.lab', 'recording_62.wav', 'recording_62.lab', 'recording_91.wav', 'recording_91.lab', 'recording_110.wav', 'recording_110.lab']

SEARCHING FOR METADATA 

In [6]:
# ================================================================
# ENGLISH FASTSPEECH 2 — FULL PROJECT RECOVERY AUDIT
# RUN NOW — READ ONLY. NO INSTALLATION, DELETION OR OVERWRITING.
# ================================================================

from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/English"
)

def show_files(folder, extensions=None, limit=12):
    p = ROOT / folder
    print(f"\n[{folder}]")
    if not p.exists():
        print("NOT FOUND")
        return
    files = [x for x in p.rglob("*") if x.is_file()]
    if extensions:
        files = [x for x in files if x.suffix.lower() in extensions]
    print("File count:", len(files))
    for f in files[:limit]:
        print(f"  {f.relative_to(ROOT)} ({f.stat().st_size:,} bytes)")

# 1. Existing alignments
print("\n" + "="*65)
print("1. EXISTING ALIGNMENT INVENTORY")
print("="*65)

for folder in ["alignment", "alignment_pilot"]:
    p = ROOT / folder
    grids = (
        list(p.rglob("*.TextGrid")) +
        list(p.rglob("*.textgrid"))
    ) if p.exists() else []
    print(f"\n{folder}: {len(grids)} TextGrid files")
    for f in grids[:5]:
        print(" ", f.relative_to(ROOT))

# 2. Processed features
print("\n" + "="*65)
print("2. PROCESSED FEATURES")
print("="*65)

for folder in ["processed/train", "processed/validation"]:
    show_files(folder, {".npz", ".npy", ".csv", ".json"})
    p = ROOT / folder
    if p.exists():
        for ext in [".npz", ".npy"]:
            print(f"{ext} total:", len(list(p.rglob("*" + ext))))

# 3. Checkpoints
print("\n" + "="*65)
print("3. CHECKPOINTS")
print("="*65)

show_files("checkpoints", {".pth", ".pt", ".ckpt", ".tar"}, limit=30)

# 4. Configurations and logs
print("\n" + "="*65)
print("4. CONFIGURATIONS AND LOGS")
print("="*65)

show_files("config", {".yaml", ".yml", ".json", ".txt"})
show_files("logs", {".csv", ".json", ".txt", ".log"})

# 5. Audit reports
print("\n" + "="*65)
print("5. EXISTING AUDIT REPORTS")
print("="*65)

audit_names = [
    "english_dataset_readiness_audit.json",
    "english_full_audit.json",
    "english_mfa_environment_audit.json",
    "processed/preprocessing_manifest.json",
    "processed/stats.json",
    "config/english_ipa_vocabulary.json"
]

for name in audit_names:
    p = ROOT / name
    print(f"\n--- {name} ---")
    if p.exists():
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])
        except Exception as e:
            print("Could not parse:", e)
    else:
        print("NOT FOUND")

# 6. Existing metadata summaries
print("\n" + "="*65)
print("6. PROCESSED METADATA")
print("="*65)

for name in [
    "processed/train/metadata.csv",
    "processed/validation/metadata.csv",
    "logs/english_training_history.csv"
]:
    p = ROOT / name
    print(f"\n{name}")
    if p.exists():
        df = pd.read_csv(p)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print(df.head(3).to_string(index=False))
    else:
        print("NOT FOUND")

print("\n" + "="*65)
print("RECOVERY AUDIT FINISHED — NO FILES MODIFIED")
print("="*65)


1. EXISTING ALIGNMENT INVENTORY

alignment: 3178 TextGrid files
  alignment/retry_17/output/recording_6486.TextGrid
  alignment/retry_17/output/recording_1558.TextGrid
  alignment/retry_17/output/recording_3983.TextGrid
  alignment/retry_17/output/recording_4707.TextGrid
  alignment/retry_17/output/recording_5283.TextGrid

alignment_pilot: 10 TextGrid files
  alignment_pilot/output/recording_1010.TextGrid
  alignment_pilot/output/recording_101.TextGrid
  alignment_pilot/output/recording_0.TextGrid
  alignment_pilot/output/recording_10.TextGrid
  alignment_pilot/output/recording_1008.TextGrid

2. PROCESSED FEATURES

[processed/train]
File count: 2664
  processed/train/recording_4757.npz (311,099 bytes)
  processed/train/recording_4761.npz (109,235 bytes)
  processed/train/recording_4762.npz (207,316 bytes)
  processed/train/recording_4763.npz (154,940 bytes)
  processed/train/recording_4767.npz (101,084 bytes)
  processed/train/recording_4768.npz (67,422 bytes)
  processed/train/record

In [7]:
# ================================================================
# ENGLISH FASTSPEECH 2 — CHECKPOINT + ALIGNMENT RECOVERY AUDIT
# RUN NOW IN A NEW CELL
# READ ONLY: DOES NOT MODIFY OR DELETE PROJECT FILES
# ================================================================

from pathlib import Path
import json, re, os
import numpy as np
import pandas as pd

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

# ---------- 1. ALIGNMENT INVENTORY ----------

print("\n" + "="*65)
print("1. ALIGNMENT DIRECTORY BREAKDOWN")
print("="*65)

ALIGN = ROOT / "alignment"

for folder in sorted(ALIGN.iterdir()):
    if folder.is_dir():
        grids = list(folder.rglob("*.TextGrid"))
        print(f"{folder.relative_to(ROOT)}: {len(grids)} TextGrid files")

# ---------- 2. FEATURE ID COVERAGE ----------

print("\n" + "="*65)
print("2. FEATURE ID COVERAGE")
print("="*65)

def load_ids(csv_path):
    df = pd.read_csv(csv_path)
    return set(df["id"].astype(str))

train_ids = load_ids(ROOT / "processed/train/metadata.csv")
val_ids = load_ids(ROOT / "processed/validation/metadata.csv")

def grid_ids(folder):
    return {
        p.stem for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() == ".textgrid"
    }

all_grid_ids = grid_ids(ALIGN)

train_aligned = train_ids & all_grid_ids
val_aligned = val_ids & all_grid_ids

print("Training feature IDs:", len(train_ids))
print("Validation feature IDs:", len(val_ids))
print("Training IDs with alignment:", len(train_aligned))
print("Validation IDs with alignment:", len(val_aligned))
print("Training IDs missing alignment:", len(train_ids - all_grid_ids))
print("Validation IDs missing alignment:", len(val_ids - all_grid_ids))

print("\nExamples of missing training alignments:",
      sorted(train_ids - all_grid_ids)[:10])
print("Examples of missing validation alignments:",
      sorted(val_ids - all_grid_ids)[:10])

# ---------- 3. CHECK FEATURE FILE INTEGRITY ----------

print("\n" + "="*65)
print("3. FEATURE FILE INTEGRITY")
print("="*65)

def inspect_npz(folder, expected_ids):
    good, bad = 0, []
    shapes = {}

    for item in expected_ids:
        path = folder / f"{item}.npz"
        if not path.exists():
            bad.append((item, "file missing"))
            continue

        try:
            with np.load(path, allow_pickle=True) as data:
                keys = list(data.keys())
                shapes[item] = {
                    k: data[k].shape for k in keys
                }
                good += 1
        except Exception as e:
            bad.append((item, str(e)))

    print("Valid:", good)
    print("Invalid/missing:", len(bad))
    if shapes:
        print("Example feature shapes:", shapes[next(iter(shapes))])
    if bad[:10]:
        print("Errors:", bad[:10])

    return good, bad

train_good, train_bad = inspect_npz(
    ROOT / "processed/train", train_ids
)
val_good, val_bad = inspect_npz(
    ROOT / "processed/validation", val_ids
)

# ---------- 4. CHECK CHECKPOINT METADATA ----------

print("\n" + "="*65)
print("4. CHECKPOINT FILES")
print("="*65)

checkpoint_dir = ROOT / "checkpoints"

for name in ["best_checkpoint.pth.tar", "latest_checkpoint.pth.tar"]:
    path = checkpoint_dir / name
    print(f"\n{name}")
    print("Exists:", path.exists())

    if path.exists():
        print("Size MB:", round(path.stat().st_size / (1024**2), 2))

# Load checkpoint metadata without changing anything.
try:
    import torch

    for name in ["best_checkpoint.pth.tar", "latest_checkpoint.pth.tar"]:
        path = checkpoint_dir / name
        if not path.exists():
            continue

        print(f"\nInspecting {name}")
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=False
        )

        if isinstance(checkpoint, dict):
            print("Checkpoint keys:", list(checkpoint.keys()))

            for key in [
                "step", "global_step", "epoch",
                "best_val_loss", "validation_loss"
            ]:
                if key in checkpoint:
                    print(key, ":", checkpoint[key])

            for key in ["model", "model_state_dict", "optimizer"]:
                if key in checkpoint:
                    value = checkpoint[key]
                    if isinstance(value, dict):
                        print(key, "entries:", len(value))

        del checkpoint

except Exception as e:
    print("Checkpoint inspection error:", repr(e))

# ---------- 5. TRAINING HISTORY ----------

print("\n" + "="*65)
print("5. TRAINING HISTORY")
print("="*65)

history_path = ROOT / "logs/english_training_history.csv"

if history_path.exists():
    history = pd.read_csv(history_path)

    print("Logged rows:", len(history))
    print("First step:", history["step"].min())
    print("Last step:", history["step"].max())

    if "validation_total" in history.columns:
        valid = history.dropna(subset=["validation_total"])

        if len(valid):
            best = valid.loc[valid["validation_total"].idxmin()]
            print("\nBest validation row:")
            print(best.to_string())

    print("\nLast five training rows:")
    print(history.tail().to_string(index=False))

# ---------- 6. MODEL CONFIGURATION ----------

print("\n" + "="*65)
print("6. MODEL CONFIGURATION")
print("="*65)

for name in ["model.yaml", "train.yaml", "preprocess.yaml"]:
    path = ROOT / "config" / name
    print(f"\n--- {name} ---")

    if path.exists():
        print(path.read_text(encoding="utf-8"))

# ---------- 7. FINAL SUMMARY ----------

print("\n" + "="*65)
print("7. RECOVERY SUMMARY")
print("="*65)

summary = {
    "training_features_valid": train_good,
    "validation_features_valid": val_good,
    "training_alignments_available": len(train_aligned),
    "validation_alignments_available": len(val_aligned),
    "best_checkpoint_exists":
        (checkpoint_dir / "best_checkpoint.pth.tar").exists(),
    "latest_checkpoint_exists":
        (checkpoint_dir / "latest_checkpoint.pth.tar").exists(),
    "training_history_exists": history_path.exists()
}

print(json.dumps(summary, indent=2))
print("\nAUDIT COMPLETE — NO FILES MODIFIED.")


1. ALIGNMENT DIRECTORY BREAKDOWN
alignment/full_alignment: 3163 TextGrid files
alignment/logs: 0 TextGrid files
alignment/retry_17: 15 TextGrid files
alignment/temporary: 0 TextGrid files

2. FEATURE ID COVERAGE
Training feature IDs: 2662
Validation feature IDs: 501
Training IDs with alignment: 2662
Validation IDs with alignment: 501
Training IDs missing alignment: 0
Validation IDs missing alignment: 0

Examples of missing training alignments: []
Examples of missing validation alignments: []

3. FEATURE FILE INTEGRITY
Valid: 2662
Invalid/missing: 0
Example feature shapes: {'text': (56,), 'duration': (56,), 'pitch': (56,), 'energy': (56,), 'mel': (392, 80), 'raw_text': (), 'speaker': ()}
Valid: 501
Invalid/missing: 0
Example feature shapes: {'text': (38,), 'duration': (38,), 'pitch': (38,), 'energy': (38,), 'mel': (288, 80), 'raw_text': (), 'speaker': ()}

4. CHECKPOINT FILES

best_checkpoint.pth.tar
Exists: True
Size MB: 89.38

latest_checkpoint.pth.tar
Exists: True
Size MB: 89.38

In

In [8]:
# ============================================================
# ENGLISH FASTSPEECH 2 — INFERENCE ENVIRONMENT RECOVERY
# Uses the ORIGINAL saved experiment. Does NOT retrain.
# ============================================================

import os, sys, json, subprocess, importlib
from pathlib import Path

PROJECT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")
REPO = Path("/content/FastSpeech2")

CHECKPOINT = PROJECT / "checkpoints/best_checkpoint.pth.tar"
MODEL_CONFIG = PROJECT / "config/model.yaml"
TRAIN_CONFIG = PROJECT / "config/train.yaml"
PREPROCESS_CONFIG = PROJECT / "config/preprocess.yaml"
VOCAB = PROJECT / "config/english_ipa_vocabulary.json"
STATS = PROJECT / "processed/stats.json"

# 1. Verify Drive and the original experiment
required = [
    CHECKPOINT, MODEL_CONFIG, TRAIN_CONFIG,
    PREPROCESS_CONFIG, VOCAB, STATS,
    PROJECT / "processed/train",
    PROJECT / "processed/validation",
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required English experiment files are missing:\n" +
        "\n".join(missing)
    )

print("English experiment files verified.")
print("Best checkpoint:", CHECKPOINT)
print("Checkpoint size:", round(CHECKPOINT.stat().st_size / 1024**2, 2), "MB")

# 2. Restore the FastSpeech2 repository if the runtime was reset
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ming024/FastSpeech2.git", str(REPO)],
        check=True
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("\nRepository:", REPO)

# 3. Install only the inference dependencies
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyyaml", "librosa==0.11.0", "pyworld",
    "unidecode", "soundfile", "matplotlib",
    "scipy", "tqdm"
], check=True)

# 4. Inspect the repository's inference entry points
print("\nPotential inference files:")
for p in REPO.rglob("*.py"):
    if any(word in p.name.lower() for word in
           ["synth", "inference", "model", "vocoder"]):
        print(" ", p.relative_to(REPO))

# 5. Inspect the saved model configuration and checkpoint
import torch
import yaml

with open(MODEL_CONFIG) as f:
    model_config = yaml.safe_load(f)

with open(PREPROCESS_CONFIG) as f:
    preprocess_config = yaml.safe_load(f)

with open(VOCAB) as f:
    vocabulary = json.load(f)

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\n========== ENGLISH CHECKPOINT ==========")
print("Training step:", checkpoint.get("step"))
print("Best validation loss:", checkpoint.get("best_val_loss"))
print("Vocabulary size:", len(vocabulary.get("phone_to_id", {})))
print("Model vocabulary size:", model_config.get("input_vocabulary_size"))
print("Model state entries:", len(checkpoint["model"]))

assert checkpoint.get("step") == 5000, (
    "Expected the best checkpoint at step 5000."
)
assert len(vocabulary.get("phone_to_id", {})) == 67
assert model_config.get("input_vocabulary_size") == 67

# 6. Verify feature files without modifying them
import numpy as np

train_dir = PROJECT / "processed/train"
val_dir = PROJECT / "processed/validation"

train_count = len(list(train_dir.glob("*.npz")))
val_count = len(list(val_dir.glob("*.npz")))

print("\n========== SAVED FEATURES ==========")
print("Training features:", train_count)
print("Validation features:", val_count)

assert train_count == 2662
assert val_count == 501

# 7. Find saved training history
history_file = PROJECT / "logs/english_training_history.csv"
assert history_file.exists(), "English training history is missing."

print("\nTraining history verified:", history_file)
print("\nINFERENCE ENVIRONMENT RECOVERY COMPLETE.")
print("No training, preprocessing, or alignment was rerun.")

English experiment files verified.
Best checkpoint: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/checkpoints/best_checkpoint.pth.tar
Checkpoint size: 89.38 MB

Repository: /content/FastSpeech2

Potential inference files:
  synthesize.py
  utils/model.py
  transformer/Models.py
  hifigan/models.py

========== ENGLISH CHECKPOINT ==========
Training step: 5000
Best validation loss: 3.629412801034989
Vocabulary size: 67
Model vocabulary size: 67
Model state entries: 138

========== SAVED FEATURES ==========
Training features: 2662
Validation features: 501

Training history verified: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/logs/english_training_history.csv

INFERENCE ENVIRONMENT RECOVERY COMPLETE.
No training, preprocessing, or alignment was rerun.


In [9]:
# ============================================================
# ENGLISH FASTSPEECH 2 — INFERENCE CODE INSPECTION
# Read-only: no training, preprocessing, or checkpoint changes.
# ============================================================

from pathlib import Path
import os

REPO = Path("/content/FastSpeech2")
PROJECT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

files_to_inspect = [
    REPO / "synthesize.py",
    REPO / "utils/model.py",
    REPO / "transformer/Models.py",
    REPO / "hifigan/__init__.py",
]

for file_path in files_to_inspect:
    print("\n" + "=" * 90)
    print("FILE:", file_path)
    print("=" * 90)

    if not file_path.exists():
        print("NOT FOUND")
        continue

    lines = file_path.read_text(errors="replace").splitlines()

    # Show the full inference entry point and model wrapper.
    # For the larger transformer file, show its beginning and
    # the definitions most relevant to model initialization.
    if file_path.name == "Models.py":
        matches = [
            i for i, line in enumerate(lines)
            if any(term in line for term in
                   ["class Encoder", "class Decoder", "class FastSpeech"])
        ]
        selected = set(range(min(100, len(lines))))
        for i in matches:
            selected.update(range(max(0, i-5), min(len(lines), i+45)))
        for i in sorted(selected):
            print(f"{i+1:4}: {lines[i]}")
    else:
        for i, line in enumerate(lines):
            print(f"{i+1:4}: {line}")

# Inspect available vocoder files and saved inference artifacts.
print("\n" + "=" * 90)
print("HIFIGAN / VOCODER FILES")
print("=" * 90)

for base in [REPO / "hifigan", REPO / "hifigan" / "assets",
             PROJECT / "vocoder", PROJECT / "checkpoints"]:
    if base.exists():
        print(f"\n{base}")
        for p in list(base.rglob("*"))[:60]:
            if p.is_file():
                print(" ", p.relative_to(base), f"({p.stat().st_size/1024**2:.2f} MB)")

# Show any existing English test-sentence definitions in the notebook files
# or saved project text files, without inventing replacements.
print("\n" + "=" * 90)
print("POSSIBLE SAVED TEST SENTENCES")
print("=" * 90)

for base in [PROJECT, Path("/content")]:
    for p in base.glob("**/*"):
        if p.is_file() and p.suffix.lower() in [".txt", ".json", ".yaml", ".yml"]:
            if any(k in p.name.lower() for k in
                   ["test", "sentence", "inference", "sample"]):
                print("\n", p)
                try:
                    print(p.read_text(errors="replace")[:5000])
                except Exception:
                    pass

print("\nINSPECTION COMPLETE. No files were modified.")


FILE: /content/FastSpeech2/synthesize.py
   1: import re
   2: import argparse
   3: from string import punctuation
   4: 
   5: import torch
   6: import yaml
   7: import numpy as np
   8: from torch.utils.data import DataLoader
   9: from g2p_en import G2p
  10: from pypinyin import pinyin, Style
  11: 
  12: from utils.model import get_model, get_vocoder
  13: from utils.tools import to_device, synth_samples
  14: from dataset import TextDataset
  15: from text import text_to_sequence
  16: 
  17: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  18: 
  19: 
  20: def read_lexicon(lex_path):
  21:     lexicon = {}
  22:     with open(lex_path) as f:
  23:         for line in f:
  24:             temp = re.split(r"\s+", line.strip("\n"))
  25:             word = temp[0]
  26:             phones = temp[1:]
  27:             if word.lower() not in lexicon:
  28:                 lexicon[word.lower()] = phones
  29:     return lexicon
  30: 
  31: 
  32: def prepr

In [11]:
# ============================================================
# PATCH VARIANCE ADAPTOR FOR THE SAVED STATS.JSON FORMAT
# ============================================================

from pathlib import Path
import json

REPO = Path("/content/FastSpeech2")
PROJECT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

modules_file = REPO / "model/modules.py"
source = modules_file.read_text()

old_pitch = 'pitch_min, pitch_max = stats["pitch"][:2]'
old_energy = 'energy_min, energy_max = stats["energy"][:2]'

new_pitch = '''pitch_stats = stats["pitch"]
            if isinstance(pitch_stats, dict):
                pitch_min, pitch_max = pitch_stats["min"], pitch_stats["max"]
            else:
                pitch_min, pitch_max = pitch_stats[:2]'''

new_energy = '''energy_stats = stats["energy"]
            if isinstance(energy_stats, dict):
                energy_min, energy_max = energy_stats["min"], energy_stats["max"]
            else:
                energy_min, energy_max = energy_stats[:2]'''

if old_pitch in source:
    source = source.replace(old_pitch, new_pitch)
elif new_pitch not in source:
    raise RuntimeError("Could not locate the pitch statistics code.")

if old_energy in source:
    source = source.replace(old_energy, new_energy)
elif new_energy not in source:
    raise RuntimeError("Could not locate the energy statistics code.")

modules_file.write_text(source)

# Verify the saved statistics without changing them
with open(PROJECT / "processed/stats.json") as f:
    stats = json.load(f)

print("Pitch:", stats["pitch"])
print("Energy:", stats["energy"])
print("\nUpdated model/modules.py successfully.")

# Reload model modules so the patched code is used
import sys, importlib

for name in ["model.modules", "model.fastspeech2", "model", "utils.model"]:
    if name in sys.modules:
        importlib.reload(sys.modules[name])

print("Model modules reloaded.")
print("Now rerun the model-loading section below.")

Pitch: {'mean': 179.5105018095426, 'std': 59.33071361434518, 'min': 64.46283721923828, 'max': 758.3485107421875, 'count': 151841}
Energy: {'mean': 26.42564903151013, 'std': 26.422304389107822, 'min': 0.0, 'max': 226.0308380126953, 'count': 152163}

Updated model/modules.py successfully.
Model modules reloaded.
Now rerun the model-loading section below.


In [14]:
# ============================================================
# DIRECT HIFIGAN LOADING — CPU SAFE
# Does not reload or modify the FastSpeech2 checkpoint.
# ============================================================

import os
import json
import torch
import hifigan

os.chdir("/content/FastSpeech2")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load HiFi-GAN architecture configuration
with open("hifigan/config.json", "r") as f:
    hifigan_config = hifigan.AttrDict(json.load(f))

# Create generator
vocoder = hifigan.Generator(hifigan_config)

# Load checkpoint explicitly onto CPU/GPU
vocoder_checkpoint_path = "hifigan/generator_LJSpeech.pth.tar"

vocoder_checkpoint = torch.load(
    vocoder_checkpoint_path,
    map_location=device,
    weights_only=False
)

# Restore generator weights
vocoder.load_state_dict(vocoder_checkpoint["generator"])

# Evaluation mode
vocoder.eval()
vocoder.remove_weight_norm()
vocoder.to(device)

print("FastSpeech2 model:", "LOADED")
print("HiFi-GAN vocoder:", "LOADED")
print("Device:", device)
print("Vocoder parameters:", sum(p.numel() for p in vocoder.parameters()))

# Quick vocoder shape test
with torch.no_grad():
    test_mel = torch.randn(1, 80, 20).to(device)
    test_audio = vocoder(test_mel)

print("Test mel shape:", tuple(test_mel.shape))
print("Test audio shape:", tuple(test_audio.shape))
print("\nBOTH MODELS READY FOR INFERENCE.")

Removing weight norm...
FastSpeech2 model: LOADED
HiFi-GAN vocoder: LOADED
Device: cpu
Vocoder parameters: 13926017
Test mel shape: (1, 80, 20)
Test audio shape: (1, 1, 5120)

BOTH MODELS READY FOR INFERENCE.


In [16]:
# ============================================================
# FIND ORIGINAL ENGLISH EVALUATION SENTENCES
# Read-only search across the saved assignment directory.
# ============================================================

from pathlib import Path
import json, re

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

keywords = ["E1", "E2a", "E2b", "E2(a)", "E2(b)", "E3", "E4"]

print("Searching English assignment files...\n")

found = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in [".json", ".txt", ".csv", ".yaml", ".yml", ".md"]:
        continue

    try:
        content = path.read_text(errors="ignore")
    except Exception:
        continue

    matches = [
        line.strip() for line in content.splitlines()
        if any(re.search(rf"\b{re.escape(k)}\b", line, re.I)
               for k in keywords)
    ]

    if matches:
        found.append((path, matches))

for path, matches in found:
    print("\nFILE:", path)
    for line in matches[:30]:
        print(" ", line[:1000])

if not found:
    print("No labelled English test sentences found in the saved project.")

print("\nSearch complete. No files changed.")

Searching English assignment files...

No labelled English test sentences found in the saved project.

Search complete. No files changed.


In [21]:
print("ACTUAL TRAINED IPA VOCABULARY:")
print(sorted(phone_to_id.keys()))

print("\nDH-related tokens:")
for token in phone_to_id:
    if any(x in token.lower() for x in ["d", "h", "t"]):
        print(repr(token), "->", phone_to_id[token])

print("\nARPABET phones used by your five sentences:")
for sid, sentence in TEST_SENTENCES.items():
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", sentence.lower())
    phones = []
    for word in words:
        phones.extend(lexicon.get(word, []))
    print(sid, sorted(set(phones)))

ACTUAL TRAINED IPA VOCABULARY:
['<pad>', '<sil>', '<unk>', 'a', 'aj', 'aw', 'aː', 'b', 'bʲ', 'c', 'cʷ', 'dʒ', 'd̪', 'eː', 'f', 'fʲ', 'h', 'i', 'iː', 'j', 'k', 'kʷ', 'l', 'm', 'mʲ', 'n', 'oː', 'p', 'pʲ', 's', 'spn', 'tʃ', 't̪', 'z', 'ç', 'ŋ', 'ɑ', 'ɑː', 'ɒ', 'ɒː', 'ɔj', 'ɖ', 'ə', 'ɛ', 'ɛː', 'ɜ', 'ɜː', 'ɟ', 'ɟʷ', 'ɡ', 'ɡʷ', 'ɪ', 'ɱ', 'ɲ', 'ɹ', 'ɾ', 'ʃ', 'ʈ', 'ʈʲ', 'ʈʷ', 'ʉ', 'ʉː', 'ʊ', 'ʋ', 'ʎ', 'ʒ', 'ʔ']

DH-related tokens:
'<pad>' -> 0
'dʒ' -> 11
'd̪' -> 12
'h' -> 16
'tʃ' -> 31
't̪' -> 32

ARPABET phones used by your five sentences:
E1 ['AH0', 'AH1', 'AY1', 'B', 'D', 'DH', 'EY1', 'IH0', 'IY0', 'L', 'N', 'NG', 'R', 'S', 'SH', 'T', 'Z']
E2a ['AH0', 'AW1', 'B', 'EH1', 'ER0', 'F', 'IY1', 'L', 'S', 'SH', 'T', 'UW1', 'Y', 'Z']
E2b ['AA1', 'AH0', 'AW1', 'B', 'DH', 'ER0', 'F', 'L', 'R', 'T', 'UW1', 'Y', 'Z']
E3 ['AA2', 'AH0', 'CH', 'D', 'DH', 'EH1', 'ER1', 'EY1', 'F', 'IH0', 'IH1', 'JH', 'L', 'N', 'NG', 'R', 'S', 'SH', 'T', 'W', 'Z']
E4 ['AE1', 'AH0', 'CH', 'D', 'ER0', 'IH1', 'IY0', 'IY1', 'K'

In [24]:
# ============================================================
# ENGLISH FASTSPEECH 2 — FINAL INFERENCE
# 5 WAV FILES + 15 VARIANCE ADAPTOR PLOTS
# Uses already-loaded model and vocoder.
# ============================================================

import os, re, json
import numpy as np
import torch
import soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
from collections import OrderedDict

# ------------------------------------------------------------
# 1. EDIT YOUR FIVE ASSIGNMENT SENTENCES HERE
# ------------------------------------------------------------

TEST_SENTENCES = OrderedDict([
    ("E1",  "The sun is shining brightly today."),
    ("E2a", "She sells beautiful flowers."),
    ("E2b", "The flowers are beautiful."),
    ("E3",  "Artificial intelligence is changing the world."),
    ("E4",  "Please speak clearly and naturally.")
])

assert all(
    sentence and not sentence.startswith("REPLACE WITH")
    for sentence in TEST_SENTENCES.values()
), "Replace all five placeholder sentences before running."

# ------------------------------------------------------------
# 2. PATHS AND MODEL CHECKS
# ------------------------------------------------------------

PROJECT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

AUDIO_DIR = PROJECT / "results/test_generated_audios"
PLOT_DIR = PROJECT / "results/variance_adaptor_plots"
PRED_DIR = PROJECT / "inference/test_predictions"

for folder in [
    AUDIO_DIR,
    PLOT_DIR / "duration",
    PLOT_DIR / "pitch",
    PLOT_DIR / "energy",
    PRED_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

assert model is not None and vocoder is not None

device = next(model.parameters()).device
model.eval()
vocoder.eval()

# ------------------------------------------------------------
# 3. LOAD IPA VOCABULARY AND LEXICON
# ------------------------------------------------------------

with open(PROJECT / "config/english_ipa_vocabulary.json") as f:
    vocab_data = json.load(f)

phone_to_id = vocab_data["phone_to_id"]

lexicon_path = Path(preprocess_config["path"]["lexicon_path"])

if not lexicon_path.is_absolute():
    candidates = [
        Path("/content/FastSpeech2") / lexicon_path,
        PROJECT / lexicon_path
    ]
    lexicon_path = next(p for p in candidates if p.exists())

lexicon = {}

with open(lexicon_path, encoding="utf-8", errors="ignore") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) > 1:
            lexicon.setdefault(parts[0].lower(), parts[1:])

# ------------------------------------------------------------
# 4. ARPABET TO IPA CONVERSION
# Selects an IPA candidate only if it exists in your vocabulary.
# ------------------------------------------------------------

ARPABET_TO_IPA = {
    "AA": [["ɑ"], ["ɒ"]],
    "AE": [["ɛ"]],
    "AH": [["ə"]],
    "AO": [["ɒ"]],
    "AW": [["aw"]],
    "AY": [["aj"]],
    "B": [["b"]],
    "CH": [["tʃ"]],
    "D": [["d̪"]],
    "DH": [["d̪"]],       # Approximation: voiced dental stop
    "EH": [["ɛ"]],
    "ER": [["ɜː"], ["ɜ"]],
    "EY": [["eː"]],
    "F": [["f"]],
    "G": [["ɡ"]],
    "HH": [["h"]],
    "IH": [["ɪ"]],
    "IY": [["iː"], ["i"]],
    "JH": [["dʒ"]],
    "K": [["k"]],
    "L": [["l"]],
    "M": [["m"]],
    "N": [["n"]],
    "NG": [["ŋ"]],
    "OW": [["oː"]],
    "OY": [["ɔj"]],
    "P": [["p"]],
    "R": [["ɹ"]],
    "S": [["s"]],
    "SH": [["ʃ"]],
    "T": [["t̪"]],
    "TH": [["t̪"]],       # Approximation: voiceless dental stop
    "UH": [["ʊ"]],
    "UW": [["ʉː"], ["ʉ"]],
    "V": [["ʋ"]],
    "W": [["ʋ"]],         # Approximation: labial-approximant sound
    "Y": [["j"]],
    "Z": [["z"]],
    "ZH": [["ʒ"]],
}

def convert_phone(phone):
    base = re.sub(r"[012]$", "", phone.upper())

    if base not in ARPABET_TO_IPA:
        raise ValueError(f"Unmapped ARPAbet phone: {phone}")

    for candidate in ARPABET_TO_IPA[base]:
        if all(symbol in phone_to_id for symbol in candidate):
            return candidate

    raise ValueError(
        f"No matching IPA symbol for {phone}. "
        f"Check your saved vocabulary."
    )

silence_token = next(
    (x for x in ["<sil>", "sp", "sil"] if x in phone_to_id),
    None
)

def sentence_to_ids(sentence):
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", sentence.lower())

    tokens = []
    missing = []

    for word in words:
        phones = lexicon.get(word)

        if phones is None:
            missing.append(word)
            continue

        if tokens and silence_token:
            tokens.append(silence_token)

        for phone in phones:
            tokens.extend(convert_phone(phone))

    if missing:
        raise ValueError(f"Words absent from lexicon: {missing}")

    unknown = [x for x in tokens if x not in phone_to_id]

    if unknown:
        raise ValueError(f"Unknown IPA tokens: {unknown}")

    return [phone_to_id[x] for x in tokens], tokens

# Convert all sentences first; do not partially generate if one fails.
prepared = {}

for sid, sentence in TEST_SENTENCES.items():
    ids, tokens = sentence_to_ids(sentence)
    prepared[sid] = (ids, tokens)
    print(f"{sid}: {len(ids)} IPA tokens")

print("\nAll five sentences converted successfully.")

# ------------------------------------------------------------
# 5. SAMPLE RATE AND PLOTTING FUNCTION
# ------------------------------------------------------------

sample_rate = int(
    preprocess_config["preprocessing"]["audio"]["sampling_rate"]
)

def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy().squeeze()
    return np.asarray(x).squeeze()

def create_plot(values, title, xlabel, ylabel, path):
    values = np.asarray(values).reshape(-1)

    plt.figure(figsize=(11, 4))
    plt.plot(values, linewidth=1.3)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ------------------------------------------------------------
# 6. FASTSPEECH 2 INFERENCE
# Correct output order:
# 0 postnet_mel
# 1 mel
# 2 log_duration_prediction
# 3 duration_rounded
# 4 pitch_prediction
# 5 energy_prediction
# ------------------------------------------------------------

results = []

for sid, sentence in TEST_SENTENCES.items():

    print(f"\nGenerating {sid}...")

    ids, ipa_tokens = prepared[sid]

    text_tensor = torch.LongTensor([ids]).to(device)
    text_lengths = torch.LongTensor([len(ids)]).to(device)
    speakers = torch.LongTensor([0]).to(device)

    with torch.no_grad():
        output = model(
            speakers,
            text_tensor,
            text_lengths,
            len(ids)
        )

    if len(output) < 6:
        raise RuntimeError(
            f"Unexpected FastSpeech2 output count: {len(output)}"
        )

    postnet_mel = output[0]
    mel = output[1]
    log_duration = to_numpy(output[2])
    duration = to_numpy(output[3])
    pitch = to_numpy(output[4])
    energy = to_numpy(output[5])

    # Use postnet-refined mel for audio synthesis.
    mel_for_audio = postnet_mel

    if mel_for_audio.shape[-1] != 80:
        raise RuntimeError(
            f"Unexpected mel shape: {tuple(mel_for_audio.shape)}"
        )

    # --------------------------------------------------------
    # 7. VOCODER
    # --------------------------------------------------------

    with torch.no_grad():
        audio_tensor = vocoder(
            mel_for_audio.transpose(1, 2)
        ).squeeze()

    audio = audio_tensor.detach().cpu().numpy().astype(np.float32)
    audio = np.nan_to_num(audio)

    peak = np.max(np.abs(audio)) if audio.size else 0

    if peak > 1:
        audio = audio * (0.98 / peak)

    audio_path = AUDIO_DIR / f"{sid}.wav"
    sf.write(audio_path, audio, sample_rate)

    # --------------------------------------------------------
    # 8. SAVE MODEL PREDICTIONS
    # --------------------------------------------------------

    prediction_path = PRED_DIR / f"{sid}_predictions.npz"

    np.savez(
        prediction_path,
        token_ids=np.array(ids),
        ipa_tokens=np.array(ipa_tokens, dtype=object),
        duration=duration,
        log_duration=log_duration,
        pitch=pitch,
        energy=energy,
        mel=mel.detach().cpu().numpy(),
        postnet_mel=postnet_mel.detach().cpu().numpy()
    )

    # --------------------------------------------------------
    # 9. GENERATE THREE PLOTS PER SENTENCE
    # --------------------------------------------------------

    create_plot(
        duration,
        f"{sid} — Predicted Duration",
        "IPA token index",
        "Duration (frames)",
        PLOT_DIR / "duration" / f"{sid}_duration.png"
    )

    create_plot(
        pitch,
        f"{sid} — Predicted Pitch",
        "IPA token index",
        "Pitch",
        PLOT_DIR / "pitch" / f"{sid}_pitch.png"
    )

    create_plot(
        energy,
        f"{sid} — Predicted Energy",
        "IPA token index",
        "Energy",
        PLOT_DIR / "energy" / f"{sid}_energy.png"
    )

    seconds = len(audio) / sample_rate

    results.append({
        "id": sid,
        "text": sentence,
        "ipa_tokens": ipa_tokens,
        "token_count": len(ids),
        "mel_frames": int(postnet_mel.shape[1]),
        "audio_seconds": round(seconds, 3),
        "sample_rate": sample_rate,
        "audio_file": str(audio_path),
        "prediction_file": str(prediction_path)
    })

    print(f"{sid} saved | {seconds:.2f} seconds")

# ------------------------------------------------------------
# 10. SAVE INFERENCE SUMMARY
# ------------------------------------------------------------

summary_path = PROJECT / "inference/english_inference_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": "FastSpeech 2",
        "checkpoint_step": 5000,
        "vocoder": "HiFi-GAN LJSpeech",
        "device": str(device),
        "sample_rate": sample_rate,
        "results": results
    }, f, indent=2, ensure_ascii=False)

# ------------------------------------------------------------
# 11. FINAL OUTPUT AUDIT
# ------------------------------------------------------------

audio_count = len(list(AUDIO_DIR.glob("E*.wav")))
duration_count = len(list((PLOT_DIR/"duration").glob("E*.png")))
pitch_count = len(list((PLOT_DIR/"pitch").glob("E*.png")))
energy_count = len(list((PLOT_DIR/"energy").glob("E*.png")))

print("\n" + "="*60)
print("ENGLISH INFERENCE FINISHED")
print("="*60)
print("Audio files:", audio_count, "/ 5")
print("Duration plots:", duration_count, "/ 5")
print("Pitch plots:", pitch_count, "/ 5")
print("Energy plots:", energy_count, "/ 5")
print("Summary:", summary_path)

assert audio_count == 5
assert duration_count == 5
assert pitch_count == 5
assert energy_count == 5

print("\nALL 5 AUDIOS AND 15 PLOTS SAVED SUCCESSFULLY.")

E1: 27 IPA tokens
E2a: 22 IPA tokens
E2b: 20 IPA tokens
E3: 38 IPA tokens
E4: 28 IPA tokens

All five sentences converted successfully.

Generating E1...
E1 saved | 3.20 seconds

Generating E2a...
E2a saved | 2.64 seconds

Generating E2b...
E2b saved | 2.21 seconds

Generating E3...
E3 saved | 3.91 seconds

Generating E4...
E4 saved | 3.05 seconds

ENGLISH INFERENCE FINISHED
Audio files: 5 / 5
Duration plots: 5 / 5
Pitch plots: 5 / 5
Energy plots: 5 / 5
Summary: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/inference/english_inference_summary.json

ALL 5 AUDIOS AND 15 PLOTS SAVED SUCCESSFULLY.


In [23]:
# DIAGNOSTIC ONLY — DOES NOT RUN INFERENCE

print("ACTUAL IPA VOCABULARY:")
print(list(phone_to_id.keys()))

print("\nDoes the vocabulary contain the DH sound?")
for x in ["ð", "d", "z", "θ", "t"]:
    print(repr(x), ":", x in phone_to_id)

print("\nARPABET phones in test sentences:")
for sid, sentence in TEST_SENTENCES.items():
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", sentence.lower())
    phones = []
    for word in words:
        phones.extend(lexicon.get(word, []))
    print(sid, sorted(set(phones)))

ACTUAL IPA VOCABULARY:
['<pad>', '<unk>', '<sil>', 'a', 'aj', 'aw', 'aː', 'b', 'bʲ', 'c', 'cʷ', 'dʒ', 'd̪', 'eː', 'f', 'fʲ', 'h', 'i', 'iː', 'j', 'k', 'kʷ', 'l', 'm', 'mʲ', 'n', 'oː', 'p', 'pʲ', 's', 'spn', 'tʃ', 't̪', 'z', 'ç', 'ŋ', 'ɑ', 'ɑː', 'ɒ', 'ɒː', 'ɔj', 'ɖ', 'ə', 'ɛ', 'ɛː', 'ɜ', 'ɜː', 'ɟ', 'ɟʷ', 'ɡ', 'ɡʷ', 'ɪ', 'ɱ', 'ɲ', 'ɹ', 'ɾ', 'ʃ', 'ʈ', 'ʈʲ', 'ʈʷ', 'ʉ', 'ʉː', 'ʊ', 'ʋ', 'ʎ', 'ʒ', 'ʔ']

Does the vocabulary contain the DH sound?
'ð' : False
'd' : False
'z' : True
'θ' : False
't' : False

ARPABET phones in test sentences:
E1 ['AH0', 'AH1', 'AY1', 'B', 'D', 'DH', 'EY1', 'IH0', 'IY0', 'L', 'N', 'NG', 'R', 'S', 'SH', 'T', 'Z']
E2a ['AH0', 'AW1', 'B', 'EH1', 'ER0', 'F', 'IY1', 'L', 'S', 'SH', 'T', 'UW1', 'Y', 'Z']
E2b ['AA1', 'AH0', 'AW1', 'B', 'DH', 'ER0', 'F', 'L', 'R', 'T', 'UW1', 'Y', 'Z']
E3 ['AA2', 'AH0', 'CH', 'D', 'DH', 'EH1', 'ER1', 'EY1', 'F', 'IH0', 'IH1', 'JH', 'L', 'N', 'NG', 'R', 'S', 'SH', 'T', 'W', 'Z']
E4 ['AE1', 'AH0', 'CH', 'D', 'ER0', 'IH1', 'IY0', 'IY1', 'K', '

In [25]:
# ============================================================
# ENGLISH TRAINING & VALIDATION LOSS PLOTS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

CSV_PATH = PROJECT / "logs/english_training_history.csv"
RESULTS = PROJECT / "results"

RESULTS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)

print("Available columns:", df.columns.tolist())
print("Training records:", len(df))

# Total loss: training vs validation
plt.figure(figsize=(10, 5))
plt.plot(df["step"], df["train_total"], label="Training Loss")
plt.plot(df["step"], df["validation_total"], label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Total Loss")
plt.title("English FastSpeech 2 — Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / "training_validation_loss.png", dpi=200)
plt.show()

# Individual training components
components = ["mel", "postnet", "pitch", "energy", "duration"]

plt.figure(figsize=(11, 6))
for component in components:
    column = f"train_{component}"
    if column in df.columns:
        plt.plot(df["step"], df[column], label=component)

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("English FastSpeech 2 — Training Loss Components")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / "training_loss_components.png", dpi=200)
plt.show()

# Individual validation components
plt.figure(figsize=(11, 6))
for component in components:
    column = f"validation_{component}"
    if column in df.columns:
        plt.plot(df["step"], df[column], label=component)

plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("English FastSpeech 2 — Validation Loss Components")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / "validation_loss_components.png", dpi=200)
plt.show()

print("\nENGLISH LOSS PLOTS SAVED:")
for file in RESULTS.glob("*loss*.png"):
    print(file)

Available columns: ['step', 'train_total', 'train_mel', 'train_postnet', 'train_pitch', 'train_energy', 'train_duration', 'validation_total', 'validation_mel', 'validation_postnet', 'validation_pitch', 'validation_energy', 'validation_duration']
Training records: 120

ENGLISH LOSS PLOTS SAVED:
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/results/english_training_loss.png
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/results/english_validation_loss.png
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/results/training_validation_loss.png
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/results/training_loss_components.png
/content/gdrive/MyDrive/FastSpeech2_Assignment/English/results/validation_loss_components.png


In [26]:
from pathlib import Path
import json

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/English")

checks = {
    "Best checkpoint": ROOT / "checkpoints/best_checkpoint.pth.tar",
    "Latest checkpoint": ROOT / "checkpoints/latest_checkpoint.pth.tar",
    "Training history": ROOT / "logs/english_training_history.csv",
    "Training/validation loss plot": ROOT / "results/training_validation_loss.png",
    "Training component plot": ROOT / "results/training_loss_components.png",
    "Validation component plot": ROOT / "results/validation_loss_components.png",
    "Inference summary": ROOT / "inference/english_inference_summary.json",
}

print("ENGLISH DELIVERABLE AUDIT")
print("=" * 55)

for name, path in checks.items():
    print(("OK" if path.exists() else "MISSING"), name)

folders = {
    "Generated audios": ROOT / "results/test_generated_audios",
    "Duration plots": ROOT / "results/variance_adaptor_plots/duration",
    "Pitch plots": ROOT / "results/variance_adaptor_plots/pitch",
    "Energy plots": ROOT / "results/variance_adaptor_plots/energy",
    "Prediction files": ROOT / "inference/test_predictions",
}

print("\nFILE COUNTS")
for name, folder in folders.items():
    count = len(list(folder.glob("*"))) if folder.exists() else 0
    print(f"{name}: {count}")

print("\nENGLISH AUDIT COMPLETE")

ENGLISH DELIVERABLE AUDIT
OK Best checkpoint
OK Latest checkpoint
OK Training history
OK Training/validation loss plot
OK Training component plot
OK Validation component plot
OK Inference summary

FILE COUNTS
Generated audios: 5
Duration plots: 5
Pitch plots: 5
Energy plots: 5
Prediction files: 5

ENGLISH AUDIT COMPLETE


In [27]:
# ============================================================
# TELUGU EXPERIMENT — COMPLETE DRIVE AUDIT
# READ-ONLY: DOES NOT MODIFY OR DELETE FILES
# ============================================================

from pathlib import Path
import os

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
TELUGU = ROOT / "Telugu"

print("=" * 65)
print("TELUGU EXPERIMENT AUDIT")
print("=" * 65)

if not TELUGU.exists():
    print("Telugu folder does not exist:", TELUGU)
else:
    print("Telugu folder:", TELUGU)

    for current, dirs, files in os.walk(TELUGU):
        dirs[:] = [d for d in dirs if d not in ["__pycache__", ".ipynb_checkpoints"]]

        relative = Path(current).relative_to(TELUGU)
        depth = len(relative.parts)
        indent = "  " * depth

        print(f"\n{indent}[{relative if str(relative) != '.' else 'Telugu'}]")

        for filename in sorted(files):
            filepath = Path(current) / filename
            size_mb = filepath.stat().st_size / (1024 * 1024)
            print(f"{indent}  {filename} ({size_mb:.2f} MB)")

print("\n" + "=" * 65)
print("TELUGU AUDIT FINISHED — NO FILES MODIFIED")

Streaming output truncated to the last 5000 lines.
      train_telugumale_01803.lab (0.00 MB)
      train_telugumale_01803.wav (1.15 MB)
      train_telugumale_01808.lab (0.00 MB)
      train_telugumale_01808.wav (3.04 MB)
      train_telugumale_01813.lab (0.00 MB)
      train_telugumale_01813.wav (1.42 MB)
      train_telugumale_01815.lab (0.00 MB)
      train_telugumale_01815.wav (0.62 MB)
      train_telugumale_01819.lab (0.00 MB)
      train_telugumale_01819.wav (0.88 MB)
      train_telugumale_01826.lab (0.00 MB)
      train_telugumale_01826.wav (0.45 MB)
      train_telugumale_01833.lab (0.00 MB)
      train_telugumale_01833.wav (3.69 MB)
      train_telugumale_01834.lab (0.00 MB)
      train_telugumale_01834.wav (1.25 MB)
      train_telugumale_01838.lab (0.00 MB)
      train_telugumale_01838.wav (1.13 MB)
      train_telugumale_01842.lab (0.00 MB)
      train_telugumale_01842.wav (1.54 MB)
      train_telugumale_01858.lab (0.00 MB)
      train_telugumale_01858.wav (0.62 MB)
   

In [28]:
# ============================================================
# TELUGU TRAINING + AUDIO QUALITY DIAGNOSTIC
# READ-ONLY — NO TRAINING OR FILE MODIFICATIONS
# ============================================================

import json
import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")

# 1. Training history
history_path = ROOT / "logs/training_history.json"

print("=" * 60)
print("1. TELUGU TRAINING HISTORY")
print("=" * 60)

with open(history_path, encoding="utf-8") as f:
    history = json.load(f)

if isinstance(history, dict):
    print("History keys:", list(history.keys()))
    for key, value in history.items():
        if isinstance(value, list):
            print(f"\n{key}: {len(value)} records")
            print("Last record:", value[-1] if value else "Empty")
        else:
            print(key, ":", value)
elif isinstance(history, list):
    print("Records:", len(history))
    print("First record:", history[0] if history else "Empty")
    print("Last record:", history[-1] if history else "Empty")

# 2. Audio quality audit
quality_path = ROOT / "results/audio_quality_audit/telugu_audio_quality.csv"

print("\n" + "=" * 60)
print("2. TELUGU AUDIO QUALITY")
print("=" * 60)

if quality_path.exists():
    quality = pd.read_csv(quality_path)
    print("Columns:", quality.columns.tolist())
    print(quality.to_string(index=False))
else:
    print("Audio quality CSV not found.")

# 3. Dataset manifests
print("\n" + "=" * 60)
print("3. DATASET MANIFESTS")
print("=" * 60)

for filename in ["train_manifest.csv", "validation_manifest.csv"]:
    path = ROOT / "fs2_prepared" / filename

    if path.exists():
        df = pd.read_csv(path)
        print(f"\n{filename}: {len(df)} rows")
        print("Columns:", df.columns.tolist())
        print(df.head(2).to_string(index=False))
    else:
        print(filename, "MISSING")

# 4. Inference diagnostic summary
print("\n" + "=" * 60)
print("4. VOCODER DIAGNOSTIC")
print("=" * 60)

diag_path = ROOT / "inference/vocoder_diagnostic/vocoder_diagnostic.json"

if diag_path.exists():
    with open(diag_path, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
else:
    print("Vocoder diagnostic JSON not found.")

print("\nDIAGNOSTIC READ COMPLETE.")

1. TELUGU TRAINING HISTORY
History keys: ['step', 'train_total', 'train_mel', 'train_postnet_mel', 'train_pitch', 'train_energy', 'train_duration', 'validation_total', 'validation_mel', 'validation_postnet_mel', 'validation_pitch', 'validation_energy', 'validation_duration', 'val_total', 'val_mel', 'val_postnet_mel', 'val_pitch', 'val_energy', 'val_duration']

step: 6 records
Last record: 600

train_total: 6 records
Last record: 11.308816909790039

train_mel: 6 records
Last record: 5.6544084548950195

train_postnet_mel: 6 records
Last record: 2.219219446182251

train_pitch: 6 records
Last record: 2.0743587017059326

train_energy: 6 records
Last record: 0.46613749861717224

train_duration: 6 records
Last record: 0.7418739795684814

validation_total: 3 records
Last record: 6.9097413420677185

validation_mel: 3 records
Last record: 2.734986980756124

validation_postnet_mel: 3 records
Last record: 3.114191840092341

validation_pitch: 3 records
Last record: 0.5171841631333033

validation_en

In [29]:
# ============================================================
# TELUGU — VALIDATION LOSS COMPARISON
# READ-ONLY: NO TRAINING OR FILE MODIFICATIONS
# ============================================================

import json
import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")
history_path = ROOT / "logs/training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

# Convert arrays into a table, padding shorter arrays with NaN.
df = pd.DataFrame({
    key: pd.Series(value)
    for key, value in history.items()
})

columns = [
    "step",
    "train_total",
    "validation_total",
    "val_total",
    "validation_mel",
    "val_mel",
    "validation_postnet_mel",
    "val_postnet_mel",
    "validation_pitch",
    "val_pitch",
    "validation_energy",
    "val_energy",
    "validation_duration",
    "val_duration"
]

available = [c for c in columns if c in df.columns]

print("VALIDATION LOSS COMPARISON")
print("=" * 75)
print(df[available].to_string(index=False))

print("\nARRAY LENGTHS")
for key, value in history.items():
    print(f"{key}: {len(value)}")

print("\nBEST VALUES")
for key in ["validation_total", "val_total"]:
    if key in df.columns:
        valid = df[key].dropna()
        if len(valid):
            idx = valid.idxmin()
            print(
                f"{key}: best={valid.loc[idx]:.5f}, "
                f"record={idx}, "
                f"step={df.loc[idx, 'step'] if 'step' in df else 'unknown'}"
            )

VALIDATION LOSS COMPARISON
 step  train_total  validation_total  val_total  validation_mel  val_mel  validation_postnet_mel  val_postnet_mel  validation_pitch  val_pitch  validation_energy  val_energy  validation_duration  val_duration
  100    11.093661          9.893165  12.541907        4.712234 6.270953                3.910243         2.640452          0.624183   2.535344           0.478156    0.563377             0.168349      0.400704
  200     8.841996          8.248127  12.424448        3.500096 6.212224                3.689031         2.639997          0.589574   2.543468           0.336399    0.517086             0.133027      0.392890
  300     6.579645          6.909741  12.609603        2.734987 6.304802                3.114192         2.651767          0.517184   2.544435           0.413352    0.553838             0.130026      0.406926
  400    12.609852               NaN        NaN             NaN      NaN                     NaN              NaN               NaN      

In [30]:
# ============================================================
# TELUGU — TRACE VALIDATION LOSS LOGGING
# READ-ONLY: NO TRAINING / NO FILE MODIFICATIONS
# ============================================================

from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")
SEARCH_ROOTS = [
    ROOT,
    Path("/content"),
]

terms = [
    "validation_total",
    "val_total",
    "validation_mel",
    "val_mel",
    "validation_loss",
    "evaluate(",
]

seen = set()

print("=" * 75)
print("SEARCHING TELUGU TRAINING AND EVALUATION SCRIPTS")
print("=" * 75)

for base in SEARCH_ROOTS:
    if not base.exists():
        continue

    for path in base.rglob("*.py"):
        if path in seen:
            continue
        seen.add(path)

        try:
            lines = path.read_text(
                encoding="utf-8",
                errors="ignore"
            ).splitlines()
        except Exception:
            continue

        matches = [
            (i, line.strip())
            for i, line in enumerate(lines)
            if any(term in line for term in terms)
        ]

        if matches:
            print(f"\nFILE: {path}")
            for i, line in matches[:15]:
                print(f"  Line {i+1}: {line[:220]}")

print("\n" + "=" * 75)
print("CHECKPOINT METADATA")
print("=" * 75)

import torch

checkpoint_dir = ROOT / "checkpoints"

for filename in [
    "300.pth.tar",
    "400.pth.tar",
    "500.pth.tar",
    "600.pth.tar",
    "latest.pth.tar",
]:
    path = checkpoint_dir / filename

    if not path.exists():
        print(filename, "NOT FOUND")
        continue

    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        print(
            filename,
            "| step:", ckpt.get("step"),
            "| best_val_loss:", ckpt.get("best_val_loss"),
            "| keys:", list(ckpt.keys())
        )
    except Exception as e:
        print(filename, "ERROR:", str(e)[:200])

print("\nTRACE COMPLETE.")

SEARCHING TELUGU TRAINING AND EVALUATION SCRIPTS

FILE: /content/FastSpeech2/evaluate.py
  Line 18: def evaluate(model, step, configs, logger=None, vocoder=None):
  Line 119: message = evaluate(model, args.restore_step, configs)

FILE: /content/FastSpeech2/train.py
  Line 144: message = evaluate(model, step, configs, val_logger, vocoder)

CHECKPOINT METADATA
300.pth.tar | step: 300 | best_val_loss: None | keys: ['step', 'model', 'optimizer', 'history', 'speaker_map', 'normalization']
400.pth.tar | step: 400 | best_val_loss: None | keys: ['step', 'model', 'optimizer', 'history', 'speaker_map', 'normalization']
500.pth.tar | step: 500 | best_val_loss: None | keys: ['step', 'model', 'optimizer', 'history', 'speaker_map', 'normalization']
600.pth.tar | step: 600 | best_val_loss: None | keys: ['step', 'model', 'optimizer', 'history', 'speaker_map', 'normalization']
latest.pth.tar | step: 600 | best_val_loss: None | keys: ['step', 'model', 'optimizer', 'history', 'speaker_map', 'normalizatio

In [31]:
# ============================================================
# TELUGU — EVALUATION CODE + CHECKPOINT HISTORY INSPECTION
# READ-ONLY
# ============================================================

from pathlib import Path
import torch
import json

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")
REPO = Path("/content/FastSpeech2")

# 1. Inspect evaluation implementation
print("=" * 75)
print("1. evaluate.py")
print("=" * 75)

for filename, start, end in [
    (REPO / "evaluate.py", 1, 115),
    (REPO / "train.py", 100, 165),
]:
    print(f"\nFILE: {filename}")
    if filename.exists():
        lines = filename.read_text(
            encoding="utf-8",
            errors="ignore"
        ).splitlines()

        for i in range(start - 1, min(end, len(lines))):
            print(f"{i+1:4}: {lines[i]}")
    else:
        print("FILE NOT FOUND")

# 2. Inspect checkpoint histories
print("\n" + "=" * 75)
print("2. HISTORIES STORED INSIDE CHECKPOINTS")
print("=" * 75)

for step in [100, 200, 300, 400, 500, 600]:
    path = ROOT / "checkpoints" / f"{step}.pth.tar"

    if not path.exists():
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    history = ckpt.get("history", {})

    print(f"\nCHECKPOINT {step}")
    print("History type:", type(history).__name__)

    if isinstance(history, dict):
        for key, value in history.items():
            if isinstance(value, list):
                print(
                    f"{key}: count={len(value)}, "
                    f"last={value[-1] if value else None}"
                )
            else:
                print(f"{key}: {value}")
    else:
        print(str(history)[:2000])

# 3. Inspect separate JSON log structure
print("\n" + "=" * 75)
print("3. EXTERNAL JSON HISTORY")
print("=" * 75)

history_path = ROOT / "logs" / "training_history.json"

with open(history_path, encoding="utf-8") as f:
    external_history = json.load(f)

print("Keys:", list(external_history.keys()))

for key in external_history:
    values = external_history[key]
    print(
        f"{key}: "
        f"first={values[0] if values else None}, "
        f"last={values[-1] if values else None}"
    )

print("\nINSPECTION COMPLETE.")

1. evaluate.py

FILE: /content/FastSpeech2/evaluate.py
   1: import argparse
   2: import os
   3: 
   4: import torch
   5: import yaml
   6: import torch.nn as nn
   7: from torch.utils.data import DataLoader
   8: 
   9: from utils.model import get_model, get_vocoder
  10: from utils.tools import to_device, log, synth_one_sample
  11: from model import FastSpeech2Loss
  12: from dataset import Dataset
  13: 
  14: 
  15: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  16: 
  17: 
  18: def evaluate(model, step, configs, logger=None, vocoder=None):
  19:     preprocess_config, model_config, train_config = configs
  20: 
  21:     # Get dataset
  22:     dataset = Dataset(
  23:         "val.txt", preprocess_config, train_config, sort=False, drop_last=False
  24:     )
  25:     batch_size = train_config["optimizer"]["batch_size"]
  26:     loader = DataLoader(
  27:         dataset,
  28:         batch_size=batch_size,
  29:         shuffle=False,
  30:       

In [32]:
# ============================================================
# TELUGU — CONFIGURATION AND CHECKPOINT COMPARISON
# READ-ONLY
# ============================================================

import json
import yaml
import torch
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")

print("=" * 70)
print("1. SAVED CONFIGURATIONS")
print("=" * 70)

for filename in [
    "config/model.yaml",
    "config/preprocess.yaml",
    "config/train.yaml",
]:
    path = ROOT / filename
    print(f"\n--- {filename} ---")

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("NOT FOUND")

print("\n" + "=" * 70)
print("2. CHECKPOINT 300 vs 500")
print("=" * 70)

for step in [300, 500]:
    path = ROOT / "checkpoints" / f"{step}.pth.tar"

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print(f"\nCHECKPOINT: {step}")
    print("Step:", ckpt.get("step"))
    print("Model parameter count:",
          sum(v.numel() for v in ckpt["model"].values()
              if torch.is_tensor(v)))

    print("Speaker map:", ckpt.get("speaker_map"))
    print("Normalization:", ckpt.get("normalization"))

    print("Model state sample:")
    for key, value in list(ckpt["model"].items())[:8]:
        if torch.is_tensor(value):
            print(key, tuple(value.shape))

print("\n" + "=" * 70)
print("3. PREPROCESSING STATISTICS")
print("=" * 70)

stats_path = ROOT / "fs2_prepared/model_metadata/stats.json"

if stats_path.exists():
    print(json.dumps(
        json.loads(stats_path.read_text(encoding="utf-8")),
        indent=2,
        ensure_ascii=False
    ))
else:
    print("Statistics file not found.")

print("\nCOMPARISON COMPLETE.")

1. SAVED CONFIGURATIONS

--- config/model.yaml ---
transformer:
  encoder_layer: 2
  encoder_head: 2
  encoder_hidden: 128
  decoder_layer: 2
  decoder_head: 2
  decoder_hidden: 128
  conv_filter_size: 512
  conv_kernel_size:
  - 9
  - 1
  encoder_dropout: 0.1
  decoder_dropout: 0.1
variance_predictor:
  filter_size: 128
  kernel_size: 3
  dropout: 0.2
variance_embedding:
  pitch_quantization: linear
  energy_quantization: linear
  n_bins: 256
multi_speaker: true
max_seq_len: 2048
vocoder:
  model: HiFi-GAN
  speaker: LJSpeech


--- config/preprocess.yaml ---
dataset: Telugu_MMS_FA
path:
  corpus_path: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu
  lexicon_path: ''
  raw_path: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio
  preprocessed_path: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/fs2_prepared/model_metadata
preprocessing:
  val_size: 0
  text:
    text_cleaners:
    - basic_cleaners
    language: te_romanized
  audio:
    sampling_rate: 22050
   

In [33]:
# ============================================================
# TELUGU — PREDICTION vs GROUND-TRUTH FEATURE AUDIT
# READ-ONLY — NO TRAINING / NO FILE MODIFICATIONS
# ============================================================

import numpy as np
import json
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")

def inspect_npz(path, label):
    print(f"\n--- {label} ---")
    if not path.exists():
        print("NOT FOUND:", path)
        return

    with np.load(path, allow_pickle=True) as data:
        for key in data.files:
            arr = np.asarray(data[key])
            print(f"\n{key}:")
            print(" shape:", arr.shape)
            print(" dtype:", arr.dtype)

            if np.issubdtype(arr.dtype, np.number):
                finite = arr[np.isfinite(arr)]
                if finite.size:
                    print(
                        " min:", float(finite.min()),
                        " max:", float(finite.max()),
                        " mean:", float(finite.mean()),
                        " std:", float(finite.std())
                    )

# 1. Inspect prediction files
print("=" * 70)
print("1. TELUGU INFERENCE PREDICTIONS")
print("=" * 70)

for name in ["T1", "T2a", "T2b", "T3", "T4"]:
    path = ROOT / "inference" / f"{name}_predictions.npz"
    inspect_npz(path, name)

# 2. Inspect a real training feature
print("\n" + "=" * 70)
print("2. REAL TRAINING FEATURES")
print("=" * 70)

train_dir = ROOT / "features" / "train"
feature_files = sorted(train_dir.glob("*.npz"))

if feature_files:
    inspect_npz(feature_files[0], feature_files[0].name)
else:
    print("No feature NPZ files found in:", train_dir)

# 3. Inspect prepared model statistics
print("\n" + "=" * 70)
print("3. NORMALIZATION STATISTICS")
print("=" * 70)

stats_path = ROOT / "fs2_prepared" / "model_metadata" / "stats.json"

if stats_path.exists():
    print(json.dumps(
        json.loads(stats_path.read_text(encoding="utf-8")),
        indent=2
    ))
else:
    print("Stats file not found.")

print("\nFEATURE AUDIT COMPLETE.")

1. TELUGU INFERENCE PREDICTIONS

--- T1 ---
NOT FOUND: /content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/T1_predictions.npz

--- T2a ---
NOT FOUND: /content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/T2a_predictions.npz

--- T2b ---
NOT FOUND: /content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/T2b_predictions.npz

--- T3 ---
NOT FOUND: /content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/T3_predictions.npz

--- T4 ---
NOT FOUND: /content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/T4_predictions.npz

2. REAL TRAINING FEATURES

--- train_telugufemale_00009.npz ---

mel:
 shape: (80, 1509)
 dtype: float32
 min: -11.512925148010254  max: 1.1536953449249268  mean: -6.723828315734863  std: 2.6046833992004395

pitch:
 shape: (1509,)
 dtype: float32
 min: 0.0  max: 348.48077392578125  mean: 110.892333984375  std: 111.24024200439453

energy:
 shape: (1509,)
 dtype: float32
 min: 0.00038481707451865077  max: 0.24603818356990814  mean:

In [34]:
# ============================================================
# TELUGU — LOCATE PREDICTIONS AND COMPARE FEATURES
# READ-ONLY
# ============================================================

import numpy as np
from pathlib import Path

ROOT = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu"
)

print("=" * 70)
print("1. LOCATING ALL INFERENCE PREDICTIONS")
print("=" * 70)

for name in ["T1", "T2a", "T2b", "T3", "T4"]:
    matches = list(ROOT.rglob(f"{name}*_predictions.npz"))
    print(f"\n{name}:")
    if matches:
        for p in matches:
            print(p)
    else:
        print("No matching file found.")

print("\n" + "=" * 70)
print("2. INSPECTING PREDICTION FILES")
print("=" * 70)

for name in ["T1", "T2a", "T2b", "T3", "T4"]:
    matches = list(ROOT.rglob(f"{name}*_predictions.npz"))

    for path in matches:
        print(f"\nFILE: {path}")
        with np.load(path, allow_pickle=True) as data:
            for key in data.files:
                arr = np.asarray(data[key])
                print(f"\n{key}: shape={arr.shape}, dtype={arr.dtype}")

                if np.issubdtype(arr.dtype, np.number):
                    finite = arr[np.isfinite(arr)]
                    if finite.size:
                        print(
                            " min:", float(finite.min()),
                            " max:", float(finite.max()),
                            " mean:", float(finite.mean()),
                            " std:", float(finite.std())
                        )

print("\n" + "=" * 70)
print("3. PREPARED TRAINING FEATURE FILES")
print("=" * 70)

for split in ["train", "validation"]:
    folder = ROOT / "fs2_prepared" / split
    files = sorted(folder.glob("*.npz"))

    print(f"\n{split}: {len(files)} files")

    if files:
        path = files[0]
        print("Sample:", path)

        with np.load(path, allow_pickle=True) as data:
            for key in data.files:
                arr = np.asarray(data[key])
                print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

                if np.issubdtype(arr.dtype, np.number):
                    finite = arr[np.isfinite(arr)]
                    if finite.size:
                        print(
                            " min:", float(finite.min()),
                            " max:", float(finite.max()),
                            " mean:", float(finite.mean()),
                            " std:", float(finite.std())
                        )

print("\nFILE LOCATION AND FEATURE AUDIT COMPLETE.")

1. LOCATING ALL INFERENCE PREDICTIONS

T1:
/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions/T1_predictions.npz

T2a:
/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions/T2a_predictions.npz

T2b:
/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions/T2b_predictions.npz

T3:
/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions/T3_predictions.npz
/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions/T3_checkpoint500_predictions.npz

T4:
/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions/T4_predictions.npz

2. INSPECTING PREDICTION FILES

FILE: /content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/inference/test_predictions/T1_predictions.npz

mel: shape=(588, 80), dtype=float32
 min: -6.452662944793701  max: -1.1587300300598145  mean: -3.0398356914520264  std: 0.9168557524681091

pitch: shape=(588,), dtype=float32
 min

In [35]:
# ============================================================
# TELUGU — INFERENCE PIPELINE INSPECTION
# READ-ONLY
# ============================================================

from pathlib import Path

ROOT = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu"
)

REPO = Path("/content/FastSpeech2")

# Search likely locations for inference scripts
search_roots = [ROOT, REPO]

terms = [
    "test_predictions",
    "predictions.npz",
    "postnet_mel",
    "mel_predictions",
    "p_control",
    "e_control",
    "d_control",
    "normalization",
    "pitch_mean",
    "energy_mean",
    "mel =",
    "model(",
]

seen = set()

print("=" * 75)
print("TELUGU INFERENCE SCRIPT SEARCH")
print("=" * 75)

for base in search_roots:
    for path in base.rglob("*.py"):
        if path in seen:
            continue
        seen.add(path)

        try:
            lines = path.read_text(
                encoding="utf-8",
                errors="ignore"
            ).splitlines()
        except Exception:
            continue

        matches = [
            (i, line.strip())
            for i, line in enumerate(lines)
            if any(term in line for term in terms)
        ]

        if matches:
            print(f"\nFILE: {path}")
            for i, line in matches[:20]:
                print(f"Line {i+1}: {line[:220]}")

# Inspect prediction metadata for the exact test text
print("\n" + "=" * 75)
print("SAVED TEST SENTENCE MAPPING")
print("=" * 75)

for filename in [
    "inference/test_sentence_token_mapping.json",
    "inference/test_sentences.json",
    "inference/telugu_inference_summary.json",
]:
    path = ROOT / filename

    if path.exists():
        print(f"\n--- {filename} ---")
        print(path.read_text(
            encoding="utf-8",
            errors="ignore"
        )[:7000])

print("\nINSPECTION COMPLETE.")

TELUGU INFERENCE SCRIPT SEARCH

FILE: /content/FastSpeech2/evaluate.py
Line 43: output = model(*(batch[2:]))
Line 117: model = get_model(args, configs, device, train=False).to(device)

FILE: /content/FastSpeech2/dataset.py
Line 43: mel = np.load(mel_path)

FILE: /content/FastSpeech2/synthesize.py
Line 95: output = model(
Line 97: p_control=pitch_control,
Line 98: e_control=energy_control,
Line 99: d_control=duration_control
Line 188: model = get_model(args, configs, device, train=False)

FILE: /content/FastSpeech2/train.py
Line 41: model, optimizer = get_model(args, configs, device, train=True)
Line 82: output = model(*(batch[2:]))

FILE: /content/FastSpeech2/preprocessor/preprocessor.py
Line 40: self.pitch_normalization = config["preprocessing"]["pitch"]["normalization"]
Line 41: self.energy_normalization = config["preprocessing"]["energy"]["normalization"]
Line 93: # Perform normalization if necessary
Line 94: if self.pitch_normalization:
Line 95: pitch_mean = pitch_scaler.mean_[0]
L

In [36]:
from pathlib import Path

ROOT = Path("/content/FastSpeech2")

sections = {
    "synthesize.py — inference and prediction saving": [
        (75, 125),
        (125, 175),
    ],
    "preprocessor/preprocessor.py — feature normalization": [
        (65, 140),
    ],
    "dataset.py — loading training features": [
        (20, 75),
    ],
    "utils/tools.py — inference output and vocoder": [
        (180, 245),
    ],
    "audio/tools.py — mel-to-audio conversion": [
        (1, 65),
    ],
    "model/loss.py — training loss": [
        (20, 95),
    ],
}

for filename, ranges in sections.items():
    path = ROOT / filename.split(" — ")[0]
    print("\n" + "=" * 90)
    print(filename)
    print("=" * 90)

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for start, end in ranges:
        print(f"\n--- Lines {start}–{end} ---")
        for i in range(start - 1, min(end, len(lines))):
            print(f"{i + 1:4}: {lines[i]}")

print("\nINSPECTION COMPLETE — NO FILES MODIFIED")


synthesize.py — inference and prediction saving

--- Lines 75–125 ---
  75:     phones = "{" + " ".join(phones) + "}"
  76:     print("Raw Text Sequence: {}".format(text))
  77:     print("Phoneme Sequence: {}".format(phones))
  78:     sequence = np.array(
  79:         text_to_sequence(
  80:             phones, preprocess_config["preprocessing"]["text"]["text_cleaners"]
  81:         )
  82:     )
  83: 
  84:     return np.array(sequence)
  85: 
  86: 
  87: def synthesize(model, step, configs, vocoder, batchs, control_values):
  88:     preprocess_config, model_config, train_config = configs
  89:     pitch_control, energy_control, duration_control = control_values
  90: 
  91:     for batch in batchs:
  92:         batch = to_device(batch, device)
  93:         with torch.no_grad():
  94:             # Forward
  95:             output = model(
  96:                 *(batch[2:]),
  97:                 p_control=pitch_control,
  98:                 e_control=energy_control,
  99: 

In [37]:
from pathlib import Path

ROOT = Path("/content/FastSpeech2")

checks = {
    "preprocessor/preprocessor.py": [(1, 65), (145, 260)],
    "audio/stft.py": [(1, 180)],
    "model/fastspeech2.py": [(1, 125)],
    "model/modules.py": [(1, 180)],
    "utils/model.py": [(1, 100)],
}

for filename, ranges in checks.items():
    path = ROOT / filename
    print("\n" + "=" * 95)
    print("FILE:", filename)
    print("=" * 95)

    if not path.exists():
        print("NOT FOUND")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for start, end in ranges:
        print(f"\n--- Lines {start}-{end} ---")
        for i in range(start - 1, min(end, len(lines))):
            print(f"{i + 1:4}: {lines[i]}")

print("\nREAD-ONLY INSPECTION COMPLETE")


FILE: preprocessor/preprocessor.py

--- Lines 1-65 ---
   1: import os
   2: import random
   3: import json
   4: 
   5: import tgt
   6: import librosa
   7: import numpy as np
   8: import pyworld as pw
   9: from scipy.interpolate import interp1d
  10: from sklearn.preprocessing import StandardScaler
  11: from tqdm import tqdm
  12: 
  13: import audio as Audio
  14: 
  15: 
  16: class Preprocessor:
  17:     def __init__(self, config):
  18:         self.config = config
  19:         self.in_dir = config["path"]["raw_path"]
  20:         self.out_dir = config["path"]["preprocessed_path"]
  21:         self.val_size = config["preprocessing"]["val_size"]
  22:         self.sampling_rate = config["preprocessing"]["audio"]["sampling_rate"]
  23:         self.hop_length = config["preprocessing"]["stft"]["hop_length"]
  24: 
  25:         assert config["preprocessing"]["pitch"]["feature"] in [
  26:             "phoneme_level",
  27:             "frame_level",
  28:         ]
  29:  

In [38]:
from pathlib import Path

file_path = Path("/content/FastSpeech2/model/modules.py")

source = file_path.read_text(encoding="utf-8")

old = """energy_prediction, energy_embedding = self.get_energy_embedding(
                x, energy_target, src_mask, p_control
            )"""

new = """energy_prediction, energy_embedding = self.get_energy_embedding(
                x, energy_target, src_mask, e_control
            )"""

if old in source:
    source = source.replace(old, new, 1)
    file_path.write_text(source, encoding="utf-8")
    print("SUCCESS: Energy-control correction applied.")
elif new in source:
    print("ALREADY FIXED: No changes needed.")
else:
    print("STOP: Code pattern not found. No changes made.")

SUCCESS: Energy-control correction applied.


In [39]:
from pathlib import Path
import torch

# 1. Verify the code correction
path = Path("/content/FastSpeech2/model/modules.py")
code = path.read_text(encoding="utf-8")

assert "x, energy_target, src_mask, e_control" in code
print("1. Energy-control correction: VERIFIED")

# 2. Verify saved Telugu checkpoints
from pathlib import Path

checkpoint_dir = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints"
)

for step in [300, 500, 600]:
    checkpoint = checkpoint_dir / f"{step}.pth.tar"
    print(f"{step} checkpoint:", "FOUND" if checkpoint.exists() else "MISSING")

# 3. Verify the current model is still available
print("3. FastSpeech2 model in memory:", "LOADED" if "model" in globals() else "NOT LOADED")
print("4. HiFi-GAN vocoder in memory:", "LOADED" if "vocoder" in globals() else "NOT LOADED")

print("\nVERIFICATION COMPLETE")

1. Energy-control correction: VERIFIED
300 checkpoint: FOUND
500 checkpoint: FOUND
600 checkpoint: FOUND
3. FastSpeech2 model in memory: LOADED
4. HiFi-GAN vocoder in memory: LOADED

VERIFICATION COMPLETE


In [40]:
import numpy as np
import torch
from pathlib import Path

# Telugu training feature directory
FEATURE_DIR = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/fs2_prepared/train"
)

# 1. Find a real prepared training example
files = sorted(FEATURE_DIR.glob("*.npz"))

if not files:
    raise FileNotFoundError("No Telugu training NPZ files found.")

sample_path = files[0]
data = np.load(sample_path, allow_pickle=True)

print("=" * 65)
print("TELUGU TEACHER-FORCING DIAGNOSTIC")
print("=" * 65)

print("\nSample:", sample_path.name)
print("\nFeature shapes:")

for key in data.files:
    value = data[key]
    print(f"{key}: shape={np.asarray(value).shape}, dtype={np.asarray(value).dtype}")

# 2. Check the loaded model and vocoder
print("\nLoaded objects:")
print("Model:", type(model).__name__)
print("Vocoder:", type(vocoder).__name__)
print("Device:", next(model.parameters()).device)

# 3. Check the Telugu checkpoint
CHECKPOINT_DIR = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints"
)

for step in [300, 500, 600]:
    path = CHECKPOINT_DIR / f"{step}.pth.tar"
    print(f"Checkpoint {step}:", "FOUND" if path.exists() else "MISSING")

# 4. Check configuration identity
print("\nConfiguration check:")

if "preprocess_config" in globals():
    print("Preprocessed path:",
          preprocess_config.get("path", {}).get("preprocessed_path"))
    print("Dataset:",
          preprocess_config.get("preprocessing", {}).get("dataset"))
else:
    print("preprocess_config is not available.")

if "model_config" in globals():
    print("Multi-speaker:", model_config.get("multi_speaker"))
    print("Model vocabulary size:",
          model_config.get("input_vocabulary_size"))
else:
    print("model_config is not available.")

print("\nINSPECTION COMPLETE — NO TRAINING OR FILES MODIFIED")

TELUGU TEACHER-FORCING DIAGNOSTIC

Sample: train_telugufemale_00009_chunk000.npz

Feature shapes:
mel: shape=(80, 1509), dtype=float32
pitch: shape=(1509,), dtype=float32
energy: shape=(1509,), dtype=float32
tokens: shape=(210,), dtype=<U1
text_ids: shape=(210,), dtype=int64
durations: shape=(210,), dtype=int64
original_id: shape=(), dtype=<U24
chunk_index: shape=(), dtype=int64
sample_rate: shape=(), dtype=int64
hop_length: shape=(), dtype=int64

Loaded objects:
Model: FastSpeech2
Vocoder: Generator
Device: cpu
Checkpoint 300: FOUND
Checkpoint 500: FOUND
Checkpoint 600: FOUND

Configuration check:
Preprocessed path: /content/gdrive/MyDrive/FastSpeech2_Assignment/English/processed
Dataset: None
Multi-speaker: False
Model vocabulary size: 67

INSPECTION COMPLETE — NO TRAINING OR FILES MODIFIED


In [41]:
from pathlib import Path
import yaml
import torch

TELUGU = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu"
)

print("=" * 70)
print("TELUGU MODEL CONFIGURATION AUDIT")
print("=" * 70)

# 1. Read Telugu configurations
for name in ["model.yaml", "preprocess.yaml", "train.yaml"]:
    path = TELUGU / "config" / name

    print(f"\n--- {name} ---")

    if not path.exists():
        print("NOT FOUND:", path)
        continue

    with open(path, encoding="utf-8") as f:
        config = yaml.safe_load(f)

    if name == "model.yaml":
        print("Multi-speaker:", config.get("multi_speaker"))
        print("Vocabulary size:", config.get("input_vocabulary_size"))
        print("Transformer:", config.get("transformer"))

    elif name == "preprocess.yaml":
        print("Paths:", config.get("path"))
        print("Dataset:", config.get("preprocessing", {}).get("dataset"))
        print("Text:", config.get("preprocessing", {}).get("text"))
        print("Pitch:", config.get("preprocessing", {}).get("pitch"))
        print("Energy:", config.get("preprocessing", {}).get("energy"))

    else:
        print("Training parameters:", config.get("optimizer"))
        print("Training steps:", config.get("step"))
        print("Paths:", config.get("path"))

# 2. Inspect Telugu checkpoint metadata
checkpoint_path = TELUGU / "checkpoints" / "300.pth.tar"

print("\n--- CHECKPOINT 300 ---")

if checkpoint_path.exists():
    ckpt = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint keys:", list(ckpt.keys()))
    print("Step:", ckpt.get("step"))
    print("Speaker map:", ckpt.get("speaker_map"))
    print("Normalization:", ckpt.get("normalization"))

    state = ckpt["model"]

    print("\nImportant model parameter shapes:")
    for key, value in state.items():
        if any(x in key for x in [
            "encoder.src_word_emb",
            "speaker_emb",
            "pitch_embedding",
            "energy_embedding",
            "mel_linear"
        ]):
            print(key, tuple(value.shape))
else:
    print("Checkpoint 300 not found.")

print("\nTELUGU CONFIGURATION AUDIT COMPLETE")

TELUGU MODEL CONFIGURATION AUDIT

--- model.yaml ---
Multi-speaker: True
Vocabulary size: None
Transformer: {'encoder_layer': 2, 'encoder_head': 2, 'encoder_hidden': 128, 'decoder_layer': 2, 'decoder_head': 2, 'decoder_hidden': 128, 'conv_filter_size': 512, 'conv_kernel_size': [9, 1], 'encoder_dropout': 0.1, 'decoder_dropout': 0.1}

--- preprocess.yaml ---
Paths: {'corpus_path': '/content/drive/MyDrive/FastSpeech2_Assignment/Telugu', 'lexicon_path': '', 'raw_path': '/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/raw_audio', 'preprocessed_path': '/content/drive/MyDrive/FastSpeech2_Assignment/Telugu/fs2_prepared/model_metadata'}
Dataset: None
Text: {'text_cleaners': ['basic_cleaners'], 'language': 'te_romanized'}
Pitch: {'feature': 'frame_level', 'normalization': True}
Energy: {'feature': 'frame_level', 'normalization': True}

--- train.yaml ---
Training parameters: {'batch_size': 1, 'betas': [0.9, 0.98], 'eps': 1e-09, 'weight_decay': 0.0, 'grad_clip_thresh': 1.0, 'grad_acc_step': 

In [42]:
from pathlib import Path
import yaml
import torch
import json

from model import FastSpeech2

# --------------------------------------------------
# 1. Telugu project paths
# --------------------------------------------------
TELUGU = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu"
)

CONFIG_DIR = TELUGU / "config"
METADATA_DIR = TELUGU / "fs2_prepared" / "model_metadata"
CHECKPOINT = TELUGU / "checkpoints" / "300.pth.tar"

# --------------------------------------------------
# 2. Load Telugu configurations
# --------------------------------------------------
with open(CONFIG_DIR / "preprocess.yaml", encoding="utf-8") as f:
    telugu_preprocess_config = yaml.safe_load(f)

with open(CONFIG_DIR / "model.yaml", encoding="utf-8") as f:
    telugu_model_config = yaml.safe_load(f)

with open(CONFIG_DIR / "train.yaml", encoding="utf-8") as f:
    telugu_train_config = yaml.safe_load(f)

# Use the actual mounted Drive path
telugu_preprocess_config["path"]["corpus_path"] = str(TELUGU)
telugu_preprocess_config["path"]["raw_path"] = str(TELUGU / "raw_audio")
telugu_preprocess_config["path"]["preprocessed_path"] = str(METADATA_DIR)

telugu_train_config["path"]["ckpt_path"] = str(TELUGU / "checkpoints")
telugu_train_config["path"]["log_path"] = str(TELUGU / "logs")
telugu_train_config["path"]["result_path"] = str(TELUGU / "results")

# --------------------------------------------------
# 3. Set vocabulary size from verified checkpoint
# --------------------------------------------------
telugu_model_config["input_vocabulary_size"] = 361

# --------------------------------------------------
# 4. Verify required metadata
# --------------------------------------------------
assert (METADATA_DIR / "speakers.json").exists(), "speakers.json missing"
assert (METADATA_DIR / "stats.json").exists(), "stats.json missing"
assert CHECKPOINT.exists(), "Checkpoint 300 missing"

with open(METADATA_DIR / "speakers.json") as f:
    print("Speaker mapping:", json.load(f))

# --------------------------------------------------
# 5. Create a separate Telugu model
# --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

telugu_model = FastSpeech2(
    telugu_preprocess_config,
    telugu_model_config
).to(device)

# --------------------------------------------------
# 6. Load checkpoint 300
# --------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=False
)

result = telugu_model.load_state_dict(
    checkpoint["model"],
    strict=True
)

telugu_model.eval()

# --------------------------------------------------
# 7. Verify
# --------------------------------------------------
print("\n" + "=" * 65)
print("TELUGU MODEL LOADING RESULT")
print("=" * 65)

print("Checkpoint step:", checkpoint["step"])
print("Model parameters:", sum(p.numel() for p in telugu_model.parameters()))
print("Vocabulary size:", telugu_model.encoder.src_word_emb.num_embeddings)
print("Speaker embeddings:", telugu_model.speaker_emb.num_embeddings)
print("Device:", device)
print("Strict checkpoint loading:", result)
print("\nTELUGU MODEL LOADED SUCCESSFULLY")

Speaker mapping: {'female': 0, 'male': 1}

TELUGU MODEL LOADING RESULT
Checkpoint step: 300
Model parameters: 8183361
Vocabulary size: 361
Speaker embeddings: 2
Device: cpu
Strict checkpoint loading: <All keys matched successfully>

TELUGU MODEL LOADED SUCCESSFULLY


In [43]:
import numpy as np
import torch
from pathlib import Path

# --------------------------------------------------
# 1. Load one real Telugu training sample
# --------------------------------------------------
feature_dir = (
    Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")
    / "fs2_prepared" / "train"
)

sample_path = feature_dir / "train_telugufemale_00009_chunk000.npz"

assert sample_path.exists(), f"Missing: {sample_path}"

data = np.load(sample_path, allow_pickle=True)

# --------------------------------------------------
# 2. Prepare model inputs
# --------------------------------------------------
device = next(telugu_model.parameters()).device

speaker = torch.tensor([0], dtype=torch.long, device=device)

text = torch.tensor(
    data["text_ids"].astype(np.int64),
    dtype=torch.long,
    device=device
).unsqueeze(0)

durations = torch.tensor(
    data["durations"].astype(np.int64),
    dtype=torch.long,
    device=device
).unsqueeze(0)

pitch = torch.tensor(
    data["pitch"].astype(np.float32),
    dtype=torch.float32,
    device=device
).unsqueeze(0)

energy = torch.tensor(
    data["energy"].astype(np.float32),
    dtype=torch.float32,
    device=device
).unsqueeze(0)

mel = torch.tensor(
    data["mel"].T.astype(np.float32),
    dtype=torch.float32,
    device=device
).unsqueeze(0)

src_lens = torch.tensor([text.shape[1]], device=device)
mel_lens = torch.tensor([mel.shape[1]], device=device)

# --------------------------------------------------
# 3. Run teacher-forced reconstruction
# --------------------------------------------------
telugu_model.eval()

with torch.no_grad():
    outputs = telugu_model(
        speaker,
        text,
        src_lens,
        text.shape[1],
        mel,
        mel_lens,
        mel.shape[1],
        pitch,
        energy,
        durations
    )

# --------------------------------------------------
# 4. Compare real and reconstructed mel
# --------------------------------------------------
predicted_mel = outputs[1][0].cpu().numpy()
real_mel = mel[0].cpu().numpy()

print("=" * 65)
print("TELUGU TEACHER-FORCED RECONSTRUCTION")
print("=" * 65)

print("Real mel shape:", real_mel.shape)
print("Predicted mel shape:", predicted_mel.shape)

print("\nREAL MEL")
print("Mean:", real_mel.mean())
print("Std:", real_mel.std())
print("Min:", real_mel.min())
print("Max:", real_mel.max())

print("\nPREDICTED MEL")
print("Mean:", predicted_mel.mean())
print("Std:", predicted_mel.std())
print("Min:", predicted_mel.min())
print("Max:", predicted_mel.max())

print("\nRECONSTRUCTION MAE:",
      np.mean(np.abs(real_mel - predicted_mel)))

print("\nRECONSTRUCTION TEST COMPLETE")

TELUGU TEACHER-FORCED RECONSTRUCTION
Real mel shape: (1509, 80)
Predicted mel shape: (1509, 80)

REAL MEL
Mean: -6.7238283
Std: 2.6046834
Min: -11.512925
Max: 1.1536953

PREDICTED MEL
Mean: -3.0405457
Std: 0.90952003
Min: -6.4048443
Max: -1.2364386

RECONSTRUCTION MAE: 3.8276787

RECONSTRUCTION TEST COMPLETE


In [44]:
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")

print("=" * 70)
print("TELUGU TRAINING PIPELINE FILES")
print("=" * 70)

for p in ROOT.rglob("*.py"):
    if any(x in p.name.lower() for x in ["train", "dataset", "preprocess"]):
        print(p)

print("\n" + "=" * 70)
print("LOCAL REPOSITORY TRAINING FILES")
print("=" * 70)

REPO = Path("/content/FastSpeech2")

for p in REPO.rglob("*.py"):
    if any(x in p.name.lower() for x in ["train", "dataset", "preprocess"]):
        print(p)

TELUGU TRAINING PIPELINE FILES

LOCAL REPOSITORY TRAINING FILES
/content/FastSpeech2/preprocess.py
/content/FastSpeech2/dataset.py
/content/FastSpeech2/train.py
/content/FastSpeech2/preprocessor/preprocessor.py


In [45]:
from pathlib import Path

REPO = Path("/content/FastSpeech2")

files_to_inspect = {
    "train.py": REPO / "train.py",
    "dataset.py": REPO / "dataset.py",
    "preprocessor.py": REPO / "preprocessor" / "preprocessor.py",
    "loss.py": REPO / "model" / "loss.py",
    "modules.py": REPO / "model" / "modules.py",
}

for name, path in files_to_inspect.items():
    print("\n" + "=" * 90)
    print(f"FILE: {name}")
    print("=" * 90)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(errors="replace").splitlines()

    # Print the complete file when reasonably sized.
    if len(lines) <= 300:
        print("\n".join(f"{i+1:4}: {line}" for i, line in enumerate(lines)))
    else:
        print(f"File has {len(lines)} lines. Showing first 300.")
        print("\n".join(f"{i+1:4}: {line}" for i, line in enumerate(lines[:300])))


FILE: train.py
   1: import argparse
   2: import os
   3: 
   4: import torch
   5: import yaml
   6: import torch.nn as nn
   7: from torch.utils.data import DataLoader
   8: from torch.utils.tensorboard import SummaryWriter
   9: from tqdm import tqdm
  10: 
  11: from utils.model import get_model, get_vocoder, get_param_num
  12: from utils.tools import to_device, log, synth_one_sample
  13: from model import FastSpeech2Loss
  14: from dataset import Dataset
  15: 
  16: from evaluate import evaluate
  17: 
  18: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  19: 
  20: 
  21: def main(args, configs):
  22:     print("Prepare training ...")
  23: 
  24:     preprocess_config, model_config, train_config = configs
  25: 
  26:     # Get dataset
  27:     dataset = Dataset(
  28:         "train.txt", preprocess_config, train_config, sort=True, drop_last=True
  29:     )
  30:     batch_size = train_config["optimizer"]["batch_size"]
  31:     group_size = 4  #

In [46]:
import numpy as np
import torch
from pathlib import Path

# Load the same real Telugu training sample
TELUGU = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")
sample_path = TELUGU / "fs2_prepared/train/train_telugufemale_00009_chunk000.npz"

data = np.load(sample_path, allow_pickle=True)
device = next(telugu_model.parameters()).device

# Checkpoint normalization statistics
ckpt = torch.load(
    TELUGU / "checkpoints/300.pth.tar",
    map_location="cpu",
    weights_only=False
)
norm = ckpt["normalization"]

pitch_mean = float(norm["pitch_mean"])
pitch_std = float(norm["pitch_std"])
energy_mean = float(norm["energy_mean"])
energy_std = float(norm["energy_std"])

# Normalize raw targets using checkpoint statistics
pitch_raw = data["pitch"].astype(np.float32)
energy_raw = data["energy"].astype(np.float32)

pitch_norm = (pitch_raw - pitch_mean) / pitch_std
energy_norm = (energy_raw - energy_mean) / energy_std

# Prepare tensors
speaker = torch.tensor([0], dtype=torch.long, device=device)

text = torch.tensor(
    data["text_ids"].astype(np.int64),
    dtype=torch.long,
    device=device
).unsqueeze(0)

durations = torch.tensor(
    data["durations"].astype(np.int64),
    dtype=torch.long,
    device=device
).unsqueeze(0)

pitch = torch.tensor(
    pitch_norm, dtype=torch.float32, device=device
).unsqueeze(0)

energy = torch.tensor(
    energy_norm, dtype=torch.float32, device=device
).unsqueeze(0)

mel = torch.tensor(
    data["mel"].T.astype(np.float32),
    dtype=torch.float32,
    device=device
).unsqueeze(0)

src_lens = torch.tensor([text.shape[1]], device=device)
mel_lens = torch.tensor([mel.shape[1]], device=device)

# Teacher-forced reconstruction
telugu_model.eval()

with torch.no_grad():
    outputs = telugu_model(
        speaker,
        text,
        src_lens,
        text.shape[1],
        mel,
        mel_lens,
        mel.shape[1],
        pitch,
        energy,
        durations
    )

real_mel = mel[0].cpu().numpy()
pred_mel = outputs[1][0].cpu().numpy()

print("=" * 65)
print("NORMALIZED TELUGU RECONSTRUCTION TEST")
print("=" * 65)

print("Pitch normalized range:",
      float(pitch.min()), float(pitch.max()))
print("Energy normalized range:",
      float(energy.min()), float(energy.max()))

print("\nReal mel mean/std:",
      float(real_mel.mean()), float(real_mel.std()))

print("Predicted mel mean/std:",
      float(pred_mel.mean()), float(pred_mel.std()))

print("\nReconstruction MAE:",
      float(np.mean(np.abs(real_mel - pred_mel))))

print("\nNORMALIZED RECONSTRUCTION TEST COMPLETE")

NORMALIZED TELUGU RECONSTRUCTION TEST
Pitch normalized range: -1.062591791152954 2.310774326324463
Energy normalized range: -0.9408150911331177 3.497180223464966

Real mel mean/std: -6.723828315734863 2.6046833992004395
Predicted mel mean/std: -3.7867538928985596 1.2515870332717896

Reconstruction MAE: 3.0890159606933594

NORMALIZED RECONSTRUCTION TEST COMPLETE


In [47]:
import numpy as np
import torch

# FastSpeech2 returns:
# outputs[0] = decoder mel prediction
# outputs[1] = postnet-refined mel prediction

decoder_mel = outputs[0][0].detach().cpu().numpy()
postnet_mel = outputs[1][0].detach().cpu().numpy()
target_mel = mel[0].detach().cpu().numpy()

def report(name, prediction):
    print(f"\n{name}")
    print("Shape:", prediction.shape)
    print("Mean:", float(prediction.mean()))
    print("Std:", float(prediction.std()))
    print("Min:", float(prediction.min()))
    print("Max:", float(prediction.max()))
    print("MAE vs target:", float(np.mean(np.abs(prediction - target_mel))))

print("=" * 65)
print("TELUGU DECODER VS POSTNET DIAGNOSTIC")
print("=" * 65)

print("\nTARGET MEL")
print("Mean:", float(target_mel.mean()))
print("Std:", float(target_mel.std()))
print("Min:", float(target_mel.min()))
print("Max:", float(target_mel.max()))

report("DECODER OUTPUT", decoder_mel)
report("POSTNET OUTPUT", postnet_mel)

print("\nPOSTNET CHANGE (mean absolute):",
      float(np.mean(np.abs(postnet_mel - decoder_mel))))

print("\nMODEL TRAINING MODE:", telugu_model.training)
print("CHECKPOINT STEP:", ckpt["step"])
print("\nDIAGNOSTIC COMPLETE")

TELUGU DECODER VS POSTNET DIAGNOSTIC

TARGET MEL
Mean: -6.723828315734863
Std: 2.6046833992004395
Min: -11.512925148010254
Max: 1.1536953449249268

DECODER OUTPUT
Shape: (1509, 80)
Mean: -4.451935291290283
Std: 0.5260211825370789
Min: -5.863694190979004
Max: -2.849274158477783
MAE vs target: 2.808413028717041

POSTNET OUTPUT
Shape: (1509, 80)
Mean: -3.7867538928985596
Std: 1.2515870332717896
Min: -7.192351341247559
Max: -1.2830361127853394
MAE vs target: 3.0890159606933594

POSTNET CHANGE (mean absolute): 1.252570629119873

MODEL TRAINING MODE: False
CHECKPOINT STEP: 300

DIAGNOSTIC COMPLETE


In [48]:
import torch
import numpy as np
from pathlib import Path

TELUGU = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu")
device = next(telugu_model.parameters()).device

# Preserve the currently loaded checkpoint-300 weights
original_state = {
    k: v.detach().cpu().clone()
    for k, v in telugu_model.state_dict().items()
}

# Same sample and inputs for every checkpoint
sample_path = TELUGU / "fs2_prepared/train/train_telugufemale_00009_chunk000.npz"
data = np.load(sample_path, allow_pickle=True)

ckpt300 = torch.load(
    TELUGU / "checkpoints/300.pth.tar",
    map_location="cpu",
    weights_only=False
)

norm = ckpt300["normalization"]

pitch = torch.tensor(
    ((data["pitch"].astype(np.float32) - norm["pitch_mean"])
     / norm["pitch_std"]),
    dtype=torch.float32, device=device
).unsqueeze(0)

energy = torch.tensor(
    ((data["energy"].astype(np.float32) - norm["energy_mean"])
     / norm["energy_std"]),
    dtype=torch.float32, device=device
).unsqueeze(0)

text = torch.tensor(
    data["text_ids"].astype(np.int64),
    dtype=torch.long, device=device
).unsqueeze(0)

durations = torch.tensor(
    data["durations"].astype(np.int64),
    dtype=torch.long, device=device
).unsqueeze(0)

mel = torch.tensor(
    data["mel"].T.astype(np.float32),
    dtype=torch.float32, device=device
).unsqueeze(0)

speaker = torch.tensor([0], dtype=torch.long, device=device)
src_lens = torch.tensor([text.shape[1]], device=device)
mel_lens = torch.tensor([mel.shape[1]], device=device)
target = mel[0].cpu().numpy()

print("=" * 75)
print("CHECKPOINT COMPARISON — SAME TEACHER-FORCED SAMPLE")
print("=" * 75)

try:
    for step in [300, 500, 600]:
        path = TELUGU / "checkpoints" / f"{step}.pth.tar"
        checkpoint = torch.load(
            path, map_location="cpu", weights_only=False
        )

        telugu_model.load_state_dict(checkpoint["model"], strict=True)
        telugu_model.eval()

        with torch.no_grad():
            out = telugu_model(
                speaker, text, src_lens, text.shape[1],
                mel, mel_lens, mel.shape[1],
                pitch, energy, durations
            )

        decoder = out[0][0].cpu().numpy()
        postnet = out[1][0].cpu().numpy()

        print(f"\nCHECKPOINT {step}")
        print("Decoder MAE:", round(float(np.mean(np.abs(decoder-target))), 5))
        print("Postnet MAE:", round(float(np.mean(np.abs(postnet-target))), 5))
        print("Decoder mean/std:", round(float(decoder.mean()), 4),
              round(float(decoder.std()), 4))
        print("Postnet mean/std:", round(float(postnet.mean()), 4),
              round(float(postnet.std()), 4))

finally:
    # Restore the original in-memory checkpoint-300 model
    telugu_model.load_state_dict(original_state, strict=True)
    telugu_model.eval()

print("\nOriginal checkpoint-300 model restored.")
print("No Drive checkpoint files were modified.")

CHECKPOINT COMPARISON — SAME TEACHER-FORCED SAMPLE

CHECKPOINT 300
Decoder MAE: 2.80841
Postnet MAE: 3.08902
Decoder mean/std: -4.4519 0.526
Postnet mean/std: -3.7868 1.2516

CHECKPOINT 500
Decoder MAE: 2.80841
Postnet MAE: 2.56274
Decoder mean/std: -4.4519 0.526
Postnet mean/std: -4.3674 1.4938

CHECKPOINT 600
Decoder MAE: 2.80841
Postnet MAE: 2.61717
Decoder mean/std: -4.4519 0.526
Postnet mean/std: -4.3048 1.4606

Original checkpoint-300 model restored.
No Drive checkpoint files were modified.


In [49]:
import torch
from pathlib import Path
from collections import defaultdict

TELUGU = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints")

checkpoints = {}
for step in [300, 500, 600]:
    path = TELUGU / f"{step}.pth.tar"
    checkpoints[step] = torch.load(
        path, map_location="cpu", weights_only=False
    )["model"]

def compare_weights(step_a, step_b):
    state_a = checkpoints[step_a]
    state_b = checkpoints[step_b]

    groups = defaultdict(lambda: {
        "tensors": 0,
        "changed": 0,
        "max_diff": 0.0,
        "total_diff": 0.0
    })

    for key in state_a:
        if key not in state_b:
            continue

        a = state_a[key].float()
        b = state_b[key].float()

        if a.shape != b.shape:
            continue

        diff = (a - b).abs()
        group = key.split(".")[0]

        groups[group]["tensors"] += 1
        if torch.count_nonzero(diff).item() > 0:
            groups[group]["changed"] += 1

        groups[group]["max_diff"] = max(
            groups[group]["max_diff"], float(diff.max())
        )
        groups[group]["total_diff"] += float(diff.sum())

    print(f"\nCHECKPOINT {step_a} vs {step_b}")
    print("-" * 75)
    print(f"{'Component':30} {'Changed':>10} {'Total':>10} {'Max diff':>14}")

    for name, values in sorted(groups.items()):
        print(
            f"{name:30} "
            f"{values['changed']:>10} "
            f"{values['tensors']:>10} "
            f"{values['max_diff']:>14.7f}"
        )

compare_weights(300, 500)
compare_weights(500, 600)

print("\nWEIGHT COMPARISON COMPLETE — no checkpoints modified.")


CHECKPOINT 300 vs 500
---------------------------------------------------------------------------
Component                         Changed      Total       Max diff
decoder                                 0         33      0.0000000
encoder                                 0         34      0.0000000
mel_linear                              0          2      0.0000000
postnet                                15         35    200.0000000
speaker_emb                             0          1      0.0000000
variance_adaptor                        0         34      0.0000000

CHECKPOINT 500 vs 600
---------------------------------------------------------------------------
Component                         Changed      Total       Max diff
decoder                                 0         33      0.0000000
encoder                                 0         34      0.0000000
mel_linear                              0          2      0.0000000
postnet                                15         35  

In [50]:
import torch
from pathlib import Path
from collections import defaultdict

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints")

print("=" * 75)
print("TELUGU OPTIMIZER AUDIT")
print("=" * 75)

# Model parameter counts
components = defaultdict(int)

for name, param in telugu_model.named_parameters():
    components[name.split(".")[0]] += param.numel()

print("\nMODEL PARAMETER COUNTS")
for name, count in sorted(components.items()):
    print(f"{name:30} {count:>12,}")

print("\nTOTAL MODEL PARAMETERS:", sum(components.values()))

# Inspect saved optimizer groups
for step in [300, 500, 600]:
    checkpoint = torch.load(
        ROOT / f"{step}.pth.tar",
        map_location="cpu",
        weights_only=False
    )

    optimizer = checkpoint["optimizer"]

    print(f"\n{'-' * 65}")
    print(f"CHECKPOINT {step} OPTIMIZER")
    print("-" * 65)

    total_ids = 0

    for i, group in enumerate(optimizer["param_groups"]):
        ids = group["params"]
        total_ids += len(ids)

        print(f"Parameter group {i}:")
        print("  Parameter tensors:", len(ids))
        print("  Learning rate:", group.get("lr"))
        print("  Weight decay:", group.get("weight_decay"))
        print("  Betas:", group.get("betas"))

    print("Total optimizer parameter tensors:", total_ids)
    print("Optimizer state entries:", len(optimizer["state"]))

print("\nOPTIMIZER AUDIT COMPLETE")

TELUGU OPTIMIZER AUDIT

MODEL PARAMETER COUNTS
decoder                           1,707,392
encoder                           1,753,600
mel_linear                           10,320
postnet                           4,348,144
speaker_emb                             256
variance_adaptor                    363,649

TOTAL MODEL PARAMETERS: 8183361

-----------------------------------------------------------------
CHECKPOINT 300 OPTIMIZER
-----------------------------------------------------------------
Parameter group 0:
  Parameter tensors: 124
  Learning rate: 0.0001
  Weight decay: 0.0
  Betas: (0.9, 0.98)
Total optimizer parameter tensors: 124
Optimizer state entries: 120

-----------------------------------------------------------------
CHECKPOINT 500 OPTIMIZER
-----------------------------------------------------------------
Parameter group 0:
  Parameter tensors: 124
  Learning rate: 0.0001
  Weight decay: 0.0
  Betas: (0.9, 0.98)
Total optimizer parameter tensors: 124
Optimizer state

In [51]:
import torch
from pathlib import Path

ROOT = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints"
)

# Parameter names and ordering from the loaded Telugu model
named_params = list(telugu_model.named_parameters())

print("Model parameter tensors:", len(named_params))

for step in [300, 500, 600]:
    checkpoint = torch.load(
        ROOT / f"{step}.pth.tar",
        map_location="cpu",
        weights_only=False
    )

    optimizer = checkpoint["optimizer"]
    param_ids = optimizer["param_groups"][0]["params"]
    states = optimizer["state"]

    print("\n" + "=" * 75)
    print(f"CHECKPOINT {step}: ADAM STATE BY MODEL PARAMETER")
    print("=" * 75)

    if len(param_ids) != len(named_params):
        print("WARNING: Parameter count/order mismatch.")
        print("Optimizer IDs:", len(param_ids))
        print("Model parameters:", len(named_params))
        continue

    component_stats = {}

    for (name, param), pid in zip(named_params, param_ids):
        state = states.get(pid)

        if not state:
            print(name, "— NO OPTIMIZER STATE")
            continue

        exp_avg = state.get("exp_avg")
        exp_avg_sq = state.get("exp_avg_sq")

        if exp_avg is None or exp_avg_sq is None:
            print(name, "— missing Adam moments")
            continue

        avg_norm = float(exp_avg.float().norm())
        sq_norm = float(exp_avg_sq.float().norm())
        avg_max = float(exp_avg.float().abs().max())

        component = name.split(".")[0]

        if component not in component_stats:
            component_stats[component] = {
                "tensors": 0,
                "nonzero_moments": 0,
                "max_avg": 0.0,
                "max_sq": 0.0
            }

        stats = component_stats[component]
        stats["tensors"] += 1

        if avg_norm > 0 or sq_norm > 0:
            stats["nonzero_moments"] += 1

        stats["max_avg"] = max(stats["max_avg"], avg_max)
        stats["max_sq"] = max(stats["max_sq"], sq_norm)

    print("\nCOMPONENT SUMMARY")
    for component, stats in sorted(component_stats.items()):
        print(
            f"{component:25} "
            f"tensors={stats['tensors']:3} "
            f"nonzero={stats['nonzero_moments']:3} "
            f"max_exp_avg={stats['max_avg']:.6g} "
            f"max_exp_avg_sq_norm={stats['max_sq']:.6g}"
        )

print("\nOPTIMIZER MOMENT AUDIT COMPLETE")

Model parameter tensors: 124

CHECKPOINT 300: ADAM STATE BY MODEL PARAMETER
encoder.position_enc — NO OPTIMIZER STATE
variance_adaptor.pitch_bins — NO OPTIMIZER STATE
variance_adaptor.energy_bins — NO OPTIMIZER STATE
decoder.position_enc — NO OPTIMIZER STATE

COMPONENT SUMMARY
decoder                   tensors= 32 nonzero= 32 max_exp_avg=0.00986044 max_exp_avg_sq_norm=0.000404732
encoder                   tensors= 33 nonzero= 33 max_exp_avg=0.000877534 max_exp_avg_sq_norm=0.000150799
mel_linear                tensors=  2 nonzero=  2 max_exp_avg=0.0048526 max_exp_avg_sq_norm=0.000802773
postnet                   tensors= 20 nonzero= 20 max_exp_avg=0.00172069 max_exp_avg_sq_norm=1.9122e-05
speaker_emb               tensors=  1 nonzero=  1 max_exp_avg=0.000491815 max_exp_avg_sq_norm=2.37776e-05
variance_adaptor          tensors= 32 nonzero= 32 max_exp_avg=0.078822 max_exp_avg_sq_norm=0.0776221

CHECKPOINT 500: ADAM STATE BY MODEL PARAMETER
encoder.position_enc — NO OPTIMIZER STATE
varianc

In [52]:
import torch
from pathlib import Path

ROOT = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints"
)

loaded = {}

for step in [300, 500, 600]:
    loaded[step] = torch.load(
        ROOT / f"{step}.pth.tar",
        map_location="cpu",
        weights_only=False
    )

print("=" * 75)
print("ADAM STATE STEP AND CHECKPOINT CONSISTENCY")
print("=" * 75)

for step, checkpoint in loaded.items():
    opt_state = checkpoint["optimizer"]["state"]

    recorded_steps = []
    for state in opt_state.values():
        if "step" in state:
            recorded_steps.append(float(state["step"]))

    print(f"\nCheckpoint {step}")
    print("Model training step:", checkpoint["step"])
    print("Adam state entries:", len(opt_state))
    print("Adam recorded step values:",
          sorted(set(recorded_steps)))
    print("History records:", len(checkpoint.get("history", [])))

# Compare Adam moments exactly
def compare_optimizer_states(a, b):
    sa = loaded[a]["optimizer"]["state"]
    sb = loaded[b]["optimizer"]["state"]

    common = set(sa) & set(sb)
    identical = 0
    different = 0

    for pid in common:
        for key in sa[pid]:
            if key not in sb[pid]:
                different += 1
                continue

            va, vb = sa[pid][key], sb[pid][key]

            if torch.is_tensor(va) and torch.is_tensor(vb):
                same = torch.equal(va, vb)
            else:
                same = va == vb

            if same:
                identical += 1
            else:
                different += 1

    print(f"\nOptimizer {a} vs {b}:")
    print("Identical state values:", identical)
    print("Different state values:", different)

compare_optimizer_states(300, 500)
compare_optimizer_states(500, 600)

print("\nDIAGNOSTIC COMPLETE")

ADAM STATE STEP AND CHECKPOINT CONSISTENCY

Checkpoint 300
Model training step: 300
Adam state entries: 120
Adam recorded step values: [300.0]
History records: 13

Checkpoint 500
Model training step: 500
Adam state entries: 120
Adam recorded step values: [300.0]
History records: 19

Checkpoint 600
Model training step: 600
Adam state entries: 120
Adam recorded step values: [300.0]
History records: 19

Optimizer 300 vs 500:
Identical state values: 360
Different state values: 0

Optimizer 500 vs 600:
Identical state values: 360
Different state values: 0

DIAGNOSTIC COMPLETE


In [53]:
import torch
from pathlib import Path

ROOT = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints"
)

states = {}

for step in [300, 500, 600]:
    checkpoint = torch.load(
        ROOT / f"{step}.pth.tar",
        map_location="cpu",
        weights_only=False
    )
    states[step] = checkpoint["model"]

# Names of trainable parameters, excluding buffers
parameter_names = {
    name for name, _ in telugu_model.named_parameters()
}

def compare_trainable(a, b):
    changed = []
    unchanged = []

    for name in sorted(parameter_names):
        x = states[a][name]
        y = states[b][name]

        if not torch.equal(x, y):
            changed.append((
                name,
                float((x.float() - y.float()).abs().max())
            ))
        else:
            unchanged.append(name)

    print(f"\nTRAINABLE WEIGHTS: {a} vs {b}")
    print("Changed tensors:", len(changed))
    print("Unchanged tensors:", len(unchanged))

    for name, diff in changed[:30]:
        print(f"  CHANGED: {name} | max difference: {diff:.8f}")

compare_trainable(300, 500)
compare_trainable(500, 600)

print("\nTRAINABLE WEIGHT AUDIT COMPLETE")


TRAINABLE WEIGHTS: 300 vs 500
Changed tensors: 0
Unchanged tensors: 124

TRAINABLE WEIGHTS: 500 vs 600
Changed tensors: 0
Unchanged tensors: 124

TRAINABLE WEIGHT AUDIT COMPLETE


In [54]:
import torch
from pathlib import Path

ROOT = Path(
    "/content/gdrive/MyDrive/FastSpeech2_Assignment/Telugu/checkpoints"
)

states = {}

for step in [300, 500, 600]:
    states[step] = torch.load(
        ROOT / f"{step}.pth.tar",
        map_location="cpu",
        weights_only=False
    )["model"]

for a, b in [(300, 500), (500, 600)]:
    print(f"\nCHECKPOINT {a} vs {b}")
    print("-" * 70)

    for name in states[a]:
        if not torch.equal(states[a][name], states[b][name]):
            x = states[a][name]
            y = states[b][name]

            print(
                name,
                "| shape:", tuple(x.shape),
                "| dtype:", x.dtype,
                "| max difference:",
                float((x.float() - y.float()).abs().max())
            )

print("\nBUFFER COMPARISON COMPLETE")


CHECKPOINT 300 vs 500
----------------------------------------------------------------------
postnet.convolutions.0.1.running_mean | shape: (512,) | dtype: torch.float32 | max difference: 0.31562089920043945
postnet.convolutions.0.1.running_var | shape: (512,) | dtype: torch.float32 | max difference: 0.06967969238758087
postnet.convolutions.0.1.num_batches_tracked | shape: () | dtype: torch.int64 | max difference: 200.0
postnet.convolutions.1.1.running_mean | shape: (512,) | dtype: torch.float32 | max difference: 0.01579321175813675
postnet.convolutions.1.1.running_var | shape: (512,) | dtype: torch.float32 | max difference: 0.04821494221687317
postnet.convolutions.1.1.num_batches_tracked | shape: () | dtype: torch.int64 | max difference: 200.0
postnet.convolutions.2.1.running_mean | shape: (512,) | dtype: torch.float32 | max difference: 0.01897171139717102
postnet.convolutions.2.1.running_var | shape: (512,) | dtype: torch.float32 | max difference: 0.13278430700302124
postnet.convolu

In [57]:
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")

for p in [
    ROOT / "README.md",
    ROOT / "requirements.txt",
    ROOT / "submission_audit.json",
    ROOT / "Report" / "Technical_Report.pdf",
    ROOT / "FastSpeech2_Submission.zip",
]:
    print(("FOUND  " if p.exists() else "MISSING"), p)

MISSING /content/gdrive/MyDrive/FastSpeech2_Assignment/README.md
MISSING /content/gdrive/MyDrive/FastSpeech2_Assignment/requirements.txt
MISSING /content/gdrive/MyDrive/FastSpeech2_Assignment/submission_audit.json
MISSING /content/gdrive/MyDrive/FastSpeech2_Assignment/Report/Technical_Report.pdf
MISSING /content/gdrive/MyDrive/FastSpeech2_Assignment/FastSpeech2_Submission.zip


In [60]:
# ============================================================
# FINAL SUBMISSION BUILDER — RUN ONCE
# Reuses existing English and Telugu experiment outputs.
# Does NOT run training, preprocessing, alignment or inference.
# ============================================================

from pathlib import Path
import os, json, csv, shutil, zipfile, sys

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
EN = ROOT / "English"
TE = ROOT / "Telugu"
REPORT = ROOT / "Report"
CODE = ROOT / "Code" / "FastSpeech2"
REPO = Path("/content/FastSpeech2")

assert ROOT.exists(), f"Assignment folder not found: {ROOT}"
assert EN.exists(), "English results folder is missing."
assert TE.exists(), "Telugu results folder is missing."

REPORT.mkdir(parents=True, exist_ok=True)
CODE.mkdir(parents=True, exist_ok=True)

print("1/7 — Checking existing experiment artifacts...")

def count_files(folder, suffix=None):
    if not folder.exists():
        return 0
    files = folder.rglob("*")
    if suffix:
        return sum(1 for p in files if p.is_file() and p.suffix.lower() == suffix)
    return sum(1 for p in files if p.is_file())

audit = {
    "project": "FastSpeech 2 English and Telugu",
    "datasets": {
        "English": "ai4bharat/Svarah",
        "Telugu": "SPRINGLab/IndicTTS_Telugu"
    },
    "English": {
        "checkpoints": count_files(EN / "checkpoints", ".tar"),
        "audio_files": count_files(EN, ".wav"),
        "plots": count_files(EN, ".png"),
        "feature_files": count_files(EN / "processed", ".npz")
    },
    "Telugu": {
        "checkpoints": count_files(TE / "checkpoints", ".tar"),
        "audio_files": count_files(TE, ".wav"),
        "plots": count_files(TE, ".png"),
        "feature_files": count_files(TE / "features", ".npz")
    }
}

(ROOT / "submission_audit.json").write_text(
    json.dumps(audit, indent=2), encoding="utf-8"
)

print(json.dumps(audit, indent=2))

print("\n2/7 — Creating README...")

readme_lines = [
    "# FastSpeech 2: English and Telugu",
    "",
    "This project implements and evaluates FastSpeech 2 for English and Telugu speech synthesis.",
    "",
    "## Datasets",
    "- English: ai4bharat/Svarah",
    "- Telugu: SPRINGLab/IndicTTS_Telugu",
    "",
    "## Project structure",
    "```text",
    "FastSpeech2_Assignment/",
    "├── English/",
    "├── Telugu/",
    "├── Code/FastSpeech2/",
    "├── Report/Technical_Report.pdf",
    "├── README.md",
    "├── requirements.txt",
    "└── submission_audit.json",
    "```",
    "",
    "## Experiment summary",
    "- English: the saved experiment includes training and validation features, alignments, checkpoints, training history, generated audio, and variance-adaptor plots.",
    "- English best checkpoint: step 5000; recorded validation loss: 3.6294.",
    "- Telugu: prepared data, alignments, features, checkpoints, training logs, generated audio, and variance-adaptor plots are included where present.",
    "- Telugu inference audio quality was not satisfactory. The predicted mel-spectrograms differed substantially from the real training mel-spectrograms.",
    "- Telugu checkpoint comparisons showed no changes to trainable model weights after step 300. This limitation is documented rather than presenting the later checkpoints as successful continued training.",
    "",
    "## Results",
    "The English and Telugu folders contain their respective configurations, checkpoints, logs, generated audio, and plots.",
    "Refer to the technical report for methodology, evaluation, observations, and limitations.",
    "",
    "## Reproducibility",
    "Install dependencies using:",
    "```bash",
    "pip install -r requirements.txt",
    "```",
    "",
    "The original experiment artifacts are preserved. Dataset audio is not copied into the submission ZIP.",
    "",
    "## Source code",
    "The Code/FastSpeech2 directory contains the FastSpeech2 implementation source copied from the working repository.",
    ""
]
(ROOT / "README.md").write_text("\n".join(readme_lines), encoding="utf-8")

print("\n3/7 — Creating requirements.txt...")

requirements = [
    "numpy",
    "scipy",
    "pandas",
    "matplotlib",
    "PyYAML",
    "librosa",
    "soundfile",
    "torch",
    "torchaudio",
    "torchvision",
    "scikit-learn",
    "tqdm",
    "tensorboard",
    "pyworld",
    "pillow",
    "tgt",
    "g2p-en",
    "reportlab"
]
(ROOT / "requirements.txt").write_text(
    "\n".join(requirements) + "\n", encoding="utf-8"
)

print("\n4/7 — Copying FastSpeech2 source code...")

if REPO.exists():
    copied = 0
    excluded_dirs = {
        ".git", "__pycache__", ".ipynb_checkpoints",
        "venv", ".venv", "node_modules"
    }
    allowed_suffixes = {
        ".py", ".yaml", ".yml", ".json", ".txt", ".md"
    }

    for source in REPO.rglob("*"):
        if not source.is_file():
            continue
        relative = source.relative_to(REPO)
        if any(part in excluded_dirs for part in relative.parts):
            continue
        if source.suffix.lower() not in allowed_suffixes:
            continue
        # Avoid copying local model weights and large generated files.
        if any(term in str(relative).lower() for term in
               ["checkpoint", "generator_ljspeech", "results", "output"]):
            continue

        destination = CODE / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        copied += 1

    print(f"Copied {copied} source/config/documentation files.")
else:
    print("WARNING: /content/FastSpeech2 not found; source copy skipped.")

print("\n5/7 — Creating technical report PDF...")

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, PageBreak
    )
    from reportlab.lib.units import inch

except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab"])
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, PageBreak
    )
    from reportlab.lib.units import inch

pdf_path = REPORT / "Technical_Report.pdf"
styles = getSampleStyleSheet()
story = []

def add_heading(text, style="Heading1"):
    story.append(Paragraph(text, styles[style]))
    story.append(Spacer(1, 8))

def add_para(text):
    story.append(Paragraph(text, styles["BodyText"]))
    story.append(Spacer(1, 7))

add_heading("FastSpeech 2: English and Telugu Speech Synthesis")
add_para("Technical report documenting the implementation, training artifacts, inference experiments, evaluation observations, and limitations.")

add_heading("1. Objective")
add_para("The objective was to implement and evaluate FastSpeech 2 independently for English and Telugu using the specified datasets. The workflow includes dataset preparation, text/audio processing, duration, pitch and energy supervision, model training, inference, and result visualization.")

add_heading("2. Datasets")
add_para("<b>English:</b> ai4bharat/Svarah.")
add_para("<b>Telugu:</b> SPRINGLab/IndicTTS_Telugu.")
add_para("The language experiments use separate configurations, data artifacts, and checkpoints.")

add_heading("3. Model and preprocessing")
add_para("FastSpeech 2 uses a Transformer-based encoder and decoder with a variance adaptor for duration, pitch, and energy. Mel-spectrograms are used as acoustic targets, with a HiFi-GAN vocoder for waveform generation.")
add_para("The English experiment uses IPA-based text representations. The Telugu experiment uses romanized text representations and a multilingual alignment workflow. The language-specific preprocessing configurations are retained in the experiment folders.")

add_heading("4. English experiment")
add_para("The saved English experiment contains 2,662 training feature files and 501 validation feature files, with alignment coverage for all feature IDs.")
add_para("The best recorded English checkpoint is at step 5,000, with validation loss approximately 3.6294. The saved training history, checkpoints, generated test audios, loss plots, and duration, pitch, and energy prediction plots are included in the English folder.")
add_para("The English inference artifacts include five test audio outputs and five plots for each of the three variance-adaptor predictions.")

add_heading("5. Telugu experiment")
add_para("The Telugu experiment contains prepared training and validation data, alignment outputs, feature files, checkpoints, training logs, generated audio, and variance-adaptor plots.")
add_para("The training artifacts include checkpoints through step 600. The checkpoint comparisons showed that trainable model weights were unchanged between the saved checkpoints after step 300; therefore, the later checkpoints should not be interpreted as evidence of continued parameter learning.")
add_para("Inference audio quality was unsatisfactory. Predicted mel-spectrograms were compressed and shifted relative to real training mel-spectrograms, and the generated outputs contained noise. This is an important limitation of the current Telugu experiment.")

add_heading("6. Evaluation and visualizations")
add_para("The saved results include training and validation loss plots, component loss plots where available, generated audio files, and duration, pitch, and energy prediction plots for the test sentences.")
add_para("These plots visualize the model's variance-adaptor predictions. They should be interpreted alongside the audio-quality limitations, especially for Telugu.")

add_heading("7. Limitations and future work")
add_para("The Telugu experiment requires further investigation of training-loop continuation, optimizer-state restoration, acoustic feature normalization, and predicted mel-spectrogram distributions. Telugu inference quality should be improved and reevaluated before claiming satisfactory synthesis.")
add_para("The English artifacts represent a substantially more complete experiment. The two language results should be reported separately rather than implying equivalent synthesis quality.")

add_heading("8. Deliverables")
add_para("The submission folder contains language-specific experiment artifacts, source code, configuration files, checkpoints, logs, generated audio, plots, this report, a README, dependency list, and an audit inventory.")
add_para("The submission ZIP excludes the original raw audio datasets to reduce package size. Those datasets must be obtained from their respective dataset sources.")

doc = SimpleDocTemplate(
    str(pdf_path), pagesize=A4,
    rightMargin=0.65*inch, leftMargin=0.65*inch,
    topMargin=0.65*inch, bottomMargin=0.65*inch
)
doc.build(story)
print("Created:", pdf_path)

print("\n6/7 — Building submission ZIP...")

zip_path = ROOT / "FastSpeech2_Submission.zip"
exclude_parts = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    "venv", ".venv", "node_modules"
}
exclude_names = {
    "raw_audio"
}

top_level = [
    ROOT / "English",
    ROOT / "Telugu",
    ROOT / "Code",
    ROOT / "Report",
    ROOT / "README.md",
    ROOT / "requirements.txt",
    ROOT / "submission_audit.json"
]

written = 0
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in top_level:
        if not item.exists():
            continue

        if item.is_file():
            zf.write(item, item.relative_to(ROOT))
            written += 1
            continue

        for file in item.rglob("*"):
            if not file.is_file():
                continue
            relative = file.relative_to(ROOT)

            if any(part in exclude_parts for part in relative.parts):
                continue
            if any(part.lower() in exclude_names for part in relative.parts):
                continue
            if "cache" in str(relative).lower():
                continue

            zf.write(file, relative)
            written += 1

print(f"Added {written} files to ZIP.")

print("\n7/7 — Verifying final files...")

for path in [
    ROOT / "README.md",
    ROOT / "requirements.txt",
    ROOT / "submission_audit.json",
    pdf_path,
    zip_path
]:
    assert path.exists() and path.stat().st_size > 0, f"Missing/empty: {path}"
    print("OK", path.name, f"({path.stat().st_size / (1024*1024):.2f} MB)")

print("\nFINALIZATION COMPLETE")
print("Submission folder:", ROOT)
print("ZIP:", zip_path)
print("NOTE: This creates the package locally in Google Drive. It does NOT publish to GitHub.")

1/7 — Checking existing experiment artifacts...
{
  "project": "FastSpeech 2 English and Telugu",
  "datasets": {
    "English": "ai4bharat/Svarah",
    "Telugu": "SPRINGLab/IndicTTS_Telugu"
  },
  "English": {
    "checkpoints": 2,
    "audio_files": 3202,
    "plots": 20,
    "feature_files": 3163
  },
  "Telugu": {
    "checkpoints": 7,
    "audio_files": 1543,
    "plots": 30,
    "feature_files": 1531
  }
}

2/7 — Creating README...

3/7 — Creating requirements.txt...

4/7 — Copying FastSpeech2 source code...
Copied 65 source/config/documentation files.

5/7 — Creating technical report PDF...
Created: /content/gdrive/MyDrive/FastSpeech2_Assignment/Report/Technical_Report.pdf

6/7 — Building submission ZIP...
Added 10253 files to ZIP.

7/7 — Verifying final files...
OK README.md (0.00 MB)
OK requirements.txt (0.00 MB)
OK submission_audit.json (0.00 MB)
OK Technical_Report.pdf (0.00 MB)
OK FastSpeech2_Submission.zip (2837.32 MB)

FINALIZATION COMPLETE
Submission folder: /content/gdr

In [1]:
from pathlib import Path
import torch

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")
TE = ROOT / "Telugu"

print("=" * 65)
print("TELUGU TRAINING DIAGNOSTIC")
print("=" * 65)

# 1. Locate training-related scripts
for name in ["train.py", "evaluate.py", "model/loss.py"]:
    p = REPO / name
    print(f"\n{name}: {'FOUND' if p.exists() else 'MISSING'}")
    if p.exists():
        print("Path:", p)

# 2. Inspect checkpoint trainable parameters and optimizer state
for step in [300, 400, 500, 600]:
    p = TE / "checkpoints" / f"{step}.pth.tar"
    if not p.exists():
        continue

    ckpt = torch.load(p, map_location="cpu", weights_only=False)
    state = ckpt.get("model", {})
    optimizer = ckpt.get("optimizer", {})
    opt_state = optimizer.get("state", {})

    print(f"\nCHECKPOINT {step}")
    print("Saved step:", ckpt.get("step"))
    print("Model tensors:", len(state))
    print("Optimizer state entries:", len(opt_state))

    adam_steps = set()
    for entry in opt_state.values():
        if "step" in entry:
            adam_steps.add(float(entry["step"].item()))

    print("Adam step values:", sorted(adam_steps))

# 3. Display relevant training loop
p = REPO / "train.py"
if p.exists():
    lines = p.read_text(errors="replace").splitlines()
    print("\n" + "=" * 65)
    print("TRAINING LOOP SOURCE")
    print("=" * 65)

    keywords = [
        "optimizer.step",
        "zero_grad",
        "backward",
        "load_state_dict",
        "save_checkpoint",
        "scheduler.step",
        "train_step"
    ]

    for i, line in enumerate(lines):
        if any(k in line for k in keywords):
            start = max(0, i - 3)
            end = min(len(lines), i + 5)
            print(f"\n--- Lines {start+1}-{end} ---")
            for j in range(start, end):
                print(f"{j+1}: {lines[j]}")

print("\nDIAGNOSTIC COMPLETE — no files modified.")

TELUGU TRAINING DIAGNOSTIC

train.py: MISSING

evaluate.py: MISSING

model/loss.py: MISSING

DIAGNOSTIC COMPLETE — no files modified.


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import os

print("Google Drive mount status:")
for p in [Path("/content/gdrive"), Path("/content/drive")]:
    print(p, "EXISTS:", p.exists())

print("\nPossible project locations:")
for p in [
    Path("/content/gdrive/MyDrive/FastSpeech2_Assignment"),
    Path("/content/drive/MyDrive/FastSpeech2_Assignment"),
    Path("/content/gdrive/My Drive/FastSpeech2_Assignment"),
    Path("/content/drive/My Drive/FastSpeech2_Assignment"),
]:
    print(p, "EXISTS:", p.exists())

print("\nContent directory:")
for p in Path("/content").iterdir():
    print(p)

Google Drive mount status:
/content/gdrive EXISTS: False
/content/drive EXISTS: False

Possible project locations:
/content/gdrive/MyDrive/FastSpeech2_Assignment EXISTS: False
/content/drive/MyDrive/FastSpeech2_Assignment EXISTS: False
/content/gdrive/My Drive/FastSpeech2_Assignment EXISTS: False
/content/drive/My Drive/FastSpeech2_Assignment EXISTS: False

Content directory:
/content/.config
/content/sample_data


In [7]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [8]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

print("Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Assignment folder found:", ROOT.exists())

if ROOT.exists():
    for item in sorted(ROOT.iterdir()):
        print(("📁 " if item.is_dir() else "📄 ") + item.name)

Drive mounted: True
Assignment folder found: True
📁 Code
📁 English
📄 FastSpeech2_Submission.zip
📄 README.md
📁 Report
📁 Telugu
📁 metadata
📁 raw_data
📄 requirements.txt
📄 submission_audit.json


In [10]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

for folder in [ROOT / "Code", ROOT / "Telugu"]:
    print("\n" + "=" * 65)
    print("FOLDER:", folder)
    print("EXISTS:", folder.exists())

    if not folder.exists():
        continue

    print("\nTop-level contents:")
    for p in sorted(folder.iterdir()):
        print(("📁 " if p.is_dir() else "📄 ") + p.name)

    print("\nRelevant Python files:")
    for p in sorted(folder.rglob("*.py")):
        if any(k in p.name.lower() for k in
               ["train", "loss", "model", "dataset", "evaluate"]):
            print(p.relative_to(ROOT))

print("\nCheckpoint files:")
for p in sorted((ROOT / "Telugu" / "checkpoints").glob("*.pth.tar")):
    print(p.name, f"{p.stat().st_size / (1024*1024):.1f} MB")


FOLDER: /content/drive/MyDrive/FastSpeech2_Assignment/Code
EXISTS: True

Top-level contents:
📁 FastSpeech2

Relevant Python files:
Code/FastSpeech2/dataset.py
Code/FastSpeech2/evaluate.py
Code/FastSpeech2/hifigan/models.py
Code/FastSpeech2/model/loss.py
Code/FastSpeech2/train.py
Code/FastSpeech2/transformer/Models.py
Code/FastSpeech2/utils/model.py

FOLDER: /content/drive/MyDrive/FastSpeech2_Assignment/Telugu
EXISTS: True

Top-level contents:
📁 checkpoints
📁 config
📁 features
📁 fs2_prepared
📁 inference
📁 inference_smoke_test
📁 logs
📁 raw_audio
📁 results
📁 vocoder

Relevant Python files:

Checkpoint files:
100.pth.tar 89.8 MB
200.pth.tar 89.8 MB
300.pth.tar 89.8 MB
400.pth.tar 89.8 MB
500.pth.tar 89.8 MB
600.pth.tar 89.8 MB
latest.pth.tar 89.8 MB


In [11]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2")

files = [
    BASE / "train.py",
    BASE / "utils" / "model.py",
    BASE / "evaluate.py",
    BASE / "model" / "loss.py",
]

for path in files:
    print("\n" + "=" * 85)
    print("FILE:", path.relative_to(BASE))
    print("=" * 85)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("Total lines:", len(lines))

    if path.name == "train.py":
        # Print the full training script.
        for i, line in enumerate(lines, 1):
            print(f"{i:4}: {line}")

    else:
        # Print relevant model-loading, optimizer, and evaluation logic.
        keywords = [
            "def get_model", "def get_optimizer", "load_state_dict",
            "optimizer", "checkpoint", "backward", "evaluate",
            "FastSpeech2Loss", "def forward"
        ]
        matches = [
            i for i, line in enumerate(lines)
            if any(k.lower() in line.lower() for k in keywords)
        ]

        selected = set()
        for i in matches:
            selected.update(range(max(0, i-5), min(len(lines), i+10)))

        previous = -2
        for i in sorted(selected):
            if i > previous + 1:
                print("     ...")
            print(f"{i+1:4}: {lines[i]}")
            previous = i

print("\nINSPECTION COMPLETE — no files modified.")


FILE: train.py
Total lines: 198
   1: import argparse
   2: import os
   3: 
   4: import torch
   5: import yaml
   6: import torch.nn as nn
   7: from torch.utils.data import DataLoader
   8: from torch.utils.tensorboard import SummaryWriter
   9: from tqdm import tqdm
  10: 
  11: from utils.model import get_model, get_vocoder, get_param_num
  12: from utils.tools import to_device, log, synth_one_sample
  13: from model import FastSpeech2Loss
  14: from dataset import Dataset
  15: 
  16: from evaluate import evaluate
  17: 
  18: device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  19: 
  20: 
  21: def main(args, configs):
  22:     print("Prepare training ...")
  23: 
  24:     preprocess_config, model_config, train_config = configs
  25: 
  26:     # Get dataset
  27:     dataset = Dataset(
  28:         "train.txt", preprocess_config, train_config, sort=True, drop_last=True
  29:     )
  30:     batch_size = train_config["optimizer"]["batch_size"]
  31:     

In [12]:
from pathlib import Path
import torch
import yaml

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
CODE = ROOT / "Code" / "FastSpeech2"
TE = ROOT / "Telugu"

# 1. Find optimizer implementation and other training scripts
print("=" * 70)
print("1. OPTIMIZER / TRAINING IMPLEMENTATIONS")
print("=" * 70)

for p in CODE.rglob("*.py"):
    try:
        content = p.read_text(errors="replace")
    except Exception:
        continue

    if any(term in content for term in [
        "class ScheduledOptim",
        "def step_and_update_lr",
        "optimizer.step()",
        "history.append"
    ]):
        print("\nFILE:", p.relative_to(CODE))
        lines = content.splitlines()
        for i, line in enumerate(lines):
            if any(term in line for term in [
                "class ScheduledOptim",
                "def step_and_update_lr",
                "optimizer.step()",
                "history.append"
            ]):
                for j in range(max(0, i-3), min(len(lines), i+9)):
                    print(f"{j+1:4}: {lines[j]}")
                print("...")

# 2. Read Telugu training config
print("\n" + "=" * 70)
print("2. TELUGU TRAIN CONFIG")
print("=" * 70)

config_path = TE / "config" / "train.yaml"
if config_path.exists():
    config = yaml.safe_load(config_path.read_text())
    print(yaml.safe_dump(config, sort_keys=False))
else:
    print("train.yaml not found")

# 3. Inspect exact checkpoint structure
print("\n" + "=" * 70)
print("3. CHECKPOINT STRUCTURE")
print("=" * 70)

for step in [300, 500, 600]:
    path = TE / "checkpoints" / f"{step}.pth.tar"
    if not path.exists():
        continue

    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    print(f"\nCheckpoint {step}")
    print("Top-level keys:", list(ckpt.keys()))
    print("Saved step:", ckpt.get("step"))
    print("History records:", len(ckpt.get("history", [])))

    opt = ckpt.get("optimizer", {})
    print("Optimizer keys:", list(opt.keys()))
    print("Optimizer parameter groups:", len(opt.get("param_groups", [])))

    if opt.get("param_groups"):
        group = opt["param_groups"][0]
        print("Optimizer type:", group.get("name", "not recorded"))
        print("Learning rate:", group.get("lr"))
        print("Betas:", group.get("betas"))

print("\nREAD-ONLY DIAGNOSTIC COMPLETE.")

1. OPTIMIZER / TRAINING IMPLEMENTATIONS

FILE: model/optimizer.py
   2: import numpy as np
   3: 
   4: 
   5: class ScheduledOptim:
   6:     """ A simple wrapper class for learning rate scheduling """
   7: 
   8:     def __init__(self, model, train_config, model_config, current_step):
   9: 
  10:         self._optimizer = torch.optim.Adam(
  11:             model.parameters(),
  12:             betas=train_config["optimizer"]["betas"],
  13:             eps=train_config["optimizer"]["eps"],
...
  19:         self.current_step = current_step
  20:         self.init_lr = np.power(model_config["transformer"]["encoder_hidden"], -0.5)
  21: 
  22:     def step_and_update_lr(self):
  23:         self._update_learning_rate()
  24:         self._optimizer.step()
  25: 
  26:     def zero_grad(self):
  27:         # print(self.init_lr)
  28:         self._optimizer.zero_grad()
  29: 
  30:     def load_state_dict(self, path):
...
  21: 
  22:     def step_and_update_lr(self):
  23:         

In [13]:
# ================================================================
# TELUGU FASTSPEECH2 — CONSOLIDATED TRAINING FORENSIC AUDIT
# RUN NOW — READ ONLY
# ================================================================

from pathlib import Path
import os
import json
import yaml
import torch
import numpy as np
import hashlib

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
TE = ROOT / "Telugu"
CODE = ROOT / "Code" / "FastSpeech2"
CKPT = TE / "checkpoints"

print("=" * 80)
print("TELUGU FASTSPEECH2 — COMPLETE TRAINING INVESTIGATION")
print("=" * 80)

# ------------------------------------------------
# 1. COMPLETE OPTIMIZER SOURCE
# ------------------------------------------------

print("\n[1] OPTIMIZER IMPLEMENTATION")

optimizer_file = CODE / "model" / "optimizer.py"

if optimizer_file.exists():
    print(optimizer_file.read_text(errors="replace"))
else:
    print("optimizer.py NOT FOUND")

# ------------------------------------------------
# 2. FIND ALL TRAINING SCRIPTS AND NOTEBOOKS
# ------------------------------------------------

print("\n[2] SEARCHING FOR ACTUAL TRAINING IMPLEMENTATION")

search_roots = [
    ROOT / "Code",
    ROOT / "Telugu",
    ROOT / "metadata"
]

terms = [
    "optimizer.step",
    "step_and_update_lr",
    "loss.backward",
    "total_loss.backward",
    "history.append",
    "torch.save",
    "restore_step",
    "load_state_dict"
]

for base in search_roots:
    if not base.exists():
        continue

    for file in base.rglob("*"):
        if not file.is_file():
            continue

        if file.suffix.lower() not in [".py", ".ipynb"]:
            continue

        try:
            content = file.read_text(errors="replace")
        except Exception:
            continue

        matched = [term for term in terms if term in content]

        if matched:
            print("\nFILE:", file.relative_to(ROOT))
            print("MATCHES:", matched)

            lines = content.splitlines()
            for i, line in enumerate(lines):
                if any(term in line for term in terms):
                    print(f"{i+1}: {line.strip()[:250]}")

# ------------------------------------------------
# 3. LOAD CHECKPOINTS
# ------------------------------------------------

print("\n[3] CHECKPOINT METADATA")

checkpoints = {}

for step in [300, 400, 500, 600]:
    path = CKPT / f"{step}.pth.tar"

    if not path.exists():
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    checkpoints[step] = ckpt

    print(f"\nCHECKPOINT {step}")
    print("Saved step:", ckpt.get("step"))
    print("Keys:", list(ckpt.keys()))
    print("History records:", len(ckpt.get("history", [])))

    optimizer = ckpt.get("optimizer", {})
    state = optimizer.get("state", {})
    groups = optimizer.get("param_groups", [])

    print("Optimizer state entries:", len(state))

    if groups:
        group = groups[0]
        print("Learning rate:", group.get("lr"))
        print("Betas:", group.get("betas"))
        print("Epsilon:", group.get("eps"))

    adam_steps = []

    for entry in state.values():
        if "step" in entry:
            adam_steps.append(float(entry["step"].item()))

    print("Adam step values:", sorted(set(adam_steps)))

# ------------------------------------------------
# 4. COMPARE TRAINABLE PARAMETERS
# ------------------------------------------------

print("\n[4] MODEL PARAMETER COMPARISON")

def tensor_hash(t):
    return hashlib.sha256(
        t.detach().cpu().contiguous().numpy().tobytes()
    ).hexdigest()

available_steps = sorted(checkpoints.keys())

if len(available_steps) >= 2:
    reference_step = available_steps[0]
    reference = checkpoints[reference_step]["model"]

    for step in available_steps[1:]:
        current = checkpoints[step]["model"]

        changed = []
        unchanged = []
        missing = []

        for name, tensor in reference.items():
            if name not in current:
                missing.append(name)
                continue

            if tensor_hash(tensor) == tensor_hash(current[name]):
                unchanged.append(name)
            else:
                changed.append(name)

        print(f"\n{reference_step} -> {step}")
        print("Changed tensors:", len(changed))
        print("Unchanged tensors:", len(unchanged))
        print("Missing tensors:", len(missing))

        if changed:
            print("Changed tensor examples:", changed[:10])

        if missing:
            print("Missing examples:", missing[:10])

# ------------------------------------------------
# 5. COMPARE ADAM STATES
# ------------------------------------------------

print("\n[5] ADAM OPTIMIZER STATE COMPARISON")

for a, b in [(300, 500), (500, 600)]:
    if a not in checkpoints or b not in checkpoints:
        continue

    sa = checkpoints[a]["optimizer"]["state"]
    sb = checkpoints[b]["optimizer"]["state"]

    changed_states = 0
    unchanged_states = 0
    differences = []

    for key in sa:
        if key not in sb:
            differences.append((key, "missing"))
            continue

        state_a = sa[key]
        state_b = sb[key]

        same = True

        for field in state_a:
            if field not in state_b:
                same = False
                continue

            va, vb = state_a[field], state_b[field]

            if torch.is_tensor(va) and torch.is_tensor(vb):
                if not torch.equal(va, vb):
                    same = False
                    differences.append((key, field))
            elif va != vb:
                same = False
                differences.append((key, field))

        if same:
            unchanged_states += 1
        else:
            changed_states += 1

    print(f"\n{a} -> {b}")
    print("Changed optimizer states:", changed_states)
    print("Unchanged optimizer states:", unchanged_states)
    print("Differences:", differences[:15])

# ------------------------------------------------
# 6. INSPECT TRAINING HISTORY
# ------------------------------------------------

print("\n[6] TRAINING HISTORY")

for step in [300, 500, 600]:
    if step not in checkpoints:
        continue

    history = checkpoints[step].get("history", [])

    print(f"\nCheckpoint {step}: {len(history)} history records")

    for record in history[-5:]:
        print(record)

# ------------------------------------------------
# 7. INSPECT TELUGU CONFIGURATIONS
# ------------------------------------------------

print("\n[7] TELUGU CONFIGURATION FILES")

for filename in ["train.yaml", "model.yaml", "preprocess.yaml"]:
    path = TE / "config" / filename

    print("\n---", filename, "---")

    if path.exists():
        print(path.read_text(errors="replace"))
    else:
        print("MISSING")

# ------------------------------------------------
# 8. INSPECT LOG FILES
# ------------------------------------------------

print("\n[8] TRAINING LOG FILES")

log_root = TE / "logs"

if log_root.exists():
    for file in log_root.rglob("*"):
        if file.is_file() and file.suffix.lower() in [
            ".txt", ".csv", ".json"
        ]:
            print("\nFILE:", file.relative_to(ROOT))

            try:
                content = file.read_text(errors="replace")
                print(content[-5000:])
            except Exception as e:
                print("Could not read:", e)

# ------------------------------------------------
# 9. FINAL DIAGNOSTIC SUMMARY
# ------------------------------------------------

print("\n" + "=" * 80)
print("INVESTIGATION COMPLETE")
print("=" * 80)

print("Checkpoints found:", available_steps)
print("No checkpoint was modified.")
print("No training was started.")
print("No files were deleted.")

TELUGU FASTSPEECH2 — COMPLETE TRAINING INVESTIGATION

[1] OPTIMIZER IMPLEMENTATION
import torch
import numpy as np


class ScheduledOptim:
    """ A simple wrapper class for learning rate scheduling """

    def __init__(self, model, train_config, model_config, current_step):

        self._optimizer = torch.optim.Adam(
            model.parameters(),
            betas=train_config["optimizer"]["betas"],
            eps=train_config["optimizer"]["eps"],
            weight_decay=train_config["optimizer"]["weight_decay"],
        )
        self.n_warmup_steps = train_config["optimizer"]["warm_up_step"]
        self.anneal_steps = train_config["optimizer"]["anneal_steps"]
        self.anneal_rate = train_config["optimizer"]["anneal_rate"]
        self.current_step = current_step
        self.init_lr = np.power(model_config["transformer"]["encoder_hidden"], -0.5)

    def step_and_update_lr(self):
        self._update_learning_rate()
        self._optimizer.step()

    def zero_grad(self):

KeyError: slice(-5, None, None)

In [14]:
# ================================================================
# TELUGU FASTSPEECH2 — FINAL TRAINING CODE INSPECTION
# READ ONLY: does not modify checkpoints or source files
# ================================================================

from pathlib import Path
import torch
import yaml
import json

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
TE = ROOT / "Telugu"
CODE = ROOT / "Code" / "FastSpeech2"

# 1. Correctly inspect checkpoint history
print("=" * 75)
print("1. CHECKPOINT HISTORY STRUCTURE")
print("=" * 75)

for step in [300, 400, 500, 600]:
    p = TE / "checkpoints" / f"{step}.pth.tar"
    if not p.exists():
        continue

    ckpt = torch.load(p, map_location="cpu", weights_only=False)
    history = ckpt.get("history")

    print(f"\nCHECKPOINT {step}")
    print("History type:", type(history).__name__)

    if isinstance(history, dict):
        print("History keys:", list(history.keys()))
        for key, value in history.items():
            print(f"\n{key}: type={type(value).__name__}")
            if isinstance(value, (list, tuple)):
                print("Count:", len(value))
                print("Last values:", value[-5:])
            elif isinstance(value, dict):
                print("Contents:", value)
            else:
                print("Value:", value)

    elif isinstance(history, (list, tuple)):
        print("Record count:", len(history))
        for item in history[-5:]:
            print(item)
    else:
        print("History:", history)

# 2. Inspect optimizer source completely
print("\n" + "=" * 75)
print("2. OPTIMIZER SOURCE")
print("=" * 75)

p = CODE / "model" / "optimizer.py"
if p.exists():
    print(p.read_text(errors="replace"))

# 3. Inspect model source for gradient-related operations
print("\n" + "=" * 75)
print("3. MODEL GRADIENT / DETACH INSPECTION")
print("=" * 75)

model_root = CODE / "model"

keywords = [
    "detach(", ".detach",
    "torch.no_grad",
    "requires_grad",
    "stop_gradient",
    "optimizer",
    "backward"
]

for p in model_root.rglob("*.py"):
    lines = p.read_text(errors="replace").splitlines()
    matches = [
        (i, line) for i, line in enumerate(lines)
        if any(k in line for k in keywords)
    ]

    if matches:
        print("\nFILE:", p.relative_to(CODE))
        for i, line in matches:
            print(f"{i+1}: {line.strip()[:250]}")

# 4. Inspect actual saved training logs
print("\n" + "=" * 75)
print("4. TELUGU LOG FILES")
print("=" * 75)

for p in (TE / "logs").rglob("*"):
    if p.is_file() and p.suffix.lower() in [".json", ".csv", ".txt"]:
        print("\nFILE:", p.relative_to(ROOT))
        try:
            content = p.read_text(errors="replace")
            print(content[-3500:])
        except Exception as e:
            print("Read error:", e)

# 5. Inspect model and preprocessing configuration
print("\n" + "=" * 75)
print("5. MODEL CONFIGURATION")
print("=" * 75)

for name in ["model.yaml", "preprocess.yaml", "train.yaml"]:
    p = TE / "config" / name
    print(f"\n{name}")
    if p.exists():
        print(p.read_text(errors="replace"))

print("\n" + "=" * 75)
print("INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 75)

1. CHECKPOINT HISTORY STRUCTURE

CHECKPOINT 300
History type: dict
History keys: ['step', 'train_total', 'train_mel', 'train_postnet_mel', 'train_pitch', 'train_energy', 'train_duration', 'validation_total', 'validation_mel', 'validation_postnet_mel', 'validation_pitch', 'validation_energy', 'validation_duration']

step: type=list
Count: 3
Last values: [100, 200, 300]

train_total: type=list
Count: 3
Last values: [11.093661308288574, 8.841996192932129, 6.579644680023193]

train_mel: type=list
Count: 3
Last values: [4.779876708984375, 3.9302239418029785, 2.7016289234161377]

train_postnet_mel: type=list
Count: 3
Last values: [4.743826866149902, 3.834397554397583, 2.523439645767212]

train_pitch: type=list
Count: 3
Last values: [0.7721270322799683, 0.5067895650863647, 0.7235786318778992]

train_energy: type=list
Count: 3
Last values: [0.5193660259246826, 0.339690625667572, 0.39245596528053284]

train_duration: type=list
Count: 3
Last values: [0.2784644067287445, 0.23089446127414703, 0.23

In [15]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

print("=" * 65)
print("SEARCHING FOR ORIGINAL TRAINING NOTEBOOKS")
print("=" * 65)

for p in DRIVE.rglob("*.ipynb"):
    if any(x in str(p).lower() for x in [
        "fastspeech", "telugu", "speech", "tts"
    ]):
        print(p)

print("\n" + "=" * 65)
print("SEARCHING FOR CUSTOM TRAINING SCRIPTS")
print("=" * 65)

ROOT = DRIVE / "FastSpeech2_Assignment"

for p in ROOT.rglob("*.py"):
    if any(x in str(p).lower() for x in [
        "train", "finetune", "resume", "telugu"
    ]):
        print(p)

print("\nSEARCH COMPLETE — NO FILES MODIFIED")

SEARCHING FOR ORIGINAL TRAINING NOTEBOOKS
/content/drive/MyDrive/Colab Notebooks/FastSpeech2_Assignment.ipynb

SEARCHING FOR CUSTOM TRAINING SCRIPTS
/content/drive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2/train.py

SEARCH COMPLETE — NO FILES MODIFIED


In [16]:
# ================================================================
# INSPECT ORIGINAL FASTSPEECH2 ASSIGNMENT NOTEBOOK
# READ ONLY — DOES NOT MODIFY THE NOTEBOOK OR CHECKPOINTS
# ================================================================

import json
from pathlib import Path

NB_PATH = Path(
    "/content/drive/MyDrive/Colab Notebooks/FastSpeech2_Assignment.ipynb"
)

assert NB_PATH.exists(), f"Notebook not found: {NB_PATH}"

with open(NB_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

print("Notebook:", NB_PATH.name)
print("Total cells:", len(cells))

keywords = [
    "optimizer.step",
    "optimizer.step_and_update_lr",
    "backward()",
    "zero_grad",
    "checkpoint",
    "history",
    "train_total",
    "torch.save",
    "restore_step",
    "training loop",
    "for step",
    "for epoch"
]

print("\n" + "=" * 80)
print("RELEVANT NOTEBOOK CELLS")
print("=" * 80)

for i, cell in enumerate(cells):
    source = "".join(cell.get("source", []))
    if not source.strip():
        continue

    matches = [
        k for k in keywords
        if k.lower() in source.lower()
    ]

    if matches:
        print(f"\n{'=' * 30} CELL {i} {'=' * 30}")
        print("Cell type:", cell.get("cell_type"))
        print("Matched:", matches)
        print(source[:12000])

print("\n" + "=" * 80)
print("NOTEBOOK INSPECTION COMPLETE")
print("No notebook cells were executed or modified.")
print("=" * 80)

Notebook: FastSpeech2_Assignment.ipynb
Total cells: 379

RELEVANT NOTEBOOK CELLS

============================== CELL 70 ==============================
Cell type: code
Matched: ['checkpoint']
import torch
import os

vocoder_dir = "/content/FastSpeech2/hifigan"

for filename in [
    "generator_LJSpeech.pth.tar",
    "generator_universal.pth.tar"
]:
    path = os.path.join(vocoder_dir, filename)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    print(f"\n{filename}")
    print("Checkpoint keys:", checkpoint.keys())

    if "generator" in checkpoint:
        weights = checkpoint["generator"]
        print("Number of generator parameters:", len(weights))
        first_key = next(iter(weights))
        print("First parameter:", first_key)
        print("Parameter shape:", weights[first_key].shape)

============================== CELL 79 ==============================
Cell type: code
Matched: ['checkpoint']
model_path = "/content/Fa

In [17]:
# RUN NOW — READ ONLY
from pathlib import Path

ROOT = Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
CODE = ROOT / "Code/FastSpeech2"

files = [
    CODE / "model/optimizer.py",
    CODE / "model/loss.py",
    CODE / "train.py",
]

for path in files:
    print("\n" + "=" * 75)
    print("FILE:", path)
    print("EXISTS:", path.exists())
    print("=" * 75)

    if path.exists():
        lines = path.read_text(errors="replace").splitlines()
        for i, line in enumerate(lines, 1):
            print(f"{i:4}: {line}")



FILE: /content/gdrive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2/model/optimizer.py
EXISTS: False

FILE: /content/gdrive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2/model/loss.py
EXISTS: False

FILE: /content/gdrive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2/train.py
EXISTS: False


In [18]:
import json
from pathlib import Path

NB = Path("/content/drive/MyDrive/Colab Notebooks/FastSpeech2_Assignment.ipynb")

assert NB.exists(), f"Notebook not found: {NB}"

with open(NB, encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if any(term in source for term in [
        "STEP 76A",
        "STEP 93",
        "RESUME TRAINING"
    ]):
        print("\n" + "=" * 80)
        print(f"NOTEBOOK CELL {i}")
        print("=" * 80)
        print(source)


NOTEBOOK CELL 198
# ================================================================
# STEP 76A: TELUGU FASTSPEECH 2 TRAINING
# CPU TRAINING + VALIDATION + CHECKPOINTS + LOSS PLOTS
# ================================================================

import os
import sys
import json
import yaml
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from pathlib import Path

# ------------------------------------------------
# 1. PATHS AND SETTINGS
# ------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

TRAIN_DIR = BASE / "Telugu/fs2_prepared/train"
VAL_DIR = BASE / "Telugu/fs2_prepared/validation"
META_DIR = BASE / "Telugu/fs2_prepared/model_metadata"

CKPT_DIR = BASE / "Telugu/checkpoints"
LOG_DIR = BASE / "Telugu/logs"
RESULT_DIR = BASE / "Telugu/results"

for folder in [CKPT_DIR, LOG_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MAX_STEPS = 300
VALI

In [23]:
# ================================================================
# INSTALL MISSING DEPENDENCY + TEST FASTSPEECH2 IMPORTS
# ================================================================

%pip install -q Unidecode

import sys
from pathlib import Path

REPO = Path(
    "/content/drive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2"
)

assert (REPO / "model/fastspeech2.py").exists()

if str(REPO) in sys.path:
    sys.path.remove(str(REPO))

sys.path.insert(0, str(REPO))

from model.fastspeech2 import FastSpeech2
from model.loss import FastSpeech2Loss

print("SUCCESS: FastSpeech2 imports are working.")
print("Repository:", REPO)

SUCCESS: FastSpeech2 imports are working.
Repository: /content/drive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2


In [24]:
# ================================================================
# TELUGU RECOVERY TRAINING — START FROM CHECKPOINT 300
# Fresh model + fresh optimizer + verified parameter updates
# RUN NOW — DO NOT RERUN STEP 76A OR STEP 93
# ================================================================

import json, random, yaml, copy
import numpy as np
import torch
from pathlib import Path

from model.fastspeech2 import FastSpeech2
from model.loss import FastSpeech2Loss

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = Path("/content/FastSpeech2")

TRAIN_DIR = BASE / "Telugu/fs2_prepared/train"
VAL_DIR = BASE / "Telugu/fs2_prepared/validation"
META_DIR = BASE / "Telugu/fs2_prepared/model_metadata"

SOURCE_CKPT = BASE / "Telugu/checkpoints/300.pth.tar"

RECOVERY_DIR = BASE / "Telugu/recovery_training"
CKPT_DIR = RECOVERY_DIR / "checkpoints"
LOG_DIR = RECOVERY_DIR / "logs"
RESULT_DIR = RECOVERY_DIR / "results"

for folder in [CKPT_DIR, LOG_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

assert SOURCE_CKPT.exists(), "Original step-300 checkpoint missing!"
assert len(list(TRAIN_DIR.glob("*.npz"))) == 1520
assert len(list(VAL_DIR.glob("*.npz"))) == 301

# ------------------------------------------------
# 2. CONFIGURATION
# ------------------------------------------------

with open(BASE / "Telugu/config/preprocess.yaml") as f:
    preprocess_config = yaml.safe_load(f)

with open(BASE / "Telugu/config/model.yaml") as f:
    model_config = yaml.safe_load(f)

with open(META_DIR / "speakers.json") as f:
    speaker_map = json.load(f)

with open(META_DIR / "stats.json") as f:
    stats = json.load(f)

# The Telugu model uses 361 input tokens.
model_config["input_vocabulary_size"] = 361

norm = stats["normalization_parameters"]

pitch_mean = norm["pitch_mean"]
pitch_std = norm["pitch_std"]
energy_mean = norm["energy_mean"]
energy_std = norm["energy_std"]

device = torch.device("cpu")
torch.set_num_threads(2)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

train_files = sorted(TRAIN_DIR.glob("*.npz"))
val_files = sorted(VAL_DIR.glob("*.npz"))

print("Training chunks:", len(train_files))
print("Validation chunks:", len(val_files))
print("Device:", device)

# ------------------------------------------------
# 3. SAMPLE LOADER
# ------------------------------------------------

def load_sample(path):
    with np.load(path, allow_pickle=True) as data:
        original_id = str(data["original_id"].item()).lower()

        mel = data["mel"].astype(np.float32).T
        pitch = data["pitch"].astype(np.float32)
        energy = data["energy"].astype(np.float32)

        text = data["text_ids"].astype(np.int64)
        duration = data["durations"].astype(np.int64)

    if "female" in original_id:
        speaker_id = speaker_map["female"]
    elif "male" in original_id:
        speaker_id = speaker_map["male"]
    else:
        raise ValueError(f"Unknown speaker: {original_id}")

    pitch = (pitch - pitch_mean) / pitch_std
    energy = (energy - energy_mean) / energy_std

    assert mel.shape[0] == len(pitch) == len(energy)
    assert len(text) == len(duration)
    assert int(duration.sum()) == mel.shape[0]

    assert np.isfinite(mel).all()
    assert np.isfinite(pitch).all()
    assert np.isfinite(energy).all()

    return {
        "speaker": speaker_id,
        "text": text,
        "mel": mel,
        "pitch": pitch,
        "energy": energy,
        "duration": duration
    }


def make_batch(path):
    s = load_sample(path)

    return (
        [path.stem],
        [""],
        torch.tensor([s["speaker"]], dtype=torch.long),
        torch.tensor(s["text"][None, :], dtype=torch.long),
        torch.tensor([len(s["text"])], dtype=torch.long),
        len(s["text"]),
        torch.tensor(s["mel"][None], dtype=torch.float32),
        torch.tensor([s["mel"].shape[0]], dtype=torch.long),
        s["mel"].shape[0],
        torch.tensor(s["pitch"][None], dtype=torch.float32),
        torch.tensor(s["energy"][None], dtype=torch.float32),
        torch.tensor(s["duration"][None], dtype=torch.long)
    )

# ------------------------------------------------
# 4. CREATE A COMPLETELY FRESH MODEL AND OPTIMIZER
# ------------------------------------------------

model = FastSpeech2(
    preprocess_config,
    model_config
).to(device)

criterion = FastSpeech2Loss(
    preprocess_config,
    model_config
).to(device)

# Load MODEL WEIGHTS ONLY from the trusted step-300 checkpoint.
checkpoint = torch.load(
    SOURCE_CKPT,
    map_location=device,
    weights_only=False
)

assert checkpoint["step"] == 300

model.load_state_dict(checkpoint["model"], strict=True)

# Fresh optimizer: no stale parameter references or old Adam state.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
    weight_decay=0.0
)

# Verify optimizer and model reference exactly the same parameters.
model_params = {
    id(p) for p in model.parameters() if p.requires_grad
}
optimizer_params = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
    if p.requires_grad
}

assert model_params == optimizer_params, \
    "Optimizer is not connected to the current model!"

print("Model restored from step 300.")
print("Fresh optimizer created.")
print("Optimizer/model parameter references match.")

# ------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------

def run_validation():
    model.eval()
    values = []

    selected = random.sample(
        val_files,
        min(24, len(val_files))
    )

    with torch.no_grad():
        for path in selected:
            batch = make_batch(path)
            predictions = model(*(batch[2:]))
            losses = criterion(batch, predictions)

            values.append([
                float(x.detach().item()) for x in losses
            ])

    model.train()
    return np.mean(values, axis=0).tolist()

# ------------------------------------------------
# 6. VERIFY A REAL PARAMETER UPDATE BEFORE CONTINUING
# ------------------------------------------------

model.train()

first_path = random.choice(train_files)
first_batch = make_batch(first_path)

optimizer.zero_grad(set_to_none=True)

first_predictions = model(*(first_batch[2:]))
first_losses = criterion(first_batch, first_predictions)

# Match the original STEP 76A loss convention.
first_total_loss = first_losses[0]

assert torch.isfinite(first_total_loss), "Non-finite loss!"

first_total_loss.backward()

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(), max_norm=1.0
)

assert torch.isfinite(grad_norm), "Non-finite gradient norm!"
assert grad_norm.item() > 0, "No gradients detected!"

probe = next(
    p for p in model.parameters()
    if p.requires_grad and p.grad is not None
)

before_update = probe.detach().clone()
optimizer.step()
after_update = probe.detach().clone()

parameter_change = (after_update - before_update).abs().max().item()

assert parameter_change > 0, (
    "Optimizer step did not change the model. "
    "Stopping before the training run."
)

print("\nFIRST UPDATE VERIFIED")
print("Gradient norm:", float(grad_norm))
print("Maximum parameter change:", parameter_change)

# ------------------------------------------------
# 7. RECOVERY TRAINING — 299 MORE STEPS
# ------------------------------------------------

history = {
    "step": [],
    "train_total": [],
    "train_mel": [],
    "train_postnet_mel": [],
    "train_pitch": [],
    "train_energy": [],
    "train_duration": [],
    "val_total": [],
    "val_mel": [],
    "val_postnet_mel": [],
    "val_pitch": [],
    "val_energy": [],
    "val_duration": []
}

def save_recovery_checkpoint(step):
    payload = {
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "history": history,
        "speaker_map": speaker_map,
        "normalization": norm
    }

    torch.save(
        payload,
        CKPT_DIR / f"recovery_{step}.pth.tar"
    )

    torch.save(payload, CKPT_DIR / "latest.pth.tar")

    with open(LOG_DIR / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print("Saved recovery checkpoint:", step)


model.train()

# The first verified update represents step 301.
for step in range(302, 601):

    path = random.choice(train_files)
    batch = make_batch(path)

    optimizer.zero_grad(set_to_none=True)

    predictions = model(*(batch[2:]))
    losses = criterion(batch, predictions)

    total_loss = losses[0]

    if not torch.isfinite(total_loss):
        raise RuntimeError(f"Non-finite loss at step {step}")

    total_loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=1.0
    )

    if not torch.isfinite(grad_norm):
        raise RuntimeError(f"Non-finite gradient at step {step}")

    optimizer.step()

    if step % 10 == 0:
        print(
            f"Step {step}/600 | "
            f"Total loss: {total_loss.item():.4f} | "
            f"Gradient norm: {grad_norm.item():.4f}"
        )

    if step % 100 == 0:

        train_values = [
            float(x.detach().item()) for x in losses
        ]

        val_values = run_validation()

        history["step"].append(step)

        names = [
            "total", "mel", "postnet_mel",
            "pitch", "energy", "duration"
        ]

        for i, name in enumerate(names):
            if name == "total":
                history["train_total"].append(train_values[i])
                history["val_total"].append(sum(val_values))
            else:
                history[f"train_{name}"].append(train_values[i])
                history[f"val_{name}"].append(val_values[i])

        print("\nVALIDATION:", step)
        print("Training total:", train_values[0])
        print("Validation total:", sum(val_values))

        save_recovery_checkpoint(step)

# Save the final checkpoint.
save_recovery_checkpoint(600)

print("\nRECOVERY TRAINING FINISHED")
print("Recovery checkpoints:", CKPT_DIR)
print("Recovery logs:", LOG_DIR)

Training chunks: 1520
Validation chunks: 301
Device: cpu
Model restored from step 300.
Fresh optimizer created.
Optimizer/model parameter references match.

FIRST UPDATE VERIFIED
Gradient norm: 4.329444408416748
Maximum parameter change: 0.00010001659393310547
Step 310/600 | Total loss: 6.0245 | Gradient norm: 6.8021
Step 320/600 | Total loss: 6.3477 | Gradient norm: 5.5142
Step 330/600 | Total loss: 4.7009 | Gradient norm: 6.2493
Step 340/600 | Total loss: 5.5872 | Gradient norm: 14.8853
Step 350/600 | Total loss: 6.0005 | Gradient norm: 6.5368
Step 360/600 | Total loss: 5.1436 | Gradient norm: 8.7235
Step 370/600 | Total loss: 4.7349 | Gradient norm: 6.6535
Step 380/600 | Total loss: 5.2480 | Gradient norm: 12.3472
Step 390/600 | Total loss: 4.8894 | Gradient norm: 3.7053
Step 400/600 | Total loss: 5.2937 | Gradient norm: 4.8112

VALIDATION: 400
Training total: 5.293701648712158
Validation total: 9.630045550254485
Saved recovery checkpoint: 400
Step 410/600 | Total loss: 5.2444 | Gra

In [25]:
# ================================================================
# TELUGU RECOVERY CHECKPOINT AUDIT — READ ONLY
# ================================================================

import torch
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
CKPT_DIR = BASE / "Telugu/recovery_training/checkpoints"

paths = {
    step: CKPT_DIR / f"recovery_{step}.pth.tar"
    for step in [400, 500, 600]
}

checkpoints = {}

for step, path in paths.items():
    assert path.exists(), f"Missing checkpoint: {path}"
    checkpoints[step] = torch.load(
        path, map_location="cpu", weights_only=False
    )

    print(
        f"Step {step}: checkpoint step={checkpoints[step]['step']}, "
        f"optimizer states={len(checkpoints[step]['optimizer']['state'])}"
    )

def compare_weights(a, b):
    state_a = checkpoints[a]["model"]
    state_b = checkpoints[b]["model"]

    changed = []
    unchanged = []

    for key in state_a:
        if torch.is_tensor(state_a[key]):
            if not torch.equal(state_a[key], state_b[key]):
                changed.append(key)
            else:
                unchanged.append(key)

    return changed, unchanged

for a, b in [(400, 500), (500, 600)]:
    changed, unchanged = compare_weights(a, b)
    print(f"\n{a} -> {b}")
    print("Changed tensors:", len(changed))
    print("Unchanged tensors:", len(unchanged))
    print("Examples:", changed[:5])

# Check optimizer step counters
for step, ckpt in checkpoints.items():
    optimizer_state = ckpt["optimizer"]["state"]
    adam_steps = [
        int(state["step"].item())
        for state in optimizer_state.values()
        if "step" in state
    ]
    print(
        f"\nOptimizer at checkpoint {step}:",
        "min step =", min(adam_steps) if adam_steps else None,
        "max step =", max(adam_steps) if adam_steps else None
    )

print("\nRECOVERY CHECKPOINT AUDIT COMPLETE")

Step 400: checkpoint step=400, optimizer states=120
Step 500: checkpoint step=500, optimizer states=120
Step 600: checkpoint step=600, optimizer states=120

400 -> 500
Changed tensors: 135
Unchanged tensors: 4
Examples: ['encoder.src_word_emb.weight', 'encoder.layer_stack.0.slf_attn.w_qs.weight', 'encoder.layer_stack.0.slf_attn.w_qs.bias', 'encoder.layer_stack.0.slf_attn.w_ks.weight', 'encoder.layer_stack.0.slf_attn.w_ks.bias']

500 -> 600
Changed tensors: 135
Unchanged tensors: 4
Examples: ['encoder.src_word_emb.weight', 'encoder.layer_stack.0.slf_attn.w_qs.weight', 'encoder.layer_stack.0.slf_attn.w_qs.bias', 'encoder.layer_stack.0.slf_attn.w_ks.weight', 'encoder.layer_stack.0.slf_attn.w_ks.bias']

Optimizer at checkpoint 400: min step = 100 max step = 100

Optimizer at checkpoint 500: min step = 200 max step = 200

Optimizer at checkpoint 600: min step = 300 max step = 300

RECOVERY CHECKPOINT AUDIT COMPLETE


In [26]:
# ================================================================
# TELUGU INFERENCE READINESS CHECK — READ ONLY
# ================================================================

from pathlib import Path
import json
import torch

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
TE = BASE / "Telugu"

ckpt = TE / "recovery_training/checkpoints/recovery_600.pth.tar"

assert ckpt.exists(), "Recovery checkpoint 600 is missing."

checkpoint = torch.load(
    ckpt, map_location="cpu", weights_only=False
)

print("CHECKPOINT STEP:", checkpoint["step"])
print("MODEL TENSORS:", len(checkpoint["model"]))
print("OPTIMIZER STATES:", len(checkpoint["optimizer"]["state"]))
print("SPEAKER MAP:", checkpoint["speaker_map"])
print("NORMALIZATION:", checkpoint["normalization"])

sentence_file = TE / "inference/test_sentences.json"
print("\nTest sentences file exists:", sentence_file.exists())

if sentence_file.exists():
    with open(sentence_file) as f:
        sentences = json.load(f)
    print(json.dumps(sentences, indent=2, ensure_ascii=False))

print("\nRECOVERY INFERENCE READINESS CHECK COMPLETE")

CHECKPOINT STEP: 600
MODEL TENSORS: 139
OPTIMIZER STATES: 120
SPEAKER MAP: {'female': 0, 'male': 1}
NORMALIZATION: {'pitch_mean': 109.76953432377077, 'pitch_std': 103.30357441258384, 'energy_mean': 0.05246112019682791, 'energy_std': 0.05535232817865627}

Test sentences file exists: True
{
  "T1": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
  "T2a": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
  "T2b": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
  "T3": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
  "T4": "ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి."
}

RECOVERY INFERENCE READINESS CHECK COMPLETE


In [27]:
# ================================================================
# LOAD TELUGU RECOVERY MODEL — CHECKPOINT 600
# Does not modify existing checkpoints or inference results
# ================================================================

import sys
import yaml
import torch
from pathlib import Path

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = BASE / "Code/FastSpeech2"
TE = BASE / "Telugu"

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from model.fastspeech2 import FastSpeech2

device = torch.device("cpu")

with open(TE / "config/preprocess.yaml") as f:
    telugu_preprocess_config = yaml.safe_load(f)

with open(TE / "config/model.yaml") as f:
    telugu_model_config = yaml.safe_load(f)

telugu_model_config["input_vocabulary_size"] = 361

RECOVERY_CKPT = (
    TE / "recovery_training/checkpoints/recovery_600.pth.tar"
)

checkpoint = torch.load(
    RECOVERY_CKPT,
    map_location=device,
    weights_only=False
)

assert checkpoint["step"] == 600

# Create a separate model instance.
telugu_recovery_model = FastSpeech2(
    telugu_preprocess_config,
    telugu_model_config
).to(device)

result = telugu_recovery_model.load_state_dict(
    checkpoint["model"],
    strict=True
)

telugu_recovery_model.eval()

print("TELUGU RECOVERY MODEL LOADED")
print("Checkpoint step:", checkpoint["step"])
print("Model parameters:", sum(
    p.numel() for p in telugu_recovery_model.parameters()
))
print("Device:", device)
print("Strict loading:", result)
print("Existing telugu_model preserved:", "telugu_model" in globals())

TELUGU RECOVERY MODEL LOADED
Checkpoint step: 600
Model parameters: 8183361
Device: cpu
Strict loading: <All keys matched successfully>
Existing telugu_model preserved: False


In [28]:
# ================================================================
# FIND ORIGINAL TELUGU INFERENCE CELLS — READ ONLY
# ================================================================

import json
from pathlib import Path

NB = Path(
    "/content/drive/MyDrive/Colab Notebooks/FastSpeech2_Assignment.ipynb"
)

assert NB.exists(), f"Notebook not found: {NB}"

with open(NB, encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "test_sentences.json",
    "test_sentence_token_mapping",
    "uroman",
    "T1_predictions",
    "telugu_inference_summary",
    "TELUGU INFERENCE",
    "telugu_model",
]

matches = []

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))
    hits = [k for k in keywords if k.lower() in source.lower()]

    if hits:
        matches.append((i, hits, source))

print("MATCHING CELLS:", len(matches))

for i, hits, source in matches:
    print("\n" + "=" * 80)
    print("CELL:", i)
    print("MATCHES:", hits)
    print("=" * 80)
    print(source)

MATCHING CELLS: 35

CELL: 149
MATCHES: ['uroman']
import os
import shutil
import importlib.util
import torch
import torchaudio

# Available system memory
try:
    import psutil
    memory = psutil.virtual_memory()

    print("Total RAM (GB):", round(memory.total / (1024**3), 2))
    print("Available RAM (GB):", round(memory.available / (1024**3), 2))
except ImportError:
    print("psutil is not installed")

# Available disk space
disk = shutil.disk_usage("/content")

print("\nAvailable disk space (GB):", round(disk.free / (1024**3), 2))

# Check whether uroman is already available
print("\nuroman package installed:",
      importlib.util.find_spec("uroman") is not None)

print("TorchAudio:", torchaudio.__version__)
print("MMS_FA sample rate:", torchaudio.pipelines.MMS_FA.sample_rate)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

CELL: 150
MATCHES: ['uroman']
import subprocess
from pathlib import Path
import pandas as pd
import shutil

# Clone the official Uroman rep

In [30]:
# ================================================================
# TELUGU RECOVERY INFERENCE — CHECKPOINT 600
# 5 AUDIOS + 15 PLOTS + 5 PREDICTION FILES
#
# Uses:
#   - telugu_recovery_model (recovery checkpoint 600)
#   - Original Uroman romanization
#   - Original 361-token vocabulary
#   - Existing universal HiFi-GAN vocoder
#
# Saves to a NEW recovery folder.
# Does not overwrite original Telugu inference results.
# ================================================================

import os
import sys
import re
import json
import subprocess
import numpy as np
import torch
import matplotlib.pyplot as plt
import soundfile as sf

from pathlib import Path
from datetime import datetime

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
TE = BASE / "Telugu"
REPO = TE / "PLACEHOLDER"

# Actual source repository in Drive
REPO = BASE / "Code/FastSpeech2"

RECOVERY = TE / "recovery_training"

AUDIO_DIR = RECOVERY / "inference/test_generated_audios"
PRED_DIR = RECOVERY / "inference/test_predictions"
DURATION_DIR = RECOVERY / "inference/plots/duration"
PITCH_DIR = RECOVERY / "inference/plots/pitch"
ENERGY_DIR = RECOVERY / "inference/plots/energy"
LOG_DIR = RECOVERY / "inference/logs"

for folder in [
    AUDIO_DIR, PRED_DIR, DURATION_DIR,
    PITCH_DIR, ENERGY_DIR, LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
SAMPLE_RATE = 22050
SPEAKER_ID = 0  # Female speaker, matching original STEP 85

assert REPO.exists(), f"Repository not found: {REPO}"
assert (REPO / "hifigan/config.json").exists()
assert (TE / "vocoder/generator_universal/generator_universal.pth.tar").exists()

# ------------------------------------------------
# 2. VERIFY RECOVERY MODEL
# ------------------------------------------------

assert "telugu_recovery_model" in globals(), (
    "telugu_recovery_model is not loaded. "
    "Load recovery checkpoint 600 first."
)

model = telugu_recovery_model.to(DEVICE)
model.eval()

print("FastSpeech2 recovery model ready.")
print("Checkpoint: 600")

# ------------------------------------------------
# 3. LOAD ORIGINAL VOCABULARY
# ------------------------------------------------

if str(REPO) in sys.path:
    sys.path.remove(str(REPO))

sys.path.insert(0, str(REPO))

from text.symbols import symbols

symbol_to_id = {
    symbol: idx for idx, symbol in enumerate(symbols)
}

print("Vocabulary symbols:", len(symbol_to_id))
print("Model vocabulary size:", telugu_recovery_model.encoder.src_word_emb.num_embeddings)

assert len(symbol_to_id) <= telugu_recovery_model.encoder.src_word_emb.num_embeddings, (
    "Vocabulary indices exceed the model embedding size."
)

assert min(symbol_to_id.values()) >= 0
assert max(symbol_to_id.values()) < telugu_recovery_model.encoder.src_word_emb.num_embeddings

print("Vocabulary compatibility check: PASSED")

print("Vocabulary size:", len(symbol_to_id))

# ------------------------------------------------
# 4. LOAD UROMAN
# ------------------------------------------------

try:
    from uroman.uroman import Uroman
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "uroman"
    ])
    from uroman.uroman import Uroman

uroman = Uroman()

print("Uroman loaded.")

# ------------------------------------------------
# 5. LOAD UNIVERSAL HIFI-GAN
# ------------------------------------------------

from hifigan.models import Generator
from types import SimpleNamespace

with open(REPO / "hifigan/config.json") as f:
    hifi_config = SimpleNamespace(**json.load(f))

vocoder_checkpoint_path = (
    TE / "vocoder/generator_universal/generator_universal.pth.tar"
)

vocoder_checkpoint = torch.load(
    vocoder_checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

generator = Generator(hifi_config).to(DEVICE)

generator.load_state_dict(
    vocoder_checkpoint["generator"],
    strict=True
)

generator.eval()

if hasattr(generator, "remove_weight_norm"):
    generator.remove_weight_norm()

print("Universal HiFi-GAN loaded successfully.")

# ------------------------------------------------
# 6. EXACT ORIGINAL TEST SENTENCES
# ------------------------------------------------
# T4 is expanded into spoken Telugu, as in original STEP 85.

test_sentences = {
    "T1": {
        "original": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
        "spoken": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు."
    },
    "T2a": {
        "original": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
        "spoken": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది."
    },
    "T2b": {
        "original": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
        "spoken": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది."
    },
    "T3": {
        "original": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
        "spoken": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది."
    },
    "T4": {
        "original": "ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి.",
        "spoken": "ఈ సమావేశం రెండువేల ఇరవై ఆరు నవంబర్ పన్నెండవ తేదీ ఉదయం పది గంటల నలభై ఐదు నిమిషాలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి."
    }
}

# ------------------------------------------------
# 7. TEXT NORMALIZATION AND TOKENIZATION
# ------------------------------------------------

def prepare_text(text):

    romanized = uroman.romanize_string(text)

    normalized = re.sub(r"[^\w\s]", "", romanized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    normalized = normalized.lower()

    chars = list(normalized)

    unknown = sorted(set(
        c for c in chars if c not in symbol_to_id
    ))

    if unknown:
        raise ValueError(
            f"Unknown vocabulary characters: {unknown}\n"
            f"Original: {text}\n"
            f"Romanized: {romanized}"
        )

    token_ids = [symbol_to_id[c] for c in chars]

    if not token_ids:
        raise ValueError("Empty token sequence.")

    return romanized, normalized, chars, token_ids


prepared = {}

print("\n" + "=" * 65)
print("VALIDATING TELUGU TEST SENTENCES")
print("=" * 65)

for name, item in test_sentences.items():

    romanized, normalized, chars, token_ids = prepare_text(
        item["spoken"]
    )

    prepared[name] = {
        "original": item["original"],
        "spoken": item["spoken"],
        "romanized": romanized,
        "normalized": normalized,
        "tokens": chars,
        "token_ids": token_ids
    }

    print(
        f"{name}: {len(token_ids)} tokens | "
        f"unknown characters: 0"
    )

# Save mapping in recovery folder
mapping_path = LOG_DIR / "recovery_test_sentence_token_mapping.json"

with open(mapping_path, "w", encoding="utf-8") as f:
    json.dump(prepared, f, ensure_ascii=False, indent=2)

# ------------------------------------------------
# 8. MODEL SMOKE TEST
# ------------------------------------------------

first_item = next(iter(prepared.values()))

smoke_text = torch.tensor(
    [first_item["token_ids"]],
    dtype=torch.long,
    device=DEVICE
)

smoke_length = torch.tensor(
    [smoke_text.shape[1]],
    dtype=torch.long,
    device=DEVICE
)

smoke_speaker = torch.tensor(
    [SPEAKER_ID],
    dtype=torch.long,
    device=DEVICE
)

with torch.no_grad():
    smoke_outputs = model(
        smoke_speaker,
        smoke_text,
        smoke_length,
        int(smoke_length.max().item())
    )

assert len(smoke_outputs) >= 6

print("\nModel output shapes:")
for i, output in enumerate(smoke_outputs[:6]):
    if torch.is_tensor(output):
        print(i, tuple(output.shape))

# ------------------------------------------------
# 9. INFERENCE FUNCTION
# ------------------------------------------------

def run_inference(token_ids):

    text = torch.tensor(
        [token_ids], dtype=torch.long, device=DEVICE
    )

    src_lens = torch.tensor(
        [len(token_ids)], dtype=torch.long, device=DEVICE
    )

    speakers = torch.tensor(
        [SPEAKER_ID], dtype=torch.long, device=DEVICE
    )

    with torch.no_grad():

        outputs = model(
            speakers,
            text,
            src_lens,
            int(src_lens.max().item())
        )

        mel = outputs[1][0].cpu().numpy()
        pitch = outputs[2][0].cpu().numpy()
        energy = outputs[3][0].cpu().numpy()
        log_duration = outputs[4][0].cpu().numpy()
        durations = outputs[5][0].cpu().numpy()

    return {
        "mel": mel,
        "pitch": pitch,
        "energy": energy,
        "log_duration": log_duration,
        "durations": durations
    }

# ------------------------------------------------
# 10. VOCODER FUNCTION
# ------------------------------------------------

def generate_waveform(mel):

    mel_tensor = torch.tensor(
        mel.T[None, :, :],
        dtype=torch.float32,
        device=DEVICE
    )

    with torch.no_grad():
        waveform = generator(mel_tensor)

    waveform = waveform.squeeze().cpu().numpy()
    waveform = np.nan_to_num(waveform)

    if waveform.size == 0:
        raise RuntimeError("Empty waveform.")

    peak = np.max(np.abs(waveform))

    if peak > 1e-8:
        waveform = 0.95 * waveform / peak

    return waveform

# ------------------------------------------------
# 11. PLOT FUNCTIONS
# ------------------------------------------------

def save_duration_plot(name, tokens, durations):

    durations = np.asarray(durations).reshape(-1)
    n = min(len(tokens), len(durations))

    plt.figure(figsize=(max(12, n * 0.16), 5))
    plt.bar(np.arange(n), durations[:n])

    plt.xlabel("Input character position")
    plt.ylabel("Predicted duration (mel frames)")
    plt.title(f"{name} - Predicted Duration")

    plt.xticks(
        np.arange(n),
        tokens[:n],
        rotation=90,
        fontsize=7
    )

    plt.tight_layout()
    plt.savefig(DURATION_DIR / f"{name}_duration.png", dpi=180)
    plt.close()


def save_pitch_plot(name, pitch):

    pitch = np.asarray(pitch).reshape(-1)

    plt.figure(figsize=(12, 4))
    plt.plot(pitch)

    plt.xlabel("Mel frame")
    plt.ylabel("Predicted pitch")
    plt.title(f"{name} - Predicted Pitch")

    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(PITCH_DIR / f"{name}_pitch.png", dpi=180)
    plt.close()


def save_energy_plot(name, energy):

    energy = np.asarray(energy).reshape(-1)

    plt.figure(figsize=(12, 4))
    plt.plot(energy)

    plt.xlabel("Mel frame")
    plt.ylabel("Predicted energy")
    plt.title(f"{name} - Predicted Energy")

    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(ENERGY_DIR / f"{name}_energy.png", dpi=180)
    plt.close()

# ------------------------------------------------
# 12. GENERATE ALL FIVE AUDIOS AND PLOTS
# ------------------------------------------------

results_summary = []

print("\n" + "=" * 65)
print("GENERATING TELUGU RECOVERY AUDIOS — STEP 600")
print("=" * 65)

for name, item in prepared.items():

    print(f"\nProcessing {name}...")

    prediction = run_inference(item["token_ids"])

    mel = prediction["mel"]
    pitch = prediction["pitch"]
    energy = prediction["energy"]
    durations = prediction["durations"]
    log_duration = prediction["log_duration"]

    assert mel.ndim == 2 and mel.shape[1] == 80
    assert np.isfinite(mel).all()
    assert np.isfinite(pitch).all()
    assert np.isfinite(energy).all()
    assert np.isfinite(durations).all()

    waveform = generate_waveform(mel)

    wav_path = AUDIO_DIR / f"{name}_recovery600.wav"

    sf.write(
        str(wav_path),
        waveform,
        SAMPLE_RATE,
        subtype="PCM_16"
    )

    prediction_path = PRED_DIR / f"{name}_recovery600_predictions.npz"

    np.savez_compressed(
        prediction_path,
        mel=mel,
        pitch=pitch,
        energy=energy,
        durations=durations,
        log_duration=log_duration,
        tokens=np.asarray(item["tokens"]),
        token_ids=np.asarray(item["token_ids"]),
        original_text=item["original"],
        spoken_text=item["spoken"],
        romanized_text=item["romanized"],
        normalized_text=item["normalized"],
        speaker_id=SPEAKER_ID,
        checkpoint_step=600
    )

    save_duration_plot(
        name, item["tokens"], durations
    )

    save_pitch_plot(name, pitch)
    save_energy_plot(name, energy)

    audio_seconds = len(waveform) / SAMPLE_RATE

    results_summary.append({
        "sentence": name,
        "checkpoint_step": 600,
        "input_characters": len(item["tokens"]),
        "mel_frames": int(mel.shape[0]),
        "audio_seconds": round(audio_seconds, 3),
        "audio_file": str(wav_path),
        "prediction_file": str(prediction_path)
    })

    print("Saved:", wav_path.name)
    print("Duration:", round(audio_seconds, 2), "seconds")
    print("Mel:", mel.shape)
    print("Pitch:", pitch.shape)
    print("Energy:", energy.shape)
    print("Duration:", durations.shape)

# ------------------------------------------------
# 13. SAVE SUMMARY
# ------------------------------------------------

summary_path = LOG_DIR / "telugu_recovery600_inference_summary.json"

summary_data = {
    "generated_at": datetime.now().isoformat(),
    "model": "FastSpeech 2",
    "checkpoint_step": 600,
    "vocoder": "Universal HiFi-GAN",
    "device": str(DEVICE),
    "sample_rate": SAMPLE_RATE,
    "speaker_id": SPEAKER_ID,
    "speaker_description": "Female speaker",
    "sentences": results_summary
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_data, f, ensure_ascii=False, indent=2)

# ------------------------------------------------
# 14. FINAL AUDIT
# ------------------------------------------------

audio_count = len(list(AUDIO_DIR.glob("*_recovery600.wav")))
prediction_count = len(list(PRED_DIR.glob("*_recovery600_predictions.npz")))

duration_count = len(list(DURATION_DIR.glob("*.png")))
pitch_count = len(list(PITCH_DIR.glob("*.png")))
energy_count = len(list(ENERGY_DIR.glob("*.png")))

print("\n" + "=" * 65)
print("TELUGU RECOVERY INFERENCE COMPLETE")
print("=" * 65)

print("Audio files:", audio_count, "/ 5")
print("Prediction files:", prediction_count, "/ 5")
print("Duration plots:", duration_count, "/ 5")
print("Pitch plots:", pitch_count, "/ 5")
print("Energy plots:", energy_count, "/ 5")

print("\nSummary:", summary_path)
print("Recovery output:", RECOVERY / "inference")

assert audio_count == 5
assert prediction_count == 5
assert duration_count == 5
assert pitch_count == 5
assert energy_count == 5

print("\nALL FIVE TELUGU INFERENCE OUTPUTS VERIFIED.")

FastSpeech2 recovery model ready.
Checkpoint: 600
Vocabulary symbols: 360
Model vocabulary size: 361
Vocabulary compatibility check: PASSED
Vocabulary size: 360
Uroman loaded.


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
Universal HiFi-GAN loaded successfully.

VALIDATING TELUGU TEST SENTENCES
T1: 95 tokens | unknown characters: 0
T2a: 44 tokens | unknown characters: 0
T2b: 49 tokens | unknown characters: 0
T3: 40 tokens | unknown characters: 0
T4: 167 tokens | unknown characters: 0

Model output shapes:
0 (1, 613, 80)
1 (1, 613, 80)
2 (1, 613)
3 (1, 613)
4 (1, 95)
5 (1, 95)

GENERATING TELUGU RECOVERY AUDIOS — STEP 600

Processing T1...
Saved: T1_recovery600.wav
Duration: 7.12 seconds
Mel: (613, 80)
Pitch: (613,)
Energy: (613,)
Duration: (95,)

Processing T2a...
Saved: T2a_recovery600.wav
Duration: 3.59 seconds
Mel: (309, 80)
Pitch: (309,)
Energy: (309,)
Duration: (44,)

Processing T2b...
Saved: T2b_recovery600.wav
Duration: 4.44 seconds
Mel: (382, 80)
Pitch: (382,)
Energy: (382,)
Duration: (49,)

Processing T3...
Saved: T3_recovery600.wav
Duration: 3.49 seconds
Mel: (301, 80)
Pitch: (301,)
Energy: (301,)
Duration: (40,)

Processing T4...
Saved: T4_recovery600.wav
Duration: 12.

In [32]:
!pip install -q reportlab

In [33]:
# ============================================================
# FINAL FASTSPEECH2 SUBMISSION BUILDER
# RUN NOW — one cell; does NOT rerun training or inference.
# ============================================================

from pathlib import Path
import os, json, zipfile, datetime

# Locate project
candidates = [
    Path("/content/drive/MyDrive/FastSpeech2_Assignment"),
    Path("/content/gdrive/MyDrive/FastSpeech2_Assignment")
]
ROOT = next((p for p in candidates if p.exists()), None)
assert ROOT is not None, "Project folder not found. Check that Google Drive is mounted."

EN = ROOT / "English"
TE = ROOT / "Telugu"
REPORT = ROOT / "Report"
REPORT.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)

# ---------- 1. Inventory and audit ----------
def count_files(folder, suffix=None):
    if not folder.exists():
        return 0
    files = [p for p in folder.rglob("*") if p.is_file()]
    if suffix:
        files = [p for p in files if p.suffix.lower() == suffix.lower()]
    return len(files)

def first_existing(*paths):
    return next((p for p in paths if p.exists()), None)

english_audio = first_existing(
    EN / "results/test_generated_audios",
    EN / "test_generated_audios"
)
telugu_recovery = TE / "recovery_training/inference"
telugu_audio = first_existing(
    telugu_recovery / "test_generated_audios",
    TE / "results/test_generated_audios"
)

checks = {
    "English best checkpoint": (EN / "checkpoints/best_checkpoint.pth.tar").exists(),
    "English latest checkpoint": (EN / "checkpoints/latest_checkpoint.pth.tar").exists(),
    "English training history": (EN / "logs/english_training_history.csv").exists(),
    "English inference summary": (EN / "inference/logs/english_inference_summary.json").exists(),
    "Telugu recovery checkpoint": (TE / "recovery_training/checkpoints/600.pth.tar").exists()
        or count_files(TE / "recovery_training/checkpoints", ".pth.tar") > 0,
    "Telugu recovery inference summary": (telugu_recovery / "logs/telugu_recovery600_inference_summary.json").exists(),
    "English audio outputs": count_files(english_audio, ".wav") >= 5 if english_audio else False,
    "Telugu recovery audio outputs": count_files(telugu_audio, ".wav") >= 5 if telugu_audio else False,
}

# Count plots across each project language tree
checks["English variance-adaptor plots"] = count_files(EN, ".png") >= 15
checks["Telugu variance-adaptor plots"] = (
    count_files(telugu_recovery, ".png") >= 15
)

# ---------- 2. README ----------
readme = """# FastSpeech 2 — English and Telugu

This project implements and evaluates FastSpeech 2 for English and Telugu speech synthesis.

## Languages and datasets
- English: ai4bharat/Svarah
- Telugu: SPRINGLab/IndicTTS_Telugu

## Project contents
- `English/`: English preprocessing, configuration, checkpoints, training logs and inference outputs.
- `Telugu/`: Telugu preprocessing, configuration, checkpoints, training logs and inference outputs.
- `Telugu/recovery_training/inference/`: Telugu checkpoint-600 recovery inference outputs.
- `Report/Technical_Report.pdf`: Technical report.
- `requirements.txt`: Python dependencies.

## Generated test sentences
Each language has five test outputs: E1/E2a/E2b/E3/E4 for English and T1/T2a/T2b/T3/T4 for Telugu.

For each test sentence, the project stores generated audio, predictions and duration, pitch and energy visualizations where available.

## Important Telugu training limitation
The Telugu experiment reached checkpoint step 600. An audit found that model trainable parameters and Adam optimizer state did not advance after step 300, although BatchNorm running statistics continued changing. The checkpoint-600 inference outputs were generated successfully, but successful inference does not establish high speech quality. This limitation should be considered when interpreting Telugu results.

## Reproducibility
Use the language-specific configuration files and checkpoints in their respective folders. Dataset preparation and training are computationally intensive; existing prepared features and checkpoints are included where available.

## Results
See `Results/` where present, and the language-specific inference folders for generated audio and variance-adaptor plots.
"""

(ROOT / "README.md").write_text(readme, encoding="utf-8")

# ---------- 3. Requirements ----------
requirements = """torch
torchaudio
numpy
scipy
librosa
soundfile
matplotlib
pandas
PyYAML
tqdm
g2p-en
pyworld
uroman
praat-parselmouth
scikit-learn
"""

(ROOT / "requirements.txt").write_text(requirements, encoding="utf-8")

# ---------- 4. Technical report PDF ----------
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, ListFlowable, ListItem
)

pdf_path = REPORT / "Technical_Report.pdf"
doc = SimpleDocTemplate(
    str(pdf_path), pagesize=A4,
    rightMargin=1.8*cm, leftMargin=1.8*cm,
    topMargin=1.7*cm, bottomMargin=1.7*cm
)
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    name="ReportTitle", parent=styles["Title"],
    alignment=TA_CENTER, fontSize=20, leading=25, spaceAfter=14
))
story = [
    Paragraph("FastSpeech 2: English and Telugu Speech Synthesis", styles["ReportTitle"]),
    Paragraph("Technical Report", styles["Heading2"]),
    Paragraph("Prepared: " + datetime.datetime.now().strftime("%d %B %Y"), styles["Normal"]),
    Spacer(1, 12),
    Paragraph("1. Project Overview", styles["Heading1"]),
    Paragraph(
        "This project implements and evaluates a FastSpeech 2 text-to-speech pipeline "
        "for English and Telugu. The experiments use the specified Svarah English dataset "
        "and IndicTTS Telugu dataset. The workflow includes data preparation, alignment or "
        "duration preparation, acoustic feature extraction, model training, inference and "
        "analysis of duration, pitch and energy predictions.", styles["BodyText"]),
    Paragraph("2. Datasets and Preparation", styles["Heading1"]),
    Paragraph(
        "<b>English:</b> ai4bharat/Svarah. The verified experiment used 2,662 training "
        "and 501 validation feature files, with no missing alignments among the selected "
        "feature IDs. The dataset readiness audit recorded approximately 5.00 hours of "
        "training audio and 1.00 hour of validation audio, with no train/validation ID overlap.",
        styles["BodyText"]),
    Spacer(1, 6),
    Paragraph(
        "<b>Telugu:</b> SPRINGLab/IndicTTS_Telugu. The prepared split contained approximately "
        "5.01 hours of training data and 1.00 hour of validation data. The experiment used "
        "MMS forced alignment with Uroman romanization and extracted mel, pitch, energy and "
        "duration features. The prepared features were chunked for model input.", styles["BodyText"]),
    Paragraph("3. Model and Configuration", styles["Heading1"]),
    Paragraph(
        "Both experiments use a compact FastSpeech 2 configuration with Transformer encoder "
        "and decoder layers, a variance adaptor for duration, pitch and energy, and an "
        "80-channel mel-spectrogram representation. The experiments use a 22,050 Hz audio "
        "sampling rate and a HiFi-GAN vocoder. Language-specific preprocessing and model "
        "configurations are retained in the project folders.", styles["BodyText"]),
    Paragraph("4. English Experiment", styles["Heading1"]),
    Paragraph(
        "The English experiment has a best checkpoint at step 5,000 with recorded best "
        "validation loss 3.6294. The latest checkpoint is at step 6,000. Five test utterances "
        "were synthesized, and the project contains the corresponding prediction files and "
        "duration, pitch and energy plots, alongside training history and loss plots.",
        styles["BodyText"]),
    Paragraph("5. Telugu Experiment and Limitation", styles["Heading1"]),
    Paragraph(
        "Telugu training reached checkpoint step 600. Five test utterances were synthesized "
        "from the recovery checkpoint, and five prediction files plus 15 duration, pitch and "
        "energy plots were generated. Inference completed successfully using the universal "
        "HiFi-GAN vocoder.", styles["BodyText"]),
    Paragraph(
        "<b>Training limitation:</b> A checkpoint comparison found that trainable model "
        "parameters and Adam optimizer state remained unchanged after step 300, while "
        "BatchNorm running statistics continued to change. The exact cause has not been "
        "established. Therefore, checkpoint 600 is reported as the recovery inference "
        "checkpoint, not as evidence that Telugu optimization converged. Telugu audio "
        "quality should be assessed separately and results interpreted cautiously.",
        styles["BodyText"]),
    Paragraph("6. Inference and Analysis", styles["Heading1"]),
    Paragraph(
        "The inference outputs include five utterances per language. For each utterance, "
        "the pipeline records predicted mel features, duration predictions, pitch and energy "
        "outputs, and generated waveform audio. Variance-adaptor plots are provided to "
        "support inspection of duration, pitch and energy behavior.", styles["BodyText"]),
    Paragraph("7. Deliverables", styles["Heading1"]),
    Paragraph(
        "The submission package includes language-specific experiment folders, available "
        "checkpoints and configurations, training logs, inference audio, prediction artifacts, "
        "plots, this report, README and dependency list. The accompanying audit JSON records "
        "which expected artifacts were detected at packaging time.", styles["BodyText"]),
    Paragraph("8. Reproducibility Notes", styles["Heading1"]),
    Paragraph(
        "Use the language-specific configuration files and checkpoints. The prepared feature "
        "files and alignments are retained to avoid repeating expensive preprocessing. "
        "The Python dependencies are listed in requirements.txt; exact compatibility may "
        "depend on the Colab runtime and installed PyTorch/CUDA versions.", styles["BodyText"]),
    Spacer(1, 10),
    Paragraph("Packaging audit", styles["Heading1"])
]

audit_rows = [["Artifact check", "Detected"]]
for name, ok in checks.items():
    audit_rows.append([name, "PASS" if ok else "NOT FOUND / CHECK"])
tbl = Table(audit_rows, colWidths=[12.2*cm, 3.0*cm], repeatRows=1)
tbl.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#263445")),
    ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("GRID", (0,0), (-1,-1), 0.4, colors.lightgrey),
    ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
    ("FONTSIZE", (0,0), (-1,-1), 8),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, colors.HexColor("#F2F4F7")]),
]))
story.append(tbl)
doc.build(story)

# ---------- 5. Audit JSON ----------
audit = {
    "created": datetime.datetime.now().isoformat(),
    "project_root": str(ROOT),
    "checks": checks,
    "english_audio_count": count_files(english_audio, ".wav") if english_audio else 0,
    "telugu_recovery_audio_count": count_files(telugu_audio, ".wav") if telugu_audio else 0,
    "english_png_count": count_files(EN, ".png"),
    "telugu_recovery_png_count": count_files(telugu_recovery, ".png"),
    "telugu_training_note": (
        "Trainable model parameters and Adam optimizer state were unchanged after step 300; "
        "BatchNorm running statistics continued changing. Cause not established."
    )
}
(ROOT / "submission_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")

# ---------- 6. ZIP package ----------
zip_path = ROOT / "FastSpeech2_Submission.zip"
skip_parts = {
    ".git", "__pycache__", ".ipynb_checkpoints",
    ".cache", "cache", "raw_audio"
}
include_roots = [
    ROOT / "English", ROOT / "Telugu", ROOT / "Report",
    ROOT / "Code"
]
include_files = [
    ROOT / "README.md", ROOT / "requirements.txt",
    ROOT / "submission_audit.json"
]

def should_skip(path):
    return any(part in skip_parts for part in path.parts)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as z:
    for base in include_roots:
        if not base.exists():
            continue
        for p in base.rglob("*"):
            if p.is_file() and not should_skip(p.relative_to(ROOT)):
                z.write(p, p.relative_to(ROOT))
    for p in include_files:
        if p.exists():
            z.write(p, p.relative_to(ROOT))

# ---------- 7. Final status ----------
print("\n" + "="*60)
print("FASTSpeech2 SUBMISSION PACKAGE CREATED")
print("="*60)
for name, ok in checks.items():
    print(("OK   " if ok else "CHECK") + name)

print("\nREADME:", ROOT / "README.md")
print("Requirements:", ROOT / "requirements.txt")
print("PDF report:", pdf_path)
print("Audit:", ROOT / "submission_audit.json")
print("ZIP:", zip_path)
print("ZIP size: %.2f MB" % (zip_path.stat().st_size / (1024**2)))
print("\nNOTE: GitHub upload is a separate step; this cell does not publish anything.")

Project: /content/drive/MyDrive/FastSpeech2_Assignment


KeyboardInterrupt: 

In [34]:
import os, time
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

for p in [
    ROOT / "README.md",
    ROOT / "requirements.txt",
    ROOT / "Report/Technical_Report.pdf",
    ROOT / "submission_audit.json",
    ROOT / "FastSpeech2_Submission.zip"
]:
    if p.exists():
        print("FOUND:", p.name, f"{p.stat().st_size / (1024**2):.2f} MB")
    else:
        print("MISSING:", p.name)

print("\nZIP exists:", (ROOT / "FastSpeech2_Submission.zip").exists())

FOUND: README.md 0.00 MB
FOUND: requirements.txt 0.00 MB
FOUND: Technical_Report.pdf 0.00 MB
FOUND: submission_audit.json 0.00 MB
FOUND: FastSpeech2_Submission.zip 1040.48 MB

ZIP exists: True


In [35]:
from pathlib import Path
import zipfile

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ZIP = ROOT / "FastSpeech2_Submission.zip"
PDF = ROOT / "Report/Technical_Report.pdf"

print("ZIP size:", round(ZIP.stat().st_size / (1024**2), 2), "MB")
print("ZIP integrity:", end=" ")

with zipfile.ZipFile(ZIP, "r") as z:
    bad = z.testzip()
    print("PASS" if bad is None else f"FAILED: {bad}")
    print("Files inside ZIP:", len(z.namelist()))

print("PDF size:", PDF.stat().st_size, "bytes")
print("PDF header:", PDF.open("rb").read(5))

ZIP size: 1040.48 MB
ZIP integrity: PASS
Files inside ZIP: 3516
PDF size: 5108 bytes
PDF header: b'%PDF-'


In [36]:
import zipfile
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ZIP = ROOT / "FastSpeech2_Submission.zip"

with zipfile.ZipFile(ZIP) as z:
    names = z.namelist()

checks = {
    "README": any(x.endswith("README.md") for x in names),
    "Requirements": any(x.endswith("requirements.txt") for x in names),
    "Technical report": any(x.endswith("Technical_Report.pdf") for x in names),
    "English checkpoint": any("English/" in x and "best_checkpoint" in x for x in names),
    "Telugu recovery inference": any("telugu_recovery600_inference_summary.json" in x for x in names),
    "Telugu recovery audio": any("recovery_training/inference/" in x and x.endswith(".wav") for x in names),
    "Python source code": any(x.endswith(".py") for x in names),
}

for item, passed in checks.items():
    print(("PASS" if passed else "MISSING"), item)

MISSING README
MISSING Requirements
MISSING Technical report
PASS English checkpoint
MISSING Telugu recovery inference
MISSING Telugu recovery audio
MISSING Python source code


In [37]:
from pathlib import Path
import zipfile

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ZIP = ROOT / "FastSpeech2_Submission.zip"

print("ACTUAL PROJECT FILES")
for p in [
    ROOT / "README.md",
    ROOT / "requirements.txt",
    ROOT / "Report/Technical_Report.pdf",
    ROOT / "Telugu/recovery_training/inference",
    ROOT / "Code/FastSpeech2"
]:
    print("\n", p, "EXISTS:", p.exists())
    if p.is_dir():
        for f in list(p.rglob("*"))[:8]:
            if f.is_file():
                print("  ", f.relative_to(ROOT))

print("\nZIP CONTENTS — RELEVANT ENTRIES")
with zipfile.ZipFile(ZIP, "r") as z:
    names = z.namelist()
    for keyword in [
        "README", "requirements", "Technical_Report",
        "recovery_training", "recovery600", ".py"
    ]:
        matches = [n for n in names if keyword.lower() in n.lower()]
        print(f"\n{keyword}: {len(matches)} matches")
        for n in matches[:5]:
            print(" ", n)

print("\nZIP file count:", len(names))

ACTUAL PROJECT FILES

 /content/drive/MyDrive/FastSpeech2_Assignment/README.md EXISTS: True

 /content/drive/MyDrive/FastSpeech2_Assignment/requirements.txt EXISTS: True

 /content/drive/MyDrive/FastSpeech2_Assignment/Report/Technical_Report.pdf EXISTS: True

 /content/drive/MyDrive/FastSpeech2_Assignment/Telugu/recovery_training/inference EXISTS: True
   Telugu/recovery_training/inference/test_generated_audios/T1_recovery600.wav
   Telugu/recovery_training/inference/test_generated_audios/T2a_recovery600.wav
   Telugu/recovery_training/inference/test_generated_audios/T2b_recovery600.wav
   Telugu/recovery_training/inference/test_generated_audios/T3_recovery600.wav

 /content/drive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2 EXISTS: True
   Code/FastSpeech2/evaluate.py
   Code/FastSpeech2/README.md
   Code/FastSpeech2/dataset.py
   Code/FastSpeech2/prepare_align.py
   Code/FastSpeech2/preprocess.py
   Code/FastSpeech2/synthesize.py
   Code/FastSpeech2/requirements.txt
   Code/FastSp

In [38]:
from pathlib import Path
import zipfile

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
ZIP = ROOT / "FastSpeech2_Submission.zip"

required_files = [
    ROOT / "README.md",
    ROOT / "requirements.txt",
    ROOT / "Report/Technical_Report.pdf",
    ROOT / "submission_audit.json",
]

required_dirs = [
    ROOT / "Telugu/recovery_training/inference",
    ROOT / "Code/FastSpeech2",
]

added = []
missing = []

with zipfile.ZipFile(ZIP, "a", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    existing = set(z.namelist())

    for p in required_files:
        if not p.exists():
            missing.append(str(p))
            continue
        arcname = p.relative_to(ROOT).as_posix()
        if arcname not in existing:
            z.write(p, arcname)
            added.append(arcname)

    for folder in required_dirs:
        if not folder.exists():
            missing.append(str(folder))
            continue
        for p in folder.rglob("*"):
            if not p.is_file():
                continue
            arcname = p.relative_to(ROOT).as_posix()
            if arcname not in existing:
                z.write(p, arcname)
                added.append(arcname)

print("ADDED FILES:", len(added))
print("MISSING SOURCE PATHS:", missing)

# Verify required deliverables
with zipfile.ZipFile(ZIP, "r") as z:
    names = set(z.namelist())

    checks = {
        "README": "README.md" in names,
        "Requirements": "requirements.txt" in names,
        "Technical report": "Report/Technical_Report.pdf" in names,
        "English checkpoint": any(
            "English/" in n and "best_checkpoint" in n for n in names
        ),
        "Telugu recovery summary": (
            "Telugu/recovery_training/inference/logs/"
            "telugu_recovery600_inference_summary.json" in names
        ),
        "Telugu recovery audio": any(
            n.startswith("Telugu/recovery_training/inference/")
            and n.endswith(".wav") for n in names
        ),
        "Python source code": any(
            n.startswith("Code/FastSpeech2/") and n.endswith(".py")
            for n in names
        ),
    }

    print("\nFINAL ZIP CHECK")
    for name, ok in checks.items():
        print(("PASS " if ok else "FAIL ") + name)

    print("\nZIP integrity:", "PASS" if z.testzip() is None else "FAIL")
    print("Total files:", len(names))

print("\nZIP size: %.2f MB" % (ZIP.stat().st_size / (1024**2)))

ADDED FILES: 116
MISSING SOURCE PATHS: []

FINAL ZIP CHECK
PASS README
PASS Requirements
PASS Technical report
PASS English checkpoint
PASS Telugu recovery summary
PASS Telugu recovery audio
PASS Python source code

ZIP integrity: PASS
Total files: 3632

ZIP size: 1067.88 MB


In [39]:
# ============================================================
# FASTSPEECH 2 — FINAL SOURCE CODE AUDIT
# RUN NOW — inspection only; no training or file modifications.
# ============================================================

from pathlib import Path
import ast

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")

required_files = {
    "English": [
        "English/train.py",
        "English/preprocess.py",
        "English/inference.py",
    ],
    "Telugu": [
        "Telugu/train.py",
        "Telugu/preprocess.py",
        "Telugu/inference.py",
    ],
}

print("=" * 65)
print("FASTSPEECH 2 SOURCE CODE AUDIT")
print("=" * 65)

all_passed = True

for language, files in required_files.items():
    print(f"\n{language.upper()}")

    for relative_path in files:
        path = ROOT / relative_path

        if not path.exists():
            print(f"MISSING: {relative_path}")
            all_passed = False
            continue

        source = path.read_text(encoding="utf-8", errors="replace")

        try:
            tree = ast.parse(source)
            syntax = "VALID"
        except SyntaxError as e:
            syntax = f"ERROR: {e}"
            all_passed = False
            tree = None

        functions = []
        classes = []

        if tree:
            functions = [
                n.name for n in ast.walk(tree)
                if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))
            ]
            classes = [
                n.name for n in ast.walk(tree)
                if isinstance(n, ast.ClassDef)
            ]

        print(f"\nFILE: {relative_path}")
        print(f"Size: {path.stat().st_size / 1024:.1f} KB")
        print(f"Python syntax: {syntax}")
        print(f"Functions: {', '.join(functions[:15]) or 'None detected'}")
        print(f"Classes: {', '.join(classes[:10]) or 'None detected'}")

        if not source.strip():
            print("WARNING: Empty file")
            all_passed = False

print("\n" + "=" * 65)
print("ADDITIONAL IMPLEMENTATION CHECKS")
print("=" * 65)

keywords = {
    "English/train.py": ["optimizer", "loss", "backward"],
    "English/preprocess.py": ["mel", "pitch", "energy"],
    "English/inference.py": ["inference", "vocoder"],
    "Telugu/train.py": ["optimizer", "loss", "backward"],
    "Telugu/preprocess.py": ["mel", "pitch", "energy"],
    "Telugu/inference.py": ["inference", "vocoder"],
}

for relative_path, terms in keywords.items():
    path = ROOT / relative_path

    if not path.exists():
        continue

    source = path.read_text(encoding="utf-8", errors="replace").lower()

    results = {term: term in source for term in terms}

    print(f"\n{relative_path}")
    for term, found in results.items():
        print(f"  {'FOUND' if found else 'NOT FOUND'}: {term}")

print("\n" + "=" * 65)
print("BASIC SOURCE AUDIT:", "PASSED" if all_passed else "ISSUES FOUND")
print("=" * 65)

FASTSPEECH 2 SOURCE CODE AUDIT

ENGLISH
MISSING: English/train.py
MISSING: English/preprocess.py
MISSING: English/inference.py

TELUGU
MISSING: Telugu/train.py
MISSING: Telugu/preprocess.py
MISSING: Telugu/inference.py

ADDITIONAL IMPLEMENTATION CHECKS

BASIC SOURCE AUDIT: ISSUES FOUND


In [40]:
from pathlib import Path
import ast

ROOT = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
CODE = ROOT / "Code/FastSpeech2"

print("CODE DIRECTORY:", CODE)
print("EXISTS:", CODE.exists())

required = [
    "train.py",
    "preprocess.py",
    "synthesize.py",
    "evaluate.py",
    "dataset.py",
    "prepare_align.py",
]

print("\nSOURCE FILE AUDIT")
for name in required:
    p = CODE / name
    if p.exists():
        try:
            ast.parse(p.read_text(encoding="utf-8", errors="replace"))
            print("PASS:", name, "| Syntax valid")
        except SyntaxError as e:
            print("FAIL:", name, "|", e)
    else:
        print("MISSING:", name)

print("\nLANGUAGE CONFIGURATION AUDIT")
for language in ["English", "Telugu"]:
    folder = ROOT / language
    print(f"\n{language}:")
    for name in ["config.yaml", "train.yaml", "preprocess.yaml", "model.yaml"]:
        matches = list(folder.rglob(name)) if folder.exists() else []
        print(("FOUND" if matches else "NOT FOUND"), name)
        for p in matches[:2]:
            print("  ", p.relative_to(ROOT))

print("\nPYTHON FILES IN CODE DIRECTORY:")
for p in CODE.rglob("*.py"):
    print(p.relative_to(ROOT))

CODE DIRECTORY: /content/drive/MyDrive/FastSpeech2_Assignment/Code/FastSpeech2
EXISTS: True

SOURCE FILE AUDIT
PASS: train.py | Syntax valid
PASS: preprocess.py | Syntax valid
PASS: synthesize.py | Syntax valid
PASS: evaluate.py | Syntax valid
PASS: dataset.py | Syntax valid
PASS: prepare_align.py | Syntax valid

LANGUAGE CONFIGURATION AUDIT

English:
NOT FOUND config.yaml
FOUND train.yaml
   English/config/train.yaml
   English/alignment/temporary/train/train/train.yaml
FOUND preprocess.yaml
   English/config/preprocess.yaml
FOUND model.yaml
   English/config/model.yaml

Telugu:
NOT FOUND config.yaml
FOUND train.yaml
   Telugu/config/train.yaml
FOUND preprocess.yaml
   Telugu/config/preprocess.yaml
FOUND model.yaml
   Telugu/config/model.yaml

PYTHON FILES IN CODE DIRECTORY:
Code/FastSpeech2/evaluate.py
Code/FastSpeech2/dataset.py
Code/FastSpeech2/prepare_align.py
Code/FastSpeech2/preprocess.py
Code/FastSpeech2/synthesize.py
Code/FastSpeech2/train.py
Code/FastSpeech2/text/cleaners.py


In [41]:
# ================================================================
# TELUGU STANDALONE INFERENCE SCRIPT
# Checkpoint: recovery_600.pth.tar
# Output: 5 WAV + 15 plots + 5 prediction files + summary JSON
# Does NOT modify English files or original Telugu inference results
# ================================================================

import os
import sys
import re
import json
import shutil
import subprocess
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
import soundfile as sf

from pathlib import Path
from types import SimpleNamespace

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO = BASE / "Code/FastSpeech2"
TELUGU = BASE / "Telugu"

CHECKPOINT = TELUGU / "recovery_training/checkpoints/recovery_600.pth.tar"
VOCODER_CHECKPOINT = (
    TELUGU / "vocoder/generator_universal/generator_universal.pth.tar"
)

OUT = TELUGU / "recovery_training/inference"

AUDIO_DIR = OUT / "test_generated_audios"
DURATION_DIR = OUT / "variance_adaptor_plots/duration"
PITCH_DIR = OUT / "variance_adaptor_plots/pitch"
ENERGY_DIR = OUT / "variance_adaptor_plots/energy"
PRED_DIR = OUT / "predictions"

for folder in [AUDIO_DIR, DURATION_DIR, PITCH_DIR, ENERGY_DIR, PRED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
SAMPLE_RATE = 22050
SPEAKER_ID = 0  # Female speaker

assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"
assert VOCODER_CHECKPOINT.exists(), f"Missing vocoder: {VOCODER_CHECKPOINT}"

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Paths verified.")
print("Device:", DEVICE)

# ------------------------------------------------
# 2. LOAD FASTSPEECH 2 CHECKPOINT 600
# ------------------------------------------------

from model.fastspeech2 import FastSpeech2

with open(TELUGU / "config/preprocess.yaml", encoding="utf-8") as f:
    preprocess_config = yaml.safe_load(f)

with open(TELUGU / "config/model.yaml", encoding="utf-8") as f:
    model_config = yaml.safe_load(f)

# The trained Telugu embedding has 361 entries.
model_config["input_vocabulary_size"] = 361

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

assert checkpoint["step"] == 600, (
    f"Expected checkpoint 600, got {checkpoint['step']}"
)

telugu_model = FastSpeech2(
    preprocess_config,
    model_config
).to(DEVICE)

telugu_model.load_state_dict(checkpoint["model"], strict=True)
telugu_model.eval()

print("FastSpeech 2 loaded.")
print("Checkpoint step:", checkpoint["step"])

# ------------------------------------------------
# 3. LOAD VOCABULARY AND UROMAN
# ------------------------------------------------

from text.symbols import symbols

symbol_to_id = {
    symbol: idx for idx, symbol in enumerate(symbols)
}

assert max(symbol_to_id.values()) < model_config["input_vocabulary_size"]

try:
    from uroman.uroman import Uroman
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "uroman"],
        check=True
    )
    from uroman.uroman import Uroman

uroman = Uroman()

print("Vocabulary entries:", len(symbol_to_id))
print("Uroman loaded.")

# ------------------------------------------------
# 4. LOAD UNIVERSAL HIFI-GAN
# ------------------------------------------------

from hifigan.models import Generator

with open(REPO / "hifigan/config.json", encoding="utf-8") as f:
    hifi_config = SimpleNamespace(**json.load(f))

generator = Generator(hifi_config).to(DEVICE)

vocoder_data = torch.load(
    VOCODER_CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

generator.load_state_dict(
    vocoder_data["generator"],
    strict=True
)

generator.eval()

print("Universal HiFi-GAN loaded.")

# ------------------------------------------------
# 5. TEST SENTENCES
# T4 spoken form expands the date and time for synthesis.
# ------------------------------------------------

test_sentences = {
    "T1": {
        "original": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.",
        "spoken": "న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు."
    },
    "T2a": {
        "original": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది.",
        "spoken": "తాడుపై అతని పట్టు చాలా బలంగా ఉంది."
    },
    "T2b": {
        "original": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.",
        "spoken": "ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది."
    },
    "T3": {
        "original": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది.",
        "spoken": "పులి అడవిలో నిశ్శబ్దంగా నడిచింది."
    },
    "T4": {
        "original": "ఈ సమావేశం 2026 నవంబర్ 12వ తేదీ ఉదయం 10:45 గంటలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి.",
        "spoken": "ఈ సమావేశం రెండువేల ఇరవై ఆరు నవంబర్ పన్నెండవ తేదీ ఉదయం పది గంటల నలభై ఐదు నిమిషాలకు ప్రారంభమవుతుంది, సమయానికి సిద్ధంగా ఉంచండి."
    }
}

# ------------------------------------------------
# 6. TEXT PREPARATION
# ------------------------------------------------

def prepare_text(telugu_text):

    romanized = uroman.romanize_string(telugu_text)

    normalized = re.sub(r"[^\w\s]", "", romanized)
    normalized = re.sub(r"\s+", " ", normalized).strip().lower()

    tokens = list(normalized)

    unknown = sorted(set(c for c in tokens if c not in symbol_to_id))

    if unknown:
        raise ValueError(f"Unknown Telugu romanized characters: {unknown}")

    token_ids = [symbol_to_id[c] for c in tokens]

    if not token_ids:
        raise ValueError("Empty token sequence.")

    return romanized, normalized, tokens, token_ids

# ------------------------------------------------
# 7. FASTSPEECH 2 INFERENCE
# ------------------------------------------------

def run_inference(token_ids):

    text = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=DEVICE
    )

    src_lens = torch.tensor(
        [len(token_ids)],
        dtype=torch.long,
        device=DEVICE
    )

    speakers = torch.tensor(
        [SPEAKER_ID],
        dtype=torch.long,
        device=DEVICE
    )

    with torch.no_grad():
        outputs = telugu_model(
            speakers,
            text,
            src_lens,
            int(src_lens.max().item())
        )

    if len(outputs) < 6:
        raise RuntimeError(f"Unexpected output count: {len(outputs)}")

    mel = outputs[1][0].detach().cpu().numpy()
    pitch = outputs[2][0].detach().cpu().numpy()
    energy = outputs[3][0].detach().cpu().numpy()
    durations = outputs[5][0].detach().cpu().numpy()

    return mel, pitch, energy, durations

# ------------------------------------------------
# 8. MEL TO AUDIO
# ------------------------------------------------

def generate_audio(mel):

    mel_tensor = torch.tensor(
        mel.T,
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    with torch.no_grad():
        waveform = generator(mel_tensor)

    waveform = waveform.squeeze().detach().cpu().numpy()
    waveform = np.nan_to_num(waveform)

    peak = np.max(np.abs(waveform))

    if peak > 1e-8:
        waveform = 0.95 * waveform / peak

    return waveform

# ------------------------------------------------
# 9. PLOT FUNCTIONS
# ------------------------------------------------

def save_duration_plot(name, tokens, durations):

    durations = np.asarray(durations).reshape(-1)
    n = min(len(tokens), len(durations))

    plt.figure(figsize=(max(12, n * 0.16), 5))
    plt.bar(np.arange(n), durations[:n])
    plt.xlabel("Input character position")
    plt.ylabel("Predicted duration (mel frames)")
    plt.title(f"{name} - Predicted Duration")
    plt.xticks(
        np.arange(n),
        tokens[:n],
        rotation=90,
        fontsize=7
    )
    plt.tight_layout()
    plt.savefig(DURATION_DIR / f"{name}_duration.png", dpi=180)
    plt.close()


def save_frame_plot(name, values, label, folder):

    values = np.asarray(values).reshape(-1)

    plt.figure(figsize=(14, 4))
    plt.plot(values)
    plt.xlabel("Mel frame")
    plt.ylabel(label)
    plt.title(f"{name} - Predicted {label}")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(folder / f"{name}_{label.lower()}.png", dpi=180)
    plt.close()

# ------------------------------------------------
# 10. GENERATE ALL FIVE SENTENCES
# ------------------------------------------------

summary = []

for name, item in test_sentences.items():

    print("\n" + "=" * 60)
    print("GENERATING:", name)
    print("=" * 60)

    romanized, normalized, tokens, token_ids = prepare_text(
        item["spoken"]
    )

    print("Original:", item["original"])
    print("Romanized:", romanized)
    print("Tokens:", len(token_ids))

    mel, pitch, energy, durations = run_inference(token_ids)

    assert mel.ndim == 2 and mel.shape[1] == 80
    assert np.isfinite(mel).all()
    assert np.isfinite(pitch).all()
    assert np.isfinite(energy).all()

    waveform = generate_audio(mel)

    wav_path = AUDIO_DIR / f"{name}_recovery600.wav"
    sf.write(str(wav_path), waveform, SAMPLE_RATE)

    # Save model predictions.
    prediction_path = PRED_DIR / f"{name}_predictions.npz"

    np.savez_compressed(
        prediction_path,
        original_text=item["original"],
        spoken_text=item["spoken"],
        romanized_text=romanized,
        normalized_text=normalized,
        tokens=np.asarray(tokens),
        mel=mel,
        pitch=pitch,
        energy=energy,
        durations=durations
    )

    # Save 3 variance-adaptor plots.
    save_duration_plot(name, tokens, durations)
    save_frame_plot(name, pitch, "Pitch", PITCH_DIR)
    save_frame_plot(name, energy, "Energy", ENERGY_DIR)

    audio_seconds = len(waveform) / SAMPLE_RATE

    summary.append({
        "sentence": name,
        "original_text": item["original"],
        "spoken_text": item["spoken"],
        "romanized_text": romanized,
        "token_count": len(token_ids),
        "mel_frames": int(mel.shape[0]),
        "audio_seconds": round(audio_seconds, 3),
        "sample_rate": SAMPLE_RATE,
        "audio_file": str(wav_path),
        "prediction_file": str(prediction_path)
    })

    print("Audio:", wav_path.name)
    print("Duration:", round(audio_seconds, 2), "seconds")
    print("Mel shape:", mel.shape)
    print("Pitch shape:", pitch.shape)
    print("Energy shape:", energy.shape)
    print("Duration shape:", durations.shape)

# ------------------------------------------------
# 11. SAVE SUMMARY
# ------------------------------------------------

summary_path = OUT / "telugu_recovery_inference_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": "FastSpeech 2",
            "checkpoint_step": 600,
            "vocoder": "Universal HiFi-GAN",
            "device": str(DEVICE),
            "sample_rate": SAMPLE_RATE,
            "speaker_id": SPEAKER_ID,
            "sentences": summary
        },
        f,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------
# 12. FINAL AUDIT
# ------------------------------------------------

audio_count = len(list(AUDIO_DIR.glob("*.wav")))
duration_count = len(list(DURATION_DIR.glob("*.png")))
pitch_count = len(list(PITCH_DIR.glob("*.png")))
energy_count = len(list(ENERGY_DIR.glob("*.png")))
prediction_count = len(list(PRED_DIR.glob("*_predictions.npz")))

print("\n" + "=" * 65)
print("TELUGU INFERENCE AUDIT")
print("=" * 65)

print("Audio files:", audio_count)
print("Duration plots:", duration_count)
print("Pitch plots:", pitch_count)
print("Energy plots:", energy_count)
print("Prediction files:", prediction_count)

assert audio_count == 5
assert duration_count == 5
assert pitch_count == 5
assert energy_count == 5
assert prediction_count == 5

print("\nALL INFERENCE FILES VERIFIED.")
print("Summary:", summary_path)
print("Output directory:", OUT)

Paths verified.
Device: cpu
FastSpeech 2 loaded.
Checkpoint step: 600
Vocabulary entries: 360
Uroman loaded.


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Universal HiFi-GAN loaded.

GENERATING: T1
Original: న్యూ రోఫార్మ కోలాజికల్ పరిశోధనలో నూతన ఔషధాల ప్రభావాన్ని విశ్లేషించారు.
Romanized: nyuu roophaarma koolaajikal parishoodhanaloo nuutana aussadhaala prabhaavaanni vishleessimcaaru.
Tokens: 95
Audio: T1_recovery600.wav
Duration: 7.12 seconds
Mel shape: (613, 80)
Pitch shape: (613,)
Energy shape: (613,)
Duration shape: (95,)

GENERATING: T2a
Original: తాడుపై అతని పట్టు చాలా బలంగా ఉంది.
Romanized: taaddupai atani pattttu caalaa balamgaa umdi.
Tokens: 44
Audio: T2a_recovery600.wav
Duration: 3.59 seconds
Mel shape: (309, 80)
Pitch shape: (309,)
Energy shape: (309,)
Duration shape: (44,)

GENERATING: T2b
Original: ఆమె పట్టు చీరను ధరించి వేడుకకు వచ్చింది.
Romanized: aame pattttu ciiranu dharimci veeddukaku vaccimdi.
Tokens: 49
Audio: T2b_recovery600.wav
Duration: 4.44 seconds
Mel shape: (382, 80)
Pitch shape: (382,)
Energy shape: (382,)
Duration shape: (49,)

GENERATING: T3
Original: పులి అడవిలో నిశ్శబ్దంగా నడిచింది.
Romanized: puli addaviloo

In [ ]:
# RUN NOW — Push FastSpeech2 project to your existing GitHub repository.
# This does NOT rerun training, preprocessing, or inference.

import os
import sys
import shutil
import subprocess
import getpass
from pathlib import Path

# -------------------- PATHS --------------------
BASE = Path("/content/drive/MyDrive/FastSpeech2_Assignment")
REPO_URL = "https://github.com/madhuri2006/FastSpeech2-Multilingual-TTS.git"
WORK = Path("/content/github_publish")
DEST = WORK / "FastSpeech2-Multilingual-TTS"

assert BASE.exists(), f"Project folder not found: {BASE}"

# -------------------- GITHUB AUTHENTICATION --------------------
print("Enter your GitHub Personal Access Token (input is hidden).")
print("It needs permission to write to this repository.")
GITHUB_TOKEN = getpass.getpass("GitHub token: ").strip()

if not GITHUB_TOKEN:
    raise ValueError("No GitHub token entered. Nothing was pushed.")

# Use a temporary askpass script; token is not written into the Git remote URL.
ASKPASS = WORK / "git_askpass.sh"
WORK.mkdir(parents=True, exist_ok=True)

ASKPASS.write_text("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
  *) echo "$GITHUB_TOKEN" ;;
esac
""")
ASKPASS.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = GITHUB_TOKEN
env["GIT_ASKPASS"] = str(ASKPASS)
env["GIT_TERMINAL_PROMPT"] = "0"

def run(cmd, cwd=None):
    print("\\n$", " ".join(map(str, cmd)))
    subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True
    )

# -------------------- CLONE EXISTING REPOSITORY --------------------
if DEST.exists():
    shutil.rmtree(DEST)

WORK.mkdir(parents=True, exist_ok=True)
run(["git", "clone", "--branch", "main", REPO_URL, DEST])

# -------------------- COPY PROJECT CONTENTS --------------------
# Exclude large source datasets and generated preprocessing caches.
excluded_dirs = {
    "raw_audio", "processed", "alignment", "mfa_corpus",
    "__pycache__", ".ipynb_checkpoints", ".git", "wandb",
    "huggingface_cache", ".cache"
}

def ignore_project(folder, names):
    return [name for name in names if name in excluded_dirs]

def merge_folder(source, destination):
    if source.exists():
        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True,
            ignore=ignore_project
        )
        print("Copied:", source.name)

# Main implementation and language-specific experiment folders.
merge_folder(BASE / "Code" / "FastSpeech2", DEST / "Code" / "FastSpeech2")
merge_folder(BASE / "English", DEST / "English")
merge_folder(BASE / "Telugu", DEST / "Telugu")
merge_folder(BASE / "Results", DEST / "Results")
merge_folder(BASE / "Report", DEST / "Report")

# Root-level documentation and audit files.
for filename in [
    "README.md",
    "requirements.txt",
    "submission_audit.json"
]:
    source = BASE / filename
    if source.is_file():
        shutil.copy2(source, DEST / filename)
        print("Copied:", filename)

# Include the original Colab notebook if present.
notebook = Path("/content/drive/MyDrive/Colab Notebooks/FastSpeech2_Assignment.ipynb")
if notebook.is_file():
    (DEST / "Notebooks").mkdir(exist_ok=True)
    shutil.copy2(notebook, DEST / "Notebooks" / notebook.name)
    print("Copied: Colab notebook")

# -------------------- GITIGNORE --------------------
gitignore = DEST / ".gitignore"
existing = gitignore.read_text(encoding="utf-8") if gitignore.exists() else ""
gitignore.write_text(existing + """

# Large archives and temporary files
*.zip
*.7z
*.rar
*.tmp
*.log
__pycache__/
.ipynb_checkpoints/
.DS_Store

# Do not publish raw dataset audio or local caches
raw_audio/
mfa_corpus/
""", encoding="utf-8")

# -------------------- CHECK FILE SIZES --------------------
oversized = []
for file in DEST.rglob("*"):
    if file.is_file() and ".git" not in file.parts:
        if file.stat().st_size > 95 * 1024 * 1024:
            oversized.append((file, file.stat().st_size / (1024 * 1024)))

if oversized:
    print("\\nFiles above the 95 MB safety limit:")
    for file, size in oversized:
        print(f"{size:.1f} MB  {file.relative_to(DEST)}")
    raise RuntimeError(
        "Push stopped to avoid GitHub's 100 MB per-file limit. "
        "These files need Git LFS or separate handling."
    )

# -------------------- COMMIT AND PUSH --------------------
run(["git", "config", "user.name", "Madhuri"])
run(["git", "config", "user.email", "madhuri2006@users.noreply.github.com"], cwd=DEST)

run(["git", "add", "-A"], cwd=DEST)

# Show what will be committed.
subprocess.run(["git", "status", "--short"], cwd=DEST, check=True, env=env)

status = subprocess.run(
    ["git", "diff", "--cached", "--quiet"],
    cwd=DEST,
    env=env
)

if status.returncode == 0:
    print("\\nNo new changes to commit. Checking/pushing existing branch.")
else:
    run([
        "git", "commit", "-m",
        "Add multilingual FastSpeech2 implementation and experiment results"
    ], cwd=DEST)

run(["git", "push", "origin", "main"], cwd=DEST)

# -------------------- VERIFY REMOTE COMMIT --------------------
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=DEST,
    env=env,
    text=True
).strip()

# Remove token from this cell's environment and delete temporary askpass.
env.pop("GITHUB_TOKEN", None)
os.environ.pop("GITHUB_TOKEN", None)
GITHUB_TOKEN = None
ASKPASS.unlink(missing_ok=True)

print("\\n" + "=" * 55)
print("GITHUB PUSH COMMAND COMPLETED")
print("Repository:", REPO_URL)
print("Branch: main")
print("Commit:", commit)
print("Open: https://github.com/madhuri2006/FastSpeech2-Multilingual-TTS")
print("=" * 55)

Enter your GitHub Personal Access Token (input is hidden).
It needs permission to write to this repository.
GitHub token: ··········
\n$ git clone --branch main https://github.com/madhuri2006/FastSpeech2-Multilingual-TTS.git /content/github_publish/FastSpeech2-Multilingual-TTS
Copied: FastSpeech2
Copied: English
